# The Whole Project, One Notebook

**Adaptive Quality-Aware Synthetic Data Selection (ASISM)** - master's thesis, AI, Zewail City
University. SDXL fine-tuned with LoRA on CheXpert chest X-rays, used to generate synthetic images,
which are then *adaptively selected* before classifier training and final evaluation.

## How to read this

**One cell = one file.** Every source file in the repository appears exactly once, in full, in a
cell of its own. Each file's cell is preceded by a short explanation of what it does, where it sits
in the pipeline, what it reads and writes, and anything about it that is easy to misread.

**The order is execution order, not alphabetical order.** Reading top to bottom walks the pipeline:
config -> shared utilities -> data -> Stage 1 -> Stage 2 -> Stage 3 -> Stage 4 -> Stage 5 -> smoke
-> tests -> cloud. The design documents are appended at the end, unmodified.

**The repository is the source of truth.** This notebook is generated from it by
`notebooks/build_full_project_notebook.py`. Re-run that script after changing any file - do not
edit source inside this notebook.

> **Do not execute a source cell in isolation.** Almost every script takes command-line arguments
> and enforces upstream artifact gates. Use the launcher cell below to run them properly, from the
> repository root. Source cells carry the tag `project-source`.

## The pipeline in one picture

```
                        configs/*.yaml  ------------ the only place tunables live
                              |
  scripts/data/00,01          v
  download + verify CheXpert ---> 02b: SIX patient-disjoint splits --> split_manifest_v2.json
                                        |                                   (everything
        +-------------------------------+                                 downstream is
        |                               |                                hashed against
        v                               v                                   this file)
  gen_train / gen_val          classifier_train / classifier_val
        |                               |        asism_tuning_heldout     final_eval_heldout
        |                               |                 |                       |
        v                               |                 |                       |
  03 preprocess --> 04 captions         |                 |                  (SEALED until
        |                               |                 |                    Stage 5)
        v                               |                 |                       |
  STAGE 1  train_lora_sdxl.py           |                 |                       |
        |  +--> frozen LoRA adapter     |                 |                       |
        v                               |                 |                       |
  STAGE 2  01 label recipes --> 02 generate (pilot gate)  |                       |
        |  +--> ~5k synthetic images + intended_label_vector                      |
        v                               |                 |                       |
  STAGE 3  classify/00 auxiliary classifier (gen_train / gen_val only)            |
        |  asism/00c empirically derived expected regions |                       |
        |  asism/01 six signals --> asism/02 GO/NO-GO gate|                       |
        |         |                                       |                       |
        |         v                                       |                       |
        |  LEARNED ASISM                                  |                       |
        |  04  design ~120 controlled subsets             |                       |
        |  04b MEASURE each subset's real utility --------+ (proxy training,      |
        |  05  set-utility net --> per-image ranker       |  evaluated here ONLY) |
        |  06  fixed-ratio threshold (ablation only)      |                       |
        |  07  bootstrap class contexts + grid search ----+                       |
        |  07b proxy-verify thresholds                    |                       |
        |  08  Adaptive Threshold Network + governance    |                       |
        |  08b proxy-verify whole POLICIES ---------------+                       |
        |  09  pre-registered tie rule --> ONE policy     |                       |
        |      +--> adaptive_selected_manifest.jsonl      |                       |
        v                                                                         |
  STAGE 4  classify/01    A: real only                                            |
                          B: real + ALL synthetic     identical protocol,         |
                          C: real + ASISM-selected    equal OPTIMIZER STEPS       |
        |                                                                         |
        v                                                                         v
  STAGE 5  eval/stage5_evaluate.py --- opens final_eval_heldout ONCE <------------ +
        +--> eval/compare_conditions.py --- Holm-Bonferroni / BH, patient bootstrap

  PRIMARY COMPARISON: C vs B - does SELECTING synthetic images beat using all of them?
```

### Four invariants to hold in your head while reading

1. **Image-disjoint everywhere.** Train-role and val-role subsets never share an image; threshold
   contexts come from one pool only.
2. **Proxy-only tuning.** Every Stage 3 decision is made on `asism_tuning_heldout`.
   `final_eval_heldout` is opened once, in Stage 5, behind a run-id guard.
3. **No pseudo-replication.** The ranker is never trained on a subset's AUROC copied onto its member
   images; it is distilled from measured *set-level* utility.
4. **Frozen manifests + hashes at every step.** A config change alters the config hash and forces a
   re-run from the affected stage forward.

## Launcher

The cell below finds the repository root and gives you `run_script()`, which runs the original
project scripts from the repository root with their normal command-line behaviour. Use it instead of
executing source cells.

In [ ]:
from pathlib import Path
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from the project checkout.')

REPO = find_repo_root()

def run_script(relative_path, *args, env=None):
    command = [sys.executable, str(REPO / relative_path), *map(str, args)]
    print('Running:', subprocess.list2cmdline(command))
    return subprocess.run(command, cwd=REPO, env=env, check=True)

print('Repository:', REPO)

## Safe local checks

Both switches default to `False` so nothing long-running starts by accident. The smoke fixture is
explicitly non-scientific and does not represent thesis results.

In [ ]:
RUN_TESTS = False
RUN_LOCAL_SMOKE = False

if RUN_TESTS:
    run_script('tests/run_all.py')
if RUN_LOCAL_SMOKE:
    run_script('scripts/smoke/run_smoke_pipeline.py', '--phase', 'local')
if not (RUN_TESTS or RUN_LOCAL_SMOKE):
    print('Nothing executed. Change a switch to True when ready.')

---

# Part 1 - The configuration layer

Every tunable in this project lives in YAML, not in code. Read these first: once you know what is configurable, the scripts read as plumbing around these numbers. Nothing here is loaded by hand - `scripts/utils/config.py` is the only reader.

## `configs/dataset_config.yaml`

*57 lines*

**What it is:** a description of the CheXpert dataset itself - not of the pipeline.

Holds the Kaggle slug used to download it, the expected row counts used as an integrity check,
the column names (`Path`, `Sex`, `Age`, `Frontal/Lateral`, `AP/PA`), and the canonical order of
the **14 pathology columns**. That order is the serialization order used everywhere else.

**Why it is a separate file:** it should stay stable even when training hyperparameters change.
A change here means "the dataset changed", which is a much bigger event than "I tuned the LR".

**Read by:** `scripts/utils/config.load_dataset_config()`, so every script sees the same schema.

```yaml
# CheXpert-v1.0-small schema and acquisition reference. Kept separate from stage1_lora_sdxl.yaml
# because this describes the *dataset itself*, not a tunable pipeline choice — it should stay
# stable even as training hyperparameters change.

source:
  provider: "kaggle"
  # Downloaded via kagglehub.dataset_download() by scripts/data/00_download_dataset.py.
  kaggle_dataset_slug: "ashery/chexpert"
  expected_train_rows: 224316
  expected_valid_rows: 234

schema:
  path_column: "Path"
  patient_id_regex: "patient\\d+"   # extracted from Path, e.g. CheXpert-v1.0-small/train/patient00001/...
  sex_column: "Sex"
  age_column: "Age"
  frontal_lateral_column: "Frontal/Lateral"
  ap_pa_column: "AP/PA"
  # All 14 pathology label columns, ternary-valued in {-1, 0, 1, <blank>}. Order here is the
  # canonical serialization order used by scripts/utils/caption_builder.py.
  pathology_columns:
    - "No Finding"
    - "Enlarged Cardiomediastinum"
    - "Cardiomegaly"
    - "Lung Opacity"
    - "Lung Lesion"
    - "Edema"
    - "Consolidation"
    - "Pneumonia"
    - "Atelectasis"
    - "Pneumothorax"
    - "Pleural Effusion"
    - "Pleural Other"
    - "Fracture"
    - "Support Devices"
  device_column: "Support Devices"  # handled separately from pathology findings in captions (plan §7)

expected_directory_structure: |
  data/chexpert/raw/
    train.csv
    valid.csv
    train/
      patient00001/
        study1/
          view1_frontal.jpg
          ...
    valid/
      patient64541/
        ...

integrity_checks:
  # Executed by scripts/data/01_verify_download.py — see docs/stage1_plan.md §6.
  - "train.csv row count within 1% of expected_train_rows"
  - "valid.csv row count matches expected_valid_rows"
  - "all pathology_columns present with values in {-1, 0, 1, NaN}"
  - "spot-check N random images open correctly and have plausible dimensions"
  - "per-label positive-rate roughly consistent with published CheXpert statistics"

```

## `configs/splits.yaml`

*63 lines*

**What it is:** the frozen six-way patient-level split policy - the most safety-critical config
in the project.

Six partitions are cut **directly** from one deterministic patient list with one seed; no split
is derived from another, and the fractions must sum to exactly 1.0 (asserted at build time):

| Split | Fraction | Used for |
|---|---:|---|
| `gen_train` | 0.48 | Stage 1 LoRA training; ASISM real-reference pool; auxiliary classifier |
| `gen_val` | 0.10 | Stage 1 monitoring; auxiliary classifier validation |
| `classifier_train` | 0.15 | real training data for every proxy and every headline classifier |
| `classifier_val` | 0.09 | model selection only (early stop, checkpoint, threshold, hparams) |
| `asism_tuning_heldout` | 0.09 | **all** Stage 3 tuning evidence |
| `final_eval_heldout` | 0.09 | Stage 5 only, opened once |

**The support rule** (`min_positive_patients: 50`, `min_negative_patients: 50`) is the reason the
fractions look odd. The original 5% decision-bearing splits left *Lung Lesion* and *Atelectasis*
short of 50 negative patients; raising those three splits to 9% fixed both. *Pleural Other* could
not be fixed by any fraction (only ~100 patients dataset-wide carry a confident negative for it),
so it was removed from the primary endpoint instead - see `scripts/utils/labels.py`.

**Namespacing** (`splits_root/<namespace>/`) is what makes it structurally impossible for a dev
subset split to overwrite a production split.

```yaml
# Six-way patient-level split policy (docs/stages2_to_5_plan.md §1) — FROZEN v1.
#
# Supersedes the schema-v1 three-way split in configs/stage1_lora_sdxl.yaml `split:` (which built
# gen_train/gen_val/classifier_heldout and derived asism_tuning_heldout/final_eval_heldout as
# children of classifier_heldout). That architecture left no population for a classifier
# development split. See docs/stages2_to_5_plan.md §1.1 for why deriving one from gen_train was
# rejected.
#
# The v1 artifacts are retained and marked superseded; they are never read by Stage 2-5 code.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  raw_dir: ${paths.data_root}/raw
  processed_dir: ${paths.data_root}/processed
  # Namespaced so a dev-subset split can never overwrite a production split (plan §1.4).
  splits_root: ${paths.processed_dir}/splits

manifest_version: 2

# Six DIRECT top-level partitions from one deterministic patient list and one seed.
# No split is derived from another. Must sum to exactly 1.0 (asserted at build time).
#
# Revised from the original 60/10/15/5/5/5 (docs/stages2_to_5_plan.md §1.2 revision note): on the
# full production CheXpert cohort, the original 5% decision-bearing fractions left "Lung Lesion"
# and "Atelectasis" short of the frozen ≥50-negative-patient support rule (§1.3) in
# classifier_val/asism_tuning_heldout/final_eval_heldout. Raising those three splits to 9% (taken
# from gen_train) clears both with margin. "Pleural Other" cannot be fixed by any fraction choice
# (only ~100 patients dataset-wide carry a confident negative label for it) and is excluded from
# PRIMARY_ENDPOINT_LABELS instead (scripts/utils/labels.py) rather than distorting the split policy
# further for an unfixable label.
fractions:
  gen_train: 0.48             # Stage 1 LoRA training; ASISM real-reference pool; auxiliary classifier
  gen_val: 0.10               # Stage 1 monitoring; auxiliary classifier validation
  classifier_train: 0.15      # A/B/C real training component; ASISM proxy real component
  classifier_val: 0.09        # A/B/C model selection ONLY (early stop, checkpoint, threshold, hparams)
  asism_tuning_heldout: 0.09  # ASISM Go/No-Go evidence + proxy-search evaluation
  final_eval_heldout: 0.09    # Stage 5 only

split_seed: 42
split_unit: "patient_id"      # never image level — CheXpert patients have multiple studies

# Minimum-support rule (plan §1.3) — FROZEN. Patient counts, not image counts, decide eligibility.
# Applied to every label in the primary endpoint set, in every split where that label drives a
# decision or a metric.
support_rule:
  min_positive_patients: 50
  min_negative_patients: 50
  decision_bearing_splits:
    - classifier_train
    - classifier_val
    - asism_tuning_heldout
    - final_eval_heldout

# Production namespace requires the source CSV to match the full-cohort integrity contract in
# configs/dataset_config.yaml (source.expected_train_rows) within this tolerance. The dev namespace
# skips that requirement and marks rare-label support insufficiency in its manifest instead.
namespaces:
  production:
    require_full_cohort: true
    full_cohort_row_tolerance_pct: 1.0
  dev:
    require_full_cohort: false

```

## `configs/stage1_lora_sdxl.yaml`

*152 lines*

**What it is:** every tunable for Stage 1 (SDXL + LoRA fine-tuning), plus the `paths:` block that
the whole repository resolves against `PROJECT_ROOT`.

Notable sections: `paths` (everything that must survive a pod restart lives under `PROJECT_ROOT`),
`lora` (rank/alpha, UNet-only targets), `training` (precision, gradient accumulation/checkpointing,
latent+embedding caching, EMA), `optimizer`, `checkpointing`, `validation`, `captions`, `data`
(resolution, letterbox padding, quality thresholds), `dev_subset`, and the legacy `split:` block.

**One thing to know:** the `split:` block here is the **schema-v1 three-way** policy
(gen_train / gen_val / classifier_heldout). It is superseded by `configs/splits.yaml`. Stage 2-5
code never reads it; only `scripts/data/02_build_patient_splits.py` (itself superseded) does.

```yaml
# Stage 1 — SDXL + LoRA fine-tuning on CheXpert. Single source of truth for tunables.
# See docs/stage1_plan.md for the full rationale behind every value here.
#
# All paths are relative to PROJECT_ROOT, which defaults to the current working directory.
# On RunPod, set PROJECT_ROOT=/workspace/chest-synth-thesis (or wherever this repo is cloned on
# the persistent volume) before launching any script, so nothing resolves onto the ephemeral
# container disk.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  raw_dir: ${paths.data_root}/raw
  processed_dir: ${paths.data_root}/processed
  images_dir: ${paths.processed_dir}/images_768
  splits_dir: ${paths.processed_dir}/splits
  captions_dir: ${paths.processed_dir}/captions
  checkpoints_dir: ${paths.project_root}/checkpoints/stage1_lora_sdxl
  logs_dir: ${paths.project_root}/logs/stage1_lora_sdxl
  outputs_dir: ${paths.project_root}/outputs/stage1_samples
  hf_cache_dir: ${paths.project_root}/.cache/huggingface

run:
  # run_id is composed at launch time as "<timestamp>_<git_sha>_<config_hash>" by scripts/utils/manifest.py.
  # Leave null here; set explicitly only when resuming a specific prior run.
  run_id: null
  seed: 42

dev_subset:
  # Fast-iteration development subset (docs/stage1_plan.md's "small subset" made concrete/configurable).
  # This ONLY exists to validate the pipeline quickly; the full CheXpert dataset generates final
  # thesis results. Never hardcode a sample size in code — this is the single source of truth for it.
  enabled: false            # production-thesis-v1 runs on the full cohort; 01b_build_dev_subset.py
                            # is a no-op. Set true only for a dev-subset run.
  target_size: 10000        # approximate output image count; configurable (5k/10k/20k/...).
  seed: 42                  # independent of split_seed so subset draws can vary without perturbing
                             # the patient-splitting logic (now the six-way policy in
                             # configs/splits.yaml, not the superseded v1 70/10/20).
  prevalence_tolerance_pct: 5.0   # report-only threshold; deviations beyond this are WARNINGs, not
                                   # fatal — the subset only needs to be "reasonably representative."
  output_csv: ${paths.processed_dir}/dev_subset/train_subset.csv
  manifest_path: ${paths.processed_dir}/dev_subset/dev_subset_manifest.json
  prevalence_report_path: ${paths.processed_dir}/dev_subset/prevalence_report.json

model:
  base_model_id: "stabilityai/stable-diffusion-xl-base-1.0"
  revision: "462165984030d82259a11f4367a4eed129e94a7b"
  vae_upcast_fp32: true  # avoids the known SDXL-VAE fp16 NaN/black-image bug (plan §2)
  use_refiner: false

lora:
  rank: 32
  alpha: 32
  dropout: 0.0
  target_modules: ["to_q", "to_k", "to_v", "to_out.0"]  # UNet self+cross attention projections only
  train_text_encoders: false  # documented fallback: set true (+ text_encoder_lr) if rare-label conditioning is weak

optimizer:
  name: "adamw_8bit"        # falls back to "adamw" automatically if bitsandbytes/CUDA mismatch is detected
  learning_rate: 1.0e-4
  lr_scheduler: "cosine"
  lr_warmup_steps: 750
  weight_decay: 1.0e-2
  max_grad_norm: 1.0

data:
  view_filter: "frontal"   # "frontal" | "both" — lateral images are flagged, not deleted, when filtered
  resolution: 768
  aspect_mode: "letterbox" # aspect-preserving resize + minimal pad; never non-uniform stretch, never center-crop
  normalize_mean: 0.5
  normalize_std: 0.5
  horizontal_flip: false   # MUST stay false — flipping inverts CXR left/right anatomy
  min_source_resolution: 200  # px, below this an image is flagged/dropped as low quality

split:
  # MIXED BLOCK -- read this before changing anything here.
  #
  # LIVE keys, read by the current pipeline:
  #   namespace        -> 03_preprocess_images.py, 04_generate_captions.py, train_lora_sdxl.py
  #   input_csv        -> 02b_build_sixway_splits.py (default when --input-csv is omitted)
  #   manifest_version, split_unit -> documentation of the v2 contract
  #
  # SUPERSEDED keys, read ONLY by scripts/data/02_build_patient_splits.py (the schema-v1
  # three-way builder, itself not a pipeline step): gen_train_fraction, gen_val_fraction,
  # classifier_heldout_fraction, asism_tuning_fraction, heldout_split_seed. The fractions the
  # pipeline actually uses are the six-way ones in configs/splits.yaml. These are retained, not
  # deleted, so the v1 artifacts stay reproducible -- but editing them changes nothing in
  # Stage 1-5.
  #
  # There is deliberately no fallback to the superseded root-level schema-v1 split files.
  namespace: production-thesis-v1
  manifest_version: 2
  input_csv: ${paths.raw_dir}/train.csv   # FULL COHORT: the ~224k-image CheXpert train.csv. For a
                                            # dev-subset run use ${dev_subset.output_csv} (the ~10k
                                            # train_subset.csv from 01b_build_dev_subset.py) — a
                                            # config-only switch, no code change required.
  gen_train_fraction: 0.70
  gen_val_fraction: 0.10
  classifier_heldout_fraction: 0.20
  split_seed: 42
  split_unit: "patient_id"  # never split at image level — CheXpert patients have multiple studies
  # classifier_heldout is further split so ASISM's weight/threshold search (Stage 3) never tunes
  # against the same data Stage 5 reports its headline numbers on (leakage/optimistic-bias risk).
  asism_tuning_fraction: 0.35   # fraction of classifier_heldout patients used only by Stage 3's search
  heldout_split_seed: 43        # separate seed from split_seed so this sub-split can be reseeded
                                 # independently of the top-level gen_train/gen_val/heldout split

captions:
  template_version: "v1"
  num_paraphrase_variants: 4
  age_bucket_width_years: 10
  uncertain_label_policy: "omit"  # -1 labels are omitted from the finding clause, not asserted present/absent
  no_finding_overrides_positives: true  # treat "No Finding"==1 as authoritative over spurious co-positive labels

training:
  precision: "bf16"
  attention_backend: "sdpa"  # PyTorch 2.x scaled_dot_product_attention; xformers intentionally not used (plan §5)
  train_batch_size: 8
  gradient_accumulation_steps: 2
  gradient_checkpointing: true
  cache_latents_and_embeddings: true
  ema:
    enabled: true
    decay: 0.9999   # upper bound; with warmup the effective decay is min(decay, (1+k)/(10+k))
    warmup: true    # false = fixed decay from step 1, which dilutes the final LoRA toward its
                    # zero init (~82% after 2k steps, ~22% after 15k); see LoraEMA
  max_train_steps: 30000    # upper bound; actual stop driven by val-loss plateau + stable probe samples (plan §9)
  min_train_steps: 15000

checkpointing:
  save_every_n_steps: 1000
  keep_last_n_full_checkpoints: 3
  keep_all_lora_weight_snapshots: true

validation:
  val_every_n_steps: 1000
  # Validation loss is a monitoring signal only (checkpoint choice uses probe samples + FID), so it
  # runs on a fixed seeded subset of gen_val instead of all of it: the full-cohort gen_val is ~19k
  # images, ~35 min per pass on an A40 and ~17 GPU-hours over 30k steps. null = all of gen_val.
  max_val_samples: 1000
  val_seed: 2024            # fixes the subset draw AND each pass's latent/noise/timestep draws, so
                            # val loss is comparable across steps; training RNG is left untouched
  probe_seed: 123
  fid:
    inception: true
    domain_torchxrayvision: true
  clip_score: true

logging:
  tensorboard: true
  wandb:
    enabled: false  # set true + WANDB_API_KEY env var to enable the optional supplementary tracker
    project: "chexpert-sdxl-lora-stage1"

```

## `configs/stage2_generation.yaml`

*75 lines*

**What it is:** the Stage 2 synthetic-generation contract - label recipes and the sampler.

`recipes`: rarity-aware per-class quotas (`base_quota`, `min_per_label`, `max_per_label`),
co-occurrence support thresholds, medical block rules (combinations that must never be generated),
and a cap on positives per recipe. `generation`: the diffusion sampler (DPM-Solver multistep,
~40 steps, guidance ~ 7, resolution, batch size, seed). `checkpoint.lora_weights_dir`: the
**explicitly pinned** Stage 1 adapter - there is no "latest checkpoint" magic, because a silently
moving generator would invalidate every downstream artifact hash. `pilot`: the human-review gate
that must be approved before a full generation run is permitted.

```yaml
# Stage 2 — synthetic generation (docs/stages2_to_5_plan.md §3).
#
# Conditioning labels are `intended_label_vector` — what the generator was ASKED for, never
# "ground truth". Stage 3 §4.5 measures whether the pixels agree; nothing here establishes it.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  synthetic_root: ${paths.data_root}/synthetic
  images_dir: ${paths.synthetic_root}/images
  recipes_path: ${paths.synthetic_root}/label_recipes.csv
  recipe_decisions_path: ${paths.synthetic_root}/recipe_decisions.jsonl
  manifest_path: ${paths.synthetic_root}/generation_manifest.jsonl
  pilot_dir: ${paths.synthetic_root}/pilot
  pilot_approval_manifest: ${paths.synthetic_root}/pilot/pilot_approval_manifest.json

split_namespace: "production-thesis-v1"  # immutable run ID; the smoke overlay sets dev-smoke-v1

checkpoint:
  # Explicit LoRA checkpoint — NOT "latest". Generation must be reproducible and decoupled from
  # any training still in progress (docs/stage1_plan.md §11 handoff contract).
  lora_weights_dir: null      # e.g. checkpoints/stage1_lora_sdxl/<run_id>/lora_weights/step_20000
  base_model_id: "stabilityai/stable-diffusion-xl-base-1.0"
  revision: "462165984030d82259a11f4367a4eed129e94a7b"

generation:
  resolution: 768             # must match Stage 1 data.resolution
  num_inference_steps: 40
  guidance_scale: 7.0
  scheduler: "DPMSolverMultistep"
  batch_size: 4
  seed: 20260807              # base seed; per-image seed is derived deterministically from image_id
  negative_prompt: ""

recipes:
  # Empirical co-occurrence mining from gen_train (§3.1)
  min_support_patients: 25          # a label combination must occur in >= this many real patients
  # Target recipe mix (§3)
  single_label_proportion: 0.55     # fraction of disease recipes carrying exactly one positive label
  multi_label_proportion: 0.45
  no_finding_proportion: 0.20       # fraction of ALL recipes that are the all-zero (No Finding) vector
  max_positive_labels: 3            # cap on simultaneous positive disease labels per recipe

  # Per-label generation quota, oversampling rare classes relative to real prevalence (§3.7).
  # target_count = base_quota * (median_prevalence / label_prevalence) ** rarity_exponent,
  # clipped to [min_per_label, max_per_label]. rarity_exponent=0 disables oversampling.
  base_quota: 400
  rarity_exponent: 0.5
  min_per_label: 100
  max_per_label: 1500

  # Demographic coverage: recipes are sampled across these buckets rather than left incidental.
  age_buckets: [20, 30, 40, 50, 60, 70, 80]
  sexes: ["Male", "Female"]
  balance_demographics: true

  # Support Devices is carried as a context attribute only. It never affects the primary
  # disease-agreement score (§4.5) unless secondary_agreement is explicitly enabled in Stage 3.
  support_devices_proportion: 0.35

  # Medical-rule overrides (§3.2). Both directions, both auditable.
  # allow: clinically plausible combinations permitted even below min_support_patients.
  # block: combinations judged likely CheXpert NLP-extraction noise, rejected even if observed.
  medical_rule_allow: []
  medical_rule_block: []

pilot:
  num_images: 48              # small spanning batch for manual visual review before full generation
  # Automatic artifact checks the pilot must pass before it can be approved.
  checks:
    min_valid_image_fraction: 0.95
    require_correct_resolution: true
    require_rgb: true
    max_near_uniform_fraction: 0.02   # near-blank outputs indicate a broken generation config
    blank_std_threshold: 5.0

```

## `configs/stage3_asism.yaml`

*355 lines*

**What it is:** the largest config in the project - all of ASISM. Worth reading slowly, because
almost every number in Stage 3 is here rather than in code.

- `auxiliary_classifier`: architecture/resolution/dropout of the reference classifier that the
  uncertainty, explainability and agreement signals query.
- `signals`: per-signal parameters - k for the similarity k-NN, the reference sampling policy and
  hierarchical fallback threshold, the `near_duplicate_similarity: 0.95` memorisation cut-off,
  IQA thresholds, MC-Dropout pass count, Grad-CAM settings, the agreement penalty.
- `gonogo`: the thresholds each signal must clear to be admitted (missingness, distinct values,
  directionality, redundancy correlation, reproducibility).
- `tuning` / `selection`: the legacy weighted-score selector's search space (used only by
  `03_tune_freeze_select.py`, which is **not** a pipeline step).
- `learned_asism`: the actual thesis contribution - `feature_columns` (the 9 admitted feature
  columns), `subset_design` (sizes, counts, feasibility thresholds), `set_utility_network`,
  `ranking_network` (including `target_source` and the C2 target-normalization options), and
  `threshold_network` (bootstrap contexts, verification budgets, acceptance criteria, the
  full-policy variant list and tie rule).

Every compute-spending step has its own `compute_budget` with `max_gpu_hours` - a gate that must
pass **before** any results are seen, so a search space can never be widened after looking at
performance.

```yaml
# Stage 3 — ASISM (docs/stages2_to_5_plan.md §4).
#
# Five independent signals, each written to its own versioned parquet artifact, each subject to the
# SAME Go/No-Go gate (§4.6). No signal is admitted to the frozen selector merely because it was
# implemented.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  data_root: ${paths.project_root}/data/chexpert
  synthetic_root: ${paths.data_root}/synthetic
  scores_dir: ${paths.synthetic_root}/scores
  asism_dir: ${paths.synthetic_root}/asism
  gonogo_report: ${paths.synthetic_root}/asism/gonogo_report.json
  search_log: ${paths.synthetic_root}/asism/proxy_search_log.jsonl
  frozen_manifest: ${paths.synthetic_root}/asism/asism_frozen_manifest.json
  fold_manifest: ${paths.synthetic_root}/asism/asism_patient_folds.json
  selected_manifest: ${paths.synthetic_root}/asism/selected_manifest.jsonl
  rejected_log: ${paths.synthetic_root}/asism/rejected_log.jsonl
  learned_dir: ${paths.synthetic_root}/asism/learned
  subset_design_report: ${paths.synthetic_root}/asism/learned/subset_design_report.json
  utility_subsets: ${paths.synthetic_root}/asism/learned/utility_subsets.jsonl
  utility_results: ${paths.synthetic_root}/asism/learned/utility_results.jsonl
  learned_selected_manifest: ${paths.synthetic_root}/asism/learned/selected_manifest.jsonl
  adaptive_selected_manifest: ${paths.synthetic_root}/asism/learned/adaptive_selected_manifest.jsonl
  adaptive_selection_manifest: ${paths.synthetic_root}/asism/learned/adaptive_selection_manifest.json
  learned_rejected_log: ${paths.synthetic_root}/asism/learned/rejected_log.jsonl
  threshold_contexts: ${paths.synthetic_root}/asism/learned/threshold_contexts.jsonl
  threshold_candidate_evaluations: ${paths.synthetic_root}/asism/learned/threshold_candidate_evaluations.jsonl
  proxy_verification_plan: ${paths.synthetic_root}/asism/learned/proxy_verification_plan.json
  # Written ONLY by 07b, NEVER by 07 — critic-only evaluations and proxy measurements are separate,
  # immutable files.
  threshold_proxy_measurements: ${paths.synthetic_root}/asism/learned/threshold_proxy_measurements.jsonl
  # Verified (official) and exploratory training targets are SEPARATE files, not one file with a
  # "path" column — a loader that opens the wrong file entirely fails on a missing path, rather than
  # a loader that reads the right file but forgets to filter by "path" silently mixing them.
  threshold_training_targets_verified: ${paths.synthetic_root}/asism/learned/threshold_training_targets_verified.jsonl
  threshold_training_targets_exploratory: ${paths.synthetic_root}/asism/learned/threshold_training_targets_exploratory.jsonl
  official_threshold_checkpoint: ${paths.synthetic_root}/asism/learned/official_threshold_model.pt
  exploratory_threshold_checkpoint: ${paths.synthetic_root}/asism/learned/exploratory_threshold_model.pt
  adaptive_threshold_manifest: ${paths.synthetic_root}/asism/learned/adaptive_threshold_manifest.json
  full_policy_verification_plan: ${paths.synthetic_root}/asism/learned/full_policy_verification_plan.json
  full_policy_verification_results: ${paths.synthetic_root}/asism/learned/full_policy_verification_results.jsonl

split_namespace: "production-thesis-v1"  # the smoke overlay sets dev-smoke-v1

auxiliary_classifier:
  checkpoint_path: ${paths.project_root}/checkpoints/auxiliary_classifier/auxiliary_classifier.pt
  resolution: 320
  dropout_p: 0.2
  pretrained_source: "imagenet"   # CheXpert-pretrained is rejected in code (§2)

signals:
  similarity:
    encoder: "vit_small_patch14_dinov2"  # architecture; pinned weights are declared below
    weights_repo_id: "timm/vit_small_patch14_dinov2.lvd142m"
    weights_filename: "model.safetensors"
    weights_revision: "936966a8732c5442c9def5d126f2cc4ad4243dba"
    batch_size: 32
    k_neighbors: 15               # k-NN aggregation, never single-nearest-neighbor
    reference_sample_per_label: 400
    min_exact_match_references: 25    # below this, fall back down the hierarchy (§4.1)
    # Memorization flag: high single-NN similarity with low rank-k spread means the image is
    # suspiciously close to ONE real training image rather than to the class in general.
    # The 0.95 cutoff is literature-grounded, not hand-tuned: DINOv2 cosine >= 0.95 == "almost
    # identical" for medical latent-diffusion output, validated against real patient replication by
    # Dar et al., "Unconditional latent diffusion models memorize patient imaging data", Nature
    # Biomedical Engineering (2025), doi:10.1038/s41551-025-01468-8.
    near_duplicate_similarity: 0.95
    near_duplicate_spread: 0.02

  distinctiveness:
    # WITHIN-SYNTHETIC redundancy, reported as distinctiveness (higher = more unique = better, the
    # same direction as every other admitted signal). Distinct from similarity's novelty score,
    # which measures closeness to REAL images; this measures closeness to OTHER SYNTHETIC images.
    # Neighbours are drawn ONLY from images sharing the exact intended_label_vector: comparing
    # across classes would turn the score into a proxy for the Stage 2 per-class quota (a common
    # class looks redundant, a rare one looks distinct) and would then feed the class-aware
    # threshold that is itself correcting class imbalance.
    k_neighbors: 10
    # Pairs at or above this cosine similarity are treated as near-identical. Used both for the
    # duplicate COUNT reported per image and for the pre-scoring deduplication clusters, because a
    # per-image threshold provably cannot choose one representative out of a duplicate cluster.
    duplicate_similarity: 0.95
    # Deduplication keeps one representative per cluster BEFORE signals are scored. Disabled by
    # default: enabling it changes the candidate pool itself, so it must be an explicit decision.
    deduplicate_before_scoring: false

  iqa:
    # Generic NR-IQA is a BASELINE component; the domain checks below carry the signal (§4.2).
    blank_std_threshold: 5.0
    clipping_percentile: 0.995
    low_contrast_std: 15.0
    border_uniform_fraction: 0.30
    laplacian_blur_threshold: 40.0

  uncertainty:
    mc_dropout_passes: 20
    # Band edges on predictive std over the 11 primary labels (§4.3). Moderate uncertainty is NOT
    # penalized: it may be the informative band (TSynD's premise).
    low_band_max: 0.05
    moderate_band_max: 0.15

  explainability:
    method: "gradcam"
    # Fixed central-thorax box is an explicit BASELINE (§4.4). Pathology-specific expected regions
    # are relative (x0, y0, x1, y1) fractions of the image.
    region_mode: "pathology_specific"
    baseline_box: [0.20, 0.15, 0.80, 0.85]
    pathology_regions:
      Cardiomegaly: [0.30, 0.40, 0.75, 0.85]
      Enlarged Cardiomediastinum: [0.30, 0.30, 0.75, 0.80]
      Pleural Effusion: [0.10, 0.60, 0.90, 0.95]
      Pneumothorax: [0.05, 0.15, 0.95, 0.70]
      Edema: [0.15, 0.25, 0.85, 0.80]
      Consolidation: [0.10, 0.25, 0.90, 0.85]
      Pneumonia: [0.10, 0.25, 0.90, 0.85]
      Atelectasis: [0.10, 0.30, 0.90, 0.90]
      Lung Opacity: [0.10, 0.20, 0.90, 0.85]
      Lung Lesion: [0.10, 0.15, 0.90, 0.85]
      Pleural Other: [0.05, 0.50, 0.95, 0.95]
      Fracture: [0.05, 0.10, 0.95, 0.60]

  agreement:
    # agreement = mean P(intended positives) - penalty * mean P(confident unintended positives)
    # For a No Finding recipe (all-zero intended vector) this reduces to: all 12 probabilities low.
    unintended_confidence_threshold: 0.5
    penalty_weight: 0.5
    # Support Devices stays OUT of the primary disease-agreement score (§3.4, §4.5).
    secondary_agreement:
      enabled: false
      labels: ["Support Devices"]

gonogo:
  # A signal must clear every applicable check to enter the frozen selector (§4.6).
  max_missing_fraction: 0.05
  min_unique_values: 20            # a near-constant score carries no information
  directionality_probe_size: 200
  redundancy_abs_correlation_max: 0.95   # above this, a signal is near-duplicate of another
  reproducibility_tolerance: 1.0e-6

tuning:
  # Two-stage bounded search (§4.7) for the SUPERSEDED weighted-score selector
  # (03_tune_freeze_select.py, not a pipeline step). Proxy trains on classifier_train +
  # candidate synthetic subset, evaluates on asism_tuning_heldout ONLY. classifier_val is NOT
  # used to rank candidates.
  #
  # DO NOT DELETE THIS BLOCK. Two sub-keys are load-bearing for the LEARNED pipeline, which
  # reads its proxy-classifier settings from here rather than from learned_asism:
  #   tuning.proxy.*                 -> 04b, 07b, 08b (batch_size, learning_rate, resolution,
  #                                     dropout_p, pretrained_source)
  #   tuning.coarse.proxy_max_steps  -> 04b, 07b, 08b (the training length every measured
  #                                     subset utility is computed at)
  # Everything else here (coarse.trials/folds/seeds, the whole shortlist block, and
  # compute_budget below) is read only by 03 and therefore currently unused.
  coarse:
    trials: 12
    folds: 1
    seeds: 1
    proxy_max_steps: 300
  shortlist:
    size: 4
    folds: 3
    seeds: 2
    proxy_max_steps: 800
  proxy:
    batch_size: 32
    learning_rate: 1.0e-4
    resolution: 320
    dropout_p: 0.2
    pretrained_source: "imagenet"
  # Compute-budget gate: if the pre-execution estimate exceeds this, the search space must be
  # reduced BEFORE any results are seen — never adjusted afterward.
  # UNUSED: this gate belongs to 03_tune_freeze_select.py only. The learned pipeline's GPU
  # gates are learned_asism.compute_budget (04b),
  # learned_asism.threshold_network.verification_compute_budget (07b), and
  # learned_asism.threshold_network.full_policy_verification.compute_budget (08b) — none of
  # which is ever summed with this one.
  compute_budget:
    max_gpu_hours: 8.0
    hours_per_proxy_run_estimate: 0.15
  # Ties within this band resolve to the SIMPLER configuration (fewer signals, less extreme weights).
  tie_noise_band: 0.002
  search_seed: 42

selection:
  # Ordered policy (§4.8): quality floor FIRST, then class quota. Rarity never bypasses the floor.
  quality_floor_percentile: 25      # reject the bottom N% by composite score, absolutely
  target_synthetic_to_real_ratio: 1.0
  min_accepted_per_label: 50
  max_accepted_per_label: 2000
  uncertainty_band_preference: "moderate"   # "any" | "low" | "moderate"

# Learned ASISM is an additional proposed method. It does not replace the weighted baseline above.
learned_asism:
  enabled: true
  feature_columns:
    - similarity_knn_mean
    - similarity_top1
    - similarity_topk_spread
    - iqa_composite
    - iqa_sharpness
    - iqa_contrast_std
    - uncertainty_mean_std
    - explainability_region_overlap
    - agreement_score
  subset_design:
    # The validation image pool is 20% and a single-signal subset draws from
    # one of three quantile bands without replacement.  With the Stage-2
    # quota's expected ~5k candidates, 1000 would require ~15k candidates.
    # These sizes keep the largest single-band validation draw feasible while
    # retaining small/medium/large utility-set variation.
    subset_sizes: [100, 200, 250]
    total_subsets: 120
    random_fraction: 0.20
    single_signal_fraction: 0.30
    mixed_fraction: 0.50
    quantile_bins: 3
    minimum_image_exposures: 3
    seed: 42
    # val_pool_fraction_status = provisional_default (see pool_feasibility_report's own field of the
    # same name, which is what --phase build actually checks). This YAML value is only a starting
    # point for --phase feasibility; the number that governs --phase build is whatever the
    # feasibility report most recently computed against the REAL candidate pool. Never edited to
    # "make the gate pass" after seeing a failing report — the report drives the number, not the
    # other way around.
    val_pool_fraction: 0.20
    # --phase build refuses to run unless the most recent --phase feasibility report (a) matches
    # this config (config_hash check) and (b) shows zero failures against these thresholds. No
    # ratio or size is ever adjusted automatically/silently to make a failing report pass.
    feasibility_thresholds:
      min_candidates_per_class: 20
      min_candidates_per_quantile_band: 20
      min_val_subsets: 10
  set_utility_network:
    image_hidden_dims: [128, 64]
    utility_hidden_dims: [32]
    pooling: "mean"
    # Ablation only (Phase 11), default false = frozen mean-only pipeline. When true the set is
    # summarized by masked mean AND std pooling and a fixed full-pool summary is concatenated to the
    # utility head (Xie et al., ICLR 2024). Caveat: the (n-1) marginal-target size-normalization in
    # 05_train_learned_asism.py is only first-order correct for the mean component under this flag.
    superset_conditioning: false
    epochs: 100
    batch_size: 8
    learning_rate: 1.0e-4
    weight_decay: 1.0e-5
    utility_mse_weight: 1.0
    pairwise_ranking_weight: 0.5
    early_stopping_patience: 12
  ranking_network:
    # MultiSignalUtilityRankingNetwork (models.py): admitted signal feature columns
    # (9 with the default feature_columns below) -> 128 -> 64 -> 32 -> 1 scalar utility.
    # One target, two loss terms (Smooth-L1 + pairwise ranking). NOT multi-objective.
    hidden_dims: [128, 64, 32]
    dropout: 0.20
    epochs: 100
    batch_size: 128
    learning_rate: 1.0e-4
    pairwise_ranking_weight: 0.5
    # Ranking-target generator. "leave_one_out" = mean size-normalized LOO marginal set-utility (the
    # frozen default). "banzhaf_msr" = Data-Banzhaf value from the SAME measured subsets (Wang & Jia,
    # AISTATS 2023) — no extra proxy training, no GPU. 05 always computes both and records their
    # Spearman rank agreement (learned_training_manifest.json -> target_generation); banzhaf_msr stays
    # opt-in until that diagnostic is reviewed.
    target_source: "leave_one_out"
    # Transform the distilled per-image target before the Smooth-L1 + pairwise loss so one noisy
    # proxy-subset measurement cannot dominate training on ~96 images (2D-OOB, NeurIPS 2024; Chi et
    # al., ICML 2026; Banzhaf robustness rationale). Every option is order-preserving.
    # "standardize" (default) | "rank" | "none". target_winsorize_quantile clips each tail (0.0 = off).
    target_normalization: "standardize"
    target_winsorize_quantile: 0.0
  threshold_network:
    # Architecture (AdaptiveThresholdNetwork, trained via supervised distillation in
    # 08_train_threshold_network.py — NOT the soft-gate end-to-end design; that path was rejected
    # after review, so no temperature-annealing knobs remain here).
    class_embedding_dim: 16
    hidden_dims: [64, 32]
    dropout: 0.10
    epochs: 80
    learning_rate: 1.0e-4
    # Hard grid search objective (07_build_threshold_contexts.py): objective(t) =
    # critic_predicted_utility(t) - budget_weight * (selected_count / candidate_count).
    budget_weight: 0.25
    min_selected_per_label: 50
    max_selected_per_label: 2000
    target_synthetic_to_real_ratio: 1.0
    # Context generation (07): bootstrap resamples per class, honestly tagged as NOT independent
    # clinical samples — see bootstrap_class_contexts's docstring.
    n_bootstrap_contexts_per_class: 25
    bootstrap_min_fraction: 0.30
    bootstrap_max_fraction: 1.00
    t_grid_step: 0.01
    # Proxy verification candidates (07 plans, 07b executes — NOT run in this pass): critic top-K is
    # NEVER the only source, or verification would be circular (only ever checking where the critic
    # already thinks is good). diversify_verification_candidates also adds quantile-spaced
    # selection-count targets, the two boundary (smallest/largest) candidates, and the OLD
    # baseline's own ~50%-of-pool operating point.
    top_k_per_context: 2
    verification_quantile_fractions: [0.10, 0.30, 0.50, 0.70, 0.90]
    verification_compute_budget:
      max_gpu_hours: 20.0
      hours_per_proxy_run_estimate: 0.15
    # Full-policy verification (08's plan_full_policy_verification): PLAN ONLY. Trains the ACTUAL
    # final multi-label selected_manifest for each candidate policy (fixed-ratio-baseline,
    # hard-proxy-best, network-where-eligible, ...) — its OWN independent GPU budget, separate from
    # verification_compute_budget above, because it is a distinct, separately-approved production
    # execution step (08b_verify_full_policy_proxy.py --phase run, not run in this pass).
    # n_variants/n_seeds are provisional until 08 has run and the real number of policy variants is
    # known.
    # Training-free FreeMatch-style class-adaptive percentile threshold
    # (learned.py::freematch_style_percentile_per_class). Registered as a competing full-policy
    # variant (freematch_style_adaptive_percentile), NEVER a replacement for the learned network.
    # Leniency is RANGE-normalised across the observed per-class real prevalences, and a hard
    # per-class floor (threshold_network.min_selected_per_label, via enforce_per_class_selection_floor
    # in 08b/09) is applied after thresholding so a rare class is never selected down to zero — the
    # one-time-curation analogue of SST's class-fairness term (Zhao et al., IP&M, 2025).
    freematch_style:
      base_percentile: 50.0
      min_percentile: 10.0
    full_policy_verification:
      n_variants: 5
      n_seeds: 3
      compute_budget:
        max_gpu_hours: 10.0
        hours_per_proxy_run_estimate: 0.15
      finalization:
        tie_noise_band: 0.002
        simplicity_order:
          - fixed_target_ratio_threshold_distillation_baseline_v1
          - hard_proxy_best_among_verified
          - adaptive_threshold_network
          - literal_top_50_percent
          - freematch_style_adaptive_percentile
    # Governance (08) — three-way, no other outcome possible:
    #   ZERO verified TRAIN contexts (regardless of held-out)           -> fixed-ratio baseline
    #   verified train contexts, but below this minimum on either side  -> hard_proxy_best_among_verified
    #     (train or image-disjoint held-out)
    #   >= this minimum on BOTH sides AND acceptance_criteria pass      -> adaptive_threshold_network
    min_verified_contexts_per_class: 3
    # Frozen BEFORE any held-out evaluation is run (08's freeze_acceptance_criteria) — tagged
    # provisional_pre_registered_defaults and NEVER adjusted after seeing results. Units/definitions
    # for each key are in scripts/asism/learned.py's CRITERIA_DEFINITIONS.
    acceptance_criteria:
      critic_proxy_median_spearman_min: 0.30
      median_critic_predicted_utility_regret_max: 0.01
      threshold_stability_std_max: 0.05
  safety:
    reject_invalid_iqa: true
    reject_near_duplicates: true
  # Compute-budget gate for 04b_evaluate_utility_subsets.py, mirroring tuning.compute_budget above
  # but scoped independently: this search evaluates subset_design.total_subsets recipes, not the
  # weighted baseline's coarse/shortlist trial count.
  compute_budget:
    max_gpu_hours: 20.0
    hours_per_proxy_run_estimate: 0.15

```

## `configs/stage4_classifier.yaml`

*65 lines*

**What it is:** the frozen Stage 4 protocol. The point of this file is that architecture,
initialization, label policy, uncertainty policy, optimizer, batch size, augmentation, checkpoint
selection, validation rule and threshold rule are **identical across conditions**. Only the
training data differs.

- `conditions: [A, B, C]` - A = real only, B = real + all synthetic, C = real + ASISM-selected.
- `training.max_steps` - the budget is in **optimizer steps, not epochs**. At fixed epochs a larger
  dataset silently gets more gradient updates, which would conflate "more data" with "more
  training". This matters most for B (all synthetic) vs C (a selected subset).
- `augmentation: "none"` - horizontal flip is forbidden; it inverts chest-X-ray anatomy.
- `seeds: [42, 43, 44]` -> 3 conditions x 3 seeds = 9 runs.
- `condition_d` - **inactive but retained**. The matched-random control is the documented fix for
  the one structural gap in the comparison (see the Stage 4 script's KNOWN LIMITATION note).
  Re-enabling it needs supervisor sign-off and `n_draws x 3` extra runs.

```yaml
# Stage 4 — thesis conditions A/B/C. FROZEN protocol.
#
# Architecture, initialization, label policy, uncertainty policy, optimizer family, batch size,
# augmentation, checkpoint-selection rule, validation rule, and threshold-selection rule are
# IDENTICAL across all conditions. Only the training data differs.

paths:
  project_root: ${oc.env:PROJECT_ROOT,.}
  checkpoints_dir: ${paths.project_root}/checkpoints/stage4_conditions
  results_dir: ${paths.project_root}/outputs/stage4
  protocol_manifest: ${paths.project_root}/outputs/stage4/frozen_experiment_protocol.json
  label_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_label_policy.json
  threshold_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_threshold_policy.json
  seed_plan_manifest: ${paths.project_root}/outputs/stage4/frozen_seed_run_plan.json
  d_matching_policy_manifest: ${paths.project_root}/outputs/stage4/frozen_d_matching_policy.json

split_namespace: "production-thesis-v1"  # the smoke overlay sets dev-smoke-v1
conditions: [A, B, C]

model:
  architecture: "densenet121"
  pretrained_source: "imagenet"    # CheXpert-pretrained is rejected in code (§2)
  dropout_p: 0.2
  resolution: 320

# FROZEN fairness protocol (§7): equal OPTIMIZER STEPS across every enabled condition and seed.
# Not equal epochs — at fixed epochs a larger dataset silently receives more gradient updates,
# conflating "more data" with "more training". This matters most for B (all synthetic) vs C
# (selected subset), which differ in dataset size by construction.
training:
  max_steps: 6000
  batch_size: 32
  learning_rate: 1.0e-4
  weight_decay: 1.0e-4
  eval_every_n_steps: 500
  sampling_policy: "shuffled_concat"   # recorded per run; identical across conditions
  augmentation: "none"                 # horizontal flip is forbidden: it inverts CXR anatomy

# FROZEN seed policy (§7, v3 revision note 2026-08-21): every enabled condition gets these 3 seeds.
# Total runs = 3 conditions x 3 seeds = 9 (superseding the earlier A-G counts of 33 and 27).
# The surviving three were renamed A/B/C; the ASISM-selected arm was condition F under A-G.
seeds: [42, 43, 44]

# INACTIVE — retained, not deleted. No condition currently consumes this block: under the old A-G
# set the matched-random control was matched to a condition that has since been removed (§7 v3
# revision note). It is kept intact because §7.1 identifies re-enabling it — re-pointed at the
# current condition C — as the fix for the thesis's one structural gap: without it, a C-over-B
# gain cannot be attributed to ASISM's ranking rather than to simply using fewer synthetic
# images. Re-enabling requires supervisor sign-off and n_draws*3 additional runs.
condition_d:
  n_draws: 5                    # FROZEN — exactly five independent matched-random draws
  draw_seeds: [101, 102, 103, 104, 105]
  # Each draw must match the reference condition on:
  matching:
    match_total_count: true
    match_per_label_marginal: true
    per_label_tolerance_fraction: 0.10   # frozen tolerance on per-label positive counts
    match_single_multi_proportion: true
    single_multi_tolerance_fraction: 0.10
    attempt_joint_vector_matching: true  # only where support permits; never made infeasible

# Classification-threshold selection policy (frozen before Stage 5).
threshold_policy:
  method: "max_youden_j"       # chosen on classifier_val ONLY, never on final_eval_heldout
  selection_split: "classifier_val"

```

## `configs/smoke_e2e.yaml`

*182 lines*

**What it is:** a fixture-only overlay, activated *only* through the `THESIS_CONFIG_OVERLAY`
environment variable (see `scripts/utils/config._smoke_overlay`). It shrinks everything - 64px
images, LoRA rank 2, 2 training steps, 4 inference steps, 2 MC-Dropout passes, tiny subset design -
so the whole Stage 1->5 chain can run on CPU in minutes.

**It cannot be activated by accident**: nothing loads it unless the env var points at it, and
`smoke_only: true` plus the `dev-smoke-v1` namespace mark every artifact it produces as
non-scientific.

The comments in this file are unusually valuable: they record exactly which smoke values had to be
raised or lowered and why (e.g. why `base_quota` went from 1 to 24, why the verification budget
went from 0.5 h to 1.5 h).

```yaml
# Explicit fixture-only overrides. Activated only through THESIS_CONFIG_OVERLAY.
smoke_only: true
namespace: dev-smoke-v1

dataset:
  source:
    expected_train_rows: 60
    expected_valid_rows: 4

splits:
  namespaces:
    dev:
      require_full_cohort: false

stage1:
  # The base config points split.input_csv at the dev-subset CSV (commit d8e4151); the smoke
  # fixture writes a full raw train.csv instead, so read that directly.
  dev_subset:
    enabled: false
  split:
    namespace: dev-smoke-v1
    input_csv: ${paths.raw_dir}/train.csv
  data:
    resolution: 64
    min_source_resolution: 32
  lora:
    rank: 2
    alpha: 2
  optimizer:
    name: adamw
    lr_warmup_steps: 0
  training:
    precision: fp32
    train_batch_size: 1
    gradient_accumulation_steps: 1
    gradient_checkpointing: false
    cache_latents_and_embeddings: false
    max_train_steps: 2
    min_train_steps: 0
    ema:
      enabled: false
  checkpointing:
    save_every_n_steps: 1
    keep_last_n_full_checkpoints: 2
  validation:
    val_every_n_steps: 1

stage2:
  split_namespace: dev-smoke-v1
  checkpoint:
    lora_weights_dir: ${oc.env:SMOKE_LORA_DIR,null}
  generation:
    resolution: 64
    num_inference_steps: 4
    guidance_scale: 3.0
    batch_size: 1
    seed: 20260807
  recipes:
    min_support_patients: 1
    # Raised from 1: the learned-ASISM GPU smoke (scripts/smoke/run_smoke_pipeline.py, added
    # 2026-08-21) needs real candidates across all 11 PRIMARY_ENDPOINT_LABELS to clear
    # stage3.learned_asism's feasibility gate below, not just a handful of pilot-review images.
    # 8 per label (40 recipes after --limit) left the 20% val image pool with 0-1 candidates for
    # most labels, so the feasibility gate correctly failed closed. 24 per label (~270 recipes)
    # gives every label several val-pool candidates; 64 px / 4 steps keeps generation ~2 min.
    base_quota: 24
    min_per_label: 24
    max_per_label: 24
    age_buckets: [40, 60]
  pilot:
    num_images: 4
    checks:
      min_valid_image_fraction: 1.0
      max_near_uniform_fraction: 1.0

stage3:
  split_namespace: dev-smoke-v1
  auxiliary_classifier:
    resolution: 64
    dropout_p: 0.2
    pretrained_source: random
  signals:
    similarity:
      batch_size: 2
      k_neighbors: 2
      reference_sample_per_label: 2
      min_exact_match_references: 1
    uncertainty:
      mc_dropout_passes: 2
  gonogo:
    min_unique_values: 2
    directionality_probe_size: 4
  tuning:
    coarse:
      trials: 10
      folds: 1
      seeds: 1
      proxy_max_steps: 1
    shortlist:
      size: 1
      folds: 2
      seeds: 1
      proxy_max_steps: 1
    proxy:
      batch_size: 2
      resolution: 64
      pretrained_source: random
    compute_budget:
      max_gpu_hours: 1.0
      hours_per_proxy_run_estimate: 0.01
  selection:
    quality_floor_percentile: 0
    target_synthetic_to_real_ratio: 0.25
    min_accepted_per_label: 0
    max_accepted_per_label: 4
  # Shared by TWO smoke tiers with very different pool sizes — values are picked to be the smallest
  # safe for the GPU tier's small REAL pool, which is also automatically safe for the CPU tier's
  # larger FABRICATED pool (a lower threshold is satisfied by a bigger pool a fortiori):
  #   - scripts/smoke/01b_learned_asism_cpu_smoke.py: ~900 fabricated candidates, CPU-only, verified
  #     passing at these thresholds (2026-08-21).
  #   - scripts/smoke/run_smoke_pipeline.py (GPU phase, added 2026-08-21): real SDXL-generated
  #     candidates from stage2.recipes above (target ~150-300 images across 11 labels).
  #     UNVERIFIED BY EXECUTION — no GPU was available to run this locally. If `04 --phase
  #     feasibility` fails on the real pool, these are the first numbers to lower further; it needs
  #     no GPU and does not repeat Stage 1/2/aux/signals.
  learned_asism:
    subset_design:
      subset_sizes: [3, 4, 5]
      total_subsets: 24
      quantile_bins: 2  # build_controlled_subsets() hard-requires >= 2 (learned.py)
      minimum_image_exposures: 2
      feasibility_thresholds:
        min_candidates_per_class: 2
        min_candidates_per_quantile_band: 2
        min_val_subsets: 4
    set_utility_network:
      epochs: 3
      batch_size: 4
      early_stopping_patience: 3
    ranking_network:
      epochs: 3
      batch_size: 4
    threshold_network:
      epochs: 3
      # Production defaults (25 contexts/class x 11 classes, up to 5 verified candidates/context,
      # 5 full-policy variants x 3 seeds) imply hundreds of real tiny proxy-classifier training
      # runs — fine for a real budgeted production pass, not for a smoke test. Cut to the minimum
      # that still exercises the adaptive_threshold_network code path at least once, not just its
      # fallbacks.
      n_bootstrap_contexts_per_class: 2
      top_k_per_context: 1
      verification_quantile_fractions: [0.5]
      min_verified_contexts_per_class: 1
      verification_compute_budget:
        # 0.5 h fit the 40-image fixture; the ~360-image fixture yields 132 verification runs
        # (1.32 h at the 0.01 h/run estimate), so 07b's gate correctly refused. Smoke-only value.
        max_gpu_hours: 1.5
        hours_per_proxy_run_estimate: 0.01
      full_policy_verification:
        n_variants: 2
        n_seeds: 1
        compute_budget:
          max_gpu_hours: 0.5
          hours_per_proxy_run_estimate: 0.01
    compute_budget:
      max_gpu_hours: 1.0
      hours_per_proxy_run_estimate: 0.01

stage4:
  split_namespace: dev-smoke-v1
  conditions: [A, B, C]  # C (formerly F) = the finalized Learned-ASISM selection
  model:
    pretrained_source: random
    resolution: 64
  training:
    max_steps: 2
    batch_size: 2
    eval_every_n_steps: 1
  seeds: [42]
  condition_d:
    n_draws: 0
    draw_seeds: []

```

## `configs/accelerate_config.yaml`

*18 lines*

**What it is:** the HuggingFace Accelerate launch configuration for Stage 1 - single process,
single GPU, bf16 mixed precision. Used as
`accelerate launch --config_file configs/accelerate_config.yaml ...`.

```yaml
# Generated by `accelerate config` conventions, checked in so training is reproducible without
# an interactive setup step on a fresh RunPod pod. Single A40, single process — no multi-GPU/
# multi-node config needed (plan §8).
compute_environment: LOCAL_MACHINE
distributed_type: "NO"
downcast_bf16: "no"
gpu_ids: "0"
machine_rank: 0
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

```

---

# Part 2 - Shared utilities (`scripts/utils/`)

The layer every stage stands on: config loading, provenance hashing, the label policy, the split access guard, artifact contracts, metrics, and the one shared classifier. If two stages must agree about something, that something is defined here exactly once.

## `scripts/utils/config.py`

*77 lines*

**Role:** the single entry point for reading YAML. Nothing else in the repository parses YAML
directly, so `PROJECT_ROOT` resolution stays consistent everywhere.

- `load_named_config(filename, section)` - the general loader used by Stage 2-5.
- `load_stage1_config(overrides)` - Stage 1, and accepts OmegaConf dotlist overrides
  (`training.train_batch_size=4`) so CLI experiments never edit the checked-in YAML.
- `_smoke_overlay(section)` - if `THESIS_CONFIG_OVERLAY` is set, merge that section over the base
  config. This is the **only** way smoke values enter the system.
- `ensure_dirs(config)` - idempotently create the output directories a script writes into.

In [ ]:
# ===== FILE: scripts/utils/config.py =====
"""Load and resolve the Stage 1 YAML configs (configs/stage1_lora_sdxl.yaml,
configs/dataset_config.yaml) via OmegaConf, with PROJECT_ROOT-relative path resolution.

Every script under scripts/ should load config through this module rather than parsing YAML
directly, so path resolution (PROJECT_ROOT env var, see docs/stage1_plan.md §4) stays consistent.
"""

from __future__ import annotations

from pathlib import Path
import os

from omegaconf import OmegaConf, DictConfig

REPO_ROOT = Path(__file__).resolve().parents[2]
CONFIGS_DIR = REPO_ROOT / "configs"


def _smoke_overlay(section: str) -> DictConfig | None:
    path = os.environ.get("THESIS_CONFIG_OVERLAY")
    if not path:
        return None
    overlay_path = Path(path)
    if not overlay_path.is_file():
        raise SystemExit(f"THESIS_CONFIG_OVERLAY does not exist: {overlay_path}")
    overlay = OmegaConf.load(overlay_path)
    return overlay.get(section)


def load_named_config(filename: str, section: str) -> DictConfig:
    config = OmegaConf.load(CONFIGS_DIR / filename)
    overlay = _smoke_overlay(section)
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    OmegaConf.resolve(config)
    return config


def load_stage1_config(overrides: list[str] | None = None) -> DictConfig:
    """Load configs/stage1_lora_sdxl.yaml with PROJECT_ROOT-based interpolation resolved.

    `overrides` accepts OmegaConf dotlist entries, e.g. ["training.train_batch_size=4"], for
    CLI-driven experiment overrides without editing the checked-in YAML.
    """
    config = OmegaConf.load(CONFIGS_DIR / "stage1_lora_sdxl.yaml")
    overlay = _smoke_overlay("stage1")
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    if overrides:
        config = OmegaConf.merge(config, OmegaConf.from_dotlist(overrides))
    OmegaConf.resolve(config)
    return config


def load_dataset_config() -> DictConfig:
    config = OmegaConf.load(CONFIGS_DIR / "dataset_config.yaml")
    overlay = _smoke_overlay("dataset")
    if overlay is not None:
        config = OmegaConf.merge(config, overlay)
    OmegaConf.resolve(config)
    return config


def ensure_dirs(config: DictConfig) -> None:
    """Create the directories under config.paths that scripts write into (idempotent)."""
    for key in (
        "raw_dir",
        "processed_dir",
        "images_dir",
        "splits_dir",
        "captions_dir",
        "checkpoints_dir",
        "logs_dir",
        "outputs_dir",
        "hf_cache_dir",
    ):
        Path(config.paths[key]).mkdir(parents=True, exist_ok=True)


## `scripts/utils/manifest.py`

*157 lines*

**Role:** provenance. Every artifact in the project records enough to trace it back to the exact
code + config + data that produced it.

- `code_identity()` - hashes the *contents* of `scripts/`, `configs/`, `environment/`, including
  uncommitted and untracked files. A git SHA alone is not enough in a dirty research worktree; this
  is what lets a downstream stage refuse artifacts produced by a different source tree.
- `hash_dict` / `sha256_file` / `sha256_directory` - stable hashes for configs, files, directories.
- `make_run_id(config)` - `<UTC timestamp>_<git sha>_<config hash>`.
- `write_json` - **atomic** (temp file + `fsync` + `os.replace`), so an interrupted pod can never
  leave a half-written manifest.
- `write_frozen_json` - writes once and then refuses any overwrite, identical or not. This is the
  mechanism behind every "frozen" artifact in the project.
- `build_checkpoint_metadata` - the metadata block stored next to every Stage 1 checkpoint.

In [ ]:
# ===== FILE: scripts/utils/manifest.py =====
"""Provenance helpers: run_id composition, git/config hashing, JSON manifest read/write.

Used throughout Stage 1 (dataset splits, captions, checkpoints) so every artifact records enough
to be traced back to the exact code + config + data that produced it (docs/stage1_plan.md:
reproducibility, checkpoint metadata, handoff to Stage 2).
"""

from __future__ import annotations

import hashlib
import json
import os
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def get_git_commit_hash(repo_dir: str | Path = ".", short: bool = True) -> str:
    try:
        args = ["git", "-C", str(repo_dir), "rev-parse"]
        args.append("--short" if short else "HEAD")
        if short:
            args.append("HEAD")
        result = subprocess.run(args, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return "nogit"


def hash_dict(data: dict[str, Any], length: int = 8) -> str:
    """Stable short hash of a (JSON-serializable) config dict, for use in run_ids and provenance."""
    serialized = json.dumps(data, sort_keys=True, default=str)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:length]


def make_run_id(config: dict[str, Any], repo_dir: str | Path = ".") -> str:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    git_sha = get_git_commit_hash(repo_dir)
    config_hash = hash_dict(config)
    return f"{timestamp}_{git_sha}_{config_hash}"


def get_library_versions() -> dict[str, str]:
    versions: dict[str, str] = {}
    for module_name in (
        "torch",
        "diffusers",
        "transformers",
        "accelerate",
        "peft",
        "bitsandbytes",
    ):
        try:
            module = __import__(module_name)
            versions[module_name] = getattr(module, "__version__", "unknown")
        except ImportError:
            versions[module_name] = "not_installed"
    return versions


def code_identity(repo_dir: str | Path = ".") -> dict[str, Any]:
    """Hash the executable/config source, including uncommitted and untracked files.

    A Git SHA alone is insufficient in a dirty research worktree. Data and generated artifacts are
    intentionally excluded; their identities are recorded by their own manifests.
    """
    root = Path(repo_dir).resolve()
    included_roots = ("scripts", "configs", "environment")
    files = []
    for relative_root in included_roots:
        directory = root / relative_root
        if not directory.exists():
            continue
        for path in sorted(p for p in directory.rglob("*") if p.is_file() and "__pycache__" not in p.parts):
            files.append({"path": path.relative_to(root).as_posix(), "sha256": sha256_file(path)})
    digest = hashlib.sha256(json.dumps(files, sort_keys=True).encode("utf-8")).hexdigest()
    return {"git_commit": get_git_commit_hash(root, short=False), "source_tree_sha256": digest, "files": files}


def sha256_directory(path: str | Path) -> str:
    root = Path(path)
    if not root.is_dir():
        raise FileNotFoundError(root)
    entries = [
        {"path": file.relative_to(root).as_posix(), "sha256": sha256_file(file)}
        for file in sorted(p for p in root.rglob("*") if p.is_file())
    ]
    return hashlib.sha256(json.dumps(entries, sort_keys=True).encode("utf-8")).hexdigest()


def sha256_file(path: str | Path) -> str:
    """Stream-hash a file's contents (used for provenance of split/checkpoint/manifest inputs)."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: str | Path, data: dict[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, sort_keys=True, default=str)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temporary_name, path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)


def write_frozen_json(path: str | Path, data: dict[str, Any]) -> None:
    """Create a freeze artifact exactly once; identical or changed overwrites are refused."""
    path = Path(path)
    if path.exists():
        raise FileExistsError(
            f"Frozen artifact already exists and is immutable: {path}. "
            "Use a new explicit protocol/run version and record its parent."
        )
    payload = dict(data)
    payload["frozen"] = True
    write_json(path, payload)


def read_json(path: str | Path) -> dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def build_checkpoint_metadata(
    step: int,
    epoch: int,
    config: dict[str, Any],
    split_manifest_hash: str,
    seed: int,
    repo_dir: str | Path = ".",
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Metadata recorded alongside every checkpoint (docs/stage1_plan.md §10)."""
    metadata = {
        "step": step,
        "epoch": epoch,
        "wall_clock_utc": datetime.now(timezone.utc).isoformat(),
        "git_commit_hash": get_git_commit_hash(repo_dir),
        "config": config,
        "config_hash": hash_dict(config),
        "split_manifest_hash": split_manifest_hash,
        "seed": seed,
        "library_versions": get_library_versions(),
    }
    if extra:
        metadata.update(extra)
    return metadata


## `scripts/utils/identifiers.py`

*15 lines*

**Role:** one tiny function, `sanitize_image_id`, that turns a CheXpert path into a flat id
(`patient00001__study1__view1_frontal`).

**Why it exists as its own module:** preprocessing (03) and caption generation (04) must produce
*identical* ids or the two outputs cannot be joined. Sharing the function makes drift impossible.

In [ ]:
# ===== FILE: scripts/utils/identifiers.py =====
"""Shared image-id derivation, used identically by preprocessing (03) and caption generation (04)
so the two stay joinable on image_id."""

from __future__ import annotations

import re


def sanitize_image_id(raw_path: str) -> str:
    p = str(raw_path).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if p.startswith(prefix):
            p = p[len(prefix):]
    p = re.sub(r"\.(jpg|jpeg|png)$", "", p, flags=re.IGNORECASE)
    return p.replace("/", "__")


## `scripts/utils/seed.py`

*27 lines*

**Role:** `set_seed(seed)` - seeds `random`, `PYTHONHASHSEED`, NumPy and torch (including CUDA) in
one call, tolerating missing optional dependencies. Called by every stage that has any randomness.

In [ ]:
# ===== FILE: scripts/utils/seed.py =====
"""Deterministic seeding helper, shared by every Stage 1 script (docs/stage1_plan.md: reproducibility)."""

from __future__ import annotations

import os
import random


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass

    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


## `scripts/utils/caption_builder.py`

*220 lines*

**Role:** the single source of truth for turning a CheXpert label row into a text prompt. Imported
**verbatim** by Stage 1 training (`04_generate_captions.py`), Stage 2 generation, and the Stage 1
probe sampler - any phrasing drift between train time and generation time directly degrades
conditioning fidelity.

Encoded policy decisions:

- **Uncertain (-1) labels are omitted** from the findings clause - never asserted present or absent.
- **`No Finding == 1` is authoritative** over any spuriously co-positive pathology column (a known
  artifact of CheXpert's NLP label extraction).
- **Support Devices is a device, not a pathology**, and gets its own separate clause.
- **Age is decade-bucketed** to reduce caption-vocabulary sparsity.
- **Paraphrase variant selection is deterministic** - `sha256(image_id + epoch) % n`, not an
  unseeded RNG, so captions are reproducible.
- `serialize_natural_list` produces "X, Y, and Z" grammar rather than a comma token-dump, which
  gives more stable CLIP embeddings.

In [ ]:
# ===== FILE: scripts/utils/caption_builder.py =====
"""Structured label-to-text caption builder for CheXpert chest X-rays (docs/stage1_plan.md §7).

This module is the single source of truth for turning a CheXpert label row into a text prompt.
It is imported verbatim by both the Stage 1 training data pipeline (scripts/data/04_generate_captions.py)
and, later, Stage 2's generation pipeline — any drift between train-time and generation-time
phrasing directly degrades conditioning fidelity, so nothing caption-related should be
reimplemented elsewhere.

Design decisions encoded here (see docs/stage1_plan.md §7 for full rationale):
- Uncertain (-1) labels are OMITTED from the finding clause, never asserted present or absent.
- "No Finding" == 1 is authoritative over any spuriously co-positive pathology columns.
- "Support Devices" is a device, not a pathology, and gets its own clause.
- Age is decade-bucketed to reduce caption vocabulary sparsity.
- A small set of paraphrase templates is selected deterministically (hash of image id + epoch),
  not by an unseeded RNG, so caption generation is reproducible.
"""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

# Canonical order for serializing multi-label findings. Must match configs/dataset_config.yaml's
# `schema.pathology_columns` — kept as a plain constant here (rather than loaded from YAML) so this
# module has no config-parsing dependency and can be imported standalone by Stage 2.
PATHOLOGY_COLUMNS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

DEVICE_COLUMN = "Support Devices"
NO_FINDING_COLUMN = "No Finding"

# Human-readable phrasing for each pathology column (lowercase, fit for "Findings: X, Y, and Z.")
FINDING_TEXT = {
    "Enlarged Cardiomediastinum": "an enlarged cardiomediastinum",
    "Cardiomegaly": "cardiomegaly",
    "Lung Opacity": "lung opacity",
    "Lung Lesion": "a lung lesion",
    "Edema": "pulmonary edema",
    "Consolidation": "consolidation",
    "Pneumonia": "pneumonia",
    "Atelectasis": "atelectasis",
    "Pneumothorax": "pneumothorax",
    "Pleural Effusion": "pleural effusion",
    "Pleural Other": "other pleural abnormality",
    "Fracture": "a fracture",
}

NO_FINDING_PHRASE = "no acute cardiopulmonary abnormality"
DEVICE_PHRASE = "Support devices are present."

# Paraphrase templates (plan §7: 3-5 variants, config default 4). All must use exactly the
# {view}, {age_bucket}, {sex}, {finding_clause} placeholders; {device_clause} is appended
# separately by build_caption(), never interpolated inside these.
TEMPLATES = [
    "A {view} chest X-ray of a {age_bucket} {sex} patient. Findings: {finding_clause}.",
    "Chest radiograph, {view} view, {age_bucket} {sex} patient. Findings: {finding_clause}.",
    "A {view} view chest X-ray from a {age_bucket} {sex} patient showing {finding_clause}.",
    "{age_bucket} {sex} patient, {view} chest X-ray. Findings: {finding_clause}.",
]


@dataclass(frozen=True)
class CaptionConfig:
    age_bucket_width_years: int = 10
    uncertain_label_policy: str = "omit"  # only "omit" is implemented (plan §7 decision)
    no_finding_overrides_positives: bool = True
    num_paraphrase_variants: int = 4
    template_version: str = "v1"


def bucket_age(age, bucket_width_years: int = 10) -> str:
    """Decade-bucket a numeric age into a phrase like '40-49-year-old'."""
    try:
        age_int = int(float(age))
    except (TypeError, ValueError):
        return "adult"
    if age_int <= 0:
        return "adult"
    low = (age_int // bucket_width_years) * bucket_width_years
    high = low + bucket_width_years - 1
    return f"{low}-{high}-year-old"


def sex_phrase(sex_value) -> str:
    if sex_value is None:
        return "patient"
    s = str(sex_value).strip().lower()
    if s.startswith("m"):
        return "male"
    if s.startswith("f"):
        return "female"
    return "patient"


def view_phrase(frontal_lateral_value, ap_pa_value=None) -> str:
    fl = str(frontal_lateral_value).strip().lower() if frontal_lateral_value is not None else "frontal"
    if fl.startswith("lat"):
        return "lateral"
    return "frontal"


def serialize_natural_list(items: list[str]) -> str:
    """Join items with natural 'X', 'X and Y', or 'X, Y, and Z' grammar (plan §7: more stable
    CLIP embeddings than a raw comma token-dump)."""
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"


def _label_value(row: dict, column: str):
    """Normalize a raw CheXpert label cell to one of {-1, 0, 1, None}."""
    value = row.get(column)
    if value is None or value == "":
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    if f != f:  # NaN
        return None
    return int(f)


def build_finding_clause(row: dict, config: CaptionConfig = CaptionConfig()) -> str:
    """Build the 'Findings: ...' clause per the uncertainty and No-Finding policies in plan §7."""
    if config.uncertain_label_policy != "omit":
        raise NotImplementedError(
            f"uncertain_label_policy={config.uncertain_label_policy!r} not implemented; "
            "the plan's chosen policy is 'omit' — see docs/stage1_plan.md §7."
        )

    no_finding = _label_value(row, NO_FINDING_COLUMN)
    positive_pathologies = [
        col
        for col in PATHOLOGY_COLUMNS
        if col not in (NO_FINDING_COLUMN, DEVICE_COLUMN) and _label_value(row, col) == 1
    ]

    if config.no_finding_overrides_positives and no_finding == 1:
        # A positive "No Finding" is treated as authoritative over any spuriously co-positive
        # pathology columns (known NLP-label-extraction artifact) — see plan §7.
        return NO_FINDING_PHRASE

    if not positive_pathologies:
        if no_finding == 1:
            return NO_FINDING_PHRASE
        # No positive findings and No Finding not asserted (e.g. only uncertain/negative labels):
        # still emit the normal-chest phrasing rather than an empty clause.
        return NO_FINDING_PHRASE

    phrases = [FINDING_TEXT[col] for col in positive_pathologies]
    return serialize_natural_list(phrases)


def build_device_clause(row: dict) -> str:
    if _label_value(row, DEVICE_COLUMN) == 1:
        return DEVICE_PHRASE
    return ""


def select_variant_index(image_id: str, epoch: int, num_variants: int) -> int:
    """Deterministic (not process-RNG-dependent) paraphrase variant selection, so caption
    generation is reproducible across runs given the same image id and epoch (plan §7)."""
    key = f"{image_id}:{epoch}".encode("utf-8")
    digest = hashlib.sha256(key).hexdigest()
    return int(digest, 16) % num_variants


def build_caption(
    row: dict,
    config: CaptionConfig = CaptionConfig(),
    variant_index: int = 0,
) -> str:
    """Build a single caption string from a CheXpert label row.

    `row` is expected to expose (at minimum) the columns listed in
    configs/dataset_config.yaml `schema`: Sex, Age, Frontal/Lateral, AP/PA, and the 14
    pathology columns in PATHOLOGY_COLUMNS, with ternary values in {-1, 0, 1, NaN/None}.
    """
    template = TEMPLATES[variant_index % len(TEMPLATES)]
    view = view_phrase(row.get("Frontal/Lateral"), row.get("AP/PA"))
    age_bucket = bucket_age(row.get("Age"), config.age_bucket_width_years)
    sex = sex_phrase(row.get("Sex"))
    finding_clause = build_finding_clause(row, config)
    device_clause = build_device_clause(row)

    caption = template.format(
        view=view,
        age_bucket=age_bucket,
        sex=sex,
        finding_clause=finding_clause,
    )
    if device_clause:
        caption = f"{caption} {device_clause}"
    return caption


def build_caption_variants(row: dict, config: CaptionConfig = CaptionConfig()) -> list[str]:
    """Return all configured paraphrase variants for a row (useful for prompt-cache precomputation,
    plan §8, where every variant an image might use during training is embedded once up front)."""
    n = min(config.num_paraphrase_variants, len(TEMPLATES))
    return [build_caption(row, config, variant_index=i) for i in range(n)]


## `scripts/utils/labels.py`

*245 lines*

**Role:** the frozen label policy. Three label sets that are **deliberately not interchangeable**:

| Set | Size | Meaning |
|---|---:|---|
| `CLASSIFIER_TARGET_LABELS` | 14 | the classifier's output space; everything is predicted and reported |
| `PRIMARY_ENDPOINT_LABELS` | 11 | what macro-AUROC (the primary endpoint) is computed over |
| `GENERATION_TARGET_LABELS` | 12 | what Stage 2 is allowed to intentionally synthesize |

The 11 excludes `No Finding` (an absence-of-disease meta-label), `Support Devices` (a device label,
and the easiest/highest-prevalence column - including it would flatter the macro-average), and
`Pleural Other` (cannot meet the 50-negative-patient support rule in any split). The 12 adds
`Pleural Other` back, because support scarcity is a property of what can be reliably *evaluated*,
not a reason to stop *generating* examples of a condition.

**The uncertainty policy lives here** and is the most important thing in the file:
`build_label_arrays` returns three aligned arrays - `raw_label` (with `-2` for blank so the array
stays integer), `training_target`, and `mask`. Positions where the mask is False are meaningless
placeholders, **not** negatives. `-1` and blank contribute to neither loss nor metrics and are
never silently mapped to 0 or 1.

`patient_level_support` + `check_support_rule` implement the support rule at **patient** level -
image counts would let one patient with many studies manufacture apparent support.

In [ ]:
# ===== FILE: scripts/utils/labels.py =====
"""Frozen CheXpert label policy, shared by every Stage 2-5 script
(docs/stages2_to_5_plan.md §5 and §6).

Three label sets, deliberately not interchangeable:

  CLASSIFIER_TARGET_LABELS (14)  - the classifier's output space. Everything is predicted and
                                   reported.
  PRIMARY_ENDPOINT_LABELS  (11)  - the disease labels the PRIMARY endpoint (macro-AUROC) is
                                   computed over. Excludes "No Finding" (an absence-of-disease
                                   meta-label, not a pathology) and "Support Devices" (a device
                                   label, and the highest-prevalence/easiest column in CheXpert —
                                   including it would flatter the macro-average without measuring
                                   diagnostic performance). Also excludes "Pleural Other"
                                   (docs/stages2_to_5_plan.md §5.2 revision note): on the full
                                   production cohort only ~100 patients dataset-wide carry a
                                   confident negative label for it, so no split-fraction choice can
                                   give every decision-bearing split the frozen ≥50-negative-patient
                                   support rule (§1.3) — unlike "Lung Lesion"/"Atelectasis", which
                                   were fixed by revising configs/splits.yaml fractions instead. All
                                   three remain secondary outcomes.
  GENERATION_TARGET_LABELS (12)  - PRIMARY_ENDPOINT_LABELS plus INSUFFICIENT_SUPPORT_LABELS
                                   (docs/stages2_to_5_plan.md §3 revision note). Real-data support
                                   scarcity is a property of what can be reliably EVALUATED on the
                                   real cohort; it is not a reason to also stop generating or
                                   training on synthetic examples of the condition. Stage 2 recipe
                                   eligibility (co-occurrence mining, quotas, captions) keys off
                                   this list, not PRIMARY_ENDPOINT_LABELS. ASISM's agreement
                                   scoring (§4.5) and Stage 4 condition D's marginal matching (§7)
                                   deliberately stay scoped to PRIMARY_ENDPOINT_LABELS only.

Uncertainty policy (plan §6): raw labels are preserved; -1 and blank are MASKED out of loss and
metrics, never silently mapped to 0 or 1. Three quantities are always kept separate:
`raw_label`, `training_target`, and `loss_mask`/`eval_mask`.

`caption_builder.PATHOLOGY_COLUMNS` is the same 14 columns in the same canonical order; it is
imported here rather than re-typed so the two modules cannot drift.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from scripts.utils.caption_builder import (
    DEVICE_COLUMN,
    NO_FINDING_COLUMN,
    PATHOLOGY_COLUMNS,
)

# The classifier's full output space: all 14 CheXpert observations, canonical order.
CLASSIFIER_TARGET_LABELS: list[str] = list(PATHOLOGY_COLUMNS)

# Excluded from the primary endpoint due to insufficient patient-level negative support under the
# frozen §1.3 support rule (measured on the full production cohort: see module docstring). Reported
# as a secondary outcome with its support counts (plan §5.3), same treatment as No Finding /
# Support Devices.
INSUFFICIENT_SUPPORT_LABELS: list[str] = ["Pleural Other"]

# The 11 disease labels the primary endpoint is computed over (plan §5.2).
PRIMARY_ENDPOINT_LABELS: list[str] = [
    column
    for column in CLASSIFIER_TARGET_LABELS
    if column not in (NO_FINDING_COLUMN, DEVICE_COLUMN, *INSUFFICIENT_SUPPORT_LABELS)
]

# Every disease label worth intentionally synthesizing in Stage 2, whether or not it currently has
# enough REAL-data patient support to be scored as a primary endpoint (module docstring).
GENERATION_TARGET_LABELS: list[str] = [*PRIMARY_ENDPOINT_LABELS, *INSUFFICIENT_SUPPORT_LABELS]

# Sentinel used in integer label arrays for "uncertain or not mentioned" (i.e. masked).
MASKED = -1


def normalize_label(value) -> int | None:
    """Normalize one raw CheXpert cell to exactly one of {1, 0, -1, None}.

    None means blank/not-mentioned. -1 means the radiologist recorded uncertainty. Both are masked
    downstream, but they are distinct facts and are kept distinct in `raw_label`.
    """
    if value is None or value == "":
        return None
    try:
        numeric = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"invalid CheXpert label value {value!r}; expected -1, 0, 1, or blank") from exc
    if numeric != numeric:  # NaN
        return None
    if numeric not in (-1.0, 0.0, 1.0):
        raise ValueError(f"invalid CheXpert label value {value!r}; expected -1, 0, 1, or blank")
    return int(numeric)


def validate_label_domain(frame: pd.DataFrame, labels: list[str], identifier_column: str = "Path") -> None:
    """Validate labels with enough row context to diagnose a bad source cohort."""
    for label in labels:
        if label not in frame.columns:
            raise ValueError(f"missing required CheXpert label column {label!r}")
        for index, value in frame[label].items():
            try:
                normalize_label(value)
            except ValueError as exc:
                identifier = frame.at[index, identifier_column] if identifier_column in frame else index
                raise ValueError(f"label={label!r}, value={value!r}, row/image={identifier!r}: {exc}") from exc


def build_label_arrays(
    frame: pd.DataFrame,
    labels: list[str] | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (raw_label, training_target, mask) as three aligned (n_rows, n_labels) arrays.

    raw_label       int8, values in {1, 0, -1, -2}; -2 encodes blank/not-mentioned so the raw
                    array stays integer while remaining distinguishable from an explicit -1.
    training_target float32, values in {0.0, 1.0}; positions where mask is False are 0.0 and MUST
                    NOT be read (they are meaningless placeholders, not negatives).
    mask            bool, True only where the label is confidently 0 or 1.

    The mask is what enforces plan §6: -1 and blank contribute to neither loss nor metrics, and are
    never converted into a 0 or a 1 that the model would then be trained to reproduce.
    """
    labels = labels if labels is not None else CLASSIFIER_TARGET_LABELS
    n_rows, n_labels = len(frame), len(labels)

    raw = np.full((n_rows, n_labels), -2, dtype=np.int8)
    target = np.zeros((n_rows, n_labels), dtype=np.float32)
    mask = np.zeros((n_rows, n_labels), dtype=bool)

    for column_index, label in enumerate(labels):
        if label not in frame.columns:
            # Column entirely absent: leave as blank/masked rather than inventing negatives.
            continue
        values = [normalize_label(value) for value in frame[label].to_numpy()]
        for row_index, value in enumerate(values):
            if value is None:
                continue
            raw[row_index, column_index] = value
            if value in (0, 1):
                target[row_index, column_index] = float(value)
                mask[row_index, column_index] = True

    return raw, target, mask


def patient_level_support(
    frame: pd.DataFrame,
    labels: list[str] | None = None,
    patient_column: str = "patient_id",
) -> pd.DataFrame:
    """Per-label positive/negative/masked PATIENT counts (plan §1.3 — patient counts, not image
    counts, determine primary support eligibility).

    A patient counts as positive for a label if ANY of their studies carries a confident 1, and as
    negative if they are not positive and any study carries a confident 0. Patients whose every
    study is uncertain/blank for that label count as masked-only and support neither side.

    Returns one row per label with columns:
    label, positive_patients, negative_patients, masked_only_patients, positive_images,
    negative_images, total_patients.
    """
    labels = labels if labels is not None else PRIMARY_ENDPOINT_LABELS
    if patient_column not in frame.columns:
        raise KeyError(
            f"patient_level_support requires a {patient_column!r} column; got {list(frame.columns)[:8]}..."
        )

    records = []
    total_patients = frame[patient_column].nunique()

    for label in labels:
        if label not in frame.columns:
            records.append(
                {
                    "label": label,
                    "positive_patients": 0,
                    "negative_patients": 0,
                    "masked_only_patients": total_patients,
                    "positive_images": 0,
                    "negative_images": 0,
                    "total_patients": total_patients,
                }
            )
            continue

        normalized = frame[label].map(normalize_label)
        is_positive = normalized == 1
        is_negative = normalized == 0

        positive_patients = set(frame.loc[is_positive, patient_column].unique())
        negative_candidates = set(frame.loc[is_negative, patient_column].unique())
        # "Any positive" wins: a patient with one positive study is a positive patient.
        negative_patients = negative_candidates - positive_patients
        masked_only = total_patients - len(positive_patients) - len(negative_patients)

        records.append(
            {
                "label": label,
                "positive_patients": len(positive_patients),
                "negative_patients": len(negative_patients),
                "masked_only_patients": masked_only,
                "positive_images": int(is_positive.sum()),
                "negative_images": int(is_negative.sum()),
                "total_patients": total_patients,
            }
        )

    return pd.DataFrame.from_records(records)


def check_support_rule(
    support: pd.DataFrame,
    min_positive_patients: int,
    min_negative_patients: int,
) -> tuple[bool, list[dict]]:
    """Apply the frozen §1.3 support rule to a `patient_level_support` frame.

    Returns (passed, failures), where each failure records the label and both observed counts so
    the caller can emit a complete report rather than a first-failure message.
    """
    failures = []
    for row in support.to_dict("records"):
        positive_ok = row["positive_patients"] >= min_positive_patients
        negative_ok = row["negative_patients"] >= min_negative_patients
        if not (positive_ok and negative_ok):
            failures.append(
                {
                    "label": row["label"],
                    "positive_patients": row["positive_patients"],
                    "negative_patients": row["negative_patients"],
                    "min_positive_patients": min_positive_patients,
                    "min_negative_patients": min_negative_patients,
                    "positive_ok": positive_ok,
                    "negative_ok": negative_ok,
                }
            )
    return (len(failures) == 0), failures


def intended_vector_to_labels(intended: dict[str, int]) -> list[str]:
    """The primary disease labels a Stage 2 recipe intends to be positive.

    A `No Finding` recipe is the all-zero intended vector over the 11 primary labels (plan §3), so
    this correctly returns an empty list for it — agreement (§4.5) then scores it as "all 11
    primary predicted probabilities should be low."
    """
    return [label for label in PRIMARY_ENDPOINT_LABELS if int(intended.get(label, 0)) == 1]


## `scripts/utils/chexpert_schema.py`

*46 lines*

**Role:** validate that a CheXpert CSV really is a CheXpert CSV before anything trusts it.

`validate_chexpert_frame` checks required columns exist, every label value is in {-1, 0, 1, blank},
no blank or duplicate image paths, every path yields a patient id, and (optionally) that every
referenced image file actually exists. Returns a `source_schema_hash` that downstream manifests
record. `resolve_chexpert_image` strips the `CheXpert-v1.0-small/` prefix variants and resolves a
CSV path value against the raw directory.

In [ ]:
# ===== FILE: scripts/utils/chexpert_schema.py =====
"""Shared CheXpert source-schema and path validation used by verification and split building."""
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

from scripts.utils.labels import validate_label_domain
from scripts.utils.manifest import hash_dict


def resolve_chexpert_image(raw_dir: Path, value: object) -> Path:
    relative = str(value).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if relative.startswith(prefix):
            relative = relative[len(prefix):]
    return raw_dir / relative


def validate_chexpert_frame(frame: pd.DataFrame, dataset_cfg, raw_dir: Path | None = None,
                            require_all_images: bool = False) -> dict:
    schema = dataset_cfg.schema
    required = [schema.path_column, schema.sex_column, schema.age_column,
                schema.frontal_lateral_column, *list(schema.pathology_columns)]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"missing required columns: {missing}")
    validate_label_domain(frame, list(schema.pathology_columns), str(schema.path_column))
    paths = frame[schema.path_column]
    if paths.isna().any() or paths.astype(str).str.strip().eq("").any():
        raise ValueError("blank image path")
    duplicates = paths[paths.duplicated()].astype(str).head().tolist()
    if duplicates:
        raise ValueError(f"duplicate image paths: {duplicates}")
    invalid_patient = [value for value in paths.astype(str) if not re.search(str(schema.patient_id_regex), value)]
    if invalid_patient:
        raise ValueError(f"invalid patient ID/path: {invalid_patient[:5]}")
    if require_all_images:
        if raw_dir is None:
            raise ValueError("raw_dir is required for full image-path validation")
        missing_images = [str(resolve_chexpert_image(raw_dir, value)) for value in paths if not resolve_chexpert_image(raw_dir, value).is_file()]
        if missing_images:
            raise ValueError(f"{len(missing_images)} image paths do not exist; examples={missing_images[:5]}")
    return {"source_schema_hash": hash_dict({"columns": list(frame.columns),
                                              "dtypes": {c: str(t) for c, t in frame.dtypes.items()}}, length=64)}


## `scripts/utils/splits.py`

*270 lines*

**Role:** split loading, namespace resolution, and the **`final_eval_heldout` access guard** - the
single most important integrity mechanism in the project.

The honest version of the leakage rule is not "final_eval_heldout is never touched". It *is*
touched exactly once, during split construction, to assign patients, assert disjointness, run the
support check and hash the manifest. What must never happen is an **outcome-bearing read** before
Stage 5.

So access is mediated by an explicit **purpose token**:

- `NON_OUTCOME_PURPOSES` (split construction, support check, schema validation, existence check,
  hash verification) - always allowed.
- `OUTCOME_PURPOSES` (load images, load labels for evaluation, compute outcome statistics, model
  selection) - require a Stage 5 `--final-eval-run-id` **and** a registered, validated evaluation
  context. Any other caller fails loudly with `FinalEvalAccessViolation`.

Also here: `read_split_manifest` (version + namespace + self-hash integrity checks),
`split_provenance` (the small provenance block every downstream manifest embeds), and
`require_frozen_production_split_run` (the Stage 5 precondition: production class, frozen, support
check passed).

In [ ]:
# ===== FILE: scripts/utils/splits.py =====
"""Split loading, namespace resolution, and the programmatic `final_eval_heldout` access guard
(docs/stages2_to_5_plan.md §1.4 and §1.6).

The guard exists because the honest version of the leakage rule is not "final_eval_heldout is never
touched" — it IS touched, exactly once, during split construction, to assign patients, assert
disjointness, run the support check, and hash the manifest. What must never happen is an
outcome-bearing read before Stage 5: loading its images, loading its labels to compute outcome
statistics, or letting any model/method decision depend on it.

So access is mediated by an explicit purpose token. Non-outcome-bearing purposes are always
allowed; outcome-bearing purposes require a Stage 5 final-evaluation run id, which only the Stage 5
entry point supplies. Any other caller asking for an outcome-bearing read fails loudly.
"""

from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from omegaconf import DictConfig, OmegaConf

from scripts.utils.config import load_named_config
from scripts.utils.manifest import hash_dict, read_json, sha256_file, write_frozen_json
import random

SPLIT_NAMES = [
    "gen_train",
    "gen_val",
    "classifier_train",
    "classifier_val",
    "asism_tuning_heldout",
    "final_eval_heldout",
]

FINAL_EVAL_SPLIT = "final_eval_heldout"

SPLIT_MANIFEST_FILENAME = "split_manifest_v2.json"

# Purposes that do NOT read outcomes: allowed at any time, including before Stage 5.
NON_OUTCOME_PURPOSES = {
    "split_construction",    # deterministic assignment, disjointness/partition assertions
    "support_check",         # positive/negative patient counts (a property of the split, not a result)
    "schema_validation",     # column/dtype checks
    "existence_check",       # does the file exist
    "hash_verification",     # manifest hashing / hash comparison
}

# Purposes that DO read outcomes: require an explicit Stage 5 final-evaluation run id.
OUTCOME_PURPOSES = {
    "load_images",
    "load_labels_for_evaluation",
    "compute_outcome_statistics",
    "model_selection",
}


class FinalEvalAccessViolation(SystemExit):
    """Raised (as SystemExit, so it surfaces as a clean CLI failure) when code attempts an
    outcome-bearing read of final_eval_heldout outside a declared Stage 5 evaluation run."""


def assert_final_eval_access_allowed(
    purpose: str,
    final_eval_run_id: str | None = None,
    caller: str = "unknown",
) -> None:
    """Gate every access to final_eval_heldout behind an explicit declared purpose.

    Non-outcome purposes (§1.6's enumerated split-construction activities) pass unconditionally.
    Outcome purposes require `final_eval_run_id` — supplied only by the Stage 5 entry point.
    """
    if purpose in NON_OUTCOME_PURPOSES:
        return

    if purpose not in OUTCOME_PURPOSES:
        raise FinalEvalAccessViolation(
            f"[{caller}] Unrecognized final_eval_heldout access purpose {purpose!r}. "
            f"Declare one of: non-outcome {sorted(NON_OUTCOME_PURPOSES)} "
            f"or outcome-bearing {sorted(OUTCOME_PURPOSES)}."
        )

    if not final_eval_run_id:
        raise FinalEvalAccessViolation(
            f"[{caller}] Refusing outcome-bearing access to {FINAL_EVAL_SPLIT} "
            f"(purpose={purpose!r}) without an explicit final-evaluation run id.\n"
            "final_eval_heldout may only be read for outcomes inside a declared Stage 5 run "
            "(docs/stages2_to_5_plan.md §1.6). If this is Stage 5, pass --final-eval-run-id."
        )


def load_splits_config() -> DictConfig:
    return load_named_config("splits.yaml", "splits")


def resolve_splits_dir(namespace: str, config: DictConfig | None = None) -> Path:
    """Namespaced split directory: <splits_root>/<namespace>/ (plan §1.4).

    Keeping `dev` and `production` in separate directories is what makes it structurally impossible
    for a dev-subset split to overwrite a production split.
    """
    config = config if config is not None else load_splits_config()
    if not namespace or namespace in {".", ".."} or any(ch in namespace for ch in "/\\"):
        raise ValueError(f"Invalid explicit split namespace/run id {namespace!r}")
    return Path(config.paths.splits_root) / namespace


def split_manifest_path(namespace: str, config: DictConfig | None = None) -> Path:
    return resolve_splits_dir(namespace, config) / SPLIT_MANIFEST_FILENAME


def read_split_manifest(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    path = split_manifest_path(namespace, config)
    if not path.is_file():
        raise SystemExit(
            f"Missing split manifest: {path}\n"
            f"Build it first: python scripts/data/02b_build_sixway_splits.py --namespace {namespace}"
        )
    manifest = read_json(path)
    expected = int((config or load_splits_config()).manifest_version)
    if manifest.get("manifest_version") != expected:
        raise SystemExit(f"Split manifest version mismatch at {path}: expected {expected}, got {manifest.get('manifest_version')!r}")
    if manifest.get("split_namespace") != namespace:
        raise SystemExit(f"Split namespace mismatch at {path}: expected {namespace!r}, got {manifest.get('split_namespace')!r}")
    if manifest.get("manifest_hash_scope") == "full_manifest_without_manifest_hash_v2":
        declared = manifest.get("manifest_hash")
        unhashed = {key: value for key, value in manifest.items() if key != "manifest_hash"}
        actual = hash_dict(unhashed, length=64)
        if declared != actual:
            raise SystemExit(f"Split manifest integrity mismatch at {path}: declared {declared}, computed {actual}")
    return manifest


def split_metadata(split_name: str, namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """Return non-outcome metadata without parsing a label-bearing CSV."""
    manifest = read_split_manifest(namespace, config)
    path = resolve_splits_dir(namespace, config) / f"{split_name}.csv"
    if not path.is_file():
        raise SystemExit(f"Missing split file: {path}")
    outputs = manifest.get("output_files", {})
    declared = outputs.get(split_name, {})
    actual_hash = sha256_file(path)
    expected_hash = declared.get("sha256")
    if expected_hash and actual_hash != expected_hash:
        raise SystemExit(f"Split hash mismatch for {split_name}: expected {expected_hash}, got {actual_hash}")
    return {
        "exists": True, "path": str(path), "sha256": actual_hash,
        "row_count": declared.get("row_count", manifest.get("images_per_split", {}).get(split_name)),
        "schema_hash": manifest.get("source_schema_hash"),
        "manifest_hash": manifest.get("manifest_hash"), "namespace": namespace,
    }


def validate_final_eval_context(context_path: str | Path, run_id: str, namespace: str) -> dict[str, Any]:
    path = Path(context_path)
    if not path.is_file():
        raise FinalEvalAccessViolation(f"Final-evaluation context is not registered: {path}")
    record = read_json(path)
    if record.get("final_eval_run_id") != run_id or record.get("namespace") != namespace:
        raise FinalEvalAccessViolation("Final-evaluation context run ID/namespace mismatch")
    if record.get("status") not in {"registered", "outcome_accessed", "in_progress", "complete"}:
        raise FinalEvalAccessViolation(f"Final-evaluation context has invalid status {record.get('status')!r}")
    required = ("split_manifest_hash", "asism_manifest_hash", "protocol_manifest_hash", "threshold_policy_hash", "checkpoint_hashes", "code_git_version", "code_identity_sha256")
    missing = [key for key in required if not record.get(key)]
    if missing:
        raise FinalEvalAccessViolation(f"Final-evaluation context is incomplete: {missing}")
    return record


def load_split(
    split_name: str,
    namespace: str,
    config: DictConfig | None = None,
    purpose: str = "load_labels_for_evaluation",
    final_eval_run_id: str | None = None,
    final_eval_context_path: str | Path | None = None,
    caller: str = "unknown",
) -> pd.DataFrame:
    """Load one split CSV, enforcing the final_eval_heldout access rule.

    `purpose` is only consulted for final_eval_heldout; the other five splits are ordinary
    development data with no such restriction.
    """
    if split_name not in SPLIT_NAMES:
        raise ValueError(f"Unknown split {split_name!r}; expected one of {SPLIT_NAMES}")

    if split_name == FINAL_EVAL_SPLIT:
        assert_final_eval_access_allowed(
            purpose=purpose, final_eval_run_id=final_eval_run_id, caller=caller
        )
        if purpose in NON_OUTCOME_PURPOSES:
            raise FinalEvalAccessViolation(
                "Metadata-only final-evaluation checks must use split_metadata(); load_split() "
                "never returns final_eval_heldout rows before a validated evaluation context"
            )
        if purpose in OUTCOME_PURPOSES:
            if not final_eval_context_path or not final_eval_run_id:
                raise FinalEvalAccessViolation("Outcome-bearing final data access requires a validated registered context")
            validate_final_eval_context(final_eval_context_path, final_eval_run_id, namespace)

    path = resolve_splits_dir(namespace, config) / f"{split_name}.csv"
    if not path.is_file():
        raise SystemExit(
            f"Missing split file: {path}\n"
            f"Build it first: python scripts/data/02b_build_sixway_splits.py --namespace {namespace}"
        )
    return pd.read_csv(path)


def split_provenance(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """The provenance block every downstream stage records and validates against, so a stage can
    never silently consume outputs built from a different split partition.

    Deliberately small: the manifest hash pins the entire partition (seed, fractions, patient
    assignment) without every downstream manifest having to embed all of it.
    """
    manifest = read_split_manifest(namespace, config)
    return {
        "split_namespace": namespace,
        "split_manifest_version": manifest.get("manifest_version"),
        "split_manifest_hash": manifest.get("manifest_hash"),
        "split_seed": manifest.get("split_seed"),
        "frozen": manifest.get("frozen", False),
    }


def build_patient_folds(patient_ids: list[str], fold_count: int, seed: int) -> dict[int, set[str]]:
    """Deterministic patient-level validation folds; disjoint with an exact union."""
    if fold_count < 2:
        raise ValueError("fold_count must be at least 2")
    ordered = sorted(set(map(str, patient_ids)))
    random.Random(seed).shuffle(ordered)
    folds = {index: set() for index in range(fold_count)}
    for index, patient_id in enumerate(ordered):
        folds[index % fold_count].add(patient_id)
    union = set().union(*folds.values())
    if union != set(ordered) or sum(len(v) for v in folds.values()) != len(union):
        raise AssertionError("fold construction is not an exact disjoint patient partition")
    return folds


def freeze_patient_folds(path: str | Path, patient_ids: list[str], fold_count: int, seed: int, split_manifest_hash: str) -> dict[str, Any]:
    """Freeze a patient-level fold assignment as an immutable artifact.

    NOT ON THE CURRENT PATH: the only caller is 03_tune_freeze_select.py, the superseded
    weighted-score selector. The learned stages (04-09) split by subset role instead
    (learned.split_image_pool), which is image-disjoint by construction.
    """
    folds = build_patient_folds(patient_ids, fold_count, seed)
    core = {
        "schema_version": 1, "assignment_unit": "patient_id", "fold_count": fold_count,
        "assignment_seed": seed, "split_manifest_hash": split_manifest_hash,
        "pool_patient_set_hash": hash_dict({"patient_ids": sorted(set(map(str, patient_ids)))}, length=64),
        "folds": {str(i): {"patient_ids": sorted(ids), "patient_set_hash": hash_dict({"patient_ids": sorted(ids)}, length=64)} for i, ids in folds.items()},
    }
    core["fold_manifest_hash"] = hash_dict(core, length=64)
    write_frozen_json(path, core)
    return {**core, "frozen": True}


def require_frozen_production_split_run(namespace: str, config: DictConfig | None = None) -> dict[str, Any]:
    """Require production class, freeze state, and support for any versioned production run ID."""
    manifest = read_split_manifest(namespace, config)
    if manifest.get("namespace_class") != "production":
        raise SystemExit(f"Split run {namespace!r} is not a production namespace (namespace_class={manifest.get('namespace_class')!r})")
    if not manifest.get("frozen", False):
        raise SystemExit(f"Production split run {namespace!r} is not frozen")
    if not manifest.get("support_check", {}).get("passed", False):
        raise SystemExit(f"Production split run {namespace!r} failed the frozen patient-support rule")
    return manifest


## `scripts/utils/artifact_contracts.py`

*163 lines*

**Role:** the actual enforcement layer for "a stage must validate its upstream artifacts". Two
jobs:

1. **Namespace-safe paths.** `stage2_paths`, `stage3_paths`, `stage4_paths`,
   `auxiliary_checkpoint_path` - every artifact path in Stage 2-4 is derived here, always under a
   namespace directory. This is why a dev run and a production run cannot collide.
2. **Strict provenance contracts.** `require_manifest_fields` compares a manifest against expected
   values and lists every mismatch. `require_generation_complete` refuses to proceed unless the
   Stage 2 completion record matches the namespace, the split manifest hash, the generation
   manifest's own file hash, **and** the current `code_identity` hash. `require_score_artifact`
   does the same for each ASISM signal Parquet via its `.provenance.json` sidecar.
   `asism_score_provenance` composes the full expected chain: split -> generation -> LoRA checkpoint
   -> ASISM config -> code identity.

A config change alters the config hash, which forces a re-run from the affected stage forward
rather than silently mixing incompatible outputs.

In [ ]:
# ===== FILE: scripts/utils/artifact_contracts.py =====
"""Namespace-safe Stage 2/3 artifact paths and strict provenance contracts."""
from __future__ import annotations

from pathlib import Path
from typing import Any

from omegaconf import OmegaConf

from scripts.utils.manifest import code_identity, hash_dict, read_json, sha256_file
from scripts.utils.splits import read_split_manifest, resolve_splits_dir


class ArtifactContractError(SystemExit):
    pass


def validate_namespace_run_id(namespace: str, namespace_class: str) -> None:
    prefix = f"{namespace_class}-"
    if namespace_class not in {"dev", "production"} or not namespace.startswith(prefix) or len(namespace) <= len(prefix):
        raise ArtifactContractError(
            f"Invalid immutable split run ID {namespace!r}: a {namespace_class!r} run must be versioned as {prefix}<run-or-version>"
        )


def namespace_identity(namespace: str) -> dict[str, Any]:
    manifest = read_split_manifest(namespace)
    namespace_class = manifest.get("namespace_class")
    if namespace_class not in {"dev", "production"}:
        raise ArtifactContractError(
            f"Split run {namespace!r} has no valid namespace_class; rebuild it with the current six-way builder"
        )
    validate_namespace_run_id(namespace, namespace_class)
    return {
        "split_namespace": namespace,
        "namespace_class": namespace_class,
        "split_manifest_version": manifest["manifest_version"],
        "split_manifest_hash": manifest["manifest_hash"],
        "split_manifest_file_sha256": sha256_file(resolve_splits_dir(namespace) / "split_manifest_v2.json"),
        "split_frozen": bool(manifest.get("frozen", False)),
    }


def stage2_paths(config, namespace: str) -> dict[str, Path]:
    root = Path(config.paths.synthetic_root) / namespace
    pilot = root / "pilot"
    return {
        "root": root,
        "images_dir": root / "images",
        "recipes_path": root / "label_recipes.csv",
        "recipe_decisions_path": root / "recipe_decisions.jsonl",
        "recipes_manifest": root / "recipes_manifest.json",
        "manifest_path": root / "generation_manifest.jsonl",
        "generation_provenance": root / "generation_provenance.json",
        "generation_completion": root / "generation_complete.json",
        "pilot_dir": pilot,
        "pilot_images": pilot / "images",
        "pilot_manifest": pilot / "pilot_generation_manifest.jsonl",
        "pilot_approval_manifest": pilot / "pilot_approval_manifest.json",
    }


def stage3_paths(config, namespace: str) -> dict[str, Path]:
    root = Path(config.paths.synthetic_root) / namespace
    asism = root / "asism"
    return {
        "root": root,
        "scores_dir": root / "scores",
        "asism_dir": asism,
        "gonogo_report": asism / "gonogo_report.json",
        "search_log": asism / "proxy_search_log.jsonl",
        "frozen_manifest": asism / "asism_frozen_manifest.json",
        "fold_manifest": asism / "asism_patient_folds.json",
        "selected_manifest": asism / "selected_manifest.jsonl",
        "rejected_log": asism / "rejected_log.jsonl",
        "learned_dir": asism / "learned",
        "subset_design_report": asism / "learned" / "subset_design_report.json",
        "utility_subsets": asism / "learned" / "utility_subsets.jsonl",
        "utility_results": asism / "learned" / "utility_results.jsonl",
        "learned_selected_manifest": asism / "learned" / "selected_manifest.jsonl",
        "adaptive_selected_manifest": asism / "learned" / "adaptive_selected_manifest.jsonl",
        "adaptive_selection_manifest": asism / "learned" / "adaptive_selection_manifest.json",
        "learned_rejected_log": asism / "learned" / "rejected_log.jsonl",
        "threshold_contexts": asism / "learned" / "threshold_contexts.jsonl",
        "threshold_candidate_evaluations": asism / "learned" / "threshold_candidate_evaluations.jsonl",
        "proxy_verification_plan": asism / "learned" / "proxy_verification_plan.json",
        "threshold_proxy_measurements": asism / "learned" / "threshold_proxy_measurements.jsonl",
        "threshold_training_targets_verified": asism / "learned" / "threshold_training_targets_verified.jsonl",
        "threshold_training_targets_exploratory": asism / "learned" / "threshold_training_targets_exploratory.jsonl",
        "official_threshold_checkpoint": asism / "learned" / "official_threshold_model.pt",
        "exploratory_threshold_checkpoint": asism / "learned" / "exploratory_threshold_model.pt",
        "adaptive_threshold_manifest": asism / "learned" / "adaptive_threshold_manifest.json",
        "full_policy_verification_plan": asism / "learned" / "full_policy_verification_plan.json",
        "full_policy_verification_results": asism / "learned" / "full_policy_verification_results.jsonl",
    }


def auxiliary_checkpoint_path(config, namespace: str) -> Path:
    configured = Path(config.auxiliary_classifier.checkpoint_path)
    return configured.parent / namespace / configured.name


def stage4_paths(config, namespace: str) -> dict[str, Path]:
    checkpoints = Path(config.paths.checkpoints_dir) / namespace
    results = Path(config.paths.results_dir) / namespace
    return {
        "checkpoints_dir": checkpoints, "results_dir": results,
        "protocol_manifest": results / "frozen_experiment_protocol.json",
        "label_policy_manifest": results / "frozen_label_policy.json",
        "threshold_policy_manifest": results / "frozen_threshold_policy.json",
        "seed_plan_manifest": results / "frozen_seed_run_plan.json",
        "d_matching_policy_manifest": results / "frozen_d_matching_policy.json",
    }


def config_sha256(config_section) -> str:
    return hash_dict(OmegaConf.to_container(config_section, resolve=True), length=64)


def require_manifest_fields(path: Path, expected: dict[str, Any], label: str) -> dict[str, Any]:
    if not path.is_file():
        raise ArtifactContractError(f"Missing {label} manifest: {path}")
    actual = read_json(path)
    mismatches = {key: {"expected": value, "actual": actual.get(key)} for key, value in expected.items() if actual.get(key) != value}
    if mismatches:
        raise ArtifactContractError(f"Stale/incompatible {label} artifact at {path}: {mismatches}")
    return actual


def current_code_identity_hash() -> str:
    return code_identity()["source_tree_sha256"]


def require_generation_complete(config, namespace: str) -> tuple[dict[str, Path], dict[str, Any]]:
    paths = stage2_paths(config, namespace)
    identity = namespace_identity(namespace)
    expected = {"split_namespace": namespace, "namespace_class": identity["namespace_class"],
                "split_manifest_hash": identity["split_manifest_hash"], "status": "complete"}
    completion = require_manifest_fields(paths["generation_completion"], expected, "Stage 2 generation completion")
    if not paths["manifest_path"].is_file() or completion.get("generation_manifest_sha256") != sha256_file(paths["manifest_path"]):
        raise ArtifactContractError("Generation manifest is missing or differs from its completion record")
    if completion.get("code_identity_sha256") != current_code_identity_hash():
        raise ArtifactContractError("Generation artifacts were produced by a different source-tree identity")
    return paths, completion


def require_score_artifact(path: Path, signal: str, expected_provenance: dict[str, Any]) -> dict[str, Any]:
    sidecar = path.with_suffix(".provenance.json")
    expected = {"schema_version": 2, "signal": signal, **expected_provenance}
    manifest = require_manifest_fields(sidecar, expected, f"ASISM {signal} score")
    if not path.is_file() or manifest.get("parquet_sha256") != sha256_file(path):
        raise ArtifactContractError(f"ASISM {signal} Parquet hash mismatch: {path}")
    return manifest


def asism_score_provenance(stage3_config, stage2_config, namespace: str) -> dict[str, Any]:
    stage2, generation = require_generation_complete(stage2_config, namespace)
    return {
        **namespace_identity(namespace), "asism_config_sha256": config_sha256(stage3_config),
        "generation_manifest_sha256": sha256_file(stage2["manifest_path"]),
        "generation_completion_sha256": sha256_file(stage2["generation_completion"]),
        "lora_checkpoint_sha256": generation["lora_checkpoint_sha256"],
        "code_identity_sha256": current_code_identity_hash(),
    }


## `scripts/utils/experiment_registry.py`

*162 lines*

**Role:** one Parquet row per run, so every Stage 1-5 run is queryable in one place
(`outputs/experiments/experiment_registry.parquet`) instead of one manifest file at a time.

`parent_experiment_id` is the useful part: a Stage 4 run points at the Stage 3 selection run it
consumed, which points at Stage 2 generation, which points at the Stage 1 checkpoint - the whole
provenance chain as a single table.

`ExperimentRun` is a context manager: it appends a `running` row on entry and updates it to
`completed` or `failed` on exit (it never swallows the exception). Writes are
read -> modify -> atomic-replace; there is deliberately no locking layer, because the real execution
model is one researcher, one GPU, sequential runs.

In [ ]:
# ===== FILE: scripts/utils/experiment_registry.py =====
"""Global, cross-stage experiment registry: one parquet row per run, so every Stage 1-5 run is
queryable in a single place (outputs/experiments/experiment_registry.parquet) and nothing
overwrites a previous run's record.

Generalizes the per-artifact manifest provenance already produced by every stage (see
scripts/utils/manifest.py: make_run_id, build_checkpoint_metadata) into something queryable
*across* runs, rather than only inspectable one manifest file at a time. `parent_experiment_id`
lets a Stage 4 classifier run point at the Stage 3 selection run it consumed, which points at the
Stage 2 generation run, which points at the Stage 1 checkpoint — the full provenance chain as one
table.

Concurrency note: appends/updates are implemented as read-existing -> modify -> atomic replace
(temp file + os.replace, matching the write_json idiom in manifest.py). This is correct and
sufficient for this project's actual execution model (a single researcher, a single GPU,
sequential runs) and deliberately does not add a locking layer or a real database, which would be
over-engineering for a use case with no concurrent writers.
"""

from __future__ import annotations

import json
import os
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from scripts.utils.manifest import get_git_commit_hash, hash_dict, make_run_id

DEFAULT_REGISTRY_RELATIVE_PATH = Path("outputs") / "experiments" / "experiment_registry.parquet"

REGISTRY_COLUMNS = [
    "experiment_id",
    "parent_experiment_id",
    "timestamp",
    "stage",
    "git_sha",
    "config_hash",
    "dataset_version",
    "checkpoint_path",
    "execution_time_seconds",
    "metrics",
    "status",
]


def resolve_registry_path(project_root: str | Path | None = None) -> Path:
    """Default registry location: <PROJECT_ROOT>/outputs/experiments/experiment_registry.parquet."""
    root = Path(project_root) if project_root is not None else Path(os.environ.get("PROJECT_ROOT", "."))
    return root / DEFAULT_REGISTRY_RELATIVE_PATH


def _atomic_write_parquet(path: Path, df: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        df.to_parquet(temporary, index=False)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _append_row(registry_path: Path, row: dict[str, Any]) -> None:
    if registry_path.exists():
        existing = pd.read_parquet(registry_path)
        updated = pd.concat([existing, pd.DataFrame([row], columns=REGISTRY_COLUMNS)], ignore_index=True)
    else:
        updated = pd.DataFrame([row], columns=REGISTRY_COLUMNS)
    _atomic_write_parquet(registry_path, updated)


def _update_row(registry_path: Path, experiment_id: str, updates: dict[str, Any]) -> None:
    if not registry_path.exists():
        raise FileNotFoundError(f"Registry not found: {registry_path}")
    df = pd.read_parquet(registry_path)
    mask = df["experiment_id"] == experiment_id
    if not mask.any():
        raise KeyError(f"experiment_id {experiment_id!r} not found in {registry_path}")
    for key, value in updates.items():
        df.loc[mask, key] = json.dumps(value, sort_keys=True, default=str) if isinstance(value, (dict, list)) else value
    _atomic_write_parquet(registry_path, df)


class ExperimentRun:
    """Context manager registering one experiment-registry row per run.

    Usage:
        with ExperimentRun(stage="stage2_generation", config=cfg_dict,
                            parent_experiment_id=stage1_checkpoint_run_id) as run:
            ... do the stage's work ...
            run.set_checkpoint_path(output_dir)
            run.set_metrics({"num_images": 5000})

    On normal exit, status is recorded as "completed"; on any exception, "failed" (the exception
    still propagates — this context manager never swallows errors).
    """

    def __init__(
        self,
        stage: str,
        config: dict[str, Any],
        parent_experiment_id: str | None = None,
        dataset_version: str | None = None,
        experiment_id: str | None = None,
        registry_path: str | Path | None = None,
        repo_dir: str | Path = ".",
    ) -> None:
        self.stage = stage
        self.config = config
        self.parent_experiment_id = parent_experiment_id
        self.dataset_version = dataset_version
        self.experiment_id = experiment_id or make_run_id(config, repo_dir=repo_dir)
        self.registry_path = Path(registry_path) if registry_path is not None else resolve_registry_path()
        self.repo_dir = repo_dir
        self.checkpoint_path: str | None = None
        self.metrics: dict[str, Any] = {}
        self.status = "running"
        self._start_time: float | None = None

    def set_checkpoint_path(self, path: str | Path) -> None:
        self.checkpoint_path = str(path)

    def set_metrics(self, metrics: dict[str, Any]) -> None:
        self.metrics.update(metrics)

    def __enter__(self) -> "ExperimentRun":
        self._start_time = time.monotonic()
        row = {
            "experiment_id": self.experiment_id,
            "parent_experiment_id": self.parent_experiment_id,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "stage": self.stage,
            "git_sha": get_git_commit_hash(self.repo_dir),
            "config_hash": hash_dict(self.config),
            "dataset_version": self.dataset_version,
            "checkpoint_path": None,
            "execution_time_seconds": None,
            "metrics": "{}",
            "status": self.status,
        }
        _append_row(self.registry_path, row)
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        elapsed = time.monotonic() - self._start_time if self._start_time is not None else None
        self.status = "failed" if exc_type is not None else "completed"
        _update_row(
            self.registry_path,
            self.experiment_id,
            {
                "checkpoint_path": self.checkpoint_path,
                "execution_time_seconds": elapsed,
                "metrics": self.metrics,
                "status": self.status,
            },
        )
        return False

## `scripts/utils/metrics.py`

*443 lines*

**Role:** every Stage 5 metric and statistic. Pure NumPy - no torch, no GPU - which is what makes
the whole analysis layer developable and testable locally, off the pod.

- `auroc` - via the Mann-Whitney rank identity with correct mid-rank tie handling. Returns **NaN**
  when a class is absent, not a silent 0.5, so an undefined AUROC is excluded from a macro-average
  instead of dragging it toward chance.
- `average_precision`, `brier_score`, `expected_calibration_error`, `sensitivity_specificity_f1`.
- `macro_auroc_masked`, `full_metric_suite` - per-label metrics **with effective N attached**, so a
  metric over 40 usable patients is never silently compared to one over 400.
- `patient_level_bootstrap`, `paired_bootstrap_difference` - resampling is at **patient** level;
  bootstrapping images would let a patient with many studies dominate a confidence interval purely
  by study count.
- `holm_bonferroni` (confirmatory family) and `benjamini_hochberg` (exploratory) - the frozen
  multiplicity policy.

Everything is **masked**: uncertain and blank labels are excluded, never mapped to 0 or 1.

In [ ]:
# ===== FILE: scripts/utils/metrics.py =====
"""Masked metrics and patient-level inference for Stage 5 (docs/stages2_to_5_plan.md §5, §6, §8).

Every metric here is MASKED: uncertain (-1) and blank labels are excluded rather than mapped to 0
or 1 (§6), and every result carries its effective N so a metric computed over 40 usable patients is
never silently compared against one computed over 400.

Resampling is PATIENT-level (§8). Bootstrapping images would let a patient with many studies
dominate a confidence interval purely by study count, which understates the true uncertainty.

Pure NumPy — no torch, no GPU. This module is fully testable and runnable locally, which is what
makes the whole Stage 5 analysis layer developable off the pod.
"""

from __future__ import annotations

import numpy as np

from scripts.utils.labels import (
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
)


def auroc(scores: np.ndarray, labels: np.ndarray) -> float:
    """AUROC via the rank (Mann-Whitney U) identity, with correct mid-rank tie handling.

    Returns NaN when either class is absent — an undefined AUROC, not a silent 0.5, so it can be
    excluded from a macro-average rather than dragging it toward chance.
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    n_positive = int((labels == 1).sum())
    n_negative = int((labels == 0).sum())
    if n_positive == 0 or n_negative == 0:
        return float("nan")

    order = np.argsort(scores, kind="mergesort")
    ranks = np.empty(len(scores), dtype=np.float64)
    sorted_scores = scores[order]

    index = 0
    while index < len(sorted_scores):
        end = index
        while end + 1 < len(sorted_scores) and sorted_scores[end + 1] == sorted_scores[index]:
            end += 1
        mid_rank = (index + end) / 2.0 + 1.0
        ranks[order[index : end + 1]] = mid_rank
        index = end + 1

    positive_rank_sum = ranks[labels == 1].sum()
    u_statistic = positive_rank_sum - n_positive * (n_positive + 1) / 2.0
    return float(u_statistic / (n_positive * n_negative))


def average_precision(scores: np.ndarray, labels: np.ndarray) -> float:
    """Area under the precision-recall curve (step interpolation, the standard AP definition)."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    if (labels == 1).sum() == 0:
        return float("nan")

    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    true_positives = np.cumsum(sorted_labels == 1)
    predicted_positives = np.arange(1, len(sorted_labels) + 1)
    precision = true_positives / predicted_positives
    total_positives = (labels == 1).sum()
    return float((precision * (sorted_labels == 1)).sum() / total_positives)


def brier_score(scores: np.ndarray, labels: np.ndarray) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    if len(scores) == 0:
        return float("nan")
    return float(np.mean((scores - labels) ** 2))


def expected_calibration_error(scores: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """ECE with equal-width bins."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    if len(scores) == 0:
        return float("nan")

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        in_bin = (scores > lower) & (scores <= upper) if lower > 0 else (scores >= lower) & (scores <= upper)
        count = int(in_bin.sum())
        if count == 0:
            continue
        error += (count / len(scores)) * abs(labels[in_bin].mean() - scores[in_bin].mean())
    return float(error)


def sensitivity_specificity_f1(
    scores: np.ndarray, labels: np.ndarray, threshold: float
) -> tuple[float, float, float]:
    predicted = (np.asarray(scores) >= threshold).astype(np.int64)
    labels = np.asarray(labels, dtype=np.int64)

    true_positive = int(((predicted == 1) & (labels == 1)).sum())
    false_positive = int(((predicted == 1) & (labels == 0)).sum())
    true_negative = int(((predicted == 0) & (labels == 0)).sum())
    false_negative = int(((predicted == 0) & (labels == 1)).sum())

    sensitivity = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else float("nan")
    specificity = true_negative / (true_negative + false_positive) if (true_negative + false_positive) else float("nan")
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else float("nan")
    if np.isnan(precision) or np.isnan(sensitivity) or (precision + sensitivity) == 0:
        f1 = float("nan")
    else:
        f1 = 2 * precision * sensitivity / (precision + sensitivity)
    return float(sensitivity), float(specificity), float(f1)


def macro_auroc_masked(
    probabilities: np.ndarray,
    targets: np.ndarray,
    masks: np.ndarray,
    labels: list[str] | None = None,
    primary_labels: list[str] | None = None,
    min_positive: int = 1,
    min_negative: int = 1,
) -> dict:
    """Per-label and macro AUROC over the PRIMARY endpoint labels, masked.

    Labels failing the minimum-support rule are excluded from the macro-average by the
    pre-specified rule (§5.3) and reported with their support counts rather than an unstable value.
    """
    labels = labels or CLASSIFIER_TARGET_LABELS
    primary_labels = primary_labels or PRIMARY_ENDPOINT_LABELS
    index_of = {label: position for position, label in enumerate(labels)}

    per_label = {}
    usable = []
    for label in primary_labels:
        if label not in index_of:
            continue
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        y_true = targets[mask, column].astype(np.int64)
        y_score = probabilities[mask, column]

        n_positive = int((y_true == 1).sum())
        n_negative = int((y_true == 0).sum())
        eligible = n_positive >= min_positive and n_negative >= min_negative
        value = auroc(y_score, y_true) if eligible else float("nan")

        per_label[label] = {
            "auroc": value,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "effective_n": int(mask.sum()),
            "eligible": bool(eligible and not np.isnan(value)),
        }
        if per_label[label]["eligible"]:
            usable.append(value)

    return {
        "macro_auroc": float(np.mean(usable)) if usable else float("nan"),
        "n_labels_in_macro": len(usable),
        "n_labels_excluded": len(primary_labels) - len(usable),
        "per_label": per_label,
    }


def full_metric_suite(
    probabilities: np.ndarray,
    targets: np.ndarray,
    masks: np.ndarray,
    thresholds: dict[str, float] | None = None,
    labels: list[str] | None = None,
    min_positive: int = 1,
    min_negative: int = 1,
) -> dict:
    """Every Stage 5 metric (§8) for one condition, per label, with effective N attached."""
    labels = labels or CLASSIFIER_TARGET_LABELS
    thresholds = thresholds or {}
    index_of = {label: position for position, label in enumerate(labels)}

    per_label = {}
    for label in labels:
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        y_true = targets[mask, column].astype(np.int64)
        y_score = probabilities[mask, column]

        n_positive = int((y_true == 1).sum())
        n_negative = int((y_true == 0).sum())
        eligible = n_positive >= min_positive and n_negative >= min_negative
        threshold = float(thresholds.get(label, 0.5))
        sensitivity, specificity, f1 = (
            sensitivity_specificity_f1(y_score, y_true, threshold)
            if eligible
            else (float("nan"), float("nan"), float("nan"))
        )

        per_label[label] = {
            "auroc": auroc(y_score, y_true) if eligible else float("nan"),
            "auprc": average_precision(y_score, y_true) if eligible else float("nan"),
            "sensitivity": sensitivity,
            "specificity": specificity,
            "f1": f1,
            "brier": brier_score(y_score, y_true) if eligible else float("nan"),
            "ece": expected_calibration_error(y_score, y_true) if eligible else float("nan"),
            "threshold": threshold,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "effective_n": int(mask.sum()),
            "eligible": bool(eligible),
        }

    def macro_over(metric: str, subset: list[str]) -> float:
        values = [
            per_label[label][metric]
            for label in subset
            if label in per_label and per_label[label]["eligible"]
            and not np.isnan(per_label[label][metric])
        ]
        return float(np.mean(values)) if values else float("nan")

    primary = [label for label in PRIMARY_ENDPOINT_LABELS if label in per_label]
    secondary = [label for label in labels if label not in PRIMARY_ENDPOINT_LABELS]

    # Micro-AUROC pools all confidently-labelled (row, primary-label) pairs into one ranking.
    micro_scores, micro_targets = [], []
    for label in primary:
        column = index_of[label]
        mask = masks[:, column].astype(bool)
        micro_scores.append(probabilities[mask, column])
        micro_targets.append(targets[mask, column].astype(np.int64))

    return {
        "macro_auroc_primary": macro_over("auroc", primary),
        "macro_auprc_primary": macro_over("auprc", primary),
        "micro_auroc_primary": (
            auroc(np.concatenate(micro_scores), np.concatenate(micro_targets))
            if micro_scores else float("nan")
        ),
        "macro_f1_primary": macro_over("f1", primary),
        "mean_brier_primary": macro_over("brier", primary),
        "mean_ece_primary": macro_over("ece", primary),
        "primary_labels": primary,
        "secondary_labels": secondary,
        "per_label": per_label,
    }


def patient_level_bootstrap(
    patient_ids: np.ndarray,
    metric_fn,
    n_resamples: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
) -> dict:
    """Patient-level bootstrap CI (§8).

    Resamples PATIENTS with replacement and re-evaluates on all rows belonging to the drawn
    patients, so a patient's whole set of studies moves in or out together — the correct unit,
    since studies from one patient are not independent observations.
    """
    rng = np.random.default_rng(seed)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    rows_by_patient = {patient: np.where(patient_ids == patient)[0] for patient in unique_patients}

    point_estimate = metric_fn(np.arange(len(patient_ids)))

    samples = []
    for _ in range(n_resamples):
        drawn = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([rows_by_patient[patient] for patient in drawn])
        value = metric_fn(indices)
        if value is not None and not np.isnan(value):
            samples.append(value)

    if not samples:
        return {
            "point_estimate": point_estimate,
            "ci_lower": float("nan"),
            "ci_upper": float("nan"),
            "n_resamples_valid": 0,
            "n_patients": len(unique_patients),
        }

    samples = np.array(samples)
    return {
        "point_estimate": float(point_estimate),
        "ci_lower": float(np.percentile(samples, 100 * alpha / 2)),
        "ci_upper": float(np.percentile(samples, 100 * (1 - alpha / 2))),
        "bootstrap_mean": float(samples.mean()),
        "bootstrap_std": float(samples.std(ddof=1)) if len(samples) > 1 else float("nan"),
        "n_resamples_valid": len(samples),
        "n_patients": len(unique_patients),
    }


def paired_bootstrap_difference(
    patient_ids: np.ndarray,
    metric_fn_a,
    metric_fn_b,
    n_resamples: int = 1000,
    seed: int = 42,
    alpha: float = 0.05,
) -> dict:
    """Paired patient-level bootstrap of (A - B), with an effect size and a two-sided p-value.

    Paired: both conditions are evaluated on the SAME resampled patients each iteration, which
    removes patient-composition variance from the comparison and is why this is more sensitive than
    comparing two independent CIs.

    The p-value is the standard bootstrap proportion of resamples whose difference crosses zero.
    It is always reported WITH the effect size and CI (§8.2) — significance alone is not a result.
    """
    rng = np.random.default_rng(seed)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    rows_by_patient = {patient: np.where(patient_ids == patient)[0] for patient in unique_patients}

    all_rows = np.arange(len(patient_ids))
    observed = metric_fn_a(all_rows) - metric_fn_b(all_rows)

    differences = []
    for _ in range(n_resamples):
        drawn = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        indices = np.concatenate([rows_by_patient[patient] for patient in drawn])
        value_a, value_b = metric_fn_a(indices), metric_fn_b(indices)
        if not (np.isnan(value_a) or np.isnan(value_b)):
            differences.append(value_a - value_b)

    if not differences:
        return {
            "observed_difference": float(observed),
            "ci_lower": float("nan"),
            "ci_upper": float("nan"),
            "p_value": float("nan"),
            "n_resamples_valid": 0,
        }

    differences = np.array(differences)
    proportion_crossing = float(np.mean(differences <= 0)) if observed > 0 else float(np.mean(differences >= 0))
    p_value = float(min(1.0, 2 * proportion_crossing))

    return {
        "observed_difference": float(observed),
        "effect_size": float(observed),
        "ci_lower": float(np.percentile(differences, 100 * alpha / 2)),
        "ci_upper": float(np.percentile(differences, 100 * (1 - alpha / 2))),
        "bootstrap_mean_difference": float(differences.mean()),
        "p_value": p_value,
        "n_resamples_valid": len(differences),
        "n_patients": len(unique_patients),
    }


def holm_bonferroni(p_values: dict[str, float], alpha: float = 0.05) -> dict[str, dict]:
    """Holm-Bonferroni step-down correction for the CONFIRMATORY family (§8.2).

    Uniformly more powerful than plain Bonferroni at the same family-wise error rate, which matters
    when the confirmatory family is small and each test is expensive to obtain.
    """
    valid = {name: value for name, value in p_values.items() if not np.isnan(value)}
    ordered = sorted(valid.items(), key=lambda item: item[1])
    n_tests = len(ordered)

    results: dict[str, dict] = {}
    previous_adjusted = 0.0
    still_rejecting = True

    for rank, (name, p_value) in enumerate(ordered):
        adjusted = min(1.0, (n_tests - rank) * p_value)
        adjusted = max(adjusted, previous_adjusted)  # enforce monotonicity
        previous_adjusted = adjusted
        if adjusted > alpha:
            still_rejecting = False
        results[name] = {
            "p_value": p_value,
            "adjusted_p_value": adjusted,
            "rejected": bool(still_rejecting and adjusted <= alpha),
            "rank": rank + 1,
            "n_tests": n_tests,
            "method": "holm_bonferroni",
            "family": "confirmatory",
            "alpha": alpha,
        }

    for name, value in p_values.items():
        if name not in results:
            results[name] = {
                "p_value": value,
                "adjusted_p_value": float("nan"),
                "rejected": False,
                "method": "holm_bonferroni",
                "family": "confirmatory",
                "note": "undefined p-value (insufficient support)",
            }
    return results


def benjamini_hochberg(p_values: dict[str, float], alpha: float = 0.05) -> dict[str, dict]:
    """Benjamini-Hochberg FDR control for EXPLORATORY analyses (§8.2).

    Exploratory results corrected this way must be labelled exploratory wherever reported — FDR
    controls the expected false-discovery proportion, not the family-wise error rate, and is not
    interchangeable with the confirmatory correction.
    """
    valid = {name: value for name, value in p_values.items() if not np.isnan(value)}
    ordered = sorted(valid.items(), key=lambda item: item[1])
    n_tests = len(ordered)

    adjusted_values = []
    for rank, (_, p_value) in enumerate(ordered, start=1):
        adjusted_values.append(min(1.0, p_value * n_tests / rank))

    # Enforce monotonicity from the largest p-value downward.
    for index in range(len(adjusted_values) - 2, -1, -1):
        adjusted_values[index] = min(adjusted_values[index], adjusted_values[index + 1])

    results: dict[str, dict] = {}
    for (name, p_value), adjusted in zip(ordered, adjusted_values):
        results[name] = {
            "p_value": p_value,
            "adjusted_p_value": adjusted,
            "rejected": bool(adjusted <= alpha),
            "n_tests": n_tests,
            "method": "benjamini_hochberg_fdr",
            "family": "exploratory",
            "alpha": alpha,
        }

    for name, value in p_values.items():
        if name not in results:
            results[name] = {
                "p_value": value,
                "adjusted_p_value": float("nan"),
                "rejected": False,
                "method": "benjamini_hochberg_fdr",
                "family": "exploratory",
                "note": "undefined p-value (insufficient support)",
            }
    return results


## `scripts/utils/classifier.py`

*613 lines*

**Role:** one multi-label CXR classifier implementation with three consumers, so they cannot drift
apart: the auxiliary reference classifier (Stage 3 prerequisite), the ASISM proxy classifier
(used to measure subset utility), and the Stage 4 headline classifiers.

Two load-bearing design points:

- **MC Dropout needs dropout active at inference.** `model.eval()` disables it, so
  `enable_mc_dropout()` re-enables *only* the dropout modules and leaves BatchNorm in eval mode.
  Switching the whole model back to `train()` would make BatchNorm use batch statistics, corrupting
  the very predictions the uncertainty estimate is computed from.
- **Masked loss.** `masked_bce_loss` implements the frozen uncertainty policy: -1 and blank labels
  contribute to neither loss nor metric.

`TrainingBudget` expresses the budget in **optimizer steps, not epochs**, and `RunAccounting`
records what was actually spent - the fairness protocol that makes B vs C a data-composition
comparison rather than a training-length comparison.

In [ ]:
# ===== FILE: scripts/utils/classifier.py =====
"""Shared multi-label CXR classifier: model, dataset, masked loss, training loop, inference.

One implementation, three consumers, so they cannot drift apart:
  - the auxiliary reference classifier (Stage 3 prerequisite, docs/stages2_to_5_plan.md §2);
  - the ASISM proxy classifier used to rank candidate configurations (§4.7);
  - the Stage 4 headline experiment classifiers, conditions A/B/C (§7).

Two design points are load-bearing rather than incidental:

MC Dropout (§4.3) requires dropout ACTIVE at inference. `model.eval()` disables it, so
`enable_mc_dropout()` re-enables only the dropout modules while leaving BatchNorm in eval mode —
switching the whole model back to train() would also make BatchNorm use batch statistics, which
would corrupt the predictions the uncertainty estimate is computed from.

Masked loss (§6) implements the frozen uncertainty policy: -1 and blank labels contribute to
neither the loss nor any metric, and are never converted into a 0 or 1 the model would be trained
to reproduce.
"""

from __future__ import annotations

import json
import os
import random
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

from scripts.utils.labels import CLASSIFIER_TARGET_LABELS, build_label_arrays


@dataclass
class TrainingBudget:
    """Frozen fairness protocol (§7.1): budget is expressed in OPTIMIZER STEPS, not epochs.

    At fixed epochs a larger dataset silently receives more gradient updates, which would conflate
    "more data" with "more training" — exactly the confound condition D exists to rule out.
    """

    max_steps: int
    batch_size: int = 32
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    warmup_steps: int = 0
    eval_every_n_steps: int = 200
    # None = choose automatically (see _loader_settings). Engineering only: workers change which
    # process decodes images, never which images form a batch or their order.
    num_workers: int | None = None
    seed: int = 42


@dataclass
class RunAccounting:
    """Per-run record required by §7.1 — what was actually consumed, not what was configured."""

    optimizer_steps: int = 0
    epochs_completed: float = 0.0
    effective_dataset_size: int = 0
    real_sample_exposures: int = 0
    synthetic_sample_exposures: int = 0
    sampling_policy: str = "shuffled_concat"
    model_seed: int = 0
    dataset_id: str = ""
    config_hash: str = ""
    checkpoint_hash: str = ""
    extra: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "optimizer_steps": self.optimizer_steps,
            "epochs_completed": round(self.epochs_completed, 4),
            "effective_dataset_size": self.effective_dataset_size,
            "real_sample_exposures": self.real_sample_exposures,
            "synthetic_sample_exposures": self.synthetic_sample_exposures,
            "sampling_policy": self.sampling_policy,
            "model_seed": self.model_seed,
            "dataset_id": self.dataset_id,
            "config_hash": self.config_hash,
            "checkpoint_hash": self.checkpoint_hash,
            **self.extra,
        }


def require_torch():
    """Import torch lazily with a clean, actionable message when the GPU stack is absent."""
    try:
        import torch  # noqa: F401
        import torchvision  # noqa: F401
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the training stack is not installed here ({exc}).\n"
            "Classifier training is GPU work — run it on the RunPod pod after:\n"
            "  pip install -r environment/requirements.txt"
        ) from exc
    import torch

    return torch


def _loader_settings(device: str, requested: int | None) -> dict:
    """DataLoader worker/pinning settings for classifier training and inference.

    Decoding a 768px JPEG and resizing it to 320px costs ~10 ms of CPU per image. With
    num_workers=0 that runs serially inside the training process while the GPU waits, which makes
    every classifier run CPU-bound. Workers only change which process decodes: the sampler, and so
    the batch composition and order, stays in the main process, and __getitem__ is deterministic,
    so the batches are bit-identical to the single-process path.

    Precedence: THESIS_LOADER_WORKERS env (0 restores the original single-process path), then an
    explicit TrainingBudget.num_workers, then automatic (workers on CUDA, none on CPU so local
    tests and the CPU smoke keep their original behaviour).
    """
    env = os.environ.get("THESIS_LOADER_WORKERS", "").strip()
    if env:
        workers = int(env)
    elif requested is not None:
        workers = int(requested)
    elif str(device).startswith("cuda"):
        workers = min(8, max(1, (os.cpu_count() or 2) - 1))
    else:
        workers = 0
    settings: dict = {"num_workers": workers}
    if workers > 0:
        settings.update(pin_memory=str(device).startswith("cuda"), prefetch_factor=4)
    return settings


def _configure_cuda_kernels(device: str) -> None:
    """Let cuDNN pick the fastest convolution algorithms for the fixed 320x320 input.

    Changes only kernel selection. GPU convolutions are already non-bit-deterministic in this
    pipeline (cudnn.deterministic is never set), so this adds no new source of variation beyond
    floating-point rounding. THESIS_CUDNN_BENCHMARK=0 disables it.
    """
    import torch

    if str(device).startswith("cuda") and os.environ.get("THESIS_CUDNN_BENCHMARK", "1") != "0":
        torch.backends.cudnn.benchmark = True


class CXRRecordDataset:
    """Multi-label CXR dataset over an explicit record list.

    Each record is {"image_path": str, "labels": {label: raw_value}, "is_synthetic": bool}. Keeping
    real and synthetic rows in one uniform record type is what lets the Stage 4 conditions differ
    only by which records they are handed, rather than by separate code paths per condition.
    """

    def __init__(self, records: list[dict], resolution: int = 320, augment: bool = False):
        self.records = records
        self.resolution = resolution
        self.augment = augment
        self._labels = CLASSIFIER_TARGET_LABELS

        frame = pd.DataFrame([record["labels"] for record in records])
        for label in self._labels:
            if label not in frame.columns:
                frame[label] = np.nan
        _, self.targets, self.masks = build_label_arrays(frame[self._labels], self._labels)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        import torch
        from PIL import Image
        from torchvision.transforms import functional as TF

        record = self.records[index]
        with Image.open(record["image_path"]) as image:
            image = image.convert("RGB")
            if image.size != (self.resolution, self.resolution):
                image = image.resize((self.resolution, self.resolution), Image.BICUBIC)
            tensor = TF.to_tensor(image)

        # Horizontal flip is deliberately NOT offered: it inverts CXR left/right anatomy
        # (heart position, aortic arch, gastric bubble) — same constraint as Stage 1's data config.
        tensor = TF.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

        return {
            "image": tensor,
            "target": torch.from_numpy(self.targets[index]),
            "mask": torch.from_numpy(self.masks[index]),
            "index": index,
        }


def build_model(num_labels: int, dropout_p: float, pretrained_source: str, seed: int = 42):
    """DenseNet121 with a dropout layer before the classifier head.

    `pretrained_source` is recorded verbatim in run provenance. CheXpert-pretrained weights are
    rejected here rather than merely discouraged: most published CheXpert checkpoints do not
    publish patient-level provenance, so overlap with final_eval_heldout cannot be excluded (§2).
    """
    torch = require_torch()
    import torchvision

    if "chexpert" in str(pretrained_source).lower():
        raise SystemExit(
            f"Refusing CheXpert-pretrained initialization ({pretrained_source!r}).\n"
            "Its training patients cannot be shown disjoint from our evaluation splits, which "
            "would contaminate the headline comparison (docs/stages2_to_5_plan.md §2).\n"
            "Use 'imagenet', 'random', or a CXR checkpoint with published patient-level provenance."
        )

    torch.manual_seed(seed)
    if pretrained_source == "imagenet":
        weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
    elif pretrained_source == "random":
        weights = None
    else:
        raise SystemExit(
            f"Unsupported pretrained_source {pretrained_source!r}. "
            "Supported: 'imagenet', 'random'. A documented non-overlapping CXR checkpoint may be "
            "added here once its provenance is recorded (§2)."
        )

    model = torchvision.models.densenet121(weights=weights)
    in_features = model.classifier.in_features
    model.classifier = torch.nn.Sequential(
        torch.nn.Dropout(p=dropout_p),
        torch.nn.Linear(in_features, num_labels),
    )
    return model


def enable_mc_dropout(model) -> int:
    """Put ONLY dropout modules into train mode, leaving BatchNorm in eval mode.

    Calling model.train() instead would also switch BatchNorm to batch statistics, which changes
    the predictions themselves rather than just sampling over dropout masks — the resulting spread
    would not be the epistemic uncertainty §4.3 intends to measure.
    """
    import torch

    count = 0
    for module in model.modules():
        if isinstance(module, (torch.nn.Dropout, torch.nn.Dropout2d)):
            module.train()
            count += 1
    return count


def masked_bce_loss(logits, targets, masks):
    """Binary cross-entropy over confidently-labelled positions only (§6).

    Uncertain (-1) and blank labels are excluded from the loss entirely rather than mapped to 0/1,
    so the model is never trained to reproduce an assertion the radiologist declined to make.
    """
    import torch
    import torch.nn.functional as F

    per_element = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    masked = per_element * masks.float()
    denominator = masks.float().sum()
    if denominator.item() == 0:
        return torch.zeros((), device=logits.device, requires_grad=True)
    return masked.sum() / denominator


def train_classifier(
    train_records: list[dict],
    val_records: list[dict],
    budget: TrainingBudget,
    dropout_p: float,
    pretrained_source: str,
    resolution: int,
    device: str | None = None,
    progress_desc: str = "train",
    checkpoint_path: Path | None = None,
    resume_from: Path | None = None,
    checkpoint_metadata: dict | None = None,
) -> tuple[object, RunAccounting, list[dict]]:
    """Train to a FIXED optimizer-step budget. Returns (model, accounting, eval_history).

    Deliberately no early stopping inside this function: the ASISM proxy search forbids
    candidate-specific early stopping (§4.7), and A/B/C requires an identical protocol across
    conditions (§7.1). Checkpoint SELECTION (best-on-classifier_val) is applied by the caller that
    is entitled to do it, from the history returned here.
    """
    torch = require_torch()
    from torch.utils.data import DataLoader
    from tqdm.auto import tqdm

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    _configure_cuda_kernels(device)
    random.seed(budget.seed)
    torch.manual_seed(budget.seed)
    np.random.seed(budget.seed)

    train_dataset = CXRRecordDataset(train_records, resolution=resolution)
    sampler_generator = torch.Generator().manual_seed(budget.seed)
    loader_settings = _loader_settings(device, budget.num_workers)
    loader = DataLoader(
        train_dataset,
        batch_size=budget.batch_size,
        shuffle=True,
        drop_last=False,
        generator=sampler_generator,
        # The training loop re-iterates the loader every epoch; keep workers alive between epochs.
        persistent_workers=loader_settings["num_workers"] > 0,
        **loader_settings,
    )
    non_blocking = bool(loader_settings.get("pin_memory"))

    model = build_model(len(CLASSIFIER_TARGET_LABELS), dropout_p, pretrained_source, budget.seed)
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=budget.learning_rate, weight_decay=budget.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, budget.max_steps))

    accounting = RunAccounting(
        effective_dataset_size=len(train_records),
        real_sample_exposures=0,
        synthetic_sample_exposures=0,
        model_seed=budget.seed,
    )
    n_synthetic = sum(1 for record in train_records if record.get("is_synthetic"))
    n_real = len(train_records) - n_synthetic

    history: list[dict] = []
    step = 0
    best_metric = float("-inf")
    best_checkpoint_identity = None
    expected_meta = checkpoint_metadata or {}
    best_checkpoint_path = (
        checkpoint_path.with_name(f"{checkpoint_path.stem}.best{checkpoint_path.suffix}")
        if checkpoint_path is not None else None
    )
    if resume_from is not None:
        payload = torch.load(resume_from, map_location=device, weights_only=False)
        if payload.get("checkpoint_schema_version") != 2:
            raise ValueError("incompatible classifier checkpoint schema")
        actual_meta = payload.get("provenance", {})
        mismatches = {key: (actual_meta.get(key), value) for key, value in expected_meta.items() if actual_meta.get(key) != value}
        if mismatches:
            raise ValueError(f"incompatible classifier checkpoint provenance: {mismatches}")
        model.load_state_dict(payload["model_state"])
        optimizer.load_state_dict(payload["optimizer_state"])
        scheduler.load_state_dict(payload["scheduler_state"])
        step = int(payload["global_optimizer_step"])
        history = list(payload.get("eval_history", []))
        best_metric = float(payload.get("best_validation_metric", float("-inf")))
        best_checkpoint_identity = payload.get("best_checkpoint_identity")
        random.setstate(payload["rng_states"]["python"])
        np.random.set_state(payload["rng_states"]["numpy"])
        # ``map_location=device`` also moves the saved CPU RNG tensor to CUDA on GPU runs.
        # PyTorch requires a CPU ByteTensor when restoring the default CPU generator.
        torch_cpu_rng = payload["rng_states"]["torch_cpu"].detach().cpu()
        torch.set_rng_state(torch_cpu_rng)
        if torch.cuda.is_available() and payload["rng_states"].get("torch_cuda"):
            torch.cuda.set_rng_state_all(
                [state.detach().cpu() for state in payload["rng_states"]["torch_cuda"]]
            )
        sampler_generator.set_state(payload["sampler_state"].detach().cpu())

    def save_resumable() -> None:
        nonlocal best_checkpoint_identity
        if checkpoint_path is None:
            return
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        best_checkpoint_identity = best_checkpoint_identity or f"step-{step}"
        payload = {
            "checkpoint_schema_version": 2, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
            "global_optimizer_step": step, "step": step,
            "best_validation_metric": best_metric, "best_checkpoint_identity": best_checkpoint_identity,
            "early_stopping_state": {"enabled": False, "bad_evaluations": 0},
            "rng_states": {"python": random.getstate(), "numpy": np.random.get_state(),
                           "torch_cpu": torch.get_rng_state(),
                           "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},
            "sampler_state": sampler_generator.get_state(), "sampler_rule": "torch RandomSampler generator state",
            "eval_history": history, "provenance": expected_meta,
        }
        fd, temp_name = tempfile.mkstemp(prefix=f".{checkpoint_path.name}.", suffix=".tmp", dir=checkpoint_path.parent)
        os.close(fd)
        try:
            torch.save(payload, temp_name)
            os.replace(temp_name, checkpoint_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)

    def save_best() -> None:
        if best_checkpoint_path is None:
            return
        best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "checkpoint_schema_version": 2, "checkpoint_role": "best_validation_model",
            "model_state": model.state_dict(), "best_validation_metric": best_metric,
            "best_checkpoint_identity": best_checkpoint_identity, "provenance": expected_meta,
            "model_config": {"dropout_p": dropout_p, "pretrained_source": pretrained_source,
                             "resolution": resolution, "num_labels": len(CLASSIFIER_TARGET_LABELS)},
        }
        fd, temp_name = tempfile.mkstemp(prefix=f".{best_checkpoint_path.name}.", suffix=".tmp", dir=best_checkpoint_path.parent)
        os.close(fd)
        try:
            torch.save(payload, temp_name)
            os.replace(temp_name, best_checkpoint_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)
    model.train()
    progress = tqdm(total=budget.max_steps, initial=step, desc=progress_desc, unit="step")

    while step < budget.max_steps:
        for batch in loader:
            if step >= budget.max_steps:
                break
            images = batch["image"].to(device, non_blocking=non_blocking)
            targets = batch["target"].to(device, non_blocking=non_blocking)
            masks = batch["mask"].to(device, non_blocking=non_blocking)

            logits = model(images)
            loss = masked_bce_loss(logits, targets, masks)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            scheduler.step()

            step += 1
            progress.update(1)
            progress.set_postfix(loss=f"{loss.item():.4f}", refresh=False)

            if val_records and budget.eval_every_n_steps and step % budget.eval_every_n_steps == 0:
                metrics = evaluate_classifier(model, val_records, resolution, device)
                metrics["step"] = step
                history.append(metrics)
                if np.isfinite(metrics["macro_auroc"]) and metrics["macro_auroc"] >= best_metric:
                    best_metric = metrics["macro_auroc"]
                    best_checkpoint_identity = f"step-{step}"
                    save_best()
                save_resumable()
                model.train()

    progress.close()

    accounting.optimizer_steps = step
    seen = step * budget.batch_size
    accounting.epochs_completed = seen / max(len(train_records), 1)
    total = max(len(train_records), 1)
    accounting.real_sample_exposures = int(seen * n_real / total)
    accounting.synthetic_sample_exposures = int(seen * n_synthetic / total)

    if val_records:
        final_metrics = evaluate_classifier(model, val_records, resolution, device)
        final_metrics["step"] = step
        history.append(final_metrics)
        if np.isfinite(final_metrics["macro_auroc"]) and final_metrics["macro_auroc"] >= best_metric:
            best_metric = final_metrics["macro_auroc"]
            best_checkpoint_identity = f"step-{step}"
            save_best()
    save_resumable()

    if val_records and (best_checkpoint_path is None or not best_checkpoint_path.is_file()):
        raise RuntimeError(
            "No finite validation macro-AUROC was observed; refusing to publish an arbitrary final-step model as best"
        )

    if best_checkpoint_path is not None and best_checkpoint_path.is_file():
        best_payload = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(best_payload["model_state"])
        accounting.extra["best_checkpoint_path"] = str(best_checkpoint_path)
        accounting.extra["best_validation_metric"] = best_metric

    return model, accounting, history


def predict_probabilities(
    model,
    records: list[dict],
    resolution: int,
    device: str | None = None,
    batch_size: int = 32,
    mc_dropout_passes: int = 0,
) -> tuple[np.ndarray, np.ndarray | None]:
    """Predict per-label probabilities.

    With mc_dropout_passes > 0, returns (mean, std) over stochastic forward passes — the §4.3
    uncertainty signal. With 0, returns (probabilities, None) from a single deterministic pass.
    """
    torch = require_torch()
    from torch.utils.data import DataLoader

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    _configure_cuda_kernels(device)
    model = model.to(device)
    model.eval()

    dataset = CXRRecordDataset(records, resolution=resolution)
    loader_settings = _loader_settings(device, None)
    n_passes = max(1, mc_dropout_passes)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        # MC Dropout iterates the loader once per pass; keep the workers for all passes.
        persistent_workers=loader_settings["num_workers"] > 0 and n_passes > 1,
        **loader_settings,
    )
    non_blocking = bool(loader_settings.get("pin_memory"))

    if mc_dropout_passes > 0:
        activated = enable_mc_dropout(model)
        if activated == 0:
            raise SystemExit(
                "MC Dropout requested but the model contains no dropout modules — the uncertainty "
                "signal would silently degenerate to a constant. Check classifier.dropout_p (§2)."
            )

    all_passes = []
    with torch.no_grad():
        for _ in range(n_passes):
            batch_outputs = []
            for batch in loader:
                logits = model(batch["image"].to(device, non_blocking=non_blocking))
                batch_outputs.append(torch.sigmoid(logits).cpu().numpy())
            all_passes.append(np.concatenate(batch_outputs, axis=0))

    stacked = np.stack(all_passes, axis=0)
    mean = stacked.mean(axis=0)
    if mc_dropout_passes > 0:
        return mean, stacked.std(axis=0)
    return mean, None


def evaluate_classifier(model, records: list[dict], resolution: int, device: str | None = None) -> dict:
    """Masked macro-AUROC over the primary endpoint labels, plus effective N (§5.3, §6)."""
    from scripts.utils.metrics import macro_auroc_masked

    probabilities, _ = predict_probabilities(model, records, resolution, device)

    frame = pd.DataFrame([record["labels"] for record in records])
    for label in CLASSIFIER_TARGET_LABELS:
        if label not in frame.columns:
            frame[label] = np.nan
    _, targets, masks = build_label_arrays(frame[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)

    return macro_auroc_masked(probabilities, targets, masks)


def records_from_split(
    frame: pd.DataFrame,
    images_dir: Path,
    path_column: str = "Path",
    require_exists: bool = True,
) -> list[dict]:
    """Build classifier records from a real-data split, using Stage 1's preprocessed images.

    Rows whose preprocessed image is absent (filtered as lateral/corrupt during Stage 1
    preprocessing) are skipped rather than silently substituted.
    """
    from scripts.utils.identifiers import sanitize_image_id

    records = []
    for row in frame.to_dict("records"):
        image_id = sanitize_image_id(row[path_column])
        image_path = images_dir / f"{image_id}.jpg"
        if require_exists and not image_path.is_file():
            continue
        records.append(
            {
                "image_id": image_id,
                "image_path": str(image_path),
                "patient_id": row.get("patient_id"),
                "labels": {label: row.get(label) for label in CLASSIFIER_TARGET_LABELS},
                "is_synthetic": False,
            }
        )
    return records


def records_from_synthetic_manifest(
    manifest_path: Path,
    images_dir: Path,
    image_ids: Iterable[str] | None = None,
    require_exists: bool = True,
) -> list[dict]:
    """Build classifier records from Stage 2 output.

    The synthetic label vector is the INTENDED vector (§3) — the generator's conditioning intent,
    which Stage 3 measures agreement against and which Stage 4 then trains on. It is labelled
    `intended_label_vector` in the manifest and is never described as ground truth.
    """
    wanted = set(image_ids) if image_ids is not None else None
    records = []
    with open(manifest_path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            image_id = entry["image_id"]
            if wanted is not None and image_id not in wanted:
                continue
            image_path = images_dir / f"{image_id}.jpg"
            if require_exists and not image_path.is_file():
                continue
            labels = {label: value for label, value in entry["intended_label_vector"].items()}
            labels["No Finding"] = 1 if entry.get("is_no_finding") else 0
            labels["Support Devices"] = int(entry.get("support_devices", 0))
            records.append(
                {
                    "image_id": image_id,
                    "image_path": str(image_path),
                    "patient_id": f"synthetic::{image_id}",
                    "labels": labels,
                    "is_synthetic": True,
                }
            )
    return records


---

# Part 3 - Data preparation (`scripts/data/`)

Acquire CheXpert, prove it is intact, cut the six patient-disjoint splits, preprocess the images, and write the captions. Everything downstream is hashed against the split manifest produced here.

## `scripts/data/00_download_dataset.py`

*141 lines*

**Pipeline step 1.** Downloads CheXpert-v1.0-small from Kaggle (`ashery/chexpert`) with
`kagglehub`, locates the real data root inside whatever nesting the archive used
(`find_data_root`), links or copies it into `data/chexpert/raw/` (`link_or_copy` - prefers a
symlink so the dataset is not duplicated on the persistent volume, falls back to copying on
Windows without symlink rights), then runs `01_verify_download.py` automatically.

**Idempotent**: never overwrites an existing entry. **Needs** Kaggle credentials configured for
`kagglehub` (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME`/`KAGGLE_KEY`).

In [ ]:
# ===== FILE: scripts/data/00_download_dataset.py =====
from __future__ import annotations

import argparse
import re
import shutil
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import CONFIGS_DIR, ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402

VERIFY_SCRIPT = Path(__file__).resolve().parent / "01_verify_download.py"


def find_data_root(base: Path) -> Path:
    """Locate the directory that directly contains train.csv + a train/ subdir, searching a few
    likely nesting levels before falling back to a full recursive search."""
    candidates = [base, base / "CheXpert-v1.0-small", base / "CheXpert-v1.0"]
    for candidate in candidates:
        if (candidate / "train.csv").exists() and (candidate / "train").is_dir():
            return candidate

    for train_csv in base.rglob("train.csv"):
        if (train_csv.parent / "train").is_dir():
            return train_csv.parent

    raise FileNotFoundError(
        f"Could not locate a CheXpert 'train.csv' (with sibling 'train/' dir) anywhere under {base}"
    )


def link_or_copy(source: Path, target: Path, force_copy: bool) -> str:
    """Idempotent per-entry: never overwrites an existing target. Prefers a symlink (dataset stays
    out of the repo/persistent-volume duplication); falls back to copying if symlinking isn't
    permitted (e.g. Windows without Developer Mode / admin rights)."""
    if target.exists() or target.is_symlink():
        return "skipped (already present)"

    if not force_copy:
        try:
            target.symlink_to(source, target_is_directory=source.is_dir())
            return "symlinked"
        except (OSError, NotImplementedError) as e:
            print(
                f"  Symlink failed for {target.name} ({e}); falling back to copy. "
                "(On Windows, enable Developer Mode or run as Administrator to allow symlinks.)"
            )

    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)
    return "copied"


def populate_raw_dir(data_root: Path, raw_dir: Path, force_copy: bool) -> dict[str, str]:
    raw_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for entry in sorted(data_root.iterdir()):
        target = raw_dir / entry.name
        results[entry.name] = link_or_copy(entry, target, force_copy)
    return results


def patch_dataset_config_slug(slug: str) -> bool:
    """Update configs/dataset_config.yaml's kaggle_dataset_slug if it doesn't already match `slug`
    (idempotent text patch — preserves the rest of the file/comments, which OmegaConf round-tripping
    would not)."""
    config_path = CONFIGS_DIR / "dataset_config.yaml"
    text = config_path.read_text(encoding="utf-8")

    if f'kaggle_dataset_slug: "{slug}"' in text:
        return False

    pattern = re.compile(r'kaggle_dataset_slug:\s*".*?"')
    if not pattern.search(text):
        return False

    patched = pattern.sub(f'kaggle_dataset_slug: "{slug}"', text, count=1)
    config_path.write_text(patched, encoding="utf-8")
    return True


def run_verification(sample_images: int) -> int:
    result = subprocess.run([sys.executable, str(VERIFY_SCRIPT), "--sample-images", str(sample_images)])
    return result.returncode


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--force", action="store_true", help="Re-download and re-link even if raw_dir already looks valid")
    parser.add_argument("--copy", action="store_true", help="Copy instead of symlinking into data/chexpert/raw/")
    parser.add_argument("--sample-images", type=int, default=200, help="Passed through to 01_verify_download.py")
    args = parser.parse_args()

    stage1_cfg = load_stage1_config()
    ensure_dirs(stage1_cfg)
    raw_dir = Path(stage1_cfg.paths.raw_dir)

    if not args.force and (raw_dir / "train.csv").exists() and (raw_dir / "valid.csv").exists():
        print(f"Found existing train.csv/valid.csv in {raw_dir}; verifying before deciding whether to download...")
        if run_verification(args.sample_images) == 0:
            print("\nAlready downloaded and verified — skipping download (idempotent). Use --force to redo it anyway.")
            return 0
        print("\nExisting data did not pass verification; proceeding to (re)download.\n")

    try:
        import kagglehub
    except ImportError:
        print("kagglehub is not installed. Run: pip install kagglehub")
        return 1

    dataset_cfg = load_dataset_config()
    slug = dataset_cfg.source.kaggle_dataset_slug
    print(f"Downloading Kaggle dataset '{slug}' via kagglehub (this can take a while on first run)...")
    download_path = Path(kagglehub.dataset_download(slug))
    print(f"kagglehub cached the dataset at: {download_path}")

    data_root = find_data_root(download_path)
    print(f"Located dataset root: {data_root}")

    results = populate_raw_dir(data_root, raw_dir, force_copy=args.copy)
    for name, action in results.items():
        print(f"  {action}: {raw_dir / name}")

    if patch_dataset_config_slug(slug):
        print(f"Updated configs/dataset_config.yaml kaggle_dataset_slug to '{slug}'")

    print("\nRunning scripts/data/01_verify_download.py to confirm the download is usable...")
    rc = run_verification(args.sample_images)
    if rc != 0:
        print("\nDownload completed but verification FAILED — see errors above.")
        return rc

    print("\nOK: dataset downloaded, linked into data/chexpert/raw/, and verified.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/01_verify_download.py`

*173 lines*

**Pipeline step 2 (integrity gate).** Refuses to let a broken or wrong-version download through.

Checks: both CSVs exist; `train.csv` row count within 1% of `expected_train_rows` and `valid.csv`
exactly `expected_valid_rows`; all 14 pathology columns present with values only in
{-1, 0, 1, NaN}; a random sample of images actually opens with plausible dimensions; and per-label
positive rates are printed for eyeball comparison against published CheXpert statistics.

In [ ]:
# ===== FILE: scripts/data/01_verify_download.py =====
from __future__ import annotations

import argparse
import random
import sys
from pathlib import Path

import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402

VALID_LABEL_VALUES = {-1.0, 0.0, 1.0}
ROW_COUNT_TOLERANCE = 0.01  # 1%


def check_files_exist(raw_dir: Path) -> tuple[bool, list[str]]:
    errors = []
    train_csv = raw_dir / "train.csv"
    valid_csv = raw_dir / "valid.csv"
    if not train_csv.exists():
        errors.append(f"Missing {train_csv}")
    if not valid_csv.exists():
        errors.append(f"Missing {valid_csv}")
    return (len(errors) == 0, errors)


def check_row_counts(df_train: pd.DataFrame, df_valid: pd.DataFrame, dataset_cfg) -> tuple[bool, list[str]]:
    errors = []
    expected_train = dataset_cfg.source.expected_train_rows
    expected_valid = dataset_cfg.source.expected_valid_rows

    train_diff = abs(len(df_train) - expected_train) / expected_train
    if train_diff > ROW_COUNT_TOLERANCE:
        errors.append(
            f"train.csv has {len(df_train)} rows, expected ~{expected_train} "
            f"(off by {train_diff:.1%}, tolerance {ROW_COUNT_TOLERANCE:.0%})"
        )

    if len(df_valid) != expected_valid:
        errors.append(f"valid.csv has {len(df_valid)} rows, expected exactly {expected_valid}")

    return (len(errors) == 0, errors)


def check_schema(df: pd.DataFrame, dataset_cfg, csv_name: str) -> tuple[bool, list[str]]:
    from scripts.utils.chexpert_schema import validate_chexpert_frame
    try:
        validate_chexpert_frame(df, dataset_cfg)
        return True, []
    except ValueError as exc:
        return False, [f"{csv_name}: {exc}"]
    """Legacy body retained below for source-history readability; shared validator returns above."""
    errors = []
    schema = dataset_cfg.schema
    required_columns = [
        schema.path_column,
        schema.sex_column,
        schema.age_column,
        schema.frontal_lateral_column,
    ] + list(schema.pathology_columns)

    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        errors.append(f"{csv_name}: missing expected columns: {missing}")

    for col in schema.pathology_columns:
        if col not in df.columns:
            continue
        observed = set(df[col].dropna().unique().tolist())
        invalid = {v for v in observed if v not in VALID_LABEL_VALUES}
        if invalid:
            errors.append(f"{csv_name}: column {col!r} has out-of-domain values: {invalid}")

    return (len(errors) == 0, errors)


def check_image_spotcheck(df: pd.DataFrame, raw_dir: Path, dataset_cfg, n: int, seed: int) -> tuple[bool, list[str]]:
    errors = []
    rng = random.Random(seed)
    path_col = dataset_cfg.schema.path_column
    n = min(n, len(df))
    sample_rows = df.sample(n=n, random_state=seed)

    for _, row in sample_rows.iterrows():
        raw_path = str(row[path_col]).replace("\\", "/")
        # Paths in train.csv are typically like "CheXpert-v1.0-small/train/patient.../view1_frontal.jpg"
        for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
            if raw_path.startswith(prefix):
                raw_path = raw_path[len(prefix):]
        image_path = raw_dir / raw_path
        if not image_path.exists():
            errors.append(f"Spot-check: image not found: {image_path}")
            continue
        try:
            with Image.open(image_path) as im:
                im.verify()
            with Image.open(image_path) as im:
                w, h = im.size
                if w < 50 or h < 50:
                    errors.append(f"Spot-check: implausibly small image {image_path} ({w}x{h})")
        except Exception as e:
            errors.append(f"Spot-check: failed to open {image_path}: {e}")

    return (len(errors) == 0, errors)


def print_label_prevalence(df: pd.DataFrame, dataset_cfg) -> None:
    print("\nLabel prevalence in train.csv (compare against published CheXpert statistics manually --")
    print("this script does not hard-fail on prevalence, since exact published rates aren't asserted here):")
    for col in dataset_cfg.schema.pathology_columns:
        if col not in df.columns:
            continue
        counts = df[col].value_counts(dropna=False)
        total = len(df)
        pos = counts.get(1.0, 0)
        neg = counts.get(0.0, 0)
        unc = counts.get(-1.0, 0)
        blank = total - pos - neg - unc
        print(
            f"  {col:<28} positive={pos / total:6.1%}  negative={neg / total:6.1%}  "
            f"uncertain={unc / total:6.1%}  blank={blank / total:6.1%}"
        )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--sample-images", type=int, default=200, help="Number of images to spot-check")
    args = parser.parse_args()

    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    raw_dir = Path(stage1_cfg.paths.raw_dir)
    seed = stage1_cfg.run.seed

    all_errors: list[str] = []

    ok, errors = check_files_exist(raw_dir)
    all_errors += errors
    if not ok:
        print("FAILED: required files missing, cannot continue further checks.")
        for e in all_errors:
            print(f"  - {e}")
        return 1

    df_train = pd.read_csv(raw_dir / "train.csv")
    df_valid = pd.read_csv(raw_dir / "valid.csv")

    for check_fn, args_ in (
        (check_row_counts, (df_train, df_valid, dataset_cfg)),
        (check_schema, (df_train, dataset_cfg, "train.csv")),
        (check_schema, (df_valid, dataset_cfg, "valid.csv")),
        (check_image_spotcheck, (df_train, raw_dir, dataset_cfg, args.sample_images, seed)),
    ):
        _, errors = check_fn(*args_)
        all_errors += errors

    print_label_prevalence(df_train, dataset_cfg)

    print()
    if all_errors:
        print(f"FAILED: {len(all_errors)} check(s) did not pass:")
        for e in all_errors:
            print(f"  - {e}")
        return 1

    print("OK: all structural integrity checks passed.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/01b_build_dev_subset.py`

*207 lines*

**Optional step.** Builds a small, prevalence-faithful development subset so the pipeline can be
shaken down end to end without the full 224k-image cohort.

`select_dev_subset_patients` is **rarity-first greedy quota sampling**: sort pathology columns by
ascending prevalence (rarest first) and, for each, greedily draw *whole patients* carrying that
finding until its proportional quota is met. Because CheXpert pathologies co-occur, later quotas
are often partly satisfied for free; any shortfall is filled with uniform random patient draws.
All randomness comes from one seeded stream, and patients are never split.

`build_prevalence_report` then checks every column's prevalence against the full cohort within
`prevalence_tolerance_pct` and records the result (non-fatal, but written to a manifest).

Controlled by `stage1.dev_subset` in `configs/stage1_lora_sdxl.yaml`; a no-op when disabled.

In [ ]:
# ===== FILE: scripts/data/01b_build_dev_subset.py =====
from __future__ import annotations

import importlib
import random
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import (  # noqa: E402
    get_git_commit_hash,
    hash_dict,
    sha256_file,
    write_json,
)

ALGORITHM_VERSION = 1


def compute_prevalence(df: pd.DataFrame, pathology_columns: list[str]) -> pd.Series:
    return (df[pathology_columns] == 1).mean()


def select_dev_subset_patients(
    df: pd.DataFrame,
    patient_column: str,
    pathology_columns: list[str],
    target_size: int,
    seed: int,
) -> set[str]:
    """Rarity-first greedy quota sampling. Returns the set of selected patient IDs.

    Sorts pathology columns ascending by full-dataset image-level prevalence (rarest first),
    and for each, greedily draws whole patients carrying that finding until its proportional
    quota of `target_size` is met (or the eligible pool is exhausted). Because CheXpert
    pathologies co-occur, later quotas are frequently partially satisfied "for free" by earlier
    draws. Any shortfall against `target_size` after processing all columns is filled by
    uniform random patient draws. All randomness comes from one seeded `random.Random` stream.
    """
    prevalence = compute_prevalence(df, pathology_columns)
    rarity_order = prevalence.sort_values(ascending=True).index.tolist()
    quotas = (prevalence * target_size).round().astype(int)

    positive_counts_per_patient = (df[pathology_columns] == 1).groupby(df[patient_column]).sum()
    patient_image_counts = df.groupby(patient_column).size()
    positive_patients_by_col = {
        col: sorted(positive_counts_per_patient.index[positive_counts_per_patient[col] > 0].tolist())
        for col in pathology_columns
    }
    all_patients = sorted(patient_image_counts.index.tolist())

    rng = random.Random(seed)
    selected: set[str] = set()
    running_count = pd.Series(0, index=pathology_columns, dtype="int64")

    for col in rarity_order:
        quota = int(quotas[col])
        if quota <= 0:
            continue
        candidates = [p for p in positive_patients_by_col[col] if p not in selected]
        rng.shuffle(candidates)
        for patient in candidates:
            if running_count[col] >= quota:
                break
            selected.add(patient)
            running_count = running_count.add(positive_counts_per_patient.loc[patient], fill_value=0)

    total_images = int(patient_image_counts[list(selected)].sum()) if selected else 0
    if total_images < target_size:
        remaining = [p for p in all_patients if p not in selected]
        rng.shuffle(remaining)
        for patient in remaining:
            if total_images >= target_size:
                break
            selected.add(patient)
            total_images += int(patient_image_counts[patient])

    return selected


def build_prevalence_report(
    df_full: pd.DataFrame,
    df_subset: pd.DataFrame,
    pathology_columns: list[str],
    tolerance_pct: float,
) -> dict:
    full_pos = compute_prevalence(df_full, pathology_columns)
    subset_pos = compute_prevalence(df_subset, pathology_columns)
    full_unc = (df_full[pathology_columns] == -1).mean()
    subset_unc = (df_subset[pathology_columns] == -1).mean()

    per_column = {}
    num_exceeding = 0
    for col in pathology_columns:
        deviation_pct = abs(float(subset_pos[col]) - float(full_pos[col])) * 100.0
        within = deviation_pct <= tolerance_pct
        if not within:
            num_exceeding += 1
        per_column[col] = {
            "full_dataset_positive_rate": float(full_pos[col]),
            "subset_positive_rate": float(subset_pos[col]),
            "absolute_deviation_pct": deviation_pct,
            "within_tolerance": within,
            "full_dataset_uncertain_rate": float(full_unc[col]),
            "subset_uncertain_rate": float(subset_unc[col]),
        }

    return {
        "tolerance_pct": tolerance_pct,
        "num_columns_exceeding_tolerance": num_exceeding,
        "all_within_tolerance": num_exceeding == 0,
        "columns": per_column,
    }


def main() -> int:
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    dev_cfg = stage1_cfg.dev_subset

    if not bool(dev_cfg.enabled):
        print("dev_subset.enabled=false — 01b_build_dev_subset.py is a no-op. "
              "split.input_csv should point at the full train.csv.")
        return 0

    raw_dir = Path(stage1_cfg.paths.raw_dir)
    train_csv_path = raw_dir / "train.csv"
    if not train_csv_path.is_file():
        raise SystemExit(f"Missing {train_csv_path} (run 00_download_dataset.py / 01_verify_download.py first).")

    schema = dataset_cfg.schema
    pathology_columns = list(schema.pathology_columns)
    target_size = int(dev_cfg.target_size)
    if target_size <= 0:
        raise SystemExit(f"dev_subset.target_size must be positive, got {target_size}")

    df_full = pd.read_csv(train_csv_path)
    original_columns = list(df_full.columns)

    patient_splits_module = importlib.import_module("scripts.data.02_build_patient_splits")
    extract_patient_id = patient_splits_module.extract_patient_id

    df_full = df_full.copy()
    df_full["__patient_id"] = df_full[schema.path_column].apply(
        lambda p: extract_patient_id(p, schema.patient_id_regex)
    )

    selected_patients = select_dev_subset_patients(
        df_full,
        patient_column="__patient_id",
        pathology_columns=pathology_columns,
        target_size=target_size,
        seed=int(dev_cfg.seed),
    )

    df_subset = df_full[df_full["__patient_id"].isin(selected_patients)][original_columns].reset_index(drop=True)

    output_csv = Path(dev_cfg.output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df_subset.to_csv(output_csv, index=False)

    report = build_prevalence_report(
        df_full, df_subset, pathology_columns, float(dev_cfg.prevalence_tolerance_pct)
    )
    write_json(Path(dev_cfg.prevalence_report_path), report)

    manifest = {
        "algorithm_version": ALGORITHM_VERSION,
        "algorithm": "rarity_first_greedy_quota_patient_sampling",
        "target_size": target_size,
        "actual_num_images": len(df_subset),
        "actual_num_patients": len(selected_patients),
        "seed": int(dev_cfg.seed),
        "source_train_csv_path": str(train_csv_path),
        "source_train_csv_hash": sha256_file(train_csv_path),
        "source_num_images": len(df_full),
        "source_num_patients": int(df_full["__patient_id"].nunique()),
        "dev_subset_config_hash": hash_dict(
            {
                "target_size": target_size,
                "seed": int(dev_cfg.seed),
                "prevalence_tolerance_pct": float(dev_cfg.prevalence_tolerance_pct),
            }
        ),
        "git_commit_hash": get_git_commit_hash(),
        "all_within_tolerance": report["all_within_tolerance"],
    }
    write_json(Path(dev_cfg.manifest_path), manifest)

    print(f"Source: {len(df_full)} images / {df_full['__patient_id'].nunique()} patients")
    print(f"Dev subset: {len(df_subset)} images / {len(selected_patients)} patients (target {target_size})")
    if report["all_within_tolerance"]:
        print(f"Prevalence check: all {len(pathology_columns)} pathology columns within "
              f"{dev_cfg.prevalence_tolerance_pct}pp tolerance.")
    else:
        print(f"WARNING: {report['num_columns_exceeding_tolerance']} column(s) exceed "
              f"{dev_cfg.prevalence_tolerance_pct}pp prevalence-deviation tolerance — see "
              f"{dev_cfg.prevalence_report_path} (non-fatal).")
    print(f"Wrote {output_csv}, {dev_cfg.manifest_path}, {dev_cfg.prevalence_report_path}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## `scripts/data/02_build_patient_splits.py`

*171 lines*

**SUPERSEDED - not a pipeline step.** This is the original schema-v1 **three-way** split builder
(`gen_train` / `gen_val` / `classifier_heldout`, with `asism_tuning_heldout` and
`final_eval_heldout` derived as *children* of `classifier_heldout`).

That architecture left **no population for a classifier development split**, which is why it was
replaced by the direct six-way builder (`02b_build_sixway_splits.py`). Its outputs are never read
by Stage 2-5 code.

Today it is kept alive by exactly one thing: `01b_build_dev_subset.py` imports its
`extract_patient_id` helper - a 4-line regex function that `02b` also defines identically. See the
audit log at the end of this notebook.

In [ ]:
# ===== FILE: scripts/data/02_build_patient_splits.py =====
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json, write_json  # noqa: E402


def extract_patient_id(path_value: str, patient_id_regex: str) -> str:
    match = re.search(patient_id_regex, str(path_value))
    if not match:
        raise ValueError(f"Could not extract patient id from path: {path_value!r}")
    return match.group(0)


def split_patients(
    patient_ids: list[str],
    gen_train_fraction: float,
    gen_val_fraction: float,
    seed: int,
) -> tuple[set[str], set[str], set[str]]:
    import random

    rng = random.Random(seed)
    shuffled = list(patient_ids)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(round(n * gen_train_fraction))
    n_val = int(round(n * gen_val_fraction))

    train_ids = set(shuffled[:n_train])
    val_ids = set(shuffled[n_train : n_train + n_val])
    heldout_ids = set(shuffled[n_train + n_val :])

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(heldout_ids)
    assert val_ids.isdisjoint(heldout_ids)
    assert len(train_ids) + len(val_ids) + len(heldout_ids) == n

    return train_ids, val_ids, heldout_ids


def split_heldout_for_asism(
    heldout_patient_ids: list[str],
    asism_tuning_fraction: float,
    seed: int,
) -> tuple[set[str], set[str]]:
    """Sub-split classifier_heldout so Stage 3's ASISM weight/threshold search (tuned on
    asism_tuning_heldout) never touches the same data Stage 5 reports final numbers on
    (final_eval_heldout) — avoiding optimistic-bias leakage between tuning and evaluation."""
    import random

    rng = random.Random(seed)
    shuffled = list(heldout_patient_ids)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_tuning = int(round(n * asism_tuning_fraction))

    tuning_ids = set(shuffled[:n_tuning])
    final_eval_ids = set(shuffled[n_tuning:])

    assert tuning_ids.isdisjoint(final_eval_ids)
    assert len(tuning_ids) + len(final_eval_ids) == n

    return tuning_ids, final_eval_ids


def main() -> int:
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    ensure_dirs(stage1_cfg)

    splits_dir = Path(stage1_cfg.paths.splits_dir)
    schema = dataset_cfg.schema
    split_cfg = stage1_cfg.split

    input_csv_path = Path(split_cfg.input_csv)
    if not input_csv_path.is_file():
        raise SystemExit(f"Missing {input_csv_path} (run 00/01, and 01b if using the dev subset, first).")

    df = pd.read_csv(input_csv_path)
    df["patient_id"] = df[schema.path_column].apply(
        lambda p: extract_patient_id(p, schema.patient_id_regex)
    )

    unique_patients = sorted(df["patient_id"].unique().tolist())
    train_ids, val_ids, heldout_ids = split_patients(
        unique_patients,
        gen_train_fraction=split_cfg.gen_train_fraction,
        gen_val_fraction=split_cfg.gen_val_fraction,
        seed=split_cfg.split_seed,
    )
    asism_tuning_ids, final_eval_ids = split_heldout_for_asism(
        sorted(heldout_ids),
        asism_tuning_fraction=split_cfg.asism_tuning_fraction,
        seed=split_cfg.heldout_split_seed,
    )

    df_train = df[df["patient_id"].isin(train_ids)].reset_index(drop=True)
    df_val = df[df["patient_id"].isin(val_ids)].reset_index(drop=True)
    df_heldout = df[df["patient_id"].isin(heldout_ids)].reset_index(drop=True)
    df_asism_tuning = df[df["patient_id"].isin(asism_tuning_ids)].reset_index(drop=True)
    df_final_eval = df[df["patient_id"].isin(final_eval_ids)].reset_index(drop=True)

    # Belt-and-braces: verify no patient leaked across the image-level dataframes too.
    assert set(df_train["patient_id"]).isdisjoint(set(df_val["patient_id"]))
    assert set(df_train["patient_id"]).isdisjoint(set(df_heldout["patient_id"]))
    assert set(df_val["patient_id"]).isdisjoint(set(df_heldout["patient_id"]))
    assert set(df_asism_tuning["patient_id"]).isdisjoint(set(df_final_eval["patient_id"]))
    assert set(df_asism_tuning["patient_id"]) | set(df_final_eval["patient_id"]) == set(df_heldout["patient_id"])

    df_train.to_csv(splits_dir / "gen_train.csv", index=False)
    df_val.to_csv(splits_dir / "gen_val.csv", index=False)
    df_heldout.to_csv(splits_dir / "classifier_heldout.csv", index=False)
    df_asism_tuning.to_csv(splits_dir / "asism_tuning_heldout.csv", index=False)
    df_final_eval.to_csv(splits_dir / "final_eval_heldout.csv", index=False)

    dev_subset_manifest_hash = None
    dev_cfg = getattr(stage1_cfg, "dev_subset", None)
    if dev_cfg is not None and Path(dev_cfg.output_csv) == input_csv_path:
        dev_manifest_path = Path(dev_cfg.manifest_path)
        if dev_manifest_path.is_file():
            dev_subset_manifest_hash = hash_dict(read_json(dev_manifest_path))

    manifest = {
        "input_csv_path": str(input_csv_path),
        "dev_subset_manifest_hash": dev_subset_manifest_hash,
        "split_unit": split_cfg.split_unit,
        "split_seed": split_cfg.split_seed,
        "gen_train_fraction": split_cfg.gen_train_fraction,
        "gen_val_fraction": split_cfg.gen_val_fraction,
        "classifier_heldout_fraction": split_cfg.classifier_heldout_fraction,
        "asism_tuning_fraction": split_cfg.asism_tuning_fraction,
        "heldout_split_seed": split_cfg.heldout_split_seed,
        "num_patients_total": len(unique_patients),
        "num_patients_gen_train": len(train_ids),
        "num_patients_gen_val": len(val_ids),
        "num_patients_classifier_heldout": len(heldout_ids),
        "num_patients_asism_tuning_heldout": len(asism_tuning_ids),
        "num_patients_final_eval_heldout": len(final_eval_ids),
        "num_images_gen_train": len(df_train),
        "num_images_gen_val": len(df_val),
        "num_images_classifier_heldout": len(df_heldout),
        "num_images_asism_tuning_heldout": len(df_asism_tuning),
        "num_images_final_eval_heldout": len(df_final_eval),
        "official_valid_csv": "reserved untouched at data/chexpert/raw/valid.csv — not read by this script",
        "source_csv_hash": hash_dict({"n_rows": len(df), "n_patients": len(unique_patients)}),
    }
    write_json(splits_dir / "split_manifest.json", manifest)

    print(f"Input CSV: {input_csv_path}")
    print(f"Total patients: {len(unique_patients)}")
    print(f"  gen_train:             {len(train_ids):>6} patients, {len(df_train):>7} images")
    print(f"  gen_val:               {len(val_ids):>6} patients, {len(df_val):>7} images")
    print(f"  classifier_heldout:    {len(heldout_ids):>6} patients, {len(df_heldout):>7} images")
    print(f"    asism_tuning_heldout: {len(asism_tuning_ids):>6} patients, {len(df_asism_tuning):>7} images")
    print(f"    final_eval_heldout:   {len(final_eval_ids):>6} patients, {len(df_final_eval):>7} images")
    print(f"Wrote split CSVs and split_manifest.json to {splits_dir}")

    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/02b_build_sixway_splits.py`

*593 lines*

**Pipeline step 3 - the real split builder, and the file everything downstream is hashed against.**

What it does, in order:

1. Validates the source CSV against the CheXpert schema contract (and, for the production
   namespace, against the full-cohort row-count contract - `verify_full_cohort`).
2. Extracts patient ids and builds per-patient positive **and negative** label vectors
   (`patient_label_vectors`, `patient_label_negative_vectors`).
3. `multilabel_partition_patients` - stratified assignment of *patients* into the six splits in one
   pass from one seed. It balances negative support as well as positive support, which is what the
   50-negative-patient rule actually needs.
4. `assert_disjoint_and_exact` - the partition is exactly disjoint and its union is every patient.
5. `build_support_report` - per-label positive/negative **patient** counts in every
   decision-bearing split, checked against the frozen support rule. `propose_fraction_adjustment`
   suggests a fix when it fails, instead of just erroring.
6. Writes `split_manifest_v2.json` with a self-hash, library versions, git commit, per-file
   SHA-256s, and `namespace_class`; `--freeze` makes it immutable.
7. Leaves a `SUPERSEDED_v1.md` note next to the old v1 artifacts rather than deleting them.

It touches `final_eval_heldout` here - legitimately - via
`assert_final_eval_access_allowed(purpose="split_construction")`.

In [ ]:
# ===== FILE: scripts/data/02b_build_sixway_splits.py =====
from __future__ import annotations

import argparse
import os
import random
import re
import shutil
import sys
import tempfile
import uuid
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.artifact_contracts import validate_namespace_run_id  # noqa: E402
from scripts.utils.chexpert_schema import validate_chexpert_frame  # noqa: E402
from scripts.utils.labels import (  # noqa: E402
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
    check_support_rule,
    patient_level_support,
    normalize_label,
    validate_label_domain,
)
from scripts.utils.manifest import get_git_commit_hash, get_library_versions, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import (  # noqa: E402
    FINAL_EVAL_SPLIT,
    SPLIT_MANIFEST_FILENAME,
    SPLIT_NAMES,
    assert_final_eval_access_allowed,
    load_splits_config,
    resolve_splits_dir,
)

SUPERSEDED_NOTE_FILENAME = "SUPERSEDED_v1.md"

SUPERSEDED_NOTE = """# Superseded schema-v1 split artifacts

The files in this directory's parent (`gen_train.csv`, `gen_val.csv`, `classifier_heldout.csv`,
`asism_tuning_heldout.csv`, `final_eval_heldout.csv`, `split_manifest.json`) were produced by
`scripts/data/02_build_patient_splits.py` under the schema-v1 three-way split
(gen_train 0.70 / gen_val 0.10 / classifier_heldout 0.20, with the ASISM-tuning and final-eval
splits derived as children of classifier_heldout).

That architecture is superseded by the six-way partition in
`scripts/data/02b_build_sixway_splits.py` (docs/stages2_to_5_plan.md §1), because it left no
patient population for a classifier development split, and deriving one from `gen_train` would
have meant the headline classifiers' early-stopping and threshold decisions were made on patients the
SDXL generator had itself trained on.

These files are RETAINED for provenance and reproducibility of anything already built against
them. They are NOT read by any Stage 2-5 code. Current splits live under
`splits/production/` and `splits/dev/`.
"""


def extract_patient_id(path_value: str, patient_id_regex: str) -> str:
    match = re.search(patient_id_regex, str(path_value))
    if not match:
        raise ValueError(f"Could not extract patient id from path: {path_value!r}")
    return match.group(0)


def partition_patients(
    patient_ids: list[str],
    fractions: dict[str, float],
    seed: int,
) -> dict[str, set[str]]:
    """Six direct partitions from ONE shuffle of ONE deterministically ordered patient list.

    SUPERSEDED IN main(): multilabel_partition_patients() is used instead, because unstratified
    assignment left rare labels short of the frozen 50-negative-patient support rule (§1.3).
    Retained (and unit-tested) as the unstratified reference this project measures against.

    Sorting before shuffling is what makes this reproducible regardless of the row order the source
    CSV happens to arrive in. Remainder patients (from rounding) go to the largest split so the
    partition is exact rather than dropping anyone.
    """
    ordered = sorted(patient_ids)
    rng = random.Random(seed)
    rng.shuffle(ordered)

    total = len(ordered)
    counts = {name: int(total * fraction) for name, fraction in fractions.items()}

    # Assign rounding remainder to the largest split, keeping the partition exact.
    assigned = sum(counts.values())
    if assigned < total:
        largest = max(fractions, key=lambda name: fractions[name])
        counts[largest] += total - assigned

    result: dict[str, set[str]] = {}
    cursor = 0
    for name in SPLIT_NAMES:
        take = counts[name]
        result[name] = set(ordered[cursor : cursor + take])
        cursor += take

    assert cursor == total, f"Partition consumed {cursor} of {total} patients"
    return result


def patient_label_vectors(frame: pd.DataFrame, labels: list[str] = PRIMARY_ENDPOINT_LABELS) -> dict[str, tuple[int, ...]]:
    """Collapse image labels to deterministic patient-level any-positive vectors."""
    vectors = {}
    for patient_id, rows in frame.groupby("patient_id", sort=True):
        vectors[str(patient_id)] = tuple(
            int(any(normalize_label(value) == 1 for value in rows[label])) for label in labels
        )
    return vectors


def patient_label_negative_vectors(
    frame: pd.DataFrame, labels: list[str] = PRIMARY_ENDPOINT_LABELS
) -> dict[str, tuple[int, ...]]:
    """Collapse image labels to deterministic patient-level confident-negative vectors.

    Mirrors patient_level_support's negative definition (scripts/utils/labels.py): a patient is
    negative for a label if none of their studies is a confident 1 and at least one is a confident
    0. Kept separate from patient_label_vectors (positive-only) so multilabel_partition_patients can
    target both sides of the §1.3 support rule without changing prevalence_report's positive-only
    semantics.
    """
    vectors = {}
    for patient_id, rows in frame.groupby("patient_id", sort=True):
        vectors[str(patient_id)] = tuple(
            int(
                not any(normalize_label(value) == 1 for value in rows[label])
                and any(normalize_label(value) == 0 for value in rows[label])
            )
            for label in labels
        )
    return vectors


def multilabel_partition_patients(
    vectors: dict[str, tuple[int, ...]],
    neg_vectors: dict[str, tuple[int, ...]],
    fractions: dict[str, float],
    seed: int,
) -> dict[str, set[str]]:
    """Deterministic greedy iterative multilabel allocation.

    Targets both positive AND confident-negative patient counts per label (the §1.3 support rule
    needs both sides, but only positives drove allocation before this fix). Deficits are normalized
    by each split's own target so a small split (e.g. a 5% heldout) is judged by how much of its own
    quota is unmet, not by raw patient counts — otherwise a large split's much bigger absolute target
    (e.g. gen_train at 60%) always looks more "urgent" and starves rare labels out of the small
    splits entirely. Rarest labels (by either their positive or negative population) are placed
    first; each patient goes to the non-full split with the largest normalized deficit for that
    patient's positive+negative labels, then capacity deficit, then a seeded stable tie rank. This
    uses one recorded seed and never retries seeds.
    """
    names = list(SPLIT_NAMES)
    patients = sorted(vectors)
    total = len(patients)
    n_labels = len(next(iter(vectors.values()), ()))
    capacities = {name: int(total * fractions[name]) for name in names}
    capacities[max(fractions, key=fractions.get)] += total - sum(capacities.values())

    pos_totals = [sum(v[j] for v in vectors.values()) for j in range(n_labels)]
    neg_totals = [sum(v[j] for v in neg_vectors.values()) for j in range(n_labels)]
    pos_targets = {name: [pos_totals[j] * fractions[name] for j in range(n_labels)] for name in names}
    neg_targets = {name: [neg_totals[j] * fractions[name] for j in range(n_labels)] for name in names}

    rng = random.Random(seed)
    tie_order = patients[:]
    rng.shuffle(tie_order)
    tie_rank = {patient: rank for rank, patient in enumerate(tie_order)}

    def rarest_signal(patient: str) -> int:
        counts = [pos_totals[j] for j, value in enumerate(vectors[patient]) if value]
        counts += [neg_totals[j] for j, value in enumerate(neg_vectors[patient]) if value]
        return min(counts, default=total + 1)

    ordered = sorted(
        patients,
        key=lambda p: (
            rarest_signal(p),
            -(sum(vectors[p]) + sum(neg_vectors[p])),
            tie_rank[p],
            p,
        ),
    )
    result = {name: set() for name in names}
    pos_observed = {name: [0] * n_labels for name in names}
    neg_observed = {name: [0] * n_labels for name in names}
    for patient in ordered:
        pos_vector = vectors[patient]
        neg_vector = neg_vectors[patient]
        candidates = [name for name in names if len(result[name]) < capacities[name]]

        def score(name):
            pos_deficit = sum(
                max(0.0, (pos_targets[name][j] - pos_observed[name][j]) / max(pos_targets[name][j], 1e-9))
                for j, value in enumerate(pos_vector) if value
            )
            neg_deficit = sum(
                max(0.0, (neg_targets[name][j] - neg_observed[name][j]) / max(neg_targets[name][j], 1e-9))
                for j, value in enumerate(neg_vector) if value
            )
            capacity_deficit = (capacities[name] - len(result[name])) / max(capacities[name], 1)
            return (pos_deficit + neg_deficit, capacity_deficit, -names.index(name))

        chosen = max(candidates, key=score)
        result[chosen].add(patient)
        pos_observed[chosen] = [a + b for a, b in zip(pos_observed[chosen], pos_vector)]
        neg_observed[chosen] = [a + b for a, b in zip(neg_observed[chosen], neg_vector)]
    return result


def prevalence_report(vectors: dict[str, tuple[int, ...]], groups: dict[str, set[str]]) -> dict:
    labels = list(PRIMARY_ENDPOINT_LABELS)
    def rates(ids):
        n = max(len(ids), 1)
        return {label: sum(vectors[p][j] for p in ids) / n for j, label in enumerate(labels)}
    source = rates(set(vectors))
    per_split = {}
    for name, ids in groups.items():
        split_rates = rates(ids)
        per_split[name] = {label: {
            "prevalence": split_rates[label],
            "absolute_deviation": abs(split_rates[label] - source[label]),
            "relative_deviation": (abs(split_rates[label] - source[label]) / source[label] if source[label] else None),
        } for label in labels}
    return {"source": source, "per_split": per_split, "labels_unable_to_meet_tolerance": []}


def assert_disjoint_and_exact(groups: dict[str, set[str]], all_patients: set[str]) -> None:
    """Pairwise disjointness + exact partition. Hard assertions, not warnings — a leak here
    invalidates every downstream comparison."""
    names = list(groups.keys())
    for i, left in enumerate(names):
        for right in names[i + 1 :]:
            overlap = groups[left] & groups[right]
            if overlap:
                raise SystemExit(
                    f"SPLIT LEAK: {len(overlap)} patient(s) in both {left} and {right}; "
                    f"examples: {sorted(overlap)[:5]}"
                )

    union = set().union(*groups.values())
    if union != all_patients:
        missing = all_patients - union
        extra = union - all_patients
        raise SystemExit(
            f"SPLIT NOT EXACT: {len(missing)} patient(s) unassigned, {len(extra)} unknown. "
            f"missing examples: {sorted(missing)[:5]}"
        )


def verify_full_cohort(frame: pd.DataFrame, dataset_cfg, tolerance_pct: float) -> dict:
    """Production namespace gate (§1.4): the source must be the complete intended CheXpert cohort,
    checked against the same expected-row contract 01_verify_download.py uses."""
    expected = int(dataset_cfg.source.expected_train_rows)
    actual = len(frame)
    deviation_pct = abs(actual - expected) / expected * 100.0
    passed = deviation_pct <= tolerance_pct
    return {
        "expected_train_rows": expected,
        "actual_rows": actual,
        "deviation_pct": round(deviation_pct, 4),
        "tolerance_pct": tolerance_pct,
        "passed": passed,
    }


def build_support_report(
    frames: dict[str, pd.DataFrame],
    decision_bearing_splits: list[str],
    min_positive: int,
    min_negative: int,
) -> dict:
    """Run the frozen §1.3 support rule over every decision-bearing split and return a COMPLETE
    report (all splits, all labels, all failures) — not a first-failure abort, because the caller
    needs the whole picture to propose a fraction adjustment."""
    per_split = {}
    all_passed = True

    for split_name in decision_bearing_splits:
        frame = frames[split_name]
        if split_name == FINAL_EVAL_SPLIT:
            assert_final_eval_access_allowed(
                purpose="support_check", caller="02b_build_sixway_splits"
            )
        support = patient_level_support(frame, PRIMARY_ENDPOINT_LABELS)
        passed, failures = check_support_rule(support, min_positive, min_negative)
        all_passed = all_passed and passed
        per_split[split_name] = {
            "passed": passed,
            "failures": failures,
            "per_label": support.to_dict("records"),
        }

    return {
        "rule": {
            "min_positive_patients": min_positive,
            "min_negative_patients": min_negative,
            "unit": "patients (image counts are supplementary only)",
            "labels": PRIMARY_ENDPOINT_LABELS,
        },
        "passed": all_passed,
        "per_split": per_split,
    }


def propose_fraction_adjustment(
    support_report: dict,
    fractions: dict[str, float],
    total_patients: int,
) -> dict:
    """Rough support-scaling estimate when frozen fractions fail the support rule.

    Computes, per failing split, the scale-up factor its fraction would need for its worst-failing
    label to clear the threshold, then proposes taking the shortfall from gen_train (the largest
    split, and the one whose marginal patient is worth least — Stage 1 LoRA training is far less
    sensitive to a few hundred patients than a support-starved evaluation split is).

    This is a PROPOSAL written to the report. The frozen fractions are never modified automatically.
    """
    proposals = {}
    for split_name, result in support_report["per_split"].items():
        if result["passed"]:
            continue
        current_fraction = fractions[split_name]
        current_patients = max(1, int(total_patients * current_fraction))

        worst_ratio = 1.0
        for failure in result["failures"]:
            for observed, required in (
                (failure["positive_patients"], failure["min_positive_patients"]),
                (failure["negative_patients"], failure["min_negative_patients"]),
            ):
                if observed < required:
                    # Scale needed so this label's count reaches the threshold.
                    ratio = required / max(observed, 1)
                    worst_ratio = max(worst_ratio, ratio)

        needed_fraction = min(1.0, current_fraction * worst_ratio)
        proposals[split_name] = {
            "current_fraction": current_fraction,
            "current_patients": current_patients,
            "limiting_scale_factor": round(worst_ratio, 3),
            "proposed_fraction": round(needed_fraction, 4),
            "additional_patients_needed": int(total_patients * (needed_fraction - current_fraction)),
        }

    total_additional = sum(p["additional_patients_needed"] for p in proposals.values())
    return {
        "estimate_type": "rough_support_scaling_estimate",
        "proposals": proposals,
        "suggested_donor_split": "gen_train",
        "total_additional_patients_needed": total_additional,
        "suggested_gen_train_fraction": round(
            fractions["gen_train"] - total_additional / max(total_patients, 1), 4
        ),
        "note": (
            "PROPOSAL ONLY — frozen fractions were not modified. Applying this requires an explicit "
            "decision to change the frozen split policy in configs/splits.yaml, and re-running with "
            "--freeze."
        ),
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--namespace",
        choices=["production", "dev"],
        required=True,
        help="production = thesis splits (requires full cohort); dev = smoke-test splits",
    )
    parser.add_argument("--run-id", required=True, help="New immutable split run/version ID (for example production-v1 or dev-smoke-v3)")
    parser.add_argument(
        "--input-csv",
        default=None,
        help="Source CSV. Defaults to stage1 config's split.input_csv.",
    )
    parser.add_argument(
        "--freeze",
        action="store_true",
        help="Mark the manifest frozen. Required before Stage 5. Fails if the support check failed.",
    )
    args = parser.parse_args()

    splits_cfg = load_splits_config()
    validate_namespace_run_id(args.run_id, args.namespace)
    stage1_cfg = load_stage1_config()
    dataset_cfg = load_dataset_config()
    out_dir = resolve_splits_dir(args.run_id, splits_cfg)
    if out_dir.exists():
        raise SystemExit(f"Immutable split run already exists; refusing every in-place write: {out_dir}")
    out_dir.parent.mkdir(parents=True, exist_ok=True)

    fractions = {name: float(splits_cfg.fractions[name]) for name in SPLIT_NAMES}
    fraction_sum = round(sum(fractions.values()), 9)
    if fraction_sum != 1.0:
        raise SystemExit(f"Split fractions must sum to 1.0; got {fraction_sum} from {fractions}")

    input_csv = Path(args.input_csv) if args.input_csv else Path(stage1_cfg.split.input_csv)
    if not input_csv.is_file():
        raise SystemExit(f"Missing source CSV: {input_csv} (run scripts/data/00_download_dataset.py first)")

    frame = pd.read_csv(input_csv)
    schema = dataset_cfg.schema
    try:
        source_validation = validate_chexpert_frame(
            frame, dataset_cfg, Path(stage1_cfg.paths.raw_dir), require_all_images=args.namespace == "production"
        )
    except ValueError as exc:
        raise SystemExit(f"Source schema validation failed: {exc}") from exc
    frame["patient_id"] = frame[schema.path_column].apply(
        lambda value: extract_patient_id(value, schema.patient_id_regex)
    )

    namespace_cfg = splits_cfg.namespaces[args.namespace]
    cohort_check = None
    if bool(namespace_cfg.require_full_cohort):
        cohort_check = verify_full_cohort(
            frame, dataset_cfg, float(namespace_cfg.full_cohort_row_tolerance_pct)
        )
        if not cohort_check["passed"]:
            raise SystemExit(
                "Production splits require the complete intended CheXpert-v1.0-small training "
                f"cohort.\n  expected ~{cohort_check['expected_train_rows']} rows, got "
                f"{cohort_check['actual_rows']} ({cohort_check['deviation_pct']}% deviation, "
                f"tolerance {cohort_check['tolerance_pct']}%)\n"
                f"  source: {input_csv}\n"
                "Use --namespace dev for the development subset, or point split.input_csv at the "
                "full train.csv on a machine where it is available."
            )

    all_patients = set(frame["patient_id"].unique())
    vectors = patient_label_vectors(frame)
    neg_vectors = patient_label_negative_vectors(frame)
    groups = multilabel_partition_patients(vectors, neg_vectors, fractions, int(splits_cfg.split_seed))
    assert_disjoint_and_exact(groups, all_patients)

    frames = {
        name: frame[frame["patient_id"].isin(ids)].reset_index(drop=True)
        for name, ids in groups.items()
    }

    support_report = build_support_report(
        frames,
        list(splits_cfg.support_rule.decision_bearing_splits),
        int(splits_cfg.support_rule.min_positive_patients),
        int(splits_cfg.support_rule.min_negative_patients),
    )

    adjustment = None
    if not support_report["passed"]:
        adjustment = propose_fraction_adjustment(support_report, fractions, len(all_patients))

    # pathlib mkdir avoids Windows tempfile ACL inheritance that can make the staging directory
    # unreadable to the creating process in managed workspaces; uniqueness and same-filesystem
    # atomic publication are preserved.
    staging = out_dir.parent / f".{args.run_id}.staging-{uuid.uuid4().hex}"
    staging.mkdir()
    output_files = {}
    for name, split_frame in frames.items():
        if name == FINAL_EVAL_SPLIT:
            assert_final_eval_access_allowed(
                purpose="split_construction", caller="02b_build_sixway_splits"
            )
        csv_path = staging / f"{name}.csv"
        split_frame.to_csv(csv_path, index=False)
        reread = pd.read_csv(csv_path)
        if len(reread) != len(split_frame) or set(reread["patient_id"].astype(str)) != groups[name]:
            raise SystemExit(f"Staged split validation failed for {name}")
        output_files[name] = {"filename": csv_path.name, "sha256": sha256_file(csv_path), "row_count": len(reread), "patient_count": len(groups[name])}

    # Retain and mark the superseded v1 artifacts rather than deleting them (§1.5).
    legacy_manifest = Path(splits_cfg.paths.splits_root) / "split_manifest.json"
    if legacy_manifest.is_file():
        note_path = Path(splits_cfg.paths.splits_root) / SUPERSEDED_NOTE_FILENAME
        if not note_path.exists():
            note_path.write_text(SUPERSEDED_NOTE, encoding="utf-8")

    core = {
        "manifest_version": int(splits_cfg.manifest_version),
        "split_namespace": args.run_id,
        "namespace_class": args.namespace,
        "split_unit": str(splits_cfg.split_unit),
        "split_seed": int(splits_cfg.split_seed),
        "fractions": fractions,
        "input_csv_path": str(input_csv),
        "input_csv_rows": len(frame),
        "source_csv_sha256": sha256_file(input_csv),
        "source_schema_hash": source_validation["source_schema_hash"],
        "sorted_patient_set_hash": hash_dict({"patient_ids": sorted(all_patients)}, length=64),
        "output_files": output_files,
        "allocation_algorithm": "deterministic_greedy_iterative_multilabel_v1",
        "allocation_library": "internal",
        "prevalence": prevalence_report(vectors, groups),
        "num_patients_total": len(all_patients),
        "patients_per_split": {name: len(ids) for name, ids in groups.items()},
        "images_per_split": {name: len(f) for name, f in frames.items()},
        "patient_id_hash_per_split": {
            name: hash_dict({"patient_ids": sorted(ids)}, length=16) for name, ids in groups.items()
        },
        "primary_endpoint_labels": PRIMARY_ENDPOINT_LABELS,
    }

    manifest = dict(core)
    manifest["git_commit_hash"] = get_git_commit_hash()
    manifest["full_cohort_check"] = cohort_check
    manifest["support_check"] = support_report
    manifest["support_adjustment_proposal"] = adjustment
    manifest["frozen"] = False
    manifest["supersedes"] = "split_manifest.json (schema v1, three-way + derived children)"
    manifest["environment_versions"] = get_library_versions()

    if args.freeze:
        if not support_report["passed"]:
            shutil.rmtree(staging)
            print("\n" + "=" * 78, flush=True)
            print("SUPPORT CHECK FAILED — REFUSING TO FREEZE", flush=True)
            print("=" * 78, flush=True)
            for split_name, result in support_report["per_split"].items():
                if result["passed"]:
                    continue
                print(f"\n[{split_name}] {len(result['failures'])} label(s) below the frozen rule:", flush=True)
                for failure in result["failures"]:
                    print(
                        f"    {failure['label']:<32} "
                        f"pos={failure['positive_patients']:>5} (need {failure['min_positive_patients']})  "
                        f"neg={failure['negative_patients']:>5} (need {failure['min_negative_patients']})",
                        flush=True,
                    )
            if adjustment:
                print("\nRough support scaling estimate (not an optimized allocation):", flush=True)
                for split_name, proposal in adjustment["proposals"].items():
                    print(
                        f"    {split_name}: {proposal['current_fraction']} -> "
                        f"{proposal['proposed_fraction']} "
                        f"(+{proposal['additional_patients_needed']} patients)",
                        flush=True,
                    )
                print(
                    f"    donor: {adjustment['suggested_donor_split']} -> "
                    f"{adjustment['suggested_gen_train_fraction']}",
                    flush=True,
                )
                print(f"\n    {adjustment['note']}", flush=True)
            print("\nNo split run was published because the frozen support gate failed.", flush=True)
            print(
                f"\nThe frozen split fractions {fractions} were NOT modified. Changing them is an "
                "explicit decision (docs/stages2_to_5_plan.md §1.3).",
                flush=True,
            )
            return 2
        manifest["frozen"] = True

    manifest["manifest_hash_scope"] = "full_manifest_without_manifest_hash_v2"
    manifest["manifest_hash"] = hash_dict(manifest, length=64)
    write_json(staging / SPLIT_MANIFEST_FILENAME, manifest)
    os.replace(staging, out_dir)

    print(f"Namespace:     {args.namespace}", flush=True)
    print(f"Source CSV:    {input_csv} ({len(frame)} rows)", flush=True)
    print(f"Patients:      {len(all_patients)}", flush=True)
    print(f"Seed:          {splits_cfg.split_seed}", flush=True)
    for name in SPLIT_NAMES:
        print(
            f"  {name:<24} {len(groups[name]):>7} patients  {len(frames[name]):>8} images  "
            f"({fractions[name]:.0%})",
            flush=True,
        )
    print(f"Support check: {'PASSED' if support_report['passed'] else 'FAILED'}", flush=True)
    if not support_report["passed"]:
        failing = [
            f"{split_name}({len(result['failures'])})"
            for split_name, result in support_report["per_split"].items()
            if not result["passed"]
        ]
        print(f"  failing splits: {', '.join(failing)}", flush=True)
        if args.namespace == "dev":
            print(
                "  (expected on the dev subset — recorded in the manifest, not fatal; "
                "these are NOT production-frozen thesis splits)",
                flush=True,
            )
    print(f"Frozen:        {manifest['frozen']}", flush=True)
    print(f"Manifest hash: {manifest['manifest_hash']}", flush=True)
    print(f"Wrote:         {out_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/03_preprocess_images.py`

*224 lines*

**Pipeline step 4.** Turns raw CheXpert JPEGs into the fixed-size images Stage 1 trains on.

- **Frontal-only filter** (lateral views are dropped).
- `letterbox_resize` - aspect-preserving resize + pad. Chest anatomy must not be stretched.
- Quality gates: minimum source resolution, blank-image detection (`BLANK_STD_THRESHOLD`), decode
  failures - each rejection is counted and logged per split.
- `atomic_save_jpeg` - temp file + `os.replace`, so an interrupted run leaves no half-written JPEG.
- **Resumable and idempotent on real terms**: an existing output is skipped only after being fully
  decoded and validated (`validate_processed_jpeg` checks format, mode *and* dimensions), never on
  mere existence.
- `expected_manifest` + `check_manifest` - the staleness gate. If the config or the split hash
  changed, it refuses to mix new outputs into an old directory. It also refuses to reuse
  "unpinned" outputs that exist without a manifest.

In [ ]:
# ===== FILE: scripts/data/03_preprocess_images.py =====
from __future__ import annotations

import argparse
import json
import os
import sys
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.identifiers import sanitize_image_id  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import SPLIT_NAMES, read_split_manifest, resolve_splits_dir  # noqa: E402

BLANK_STD_THRESHOLD = 5.0
JPEG_QUALITY = 95
MANIFEST_VERSION = 2


def resolve_source_path(raw_dir: Path, raw_path: str) -> Path:
    value = str(raw_path).replace("\\", "/")
    for prefix in ("CheXpert-v1.0-small/", "CheXpert-v1.0/"):
        if value.startswith(prefix):
            value = value[len(prefix):]
    return raw_dir / value


def validate_processed_jpeg(path: Path, resolution: int) -> tuple[bool, str | None]:
    """Fully decode a JPEG and require its encoded mode and dimensions to match the config."""
    if not path.is_file():
        return False, "missing"
    try:
        with Image.open(path) as image:
            if image.format != "JPEG":
                return False, f"format_{image.format or 'unknown'}"
            if image.mode != "RGB":
                return False, f"mode_{image.mode}"
            if image.size != (resolution, resolution):
                return False, f"size_{image.size[0]}x{image.size[1]}"
            image.load()
    except Exception as exc:
        return False, f"unreadable_{type(exc).__name__}"
    return True, None


def letterbox_resize(image: Image.Image, target_size: int) -> Image.Image:
    width, height = image.size
    scale = target_size / max(width, height)
    new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    resized = image.resize(new_size, Image.Resampling.BICUBIC)
    canvas = Image.new("RGB", (target_size, target_size), (0, 0, 0))
    canvas.paste(resized, ((target_size - new_size[0]) // 2, (target_size - new_size[1]) // 2))
    return canvas


def atomic_save_jpeg(image: Image.Image, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        image.save(temporary, format="JPEG", quality=JPEG_QUALITY)
        valid, reason = validate_processed_jpeg(temporary, image.width)
        if not valid:
            raise OSError(f"temporary JPEG validation failed: {reason}")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)


def expected_manifest(stage1_cfg, split_paths: dict[str, Path], namespace: str | None = None, split_manifest: dict | None = None) -> dict:
    data = stage1_cfg.data
    namespace = namespace or str(stage1_cfg.split.namespace)
    split_manifest = split_manifest or read_split_manifest(namespace)
    settings = {
        "resolution": int(data.resolution), "aspect_mode": str(data.aspect_mode),
        "view_filter": str(data.view_filter), "min_source_resolution": int(data.min_source_resolution),
        "horizontal_flip": bool(data.horizontal_flip), "blank_std_threshold": BLANK_STD_THRESHOLD,
        "jpeg_quality": JPEG_QUALITY,
    }
    return {
        "manifest_version": MANIFEST_VERSION,
        "preprocessing_algorithm_version": 2,
        "output_schema_version": 2,
        "split_namespace": namespace,
        "split_manifest_hash": split_manifest.get("manifest_hash"),
        "preprocessing_config_hash": hash_dict(settings, length=64),
        "output_settings": settings,
        "source_splits": {
            name: {"path": path.name, "sha256": sha256_file(path), "size_bytes": path.stat().st_size}
            for name, path in sorted(split_paths.items())
        },
    }


def check_manifest(path: Path, expected: dict, adopt_legacy: bool) -> None:
    if not path.exists():
        return
    actual = read_json(path)
    if actual == expected:
        return
    if "manifest_version" not in actual and adopt_legacy:
        legacy_settings = {k: actual.get(k) for k in expected["output_settings"] if k in actual}
        current_settings = {k: expected["output_settings"][k] for k in legacy_settings}
        if legacy_settings != current_settings:
            raise SystemExit(f"Legacy preprocessing settings conflict with the current configuration:\nold={legacy_settings}\nnew={current_settings}")
        print("WARNING: adopting a legacy preprocessing manifest with no split hashes. Existing files will still be decoded and validated before skipping.", flush=True)
        return
    raise SystemExit(
        "Existing preprocessing outputs were created with incompatible or unverifiable settings/splits.\n"
        f"Manifest: {path}\nExisting: {json.dumps(actual, indent=2, sort_keys=True)}\n"
        f"Requested: {json.dumps(expected, indent=2, sort_keys=True)}\n"
        "Use the original configuration/split files. For a legacy manifest only, review it and rerun with --adopt-legacy-manifest."
    )


def process_split(split_name, df, raw_dir, images_dir, dataset_cfg, data_cfg) -> Counter:
    out_dir = images_dir / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    schema = dataset_cfg.schema
    image_ids = [sanitize_image_id(value) for value in df[schema.path_column]]
    duplicates = [item for item, count in Counter(image_ids).items() if count > 1]
    if duplicates:
        raise SystemExit(f"{split_name}: {len(duplicates)} duplicate output image IDs; first examples: {duplicates[:5]}")

    log_path = images_dir / f"{split_name}_preprocessing_log.jsonl"
    fd, temporary_name = tempfile.mkstemp(prefix=f".{log_path.name}.", suffix=".tmp", dir=images_dir)
    counts = Counter(processed=0, skipped_existing=0, filtered=0, failed=0)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", buffering=1) as log:
            progress = tqdm(df.itertuples(index=False, name=None), total=len(df), desc=f"[{split_name}] preprocess", unit="image")
            columns = {name: index for index, name in enumerate(df.columns)}
            for values in progress:
                raw_path = values[columns[schema.path_column]]
                image_id = sanitize_image_id(raw_path)
                destination = out_dir / f"{image_id}.jpg"
                frontal_value = values[columns[schema.frontal_lateral_column]] if schema.frontal_lateral_column in columns else "Frontal"
                reason = None
                detail = None
                action = "processed"

                if str(data_cfg.view_filter) == "frontal" and not str(frontal_value).strip().lower().startswith("front"):
                    action, reason = "filtered", "filtered_lateral_view"
                else:
                    valid, validation_reason = validate_processed_jpeg(destination, int(data_cfg.resolution))
                    if valid:
                        action, reason = "skipped_existing", "validated_existing"
                    else:
                        source = resolve_source_path(raw_dir, raw_path)
                        if not source.is_file():
                            action, reason = "failed", "source_file_missing"
                        else:
                            try:
                                with Image.open(source) as source_image:
                                    gray = source_image.convert("L")
                                    width, height = gray.size
                                    if min(width, height) < int(data_cfg.min_source_resolution):
                                        action, reason = "filtered", "below_min_source_resolution"
                                    elif np.asarray(gray, dtype=np.float32).std() < BLANK_STD_THRESHOLD:
                                        action, reason = "filtered", "near_uniform_blank"
                                    else:
                                        atomic_save_jpeg(letterbox_resize(gray.convert("RGB"), int(data_cfg.resolution)), destination)
                                        detail = validation_reason and f"replaced_{validation_reason}"
                            except Exception as exc:
                                action, reason, detail = "failed", "corrupt_or_unreadable", f"{type(exc).__name__}: {exc}"

                counts[action] += 1
                record = {"image_id": image_id, "raw_path": str(raw_path), "kept": action in ("processed", "skipped_existing"), "action": action, "reason": reason, "detail": detail}
                log.write(json.dumps(record, ensure_ascii=False) + "\n")
                progress.set_postfix({key: counts[key] for key in ("processed", "skipped_existing", "filtered", "failed")}, refresh=False)
            progress.close()
            log.flush()
            os.fsync(log.fileno())
        os.replace(temporary_name, log_path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)
    print(f"[{split_name}] processed={counts['processed']} skipped-existing={counts['skipped_existing']} filtered={counts['filtered']} failed={counts['failed']} log={log_path}", flush=True)
    return counts


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None, help="Explicit v2 split namespace/run (dev or production run ID)")
    parser.add_argument("--splits", nargs="+", choices=SPLIT_NAMES, default=SPLIT_NAMES)
    parser.add_argument("--adopt-legacy-manifest", action="store_true", help="Explicitly accept an old manifest lacking split hashes after checking its recorded settings")
    args = parser.parse_args()
    cfg, dataset_cfg = load_stage1_config(), load_dataset_config()
    ensure_dirs(cfg)
    namespace = args.namespace or str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_dir = resolve_splits_dir(namespace)
    split_paths = {name: split_dir / f"{name}.csv" for name in args.splits}
    missing = [str(path) for path in split_paths.values() if not path.is_file()]
    if missing:
        raise SystemExit(
            f"Missing split files (run 02b_build_sixway_splits.py --namespace {namespace} first): "
            + ", ".join(missing)
        )
    images_dir = Path(cfg.paths.images_dir) / namespace
    images_dir.mkdir(parents=True, exist_ok=True)
    for name, path in split_paths.items():
        manifest = expected_manifest(cfg, {name: path}, namespace, split_manifest)
        manifest_path = images_dir / f"{name}_preprocessing_manifest.json"
        split_output_dir = images_dir / name
        if not manifest_path.is_file() and split_output_dir.is_dir() and any(split_output_dir.iterdir()):
            raise SystemExit(
                f"Unpinned preprocessing outputs exist at {split_output_dir} without {manifest_path}; "
                "refusing stale cross-assignment reuse. Preserve them and choose a new versioned namespace."
            )
        check_manifest(manifest_path, manifest, args.adopt_legacy_manifest)
        write_json(manifest_path, manifest)
        process_split(name, pd.read_csv(path), Path(cfg.paths.raw_dir), images_dir, dataset_cfg, cfg.data)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/data/04_generate_captions.py`

*159 lines*

**Pipeline step 5.** Writes one caption record per preprocessed image, using
`scripts/utils/caption_builder.py` - the same module Stage 2 later generates with.

For each row it emits all configured paraphrase **variants** (`build_caption_variants`), so Stage 1
can pre-embed every caption an image might use and pick one deterministically per epoch. It also
stores `raw_label_vector` (the untouched ternary labels) next to the caption.

It imports `expected_manifest` and `validate_processed_jpeg` from `03_preprocess_images.py` rather
than re-implementing them, and refuses to caption images that preprocessing did not successfully
produce (`load_preprocessing_status`). `previous_caption_ids` makes re-runs incremental.

In [ ]:
# ===== FILE: scripts/data/04_generate_captions.py =====
from __future__ import annotations

import argparse
import importlib
import json
import os
import sys
import tempfile
from collections import Counter
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, PATHOLOGY_COLUMNS, build_caption_variants  # noqa: E402
from scripts.utils.config import ensure_dirs, load_dataset_config, load_stage1_config  # noqa: E402
from scripts.utils.identifiers import sanitize_image_id  # noqa: E402
from scripts.utils.manifest import read_json, write_json  # noqa: E402
from scripts.utils.splits import read_split_manifest, resolve_splits_dir  # noqa: E402

_preprocess = importlib.import_module("scripts.data.03_preprocess_images")
expected_manifest = _preprocess.expected_manifest
validate_processed_jpeg = _preprocess.validate_processed_jpeg


def build_caption_config(cfg) -> CaptionConfig:
    value = cfg.captions
    return CaptionConfig(value.age_bucket_width_years, value.uncertain_label_policy, value.no_finding_overrides_positives, value.num_paraphrase_variants, value.template_version)


def raw_label_vector(row: pd.Series) -> dict:
    result = {}
    for column in PATHOLOGY_COLUMNS:
        value = row.get(column)
        result[column] = None if value is None or pd.isna(value) else int(value)
    return result


def load_preprocessing_status(path: Path) -> dict[str, dict]:
    if not path.is_file():
        raise SystemExit(f"Missing preprocessing log: {path}. Complete preprocessing for this split first.")
    result = {}
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            try:
                record = json.loads(line)
            except Exception as exc:
                raise SystemExit(f"Invalid preprocessing log JSON at {path}:{line_number}: {exc}") from exc
            image_id = record.get("image_id")
            if not image_id or image_id in result:
                raise SystemExit(f"Missing/duplicate image_id in {path}:{line_number}: {image_id!r}")
            result[image_id] = record
    return result


def previous_caption_ids(path: Path) -> tuple[set[str], int]:
    ids, duplicates = set(), 0
    if path.is_file():
        with path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                image_id = json.loads(line).get("image_id")
                duplicates += image_id in ids
                ids.add(image_id)
    return ids, duplicates


def process_split(name, df, images_dir, captions_dir, schema, caption_cfg, resolution) -> Counter:
    ids = [sanitize_image_id(value) for value in df[schema.path_column]]
    duplicate_source_ids = [key for key, count in Counter(ids).items() if count > 1]
    if duplicate_source_ids:
        raise SystemExit(f"{name}: duplicate image IDs in split CSV: {duplicate_source_ids[:5]}")
    status = load_preprocessing_status(images_dir / f"{name}_preprocessing_log.jsonl")
    split_ids = set(ids)
    unknown_log_ids = set(status) - split_ids
    missing_log_ids = split_ids - set(status)
    if unknown_log_ids or missing_log_ids:
        raise SystemExit(f"{name}: stale/incomplete preprocessing log: missing={len(missing_log_ids)}, not-in-split={len(unknown_log_ids)}")

    destination = captions_dir / f"{name}_captions.jsonl"
    previous_ids, previous_duplicates = previous_caption_ids(destination)
    fd, temporary_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=captions_dir)
    counts = Counter(written=0, filtered_or_failed=0, missing_or_invalid=0)
    written_ids = set()
    try:
        with os.fdopen(fd, "w", encoding="utf-8", buffering=1) as output:
            for (_, row), image_id in tqdm(zip(df.iterrows(), ids), total=len(df), desc=f"[{name}] captions", unit="record"):
                preprocess_record = status[image_id]
                if not preprocess_record.get("kept", False):
                    counts["filtered_or_failed"] += 1
                    continue
                image_path = images_dir / name / f"{image_id}.jpg"
                valid, reason = validate_processed_jpeg(image_path, resolution)
                if not valid:
                    counts["missing_or_invalid"] += 1
                    print(f"[{name}] WARNING: excluding {image_id}: processed image is {reason}", flush=True)
                    continue
                record = {
                    "image_id": image_id, "split": name,
                    "image_relpath": (Path(name) / f"{image_id}.jpg").as_posix(),
                    "patient_id": row.get("patient_id"),
                    "caption_variants": build_caption_variants(row.to_dict(), caption_cfg),
                    "raw_labels": raw_label_vector(row), "sex": row.get(schema.sex_column),
                    "age": row.get(schema.age_column), "frontal_lateral": row.get(schema.frontal_lateral_column),
                    "ap_pa": row.get(schema.ap_pa_column),
                }
                output.write(json.dumps(record, ensure_ascii=False, default=lambda value: None if pd.isna(value) else str(value)) + "\n")
                written_ids.add(image_id)
                counts["written"] += 1
            output.flush()
            os.fsync(output.fileno())
        os.replace(temporary_name, destination)
    finally:
        Path(temporary_name).unlink(missing_ok=True)
    stale = previous_ids - written_ids
    missing_previous = written_ids - previous_ids if previous_ids else set()
    print(f"[{name}] written={counts['written']} filtered/failed={counts['filtered_or_failed']} missing/invalid={counts['missing_or_invalid']} prior-duplicates={previous_duplicates} stale-prior={len(stale)} new-vs-prior={len(missing_previous)} -> {destination}", flush=True)
    return counts


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--splits", nargs="+", default=["gen_train", "gen_val"])
    args = parser.parse_args()
    cfg, dataset_cfg = load_stage1_config(), load_dataset_config()
    ensure_dirs(cfg)
    namespace = args.namespace or str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_paths = {name: resolve_splits_dir(namespace) / f"{name}.csv" for name in args.splits}
    missing = [str(path) for path in split_paths.values() if not path.is_file()]
    if missing:
        raise SystemExit("Missing split files: " + ", ".join(missing))
    images_dir = Path(cfg.paths.images_dir) / namespace
    captions_dir = Path(cfg.paths.captions_dir) / namespace
    captions_dir.mkdir(parents=True, exist_ok=True)
    for name, path in split_paths.items():
        expected = expected_manifest(cfg, {name: path}, namespace, split_manifest)
        manifest_path = images_dir / f"{name}_preprocessing_manifest.json"
        if not manifest_path.is_file() or read_json(manifest_path) != expected:
            raise SystemExit(f"Preprocessing provenance for {name} does not match the current namespace/settings. Rerun preprocessing first.")
        process_split(name, pd.read_csv(path), images_dir, captions_dir, dataset_cfg.schema, build_caption_config(cfg), int(cfg.data.resolution))
    write_json(captions_dir / "caption_template_version.json", {
        "template_version": cfg.captions.template_version,
        "num_paraphrase_variants": cfg.captions.num_paraphrase_variants,
        "age_bucket_width_years": cfg.captions.age_bucket_width_years,
        "uncertain_label_policy": cfg.captions.uncertain_label_policy,
        "no_finding_overrides_positives": cfg.captions.no_finding_overrides_positives,
        "preprocessing_manifests": {name: read_json(images_dir / f"{name}_preprocessing_manifest.json") for name in args.splits},
        "split_namespace": namespace,
        "split_manifest_hash": split_manifest.get("manifest_hash"),
    })
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 4 - Stage 1: SDXL + LoRA fine-tuning

Fine-tune the generator. One long GPU job, built to survive pod restarts and SSH drops, plus the two monitoring scripts used to watch it.

## `scripts/train/train_lora_sdxl.py`

*881 lines*

**Stage 1 - the generator.** SDXL fine-tuned with LoRA on CheXpert-derived captions. Adapted from
the diffusers `text_to_image_lora_sdxl` reference pattern, with the production concerns added:

- **UNet-only LoRA** on self- and cross-attention projections; both text encoders stay frozen.
- **bf16** mixed precision, PyTorch SDPA attention, gradient checkpointing.
- **One-time VAE latent + text-embedding caching** (`build_or_load_cache`). Captions are
  deterministic from labels, so nothing needs re-encoding every epoch - this is the single biggest
  speed win in the stage. `CachedSDXLDataset` serves from that cache and `set_epoch` selects the
  caption variant for the epoch.
- **`LoraEMA`** - exponential moving average of just the LoRA weights.
- **Resumable checkpointing** through Accelerate `save_state`/`load_state`, all paths under
  `PROJECT_ROOT`; `save_checkpoint` also writes `latest.json` and full metadata (config hash, split
  manifest hash, seed, library versions) and prunes to `keep_last_n`.
- **`validate_training_inputs`** - the upstream gate: split manifest, caption files and
  preprocessing manifests must all match what this run expects before a single step is taken.
- TensorBoard (and optional W&B) logging via `tracker_hparams`.

In [ ]:
# ===== FILE: scripts/train/train_lora_sdxl.py =====
#!/usr/bin/env python3
"""SDXL + LoRA fine-tuning on CheXpert (docs/stage1_plan.md §3, §8).

Adapted from diffusers' text_to_image_lora_sdxl reference pattern, with:
  - UNet-only LoRA (self+cross attention projections), text encoders frozen
  - bf16 mixed precision, PyTorch SDPA attention, gradient checkpointing
  - one-time VAE latent + text embedding caching (captions are deterministic-from-labels)
  - EMA of the LoRA adapter weights
  - resumable checkpointing via Accelerate save_state/load_state, all paths under PROJECT_ROOT
    (expected to be the persistent volume on RunPod — see configs/stage1_lora_sdxl.yaml).
    save_model_hook/load_model_hook keep only the LoRA adapter in the resumable state, so a
    checkpoint is ~11 MB rather than ~9.5 GB of re-serialized frozen SDXL base weights.
  - TensorBoard (+ optional Weights & Biases) logging

Run directly, or via scripts/train/launch_resumable.sh which handles the resume-on-restart logic.

Usage:
    accelerate launch --config_file configs/accelerate_config.yaml scripts/train/train_lora_sdxl.py \\
        [--resume-from CHECKPOINT_DIR] [--run-id RUN_ID] [override.key=value ...]
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from omegaconf import OmegaConf
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import select_variant_index  # noqa: E402
from scripts.utils.config import ensure_dirs, load_stage1_config  # noqa: E402
from scripts.utils.manifest import build_checkpoint_metadata, make_run_id, read_json, write_json  # noqa: E402
from scripts.utils.seed import set_seed  # noqa: E402


# --------------------------------------------------------------------------------------
# Model / cache construction
# --------------------------------------------------------------------------------------

def tracker_hparams(cfg) -> dict[str, int | float | str | bool]:
    """Flatten a resolved OmegaConf config into TensorBoard-safe scalar hparams."""
    resolved = OmegaConf.to_container(cfg, resolve=True)
    flattened: dict[str, int | float | str | bool] = {}

    def visit(prefix: str, value) -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                visit(f"{prefix}.{key}" if prefix else str(key), child)
        elif isinstance(value, (list, tuple)):
            flattened[prefix] = json.dumps(value, sort_keys=True)
        elif value is None:
            flattened[prefix] = "null"
        elif isinstance(value, (int, float, str, bool)):
            flattened[prefix] = value
        else:
            flattened[prefix] = str(value)

    visit("", resolved)
    return flattened

def build_models(cfg, device_dtype: torch.dtype):
    from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
    from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer
    from peft import LoraConfig

    model_id = cfg.model.base_model_id
    cache_dir = cfg.paths.hf_cache_dir
    revision = cfg.model.revision if cfg.model.revision else None

    tokenizer_one = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer", revision=revision, cache_dir=cache_dir)
    tokenizer_two = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer_2", revision=revision, cache_dir=cache_dir)
    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler", revision=revision, cache_dir=cache_dir)

    text_encoder_one = CLIPTextModel.from_pretrained(
        model_id, subfolder="text_encoder", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )
    text_encoder_two = CLIPTextModelWithProjection.from_pretrained(
        model_id, subfolder="text_encoder_2", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )
    vae = AutoencoderKL.from_pretrained(
        model_id, subfolder="vae", revision=revision, cache_dir=cache_dir,
        torch_dtype=torch.float32 if cfg.model.vae_upcast_fp32 else device_dtype,
    )
    unet = UNet2DConditionModel.from_pretrained(
        model_id, subfolder="unet", revision=revision, cache_dir=cache_dir, torch_dtype=device_dtype
    )

    for model in (vae, text_encoder_one, text_encoder_two, unet):
        model.requires_grad_(False)

    if cfg.lora.train_text_encoders:
        raise NotImplementedError(
            "lora.train_text_encoders=true is a documented fallback (docs/stage1_plan.md §3) "
            "for weak rare-label conditioning, not implemented in v1. Set it back to false, or "
            "implement CLIPText LoRA attachment here before enabling it."
        )

    lora_config = LoraConfig(
        r=cfg.lora.rank,
        lora_alpha=cfg.lora.alpha,
        lora_dropout=cfg.lora.dropout,
        init_lora_weights="gaussian",
        target_modules=list(cfg.lora.target_modules),
    )
    unet.add_adapter(lora_config)
    for p in unet.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    if cfg.training.gradient_checkpointing:
        unet.enable_gradient_checkpointing()

    return {
        "tokenizer_one": tokenizer_one,
        "tokenizer_two": tokenizer_two,
        "text_encoder_one": text_encoder_one,
        "text_encoder_two": text_encoder_two,
        "vae": vae,
        "unet": unet,
        "noise_scheduler": noise_scheduler,
    }


def _load_captions(captions_path: Path) -> list[dict]:
    records = []
    seen = set()
    with open(captions_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                record = json.loads(line)
                image_id = record.get("image_id")
                if not image_id or image_id in seen:
                    raise ValueError(f"Missing or duplicate image_id {image_id!r} in {captions_path}")
                if not record.get("caption_variants"):
                    raise ValueError(f"Caption record {image_id!r} has no variants")
                seen.add(image_id)
                records.append(record)
    return records


def validate_training_inputs(cfg) -> None:
    """Fail before model downloads when captions/preprocessing no longer match the config."""
    namespace = str(cfg.split.namespace)
    metadata_path = Path(cfg.paths.captions_dir) / namespace / "caption_template_version.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Missing caption metadata: {metadata_path}; run 04_generate_captions.py")
    metadata = read_json(metadata_path)
    expected_caption = {
        "template_version": cfg.captions.template_version,
        "num_paraphrase_variants": cfg.captions.num_paraphrase_variants,
        "age_bucket_width_years": cfg.captions.age_bucket_width_years,
        "uncertain_label_policy": cfg.captions.uncertain_label_policy,
        "no_finding_overrides_positives": cfg.captions.no_finding_overrides_positives,
    }
    mismatches = {key: (metadata.get(key), value) for key, value in expected_caption.items() if metadata.get(key) != value}
    if metadata.get("split_namespace") != namespace:
        mismatches["split_namespace"] = (metadata.get("split_namespace"), namespace)
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    split_manifest = read_split_manifest(namespace)
    if metadata.get("split_manifest_hash") != split_manifest.get("manifest_hash"):
        mismatches["split_manifest_hash"] = (metadata.get("split_manifest_hash"), split_manifest.get("manifest_hash"))
    preprocessing_by_split = metadata.get("preprocessing_manifests", {})
    preprocessing = preprocessing_by_split.get("gen_train", {})
    output_settings = preprocessing.get("output_settings", {})
    if output_settings.get("resolution") != cfg.data.resolution:
        mismatches["preprocessing_resolution"] = (output_settings.get("resolution"), cfg.data.resolution)
    for split_name in ("gen_train", "gen_val"):
        split_path = resolve_splits_dir(namespace) / f"{split_name}.csv"
        recorded = preprocessing_by_split.get(split_name, {}).get("source_splits", {}).get(split_name, {}).get("sha256")
        current = hashlib.sha256(split_path.read_bytes()).hexdigest() if split_path.is_file() else None
        if recorded != current:
            mismatches[f"{split_name}_sha256"] = (recorded, current)
    if mismatches:
        raise ValueError(f"Caption/preprocessing provenance is stale or incompatible: {mismatches}. Rerun preprocessing and captions.")


def select_subset_indices(n_records: int, max_samples: int | None, seed: int) -> list[int] | None:
    """Sorted indices of a seeded random subset of `max_samples` records, or None for all of them.

    Sorted so the subset keeps the caption file's order, and drawn with its own generator so the
    choice depends only on (n_records, max_samples, seed), never on the global RNG state.
    """
    if max_samples is None or max_samples >= n_records:
        return None
    if max_samples < 1:
        raise ValueError(f"max_val_samples must be positive or null, got {max_samples}")
    generator = torch.Generator().manual_seed(seed)
    return sorted(torch.randperm(n_records, generator=generator)[:max_samples].tolist())


def build_or_load_cache(
    cfg, models, split_name: str, device: torch.device, max_samples: int | None = None,
    subset_size: int | None = None, subset_seed: int = 0,
):
    """One-time VAE latent + text embedding cache per split (docs/stage1_plan.md §8).

    Latents are cached per image (as encoder-distribution moments, so a fresh sample is still
    drawn each epoch). Text embeddings are cached per unique caption string across all
    paraphrase variants of all images, since many images share the low-cardinality caption
    vocabulary.

    `subset_size` keeps a seeded random subset of the records (see select_subset_indices) before
    anything is encoded, so only that subset is cached; the cache key covers the kept image ids.
    """
    namespace = str(cfg.split.namespace)
    images_dir = Path(cfg.paths.images_dir) / namespace
    captions_path = Path(cfg.paths.captions_dir) / namespace / f"{split_name}_captions.jsonl"
    cache_dir = Path(cfg.paths.processed_dir) / "cache" / namespace
    cache_dir.mkdir(parents=True, exist_ok=True)
    records = _load_captions(captions_path)
    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("--max-samples-per-split must be positive")
        records = records[:max_samples]
    subset = select_subset_indices(len(records), subset_size, subset_seed)
    if subset is not None:
        records = [records[i] for i in subset]
    if not records:
        raise FileNotFoundError(
            f"No caption records found at {captions_path} — run 03_preprocess_images.py and "
            "04_generate_captions.py first."
        )

    resolution = cfg.data.resolution
    record_identity = json.dumps([record["image_id"] for record in records], separators=(",", ":"))
    captions_hash = hashlib.sha256((hashlib.sha256(captions_path.read_bytes()).hexdigest() + record_identity).encode()).hexdigest()[:12]
    model_key = hashlib.sha256(f"{cfg.model.base_model_id}:{cfg.model.revision}:{resolution}".encode()).hexdigest()[:8]
    cache_key = f"{captions_hash}_{model_key}"
    latent_cache_path = cache_dir / f"{split_name}_latent_cache_{cache_key}.pt"
    prompt_cache_path = cache_dir / f"{split_name}_prompt_cache_{cache_key}.pt"

    for record in records:
        image_path = images_dir / record["image_relpath"]
        if not image_path.is_file():
            raise FileNotFoundError(f"Caption record {record['image_id']} references missing image: {image_path}")

    if not latent_cache_path.exists():
        from PIL import Image
        from torchvision.transforms import functional as TF

        vae = models["vae"].to(device, dtype=torch.float32).eval()
        moments = torch.empty((len(records), 8, resolution // 8, resolution // 8), dtype=torch.float16)
        with torch.inference_mode():
            for i, rec in enumerate(tqdm(records, desc=f"[{split_name}] VAE latent cache")):
                image_path = images_dir / rec["image_relpath"]
                with Image.open(image_path) as im:
                    pixels = TF.normalize(TF.to_tensor(im.convert("RGB")), [0.5] * 3, [0.5] * 3)
                params = vae.encode(pixels.unsqueeze(0).to(device, torch.float32)).latent_dist.parameters
                moments[i] = params[0].cpu().half()
        torch.save(
            {
                "image_ids": [r["image_id"] for r in records],
                "moments": moments,
                "scaling_factor": vae.config.scaling_factor,
                "target_size": resolution,
            },
            latent_cache_path,
        )
        vae.to("cpu")
        del moments
        torch.cuda.empty_cache()
    else:
        print(f"[{split_name}] using existing latent cache: {latent_cache_path}")

    if not prompt_cache_path.exists():
        text_dtype = {
            "fp32": torch.float32,
            "fp16": torch.float16,
            "bf16": torch.bfloat16,
        }[str(cfg.training.precision).lower()]
        # Some SDXL checkpoints contain mixed stored dtypes (notably text_projection).  Cast the
        # complete frozen encoders explicitly so their hidden states and projection weights agree.
        text_encoder_one = models["text_encoder_one"].to(device=device, dtype=text_dtype).eval()
        text_encoder_two = models["text_encoder_two"].to(device=device, dtype=text_dtype).eval()
        tokenizer_one, tokenizer_two = models["tokenizer_one"], models["tokenizer_two"]

        unique_captions = sorted({c for r in records for c in r["caption_variants"]})
        prompt_cache: dict[str, dict] = {}
        with torch.inference_mode():
            for caption in tqdm(unique_captions, desc=f"[{split_name}] text embedding cache"):
                ids1 = tokenizer_one(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(device)
                ids2 = tokenizer_two(caption, padding="max_length", truncation=True, max_length=77, return_tensors="pt").input_ids.to(device)
                o1 = text_encoder_one(ids1, output_hidden_states=True, return_dict=True)
                o2 = text_encoder_two(ids2, output_hidden_states=True, return_dict=True)
                prompt_cache[caption] = {
                    "prompt_embeds": torch.cat([o1.hidden_states[-2], o2.hidden_states[-2]], dim=-1)[0].cpu().half(),
                    "pooled_embeds": o2.text_embeds[0].cpu().half(),
                }
        torch.save(prompt_cache, prompt_cache_path)
        text_encoder_one.to("cpu")
        text_encoder_two.to("cpu")
        del prompt_cache
        torch.cuda.empty_cache()
    else:
        print(f"[{split_name}] using existing prompt cache: {prompt_cache_path}")

    return latent_cache_path, prompt_cache_path, records


class CachedSDXLDataset(Dataset):
    """Reads precomputed latents + text embeddings; draws a fresh VAE-latent sample and a
    deterministic (per image id + epoch) caption-variant selection each access."""

    def __init__(self, latent_cache_path: Path, prompt_cache_path: Path, records: list[dict], target_size: int):
        self.latent_cache = torch.load(latent_cache_path, map_location="cpu", weights_only=False)
        self.prompt_cache = torch.load(prompt_cache_path, map_location="cpu", weights_only=False)
        self.records = records
        self.target_size = target_size
        self.epoch = 0
        expected_ids = [record["image_id"] for record in records]
        if self.latent_cache.get("image_ids") != expected_ids:
            raise ValueError("Latent cache image order does not match the caption records; refusing stale cache")
        if self.latent_cache.get("target_size") != target_size:
            raise ValueError("Latent cache resolution does not match the configured target size")

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        rec = self.records[index]
        moments = self.latent_cache["moments"][index]
        mean, logvar = moments[:4].float(), moments[4:].float().clamp(-30, 20)
        std = torch.exp(0.5 * logvar)
        latent = (mean + std * torch.randn_like(mean)) * self.latent_cache["scaling_factor"]

        variant_idx = select_variant_index(rec["image_id"], self.epoch, len(rec["caption_variants"]))
        caption = rec["caption_variants"][variant_idx]
        prompt = self.prompt_cache[caption]

        time_ids = torch.tensor(
            [self.target_size, self.target_size, 0, 0, self.target_size, self.target_size], dtype=torch.float16
        )
        return {
            "latents": latent.half(),
            "prompt_embeds": prompt["prompt_embeds"],
            "pooled_embeds": prompt["pooled_embeds"],
            "time_ids": time_ids,
        }


# --------------------------------------------------------------------------------------
# EMA
# --------------------------------------------------------------------------------------

class LoraEMA:
    """EMA over the UNet's LoRA parameters only (docs/stage1_plan.md §8: cheap since LoRA
    weights are small; base/frozen weights are never touched).

    With warmup the decay follows diffusers' EMAModel schedule, min(decay, (1 + k) / (10 + k))
    for the k-th update after the first (which copies the weights). Without it, a fixed 0.9999
    started from the zero-initialised (no-op) LoRA keeps 0.9999**n of that init in the shadow:
    ~82% after 2,000 steps and ~22% after 15,000, so the exported final LoRA is badly diluted."""

    def __init__(self, unet, decay: float, warmup: bool = True):
        self.decay = decay
        self.warmup = warmup
        self.num_updates = 0
        self.shadow = {
            name: param.detach().clone().float()
            for name, param in unet.named_parameters()
            if param.requires_grad
        }

    def current_decay(self) -> float:
        """Decay used by the most recent update (num_updates counts updates already applied)."""
        if not self.warmup:
            return self.decay
        step = self.num_updates - 1
        if step <= 0:
            return 0.0
        return min(self.decay, (1 + step) / (10 + step))

    @torch.no_grad()
    def update(self, unet) -> None:
        self.num_updates += 1
        decay = self.current_decay()
        for name, param in unet.named_parameters():
            if not param.requires_grad:
                continue
            self.shadow[name].mul_(decay).add_(param.detach().float(), alpha=1 - decay)

    @torch.no_grad()
    def copy_to(self, unet) -> None:
        for name, param in unet.named_parameters():
            if param.requires_grad and name in self.shadow:
                param.data.copy_(self.shadow[name].to(param.dtype))

    def state_dict(self) -> dict:
        return {
            "decay": self.decay,
            "warmup": self.warmup,
            "num_updates": self.num_updates,
            "shadow": {k: v.cpu() for k, v in self.shadow.items()},
        }

    def load_state_dict(self, state: dict) -> None:
        if set(state["shadow"]) != set(self.shadow):
            raise ValueError("EMA checkpoint parameters do not match the current LoRA adapter")
        if bool(state.get("warmup", False)) != self.warmup:
            raise ValueError(
                f"EMA checkpoint warmup={state.get('warmup', False)} differs from the configured "
                f"training.ema.warmup={self.warmup}; refusing to resume with a different EMA schedule"
            )
        if self.warmup and "num_updates" not in state:
            raise ValueError("EMA checkpoint has no num_updates, so the warmup schedule cannot resume")
        self.decay = float(state["decay"])
        self.num_updates = int(state.get("num_updates", 0))
        # state_dict() saves the shadow on CPU; keep each tensor on the device its live counterpart
        # is on, or the first update after a resume adds CUDA params to CPU shadows and crashes.
        self.shadow = {
            name: value.float().to(self.shadow[name].device) for name, value in state["shadow"].items()
        }


# --------------------------------------------------------------------------------------
# Checkpointing
# --------------------------------------------------------------------------------------

def _adamw_8bit_step_works(device: torch.device) -> tuple[bool, str]:
    """Actually run one AdamW8bit step on `device` before committing the real run to it.

    Constructing bnb.optim.AdamW8bit succeeds whenever the package imports; the 8-bit CUDA kernel
    is only invoked on .step(). The RunPod pytorch-2.8/cu128 image ships a bitsandbytes build
    without a matching CUDA binary, so construction passed and training then died at the first
    optimizer step -- after the latent cache had been built. Probing a real step here moves that
    failure to before any training and lets the configured "adamw" fallback actually engage
    (docs/stage1_plan.md §3/§12).

    Deterministic by construction: fixed tensors only, so no RNG draw perturbs the seeded run.
    """
    try:
        import bitsandbytes as bnb

        probe = torch.ones(1, device=device, requires_grad=True)
        probe_optimizer = bnb.optim.AdamW8bit([probe], lr=1.0e-4, weight_decay=0.0)
        (probe * probe).sum().backward()
        probe_optimizer.step()
        probe_optimizer.zero_grad(set_to_none=True)
        del probe_optimizer, probe
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return True, ""
    except Exception as exc:  # ImportError, CUDA kernel/symbol errors, anything else
        return False, f"{type(exc).__name__}: {exc}"


def build_optimizer(cfg, trainable_params, device: torch.device) -> tuple[torch.optim.Optimizer, str]:
    """Build the configured optimizer, honouring the documented adamw_8bit -> adamw fallback.

    Returns (optimizer, resolved_name) so the checkpoint metadata records which optimizer actually
    ran rather than which one was requested.
    """
    learning_rate, weight_decay = cfg.optimizer.learning_rate, cfg.optimizer.weight_decay
    if cfg.optimizer.name == "adamw_8bit":
        works, reason = _adamw_8bit_step_works(device)
        if works:
            import bitsandbytes as bnb

            print("optimizer: bitsandbytes AdamW8bit (8-bit step verified on this device).", flush=True)
            return bnb.optim.AdamW8bit(trainable_params, lr=learning_rate, weight_decay=weight_decay), "adamw_8bit"
        print(
            f"optimizer: bitsandbytes AdamW8bit failed its step probe ({reason}); "
            "falling back to torch.optim.AdamW (docs/stage1_plan.md §3/§12).",
            flush=True,
        )
    elif cfg.optimizer.name != "adamw":
        raise ValueError(
            f"Unsupported optimizer.name={cfg.optimizer.name!r}; expected \"adamw_8bit\" or \"adamw\" "
            "(configs/stage1_lora_sdxl.yaml)."
        )
    return torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay), "adamw"


def save_checkpoint(accelerator, unet, ema, cfg, run_dir: Path, step: int, epoch: int, split_manifest_hash: str, seed: int, keep_last_n: int, resolved_optimizer_name: str | None = None) -> None:
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    from diffusers import StableDiffusionXLPipeline

    accelerator.wait_for_everyone()
    if not accelerator.is_main_process:
        return

    # Resumable state: LoRA adapter + optimizer/scheduler/RNG/EMA. The registered
    # save_model_hook (see main()) is what keeps the frozen UNet base weights out of this.
    full_checkpoint_dir = run_dir / f"checkpoint-{step}"
    accelerator.save_state(str(full_checkpoint_dir))

    # Separate inference-ready export, kept for every step (keep_all_lora_weight_snapshots) and
    # pinned by Stage 2's checkpoint.lora_weights_dir. Distinct purpose from the resumable state
    # above, so both are written even though both now contain the same adapter.
    lora_dir = run_dir / "lora_weights" / f"step_{step}"
    raw_unet = accelerator.unwrap_model(unet)
    state = convert_state_dict_to_diffusers(get_peft_model_state_dict(raw_unet))
    StableDiffusionXLPipeline.save_lora_weights(str(lora_dir), unet_lora_layers=state, safe_serialization=True)

    if ema is not None:
        torch.save(ema.state_dict(), lora_dir / "ema_shadow.pt")

    from scripts.utils.manifest import sha256_file
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    namespace = str(cfg.split.namespace)
    manifest = read_split_manifest(namespace)
    split_dir = resolve_splits_dir(namespace)
    source_csv = Path(manifest["input_csv_path"])
    metadata = build_checkpoint_metadata(
        step=step,
        epoch=epoch,
        config=dict(cfg),
        split_manifest_hash=split_manifest_hash,
        seed=seed,
        extra={
            "split_namespace": namespace,
            "split_manifest_version": manifest["manifest_version"],
            "source_csv_hash": manifest.get("source_csv_sha256") or (sha256_file(source_csv) if source_csv.is_file() else None),
            "gen_train_csv_hash": sha256_file(split_dir / "gen_train.csv"),
            "gen_val_csv_hash": sha256_file(split_dir / "gen_val.csv"),
            # What actually ran, which is not always config.optimizer.name -- see build_optimizer.
            "resolved_optimizer_name": resolved_optimizer_name,
        },
    )
    write_json(full_checkpoint_dir / "metadata.json", metadata)
    write_json(lora_dir / "metadata.json", metadata)

    write_json(run_dir / "latest.json", {"checkpoint_dir": str(full_checkpoint_dir), "step": step, "epoch": epoch})

    # Retention: keep only the last N full resumable checkpoints; lora_weights/* is kept in full.
    full_checkpoints = sorted(
        [p for p in run_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    for old in full_checkpoints[:-keep_last_n]:
        import shutil

        shutil.rmtree(old, ignore_errors=True)

    print(f"Saved checkpoint at step {step} -> {full_checkpoint_dir} (+ lora_weights snapshot)")


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------

def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--resume-from", type=str, default=None, help="Full checkpoint directory to resume from")
    parser.add_argument("--run-id", type=str, default=None, help="Existing run_id to continue writing into")
    parser.add_argument("--max-samples-per-split", type=int, default=None, help="Diagnostic/smoke-test limit; uses separate caches")
    parser.add_argument("overrides", nargs="*", help="OmegaConf dotlist overrides, e.g. training.train_batch_size=4")
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    from accelerate import Accelerator
    from accelerate.utils import ProjectConfiguration

    cfg = load_stage1_config(overrides=args.overrides)
    ensure_dirs(cfg)
    validate_training_inputs(cfg)
    set_seed(cfg.run.seed)

    checkpoints_root = Path(cfg.paths.checkpoints_dir)
    run_id = args.run_id or make_run_id(dict(cfg))
    run_dir = checkpoints_root / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    write_json(checkpoints_root / "latest_run.json", {"run_id": run_id})

    from scripts.utils.manifest import sha256_file
    from scripts.utils.splits import read_split_manifest, resolve_splits_dir
    namespace = str(cfg.split.namespace)
    split_manifest = read_split_manifest(namespace)
    split_manifest_hash = split_manifest["manifest_hash"]
    split_dir = resolve_splits_dir(namespace)
    source_csv = Path(split_manifest["input_csv_path"])
    stage1_provenance = {
        "split_namespace": namespace,
        "split_manifest_version": split_manifest["manifest_version"],
        "split_manifest_hash": split_manifest_hash,
        "source_csv_hash": split_manifest.get("source_csv_sha256") or (sha256_file(source_csv) if source_csv.is_file() else None),
        "gen_train_csv_hash": sha256_file(split_dir / "gen_train.csv"),
        "gen_val_csv_hash": sha256_file(split_dir / "gen_val.csv"),
    }

    log_dir = Path(cfg.paths.logs_dir) / run_id
    trackers = []
    if cfg.logging.tensorboard:
        trackers.append("tensorboard")
    if cfg.logging.wandb.enabled:
        trackers.append("wandb")

    precision = str(cfg.training.precision).lower()
    accelerate_precision = "no" if precision == "fp32" else precision
    if accelerate_precision not in {"no", "fp16", "bf16"}:
        raise ValueError(
            f"Unsupported training.precision={cfg.training.precision!r}; "
            "expected one of: fp32, fp16, bf16"
        )

    accelerator = Accelerator(
        gradient_accumulation_steps=cfg.training.gradient_accumulation_steps,
        mixed_precision=accelerate_precision,
        log_with=trackers or None,
        project_config=ProjectConfiguration(project_dir=str(run_dir), logging_dir=str(log_dir)),
    )
    if trackers:
        accelerator.init_trackers("stage1_lora_sdxl", config=tracker_hparams(cfg))

    weight_dtype = {
        "fp32": torch.float32,
        "fp16": torch.float16,
        "bf16": torch.bfloat16,
    }[precision]
    models = build_models(cfg, weight_dtype)

    train_latent_cache, train_prompt_cache, train_records = build_or_load_cache(cfg, models, "gen_train", accelerator.device, args.max_samples_per_split)
    val_latent_cache, val_prompt_cache, val_records = build_or_load_cache(
        cfg, models, "gen_val", accelerator.device, args.max_samples_per_split,
        subset_size=cfg.validation.max_val_samples, subset_seed=int(cfg.validation.val_seed),
    )
    # Training reads only the caches from here on; validation and the final export use the UNet
    # alone. Drop the VAE and both text encoders (~2 GB of host RAM) for the rest of the run.
    for frozen_only in ("vae", "text_encoder_one", "text_encoder_two"):
        models.pop(frozen_only, None)
    gc.collect()
    torch.cuda.empty_cache()

    train_dataset = CachedSDXLDataset(train_latent_cache, train_prompt_cache, train_records, cfg.data.resolution)
    val_dataset = CachedSDXLDataset(val_latent_cache, val_prompt_cache, val_records, cfg.data.resolution)
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.train_batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.training.train_batch_size, shuffle=False, num_workers=0, pin_memory=True)

    unet = models["unet"]
    trainable_params = [p for p in unet.parameters() if p.requires_grad]

    optimizer, resolved_optimizer_name = build_optimizer(cfg, trainable_params, accelerator.device)

    from diffusers.optimization import get_scheduler

    lr_scheduler = get_scheduler(
        cfg.optimizer.lr_scheduler,
        optimizer=optimizer,
        num_warmup_steps=cfg.optimizer.lr_warmup_steps,
        num_training_steps=cfg.training.max_train_steps,
    )

    unet, optimizer, train_loader, val_loader, lr_scheduler = accelerator.prepare(
        unet, optimizer, train_loader, val_loader, lr_scheduler
    )

    # Accelerate's save_state serializes every prepared model IN FULL. For a LoRA run that means
    # ~9.5 GB of FROZEN SDXL UNet base weights written into every checkpoint alongside ~11 MB of
    # actually-trainable LoRA weights — 99.9% of each checkpoint duplicating weights already pinned
    # immutably by model.revision in the HF cache. At save_every_n_steps=1000 over 30k steps that is
    # ~272 GB written and ~27 GB resident (keep_last_n_full_checkpoints=3), plus the GPU sitting idle
    # through each multi-minute serialization.
    #
    # These hooks are the diffusers text_to_image_lora_sdxl reference pattern this script is adapted
    # from (§docstring); they were the one piece of it that was missing. `weights.pop()` is what
    # stops Accelerate writing the full model, and the LoRA adapter is written instead, so the
    # checkpoint stays fully resumable. Optimizer/scheduler/RNG/EMA state are untouched by this and
    # are still saved by Accelerate as before.
    def save_model_hook(models, weights, output_dir):
        if not accelerator.is_main_process:
            return
        from diffusers import StableDiffusionXLPipeline
        from diffusers.utils import convert_state_dict_to_diffusers
        from peft.utils import get_peft_model_state_dict

        unet_lora_layers = None
        unet_type = type(accelerator.unwrap_model(unet))
        for model in models:
            if not isinstance(accelerator.unwrap_model(model), unet_type):
                raise ValueError(f"save_model_hook got an unexpected model: {model.__class__.__name__}")
            unet_lora_layers = convert_state_dict_to_diffusers(get_peft_model_state_dict(model))
            # Drop the full-model weights Accelerate queued for this model.
            weights.pop()
        if unet_lora_layers is None:
            raise ValueError("save_model_hook received no UNet; refusing to write a checkpoint with no LoRA weights")
        StableDiffusionXLPipeline.save_lora_weights(
            str(output_dir), unet_lora_layers=unet_lora_layers, safe_serialization=True
        )

    def load_model_hook(models, input_dir):
        from diffusers import StableDiffusionXLPipeline
        from diffusers.utils import convert_unet_state_dict_to_peft
        from peft import set_peft_model_state_dict

        unet_type = type(accelerator.unwrap_model(unet))
        target = None
        while models:
            model = models.pop()
            if not isinstance(accelerator.unwrap_model(model), unet_type):
                raise ValueError(f"load_model_hook got an unexpected model: {model.__class__.__name__}")
            target = model
        if target is None:
            raise ValueError("load_model_hook received no UNet to load LoRA weights into")

        lora_state_dict, _ = StableDiffusionXLPipeline.lora_state_dict(str(input_dir))
        unet_state_dict = {
            key.removeprefix("unet."): value
            for key, value in lora_state_dict.items()
            if key.startswith("unet.")
        }
        if not unet_state_dict:
            raise ValueError(f"No UNet LoRA weights found in {input_dir}; cannot resume")
        incompatible = set_peft_model_state_dict(
            target, convert_unet_state_dict_to_peft(unet_state_dict), adapter_name="default"
        )
        unexpected = getattr(incompatible, "unexpected_keys", None)
        if unexpected:
            raise ValueError(f"Unexpected LoRA keys while resuming from {input_dir}: {unexpected}")

    accelerator.register_save_state_pre_hook(save_model_hook)
    accelerator.register_load_state_pre_hook(load_model_hook)

    ema = (
        LoraEMA(accelerator.unwrap_model(unet), cfg.training.ema.decay, warmup=bool(cfg.training.ema.warmup))
        if cfg.training.ema.enabled else None
    )
    if ema is not None:
        accelerator.register_for_checkpointing(ema)

    global_step = 0
    if args.resume_from:
        resumed_metadata = read_json(Path(args.resume_from) / "metadata.json")
        for key, value in stage1_provenance.items():
            if resumed_metadata.get(key) != value:
                raise ValueError(f"Checkpoint {key} differs from current data ({resumed_metadata.get(key)!r} != {value!r}); refusing unsafe resume")
        accelerator.load_state(args.resume_from)
        global_step = resumed_metadata["step"]
        print(f"Resumed from {args.resume_from} at step {global_step}")

    noise_scheduler = models["noise_scheduler"]
    progress = tqdm(total=cfg.training.max_train_steps, initial=global_step, disable=not accelerator.is_local_main_process)
    epoch = global_step // max(1, len(train_loader))

    unet.train()
    while global_step < cfg.training.max_train_steps:
        train_dataset.set_epoch(epoch)
        for batch in train_loader:
            with accelerator.accumulate(unet):
                latents = batch["latents"].to(accelerator.device, weight_dtype)
                noise = torch.randn_like(latents)
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device
                ).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                added_cond_kwargs = {
                    "text_embeds": batch["pooled_embeds"].to(accelerator.device, weight_dtype),
                    "time_ids": batch["time_ids"].to(accelerator.device, weight_dtype),
                }
                model_pred = unet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states=batch["prompt_embeds"].to(accelerator.device, weight_dtype),
                    added_cond_kwargs=added_cond_kwargs,
                    return_dict=False,
                )[0]
                target = (
                    noise if noise_scheduler.config.prediction_type == "epsilon"
                    else noise_scheduler.get_velocity(latents, noise, timesteps)
                )
                loss = F.mse_loss(model_pred.float(), target.float())
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(trainable_params, cfg.optimizer.max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1
                progress.update(1)
                progress.set_postfix(loss=f"{loss.item():.4f}")
                if ema is not None:
                    ema.update(accelerator.unwrap_model(unet))

                if trackers:
                    accelerator.log({"train/loss": loss.item(), "train/lr": lr_scheduler.get_last_lr()[0]}, step=global_step)

                if global_step % cfg.checkpointing.save_every_n_steps == 0:
                    save_checkpoint(
                        accelerator, unet, ema, cfg, run_dir, global_step, epoch,
                        split_manifest_hash, cfg.run.seed, cfg.checkpointing.keep_last_n_full_checkpoints,
                        resolved_optimizer_name,
                    )

                if global_step % cfg.validation.val_every_n_steps == 0:
                    val_loss = run_validation(accelerator, unet, val_loader, noise_scheduler, weight_dtype, int(cfg.validation.val_seed))
                    if trackers:
                        accelerator.log({"val/loss": val_loss}, step=global_step)
                    print(f"[step {global_step}] val_loss={val_loss:.4f}")
                    unet.train()

            if global_step >= cfg.training.max_train_steps:
                break
        epoch += 1

    progress.close()
    save_checkpoint(
        accelerator, unet, ema, cfg, run_dir, global_step, epoch,
        split_manifest_hash, cfg.run.seed, cfg.checkpointing.keep_last_n_full_checkpoints,
        resolved_optimizer_name,
    )
    final_dir = run_dir / "final"
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft.utils import get_peft_model_state_dict
    from diffusers import StableDiffusionXLPipeline

    if accelerator.is_main_process:
        raw_unet = accelerator.unwrap_model(unet)
        if ema is not None:
            ema.copy_to(raw_unet)
        state = convert_state_dict_to_diffusers(get_peft_model_state_dict(raw_unet))
        StableDiffusionXLPipeline.save_lora_weights(str(final_dir), unet_lora_layers=state, safe_serialization=True)
        write_json(
            final_dir / "metadata.json",
            build_checkpoint_metadata(global_step, epoch, dict(cfg), split_manifest_hash, cfg.run.seed,
                                      extra={"ema_applied": ema is not None, **stage1_provenance}),
        )
        print(f"Final LoRA weights (EMA applied: {ema is not None}) -> {final_dir}")

    if trackers:
        accelerator.end_training()

    return 0


@torch.no_grad()
def run_validation(accelerator, unet, val_loader, noise_scheduler, weight_dtype, seed: int) -> float:
    """Mean noise-prediction loss over val_loader, identical draws on every call.

    The latent samples (drawn in CachedSDXLDataset), noise and timesteps all come from the global
    RNGs, so each pass reseeds them inside fork_rng: repeated passes see the same draws, and the
    training RNG stream resumes exactly where it was, as if validation had never run.
    """
    device = accelerator.device
    fork_devices = [device] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=fork_devices), torch.no_grad():
        torch.manual_seed(seed)
        return _validation_loss(accelerator, unet, val_loader, noise_scheduler, weight_dtype)


def _validation_loss(accelerator, unet, val_loader, noise_scheduler, weight_dtype) -> float:
    unet.eval()
    total_loss = 0.0
    n_batches = 0
    for batch in val_loader:
        latents = batch["latents"].to(accelerator.device, weight_dtype)
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        added_cond_kwargs = {
            "text_embeds": batch["pooled_embeds"].to(accelerator.device, weight_dtype),
            "time_ids": batch["time_ids"].to(accelerator.device, weight_dtype),
        }
        model_pred = unet(
            noisy_latents, timesteps, encoder_hidden_states=batch["prompt_embeds"].to(accelerator.device, weight_dtype),
            added_cond_kwargs=added_cond_kwargs, return_dict=False,
        )[0]
        target = noise if noise_scheduler.config.prediction_type == "epsilon" else noise_scheduler.get_velocity(latents, noise, timesteps)
        loss = F.mse_loss(model_pred.float(), target.float())
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/train/launch_resumable.sh`

*54 lines*

**Stage 1 launcher, built for RunPod reality.** Reads
`checkpoints/stage1_lora_sdxl/latest_run.json` for an existing `run_id` and that run's
`latest.json` for the newest full checkpoint; resumes from it if present, otherwise starts fresh.
Runs under `nohup` so a dropped SSH session does not kill training, appending to a log on the
persistent volume.

**Note:** it *backgrounds* training and returns immediately - which is why `run_dev_subset.sh`
deliberately does **not** use it, and calls `accelerate launch` in the foreground instead so the
pipeline waits for Stage 1 to finish before Stage 2 starts.

```bash
#!/usr/bin/env bash
# Launch (or resume) Stage 1 SDXL LoRA training, resilient to RunPod pod restarts and SSH
# disconnects (docs/stage1_plan.md §10).
#
# On start: checks checkpoints/stage1_lora_sdxl/latest_run.json for an existing run_id, and that
# run's latest.json for the newest full-resumable checkpoint. If found, resumes from it;
# otherwise starts a fresh run. Runs under nohup so a dropped VS Code SSH session doesn't kill
# training; all output is appended to a log file on the persistent volume.
#
# Usage:
#   bash scripts/train/launch_resumable.sh [extra overrides passed through to train_lora_sdxl.py]
#
# Recommended before trusting a long run: kill this process deliberately once, then re-run this
# script and confirm it resumes from the same step (docs/stage1_plan.md §10).

set -euo pipefail

REPO_ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")/../.." && pwd)"
: "${PROJECT_ROOT:=$REPO_ROOT}"
export PROJECT_ROOT
export PYTHONUNBUFFERED=1

CHECKPOINTS_DIR="${PROJECT_ROOT}/checkpoints/stage1_lora_sdxl"
LOGS_DIR="${PROJECT_ROOT}/logs/stage1_lora_sdxl"
mkdir -p "$CHECKPOINTS_DIR" "$LOGS_DIR"

LATEST_RUN_FILE="${CHECKPOINTS_DIR}/latest_run.json"
RESUME_ARGS=()

if [[ -f "$LATEST_RUN_FILE" ]]; then
  RUN_ID=$(python3 -u -c "import json; print(json.load(open('${LATEST_RUN_FILE}'))['run_id'])")
  RUN_LATEST_JSON="${CHECKPOINTS_DIR}/${RUN_ID}/latest.json"
  if [[ -f "$RUN_LATEST_JSON" ]]; then
    CHECKPOINT_DIR=$(python3 -u -c "import json; print(json.load(open('${RUN_LATEST_JSON}'))['checkpoint_dir'])")
    echo "Found existing run ${RUN_ID}; resuming from ${CHECKPOINT_DIR}"
    RESUME_ARGS=(--resume-from "$CHECKPOINT_DIR" --run-id "$RUN_ID")
  else
    echo "Found run ${RUN_ID} but no checkpoint yet; continuing to write into it."
    RESUME_ARGS=(--run-id "$RUN_ID")
  fi
else
  echo "No prior run found; starting fresh."
fi

LOG_FILE="${LOGS_DIR}/train_$(date -u +%Y%m%dT%H%M%SZ).log"
echo "Logging to ${LOG_FILE}"

cd "$PROJECT_ROOT"
nohup accelerate launch --config_file configs/accelerate_config.yaml \
  scripts/train/train_lora_sdxl.py "${RESUME_ARGS[@]}" "$@" \
  >> "$LOG_FILE" 2>&1 &

echo "Training launched in background, PID $!. Tail progress with:"
echo "  tail -f ${LOG_FILE}"

```

## `scripts/eval/generate_probe_samples.py`

*121 lines*

**Stage 1 monitoring.** Generates a fixed-seed probe grid from a LoRA checkpoint: one prompt per
pathology positive, one no-finding prompt, and a few multi-label combinations
(`MULTI_LABEL_COMBOS`), regenerated with the **same seeds at every checkpoint** so the grids are
directly comparable checkpoint-to-checkpoint.

Prompts are built with `scripts/utils/caption_builder.py` - the same module used at training time -
so probe conditioning matches train-time phrasing exactly. This is a qualitative plausibility and
informal directional-conditioning check, not a metric.

In [ ]:
# ===== FILE: scripts/eval/generate_probe_samples.py =====
#!/usr/bin/env python3
"""Generate a fixed-seed probe-prompt sample grid from a Stage 1 LoRA checkpoint
(docs/stage1_plan.md §9 — qualitative plausibility + informal directional-conditioning check).

The probe prompt set is fixed and regenerated with the same seeds at every checkpoint, so samples
are directly comparable checkpoint-to-checkpoint: one prompt per pathology positive, one no-finding
prompt, and a few multi-label combinations. Prompts are built with scripts/utils/caption_builder.py
— the same module used at training time — so probe conditioning matches train-time phrasing exactly.

Usage:
    python scripts/eval/generate_probe_samples.py --checkpoint checkpoints/stage1_lora_sdxl/<run_id>/final
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, PATHOLOGY_COLUMNS, build_caption  # noqa: E402
from scripts.utils.config import load_stage1_config  # noqa: E402
from scripts.utils.manifest import write_json  # noqa: E402

DEVICE_COLUMN = "Support Devices"
NO_FINDING_COLUMN = "No Finding"

MULTI_LABEL_COMBOS = [
    ["Cardiomegaly", "Pleural Effusion"],
    ["Atelectasis", "Consolidation"],
    ["Edema", "Pneumonia"],
]


def build_probe_rows(caption_config: CaptionConfig) -> list[dict]:
    """One synthetic label row per single-pathology positive, one No Finding, a few multi-label
    combos. Age/sex are fixed to a neutral reference point so only the finding clause varies."""
    base_row = {"Sex": "Male", "Age": 50, "Frontal/Lateral": "Frontal", "AP/PA": "AP"}
    rows = []

    no_finding_row = dict(base_row, **{NO_FINDING_COLUMN: 1})
    rows.append({"name": "no_finding", "row": no_finding_row})

    for col in PATHOLOGY_COLUMNS:
        if col in (NO_FINDING_COLUMN, DEVICE_COLUMN):
            continue
        row = dict(base_row, **{c: 0 for c in PATHOLOGY_COLUMNS})
        row[col] = 1
        rows.append({"name": col.replace(" ", "_").lower(), "row": row})

    for combo in MULTI_LABEL_COMBOS:
        row = dict(base_row, **{c: 0 for c in PATHOLOGY_COLUMNS})
        for col in combo:
            row[col] = 1
        rows.append({"name": "_".join(c.replace(" ", "").lower() for c in combo), "row": row})

    return rows


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--checkpoint", type=str, required=True, help="LoRA weights dir (e.g. .../lora_weights/step_1000 or .../final)")
    parser.add_argument("--steps", type=int, default=30)
    parser.add_argument("--guidance-scale", type=float, default=5.0)
    args = parser.parse_args()

    cfg = load_stage1_config()
    caption_config = CaptionConfig(
        age_bucket_width_years=cfg.captions.age_bucket_width_years,
        uncertain_label_policy=cfg.captions.uncertain_label_policy,
        no_finding_overrides_positives=cfg.captions.no_finding_overrides_positives,
        num_paraphrase_variants=cfg.captions.num_paraphrase_variants,
        template_version=cfg.captions.template_version,
    )

    checkpoint_path = Path(args.checkpoint)
    # e.g. ".../lora_weights/step_1000" -> "lora_weights_step_1000", ".../final" -> "final"
    checkpoint_name = "_".join(checkpoint_path.parts[-2:]) if checkpoint_path.parent.name == "lora_weights" else checkpoint_path.name
    out_dir = Path(cfg.paths.outputs_dir) / checkpoint_name
    out_dir.mkdir(parents=True, exist_ok=True)

    from diffusers import StableDiffusionXLPipeline

    pipeline = StableDiffusionXLPipeline.from_pretrained(
        cfg.model.base_model_id, torch_dtype=torch.bfloat16, cache_dir=cfg.paths.hf_cache_dir
    )
    pipeline.load_lora_weights(str(checkpoint_path))
    pipeline.to("cuda" if torch.cuda.is_available() else "cpu")
    pipeline.set_progress_bar_config(disable=False)

    negative_prompt = "text, watermark, labels, color image, cropped anatomy, deformed chest, low quality"
    probe_rows = build_probe_rows(caption_config)

    records = []
    for i, probe in enumerate(probe_rows):
        prompt = build_caption(probe["row"], caption_config, variant_index=0)
        seed = cfg.validation.probe_seed + i
        generator = torch.Generator(device=pipeline.device).manual_seed(seed)
        image = pipeline(
            prompt=prompt,
            negative_prompt=negative_prompt,
            height=cfg.data.resolution,
            width=cfg.data.resolution,
            num_inference_steps=args.steps,
            guidance_scale=args.guidance_scale,
            generator=generator,
        ).images[0]
        image_path = out_dir / f"{probe['name']}.png"
        image.save(image_path)
        records.append({"name": probe["name"], "prompt": prompt, "seed": seed, "image_path": str(image_path)})
        print(f"[{i + 1}/{len(probe_rows)}] {probe['name']}: {prompt}")

    write_json(out_dir / "probe_manifest.json", {"checkpoint": str(checkpoint_path), "records": records})
    print(f"Wrote {len(records)} probe samples -> {out_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/eval/compute_fid_clipscore.py`

*145 lines*

**Stage 1 monitoring - quantitative trend.** Computes **two** FID variants deliberately:

- **Inception FID** (`compute_inception_fid`) - field-standard and comparable to other work, but a
  poor fit for grayscale medical images, since Inception features are ImageNet-domain.
- **Domain FID** (`compute_domain_fid`) - `torchxrayvision` DenseNet121 features. The more
  literature-appropriate signal for this project, because standard FID does not reflect diagnostic
  content.

Plus CLIP score for caption-image agreement. **Both FIDs are tracked only as relative trends across
checkpoints, never as absolute quality bars.** The real reference set is `gen_val` - held out from
training but already touched by Stage 1; `classifier_heldout` must never be read here.

In [ ]:
# ===== FILE: scripts/eval/compute_fid_clipscore.py =====
#!/usr/bin/env python3
"""FID and CLIP-score trend for a generated probe sample set (docs/stage1_plan.md §9).

Two FID variants are computed:
  - Standard Inception-based FID (torchmetrics) — field-standard, comparable to other work, but a
    poor fit for grayscale medical images (Inception features are ImageNet-domain).
  - Domain FID using torchxrayvision's DenseNet121 features — the more literature-appropriate
    signal for this project (docs/stage1_plan.md: "FID doesn't reflect diagnostic content").

Both are tracked ONLY as relative trends across checkpoints, not absolute quality bars (plan §9).
The real reference set is gen_val (held out from training but touched by Stage 1) —
classifier_heldout must never be read here, since it is reserved untouched for Stage 4/5.

Usage:
    python scripts/eval/compute_fid_clipscore.py --generated-dir outputs/stage1_samples/<checkpoint_name>
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_stage1_config  # noqa: E402
from scripts.utils.manifest import read_json, write_json  # noqa: E402


def load_images_as_uint8_tensor(paths: list[Path], size: int) -> torch.Tensor:
    from torchvision.transforms import functional as TF

    tensors = []
    for p in paths:
        with Image.open(p) as im:
            im = im.convert("RGB").resize((size, size))
            tensors.append(TF.pil_to_tensor(im))
    return torch.stack(tensors)


def frechet_distance(mu1: np.ndarray, sigma1: np.ndarray, mu2: np.ndarray, sigma2: np.ndarray) -> float:
    from scipy import linalg

    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))


def compute_domain_fid(real_paths: list[Path], generated_paths: list[Path], device: torch.device) -> float:
    import torchxrayvision as xrv
    import torchvision.transforms as T

    model = xrv.models.DenseNet(weights="densenet121-res224-all").to(device).eval()
    transform = T.Compose([T.Resize((224, 224))])

    def extract_features(paths: list[Path]) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for p in paths:
                with Image.open(p) as im:
                    arr = np.array(im.convert("L"), dtype=np.float32)
                arr = xrv.datasets.normalize(arr, 255)
                tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)
                tensor = transform(tensor)
                features = model.features(tensor)
                feats.append(features.mean(dim=[2, 3]).squeeze(0).cpu().numpy())
        return np.stack(feats)

    real_feats = extract_features(real_paths)
    gen_feats = extract_features(generated_paths)

    mu1, sigma1 = real_feats.mean(axis=0), np.cov(real_feats, rowvar=False)
    mu2, sigma2 = gen_feats.mean(axis=0), np.cov(gen_feats, rowvar=False)
    return frechet_distance(mu1, sigma1, mu2, sigma2)


def compute_inception_fid(real_paths: list[Path], generated_paths: list[Path], device: torch.device) -> float:
    from torchmetrics.image.fid import FrechetInceptionDistance

    fid = FrechetInceptionDistance(normalize=False).to(device)
    fid.update(load_images_as_uint8_tensor(real_paths, 299).to(device), real=True)
    fid.update(load_images_as_uint8_tensor(generated_paths, 299).to(device), real=False)
    return float(fid.compute().item())


def compute_clip_score(generated_paths: list[Path], prompts: list[str], device: torch.device) -> float:
    from torchmetrics.multimodal.clip_score import CLIPScore

    metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch32").to(device)
    images = load_images_as_uint8_tensor(generated_paths, 224).to(device)
    score = metric(images, prompts)
    return float(score.item())


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--generated-dir", type=str, required=True, help="Directory with probe_manifest.json (from generate_probe_samples.py)")
    parser.add_argument("--num-real-reference", type=int, default=200)
    args = parser.parse_args()

    cfg = load_stage1_config()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    generated_dir = Path(args.generated_dir)
    manifest = read_json(generated_dir / "probe_manifest.json")
    generated_paths = [Path(r["image_path"]) for r in manifest["records"]]
    prompts = [r["prompt"] for r in manifest["records"]]

    val_images_dir = Path(cfg.paths.images_dir) / "gen_val"
    real_paths = sorted(val_images_dir.glob("*.jpg"))[: args.num_real_reference]
    if not real_paths:
        print(f"No real reference images found under {val_images_dir} — run the preprocessing pipeline first.")
        return 1

    results = {"generated_dir": str(generated_dir), "num_real_reference": len(real_paths)}

    if cfg.validation.clip_score:
        results["clip_score"] = compute_clip_score(generated_paths, prompts, device)
        print(f"CLIP score: {results['clip_score']:.4f}")

    if cfg.validation.fid.inception:
        try:
            results["fid_inception"] = compute_inception_fid(real_paths, generated_paths, device)
            print(f"FID (Inception): {results['fid_inception']:.2f}")
        except Exception as e:
            print(f"Inception FID failed ({e}); skipping.")

    if cfg.validation.fid.domain_torchxrayvision:
        try:
            results["fid_domain_torchxrayvision"] = compute_domain_fid(real_paths, generated_paths, device)
            print(f"FID (domain, torchxrayvision): {results['fid_domain_torchxrayvision']:.2f}")
        except Exception as e:
            print(f"Domain FID failed ({e}); skipping.")

    write_json(generated_dir / "fid_clipscore_results.json", results)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 5 - Stage 2: Synthetic generation

Decide what to generate (auditable label recipes), then generate it - behind a human pilot approval gate, with every image's intended label vector recorded.

## `scripts/generate/01_sample_label_recipes.py`

*374 lines*

**Stage 2a - decide *what* to generate, before generating anything.** The output is an auditable
recipe table, not images.

- `mine_cooccurrence` - counts how many distinct real **patients** exhibit each positive-label
  combination. Patient-level, so one patient with many studies cannot manufacture apparent support
  for a combination.
- `compute_label_quotas` - **rarity-aware** per-class quotas from real patient-level support, so
  rare findings are not drowned out by common ones.
- `blocked_by_medical_rule` - combinations forbidden by the configured medical block rules are
  never emitted.
- `build_recipes` - samples single- and multi-label compositions under the quotas, the
  co-occurrence support constraint and a cap on positives per recipe; attaches demographics
  (age bucket, sex, view) to each recipe.

Recipe eligibility keys off `GENERATION_TARGET_LABELS` (12 labels) - **not** the 11 primary
endpoint labels. Writes `label_recipes.csv`, `recipe_decisions.jsonl` (why each candidate was
accepted or rejected) and `recipes_manifest.json`, all atomically and all hashed.

In [ ]:
# ===== FILE: scripts/generate/01_sample_label_recipes.py =====
from __future__ import annotations

import argparse
import json
import os
import random
import sys
import tempfile
from collections import Counter
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import (  # noqa: E402
    GENERATION_TARGET_LABELS,
    normalize_label,
    patient_level_support,
)
from scripts.utils.artifact_contracts import config_sha256, current_code_identity_hash, namespace_identity, stage2_paths  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import resolve_splits_dir  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402



def load_stage2_config():
    return load_named_config("stage2_generation.yaml", "stage2")


def mine_cooccurrence(frame: pd.DataFrame, min_support_patients: int) -> dict[frozenset, int]:
    """Count how many distinct real PATIENTS exhibit each positive-label combination.

    Patient-level (not image-level) so a single patient with many studies cannot manufacture
    apparent support for a combination on their own.
    """
    combo_patients: dict[frozenset, set[str]] = {}

    for row in frame.to_dict("records"):
        positives = frozenset(
            label
            for label in GENERATION_TARGET_LABELS
            if normalize_label(row.get(label)) == 1
        )
        if not positives:
            continue
        combo_patients.setdefault(positives, set()).add(row["patient_id"])

    return {
        combo: len(patients)
        for combo, patients in combo_patients.items()
        if len(patients) >= min_support_patients
    }


def compute_label_quotas(support: pd.DataFrame, recipe_cfg) -> dict[str, int]:
    """Per-label target counts, oversampling rare labels relative to real prevalence (§3.7).

    target = base_quota * (median_prevalence / prevalence) ** rarity_exponent, clipped.
    rarity_exponent=0 reduces this to a flat base_quota for every label.
    """
    prevalence = {}
    for row in support.to_dict("records"):
        total = max(row["positive_patients"] + row["negative_patients"], 1)
        prevalence[row["label"]] = max(row["positive_patients"] / total, 1e-6)

    values = sorted(prevalence.values())
    median = values[len(values) // 2] if values else 1.0

    quotas = {}
    for label, rate in prevalence.items():
        scale = (median / rate) ** float(recipe_cfg.rarity_exponent)
        target = int(round(float(recipe_cfg.base_quota) * scale))
        quotas[label] = int(
            min(max(target, int(recipe_cfg.min_per_label)), int(recipe_cfg.max_per_label))
        )
    return quotas


def blocked_by_medical_rule(combo: frozenset, block_rules: list[list[str]]) -> str | None:
    """A block rule fires when the recipe contains ALL labels in the rule — i.e. the rule describes
    a contradictory co-presentation, not merely one suspicious label."""
    for rule in block_rules:
        rule_set = frozenset(rule)
        if rule_set and rule_set.issubset(combo):
            return f"medical_rule_block:{sorted(rule_set)}"
    return None


def build_recipes(
    frame: pd.DataFrame,
    config,
    seed: int,
) -> tuple[list[dict], list[dict], dict]:
    """Return (accepted_recipes, decisions, stats). `decisions` records every candidate considered
    with an explicit accept/reject reason."""
    recipe_cfg = config.recipes
    rng = random.Random(seed)

    support = patient_level_support(frame, GENERATION_TARGET_LABELS)
    quotas = compute_label_quotas(support, recipe_cfg)
    cooccurrence = mine_cooccurrence(frame, int(recipe_cfg.min_support_patients))

    allow_rules = [frozenset(rule) for rule in recipe_cfg.medical_rule_allow]
    block_rules = [list(rule) for rule in recipe_cfg.medical_rule_block]
    max_positive = int(recipe_cfg.max_positive_labels)

    # Candidate pool: mined combinations within the size cap, plus every single label (so a rare
    # label is never unreachable just because it rarely co-occurs), plus explicit allow-rules.
    candidates: dict[frozenset, str] = {}
    for combo in cooccurrence:
        if len(combo) <= max_positive:
            candidates[combo] = "empirical_cooccurrence"
    for label in GENERATION_TARGET_LABELS:
        candidates.setdefault(frozenset({label}), "single_label_guaranteed")
    for rule in allow_rules:
        if rule:
            candidates[rule] = "medical_rule_allow"

    decisions: list[dict] = []
    accepted_combos: list[tuple[frozenset, str]] = []

    for combo, origin in sorted(candidates.items(), key=lambda item: (len(item[0]), sorted(item[0]))):
        block_reason = blocked_by_medical_rule(combo, block_rules)
        if block_reason and origin != "medical_rule_allow":
            decisions.append(
                {
                    "combination": sorted(combo),
                    "origin": origin,
                    "accepted": False,
                    "reason": block_reason,
                    "support_patients": cooccurrence.get(combo, 0),
                }
            )
            continue
        if len(combo) > max_positive:
            decisions.append(
                {
                    "combination": sorted(combo),
                    "origin": origin,
                    "accepted": False,
                    "reason": f"exceeds_max_positive_labels:{max_positive}",
                    "support_patients": cooccurrence.get(combo, 0),
                }
            )
            continue

        support_patients = cooccurrence.get(combo, 0)
        if origin == "single_label_guaranteed" and support_patients == 0:
            reason = "accepted_single_label_below_support_but_required_for_coverage"
        elif origin == "medical_rule_allow":
            reason = "accepted_medical_rule_allow"
        else:
            reason = "accepted_empirical_support"

        decisions.append(
            {
                "combination": sorted(combo),
                "origin": origin,
                "accepted": True,
                "reason": reason,
                "support_patients": support_patients,
            }
        )
        accepted_combos.append((combo, reason))

    if not accepted_combos:
        raise SystemExit(
            "No label combinations survived recipe validation — check recipes.min_support_patients "
            "and medical_rule_block in configs/stage2_generation.yaml."
        )

    singles = [c for c, _ in accepted_combos if len(c) == 1]
    multis = [c for c, _ in accepted_combos if len(c) > 1]

    total_disease_target = sum(quotas.values())
    no_finding_share = float(recipe_cfg.no_finding_proportion)
    total_target = int(round(total_disease_target / max(1.0 - no_finding_share, 1e-6)))
    no_finding_target = total_target - total_disease_target

    age_buckets = list(recipe_cfg.age_buckets)
    sexes = list(recipe_cfg.sexes)
    device_rate = float(recipe_cfg.support_devices_proportion)
    single_share = float(recipe_cfg.single_label_proportion)

    recipes: list[dict] = []
    remaining = dict(quotas)
    index = 0

    def emit(combo: frozenset) -> None:
        nonlocal index
        age = rng.choice(age_buckets)
        sex = rng.choice(sexes)
        has_device = rng.random() < device_rate
        intended = {label: (1 if label in combo else 0) for label in GENERATION_TARGET_LABELS}
        recipe_id = f"recipe_{index:07d}"
        recipes.append(
            {
                "recipe_id": recipe_id,
                "intended_label_vector": json.dumps(intended, sort_keys=True),
                "positive_labels": json.dumps(sorted(combo)),
                "num_positive_labels": len(combo),
                "is_no_finding": len(combo) == 0,
                "age_bucket_start": age,
                "sex": sex,
                "support_devices": int(has_device),
                "view": "Frontal",
            }
        )
        index += 1

    # Disease recipes: draw until every label's quota is met, respecting the single/multi mix.
    guard = 0
    max_iterations = total_disease_target * 20 + 1000
    while any(count > 0 for count in remaining.values()) and guard < max_iterations:
        guard += 1
        use_single = rng.random() < single_share or not multis
        pool = singles if use_single else multis
        # Prefer combinations that still serve an unmet quota.
        hungry = [c for c in pool if any(remaining.get(label, 0) > 0 for label in c)]
        combo = rng.choice(hungry) if hungry else rng.choice(pool)
        emit(combo)
        for label in combo:
            if label in remaining:
                remaining[label] = max(0, remaining[label] - 1)

    for _ in range(max(0, no_finding_target)):
        emit(frozenset())

    rng.shuffle(recipes)
    for position, recipe in enumerate(recipes):
        recipe["recipe_id"] = f"recipe_{position:07d}"

    stats = {
        "num_recipes": len(recipes),
        "num_disease_recipes": sum(1 for r in recipes if not r["is_no_finding"]),
        "num_no_finding_recipes": sum(1 for r in recipes if r["is_no_finding"]),
        "num_candidate_combinations": len(candidates),
        "num_accepted_combinations": len(accepted_combos),
        "num_rejected_combinations": sum(1 for d in decisions if not d["accepted"]),
        "per_label_quota": quotas,
        "per_label_emitted": {
            label: sum(
                1
                for r in recipes
                if json.loads(r["intended_label_vector"]).get(label) == 1
            )
            for label in GENERATION_TARGET_LABELS
        },
        "single_vs_multi": dict(
            Counter("single" if r["num_positive_labels"] == 1 else
                    ("no_finding" if r["is_no_finding"] else "multi") for r in recipes)
        ),
        "quota_satisfied": all(count == 0 for count in remaining.values()),
    }
    return recipes, decisions, stats


def atomic_write_jsonl(path: Path, records: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        frame.to_csv(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None, help="Override split_namespace from config")
    parser.add_argument("--limit", type=int, default=None, help="Cap recipe count (smoke testing)")
    args = parser.parse_args()

    config = load_stage2_config()
    namespace = args.namespace or str(config.split_namespace)

    provenance = namespace_identity(namespace)
    gen_train = load_split(
        "gen_train", namespace, purpose="schema_validation", caller="01_sample_label_recipes"
    )

    recipes, decisions, stats = build_recipes(gen_train, config, seed=int(config.generation.seed))

    if args.limit is not None and len(recipes) > args.limit:
        recipes = recipes[: args.limit]
        for position, recipe in enumerate(recipes):
            recipe["recipe_id"] = f"recipe_{position:07d}"
        # Recompute every count that describes the emitted set, so the manifest can never record
        # a mix that contradicts num_recipes.
        stats["limited_to"] = args.limit
        stats["quota_satisfied"] = False  # truncation invalidates the quota guarantee
        stats["num_recipes"] = len(recipes)
        stats["num_disease_recipes"] = sum(1 for r in recipes if not r["is_no_finding"])
        stats["num_no_finding_recipes"] = sum(1 for r in recipes if r["is_no_finding"])
        stats["per_label_emitted"] = {
            label: sum(
                1 for r in recipes if json.loads(r["intended_label_vector"]).get(label) == 1
            )
            for label in GENERATION_TARGET_LABELS
        }
        stats["single_vs_multi"] = dict(
            Counter(
                "single" if r["num_positive_labels"] == 1
                else ("no_finding" if r["is_no_finding"] else "multi")
                for r in recipes
            )
        )

    paths = stage2_paths(config, namespace)
    recipes_path = paths["recipes_path"]
    decisions_path = paths["recipe_decisions_path"]

    atomic_write_csv(recipes_path, pd.DataFrame(recipes))
    atomic_write_jsonl(decisions_path, decisions)

    recipe_config_block = {
        "recipes": OmegaConf.to_container(config.recipes, resolve=True),
        "split_provenance": provenance,
    }
    manifest = {
        "stage": "stage2_recipes",
        "recipes_path": str(recipes_path),
        "decisions_path": str(decisions_path),
        "schema_version": 2,
        "config_hash": hash_dict(recipe_config_block, length=64),
        "recipe_config_sha256": config_sha256(config.recipes),
        "git_commit_hash": get_git_commit_hash(),
        "code_identity_sha256": current_code_identity_hash(),
        "split_provenance": provenance,
        "split_namespace": namespace,
        "namespace_class": provenance["namespace_class"],
        "split_manifest_hash": provenance["split_manifest_hash"],
        "source_gen_train_csv_sha256": sha256_file(resolve_splits_dir(namespace) / "gen_train.csv"),
        "recipes_csv_sha256": sha256_file(recipes_path),
        "decisions_jsonl_sha256": sha256_file(decisions_path),
        "recipe_config": recipe_config_block["recipes"],
        "stats": stats,
    }
    write_json(paths["recipes_manifest"], manifest)

    print(f"Split namespace:    {namespace} (manifest {provenance['split_manifest_hash']})", flush=True)
    print(f"Candidate combos:   {stats['num_candidate_combinations']}", flush=True)
    print(f"  accepted:         {stats['num_accepted_combinations']}", flush=True)
    print(f"  rejected:         {stats['num_rejected_combinations']}", flush=True)
    print(f"Recipes emitted:    {stats['num_recipes']}", flush=True)
    print(f"  disease:          {stats['num_disease_recipes']}", flush=True)
    print(f"  no-finding:       {stats['num_no_finding_recipes']}", flush=True)
    print(f"  mix:              {stats['single_vs_multi']}", flush=True)
    print(f"  quotas satisfied: {stats['quota_satisfied']}", flush=True)
    print(f"Wrote:              {recipes_path}", flush=True)
    print(f"Decisions log:      {decisions_path}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/generate/02_generate_synthetic_images.py`

*586 lines*

**Stage 2b - generate the images.** Loads base SDXL plus the **explicitly pinned** Stage 1 LoRA
weights and produces one image per recipe.

- Captions come from `scripts/utils/caption_builder.py` **verbatim** - the same module Stage 1
  trained with. `recipe_to_caption_row` adapts a recipe into the row shape that module expects.
- `derive_seed(base_seed, recipe_id)` - a per-image deterministic seed, so any single image can be
  regenerated exactly.
- **The pilot gate.** `--mode pilot` generates a handful of images; `run_pilot_checks` computes
  automated sanity statistics; a human then runs `--approve-pilot --reviewer ... --notes ...`, which
  writes a signed approval manifest. `enforce_pilot_gate` makes `--mode full` refuse to run without
  it. Generating 5,000 images from a broken adapter is exactly the failure this prevents.
- **Resumable and idempotent** on the same terms as preprocessing: an existing image is skipped
  only after being decoded and validated (`validate_generated_image`), and every write is atomic.
- `validate_generation_inputs` / `check_upstream_ready` gate on the recipe manifest, split
  provenance and LoRA checkpoint hash. On success it writes `generation_complete.json` - the
  completion record that every Stage 3 script checks.

Each manifest row carries the image's `intended_label_vector`, which is the ground truth ASISM's
agreement signal scores against.

In [ ]:
# ===== FILE: scripts/generate/02_generate_synthetic_images.py =====
#!/usr/bin/env python3
"""Stage 2b — generate synthetic CXRs from the recipe table (docs/stages2_to_5_plan.md §3).

Loads base SDXL + the explicitly-pinned Stage 1 LoRA weights and generates one image per recipe.
Captions come from scripts/utils/caption_builder.py VERBATIM — the same module Stage 1 trained
with, because any train/generation phrasing drift degrades conditioning fidelity
(docs/stage1_plan.md §11).

Resumable and idempotent on the same terms as 03_preprocess_images.py: an existing output is
skipped only after being decoded and validated (correct format, mode, and resolution), never on
mere existence; every write is atomic (temp file + os.replace) so an interrupted pod cannot leave a
half-written JPEG.

PILOT GATE (§3, FROZEN): `--mode full` refuses to start unless a pilot_approval_manifest.json
exists, records a completed pilot, records passing automatic checks, and carries an explicit human
approval record. Approval is a separate deliberate command; this script never self-approves.

Usage:
    python scripts/generate/02_generate_synthetic_images.py --mode pilot
    python scripts/generate/02_generate_synthetic_images.py --approve-pilot --reviewer "Walaa"
    python scripts/generate/02_generate_synthetic_images.py --mode full
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.caption_builder import CaptionConfig, build_caption  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.labels import GENERATION_TARGET_LABELS  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    ArtifactContractError, config_sha256, current_code_identity_hash, namespace_identity,
    require_manifest_fields, stage2_paths,
)
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, sha256_directory, sha256_file, write_json  # noqa: E402

SCHEMA_VERSION = 2


def load_stage2_config():
    return load_named_config("stage2_generation.yaml", "stage2")


def derive_seed(base_seed: int, recipe_id: str) -> int:
    """Deterministic per-image seed: same recipe id always regenerates the same image, so a resumed
    run reproduces exactly what an uninterrupted run would have produced."""
    digest = hashlib.sha256(f"{base_seed}:{recipe_id}".encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def recipe_to_caption_row(recipe: dict) -> dict:
    """Map a recipe row into the dict shape caption_builder expects.

    Support Devices is injected here as a CONTEXT attribute (it drives the device clause) while
    staying out of the primary disease vector — §3.4.
    """
    intended = json.loads(recipe["intended_label_vector"])
    row = {label: intended.get(label, 0) for label in GENERATION_TARGET_LABELS}
    row["No Finding"] = 1 if bool(recipe["is_no_finding"]) else 0
    row["Support Devices"] = int(recipe["support_devices"])
    row["Age"] = int(recipe["age_bucket_start"])
    row["Sex"] = str(recipe["sex"])
    row["Frontal/Lateral"] = str(recipe["view"])
    row["AP/PA"] = "AP"
    return row


def validate_generated_image(path: Path, resolution: int) -> tuple[bool, str | None]:
    """Fully decode and check an existing output. Existence alone is never sufficient."""
    if not path.is_file():
        return False, "missing"
    try:
        with Image.open(path) as image:
            if image.format != "JPEG":
                return False, f"format_{image.format or 'unknown'}"
            if image.mode != "RGB":
                return False, f"mode_{image.mode}"
            if image.size != (resolution, resolution):
                return False, f"size_{image.size[0]}x{image.size[1]}"
            image.load()
    except Exception as exc:
        return False, f"unreadable_{type(exc).__name__}"
    return True, None


def atomic_save_jpeg(image: Image.Image, destination: Path, quality: int = 95) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent)
    os.close(fd)
    temp_path = Path(temp_name)
    try:
        image.save(temp_path, format="JPEG", quality=quality)
        os.replace(temp_path, destination)
    finally:
        temp_path.unlink(missing_ok=True)


def check_upstream_ready(config) -> Path:
    """Validate Stage 2's real upstream gate (an explicit, existing Stage 1 LoRA checkpoint) using
    only config and the filesystem — before importing the GPU stack, so a missing checkpoint fails
    with a clear message rather than an unrelated ImportError."""
    lora_dir = config.checkpoint.lora_weights_dir
    if not lora_dir:
        raise SystemExit(
            "UPSTREAM GATE: checkpoint.lora_weights_dir is not set in configs/stage2_generation.yaml.\n"
            "Stage 2 requires an EXPLICIT Stage 1 LoRA checkpoint (never 'latest') so generation is "
            "reproducible — see docs/stages2_to_5_plan.md §3.\n"
            "Set it to e.g. checkpoints/stage1_lora_sdxl/<run_id>/lora_weights/step_20000"
        )
    lora_path = Path(lora_dir)
    if not lora_path.is_dir():
        raise SystemExit(
            f"UPSTREAM GATE: LoRA weights directory not found: {lora_path}\n"
            "Stage 1 must produce a checkpoint before Stage 2 can generate."
        )
    return lora_path


def validate_generation_inputs(config, namespace: str, paths: dict[str, Path]) -> dict:
    if not paths["recipes_path"].is_file():
        legacy = Path(config.paths.synthetic_root) / "label_recipes.csv"
        if legacy.is_file():
            raise ArtifactContractError(
                f"Refusing stale unnamespaced Stage 2 recipes at {legacy}. Regenerate recipes into {paths['root']} for split run {namespace!r}."
            )
        raise ArtifactContractError(f"Missing namespaced recipe table: {paths['recipes_path']}")
    identity = namespace_identity(namespace)
    expected = {
        "schema_version": 2,
        "split_namespace": namespace,
        "namespace_class": identity["namespace_class"],
        "split_manifest_hash": identity["split_manifest_hash"],
        "recipe_config_sha256": config_sha256(config.recipes),
        "recipes_csv_sha256": sha256_file(paths["recipes_path"]),
        "code_identity_sha256": current_code_identity_hash(),
    }
    recipe_manifest = require_manifest_fields(paths["recipes_manifest"], expected, "Stage 2 recipe")
    lora_path = check_upstream_ready(config)
    lora_hash = sha256_directory(lora_path)
    metadata_path = lora_path / "metadata.json"
    if not metadata_path.is_file():
        raise ArtifactContractError(f"LoRA checkpoint has no metadata.json: {lora_path}")
    metadata = read_json(metadata_path)
    for key, value in (("split_namespace", namespace), ("split_manifest_hash", identity["split_manifest_hash"])):
        if metadata.get(key) != value:
            raise ArtifactContractError(f"LoRA checkpoint {key} mismatch: expected {value!r}, got {metadata.get(key)!r}")
    return {**identity, "recipes_csv_sha256": expected["recipes_csv_sha256"],
            "recipes_manifest_sha256": sha256_file(paths["recipes_manifest"]),
            "lora_checkpoint_sha256": lora_hash, "lora_metadata_sha256": sha256_file(metadata_path),
            "generation_config_sha256": config_sha256(config.generation),
            "stage2_config_sha256": config_sha256(config), "code_identity_sha256": current_code_identity_hash()}


def build_pipeline(config, stage1_cfg):
    """Load SDXL + the pinned LoRA weights. Imported lazily so the pilot gate and manifest logic
    remain testable on a machine with no GPU stack installed."""
    lora_path = check_upstream_ready(config)

    try:
        import torch
        from diffusers import DPMSolverMultistepScheduler, StableDiffusionXLPipeline
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the generation stack is not installed here ({exc}).\n"
            "Stage 2 generation is GPU work — run it on the RunPod pod after:\n"
            "  pip install -r environment/requirements.txt"
        ) from exc

    # Keep every SDXL component and the injected LoRA adapter in the same dtype.
    # The pinned SDXL checkpoint/LoRA path is float16-compatible; forcing bfloat16 here can
    # leave projection weights in fp16 and fail at inference with Half != BFloat16.
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    pipeline = StableDiffusionXLPipeline.from_pretrained(
        config.checkpoint.base_model_id,
        revision=config.checkpoint.revision,
        torch_dtype=dtype,
        cache_dir=str(stage1_cfg.paths.hf_cache_dir),
    )
    pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
    pipeline.load_lora_weights(str(lora_path))
    # Cast on CPU too: from_pretrained(torch_dtype=float32) leaves text_encoder_2 in float16 at the
    # pinned revision, which fails at inference with Half != float.
    pipeline = pipeline.to(device="cuda" if torch.cuda.is_available() else "cpu", dtype=dtype)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


def run_generation(
    recipes: pd.DataFrame,
    config,
    stage1_cfg,
    images_dir: Path,
    manifest_path: Path,
    mode: str,
    artifact_provenance: dict,
) -> dict:
    """Generate images for `recipes`, skipping already-valid outputs. Returns counters."""
    resolution = int(config.generation.resolution)
    caption_config = CaptionConfig(
        age_bucket_width_years=int(stage1_cfg.captions.age_bucket_width_years),
        uncertain_label_policy=str(stage1_cfg.captions.uncertain_label_policy),
        no_finding_overrides_positives=bool(stage1_cfg.captions.no_finding_overrides_positives),
        num_paraphrase_variants=int(stage1_cfg.captions.num_paraphrase_variants),
        template_version=str(stage1_cfg.captions.template_version),
    )

    # Determine what still needs generating BEFORE loading the model, so a fully-complete resume
    # costs nothing and never touches the GPU.
    existing_rows: dict[str, dict] = {}
    if manifest_path.is_file():
        for line_number, line in enumerate(manifest_path.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip():
                continue
            row = json.loads(line)
            image_id = str(row.get("image_id", ""))
            if not image_id or image_id in existing_rows:
                raise SystemExit(f"Generation manifest has a missing/duplicate image_id at line {line_number}: {manifest_path}")
            for key, expected in artifact_provenance.items():
                if row.get(key) != expected:
                    raise SystemExit(f"Generation manifest provenance mismatch for {image_id} at {key}; refusing stale reuse")
            existing_rows[image_id] = row

    pending = []
    retained_rows = []
    counts = {"generated": 0, "skipped_valid": 0, "failed": 0}
    for recipe in recipes.to_dict("records"):
        destination = images_dir / f"{recipe['recipe_id']}.jpg"
        valid, _ = validate_generated_image(destination, resolution)
        existing = existing_rows.get(str(recipe["recipe_id"]))
        if valid and existing is None:
            raise SystemExit(
                f"Unregistered generated image exists at {destination}; refusing to treat a stale/unpinned image as complete"
            )
        if valid:
            counts["skipped_valid"] += 1
            retained_rows.append(existing)
        else:
            pending.append(recipe)

    unknown = sorted(set(existing_rows) - set(recipes["recipe_id"].astype(str)))
    if unknown:
        raise SystemExit(f"Generation manifest contains {len(unknown)} IDs absent from the current recipe set: {unknown[:5]}")

    # Remove rows whose image failed validation before appending replacements. Publication is
    # atomic, so an interruption cannot create duplicate manifest rows on the next resume.
    if manifest_path.is_file() and len(retained_rows) != len(existing_rows):
        fd, temp_name = tempfile.mkstemp(prefix=f".{manifest_path.name}.", suffix=".tmp", dir=manifest_path.parent)
        os.close(fd)
        try:
            with open(temp_name, "w", encoding="utf-8") as handle:
                for row in retained_rows:
                    handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            os.replace(temp_name, manifest_path)
        finally:
            Path(temp_name).unlink(missing_ok=True)

    print(
        f"[{mode}] {counts['skipped_valid']} already valid, {len(pending)} to generate",
        flush=True,
    )
    if not pending:
        return counts

    pipeline = build_pipeline(config, stage1_cfg)
    import torch  # available by construction once build_pipeline() returned

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    generation_config_hash = hash_dict(OmegaConf.to_container(config.generation, resolve=True))
    base_seed = int(config.generation.seed)

    with open(manifest_path, "a", encoding="utf-8", buffering=1) as manifest_handle:
        progress = tqdm(pending, desc=f"[{mode}] generate", unit="image")
        for recipe in progress:
            recipe_id = recipe["recipe_id"]
            destination = images_dir / f"{recipe_id}.jpg"
            caption_row = recipe_to_caption_row(recipe)
            caption = build_caption(caption_row, caption_config, variant_index=0)
            seed = derive_seed(base_seed, recipe_id)

            try:
                generator = torch.Generator(
                    device="cuda" if torch.cuda.is_available() else "cpu"
                ).manual_seed(seed)
                result = pipeline(
                    prompt=caption,
                    negative_prompt=str(config.generation.negative_prompt) or None,
                    num_inference_steps=int(config.generation.num_inference_steps),
                    guidance_scale=float(config.generation.guidance_scale),
                    height=resolution,
                    width=resolution,
                    generator=generator,
                )
                image = result.images[0].convert("RGB")
                atomic_save_jpeg(image, destination)

                valid, reason = validate_generated_image(destination, resolution)
                if not valid:
                    counts["failed"] += 1
                    progress.set_postfix(counts, refresh=False)
                    continue

                counts["generated"] += 1
                manifest_handle.write(
                    json.dumps(
                        {
                            "image_id": recipe_id,
                            "schema_version": SCHEMA_VERSION,
                            "intended_label_vector": json.loads(recipe["intended_label_vector"]),
                            "is_no_finding": bool(recipe["is_no_finding"]),
                            "support_devices": int(recipe["support_devices"]),
                            "caption": caption,
                            "seed": seed,
                            "generation_config_hash": generation_config_hash,
                            "lora_weights_dir": str(config.checkpoint.lora_weights_dir),
                            **artifact_provenance,
                            "mode": mode,
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
            except Exception as exc:  # noqa: BLE001 - one bad recipe must not kill a long run
                counts["failed"] += 1
                print(f"[{mode}] FAILED {recipe_id}: {type(exc).__name__}: {exc}", flush=True)
            progress.set_postfix(counts, refresh=False)
        progress.close()

    return counts


def run_pilot_checks(images_dir: Path, recipe_ids: list[str], config) -> dict:
    """Automatic artifact checks the pilot must pass before it is eligible for approval."""
    checks_cfg = config.pilot.checks
    resolution = int(config.generation.resolution)

    total = len(recipe_ids)
    valid = 0
    wrong_resolution = 0
    not_rgb = 0
    near_uniform = 0
    missing = 0

    for recipe_id in recipe_ids:
        path = images_dir / f"{recipe_id}.jpg"
        if not path.is_file():
            missing += 1
            continue
        try:
            with Image.open(path) as image:
                if image.mode != "RGB":
                    not_rgb += 1
                if image.size != (resolution, resolution):
                    wrong_resolution += 1
                array = np.asarray(image.convert("L"), dtype=np.float32)
                if array.std() < float(checks_cfg.blank_std_threshold):
                    near_uniform += 1
                valid += 1
        except Exception:
            missing += 1

    valid_fraction = valid / total if total else 0.0
    near_uniform_fraction = near_uniform / total if total else 1.0

    results = {
        "total": total,
        "valid": valid,
        "missing_or_unreadable": missing,
        "wrong_resolution": wrong_resolution,
        "not_rgb": not_rgb,
        "near_uniform": near_uniform,
        "valid_fraction": round(valid_fraction, 4),
        "near_uniform_fraction": round(near_uniform_fraction, 4),
    }
    passed = (
        valid_fraction >= float(checks_cfg.min_valid_image_fraction)
        and near_uniform_fraction <= float(checks_cfg.max_near_uniform_fraction)
        and (wrong_resolution == 0 if bool(checks_cfg.require_correct_resolution) else True)
        and (not_rgb == 0 if bool(checks_cfg.require_rgb) else True)
    )
    results["passed"] = passed
    results["thresholds"] = OmegaConf.to_container(checks_cfg, resolve=True)
    return results


def approve_pilot(config, paths: dict[str, Path], provenance: dict, reviewer: str, notes: str) -> int:
    """Record explicit human approval. Refuses if the pilot did not run or its checks did not pass.

    This is deliberately a separate command a person runs after looking at the images — the
    generation script never approves its own output.
    """
    approval_path = paths["pilot_approval_manifest"]
    if not approval_path.is_file():
        raise SystemExit(
            f"No pilot manifest at {approval_path}. Run --mode pilot first, review the images, "
            "then approve."
        )
    manifest = read_json(approval_path)
    if manifest.get("split_provenance") != provenance:
        raise SystemExit("Pilot provenance no longer matches the current split/recipe/checkpoint/config; rerun the pilot")
    pilot_manifest = paths["pilot_manifest"]
    if not pilot_manifest.is_file() or manifest.get("pilot_manifest_sha256") != sha256_file(pilot_manifest):
        raise SystemExit("Pilot generation manifest is missing or changed; approval refused")
    if not manifest.get("pilot_completed", False):
        raise SystemExit("Pilot did not complete — cannot approve.")
    if not manifest.get("checks", {}).get("passed", False):
        raise SystemExit(
            "Pilot automatic checks did not pass — cannot approve.\n"
            f"{json.dumps(manifest.get('checks', {}), indent=2)}"
        )

    manifest["approval"] = {
        "approved": True,
        "reviewer": reviewer,
        "notes": notes,
        "approved_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(approval_path, manifest)
    print(f"Pilot APPROVED by {reviewer} at {manifest['approval']['approved_at_utc']}", flush=True)
    print(f"Manifest: {approval_path}", flush=True)
    print("Full generation is now unblocked.", flush=True)
    return 0


def enforce_pilot_gate(config, paths: dict[str, Path], provenance: dict) -> dict:
    """FROZEN gate (§3): full generation refuses to start without a completed, checked, and
    explicitly approved pilot."""
    approval_path = paths["pilot_approval_manifest"]
    if not approval_path.is_file():
        raise SystemExit(
            "PILOT GATE: full generation refused.\n"
            f"  No pilot approval manifest at {approval_path}\n"
            "  Run:  python scripts/generate/02_generate_synthetic_images.py --mode pilot\n"
            "  Then review the pilot images and approve with --approve-pilot --reviewer <name>."
        )
    manifest = read_json(approval_path)
    if manifest.get("split_provenance") != provenance:
        raise SystemExit("PILOT GATE: approval provenance differs from current production inputs")
    if manifest.get("pilot_manifest_sha256") != sha256_file(paths["pilot_manifest"]):
        raise SystemExit("PILOT GATE: pilot manifest hash mismatch")
    if not manifest.get("pilot_completed", False):
        raise SystemExit("PILOT GATE: full generation refused — pilot did not complete.")
    if not manifest.get("checks", {}).get("passed", False):
        raise SystemExit(
            "PILOT GATE: full generation refused — pilot automatic checks did not pass.\n"
            f"{json.dumps(manifest.get('checks', {}), indent=2)}"
        )
    if not manifest.get("approval", {}).get("approved", False):
        raise SystemExit(
            "PILOT GATE: full generation refused — pilot completed and passed checks, but carries "
            "no explicit approval record.\n"
            "  Approve with: --approve-pilot --reviewer <name>"
        )
    return manifest


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=["pilot", "full"], default=None)
    parser.add_argument("--approve-pilot", action="store_true", help="Record explicit pilot approval")
    parser.add_argument("--reviewer", default=None, help="Required with --approve-pilot")
    parser.add_argument("--notes", default="", help="Optional reviewer notes")
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_stage2_config()
    stage1_cfg = load_stage1_config()
    namespace = args.namespace or str(config.split_namespace)
    paths = stage2_paths(config, namespace)

    if args.approve_pilot:
        if not args.reviewer:
            raise SystemExit("--approve-pilot requires --reviewer <name>")
        provenance = validate_generation_inputs(config, namespace, paths)
        return approve_pilot(config, paths, provenance, args.reviewer, args.notes)

    if not args.mode:
        raise SystemExit("--mode {pilot,full} is required (or use --approve-pilot)")

    paths = stage2_paths(config, namespace)
    provenance = validate_generation_inputs(config, namespace, paths)
    recipes_path = paths["recipes_path"]
    recipes = pd.read_csv(recipes_path)

    if args.mode == "pilot":
        images_dir = paths["pilot_images"]
        # Deterministic spanning subset: evenly strided so the pilot covers the recipe space
        # rather than sampling one corner of it.
        pilot_n = min(int(config.pilot.num_images), len(recipes))
        stride = max(1, len(recipes) // pilot_n)
        selected = recipes.iloc[::stride].head(pilot_n).reset_index(drop=True)
        manifest_path = paths["pilot_manifest"]
    else:
        enforce_pilot_gate(config, paths, provenance)
        images_dir = paths["images_dir"]
        selected = recipes
        manifest_path = paths["manifest_path"]

    images_dir.mkdir(parents=True, exist_ok=True)

    registry_config = {
        "stage2": OmegaConf.to_container(config, resolve=True),
        "split_provenance": provenance,
        "mode": args.mode,
    }
    with ExperimentRun(
        stage=f"stage2_generation_{args.mode}",
        config=registry_config,
        dataset_version=provenance["split_manifest_hash"],
    ) as run:
        counts = run_generation(
            selected, config, stage1_cfg, images_dir, manifest_path, args.mode, provenance
        )
        run.set_checkpoint_path(images_dir)
        run.set_metrics(counts)

    if args.mode == "pilot":
        checks = run_pilot_checks(images_dir, selected["recipe_id"].tolist(), config)
        approval_path = paths["pilot_approval_manifest"]
        existing_approval = {}
        if approval_path.is_file():
            existing_approval = read_json(approval_path).get("approval", {})
        write_json(
            approval_path,
            {
                "schema_version": SCHEMA_VERSION,
                "pilot_completed": True,
                "pilot_generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "num_pilot_images": len(selected),
                "counts": counts,
                "checks": checks,
                "images_dir": str(images_dir),
                "split_provenance": provenance,
                "pilot_manifest_sha256": sha256_file(manifest_path),
                "git_commit_hash": get_git_commit_hash(),
                # Approval is NEVER auto-granted here; a prior approval is preserved but a new
                # pilot run invalidates it by resetting to unapproved unless it already existed
                # for this same manifest.
                "approval": existing_approval if existing_approval else {"approved": False},
            },
        )
        print(f"\nPilot checks: {'PASSED' if checks['passed'] else 'FAILED'}", flush=True)
        print(json.dumps(checks, indent=2), flush=True)
        print(f"\nManifest: {approval_path}", flush=True)
        if checks["passed"]:
            print(
                "\nNext: review the pilot images yourself, then run:\n"
                "  python scripts/generate/02_generate_synthetic_images.py "
                "--approve-pilot --reviewer <your name>",
                flush=True,
            )
        return 0 if checks["passed"] else 3

    print(f"\n[full] generated={counts['generated']} skipped_valid={counts['skipped_valid']} "
          f"failed={counts['failed']}", flush=True)
    print(f"Images:   {images_dir}", flush=True)
    print(f"Manifest: {manifest_path}", flush=True)
    write_json(paths["generation_provenance"], provenance)
    if counts["failed"] == 0:
        manifest_rows = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        actual_ids = [str(row.get("image_id")) for row in manifest_rows]
        expected_ids = selected["recipe_id"].astype(str).tolist()
        if len(actual_ids) != len(set(actual_ids)) or set(actual_ids) != set(expected_ids):
            raise SystemExit("Generation completion refused: manifest IDs are duplicate, missing, or not the exact recipe partition")
        write_json(paths["generation_completion"], {**provenance, "status": "complete",
                   "generation_manifest_sha256": sha256_file(manifest_path),
                   "num_rows": len(manifest_rows), "image_id_set_sha256": hash_dict({"image_ids": sorted(actual_ids)}, length=64)})
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 6 - Stage 3, first half: the auxiliary classifier, the signals, and the Go/No-Go gate

This is where the thesis starts. Six per-image signals are computed, then each must earn its place through a gate that runs **before** any tuning.

## `scripts/classify/00_train_auxiliary_classifier.py`

*155 lines*

**Stage 3 prerequisite - and a file it is easy to misread.** This is **not** condition A.

It exists so the uncertainty, explainability and agreement signals have a real-data classifier to
query. It is trained on `gen_train` and validated on `gen_val` - **never** on `classifier_train` /
`classifier_val` (reserved unspent for the headline conditions) and never on a heldout split.
Using the generator's own splits for a scorer of the generator's output is the point: it leaves the
entire classifier development pool untouched.

Initialization is leakage-safe by construction: an ImageNet or random init only - a
CheXpert-pretrained backbone is rejected in code, because it would have seen the evaluation
patients.

Condition A is a *separate* experiment, trained later, from scratch, under the frozen Stage 4
protocol.

In [ ]:
# ===== FILE: scripts/classify/00_train_auxiliary_classifier.py =====
#!/usr/bin/env python3
"""Train the auxiliary real-only reference classifier (docs/stages2_to_5_plan.md §2).

This is an ASISM PREREQUISITE, not condition A. It exists so Stage 3's uncertainty (§4.3),
explainability (§4.4), and agreement (§4.5) signals have a real-data classifier to query. Condition
A is a separate experiment trained later, from scratch, on classifier_train under §7's frozen
protocol.

Trained on gen_train, validated on gen_val — never classifier_train/classifier_val (those are
reserved unspent for A/B/C) and never a heldout split. Using the generator's own splits for a scorer
of the generator's output is the point: it keeps the entire classifier development pool free.

Initialization is leakage-safe by construction: CheXpert-pretrained weights are REJECTED in
scripts/utils/classifier.build_model(), because their training patients cannot be shown disjoint
from final_eval_heldout.

Usage:
    python scripts/classify/00_train_auxiliary_classifier.py [--namespace dev] [--max-steps N]
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.classifier import (  # noqa: E402
    TrainingBudget,
    records_from_split,
    require_torch,
    train_classifier,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.artifact_contracts import auxiliary_checkpoint_path, current_code_identity_hash, namespace_identity  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--max-steps", type=int, default=4000)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    config = load_named_config("stage3_asism.yaml", "stage3")
    stage1_cfg = load_stage1_config()

    namespace = args.namespace or str(config.split_namespace)
    provenance = namespace_identity(namespace)

    images_dir = Path(stage1_cfg.paths.images_dir) / namespace
    train_frame = load_split("gen_train", namespace, purpose="schema_validation", caller="aux_clf")
    val_frame = load_split("gen_val", namespace, purpose="schema_validation", caller="aux_clf")

    train_records = records_from_split(train_frame, images_dir / "gen_train")
    val_records = records_from_split(val_frame, images_dir / "gen_val")

    if not train_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_train images found under {images_dir / 'gen_train'}.\n"
            "Stage 1 preprocessing (scripts/data/03_preprocess_images.py) must run first."
        )
    if not val_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_val images found under {images_dir / 'gen_val'}."
        )

    require_torch()

    checkpoint_path = auxiliary_checkpoint_path(config, namespace)
    budget = TrainingBudget(
        max_steps=args.max_steps,
        batch_size=args.batch_size,
        seed=args.seed,
        eval_every_n_steps=max(100, args.max_steps // 10),
    )

    registry_config = {
        "auxiliary_classifier": OmegaConf.to_container(config.auxiliary_classifier, resolve=True),
        "budget": budget.__dict__,
        "split_provenance": provenance,
    }

    with ExperimentRun(
        stage="auxiliary_reference_classifier",
        config=registry_config,
        dataset_version=provenance["split_manifest_hash"],
    ) as run:
        model, accounting, history = train_classifier(
            train_records=train_records,
            val_records=val_records,
            budget=budget,
            dropout_p=float(config.auxiliary_classifier.dropout_p),
            pretrained_source=str(config.auxiliary_classifier.pretrained_source),
            resolution=int(config.auxiliary_classifier.resolution),
            progress_desc="aux-classifier",
            checkpoint_path=checkpoint_path,
            resume_from=checkpoint_path if checkpoint_path.is_file() else None,
            checkpoint_metadata={
                "split_manifest_hash": provenance["split_manifest_hash"],
                "split_namespace": namespace,
                "dataset_id": "auxiliary_gen_train_gen_val",
                "config_hash": hash_dict(registry_config),
                "label_policy": "chexpert_mask_uncertain_v1",
                "condition_or_draw_id": "auxiliary",
                "code_version": get_git_commit_hash(),
                "code_identity_sha256": current_code_identity_hash(),
            },
        )
        best = max(history, key=lambda entry: entry["macro_auroc"]) if history else {}
        accounting.config_hash = hash_dict(registry_config)

        run.set_checkpoint_path(checkpoint_path)
        run.set_metrics({"best_macro_auroc": best.get("macro_auroc"), **accounting.to_dict()})

    write_json(
        checkpoint_path.with_suffix(".manifest.json"),
        {
            "stage": "auxiliary_reference_classifier",
            "role": "ASISM prerequisite scorer — NOT condition A",
            "trained_on": "gen_train",
            "validated_on": "gen_val",
            "never_touched": ["classifier_train", "classifier_val", "asism_tuning_heldout", "final_eval_heldout"],
            "pretrained_source": str(config.auxiliary_classifier.pretrained_source),
            "pretrained_provenance": {
                "note": "ImageNet weights via torchvision; no CheXpert patients involved.",
                "chexpert_pretrained_rejected": True,
            },
            "split_provenance": provenance,
            "accounting": accounting.to_dict(),
            "eval_history": history,
            "config_hash": hash_dict(registry_config),
            "resume_checkpoint": str(checkpoint_path),
            "resume_checkpoint_sha256": sha256_file(checkpoint_path),
            "best_checkpoint": accounting.extra.get("best_checkpoint_path"),
            "best_checkpoint_sha256": sha256_file(accounting.extra["best_checkpoint_path"]),
            "code_identity_sha256": current_code_identity_hash(),
            "git_commit_hash": get_git_commit_hash(),
        },
    )

    print(f"Auxiliary classifier -> {checkpoint_path}", flush=True)
    if history:
        print(f"Best gen_val macro-AUROC: {max(e['macro_auroc'] for e in history):.4f}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/signals.py`

*629 lines*

**The five ASISM signals, as pure functions.** This is the scientific core of Stage 3 and the file
to read first when trying to understand the thesis.

| Signal | Function | What it measures |
|---|---|---|
| Similarity | `compute_similarity_scores` | k-NN similarity (k = 15) of a DINOv2 embedding against a **per-label real reference pool**; `hierarchical_reference_indices` falls back to looser label matches when exact-match references are scarce. Emits `similarity_knn_mean`, `similarity_top1`, `similarity_topk_spread`. |
| Image quality | `compute_iqa_scores` | blank / clipping / low-contrast / uniform-border / blur (`_laplacian_variance`) checks -> `iqa_composite`, `iqa_sharpness`, `iqa_contrast_std`. |
| Uncertainty | `compute_uncertainty_scores` | predictive std over MC-Dropout passes. Note the sign convention: **moderate uncertainty is not penalised** - a perfectly confident synthetic image is not automatically the most useful one. |
| Explainability | `region_overlap_score`, `expected_region_for`, `aggregate_pathology_overlaps` | Grad-CAM mass vs. the *empirically derived* expected pathology region (see `00c`), not a hand-drawn box. |
| Agreement | `compute_agreement_scores` | mean P(intended positives) minus penalty x mean P(confident **unintended** positives). |

Plus **distinctiveness** (`compute_distinctiveness_scores`): within-class k-NN redundancy,
`1 - mean top-k similarity`, and `duplicate_clusters` for grouping near-identical images.

The **near-duplicate / memorisation flag** fires at `similarity_top1 >= 0.95` (grounded in Dar et
al., *Nature Biomedical Engineering* 2025). Such images get no novelty credit and are rejected.

`write_score_artifact` writes each signal's Parquet plus a `.provenance.json` sidecar - the file
`artifact_contracts.require_score_artifact` later verifies.

**Design note:** IQA is deliberately pure NumPy/PIL. It is the one signal that needs no model and
no GPU, which is what keeps the whole selection layer developable off the pod.

In [ ]:
# ===== FILE: scripts/asism/signals.py =====
"""The five ASISM signals (docs/stages2_to_5_plan.md §4.1-§4.5).

Each signal computes a per-image score set written to its own versioned parquet artifact. All five
are subject to the same Go/No-Go gate (§4.6) — implementation is not admission.

The IQA signal is deliberately pure NumPy/PIL: it is the one signal that needs no model and no GPU,
so it stays runnable and testable locally, which is what lets the whole selection/ranking layer be
developed off the pod.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, intended_vector_to_labels

SCHEMA_VERSION = 2


def aggregate_pathology_overlaps(per_label: dict[str, float]) -> tuple[float, float]:
    """Frozen multi-label Grad-CAM rule: equal-weight mean, with minimum retained as a guardrail."""
    finite = [float(value) for value in per_label.values() if np.isfinite(value)]
    if not finite:
        return float("nan"), float("nan")
    return float(np.mean(finite)), float(min(finite))


# ----------------------------------------------------------------------------------------------
# §4.2 Image Quality Assessment — no model, no GPU
# ----------------------------------------------------------------------------------------------

def _laplacian_variance(gray: np.ndarray) -> float:
    """Variance of the Laplacian: the standard focus/blur measure. Low variance = few sharp edges."""
    kernel = np.array([[0.0, 1.0, 0.0], [1.0, -4.0, 1.0], [0.0, 1.0, 0.0]])
    padded = np.pad(gray, 1, mode="edge")
    response = np.zeros_like(gray, dtype=np.float64)
    for dy in range(3):
        for dx in range(3):
            weight = kernel[dy, dx]
            if weight != 0.0:
                response += weight * padded[dy : dy + gray.shape[0], dx : dx + gray.shape[1]]
    return float(response.var())


def compute_iqa_scores(image_path: Path, config) -> dict:
    """Multi-faceted technical quality (§4.2).

    Generic no-reference IQA is represented by the sharpness/contrast statistics; the domain checks
    (border artifact, clipping, near-uniformity, grayscale consistency, anatomical completeness
    proxy) are what the signal actually leans on. Whether the generic part adds anything beyond the
    domain part is settled by ablation, not assumed.
    """
    iqa_cfg = config.signals.iqa
    try:
        with Image.open(image_path) as image:
            rgb = image.convert("RGB")
            array = np.asarray(rgb, dtype=np.float32)
            gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    except Exception as exc:
        return {"iqa_valid": False, "iqa_error": f"{type(exc).__name__}: {exc}"}

    height, width = gray.shape
    std = float(gray.std())
    mean = float(gray.mean())

    # Grayscale consistency: a real CXR is achromatic. Channel divergence indicates colour drift
    # from the RGB generation path.
    channel_spread = float(np.mean(np.std(array, axis=2)))

    # Clipping: mass piled at the extremes of the dynamic range.
    clipped_low = float(np.mean(gray <= 1.0))
    clipped_high = float(np.mean(gray >= 254.0))

    # Border artifact: letterbox residue or a dead frame around the anatomy.
    border_width = max(1, int(min(height, width) * 0.06))
    border = np.concatenate(
        [
            gray[:border_width, :].ravel(),
            gray[-border_width:, :].ravel(),
            gray[:, :border_width].ravel(),
            gray[:, -border_width:].ravel(),
        ]
    )
    border_uniform_fraction = float(np.mean(np.abs(border - border.mean()) < 2.0))

    # Anatomical completeness proxy: a plausible CXR has substantially more signal in the central
    # region than at the frame edge. A near-empty centre means the anatomy is missing or collapsed.
    centre = gray[
        int(height * 0.25) : int(height * 0.75), int(width * 0.25) : int(width * 0.75)
    ]
    centre_energy_ratio = float(centre.std() / (std + 1e-6))

    sharpness = _laplacian_variance(gray)

    # Histogram abnormality: distance from a smooth unimodal shape. A spiky/degenerate histogram
    # indicates posterization or a broken decode.
    histogram, _ = np.histogram(gray, bins=64, range=(0, 255))
    histogram = histogram / max(histogram.sum(), 1)
    histogram_entropy = float(-np.sum(histogram[histogram > 0] * np.log(histogram[histogram > 0])))

    is_near_uniform = std < float(iqa_cfg.blank_std_threshold)
    is_blurry = sharpness < float(iqa_cfg.laplacian_blur_threshold)
    is_low_contrast = std < float(iqa_cfg.low_contrast_std)
    has_border_artifact = border_uniform_fraction > float(iqa_cfg.border_uniform_fraction)

    # Composite in [0, 1]: 1.0 is clean. Each detected defect subtracts a fixed amount, so the
    # composite degrades gracefully rather than being dominated by one continuous term.
    penalties = (
        0.40 * float(is_near_uniform)
        + 0.20 * float(is_blurry)
        + 0.10 * float(is_low_contrast)
        + 0.10 * float(has_border_artifact)
        + 0.10 * float(clipped_low + clipped_high > 0.20)
        + 0.10 * float(channel_spread > 2.0)
    )
    composite = float(max(0.0, 1.0 - penalties))

    return {
        "iqa_valid": True,
        "iqa_composite": composite,
        "iqa_sharpness": sharpness,
        "iqa_contrast_std": std,
        "iqa_mean_intensity": mean,
        "iqa_channel_spread": channel_spread,
        "iqa_clipped_low_fraction": clipped_low,
        "iqa_clipped_high_fraction": clipped_high,
        "iqa_border_uniform_fraction": border_uniform_fraction,
        "iqa_centre_energy_ratio": centre_energy_ratio,
        "iqa_histogram_entropy": histogram_entropy,
        "iqa_is_near_uniform": bool(is_near_uniform),
        "iqa_is_blurry": bool(is_blurry),
        "iqa_is_low_contrast": bool(is_low_contrast),
        "iqa_has_border_artifact": bool(has_border_artifact),
    }


# ----------------------------------------------------------------------------------------------
# §4.5 Intended-label agreement
# ----------------------------------------------------------------------------------------------

def compute_agreement_scores(
    probabilities: np.ndarray,
    intended_vectors: list[dict],
    config,
) -> pd.DataFrame:
    """Proxy label-consistency between the generator's intent and the auxiliary classifier's read.

        agreement = mean P(intended positive labels)
                  - penalty_weight * mean P(confidently predicted UNintended labels)

    For a No Finding recipe the intended positive set is empty, so the first term is defined as
    (1 - mean probability over all 11 primary labels): agreement is high exactly when the classifier
    sees no disease, which is what that recipe asked for.

    NOT clinical proof. The generator and the classifier can share a bias and agree while both are
    wrong; this measures consistency between two models, nothing more (§4.5).
    """
    agreement_cfg = config.signals.agreement
    threshold = float(agreement_cfg.unintended_confidence_threshold)
    penalty_weight = float(agreement_cfg.penalty_weight)

    label_index = {label: position for position, label in enumerate(PRIMARY_ENDPOINT_LABELS)}
    rows = []

    for row_index, intended in enumerate(intended_vectors):
        probs = probabilities[row_index]
        intended_positive = intended_vector_to_labels(intended)
        unintended = [label for label in PRIMARY_ENDPOINT_LABELS if label not in intended_positive]

        if intended_positive:
            intended_mean = float(
                np.mean([probs[label_index[label]] for label in intended_positive])
            )
            is_no_finding = False
        else:
            # No Finding recipe: "intended" outcome is uniformly low disease probability.
            intended_mean = float(
                1.0 - np.mean([probs[label_index[label]] for label in PRIMARY_ENDPOINT_LABELS])
            )
            is_no_finding = True

        unintended_probs = [probs[label_index[label]] for label in unintended]
        confident_unintended = [p for p in unintended_probs if p >= threshold]
        unintended_penalty = float(np.mean(confident_unintended)) if confident_unintended else 0.0

        rows.append(
            {
                "agreement_score": intended_mean - penalty_weight * unintended_penalty,
                "agreement_intended_mean_prob": intended_mean,
                "agreement_unintended_penalty": unintended_penalty,
                "agreement_n_intended_positive": len(intended_positive),
                "agreement_n_confident_unintended": len(confident_unintended),
                "agreement_is_no_finding_recipe": is_no_finding,
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------
# §4.3 Uncertainty
# ----------------------------------------------------------------------------------------------

def compute_uncertainty_scores(
    probability_std: np.ndarray,
    probability_mean: np.ndarray,
    config,
) -> pd.DataFrame:
    """MC-Dropout predictive spread over the 11 primary labels, banded.

    Bands are reported, NOT converted into a penalty here: moderate uncertainty may be the most
    informative band (TSynD's premise, §4.3), so whether a band is good or bad is a selection-policy
    decision made later in §4.8, not baked into the signal.
    """
    uncertainty_cfg = config.signals.uncertainty
    low_max = float(uncertainty_cfg.low_band_max)
    moderate_max = float(uncertainty_cfg.moderate_band_max)

    mean_std = probability_std.mean(axis=1)
    max_std = probability_std.max(axis=1)
    # Predictive entropy of the mean probabilities: a second, complementary uncertainty view.
    clipped = np.clip(probability_mean, 1e-6, 1 - 1e-6)
    entropy = -(clipped * np.log(clipped) + (1 - clipped) * np.log(1 - clipped)).mean(axis=1)

    bands = np.where(mean_std <= low_max, "low", np.where(mean_std <= moderate_max, "moderate", "extreme"))

    return pd.DataFrame(
        {
            "uncertainty_mean_std": mean_std,
            "uncertainty_max_std": max_std,
            "uncertainty_entropy": entropy,
            "uncertainty_band": bands,
        }
    )


# ----------------------------------------------------------------------------------------------
# §4.1 Similarity
# ----------------------------------------------------------------------------------------------

def hierarchical_reference_indices(
    intended: dict,
    reference_label_vectors: list[frozenset],
    min_exact: int,
) -> tuple[np.ndarray, str]:
    """Pick real-reference rows for one synthetic image, walking the §4.1 hierarchy.

    Tier 1 exact intended-vector match -> Tier 2 nearest multilabel neighbourhood (Jaccard) ->
    Tier 3 pathology prototypes (any shared positive) -> Tier 4 class-agnostic realism fallback.
    The fallback is always available, so no image goes unscored for lack of an exact-match pool —
    which is precisely the failure mode a rigid exact-match requirement would cause for rare
    combinations.
    """
    intended_set = frozenset(intended_vector_to_labels(intended))

    exact = np.array(
        [i for i, combo in enumerate(reference_label_vectors) if combo == intended_set], dtype=int
    )
    if len(exact) >= min_exact:
        return exact, "tier1_exact"

    if intended_set:
        similarities = np.array(
            [
                len(intended_set & combo) / max(len(intended_set | combo), 1)
                for combo in reference_label_vectors
            ]
        )
        neighbourhood = np.where(similarities >= 0.5)[0]
        if len(neighbourhood) >= min_exact:
            return neighbourhood, "tier2_neighbourhood"

        prototype = np.array(
            [i for i, combo in enumerate(reference_label_vectors) if intended_set & combo], dtype=int
        )
        if len(prototype) >= min_exact:
            return prototype, "tier3_prototype"

    return np.arange(len(reference_label_vectors)), "tier4_class_agnostic"


def compute_similarity_scores(
    synthetic_embeddings: np.ndarray,
    reference_embeddings: np.ndarray,
    reference_label_vectors: list[frozenset],
    intended_vectors: list[dict],
    config,
) -> pd.DataFrame:
    """k-NN cosine similarity to real references, plus a SEPARATE memorization flag.

    Fidelity and memorization are reported as distinct columns on purpose: an image that is nearly
    identical to one specific training image is a memorization red flag, not a fidelity success, and
    collapsing the two would make the selector reward exactly what it should catch (§4.1).
    """
    similarity_cfg = config.signals.similarity
    k = int(similarity_cfg.k_neighbors)
    min_exact = int(similarity_cfg.min_exact_match_references)
    near_dup_similarity = float(similarity_cfg.near_duplicate_similarity)
    near_dup_spread = float(similarity_cfg.near_duplicate_spread)

    synthetic_norm = synthetic_embeddings / (
        np.linalg.norm(synthetic_embeddings, axis=1, keepdims=True) + 1e-8
    )
    reference_norm = reference_embeddings / (
        np.linalg.norm(reference_embeddings, axis=1, keepdims=True) + 1e-8
    )

    rows = []
    for row_index in range(len(synthetic_norm)):
        indices, tier = hierarchical_reference_indices(
            intended_vectors[row_index], reference_label_vectors, min_exact
        )
        similarities = reference_norm[indices] @ synthetic_norm[row_index]

        effective_k = min(k, len(similarities))
        top_k = np.sort(similarities)[-effective_k:][::-1]
        top1 = float(top_k[0])
        knn_mean = float(top_k.mean())
        # Gap between the single closest reference and the rest of the neighbourhood.
        spread = float(top_k[0] - top_k[-1]) if effective_k > 1 else 0.0
        # How far top1 stands out from the remaining neighbours specifically.
        rest_mean = float(top_k[1:].mean()) if effective_k > 1 else top1
        top1_gap = top1 - rest_mean

        # Memorization = the image is near-identical to SOME real training image. top1 crossing the
        # threshold is sufficient on its own; a large top1_gap additionally indicates it collapsed
        # onto ONE specific image rather than onto a cluster of near-identical real images.
        # (Requiring a SMALL spread here would be backwards: an image memorizing one reference is
        # by construction far from the others, so its spread is large.)
        is_near_duplicate = bool(top1 >= near_dup_similarity)
        collapsed_onto_single = bool(is_near_duplicate and top1_gap >= near_dup_spread)

        rows.append(
            {
                "similarity_knn_mean": knn_mean,
                "similarity_top1": top1,
                "similarity_topk_spread": spread,
                "similarity_reference_tier": tier,
                "similarity_n_references": int(len(indices)),
                "similarity_k_used": effective_k,
                "similarity_top1_gap": top1_gap,
                "novelty_is_near_duplicate": is_near_duplicate,
                "novelty_collapsed_onto_single_reference": collapsed_onto_single,
                # Novelty score: high when the image resembles the class without being a near-copy
                # of a specific training image. Memorized images get no novelty credit.
                "novelty_score": float(knn_mean * (1.0 - float(is_near_duplicate))),
            }
        )

    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------
# §4.6 Within-synthetic distinctiveness (NEW)
# ----------------------------------------------------------------------------------------------

def compute_distinctiveness_scores(
    synthetic_embeddings: np.ndarray,
    intended_vectors: list[dict],
    config,
) -> pd.DataFrame:
    """How distinct each synthetic image is from OTHER SYNTHETIC images of the SAME intended class.

    This is deliberately NOT the same question as §4.1's similarity/novelty, and the distinction is
    the whole reason the signal exists:

        §4.1 novelty        : how close is this image to a REAL image?      (fidelity / memorization)
        §4.6 distinctiveness: how close is it to OTHER SYNTHETIC images?    (redundancy)

    A diffusion model asked repeatedly for the same condition returns near-copies. Those copies each
    score well on every §4.1-§4.5 signal — they are individually realistic, sharp, confidently
    classified, and correctly localized — while adding almost no new information to a training set.
    Nothing in the other five signals can see that, because all five score an image in isolation.

    WITHIN-CLASS BY CONSTRUCTION. Neighbours are drawn only from images sharing the exact same
    `intended_label_vector`. Comparing across classes would make the score a proxy for how many
    images that class happens to have: a common condition with 1500 generated images would look
    "redundant" and a rare one with 80 would look "distinct", purely from the Stage 2 quota. That
    would then feed the class-aware threshold, which is itself trying to correct class imbalance —
    a confound in exactly the wrong direction.

    DIRECTION: reported as DISTINCTIVENESS (higher = better = more unique), matching the direction
    of every other admitted signal, so §4.6's directionality probe needs no special case for it.
    """
    distinct_cfg = config.signals.distinctiveness
    k = int(distinct_cfg.k_neighbors)
    duplicate_similarity = float(distinct_cfg.duplicate_similarity)

    normalized = synthetic_embeddings / (
        np.linalg.norm(synthetic_embeddings, axis=1, keepdims=True) + 1e-8
    )

    # Group row indices by exact intended label vector; a No Finding recipe (all-zero) forms its
    # own group rather than being merged into any disease group.
    groups: dict[frozenset, list[int]] = {}
    for row_index, intended in enumerate(intended_vectors):
        key = frozenset(intended_vector_to_labels(intended))
        groups.setdefault(key, []).append(row_index)

    rows: list[dict] = [None] * len(normalized)
    for key, member_indices in groups.items():
        members = normalized[member_indices]
        # Cosine similarity of every member against every other member of the same class.
        similarity_matrix = members @ members.T
        np.fill_diagonal(similarity_matrix, -np.inf)  # never let an image be its own neighbour

        group_size = len(member_indices)
        effective_k = min(k, group_size - 1)

        for position, row_index in enumerate(member_indices):
            if effective_k <= 0:
                # A class with a single generated image has no within-class peer, so redundancy is
                # undefined rather than zero. Recorded explicitly so the Go/No-Go missing-value
                # check sees it instead of a fabricated "perfectly distinct" score.
                rows[row_index] = {
                    "distinctiveness_score": float("nan"),
                    "distinctiveness_mean_topk_similarity": float("nan"),
                    "distinctiveness_max_similarity": float("nan"),
                    "distinctiveness_n_duplicates": 0,
                    "distinctiveness_group_size": group_size,
                    "distinctiveness_k_used": 0,
                    "distinctiveness_is_undefined": True,
                }
                continue

            similarities = similarity_matrix[position]
            top_k = np.sort(similarities)[-effective_k:]
            mean_topk = float(top_k.mean())
            max_similarity = float(top_k[-1])
            n_duplicates = int((similarities >= duplicate_similarity).sum())

            rows[row_index] = {
                "distinctiveness_score": float(1.0 - mean_topk),
                "distinctiveness_mean_topk_similarity": mean_topk,
                "distinctiveness_max_similarity": max_similarity,
                "distinctiveness_n_duplicates": n_duplicates,
                "distinctiveness_group_size": group_size,
                "distinctiveness_k_used": effective_k,
                "distinctiveness_is_undefined": False,
            }

    return pd.DataFrame(rows)


def duplicate_clusters(
    synthetic_embeddings: np.ndarray,
    intended_vectors: list[dict],
    duplicate_similarity: float,
) -> list[list[int]]:
    """Group near-identical within-class images into clusters (single-linkage, union-find).

    NOT ON THE CURRENT PATH: selection rejects near-duplicates individually via the
    similarity_top1 >= near_duplicate_similarity flag, so no caller needs the clusters yet.
    Retained (and unit-tested) for the cluster-level de-duplication ablation.

    Exists because a per-image score cannot break a tie inside a duplicate cluster: three
    near-identical images receive three near-identical scores, so any threshold either admits all
    three or rejects all three. Keeping exactly one representative is a set-level decision, and it
    is made HERE, before scoring, rather than by pretending a threshold can make it.

    Returns clusters of size >= 2 only; a singleton needs no decision.
    """
    normalized = synthetic_embeddings / (
        np.linalg.norm(synthetic_embeddings, axis=1, keepdims=True) + 1e-8
    )
    parent = list(range(len(normalized)))

    def find(node: int) -> int:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(a: int, b: int) -> None:
        root_a, root_b = find(a), find(b)
        if root_a != root_b:
            parent[max(root_a, root_b)] = min(root_a, root_b)

    groups: dict[frozenset, list[int]] = {}
    for row_index, intended in enumerate(intended_vectors):
        groups.setdefault(frozenset(intended_vector_to_labels(intended)), []).append(row_index)

    for member_indices in groups.values():
        if len(member_indices) < 2:
            continue
        members = normalized[member_indices]
        similarity_matrix = members @ members.T
        for i in range(len(member_indices)):
            for j in range(i + 1, len(member_indices)):
                if similarity_matrix[i, j] >= duplicate_similarity:
                    union(member_indices[i], member_indices[j])

    clusters: dict[int, list[int]] = {}
    for row_index in range(len(normalized)):
        clusters.setdefault(find(row_index), []).append(row_index)
    return [sorted(members) for members in clusters.values() if len(members) >= 2]


# ----------------------------------------------------------------------------------------------
# §4.4 Explainability
# ----------------------------------------------------------------------------------------------

def region_overlap_score(cam: np.ndarray, region: tuple[float, float, float, float]) -> float:
    """Fraction of total Grad-CAM activation mass falling inside the expected region.

    An explainability-PLAUSIBILITY signal, never proof of diagnostic correctness (§4.4). High
    overlap means the classifier looked where the pathology usually appears; it says nothing about
    whether the pathology is actually present.
    """
    height, width = cam.shape
    x0, y0, x1, y1 = region
    col0, col1 = int(x0 * width), int(x1 * width)
    row0, row1 = int(y0 * height), int(y1 * height)

    cam = np.maximum(cam, 0.0)
    total = float(cam.sum())
    if total <= 0.0:
        return float("nan")
    return float(cam[row0:row1, col0:col1].sum() / total)


def region_from_cam_mass(mean_cam: np.ndarray, mass_fraction: float) -> tuple[float, float, float, float]:
    """Smallest axis-aligned box holding `mass_fraction` of the averaged Grad-CAM mass.

    Rows and columns are grown independently from their centre of mass outward, each step taking
    whichever adjacent line carries more mass, until the target fraction is enclosed. Returned as
    (x0, y0, x1, y1) fractions of the image -- the same format the hand-entered boxes used, so
    nothing downstream needs to change shape.
    """
    if not 0.0 < mass_fraction <= 1.0:
        raise ValueError(f"mass_fraction must be in (0, 1], got {mass_fraction}")
    cam = np.maximum(mean_cam, 0.0)
    if float(cam.sum()) <= 0.0:
        raise ValueError("Grad-CAM mass is zero; cannot derive a region")

    def span(profile: np.ndarray) -> tuple[int, int]:
        profile = profile / profile.sum()
        centre = int(np.argmax(np.cumsum(profile) >= 0.5))
        low = high = centre
        covered = float(profile[centre])
        while covered < mass_fraction and (low > 0 or high < len(profile) - 1):
            take_low = float(profile[low - 1]) if low > 0 else -1.0
            take_high = float(profile[high + 1]) if high < len(profile) - 1 else -1.0
            if take_high >= take_low:
                high += 1
                covered += take_high
            else:
                low -= 1
                covered += take_low
        return low, high

    row0, row1 = span(cam.sum(axis=1))
    col0, col1 = span(cam.sum(axis=0))
    height, width = cam.shape
    return (
        float(col0 / width), float(row0 / height),
        float((col1 + 1) / width), float((row1 + 1) / height),
    )


def expected_region_for(
    intended: dict, config, derived_regions: dict | None = None,
) -> tuple[float, float, float, float]:
    """Pathology-specific expected region, or the explicit baseline box for No Finding recipes.

    `derived_regions` (from 00c_derive_expected_regions.py) takes precedence per label when
    supplied: those boxes are MEASURED from Grad-CAM on real positives, whereas the config boxes are
    hand-entered constants with no derivation. Precedence is per LABEL, not all-or-nothing, so a
    pathology the derivation legitimately skipped (no real positives, or no attention) still falls
    back to its config box instead of losing its region entirely.
    """
    explain_cfg = config.signals.explainability
    positives = intended_vector_to_labels(intended)
    if str(explain_cfg.region_mode) == "baseline" or not positives:
        return tuple(explain_cfg.baseline_box)

    derived_regions = derived_regions or {}
    regions = [
        tuple(derived_regions.get(label) or explain_cfg.pathology_regions[label])
        for label in positives
        if label in derived_regions or label in explain_cfg.pathology_regions
    ]
    if not regions:
        return tuple(explain_cfg.baseline_box)

    # Union of the per-pathology regions: a multi-label recipe may legitimately activate anywhere
    # its constituent pathologies are expected.
    return (
        min(r[0] for r in regions),
        min(r[1] for r in regions),
        max(r[2] for r in regions),
        max(r[3] for r in regions),
    )


def write_score_artifact(
    frame: pd.DataFrame,
    path: Path,
    signal_name: str,
    provenance: dict,
) -> None:
    """Atomic, versioned parquet write with an embedded provenance block (§4.0)."""
    import os
    import tempfile

    path.parent.mkdir(parents=True, exist_ok=True)
    frame = frame.copy()
    frame.attrs["schema_version"] = SCHEMA_VERSION
    frame.attrs["signal"] = signal_name

    sidecar = path.with_suffix(".provenance.json")
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    os.close(fd)
    try:
        frame.to_parquet(temp_name, index=False)
        os.replace(temp_name, path)
    finally:
        Path(temp_name).unlink(missing_ok=True)

    from scripts.utils.manifest import sha256_file, write_json
    write_json(sidecar, {
        "schema_version": SCHEMA_VERSION, "signal": signal_name, "n_rows": len(frame),
        "columns": list(frame.columns), "parquet_sha256": sha256_file(path), **provenance,
    })


## `scripts/asism/00c_derive_expected_regions.py`

*157 lines*

**Stage 3 preparation - derive where each pathology actually appears, empirically.**

The explainability signal compares a synthetic image's Grad-CAM against an "expected region" for
its intended pathology. Hand-drawing those boxes from a textbook would be a researcher-injected
prior. Instead, this script averages Grad-CAM maps over **real positives** for each label and takes
the box enclosing `--mass-fraction` (default 0.80) of the averaged mass (`region_from_cam_mass`).

Two deliberate choices:

- `DERIVATION_SPLIT = "gen_train"` is **not configurable**. Deriving the regions from any
  decision-bearing split would leak.
- `_signals_module()` loads `01_compute_signals.py` by file path (its leading digit makes it
  unimportable by name) so the Grad-CAM used here is *literally the same code* the signal uses,
  never a re-implementation that could drift.

Output is written with `write_frozen_json` - derived once, then immutable.

In [ ]:
# ===== FILE: scripts/asism/00c_derive_expected_regions.py =====
from __future__ import annotations

import argparse
import importlib.util
import sys
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.signals import region_from_cam_mass  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS, PRIMARY_ENDPOINT_LABELS, normalize_label  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, write_frozen_json  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402
from scripts.utils.artifact_contracts import asism_score_provenance, stage3_paths  # noqa: E402

# SPLIT DISCIPLINE — never configurable. The expected regions are a researcher-facing prior
# baked into the explainability signal, so deriving them from any decision-bearing split
# (classifier_*, asism_tuning_heldout, final_eval_heldout) would leak. gen_train is the only
# admissible source: Stage 1 already trained on it, and no Stage 3-5 decision is measured on it.
DERIVATION_SPLIT = "gen_train"


def _signals_module():
    """01_compute_signals.py is not importable by name (leading digit); load it by path so the
    Grad-CAM here is literally the same code the signal uses, never a re-implementation that could
    drift from it."""
    path = Path(__file__).with_name("01_compute_signals.py")
    spec = importlib.util.spec_from_file_location("compute_signals_01", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--max-images-per-label", type=int, default=200,
                        help="Cap on real positives averaged per pathology (cost control).")
    parser.add_argument("--mass-fraction", type=float, default=0.80,
                        help="Fraction of averaged Grad-CAM mass the derived box must enclose.")
    args = parser.parse_args()

    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths:
            cfg.paths[key] = str(value)
    cfg.split_namespace = namespace

    output_path = Path(cfg.paths.asism_dir) / "expected_regions.json"
    if output_path.exists():
        raise SystemExit(
            f"REFUSING to overwrite existing derived regions at {output_path}.\n"
            "They are a frozen input to every explainability score. Delete the file deliberately "
            "if you intend to re-derive them, and expect downstream hashes to change."
        )

    signals = _signals_module()
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    provenance = asism_score_provenance(cfg, stage2_cfg, namespace)
    model, resolution = signals.load_auxiliary_classifier(cfg, namespace, provenance)

    from scripts.utils.classifier import CXRRecordDataset, require_torch
    from scripts.utils.identifiers import sanitize_image_id

    torch = require_torch()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    frame = load_split(DERIVATION_SPLIT, namespace, purpose="schema_validation",
                       caller="derive_expected_regions")
    images_dir = Path(load_stage1_config().paths.images_dir) / namespace / DERIVATION_SPLIT

    activations, gradients = {}, {}
    target_layer = model.features.denseblock4
    handle_f = target_layer.register_forward_hook(
        lambda _m, _i, output: activations.__setitem__("value", output.detach()))
    handle_b = target_layer.register_full_backward_hook(
        lambda _m, _gi, grad_out: gradients.__setitem__("value", grad_out[0].detach()))

    derived: dict[str, dict] = {}
    skipped: dict[str, str] = {}
    try:
        for label in PRIMARY_ENDPOINT_LABELS:
            positives = frame[frame[label].map(normalize_label) == 1].head(args.max_images_per_label)
            records = []
            for row in positives.to_dict("records"):
                path = images_dir / f"{sanitize_image_id(row['Path'])}.jpg"
                if path.is_file():
                    records.append({"image_path": str(path), "labels": {}, "is_synthetic": False})
            if not records:
                # No confident real positive with a preprocessed image. Recorded explicitly so the
                # gap is visible, rather than inventing a region for this pathology.
                skipped[label] = "no preprocessed real positives found"
                continue

            dataset = CXRRecordDataset(records, resolution=resolution)
            label_index = CLASSIFIER_TARGET_LABELS.index(label)
            accumulated = None
            contributing = 0
            for index in tqdm(range(len(dataset)), desc=f"cam:{label}", unit="img", leave=False):
                image = dataset[index]["image"].unsqueeze(0).to(device)
                model.zero_grad(set_to_none=True)
                model(image)[0, label_index].backward()
                weights = gradients["value"].mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((weights * activations["value"]).sum(dim=1)).squeeze(0)
                peak = float(cam.max())
                if peak <= 0:
                    continue  # no positive attention on this image; contributes nothing
                accumulated = (cam / peak).cpu().numpy() if accumulated is None \
                    else accumulated + (cam / peak).cpu().numpy()
                contributing += 1

            if accumulated is None:
                skipped[label] = "every real positive produced empty Grad-CAM attention"
                continue

            region = region_from_cam_mass(accumulated / contributing, args.mass_fraction)
            derived[label] = {"region": list(region), "n_real_positives_used": contributing}
            print(f"  {label:30s} n={contributing:4d}  region={[round(v, 3) for v in region]}", flush=True)
    finally:
        handle_f.remove()
        handle_b.remove()

    if not derived:
        raise SystemExit("No pathology yielded a derived region; refusing to write an empty artifact.")

    write_frozen_json(output_path, {
        "schema_version": 1,
        "method": "mean_gradcam_mass_box_from_real_positives_v1",
        "frozen": True,
        "split_namespace": namespace,
        "derivation_split": DERIVATION_SPLIT,
        "mass_fraction": args.mass_fraction,
        "max_images_per_label": args.max_images_per_label,
        "regions": {label: value["region"] for label, value in derived.items()},
        "n_real_positives_used": {label: value["n_real_positives_used"] for label, value in derived.items()},
        "skipped_labels": skipped,
        "interpretation": (
            "Where a classifier trained on REAL data attends for each pathology -- an "
            "explainability-plausibility reference, NOT anatomical ground truth, and it inherits "
            "any shortcut the classifier learned."
        ),
        "git_commit_hash": get_git_commit_hash(),
        **provenance,
    })
    print(f"\nDerived {len(derived)}/{len(PRIMARY_ENDPOINT_LABELS)} regions -> {output_path}")
    if skipped:
        print(f"Skipped (config fallback still applies): {skipped}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/01_compute_signals.py`

*477 lines*

**Stage 3a - compute the signals for every synthetic image.** Writes **five independent versioned
Parquet artifacts**, one per signal, rather than one shared table, so a signal can be recomputed,
audited or *excluded* by the Go/No-Go gate without disturbing the others.

- `--signal iqa` needs no GPU and no model. The other four query the frozen auxiliary classifier or
  the DINOv2 encoder and fail cleanly at that upstream gate when it is absent.
- `run_similarity` also produces **distinctiveness** internally, because it reuses the same
  synthetic embeddings - so distinctiveness costs no extra encoder passes and is not separately
  runnable.
- `require_generation_complete` + `asism_score_provenance` run first: the Stage 2 completion record,
  generation manifest hash, LoRA hash, split hash and code identity must all match.
- `validate_generation_manifest_rows` checks every manifest row before it is scored.

**Audit note:** this file's 19-line module docstring had been deleted by commit `0190fa6`, whose
message is only about adding research papers. Since the CLI builds its parser with
`description=__doc__`, `--help` printed nothing. Restored from `8b69933` -- see the audit log at the
end of this notebook.

In [ ]:
# ===== FILE: scripts/asism/01_compute_signals.py =====
"""Stage 3a — compute the five ASISM signals (docs/stages2_to_5_plan.md §4.1-§4.5).

Writes FIVE independent versioned parquet artifacts, one per signal:
    similarity_scores.parquet  iqa_scores.parquet  uncertainty_scores.parquet
    explainability_scores.parquet  agreement_scores.parquet

Independent artifacts (rather than one shared file) so a signal can be recomputed, audited, or
EXCLUDED by the Go/No-Go gate (§4.6) without disturbing the others — an excluded-but-valid signal's
artifact is retained for auditability.

IQA needs no model and runs anywhere. The other four query the frozen auxiliary classifier (§2) or
an embedding encoder, and fail cleanly at that upstream gate when it is absent.

Usage:
    python scripts/asism/01_compute_signals.py --signal all
    python scripts/asism/01_compute_signals.py --signal iqa      # no GPU required
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from tqdm.auto import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.signals import (  # noqa: E402
    aggregate_pathology_overlaps,
    compute_agreement_scores,
    compute_distinctiveness_scores,
    compute_iqa_scores,
    compute_similarity_scores,
    compute_uncertainty_scores,
    expected_region_for,
    region_overlap_score,
    write_score_artifact,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, normalize_label  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    auxiliary_checkpoint_path, config_sha256, current_code_identity_hash, namespace_identity,
    require_generation_complete, stage3_paths,
)
from scripts.utils.manifest import get_git_commit_hash, read_json, sha256_file  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402

# "distinctiveness" is produced INSIDE run_similarity (it reuses that step's synthetic
# embeddings, so it costs no extra encoder passes) and is therefore not separately runnable.
ALL_SIGNALS = ["similarity", "iqa", "uncertainty", "explainability", "agreement"]


def load_config():
    return load_named_config("stage3_asism.yaml", "stage3")


def load_synthetic_manifest(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: missing Stage 2 generation manifest at {path}\n"
            "Run scripts/generate/02_generate_synthetic_images.py --mode full first."
        )
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    if not rows:
        raise SystemExit(f"UPSTREAM GATE: generation manifest {path} is empty.")
    return pd.DataFrame(rows)


def validate_generation_manifest_rows(manifest: pd.DataFrame, generation: dict) -> None:
    if manifest["image_id"].astype(str).duplicated().any() or len(manifest) != int(generation.get("num_rows", -1)):
        raise SystemExit("Stage 2 generation manifest has duplicate IDs or disagrees with its completion row count")
    keys = (
        "split_namespace", "namespace_class", "split_manifest_hash", "recipes_csv_sha256",
        "recipes_manifest_sha256", "lora_checkpoint_sha256", "lora_metadata_sha256",
        "generation_config_sha256", "stage2_config_sha256", "code_identity_sha256",
    )
    for key in keys:
        if key not in manifest.columns or not manifest[key].map(lambda value: value == generation.get(key)).all():
            raise SystemExit(f"Stage 2 generation row provenance mismatch at {key}; ASISM refused")


def load_auxiliary_classifier(config, namespace: str, expected_provenance: dict):
    """Load the frozen auxiliary classifier (§2). Fails cleanly if Stage 3's prerequisite is absent."""
    from scripts.utils.classifier import build_model, require_torch

    resume_path = auxiliary_checkpoint_path(config, namespace)
    checkpoint_path = resume_path.with_name(f"{resume_path.stem}.best{resume_path.suffix}")
    if not checkpoint_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: auxiliary classifier not found at {checkpoint_path}\n"
            "It is a Stage 3 prerequisite (docs/stages2_to_5_plan.md §2). Train it first:\n"
            "  python scripts/classify/00_train_auxiliary_classifier.py"
        )
    torch = require_torch()
    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    checkpoint_provenance = payload.get("provenance", {})
    for key in ("split_namespace", "split_manifest_hash", "code_identity_sha256"):
        expected = expected_provenance.get(key)
        if checkpoint_provenance.get(key) != expected:
            raise SystemExit(f"Auxiliary best-checkpoint provenance mismatch at {key}: expected {expected!r}, got {checkpoint_provenance.get(key)!r}")
    aux_manifest_path = resume_path.with_suffix(".manifest.json")
    if not aux_manifest_path.is_file():
        raise SystemExit(f"Auxiliary checkpoint manifest missing: {aux_manifest_path}")
    aux_manifest = read_json(aux_manifest_path)
    if aux_manifest.get("best_checkpoint_sha256") != sha256_file(checkpoint_path):
        raise SystemExit("Auxiliary best-checkpoint hash differs from its manifest")
    model_config = payload.get("model_config", {})
    model = build_model(len(CLASSIFIER_TARGET_LABELS), float(model_config["dropout_p"]),
                        str(model_config["pretrained_source"]))
    model.load_state_dict(payload["model_state"])
    model.eval()
    return model, int(model_config["resolution"])


def synthetic_records(manifest: pd.DataFrame, images_dir: Path) -> list[dict]:
    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    records = []
    for row in manifest.to_dict("records"):
        labels = dict(row["intended_label_vector"])
        labels["No Finding"] = 1 if row.get("is_no_finding") else 0
        labels["Support Devices"] = int(row.get("support_devices", 0))
        records.append(
            {
                "image_id": row["image_id"],
                "image_path": str(images_dir / f"{row['image_id']}.jpg"),
                "labels": {label: labels.get(label) for label in CLASSIFIER_TARGET_LABELS},
                "is_synthetic": True,
            }
        )
    return records


def run_iqa(manifest, images_dir, config, provenance, scores_dir) -> None:
    rows = []
    for image_id in tqdm(manifest["image_id"], desc="iqa", unit="img"):
        scores = compute_iqa_scores(images_dir / f"{image_id}.jpg", config)
        scores["image_id"] = image_id
        rows.append(scores)
    frame = pd.DataFrame(rows)
    columns = ["image_id"] + [c for c in frame.columns if c != "image_id"]
    write_score_artifact(frame[columns], scores_dir / "iqa_scores.parquet", "iqa", provenance)
    print(f"iqa: {len(frame)} rows, {int(frame['iqa_valid'].sum())} valid", flush=True)


def run_uncertainty_and_agreement(
    manifest, images_dir, config, provenance, scores_dir, which: list[str]
) -> None:
    """Both signals come from the same auxiliary-classifier forward passes, so they share the
    (expensive) MC-Dropout inference — but they are written as SEPARATE artifacts and are gated
    independently (§4.0, §4.6). They are never merged into a single score."""
    from scripts.utils.classifier import predict_probabilities

    model, resolution = load_auxiliary_classifier(config, provenance["split_namespace"], provenance)
    records = synthetic_records(manifest, images_dir)
    passes = int(config.signals.uncertainty.mc_dropout_passes)

    mean_all, std_all = predict_probabilities(
        model, records, resolution, mc_dropout_passes=passes
    )

    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    primary_idx = [CLASSIFIER_TARGET_LABELS.index(label) for label in PRIMARY_ENDPOINT_LABELS]
    mean_primary = mean_all[:, primary_idx]
    std_primary = std_all[:, primary_idx]

    if "uncertainty" in which:
        frame = compute_uncertainty_scores(std_primary, mean_primary, config)
        frame.insert(0, "image_id", manifest["image_id"].values)
        write_score_artifact(
            frame, scores_dir / "uncertainty_scores.parquet", "uncertainty", provenance
        )
        print(f"uncertainty: {len(frame)} rows, bands={frame['uncertainty_band'].value_counts().to_dict()}", flush=True)

    if "agreement" in which:
        intended = [dict(v) for v in manifest["intended_label_vector"]]
        frame = compute_agreement_scores(mean_primary, intended, config)
        frame.insert(0, "image_id", manifest["image_id"].values)
        write_score_artifact(
            frame, scores_dir / "agreement_scores.parquet", "agreement", provenance
        )
        print(f"agreement: {len(frame)} rows, mean={frame['agreement_score'].mean():.4f}", flush=True)


def run_explainability(manifest, images_dir, config, provenance, scores_dir) -> None:
    """Grad-CAM overlap with the expected region for each recipe's intended pathologies.

    An explainability-PLAUSIBILITY signal, not proof of diagnostic correctness (§4.4).
    """
    from scripts.utils.classifier import CXRRecordDataset, require_torch

    torch = require_torch()
    model, resolution = load_auxiliary_classifier(config, provenance["split_namespace"], provenance)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    from scripts.utils.labels import CLASSIFIER_TARGET_LABELS

    # Empirically derived expected regions (00c_derive_expected_regions.py) take precedence per
    # label over the hand-entered config boxes. Absent, the config boxes still apply -- so this is
    # an upgrade, not a new hard dependency.
    derived_regions, regions_source = {}, "config_hand_entered"
    regions_path = Path(config.paths.asism_dir) / "expected_regions.json"
    if regions_path.is_file():
        derived_payload = read_json(regions_path)
        derived_regions = dict(derived_payload.get("regions", {}))
        regions_source = f"derived:{derived_payload.get('method')}"
        print(
            f"explainability: using {len(derived_regions)} empirically derived region(s); "
            "any remaining label falls back to its config box",
            flush=True,
        )
    else:
        print(
            "explainability: no derived expected_regions.json -- using hand-entered config boxes. "
            "Run 00c_derive_expected_regions.py to replace them with measured ones.",
            flush=True,
        )

    records = synthetic_records(manifest, images_dir)
    dataset = CXRRecordDataset(records, resolution=resolution)

    # Hook the last dense block: standard Grad-CAM target for DenseNet.
    activations, gradients = {}, {}
    target_layer = model.features.denseblock4

    def forward_hook(_module, _input, output):
        activations["value"] = output.detach()

    def backward_hook(_module, _grad_in, grad_out):
        gradients["value"] = grad_out[0].detach()

    handle_f = target_layer.register_forward_hook(forward_hook)
    handle_b = target_layer.register_full_backward_hook(backward_hook)

    rows = []
    try:
        for index in tqdm(range(len(dataset)), desc="explainability", unit="img"):
            item = dataset[index]
            image = item["image"].unsqueeze(0).to(device)
            intended = dict(manifest.iloc[index]["intended_label_vector"])
            positives = [l for l in PRIMARY_ENDPOINT_LABELS if int(intended.get(l, 0)) == 1]
            target_labels = positives or ["__max_disease__"]
            per_label_overlap = {}
            cam_maxima = []
            for target_label in target_labels:
                model.zero_grad(set_to_none=True)
                logits = model(image)
                if target_label == "__max_disease__":
                    disease_idx = [CLASSIFIER_TARGET_LABELS.index(l) for l in PRIMARY_ENDPOINT_LABELS]
                    score = logits[0, disease_idx].max()
                    region = expected_region_for({}, config, derived_regions)
                else:
                    score = logits[0, CLASSIFIER_TARGET_LABELS.index(target_label)]
                    single_intent = {label: int(label == target_label) for label in PRIMARY_ENDPOINT_LABELS}
                    region = expected_region_for(single_intent, config, derived_regions)
                score.backward()
                weights = gradients["value"].mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((weights * activations["value"]).sum(dim=1)).squeeze(0).cpu().numpy()
                per_label_overlap[target_label] = region_overlap_score(cam, region)
                cam_maxima.append(float(cam.max()))
            # Frozen aggregation: every intended positive contributes equally; mean overlap is the
            # primary score and the minimum is retained to expose a missed co-pathology.
            overlap, minimum_overlap = aggregate_pathology_overlaps(per_label_overlap)
            rows.append(
                {
                    "image_id": manifest.iloc[index]["image_id"],
                    "explainability_region_overlap": overlap,
                    "explainability_min_pathology_overlap": minimum_overlap,
                    "explainability_target_labels": json.dumps(target_labels),
                    "explainability_per_label_overlap": json.dumps(per_label_overlap, sort_keys=True),
                    "explainability_aggregation": "mean_over_all_intended_positive_pathologies",
                    "explainability_cam_max": max(cam_maxima),
                    "explainability_is_plausibility_signal_only": True,
                }
            )
    finally:
        handle_f.remove()
        handle_b.remove()

    frame = pd.DataFrame(rows)
    write_score_artifact(
        frame, scores_dir / "explainability_scores.parquet", "explainability",
        {**provenance, "expected_regions_source": regions_source,
         "n_derived_regions_used": len(derived_regions)},
    )
    print(f"explainability: {len(frame)} rows, mean overlap={frame['explainability_region_overlap'].mean():.4f}", flush=True)


def run_similarity(manifest, images_dir, config, provenance, scores_dir, namespace) -> None:
    """DINOv2 embeddings for synthetic images and a stratified real-reference sample."""
    from scripts.utils.classifier import require_torch
    from scripts.utils.identifiers import sanitize_image_id

    torch = require_torch()
    try:
        import timm
    except ImportError as exc:
        raise SystemExit(
            f"UPSTREAM GATE: the similarity encoder needs `timm` ({exc}).\n"
            "Install it on the pod: pip install timm"
        ) from exc

    stage1_cfg = load_stage1_config()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Never let timm resolve a mutable default checkpoint. Download the exact reviewed revision,
    # then load it into the explicitly named architecture.
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file as load_safetensors

    similarity_cfg = config.signals.similarity
    weights_path = hf_hub_download(
        repo_id=str(similarity_cfg.weights_repo_id),
        filename=str(similarity_cfg.weights_filename),
        revision=str(similarity_cfg.weights_revision),
    )
    encoder = timm.create_model(str(similarity_cfg.encoder), pretrained=False, num_classes=0)
    incompatible = encoder.load_state_dict(load_safetensors(weights_path), strict=False)
    unexpected = list(incompatible.unexpected_keys)
    disallowed_missing = [key for key in incompatible.missing_keys if not key.startswith("head.")]
    if unexpected or disallowed_missing:
        raise RuntimeError(
            "Pinned DINOv2 checkpoint is incompatible with the configured architecture: "
            f"missing={disallowed_missing}, unexpected={unexpected}"
        )
    encoder = encoder.eval().to(device)
    data_config = timm.data.resolve_model_data_config(encoder)
    transform = timm.data.create_transform(**data_config, is_training=False)

    from PIL import Image

    def embed(paths: list[Path]) -> np.ndarray:
        outputs = []
        batch_size = int(config.signals.similarity.batch_size)
        for start in tqdm(range(0, len(paths), batch_size), desc="embed", unit="batch"):
            batch = []
            for path in paths[start : start + batch_size]:
                with Image.open(path) as image:
                    batch.append(transform(image.convert("RGB")))
            with torch.no_grad():
                outputs.append(encoder(torch.stack(batch).to(device)).cpu().numpy())
        return np.concatenate(outputs, axis=0)

    # Real reference pool: a stratified sample of gen_train (the generator's own training data is
    # the correct realism reference; classifier_train is reserved for A/B/C).
    real_frame = load_split("gen_train", namespace, purpose="schema_validation", caller="asism_similarity")
    per_label = int(config.signals.similarity.reference_sample_per_label)
    real_images_dir = Path(stage1_cfg.paths.images_dir) / namespace / "gen_train"

    selected_rows, seen = [], set()
    for label in PRIMARY_ENDPOINT_LABELS:
        subset = real_frame[real_frame[label].map(normalize_label) == 1].head(per_label)
        for row in subset.to_dict("records"):
            key = row["Path"]
            if key not in seen:
                seen.add(key)
                selected_rows.append(row)
    # Plus negatives, so the class-agnostic fallback tier is not all-pathology.
    for row in real_frame.head(per_label).to_dict("records"):
        if row["Path"] not in seen:
            seen.add(row["Path"])
            selected_rows.append(row)

    reference_paths, reference_vectors = [], []
    for row in selected_rows:
        path = real_images_dir / f"{sanitize_image_id(row['Path'])}.jpg"
        if not path.is_file():
            continue
        reference_paths.append(path)
        reference_vectors.append(
            frozenset(l for l in PRIMARY_ENDPOINT_LABELS if normalize_label(row.get(l)) == 1)
        )

    if not reference_paths:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed gen_train images under {real_images_dir}; "
            "Stage 1 preprocessing must run before the similarity signal."
        )

    synthetic_paths = [images_dir / f"{image_id}.jpg" for image_id in manifest["image_id"]]
    print(f"similarity: embedding {len(synthetic_paths)} synthetic, {len(reference_paths)} real", flush=True)

    # Embed ONCE and reuse. The distinctiveness signal (§4.6) is computed from these same synthetic
    # embeddings, so it adds no encoder passes at all -- only a per-class matrix multiply.
    synthetic_embeddings = embed(synthetic_paths)
    intended_vectors = [dict(v) for v in manifest["intended_label_vector"]]

    frame = compute_similarity_scores(
        synthetic_embeddings,
        embed(reference_paths),
        reference_vectors,
        intended_vectors,
        config,
    )
    frame.insert(0, "image_id", manifest["image_id"].values)
    write_score_artifact(frame, scores_dir / "similarity_scores.parquet", "similarity", provenance)
    print(
        f"similarity: {len(frame)} rows, near-duplicates flagged="
        f"{int(frame['novelty_is_near_duplicate'].sum())}",
        flush=True,
    )

    distinct = compute_distinctiveness_scores(synthetic_embeddings, intended_vectors, config)
    distinct.insert(0, "image_id", manifest["image_id"].values)
    write_score_artifact(
        distinct, scores_dir / "distinctiveness_scores.parquet", "distinctiveness", provenance
    )
    defined = distinct.loc[~distinct["distinctiveness_is_undefined"]]
    print(
        f"distinctiveness: {len(distinct)} rows, "
        f"{int(distinct['distinctiveness_is_undefined'].sum())} undefined (single-image class), "
        f"mean={defined['distinctiveness_score'].mean():.4f} "
        f"images-with-a-duplicate={int((defined['distinctiveness_n_duplicates'] > 0).sum())}",
        flush=True,
    )


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--signal", default="all", choices=ALL_SIGNALS + ["all"])
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_config()
    namespace = args.namespace or str(config.split_namespace)
    identity = namespace_identity(namespace)

    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")

    stage2_artifacts, generation = require_generation_complete(stage2_cfg, namespace)
    images_dir = stage2_artifacts["images_dir"]
    manifest = load_synthetic_manifest(stage2_artifacts["manifest_path"])
    validate_generation_manifest_rows(manifest, generation)
    scores_dir = stage3_paths(config, namespace)["scores_dir"]
    scores_dir.mkdir(parents=True, exist_ok=True)

    artifact_provenance = {
        **identity,
        "asism_config_sha256": config_sha256(config),
        "generation_manifest_sha256": sha256_file(stage2_artifacts["manifest_path"]),
        "generation_completion_sha256": sha256_file(stage2_artifacts["generation_completion"]),
        "lora_checkpoint_sha256": generation["lora_checkpoint_sha256"],
        "code_identity_sha256": current_code_identity_hash(),
        "git_commit_hash": get_git_commit_hash(),
        "n_synthetic_images": len(manifest),
    }

    which = ALL_SIGNALS if args.signal == "all" else [args.signal]

    if "iqa" in which:
        run_iqa(manifest, images_dir, config, artifact_provenance, scores_dir)
    if {"uncertainty", "agreement"} & set(which):
        run_uncertainty_and_agreement(
            manifest, images_dir, config, artifact_provenance, scores_dir, which
        )
    if "explainability" in which:
        run_explainability(manifest, images_dir, config, artifact_provenance, scores_dir)
    if "similarity" in which:
        run_similarity(manifest, images_dir, config, artifact_provenance, scores_dir, namespace)

    print(f"\nArtifacts -> {scores_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/02_gonogo.py`

*447 lines*

**Stage 3b - the Go/No-Go gate. The rule this script exists to enforce: a signal that fails its
gate is not forced into the final selector merely because it was implemented.**

Runs **before** any weights or thresholds are tuned. Each signal faces seven checks:

| Check | Question |
|---|---|
| `check_technical_validity` | is the artifact well-formed, with the expected columns? |
| `check_missing_rate` | how many images failed to score? |
| `check_numerical_stability` | enough distinct values; no degenerate constants or NaN storms |
| `check_directionality` | does the score move the way the design says it should? |
| `check_reproducibility` | does recomputing it give the same answer? |
| `check_redundancy` | is it just a copy of another signal (correlation)? |
| `check_usefulness` | does it separate anything downstream? |

`decide(checks)` then returns one of three verdicts:

- **`include`** - passed everything; eligible for the frozen selector.
- **`ablation_only`** - technically valid but weak or near-duplicate; retained for the
  leave-one-signal-out analysis, excluded from the selector.
- **`exclude`** - dropped.

Frozen into `gonogo_report.json` / `asism_frozen_manifest.json`. Everything downstream reads
`surviving_signals` from that report - which is how a rejected signal's feature columns disappear
from the learned ranker automatically.

In [ ]:
# ===== FILE: scripts/asism/02_gonogo.py =====
#!/usr/bin/env python3
"""Stage 3b — ASISM component Go/No-Go gate (docs/stages2_to_5_plan.md §4.6).

Runs BEFORE weights/thresholds are tuned and frozen. Each of the six signals is checked for
technical validity, score directionality, numerical stability, reproducibility, missing-output
rate, redundancy with the other signals, and downstream usefulness.

The rule this script exists to enforce: **a signal that fails its gate is not forced into the final
selector merely because it was implemented.** Outcomes are one of:

    include        - passed everything; eligible for the frozen selector
    ablation_only  - technically valid but weak or near-duplicate of another signal; retained for
                     the leave-one-signal-out analysis, not weighted into the selector
    exclude        - technically invalid or unstable; removed from the selector, artifact retained
                     for audit if it is at least readable, exclusion + reason recorded

Evidence comes from asism_tuning_heldout-derived artifacts only. final_eval_heldout is never read.

Usage:
    python scripts/asism/02_gonogo.py
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, read_json, write_json  # noqa: E402
from scripts.utils.artifact_contracts import asism_score_provenance, require_score_artifact, stage3_paths  # noqa: E402

# Each signal's headline score column, and whether higher is better.
PRIMARY_SCORE_COLUMN = {
    "similarity": ("similarity_knn_mean", True),
    "iqa": ("iqa_composite", True),
    "uncertainty": ("uncertainty_mean_std", None),   # None = no a-priori direction (§4.3)
    "explainability": ("explainability_region_overlap", True),
    "agreement": ("agreement_score", True),
    "distinctiveness": ("distinctiveness_score", True),
}

ARTIFACT_FILENAME = {name: f"{name}_scores.parquet" for name in PRIMARY_SCORE_COLUMN}


def check_technical_validity(frame: pd.DataFrame, signal: str) -> dict:
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    has_ids = "image_id" in frame.columns
    has_score = column in frame.columns
    return {
        "passed": bool(has_ids and has_score and len(frame) > 0),
        "has_image_id": has_ids,
        "has_primary_score_column": has_score,
        "n_rows": len(frame),
        "primary_score_column": column,
    }


def check_numerical_stability(values: np.ndarray, config) -> dict:
    finite = np.isfinite(values)
    n_unique = int(len(np.unique(values[finite]))) if finite.any() else 0
    min_unique = int(config.gonogo.min_unique_values)
    non_finite_fraction = float(1.0 - finite.mean()) if len(values) else 1.0
    return {
        "passed": bool(non_finite_fraction == 0.0 and n_unique >= min_unique),
        "non_finite_fraction": non_finite_fraction,
        "n_unique_values": n_unique,
        "min_unique_required": min_unique,
        "std": float(np.std(values[finite])) if finite.any() else float("nan"),
        "note": (
            "A near-constant score carries no selection information even if it computes cleanly."
        ),
    }


def check_missing_rate(frame: pd.DataFrame, signal: str, config) -> dict:
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    values = frame[column].to_numpy(dtype=np.float64, na_value=np.nan)
    missing_fraction = float(np.isnan(values).mean()) if len(values) else 1.0
    # IQA additionally reports explicit per-image validity.
    if signal == "iqa" and "iqa_valid" in frame.columns:
        missing_fraction = float(1.0 - frame["iqa_valid"].astype(bool).mean())
    threshold = float(config.gonogo.max_missing_fraction)
    return {
        "passed": bool(missing_fraction <= threshold),
        "missing_fraction": missing_fraction,
        "max_allowed": threshold,
    }


def check_directionality(frame: pd.DataFrame, signal: str) -> dict:
    """Does the score move the way the methodology claims it does?

    Each check uses an internal contrast whose expected direction is known a priori, rather than
    assuming the sign is right because the formula looks right.
    """
    result = {"passed": None, "method": "", "detail": {}}

    if signal == "iqa":
        # Images the domain checks flagged as near-uniform must score BELOW unflagged ones.
        if "iqa_is_near_uniform" in frame.columns and frame["iqa_is_near_uniform"].any():
            flagged = frame.loc[frame["iqa_is_near_uniform"].astype(bool), "iqa_composite"].mean()
            clean = frame.loc[~frame["iqa_is_near_uniform"].astype(bool), "iqa_composite"].mean()
            result.update(
                passed=bool(flagged < clean),
                method="near_uniform_images_score_lower_than_clean",
                detail={"flagged_mean": float(flagged), "clean_mean": float(clean)},
            )
        else:
            result.update(
                passed=True,
                method="no_flagged_images_in_sample",
                detail={"note": "No near-uniform images present; directionality not contradicted."},
            )

    elif signal == "agreement":
        # No-Finding recipes and disease recipes must not be degenerate: the score must vary, and
        # the unintended-penalty must actually reduce the score where it fires.
        if "agreement_n_confident_unintended" in frame.columns:
            penalized = frame["agreement_n_confident_unintended"] > 0
            if penalized.any() and (~penalized).any():
                with_penalty = frame.loc[penalized, "agreement_score"].mean()
                without = frame.loc[~penalized, "agreement_score"].mean()
                result.update(
                    passed=bool(with_penalty < without),
                    method="confident_unintended_predictions_lower_the_score",
                    detail={"penalized_mean": float(with_penalty), "clean_mean": float(without)},
                )
            else:
                result.update(passed=True, method="penalty_never_fired_in_sample", detail={})
        else:
            result.update(passed=False, method="missing_penalty_column", detail={})

    elif signal == "similarity":
        # Flagged near-duplicates must receive no novelty credit.
        if "novelty_is_near_duplicate" in frame.columns and frame["novelty_is_near_duplicate"].any():
            flagged_novelty = frame.loc[
                frame["novelty_is_near_duplicate"].astype(bool), "novelty_score"
            ].max()
            result.update(
                passed=bool(flagged_novelty == 0.0),
                method="memorized_images_receive_zero_novelty",
                detail={"max_novelty_among_flagged": float(flagged_novelty)},
            )
        else:
            result.update(passed=True, method="no_near_duplicates_in_sample", detail={})

    elif signal == "explainability":
        # Overlap is a fraction of activation mass: it must lie in [0, 1].
        values = frame["explainability_region_overlap"].dropna()
        in_range = bool(((values >= 0.0) & (values <= 1.0)).all()) if len(values) else False
        result.update(
            passed=in_range,
            method="region_overlap_is_a_valid_fraction",
            detail={"min": float(values.min()) if len(values) else None,
                    "max": float(values.max()) if len(values) else None},
        )

    elif signal == "distinctiveness":
        # Images with at least one within-class near-duplicate must score LOWER than images with
        # none — a duplicate cluster is redundancy the score is specifically meant to catch.
        if "distinctiveness_n_duplicates" in frame.columns:
            defined = (
                frame.loc[~frame["distinctiveness_is_undefined"].astype(bool)]
                if "distinctiveness_is_undefined" in frame.columns else frame
            )
            has_duplicate = defined["distinctiveness_n_duplicates"] > 0
            if has_duplicate.any() and (~has_duplicate).any():
                with_duplicates = defined.loc[has_duplicate, "distinctiveness_score"].mean()
                without_duplicates = defined.loc[~has_duplicate, "distinctiveness_score"].mean()
                result.update(
                    passed=bool(with_duplicates < without_duplicates),
                    method="images_with_duplicates_score_lower_than_those_without",
                    detail={
                        "with_duplicates_mean": float(with_duplicates),
                        "without_duplicates_mean": float(without_duplicates),
                    },
                )
            else:
                result.update(
                    passed=True,
                    method="no_duplicate_contrast_in_sample",
                    detail={"note": "No within-class duplicates present; directionality not contradicted."},
                )
        else:
            result.update(passed=False, method="missing_n_duplicates_column", detail={})

    elif signal == "uncertainty":
        # §4.3 forbids an a-priori direction: high uncertainty is NOT defined as bad. The check is
        # that the bands are populated and ordered, not that the score correlates with quality.
        if "uncertainty_band" in frame.columns:
            counts = frame["uncertainty_band"].value_counts().to_dict()
            result.update(
                passed=bool(len(counts) >= 2),
                method="bands_are_populated_no_direction_asserted",
                detail={"band_counts": {str(k): int(v) for k, v in counts.items()}},
            )
        else:
            result.update(passed=False, method="missing_band_column", detail={})

    return result


def check_reproducibility(scores_dir: Path, signal: str, config) -> dict:
    """Compare the artifact against its recorded provenance fingerprint.

    Full recomputation is a GPU-cost decision for the operator; what is enforced here is that the
    artifact carries the provenance needed to reproduce it and that its row count matches.
    """
    sidecar = scores_dir / f"{signal}_scores.provenance.json"
    if not sidecar.is_file():
        return {"passed": False, "reason": "missing_provenance_sidecar"}
    provenance = read_json(sidecar)
    required = {"schema_version", "signal", "n_rows", "asism_config_sha256", "git_commit_hash"}
    missing = sorted(required - set(provenance))
    return {
        "passed": bool(not missing),
        "missing_provenance_fields": missing,
        "recorded_n_rows": provenance.get("n_rows"),
        "asism_config_sha256": provenance.get("asism_config_sha256"),
    }


def check_redundancy(merged: pd.DataFrame, signal: str, config) -> dict:
    """Correlation of this signal's headline score with the other signals'.

    A signal that is a near-linear restatement of another adds little independent information, and
    weighting both would double-count the same evidence.
    """
    column, _ = PRIMARY_SCORE_COLUMN[signal]
    threshold = float(config.gonogo.redundancy_abs_correlation_max)

    correlations = {}
    for other, (other_column, _) in PRIMARY_SCORE_COLUMN.items():
        if other == signal or other_column not in merged.columns or column not in merged.columns:
            continue
        pair = merged[[column, other_column]].dropna()
        if len(pair) < 10 or pair[column].std() == 0 or pair[other_column].std() == 0:
            continue
        correlations[other] = float(pair[column].corr(pair[other_column]))

    worst = max(correlations.items(), key=lambda item: abs(item[1]), default=(None, 0.0))
    return {
        "passed": bool(abs(worst[1]) < threshold),
        "correlations": correlations,
        "most_correlated_with": worst[0],
        "max_abs_correlation": abs(worst[1]),
        "threshold": threshold,
    }


def check_usefulness(merged: pd.DataFrame, signal: str) -> dict:
    """Does including this signal change which images a simple equal-weight selector would keep?

    Compares the top-50% selection under an equal-weight composite WITH and WITHOUT this signal. A
    signal that changes nothing about the selection cannot influence downstream results, whatever
    its intrinsic quality.
    """
    available = [
        (name, col)
        for name, (col, direction) in PRIMARY_SCORE_COLUMN.items()
        if col in merged.columns and direction is True
    ]
    if len(available) < 2:
        return {"passed": True, "reason": "insufficient_signals_to_compare", "jaccard": None}

    def composite(names: list[str]) -> pd.Series:
        parts = []
        for _, col in [(n, c) for n, c in available if n in names]:
            values = merged[col].astype(float)
            spread = values.max() - values.min()
            parts.append((values - values.min()) / spread if spread > 0 else values * 0.0)
        return sum(parts) / max(len(parts), 1)

    all_names = [name for name, _ in available]
    if signal not in all_names:
        return {"passed": True, "reason": "signal_has_no_directional_score", "jaccard": None}
    without_names = [name for name in all_names if name != signal]

    n_keep = max(1, len(merged) // 2)
    with_top = set(composite(all_names).nlargest(n_keep).index)
    without_top = set(composite(without_names).nlargest(n_keep).index)

    intersection = len(with_top & without_top)
    union = len(with_top | without_top)
    jaccard = intersection / union if union else 1.0

    return {
        "passed": bool(jaccard < 0.99),
        "jaccard_overlap_of_selection": jaccard,
        "n_selected": n_keep,
        "note": "Jaccard ~1.0 means the signal does not change the selection at all.",
    }


def decide(checks: dict) -> tuple[str, str]:
    """Map check results to one of include / ablation_only / exclude (§4.6)."""
    hard_failures = [
        name
        for name in ("technical_validity", "numerical_stability", "missing_rate", "reproducibility")
        if checks[name].get("passed") is False
    ]
    if hard_failures:
        return "exclude", f"failed hard checks: {', '.join(hard_failures)}"

    if checks["directionality"].get("passed") is False:
        return "exclude", "score direction contradicts the methodology's stated meaning"

    soft_failures = []
    if checks["redundancy"].get("passed") is False:
        soft_failures.append(
            f"redundant with {checks['redundancy'].get('most_correlated_with')} "
            f"(|r|={checks['redundancy'].get('max_abs_correlation'):.3f})"
        )
    if checks["usefulness"].get("passed") is False:
        soft_failures.append("does not change selection")

    if soft_failures:
        return "ablation_only", "; ".join(soft_failures)
    return "include", "passed all checks"


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--scores-dir", default=None)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_named_config("stage3_asism.yaml", "stage3")

    namespace = args.namespace or str(config.split_namespace)
    stage2_config = load_named_config("stage2_generation.yaml", "stage2")
    expected_provenance = asism_score_provenance(config, stage2_config, namespace)
    paths = stage3_paths(config, namespace)
    scores_dir = Path(args.scores_dir) if args.scores_dir else paths["scores_dir"]

    frames: dict[str, pd.DataFrame] = {}
    absent: list[str] = []
    for signal, filename in ARTIFACT_FILENAME.items():
        path = scores_dir / filename
        if path.is_file():
            require_score_artifact(path, signal, expected_provenance)
            frames[signal] = pd.read_parquet(path)
        else:
            absent.append(signal)

    if not frames:
        raise SystemExit(
            f"UPSTREAM GATE: no ASISM score artifacts found in {scores_dir}\n"
            "Run scripts/asism/01_compute_signals.py first."
        )

    merged = None
    for signal, frame in frames.items():
        column, _ = PRIMARY_SCORE_COLUMN[signal]
        subset = frame[["image_id", column]] if column in frame.columns else frame[["image_id"]]
        merged = subset if merged is None else merged.merge(subset, on="image_id", how="outer")
    merged = merged.set_index("image_id")

    report = {
        "stage": "asism_gonogo",
        "scores_dir": str(scores_dir),
        "git_commit_hash": get_git_commit_hash(),
        "signals_present": sorted(frames),
        "signals_absent": sorted(absent),
        "evidence_split": "asism_tuning_heldout only; final_eval_heldout never read",
        "per_signal": {},
    }

    for signal, frame in frames.items():
        column, _ = PRIMARY_SCORE_COLUMN[signal]
        values = (
            frame[column].to_numpy(dtype=np.float64, na_value=np.nan)
            if column in frame.columns
            else np.array([])
        )

        checks = {
            "technical_validity": check_technical_validity(frame, signal),
            "numerical_stability": check_numerical_stability(values, config),
            "missing_rate": check_missing_rate(frame, signal, config),
            "reproducibility": check_reproducibility(scores_dir, signal, config),
            "directionality": check_directionality(frame, signal),
            "redundancy": check_redundancy(merged, signal, config),
            "usefulness": check_usefulness(merged, signal),
        }
        outcome, reason = decide(checks)
        report["per_signal"][signal] = {
            "outcome": outcome,
            "reason": reason,
            "checks": checks,
            "artifact_retained_for_audit": True,
        }

    for signal in absent:
        report["per_signal"][signal] = {
            "outcome": "exclude",
            "reason": "artifact not produced",
            "checks": {},
            "artifact_retained_for_audit": False,
        }

    included = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "include")
    ablation = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "ablation_only")
    excluded = sorted(s for s, r in report["per_signal"].items() if r["outcome"] == "exclude")

    report["surviving_signals"] = included
    report["ablation_only_signals"] = ablation
    report["excluded_signals"] = excluded
    report["asism_variant_status"] = (
        "primary" if len(included) >= 3
        else "reduced_variant — fewer than 3 signals survived; the resulting selector is reported "
             "as an alternative/ablation rather than the primary method (§4.6)"
    )

    report.update(expected_provenance)
    write_json(paths["gonogo_report"], report)

    print("ASISM Go/No-Go", flush=True)
    print("=" * 60, flush=True)
    for signal in sorted(report["per_signal"]):
        entry = report["per_signal"][signal]
        print(f"  {signal:<16} {entry['outcome']:<14} {entry['reason']}", flush=True)
    print("=" * 60, flush=True)
    print(f"  include:       {included}", flush=True)
    print(f"  ablation only: {ablation}", flush=True)
    print(f"  exclude:       {excluded}", flush=True)
    print(f"  variant:       {report['asism_variant_status']}", flush=True)
    print(f"\nReport -> {config.paths.gonogo_report}", flush=True)

    if not included:
        print(
            "\nNo signal survived the gate — the selector cannot be frozen. Investigate the "
            "failing checks before tuning.",
            flush=True,
        )
        return 2
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 7 - Stage 3, the superseded weighted-score selector

Kept for the historical record and for its engineering patterns. **Not a pipeline step** - nothing downstream consumes its output.

## `scripts/asism/03_tune_freeze_select.py`

*677 lines*

**NOT A PIPELINE STEP.** This is the *pre-learned* ASISM: a weighted-score selector that
normalises each signal, combines them with a weight vector (`composite_score`), searches the weight
grid in two bounded stages (coarse screening -> shortlist validation) with a proxy classifier
(`run_proxy_trial`), then freezes the winning weights and selects images.

It is historically important - it is the baseline the learned approach replaced - and its
engineering is worth reading: a compute-budget gate that must pass *before* any results are seen
(`enforce_compute_budget`), a resumable append-only search log (`append_search_log`,
`completed_trials`), and evaluation restricted to `asism_tuning_heldout` only.

But **nothing in the Stage 4/5 pipeline consumes its output**, and `run_dev_subset.sh` does not run
it. The `README` used to claim it was "retained for its shared signal-merge helpers only"; that was
stale -- the shared merge lives in `scripts/asism/candidate_pool.py`, and nothing imports
`load_merged_scores` except this file's own tests. See the audit log at the end.

In [ ]:
# ===== FILE: scripts/asism/03_tune_freeze_select.py =====
#!/usr/bin/env python3
"""Stage 3c — ASISM bounded tuning, freeze, and selection (docs/stages2_to_5_plan.md §4.7, §4.8).

Three phases, each independently invocable:

  --phase estimate   Compute the proxy-run count and GPU-hour estimate and apply the compute-budget
                     gate. Runs no training. This is the gate that must pass BEFORE any results are
                     seen — if the estimate is over budget, the search space is reduced here, never
                     after observing performance.

  --phase tune       Two-stage bounded search (coarse screening -> shortlist validation). Proxy
                     trains on classifier_train + the ACTUALLY-CONSTRUCTED candidate synthetic
                     subset and evaluates on asism_tuning_heldout ONLY. Fixed optimizer-step budget,
                     no candidate-specific early stopping. classifier_val is not used to rank
                     candidates. final_eval_heldout is never touched. Resumable at trial granularity.

  --phase select     Apply the frozen configuration: quality floor FIRST, then class quota. Writes
                     selected_manifest.jsonl and rejected_log.jsonl with a reason per rejected image.

NOT PART OF THE THESIS PIPELINE (docs/stages2_to_5_plan.md §4.9 / §7 v3 revision notes). This is the
weighted-score selector, which predates the learned ASISM components. The thesis defines ASISM as the
full module including the Multi-Signal Utility Ranking Network and Adaptive Threshold Learning, so there
is no weighted-baseline Stage 4 condition and nothing consumes this script's selected_manifest.jsonl.
Retained for reference and for the Stage 3 signal-merge helpers other code still imports.

Usage:
    python scripts/asism/03_tune_freeze_select.py --phase estimate
    python scripts/asism/03_tune_freeze_select.py --phase tune
    python scripts/asism/03_tune_freeze_select.py --phase select
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import tempfile
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.signals import SCHEMA_VERSION  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS, normalize_label  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, write_frozen_json, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    asism_score_provenance, require_score_artifact, require_generation_complete,
    stage2_paths, stage3_paths,
)

# Directional signals eligible for the weighted composite (uncertainty is banded, not directional).
DIRECTIONAL_SCORES = {
    "similarity": "similarity_knn_mean",
    "iqa": "iqa_composite",
    "explainability": "explainability_region_overlap",
    "agreement": "agreement_score",
}


def load_config():
    return load_named_config("stage3_asism.yaml", "stage3")


def compute_budget_estimate(config) -> dict:
    """Pre-execution accounting (§4.7). Must be computed and checked before any trial runs."""
    tuning = config.tuning
    coarse_runs = int(tuning.coarse.trials) * int(tuning.coarse.folds) * int(tuning.coarse.seeds)
    shortlist_runs = (
        int(tuning.shortlist.size) * int(tuning.shortlist.folds) * int(tuning.shortlist.seeds)
    )
    total_runs = coarse_runs + shortlist_runs
    hours_each = float(tuning.compute_budget.hours_per_proxy_run_estimate)
    estimated_hours = total_runs * hours_each
    max_hours = float(tuning.compute_budget.max_gpu_hours)

    return {
        "coarse_runs": coarse_runs,
        "shortlist_runs": shortlist_runs,
        "total_proxy_runs": total_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
        "formula": (
            "total = coarse_trials*coarse_folds*coarse_seeds "
            "+ shortlist_size*shortlist_folds*shortlist_seeds"
        ),
    }


def enforce_compute_budget(config) -> dict:
    estimate = compute_budget_estimate(config)
    if not estimate["within_budget"]:
        raise SystemExit(
            "COMPUTE BUDGET GATE: the ASISM search exceeds its declared budget.\n"
            f"  estimated: {estimate['estimated_gpu_hours']} GPU-hours "
            f"({estimate['total_proxy_runs']} proxy runs x "
            f"{estimate['hours_per_proxy_run_estimate']}h)\n"
            f"  budget:    {estimate['max_gpu_hours']} GPU-hours\n\n"
            "Reduce the search space in configs/stage3_asism.yaml (tuning.coarse.trials, "
            "shortlist.size, folds, or seeds) BEFORE running.\n"
            "Reducing it after seeing results would let the outcome influence the search design "
            "(docs/stages2_to_5_plan.md §4.7)."
        )
    return estimate


def load_merged_scores(config, surviving: list[str]) -> pd.DataFrame:
    """Merge the surviving signals' artifacts by image_id (§4.0)."""
    scores_dir = Path(config.paths.scores_dir)
    merged = None
    for signal in surviving:
        path = scores_dir / f"{signal}_scores.parquet"
        if not path.is_file():
            raise SystemExit(f"Surviving signal {signal!r} has no artifact at {path}")
        require_score_artifact(path, signal, config._expected_score_provenance)
        frame = pd.read_parquet(path)
        merged = frame if merged is None else merged.merge(frame, on="image_id", how="inner")

    # Uncertainty is always merged when present: §4.8 uses its band even though it is not part of
    # the directional composite.
    uncertainty_path = scores_dir / "uncertainty_scores.parquet"
    if uncertainty_path.is_file() and "uncertainty_band" not in (merged.columns if merged is not None else []):
        require_score_artifact(uncertainty_path, "uncertainty", config._expected_score_provenance)
        uncertainty = pd.read_parquet(uncertainty_path)
        merged = uncertainty if merged is None else merged.merge(uncertainty, on="image_id", how="left")

    if merged is None or merged.empty:
        raise SystemExit("No ASISM scores available to merge.")
    return merged


def normalize_series(values: pd.Series) -> pd.Series:
    values = values.astype(float)
    low, high = values.min(), values.max()
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - low) / (high - low)


def composite_score(frame: pd.DataFrame, weights: dict[str, float]) -> pd.Series:
    total_weight = sum(weights.values())
    if total_weight <= 0:
        return pd.Series(np.zeros(len(frame)), index=frame.index)
    score = pd.Series(np.zeros(len(frame)), index=frame.index)
    for signal, weight in weights.items():
        column = DIRECTIONAL_SCORES.get(signal)
        if weight <= 0 or column is None or column not in frame.columns:
            continue
        score = score + weight * normalize_series(frame[column])
    return score / total_weight


def candidate_weight_grid(surviving: list[str], n_trials: int, seed: int) -> list[dict[str, float]]:
    """Coarse grid over weights for the directional surviving signals (§4.7 Stage A).

    Deliberately coarse: fine-grained optimization against asism_tuning_heldout is exactly the
    overfitting risk the bounded-search design exists to limit.
    """
    directional = [s for s in surviving if s in DIRECTIONAL_SCORES]
    if not directional:
        return [{}]

    levels = [0.0, 0.5, 1.0]
    combinations = [
        dict(zip(directional, values))
        for values in product(levels, repeat=len(directional))
        if sum(values) > 0
    ]
    equal = {signal: 1.0 for signal in directional}
    required = [equal]
    required += [{name: (1.0 if name == signal else 0.0) for name in directional} for signal in directional]
    if "uncertainty" in surviving:
        # Uncertainty is intentionally banded rather than treated as monotonically good/bad. This
        # baseline therefore ranks only by the frozen preferred-band bonus inside class quotas.
        required.append({**{name: 0.0 for name in directional}, "__uncertainty_band_only__": 1.0})
    if len(directional) > 1:
        required += [{name: (0.0 if name == omitted else 1.0) for name in directional} for omitted in directional]
    # De-duplicate while retaining deterministic baseline order.
    required = list({tuple(sorted(item.items())): item for item in required}.values())
    if n_trials < len(required):
        raise ValueError(f"coarse.trials={n_trials} cannot cover {len(required)} required equal/single-signal/leave-one-out baselines")

    rng = np.random.default_rng(seed)
    if len(combinations) > n_trials:
        rest = [c for c in combinations if c not in required]
        chosen = rng.choice(len(rest), size=n_trials - len(required), replace=False)
        combinations = required + [rest[i] for i in chosen]
    else:
        combinations = required + [c for c in combinations if c not in required]
    return combinations[:n_trials]


def select_images(
    frame: pd.DataFrame,
    weights: dict[str, float],
    config,
    intended_by_id: dict[str, dict],
) -> tuple[list[str], list[dict]]:
    """§4.8 ordered policy: quality floor FIRST, then per-label quota.

    A poor image is never accepted merely because its pathology is rare — rarity changes how many
    images compete for a quota, never whether the floor applies.
    """
    selection_cfg = config.selection
    scored = frame.copy()
    scored["composite"] = composite_score(scored, weights)

    rejected: list[dict] = []

    # ---- Step 1: absolute quality floor, applied to everything.
    floor_value = np.percentile(scored["composite"], float(selection_cfg.quality_floor_percentile))
    below_floor = scored["composite"] < floor_value
    for image_id in scored.loc[below_floor, "image_id"]:
        rejected.append({"image_id": image_id, "reason": "below_quality_floor"})

    # Memorized near-duplicates are rejected outright: they add no new information and carry a
    # patient-privacy risk that no quota should be able to override.
    eligible = scored.loc[~below_floor].copy()
    if "novelty_is_near_duplicate" in eligible.columns:
        memorized = eligible["novelty_is_near_duplicate"].astype(bool)
        for image_id in eligible.loc[memorized, "image_id"]:
            rejected.append({"image_id": image_id, "reason": "near_duplicate_memorization"})
        eligible = eligible.loc[~memorized]

    # ---- Step 2: per-label quota among floor-passing images.
    preference = str(selection_cfg.uncertainty_band_preference)
    if preference != "any" and "uncertainty_band" in eligible.columns:
        # Preference re-ranks within the quota; it never overrides the floor.
        eligible["band_bonus"] = (eligible["uncertainty_band"] == preference).astype(float) * 0.05
        eligible["composite"] = eligible["composite"] + eligible["band_bonus"]

    min_per_label = int(selection_cfg.min_accepted_per_label)
    max_per_label = int(selection_cfg.max_accepted_per_label)

    accepted: set[str] = set()
    per_label_counts: dict[str, int] = {}

    for label in PRIMARY_ENDPOINT_LABELS:
        label_ids = [
            image_id
            for image_id in eligible["image_id"]
            if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1
        ]
        if not label_ids:
            per_label_counts[label] = 0
            continue
        subset = eligible[eligible["image_id"].isin(label_ids)].nlargest(max_per_label, "composite")
        chosen = subset["image_id"].tolist()
        accepted.update(chosen)
        per_label_counts[label] = len(chosen)

    # No-Finding recipes compete as their own group.
    no_finding_ids = [
        image_id
        for image_id in eligible["image_id"]
        if sum(int(v) for v in intended_by_id.get(image_id, {}).values()) == 0
    ]
    if no_finding_ids:
        subset = eligible[eligible["image_id"].isin(no_finding_ids)].nlargest(max_per_label, "composite")
        accepted.update(subset["image_id"].tolist())
        per_label_counts["__no_finding__"] = len(subset)

    for image_id in eligible["image_id"]:
        if image_id not in accepted:
            rejected.append({"image_id": image_id, "reason": "did_not_win_class_quota"})

    starved = {
        label: count
        for label, count in per_label_counts.items()
        if 0 < count < min_per_label
    }
    if starved:
        print(
            f"  NOTE: {len(starved)} label(s) below min_accepted_per_label={min_per_label}: "
            f"{starved}\n  (not back-filled — the quality floor is absolute, §4.8)",
            flush=True,
        )

    return sorted(accepted), rejected


def build_intended_lookup(config) -> dict[str, dict]:
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    manifest_path = stage2_paths(stage2_cfg, str(config.split_namespace))["manifest_path"]
    if not manifest_path.is_file():
        raise SystemExit(f"UPSTREAM GATE: missing generation manifest {manifest_path}")
    lookup = {}
    with open(manifest_path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                lookup[entry["image_id"]] = entry["intended_label_vector"]
    return lookup


def run_proxy_trial(
    weights: dict[str, float],
    merged: pd.DataFrame,
    intended_by_id: dict[str, dict],
    config,
    namespace: str,
    seed: int,
    fold: int,
    trial_key: str,
) -> float:
    """One proxy evaluation (§4.7): train on classifier_train + candidate subset, evaluate on
    asism_tuning_heldout. Returns macro-AUROC over the primary endpoint labels."""
    from scripts.utils.classifier import (
        TrainingBudget,
        evaluate_classifier,
        records_from_split,
        records_from_synthetic_manifest,
        train_classifier,
    )

    stage1_cfg = load_stage1_config()
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    tuning = config.tuning

    selected_ids, _ = select_images(merged, weights, config, intended_by_id)

    # Real component is classifier_train — NOT gen_train (§4.7 frozen data flow).
    real_frame = load_split(
        "classifier_train", namespace, purpose="schema_validation", caller="asism_proxy"
    )
    real_records = records_from_split(
        real_frame, Path(stage1_cfg.paths.images_dir) / namespace / "classifier_train"
    )
    s2paths = stage2_paths(stage2_cfg, namespace)
    synthetic_records = records_from_synthetic_manifest(s2paths["manifest_path"], s2paths["images_dir"], selected_ids)

    eval_frame = load_split(
        "asism_tuning_heldout", namespace, purpose="schema_validation", caller="asism_proxy"
    )
    fold_manifest_hash = None
    if fold > 0:
        fold_manifest = read_json(Path(config.paths.fold_manifest))
        fold_key = str(fold - 1)
        if fold_key not in fold_manifest["folds"]:
            raise SystemExit(f"ASISM fold {fold} is absent from the frozen fold manifest")
        patient_ids = set(fold_manifest["folds"][fold_key]["patient_ids"])
        eval_frame = eval_frame[eval_frame["patient_id"].astype(str).isin(patient_ids)].copy()
        fold_manifest_hash = fold_manifest["fold_manifest_hash"]
    eval_records = records_from_split(
        eval_frame, Path(stage1_cfg.paths.images_dir) / namespace / "asism_tuning_heldout"
    )

    budget = TrainingBudget(
        max_steps=int(tuning.coarse.proxy_max_steps) if fold == 0 else int(tuning.shortlist.proxy_max_steps),
        batch_size=int(tuning.proxy.batch_size),
        learning_rate=float(tuning.proxy.learning_rate),
        seed=seed,
        eval_every_n_steps=0,  # no candidate-specific early stopping (§4.7)
    )
    checkpoint_path = Path(config.paths.asism_dir) / "proxy_checkpoints" / f"{hash_dict({'trial_key': trial_key}, length=32)}.pt"
    checkpoint_provenance = {
        "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
        "split_namespace": namespace, "dataset_id": trial_key,
        "condition_or_draw_id": trial_key, "config_hash": hash_dict({"weights": weights, "budget": budget.__dict__}, length=64),
        "label_policy": "chexpert_mask_uncertain_v1", "fold_manifest_hash": fold_manifest_hash,
        "code_identity_sha256": config._expected_score_provenance["code_identity_sha256"],
    }
    model, _, _ = train_classifier(
        train_records=real_records + synthetic_records,
        val_records=[],
        budget=budget,
        dropout_p=float(tuning.proxy.dropout_p),
        pretrained_source=str(tuning.proxy.pretrained_source),
        resolution=int(tuning.proxy.resolution),
        progress_desc=f"proxy s{seed}f{fold}",
        checkpoint_path=checkpoint_path,
        resume_from=checkpoint_path if checkpoint_path.is_file() else None,
        checkpoint_metadata=checkpoint_provenance,
    )
    metrics = evaluate_classifier(model, eval_records, int(tuning.proxy.resolution))
    return float(metrics["macro_auroc"]), fold_manifest_hash


def append_search_log(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", buffering=1) as handle:
        handle.write(json.dumps(record, sort_keys=True, default=str) + "\n")


def completed_trials(path: Path, expected_provenance: dict) -> dict[str, float]:
    """Resume at trial granularity (§4.7): an interrupted search re-reads its own log."""
    if not path.is_file():
        return {}
    done = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                for key, value in expected_provenance.items():
                    if entry.get(key) != value:
                        raise SystemExit(f"Stale ASISM search log at {path}:{key}: expected {value!r}, got {entry.get(key)!r}")
                done[entry["trial_key"]] = entry["proxy_macro_auroc"]
    return done


def phase_tune(config, namespace: str) -> int:
    estimate = enforce_compute_budget(config)

    gonogo_path = Path(config.paths.gonogo_report)
    if not gonogo_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: Go/No-Go report not found at {gonogo_path}\n"
            "Signals must clear the gate before tuning (§4.6): python scripts/asism/02_gonogo.py"
        )
    gonogo = read_json(gonogo_path)
    for key, value in config._expected_score_provenance.items():
        if gonogo.get(key) != value:
            raise SystemExit(f"Go/No-Go provenance mismatch at {key}: expected {value!r}, got {gonogo.get(key)!r}")
    surviving = list(gonogo["surviving_signals"])
    if not surviving:
        raise SystemExit("No signals survived the Go/No-Go gate — nothing to tune.")

    merged = load_merged_scores(config, surviving)
    intended_by_id = build_intended_lookup(config)
    search_log = Path(config.paths.search_log)
    done = completed_trials(search_log, dict(config._expected_score_provenance))

    tuning = config.tuning
    from scripts.utils.splits import freeze_patient_folds
    fold_path = Path(config.paths.fold_manifest)
    if not fold_path.exists():
        fold_frame = load_split("asism_tuning_heldout", namespace, purpose="schema_validation", caller="asism_fold_freeze")
        freeze_patient_folds(fold_path, fold_frame["patient_id"].astype(str).tolist(), int(tuning.shortlist.folds), int(tuning.search_seed), split_provenance(namespace)["split_manifest_hash"])
    fold_manifest = read_json(fold_path)
    grid = candidate_weight_grid(surviving, int(tuning.coarse.trials), int(tuning.search_seed))

    print(f"Compute estimate: {estimate['total_proxy_runs']} runs, "
          f"{estimate['estimated_gpu_hours']}h (budget {estimate['max_gpu_hours']}h)", flush=True)
    print(f"Surviving signals: {surviving}", flush=True)
    print(f"\nStage A — coarse screening ({len(grid)} candidates)", flush=True)

    coarse_results = []
    for index, weights in enumerate(grid):
        key = f"coarse:{index}:{hash_dict(weights)}"
        if key in done:
            score = done[key]
            print(f"  [{index}] resumed  {weights} -> {score:.4f}", flush=True)
        else:
            score, _ = run_proxy_trial(
                weights, merged, intended_by_id, config, namespace,
                seed=int(tuning.search_seed), fold=0, trial_key=key,
            )
            append_search_log(search_log, {
                "trial_key": key, "phase": "coarse", "weights": weights,
                "proxy_macro_auroc": score, "seed": int(tuning.search_seed), "fold": 0,
                **dict(config._expected_score_provenance),
            })
            print(f"  [{index}] {weights} -> {score:.4f}", flush=True)
        coarse_results.append({"weights": weights, "score": score})

    coarse_results.sort(key=lambda item: item["score"], reverse=True)
    shortlist = coarse_results[: int(tuning.shortlist.size)]
    print(f"\nStage B — shortlist validation ({len(shortlist)} candidates)", flush=True)

    shortlist_results = []
    for index, candidate in enumerate(shortlist):
        scores = []
        for fold in range(1, int(tuning.shortlist.folds) + 1):
            for seed_offset in range(int(tuning.shortlist.seeds)):
                seed = int(tuning.search_seed) + seed_offset
                key = f"shortlist:{index}:{fold}:{seed}:{hash_dict(candidate['weights'])}"
                if key in done:
                    scores.append(done[key])
                    continue
                score, fold_hash = run_proxy_trial(
                    candidate["weights"], merged, intended_by_id, config, namespace,
                    seed=seed, fold=fold, trial_key=key,
                )
                append_search_log(search_log, {
                    "trial_key": key, "phase": "shortlist", "weights": candidate["weights"],
                    "proxy_macro_auroc": score, "seed": seed, "fold": fold,
                    "fold_manifest_hash": fold_hash,
                    **dict(config._expected_score_provenance),
                })
                scores.append(score)
        mean_score = float(np.mean(scores))
        shortlist_results.append({
            "weights": candidate["weights"], "mean_score": mean_score,
            "std_score": float(np.std(scores)), "n_runs": len(scores),
        })
        print(f"  [{index}] {candidate['weights']} -> {mean_score:.4f} "
              f"(+/-{np.std(scores):.4f}, n={len(scores)})", flush=True)

    # Frozen selection rule: best mean; ties within the noise band resolve to the SIMPLER
    # configuration (fewer active signals, then smaller total weight).
    band = float(tuning.tie_noise_band)
    finite_results = [item for item in shortlist_results if np.isfinite(item["mean_score"])]
    selection_fallback = None
    if finite_results:
        best_score = max(item["mean_score"] for item in finite_results)
        tied = [item for item in finite_results if best_score - item["mean_score"] <= band]
    elif os.environ.get("THESIS_SMOKE_MODE") == "1":
        # A tiny fixture fold can contain only one class for every label, making macro-AUROC
        # undefined. Keep the engineering smoke moving with the already-finite coarse result;
        # production remains fail-closed below.
        best_coarse = max(coarse_results, key=lambda item: item["score"])
        tied = [{**shortlist_results[0], "mean_score": float(best_coarse["score"])}]
        selection_fallback = "smoke_only_all_shortlist_aurocs_undefined_used_best_coarse_score"
    else:
        raise SystemExit(
            "ASISM tuning produced no finite shortlist macro-AUROC values. Increase held-out "
            "class support or revise the predeclared fold plan; no production winner was frozen."
        )
    winner = min(
        tied,
        key=lambda item: (
            sum(1 for w in item["weights"].values() if w > 0),
            sum(item["weights"].values()),
        ),
    )

    frozen = {
        "schema_version": SCHEMA_VERSION,
        "stage": "asism_frozen",
        "surviving_signals": surviving,
        "ablation_only_signals": gonogo.get("ablation_only_signals", []),
        "excluded_signals": gonogo.get("excluded_signals", []),
        "asism_variant_status": gonogo.get("asism_variant_status"),
        "frozen_weights": winner["weights"],
        "selection_policy": OmegaConf.to_container(config.selection, resolve=True),
        "tie_break_rule": "best mean proxy macro-AUROC; ties within noise band -> simpler config",
        "tie_noise_band": band,
        "n_tied_candidates": len(tied),
        "selection_fallback": selection_fallback,
        "winner_mean_proxy_macro_auroc": winner["mean_score"],
        "shortlist_results": shortlist_results,
        "coarse_results": coarse_results,
        "required_baselines": {
            "equal_weight": [item for item in coarse_results if len(set(item["weights"].values())) == 1 and any(item["weights"].values())],
            "individual_signal": [item for item in coarse_results if sum(value > 0 for value in item["weights"].values()) == 1],
            "leave_one_signal_out": [item for item in coarse_results if sum(value == 0 for value in item["weights"].values()) == 1],
            "uncertainty_band_only": [item for item in coarse_results if item["weights"].get("__uncertainty_band_only__", 0) > 0],
        },
        "compute_accounting": estimate,
        "proxy_data_flow": {
            "train_real": "classifier_train",
            "train_synthetic": "candidate subset (actually constructed)",
            "evaluate": "asism_tuning_heldout",
            "classifier_val_used_for_ranking": False,
            "final_eval_heldout_used": False,
        },
        "search_log": str(search_log),
        "fold_manifest_hash": fold_manifest["fold_manifest_hash"],
        "split_provenance": split_provenance(namespace),
        "git_commit_hash": get_git_commit_hash(),
        "frozen": True,
        **dict(config._expected_score_provenance),
    }
    write_frozen_json(Path(config.paths.frozen_manifest), frozen)

    print(f"\nFROZEN weights: {winner['weights']}", flush=True)
    print(f"  proxy macro-AUROC: {winner['mean_score']:.4f}", flush=True)
    print(f"  manifest -> {config.paths.frozen_manifest}", flush=True)
    return 0


def phase_select(config, namespace: str) -> int:
    frozen_path = Path(config.paths.frozen_manifest)
    if not frozen_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: ASISM is not frozen ({frozen_path} missing).\n"
            "Run --phase tune first."
        )
    frozen = read_json(frozen_path)
    for key, value in config._expected_score_provenance.items():
        if frozen.get(key) != value:
            raise SystemExit(f"Frozen ASISM provenance mismatch at {key}: expected {value!r}, got {frozen.get(key)!r}")
    if not frozen.get("frozen"):
        raise SystemExit("ASISM manifest exists but is not marked frozen.")

    merged = load_merged_scores(config, frozen["surviving_signals"])
    intended_by_id = build_intended_lookup(config)
    selected, rejected = select_images(merged, frozen["frozen_weights"], config, intended_by_id)

    selected_path = Path(config.paths.selected_manifest)
    rejected_path = Path(config.paths.rejected_log)
    selected_path.parent.mkdir(parents=True, exist_ok=True)

    def atomic_jsonl(path: Path, records: list[dict]) -> None:
        fd, temp = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as handle:
                for record in records:
                    handle.write(json.dumps(record, sort_keys=True) + "\n")
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temp, path)
        finally:
            Path(temp).unlink(missing_ok=True)

    atomic_jsonl(
        selected_path,
        [
            {
                "image_id": image_id,
                "intended_label_vector": intended_by_id.get(image_id, {}),
                "asism_frozen_weights": frozen["frozen_weights"],
                "asism_manifest_hash": hash_dict(frozen["frozen_weights"]),
            }
            for image_id in selected
        ],
    )
    atomic_jsonl(rejected_path, rejected)

    reasons = pd.Series([r["reason"] for r in rejected]).value_counts().to_dict() if rejected else {}
    write_json(
        selected_path.with_name("selection_summary.json"),
        {
            "n_candidates": len(merged),
            "n_selected": len(selected),
            "n_rejected": len(rejected),
            "rejection_reasons": reasons,
            "frozen_weights": frozen["frozen_weights"],
            "selection_policy": frozen["selection_policy"],
            "split_provenance": split_provenance(namespace),
            "git_commit_hash": get_git_commit_hash(),
        },
    )

    print(f"Candidates: {len(merged)}", flush=True)
    print(f"Selected:   {len(selected)}", flush=True)
    print(f"Rejected:   {len(rejected)}  {reasons}", flush=True)
    print(f"-> {selected_path}", flush=True)
    return 0


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=["estimate", "tune", "select"], required=True)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()

    config = load_config()
    namespace = args.namespace or str(config.split_namespace)
    stage2_config = load_named_config("stage2_generation.yaml", "stage2")
    expected_score_provenance = asism_score_provenance(config, stage2_config, namespace)
    config.split_namespace = namespace
    paths = stage3_paths(config, namespace)
    for key, value in paths.items():
        if key in config.paths:
            config.paths[key] = str(value)
    config._expected_score_provenance = expected_score_provenance

    if args.phase == "estimate":
        estimate = compute_budget_estimate(config)
        print(json.dumps(estimate, indent=2), flush=True)
        if not estimate["within_budget"]:
            print(
                "\nOVER BUDGET — reduce the search space in configs/stage3_asism.yaml before "
                "running --phase tune.",
                flush=True,
            )
            return 2
        print("\nWithin budget. Proceed: --phase tune", flush=True)
        return 0

    if args.phase == "tune":
        return phase_tune(config, namespace)
    return phase_select(config, namespace)


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 8 - Stage 3, second half: Learned ASISM (04 -> 09)

The novel contribution. Measure the real downstream utility of ~120 controlled subsets, learn a set-utility model from those measurements, distil it into a per-image ranker, then learn per-class admission thresholds under conservative, pre-registered governance - all on `asism_tuning_heldout`, never on `final_eval_heldout`.

## `scripts/asism/candidate_pool.py`

*43 lines*

**The shared, provenance-checked loader for every learned-ASISM stage (04-09).** Small but
load-bearing.

It reads the Go/No-Go report, takes `surviving_signals`, verifies **each** signal Parquet against
its expected provenance (`require_score_artifact`), inner-joins them on `image_id`, loads every
image's `intended_label_vector` from the Stage 2 manifest, and attaches a `__stratum` key (the
sorted intended-positive label combination, or `__no_finding__`) used for stratified sampling.

The comment about hashing a **pristine** Stage 3 config is a real bug fix worth noting: callers
rewrite `cfg.paths` to namespaced locations, and hashing the *rewritten* config made every learned
stage reject valid score artifacts.

In [ ]:
# ===== FILE: scripts/asism/candidate_pool.py =====
"""Provenance-checked input loading shared by Learned ASISM stages."""
from __future__ import annotations

import json
from pathlib import Path
import pandas as pd

from scripts.utils.artifact_contracts import asism_score_provenance, require_score_artifact, stage2_paths
from scripts.utils.config import load_named_config
from scripts.utils.manifest import read_json


def load_candidate_pool(cfg):
    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    # Hash the Stage 3 config exactly as 01_compute_signals.py and 02_gonogo.py recorded it: freshly
    # loaded, before callers rewrite cfg.paths to namespaced locations. Hashing the caller's
    # rewritten cfg made every learned stage (04-09) reject valid score artifacts.
    pristine = load_named_config("stage3_asism.yaml", "stage3")
    expected = asism_score_provenance(pristine, stage2_cfg, str(cfg.split_namespace))
    gonogo = read_json(Path(cfg.paths.gonogo_report))
    surviving = list(gonogo["surviving_signals"])
    if not surviving:
        raise SystemExit("No signals survived Go/No-Go; Learned ASISM cannot train.")
    merged = None
    for signal in surviving:
        path = Path(cfg.paths.scores_dir) / f"{signal}_scores.parquet"
        require_score_artifact(path, signal, expected)
        frame = pd.read_parquet(path)
        merged = frame if merged is None else merged.merge(frame, on="image_id", how="inner")
    if merged is None or merged.empty:
        raise SystemExit("No non-empty candidate images remain after merging admitted signals.")
    intended = {}
    manifest = stage2_paths(stage2_cfg, str(cfg.split_namespace))["manifest_path"]
    with open(manifest, encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                row = json.loads(line)
                intended[str(row["image_id"])] = row["intended_label_vector"]
    merged["__stratum"] = [
        "|".join(sorted(label for label, value in intended.get(str(image_id), {}).items() if int(value) == 1))
        or "__no_finding__" for image_id in merged["image_id"]
    ]
    return merged, intended, surviving


## `scripts/asism/learned.py`

*1338 lines*

**The pure-logic library behind learned ASISM - no torch, no I/O side effects, fully unit-testable.
The single densest file in the project.** Grouped by what it is for:

**Feature plumbing.** `FEATURE_COLUMNS_BY_SIGNAL`, `active_feature_columns` (intersect configured
columns with surviving signals - this is how Go/No-Go exclusion propagates), `contributing_signals`,
`safe_feature_frame`, `apply_feature_frame`.

**Subset design (04).** `split_image_pool` splits the candidate pool into **train-role** and
**val-role** image pools that are disjoint *by construction*. `build_controlled_subsets` /
`build_role_conditioned_subsets` build random / single-signal / mixed subsets across quantile
bands. `pool_feasibility_report` + `evaluate_subset_design_feasibility` are the fail-closed gate:
if the pool cannot support the design (too few candidates per class or per quantile band, too few
val subsets), it refuses rather than producing a degenerate design. `verify_built_subsets` checks
what was actually built.

**Ranking targets (05).** `normalize_ranking_targets` - the **C2** change: `standardize` (default),
`rank`, or `none`, plus optional winsorizing, all order-preserving, so one noisy proxy measurement
cannot dominate training on ~96 points. `banzhaf_msr_targets` - a Data-Banzhaf value estimator
computed from the *same* measured subsets at no extra GPU cost; `loo_vs_banzhaf_diagnostic` records
their Spearman agreement, and Banzhaf stays opt-in until that diagnostic is reviewed.
`validate_utility_results`, `spearman_with_reason`, `read_jsonl`.

**Threshold contexts (07).** `bootstrap_class_contexts` - per-class bootstrap resamples of one image
pool, each summarised as a 10-dim vector (score-distribution stats + real patient prevalence +
budget), honestly tagged `independent_clinical_sample: False`. `class_aware_context_vector`,
`real_class_support_context`, `hard_threshold_grid_search` (critic-guided candidate proposal),
`diversify_verification_candidates` (verify a *spread* of the grid, not just the critic's top-k).

**Governance (08).** `compute_verified_context_counts`, `eligible_classes_for_official_training`,
`determine_per_class_official_method` - the **three-tier per-class rule**: learned network only with
at least `min_verified_contexts_per_class` on *both* train and image-disjoint held-out sides *and*
passing frozen acceptance criteria; else the median best proxy-verified threshold
(`aggregate_hard_proxy_best_threshold`); else the fixed baseline. `freeze_acceptance_criteria` /
`enforce_acceptance_criteria` make those criteria pre-registered rather than chosen afterwards.

**Policies (08b/09).** `freematch_style_percentile_per_class` - the **C4** change: leniency scaled by
*range-normalised* real prevalence, so the rarest present class gets the most lenient threshold.
`enforce_per_class_selection_floor` - lowers a class's threshold if it would otherwise admit fewer
than `min_selected_per_label` (the one-time-curation analogue of a class-fairness term).
`build_policy_selected_manifest` combines per-class thresholds into the final multi-label selection.
`choose_full_policy` applies the pre-registered tie rule (`tie_noise_band` + `simplicity_order`).

`scientific_status_for_namespace` is the honesty tag: anything built in a dev/smoke namespace is
permanently marked non-scientific.

In [ ]:
# ===== FILE: scripts/asism/learned.py =====
"""Pure data-design utilities for learned ASISM."""
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from scripts.utils.labels import patient_level_support
from scripts.utils.manifest import hash_dict
from scripts.utils.splits import load_split


# A feature is eligible only when the Go/No-Go decision admits the signal that
# produced it.  Keeping this mapping here makes the learned selector obey the
# same signal-governance contract as the weighted selector.
FEATURE_COLUMNS_BY_SIGNAL = {
    "similarity": {"similarity_knn_mean", "similarity_top1", "similarity_topk_spread"},
    "iqa": {"iqa_composite", "iqa_sharpness", "iqa_contrast_std"},
    "uncertainty": {"uncertainty_mean_std"},
    "explainability": {"explainability_region_overlap"},
    "agreement": {"agreement_score"},
    "distinctiveness": {"distinctiveness_score"},
}


def active_feature_columns(configured_columns: list[str], surviving_signals: list[str]) -> list[str]:
    """Return configured learned-ASISM features admitted by Go/No-Go.

    ``ablation_only`` and ``exclude`` signals are deliberately absent from the
    merged score frame, so they must also be absent from every learned model.
    The returned ordered list is frozen in the learned-training manifest and
    reused by all downstream learned-ASISM stages.
    """
    surviving = set(surviving_signals)
    unknown = surviving - set(FEATURE_COLUMNS_BY_SIGNAL)
    if unknown:
        raise ValueError(f"Unknown Go/No-Go signal(s): {sorted(unknown)}")

    # A configured column that maps to NO known signal must be a hard error, never a silent drop.
    # Silently dropping it would let a typo, or a genuinely new score column added to
    # learned_asism.feature_columns without registering it above, shrink the learned model's input
    # space with no trace in any manifest — the model would train on fewer features than the config
    # says it uses, and nothing downstream could detect it.
    owned = {column for columns in FEATURE_COLUMNS_BY_SIGNAL.values() for column in columns}
    unmapped = [column for column in configured_columns if column not in owned]
    if unmapped:
        raise ValueError(
            f"learned_asism.feature_columns contains column(s) not registered in "
            f"FEATURE_COLUMNS_BY_SIGNAL: {unmapped}. Add each one to its producing signal there "
            "(so Go/No-Go governs it) or remove it from the config — it cannot be admitted to a "
            "learned model without a signal owner."
        )

    active = [
        column for column in configured_columns
        if any(column in columns and signal in surviving for signal, columns in FEATURE_COLUMNS_BY_SIGNAL.items())
    ]
    if not active:
        raise ValueError("Go/No-Go admitted no learned-ASISM feature columns; learned ASISM cannot train.")
    return active


def contributing_signals(active_columns: list[str]) -> list[str]:
    """Which Go/No-Go signals actually feed a given active feature set.

    Recorded in the learned-training manifest so the §4.6 `reduced_variant` judgement (fewer than
    three admitted signals -> the selector is reported as an ablation/alternative, not the primary
    method) can be made for the LEARNED selector too, not just the weighted one. A
    'Multi-Signal Utility Ranking Network' fed by one surviving signal is still runnable, but it is
    no longer multi-signal, and the manifest must say so rather than let the name imply otherwise.
    (The network is single-objective by design in every case — see MultiSignalUtilityRankingNetwork's
    docstring; what this count governs is how many SIGNALS feed it, not how many objectives.)
    """
    return sorted(
        signal for signal, columns in FEATURE_COLUMNS_BY_SIGNAL.items()
        if columns & set(active_columns)
    )


def safe_feature_frame(frame: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, dict]:
    missing = [column for column in columns if column not in frame]
    if missing:
        raise ValueError(f"Missing learned-ASISM feature columns: {missing}")
    values = frame[columns].replace([np.inf, -np.inf], np.nan)
    medians = values.median().fillna(0.0)
    values = values.fillna(medians)
    means = values.mean()
    stds = values.std(ddof=0).replace(0, 1.0).fillna(1.0)
    normalized = (values - means) / stds
    stats = {"columns": columns, "median": medians.to_dict(), "mean": means.to_dict(), "std": stds.to_dict()}
    return normalized, stats


def apply_feature_frame(frame: pd.DataFrame, columns: list[str], stats: dict) -> pd.DataFrame:
    """Normalize using FROZEN stats from a prior safe_feature_frame call — never refit at inference
    time. Refitting on whatever candidate pool happens to be present at selection time silently
    drifts the ranker's inputs away from what it was trained on."""
    missing = [column for column in columns if column not in frame]
    if missing:
        raise ValueError(f"Missing learned-ASISM feature columns: {missing}")
    medians = pd.Series({column: stats["median"][column] for column in columns})
    means = pd.Series({column: stats["mean"][column] for column in columns})
    stds = pd.Series({column: stats["std"][column] for column in columns})
    values = frame[columns].replace([np.inf, -np.inf], np.nan)
    values = values.fillna(medians)
    return (values - means) / stds


def _sample(pool: pd.DataFrame, size: int, rng: np.random.Generator, allow_partial: bool = False) -> list[str]:
    """NEVER samples with replacement.

    By default (allow_partial=False) this is a hard invariant: if the pool has fewer than `size`
    unique images, it raises rather than silently returning a smaller subset. A subset quietly
    smaller than its nominal size is exactly what this refuses to produce — that must surface as an
    explicit failure, either at --phase feasibility (before any subset exists) or, if the gate was
    bypassed or is stale, right here at build time.

    allow_partial=True exists ONLY for _sample_balanced's internal per-stratum draws, where drawing
    less than a stratum's exact quota is expected (a stratum can be small) and handled by that
    function's own backfill + final invariant check — never for a caller that needs an exact count.
    """
    if size <= 0:
        return []
    if len(pool) < size:
        if not allow_partial:
            raise ValueError(
                f"Insufficient pool: requested {size} unique images without replacement, pool has "
                f"only {len(pool)}. This must be caught by --phase feasibility before --phase build."
            )
        size = len(pool)
    positions = rng.choice(len(pool), size=size, replace=False)
    return pool.iloc[positions]["image_id"].astype(str).tolist()


def _sample_balanced(pool: pd.DataFrame, reference: pd.DataFrame, size: int,
                     rng: np.random.Generator, allow_partial: bool = False) -> list[str]:
    """Approximately preserve the reference label-recipe distribution when available.

    Per-stratum draws may legitimately undershoot (a stratum can be small); the shortfall is topped
    up from the rest of `pool`. If `pool` as a whole still can't cover `size` unique images after
    that: with allow_partial=False (the default, used wherever no further backfill exists above
    this call — the "random" and "single_signal" designs) this raises, a genuine invariant
    violation. allow_partial=True is used ONLY by build_controlled_subsets's "mixed" design, whose
    per-column contributions are intentionally allowed to fall short because THAT caller backfills
    the shortfall itself from the whole frame afterward (and raises there if even that isn't enough).
    """
    if "__stratum" not in reference or "__stratum" not in pool:
        return _sample(pool, size, rng, allow_partial=allow_partial)
    proportions = reference["__stratum"].value_counts(normalize=True)
    selected: list[str] = []
    for stratum, proportion in proportions.items():
        count = int(round(size * float(proportion)))
        selected.extend(_sample(pool.loc[pool["__stratum"] == stratum], count, rng, allow_partial=True))
    selected = list(dict.fromkeys(selected))
    if len(selected) < size:
        remaining = pool.loc[~pool.image_id.astype(str).isin(selected)]
        selected.extend(_sample(remaining, size - len(selected), rng, allow_partial=allow_partial))
    return selected[:size]


def build_controlled_subsets(frame: pd.DataFrame, feature_columns: list[str], total: int,
                             sizes: list[int], random_fraction: float, single_fraction: float,
                             mixed_fraction: float, seed: int, quantile_bins: int = 3) -> list[dict]:
    """Create random controls plus randomized single-signal and mixed compositions.

    Sampling remains random inside each pool. The controlled compositions prevent every large
    random subset from collapsing toward the same mean feature vector.
    """
    if frame["image_id"].duplicated().any():
        raise ValueError("image_id must be unique")
    if total < 3 or not sizes:
        raise ValueError("At least three subsets and one positive subset size are required")
    fraction_total = random_fraction + single_fraction + mixed_fraction
    if abs(fraction_total - 1.0) > 1e-6:
        raise ValueError(
            "random_fraction + single_signal_fraction + mixed_fraction must sum to 1.0, "
            f"got {fraction_total}"
        )
    if quantile_bins < 2:
        raise ValueError(f"quantile_bins must be >= 2, got {quantile_bins}")
    rng = np.random.default_rng(seed)
    n_random = round(total * random_fraction)
    n_single = round(total * single_fraction)
    n_mixed = total - n_random - n_single
    designs = ["random"] * n_random + ["single_signal"] * n_single + ["mixed"] * n_mixed
    rng.shuffle(designs)
    quantiles = {
        column: pd.qcut(frame[column].rank(method="first"), quantile_bins, labels=False)
        for column in feature_columns
    }
    records = []
    for index, design in enumerate(designs):
        size = min(int(rng.choice(sizes)), len(frame))
        if design == "random":
            ids = _sample_balanced(frame, frame, size, rng)
            recipe = {"type": "matched_random"}
        elif design == "single_signal":
            column = str(rng.choice(feature_columns))
            band = int(rng.integers(0, quantile_bins))
            ids = _sample_balanced(frame.loc[quantiles[column] == band], frame, size, rng)
            recipe = {"type": design, "feature": column, "quantile_band": band}
        else:
            chosen = list(rng.choice(feature_columns, size=min(3, len(feature_columns)), replace=False))
            weights = rng.dirichlet(np.ones(len(chosen)))
            parts: list[str] = []
            allocations = np.floor(weights * size).astype(int)
            allocations[0] += size - int(allocations.sum())
            bands = []
            for column, count in zip(chosen, allocations):
                band = int(rng.integers(0, quantile_bins))
                bands.append(band)
                parts.extend(_sample_balanced(frame.loc[quantiles[column] == band], frame, int(count), rng,
                                              allow_partial=True))
            ids = list(dict.fromkeys(parts))
            if len(ids) < size:
                remaining = frame.loc[~frame["image_id"].astype(str).isin(ids)]
                ids.extend(_sample(remaining, size - len(ids), rng))  # strict: raises if truly insufficient
            recipe = {"type": design, "features": chosen, "quantile_bands": bands, "mixture": weights.tolist()}
        records.append({"subset_id": f"utility_{index:04d}", "image_ids": ids[:size],
                        "size": len(ids[:size]), "seed": seed + index, "design": recipe})
    return records


def split_image_pool(
    frame: pd.DataFrame, val_fraction: float, seed: int
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Frozen, seeded image-level split into a train-image-pool and a val-image-pool.

    This must run BEFORE any subset is built. Every downstream subset is built from exactly one
    side (see build_role_conditioned_subsets), so no image can ever appear in both a train-role and
    a val-role subset — disjointness is guaranteed by construction, not by post-hoc filtering.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    if frame["image_id"].duplicated().any():
        raise ValueError("image_id must be unique")
    ids = frame["image_id"].astype(str).to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(ids)
    n_val = round(len(shuffled) * val_fraction)
    val_ids = set(shuffled[:n_val].tolist())
    is_val = frame["image_id"].astype(str).isin(val_ids)
    return frame.loc[~is_val].copy(), frame.loc[is_val].copy()


def build_role_conditioned_subsets(
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
    feature_columns: list[str],
    train_total: int,
    val_total: int,
    sizes: list[int],
    random_fraction: float,
    single_fraction: float,
    mixed_fraction: float,
    seed: int,
    quantile_bins: int = 3,
) -> list[dict]:
    """Build train-role and val-role subset populations from two DISJOINT image pools.

    Each call to build_controlled_subsets only ever sees one pool's frame, so it is structurally
    impossible for a subset to mix images from both sides.
    """
    train_records = build_controlled_subsets(
        train_frame, feature_columns, train_total, sizes, random_fraction, single_fraction,
        mixed_fraction, seed, quantile_bins=quantile_bins,
    )
    val_records = build_controlled_subsets(
        val_frame, feature_columns, val_total, sizes, random_fraction, single_fraction,
        mixed_fraction, seed + train_total, quantile_bins=quantile_bins,
    )
    for index, record in enumerate(train_records):
        record["role"] = "train"
        record["subset_id"] = f"train_{record['subset_id']}"
    for index, record in enumerate(val_records):
        record["role"] = "val"
        record["subset_id"] = f"val_{record['subset_id']}"
    return train_records + val_records


def pool_feasibility_report(
    frame: pd.DataFrame,
    feature_columns: list[str],
    intended_by_id: dict[str, dict],
    labels: list[str],
    subset_sizes: list[int],
    quantile_bins: int,
    val_fraction: float,
    seed: int,
    total_subsets: int,
    feasibility_thresholds: dict,
) -> dict:
    """Phase-1b feasibility numbers for the image-disjoint subset design.

    Pure function of whatever candidate frame is passed in: run it against a fixture to check the
    calculation is correct, or against a real merged score artifact (once Stage 2 generation + Stage
    3 signal scoring have produced one) to get the real numbers. It never invents a candidate pool.

    `val_fraction` is a PROVISIONAL DEFAULT (configs/stage3_asism.yaml's val_pool_fraction) until a
    real production candidate pool has been run through this report and reviewed — this function
    tags that explicitly rather than implying the ratio is final.
    """
    train_frame, val_frame = split_image_pool(frame, val_fraction, seed)

    # Defensive re-check: split_image_pool guarantees disjointness by construction, but a failure
    # criterion must VERIFY that, not assume it.
    train_ids = set(train_frame["image_id"].astype(str))
    val_ids = set(val_frame["image_id"].astype(str))
    overlap_ids = train_ids & val_ids

    def per_label_counts(pool: pd.DataFrame) -> dict[str, int]:
        return {
            label: sum(
                1
                for image_id in pool["image_id"]
                if int(intended_by_id.get(str(image_id), {}).get(label, 0)) == 1
            )
            for label in labels
        }

    def band_series(pool: pd.DataFrame) -> dict[str, "pd.Series"]:
        if pool.empty or len(pool) < quantile_bins:
            return {}
        return {
            column: pd.qcut(pool[column].rank(method="first"), quantile_bins, labels=False)
            for column in feature_columns
        }

    def per_band_counts(bands: dict) -> dict[str, dict]:
        if not bands:
            return {column: {} for column in feature_columns}
        return {
            column: {int(band): int(count) for band, count in series.value_counts().sort_index().items()}
            for column, series in bands.items()
        }

    def size_achievability(pool_n: int) -> dict:
        # No replacement is ever used (see _sample): a pool smaller than a requested size simply
        # produces a smaller-than-requested subset. That is what "achievable_without_replacement"
        # reports here — there is no separate "replacement rate" to report because replacement never
        # happens; a False here IS the shortfall.
        result = {}
        for size in subset_sizes:
            result[str(size)] = {
                "requested_size": size,
                "pool_size": pool_n,
                "achievable_without_replacement": bool(pool_n >= size),
                "achievable_size_if_smaller": min(size, pool_n),
            }
        return result

    train_bands = band_series(train_frame)
    val_bands = band_series(val_frame)
    train_total = round(int(total_subsets) * (1 - val_fraction))
    val_total = int(total_subsets) - train_total

    report = {
        "total_candidates": len(frame),
        "train_pool_size": len(train_frame),
        "val_pool_size": len(val_frame),
        "val_fraction_requested": val_fraction,
        "val_fraction_actual": (len(val_frame) / len(frame)) if len(frame) else 0.0,
        "val_pool_fraction_status": "provisional_default",
        "planned_train_subsets": train_total,
        "planned_val_subsets": val_total,
        "train_val_overlap_count": len(overlap_ids),
        "candidates_per_label": {"train": per_label_counts(train_frame), "val": per_label_counts(val_frame)},
        "candidates_per_quantile_band": {
            "train": per_band_counts(train_bands),
            "val": per_band_counts(val_bands),
        },
        "subset_size_achievability": {
            "train": size_achievability(len(train_frame)),
            "val": size_achievability(len(val_frame)),
        },
        "quantile_band_size_achievability": {
            "train": {
                column: {
                    str(band): size_achievability(int(count))
                    for band, count in per_band_counts(train_bands).get(column, {}).items()
                }
                for column in feature_columns
            },
            "val": {
                column: {
                    str(band): size_achievability(int(count))
                    for band, count in per_band_counts(val_bands).get(column, {}).items()
                }
                for column in feature_columns
            },
        },
        "feasibility_thresholds": dict(feasibility_thresholds),
    }
    report["failures"] = evaluate_subset_design_feasibility(report, feasibility_thresholds)
    report["passed"] = len(report["failures"]) == 0
    return report


def evaluate_subset_design_feasibility(report: dict, thresholds: dict) -> list[str]:
    """Pure evaluation of a pool_feasibility_report's numbers against config-declared thresholds.

    Every threshold here comes from `feasibility_thresholds` in configs/stage3_asism.yaml — nothing
    is hardcoded except the structural rules that have no numeric threshold to configure (no
    sampling with replacement, ever; train/val pools must be exactly disjoint, not "mostly").
    """
    failures: list[str] = []

    if report["train_val_overlap_count"] > 0:
        failures.append(
            f"train/val pool overlap: {report['train_val_overlap_count']} image(s) appear in both "
            "pools (structural rule, not configurable)"
        )

    min_per_class = int(thresholds["min_candidates_per_class"])
    for role in ("train", "val"):
        for label, count in report["candidates_per_label"][role].items():
            if count < min_per_class:
                failures.append(
                    f"insufficient candidates per class: {role} pool has {count} candidates for "
                    f"{label!r}, below feasibility_thresholds.min_candidates_per_class={min_per_class}"
                )

    min_per_band = int(thresholds["min_candidates_per_quantile_band"])
    for role in ("train", "val"):
        for column, bands in report["candidates_per_quantile_band"][role].items():
            for band, count in bands.items():
                if count < min_per_band:
                    failures.append(
                        f"insufficient candidates per quantile band: {role} pool, feature={column!r}, "
                        f"band={band}, count={count}, below "
                        f"feasibility_thresholds.min_candidates_per_quantile_band={min_per_band}"
                    )

    for role in ("train", "val"):
        for size, entry in report["subset_size_achievability"][role].items():
            if not entry["achievable_without_replacement"]:
                failures.append(
                    f"actual subset size below required: {role} pool cannot build a "
                    f"{entry['requested_size']}-image subset without replacement "
                    f"(pool has {entry['pool_size']}; would only reach "
                    f"{entry['achievable_size_if_smaller']})"
                )
        for column, bands in report["quantile_band_size_achievability"][role].items():
            for band, sizes in bands.items():
                for size, entry in sizes.items():
                    if not entry["achievable_without_replacement"]:
                        failures.append(
                            f"actual subset size below required: {role} pool, feature={column!r}, "
                            f"band={band} cannot build a {entry['requested_size']}-image "
                            f"single_signal/mixed subset without replacement (band has "
                            f"{entry['pool_size']}; would only reach {entry['achievable_size_if_smaller']})"
                        )

    min_val_subsets = int(thresholds["min_val_subsets"])
    if report["planned_val_subsets"] < min_val_subsets:
        failures.append(
            f"val subset count below required: planned {report['planned_val_subsets']} val-role "
            f"subsets, below feasibility_thresholds.min_val_subsets={min_val_subsets}"
        )

    return failures


def verify_built_subsets(records: list[dict], train_total: int, val_total: int) -> None:
    """Post-build defense-in-depth: re-verify the WRITTEN records against every invariant the
    construction was supposed to already guarantee. Construction bugs, stale assumptions, or future
    refactors should never silently ship a broken utility_subsets.jsonl — this is the last check
    before it is written to disk, and raises SystemExit naming exactly which invariant failed."""
    for record in records:
        ids = record["image_ids"]
        if len(set(ids)) != len(ids):
            raise SystemExit(
                f"POST-BUILD INVARIANT VIOLATION: subset {record['subset_id']!r} contains a "
                "duplicate image_id"
            )
        if len(ids) != record["size"]:
            raise SystemExit(
                f"POST-BUILD INVARIANT VIOLATION: subset {record['subset_id']!r} has "
                f"{len(ids)} images but declares size={record['size']}"
            )

    train_ids = {image_id for r in records if r["role"] == "train" for image_id in r["image_ids"]}
    val_ids = {image_id for r in records if r["role"] == "val" for image_id in r["image_ids"]}
    overlap = train_ids & val_ids
    if overlap:
        raise SystemExit(
            f"POST-BUILD INVARIANT VIOLATION: {len(overlap)} image(s) appear in both a train-role "
            "and a val-role subset (e.g. " + ", ".join(sorted(overlap)[:5]) + ")"
        )

    n_train = sum(1 for r in records if r["role"] == "train")
    n_val = sum(1 for r in records if r["role"] == "val")
    if n_train != train_total or n_val != val_total:
        raise SystemExit(
            f"POST-BUILD INVARIANT VIOLATION: built {n_train} train-role and {n_val} val-role "
            f"subsets, planned {train_total}/{val_total}"
        )


def read_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def validate_utility_results(subsets: list[dict], results: list[dict]) -> pd.DataFrame:
    expected = {row["subset_id"] for row in subsets}
    result_ids = [row.get("subset_id") for row in results]
    if len(result_ids) != len(set(result_ids)):
        raise ValueError("Duplicate subset_id in utility results")
    missing = expected - set(result_ids)
    if missing:
        raise ValueError(f"Utility results are incomplete; missing {len(missing)} subsets")
    frame = pd.DataFrame(results)
    required = {"subset_id", "real_only_macro_auroc", "augmented_macro_auroc", "fold", "seed"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Utility results need columns {sorted(required)}")
    frame["utility_delta"] = frame["augmented_macro_auroc"] - frame["real_only_macro_auroc"]
    return frame


def spearman_with_reason(a, b) -> tuple[float | None, str | None]:
    """Spearman rho with an explicit machine-readable reason when it is undefined.

    scipy returns NaN when either input is (numerically) constant or fewer than two points are
    present — the correlation is undefined, not zero, and NaN is not valid JSON. This returns
    ``(rho, None)`` on success and ``(None, reason)`` otherwise so callers never write a bare NaN
    into a frozen manifest.
    """
    import math as _math

    import scipy.stats as _stats

    a = np.asarray(list(a), dtype=np.float64)
    b = np.asarray(list(b), dtype=np.float64)
    if a.size < 2 or b.size < 2 or a.size != b.size:
        return None, "fewer_than_two_paired_points"
    correlation = _stats.spearmanr(a, b)
    statistic = getattr(correlation, "statistic", None)
    value = float(statistic) if statistic is not None else float(correlation[0])
    if _math.isnan(value):
        return None, "constant_predictions_or_targets"
    return value, None


def normalize_ranking_targets(
    targets: dict[str, float], method: str = "standardize", winsorize_quantile: float = 0.0
) -> dict[str, float]:
    """Transform the distilled per-image ranking targets before the Smooth-L1 + pairwise loss.

    On ~96 training images a single noisy proxy-subset measurement can produce an outlier marginal
    value that dominates the regression term. ``"standardize"`` (zero mean, unit std) and ``"rank"``
    (map to evenly-spaced ranks in [0, 1]) both bound that influence; ``"none"`` keeps the raw
    values. ``winsorize_quantile`` in (0, 0.5) additionally clips each tail before the transform.
    Every method is order-preserving, so the pairwise term is unaffected. Motivated by the
    robustness rationale of the Banzhaf value (Wang & Jia, AISTATS 2023) and recent
    marginal-contribution estimators (2D-OOB, NeurIPS 2024; Chi et al., ICML 2026).
    """
    if not targets:
        return {}
    keys = list(targets)
    values = np.asarray([float(targets[key]) for key in keys], dtype=np.float64)
    if winsorize_quantile:
        if not 0.0 < winsorize_quantile < 0.5:
            raise ValueError(f"winsorize_quantile must be in (0, 0.5), got {winsorize_quantile}")
        low, high = np.quantile(values, [winsorize_quantile, 1.0 - winsorize_quantile])
        values = np.clip(values, low, high)
    if method == "none":
        pass
    elif method == "standardize":
        std = float(values.std())
        values = (values - values.mean()) / (std if std > 1e-12 else 1.0)
    elif method == "rank":
        import scipy.stats as _stats
        ranks = _stats.rankdata(values, method="average")
        values = (ranks - 1.0) / max(len(ranks) - 1, 1)
    else:
        raise ValueError(f"unknown target normalization method {method!r}")
    return dict(zip(keys, values.tolist()))


def banzhaf_msr_targets(
    subsets: list[dict],
    utility_by_id: dict[str, float],
    minimum_in_subsets: int = 1,
    minimum_out_subsets: int = 1,
) -> tuple[dict[str, float], dict[str, dict[str, int]]]:
    """Per-image Data-Banzhaf value via the Maximum-Sample-Reuse (MSR) estimator.

    Uses ONLY the subsets already measured for SetUtilityNetwork (utility_subsets.jsonl +
    utility_results.jsonl) — no extra proxy-model training, no additional GPU cost. For image i:

        banzhaf(i) = mean( U(S) for S in subsets if i in S )
                   - mean( U(S) for S in subsets if i not in S )

    This is the one-pass MSR estimator of the Banzhaf semivalue (Wang & Jia, "Data Banzhaf: A
    Robust Data Valuation Framework for Machine Learning", AISTATS 2023): every measured subset
    contributes to exactly one of the two averages for every image, so all ``len(subsets)``
    measurements are reused for all images simultaneously.

    An image gets a value ONLY if it appears in at least ``minimum_in_subsets`` measured subsets AND
    is absent from at least ``minimum_out_subsets`` of them — otherwise one mean is undefined and the
    image is simply absent from the returned targets (never 0.0, never a one-sided value). This
    mirrors the exposure filter that ``05_train_learned_asism.py`` already applies to the
    leave-one-out targets. ``counts`` reports ``{"in": ..., "out": ...}`` for every image seen.
    """
    if not subsets:
        return {}, {}
    total_utility = 0.0
    n_subsets = 0
    in_sum: dict[str, float] = {}
    in_count: dict[str, int] = {}
    for subset in subsets:
        subset_id = subset["subset_id"]
        if subset_id not in utility_by_id:
            raise ValueError(f"banzhaf_msr_targets: no measured utility for subset {subset_id!r}")
        utility = float(utility_by_id[subset_id])
        total_utility += utility
        n_subsets += 1
        for image_id in set(map(str, subset["image_ids"])):
            in_sum[image_id] = in_sum.get(image_id, 0.0) + utility
            in_count[image_id] = in_count.get(image_id, 0) + 1

    targets: dict[str, float] = {}
    counts: dict[str, dict[str, int]] = {}
    for image_id, appearances in in_count.items():
        out_appearances = n_subsets - appearances
        counts[image_id] = {"in": appearances, "out": out_appearances}
        if appearances < minimum_in_subsets or out_appearances < minimum_out_subsets:
            continue
        mean_in = in_sum[image_id] / appearances
        mean_out = (total_utility - in_sum[image_id]) / out_appearances
        targets[image_id] = mean_in - mean_out
    return targets, counts


def loo_vs_banzhaf_diagnostic(
    leave_one_out_targets: dict[str, float], banzhaf_targets: dict[str, float]
) -> dict:
    """Rank agreement between the leave-one-out marginal targets (the current supervision for the
    ranking network) and the Banzhaf MSR targets, over the images both methods scored.

    Recorded in the learned-training manifest so the choice to switch the ranking network onto
    Banzhaf supervision is made from evidence, not by default — the plan keeps Banzhaf optional
    "until the diagnostic is reviewed".
    """
    shared = sorted(set(leave_one_out_targets) & set(banzhaf_targets))
    rho, reason = spearman_with_reason(
        [leave_one_out_targets[image_id] for image_id in shared],
        [banzhaf_targets[image_id] for image_id in shared],
    )
    return {
        "n_images_compared": len(shared),
        "n_leave_one_out_targets": len(leave_one_out_targets),
        "n_banzhaf_targets": len(banzhaf_targets),
        "spearman": rho,
        "spearman_undefined_reason": reason,
    }


def freematch_style_percentile_per_class(
    ranking_scores: dict[str, float],
    intended_by_id: dict[str, dict],
    image_ids: list[str],
    primary_labels: list[str],
    real_prevalence_by_label: dict[str, dict],
    base_percentile: float = 50.0,
    min_percentile: float = 10.0,
) -> dict[str, float | None]:
    """Training-free class-adaptive percentile threshold, adapted from FreeMatch (Wang et al.,
    ICLR 2023) self-adaptive thresholding.

    FreeMatch lowers the confidence threshold for classes the model has learned less well so their
    pseudo-labels are not all filtered out. ASISM applies the same idea as a ONE-TIME data-curation
    decision (never a per-training-step recompute): a class's admission percentile is scaled DOWN
    (more lenient) the rarer that class is in the REAL patient population
    (``real_class_support_context`` prevalence), so a single global cut does not starve rare classes
    of synthetic samples. The rarest class present maps to ``min_percentile`` and the most common to
    ``base_percentile`` — leniency is RANGE-normalised across the observed per-class prevalences
    (SST self-adaptive style: Zhao et al., Information Processing & Management, 2025), not merely
    divided by the maximum.

        ratio      = (prevalence(class) - min_prevalence) / (max_prevalence - min_prevalence)   in [0, 1]
        percentile = min_percentile + (base_percentile - min_percentile) * ratio
        threshold  = that percentile of the class's own candidate ranking scores

    A class with unknown or zero real prevalence falls back to ``base_percentile`` — missing
    information never buys a more lenient cut. A class with zero candidates returns ``None``. When
    every class shares the same prevalence, all get ``base_percentile``. No neural network, no
    gradient step; it is a competing threshold policy, not a replacement for the learned network.
    Pair with ``enforce_per_class_selection_floor`` so a rare class is never selected down to zero.
    """
    if not 0.0 <= min_percentile <= base_percentile <= 100.0:
        raise ValueError(
            f"require 0 <= min_percentile <= base_percentile <= 100, got {min_percentile}, {base_percentile}"
        )
    prevalences = {
        label: (real_prevalence_by_label.get(label) or {}).get("real_prevalence")
        for label in primary_labels
    }
    positive_prevalences = [value for value in prevalences.values() if value is not None and value > 0]
    max_prevalence = max(positive_prevalences) if positive_prevalences else None
    min_prevalence = min(positive_prevalences) if positive_prevalences else None
    prevalence_span = (max_prevalence - min_prevalence) if positive_prevalences else 0.0

    result: dict[str, float | None] = {}
    for label in primary_labels:
        class_scores = [
            ranking_scores[image_id]
            for image_id in image_ids
            if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1
        ]
        if not class_scores:
            result[label] = None
            continue
        prevalence = prevalences[label]
        if max_prevalence is None or prevalence is None or prevalence <= 0 or prevalence_span <= 0:
            percentile = base_percentile
        else:
            ratio = min(1.0, max(0.0, (float(prevalence) - min_prevalence) / prevalence_span))
            percentile = min_percentile + (base_percentile - min_percentile) * ratio
        result[label] = float(np.percentile(class_scores, percentile))
    return result


def enforce_per_class_selection_floor(
    thresholds: dict[str, float | None],
    ranking_scores: dict[str, float],
    intended_by_id: dict[str, dict],
    image_ids: list[str],
    primary_labels: list[str],
    min_selected_per_label: int,
) -> dict[str, float | None]:
    """Lower a class's threshold just enough to admit ``min_selected_per_label`` of its own
    candidates whenever the class-adaptive percentile would otherwise select fewer.

    The one-time-curation analogue of SST's class-fairness term (Zhao et al., Information Processing
    & Management, 2025): a class-adaptive threshold must not starve a rare class. A class with no
    threshold set is left untouched; a class with fewer than ``min_selected_per_label`` candidates
    in total has its threshold dropped to admit every candidate it has. Thresholds already
    admitting enough are unchanged.
    """
    adjusted = dict(thresholds)
    for label in primary_labels:
        threshold = adjusted.get(label)
        if threshold is None:
            continue
        class_scores = sorted(
            (ranking_scores[image_id] for image_id in image_ids
             if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1),
            reverse=True,
        )
        if not class_scores:
            continue
        selected = sum(1 for score in class_scores if score >= threshold)
        if selected >= min_selected_per_label:
            continue
        floor_index = min(min_selected_per_label, len(class_scores)) - 1
        adjusted[label] = float(class_scores[floor_index])
    return adjusted


def scientific_status_for_namespace(namespace: str) -> str:
    """Only the 'production' namespace is production evidence. Every other namespace (dev,
    dev-smoke-v1, ...) is development-only and must never be reported as real-world prevalence
    without this label attached."""
    return "production" if namespace == "production" else "non-production / development-only"


def real_class_support_context(
    namespace: str, labels: list[str], frame: pd.DataFrame | None = None
) -> dict:
    """Real patient-level prevalence/support per label from classifier_train (chexpert_mask_uncertain_v1).

    This is the REAL prevalence in the actual patient population of whatever `namespace` was loaded
    — NOT the fraction of synthetic candidate images that happen to carry a label, and NOT
    necessarily production evidence: check `scientific_status` before treating the numbers as
    anything more than development-only. `frame` may be pre-loaded (e.g. in tests) to avoid
    re-reading the split file.
    """
    if frame is None:
        frame = load_split(
            "classifier_train", namespace, purpose="schema_validation", caller="learned_asism_threshold"
        )
    support = patient_level_support(frame, labels)
    labels_context: dict[str, dict] = {}
    for _, row in support.iterrows():
        positive = float(row["positive_patients"])
        negative = float(row["negative_patients"])
        denominator = positive + negative
        labels_context[row["label"]] = {
            "positive_patients": positive,
            "negative_patients": negative,
            "real_prevalence": positive / denominator if denominator > 0 else None,
        }
    return {
        "namespace": namespace,
        "scientific_status": scientific_status_for_namespace(namespace),
        "labels": labels_context,
    }


def class_aware_context_vector(scores: list[float], real_prevalence: dict, budget_context: dict) -> list[float]:
    """Fixed-length (10-dim) context feature vector: candidate-pool score distribution stats + the
    REAL patient-level prevalence for this class (from real_class_support_context, not a fraction of
    synthetic candidates) + budget context. Order is part of the schema — do not reorder without
    also updating threshold_contexts.jsonl's "context_features" consumers."""
    array = np.asarray(scores, dtype=np.float64)
    if array.size:
        q25, q50, q75 = np.quantile(array, [0.25, 0.5, 0.75])
        mean, std = float(array.mean()), float(array.std())
    else:
        q25 = q50 = q75 = mean = std = 0.0
    real_prevalence_value = real_prevalence.get("real_prevalence")
    return [
        mean, std, float(q25), float(q50), float(q75),
        float(array.size), float(np.log1p(array.size)),
        float(real_prevalence_value) if real_prevalence_value is not None else 0.0,
        float(real_prevalence.get("positive_patients") or 0.0),
        float(budget_context.get("target_synthetic_to_real_ratio", 1.0)),
    ]


def bootstrap_class_contexts(
    pool: pd.DataFrame,
    class_id: int,
    label: str,
    intended_by_id: dict,
    score_column: str,
    context_source: str,
    n_replicates: int,
    min_fraction: float,
    max_fraction: float,
    ranking_checkpoint_hash: str,
    critic_checkpoint_hash: str,
    real_prevalence_context: dict,
    budget_context: dict,
    seed: int,
) -> list[dict]:
    """Resampled contexts for one class, from ONE image pool only (train-role or val-role — never
    both; `context_source` records which). These are honestly NOT independent clinical samples: they
    are repeated subsamples of the same underlying candidate pool, using the same ranking/critic
    checkpoints. `independent_clinical_sample: False` is stamped on every record so nothing
    downstream can mistake resampling diversity for genuine independent evidence.

    No sampling with replacement: each context is a distinct, duplicate-free subsample (via
    _sample's strict default), never a padded/duplicated set.
    """
    if not 0.0 < min_fraction <= max_fraction <= 1.0:
        raise ValueError(f"require 0 < min_fraction <= max_fraction <= 1.0, got {min_fraction}, {max_fraction}")
    class_ids = {
        str(image_id) for image_id in pool["image_id"]
        if int(intended_by_id.get(str(image_id), {}).get(label, 0)) == 1
    }
    class_pool = pool.loc[pool["image_id"].astype(str).isin(class_ids)]
    if class_pool.empty:
        return []
    rng = np.random.default_rng(seed)
    contexts = []
    for index in range(n_replicates):
        fraction = float(rng.uniform(min_fraction, max_fraction))
        size = max(1, round(len(class_pool) * fraction))
        image_ids = _sample(class_pool, size, rng)  # strict: no replacement, no silent undersizing
        scores = class_pool.loc[class_pool["image_id"].astype(str).isin(image_ids), score_column].tolist()
        context_features = class_aware_context_vector(scores, real_prevalence_context, budget_context)
        contexts.append({
            "context_id": f"ctx_{label}_{context_source}_{index:04d}",
            "class_id": class_id,
            "label": label,
            "context_source": context_source,
            "independent_clinical_sample": False,
            "candidate_pool_hash": hash_dict({"image_ids": sorted(image_ids)}, length=32),
            "ranking_checkpoint_hash": ranking_checkpoint_hash,
            "critic_checkpoint_hash": critic_checkpoint_hash,
            "fold": None,
            "seed": seed + index,
            "budget_context": dict(budget_context),
            "real_prevalence_context": dict(real_prevalence_context),
            "context_features": context_features,
            "image_ids": image_ids,
        })
    return contexts


def hard_threshold_grid_search(
    critic_utility_fn,
    ranking_scores: dict[str, float],
    image_ids: list[str],
    t_grid,
    budget_weight: float,
    max_selected: int,
    min_selected: int = 0,
) -> list[dict]:
    """Grid search over HARD thresholds only (boolean selection — critic_utility_fn must never be
    called with soft/continuous weights here). Empty subsets are rejected outright, never scored.
    `critic_predicted_utility` is exactly what the name says: the frozen critic's prediction, NOT a
    measured/ground-truth value — callers must never treat it as such.
    """
    evaluations = []
    for threshold in t_grid:
        selected = [image_id for image_id in image_ids if ranking_scores.get(image_id, 0.0) >= threshold]
        if not selected:
            continue  # reject empty subsets outright — never a valid candidate at any threshold
        if len(selected) > max_selected:
            selected = sorted(selected, key=lambda i: -ranking_scores.get(i, 0.0))[:max_selected]
        critic_predicted_utility = float(critic_utility_fn(selected))
        size_cost = len(selected) / max(len(image_ids), 1)
        objective = critic_predicted_utility - float(budget_weight) * size_cost
        evaluations.append({
            "candidate_threshold": float(threshold),
            "selected_count": len(selected),
            "critic_predicted_utility": critic_predicted_utility,
            "below_min_selected": len(selected) < min_selected,
            "_objective": objective,
        })
    if not evaluations:
        raise ValueError(
            "hard_threshold_grid_search: every threshold in t_grid produced an empty subset — "
            "the grid or the candidate pool is misconfigured for this context."
        )
    evaluations.sort(key=lambda entry: -entry["_objective"])
    for rank, entry in enumerate(evaluations, start=1):
        entry["rank_within_context"] = rank
        del entry["_objective"]
    return evaluations


def resolve_verified_only_targets(contexts: list[dict], measurements: list[dict]) -> list[dict]:
    """OFFICIAL training path. A context contributes a target ONLY if it has at least one
    proxy-MEASURED candidate — never a critic guess, never blended, never weighted-down-but-included.
    A context with zero measurements is simply absent from this function's output; there is no
    fallback inside this function to anything critic-derived. `measurements` must come from
    threshold_proxy_measurements.jsonl (07b's output), never from threshold_candidate_evaluations.jsonl
    (07's critic-only output) — mixing the two files here would defeat the whole point of the split.

    `proxy_best_among_verified_candidates` is deliberately NOT called "measured_best_threshold": only
    a subset of the grid (however it was chosen — see diversify_verification_candidates) was ever
    measured, so this is the best of what was checked, not a claim about the true grid optimum.
    """
    by_context: dict[str, list[dict]] = {}
    for row in measurements:
        by_context.setdefault(row["context_id"], []).append(row)

    targets = []
    for context in contexts:
        rows = by_context.get(context["context_id"], [])
        if not rows:
            continue
        best = max(rows, key=lambda row: row["measured_utility"])
        targets.append({
            "context_id": context["context_id"],
            "class_id": context["class_id"],
            "label": context["label"],
            "context_source": context["context_source"],
            "target_threshold": best["candidate_threshold"],
            "proxy_best_among_verified_candidates": best["candidate_threshold"],
            "n_verified_candidates_in_context": len(rows),
            "target_source": "proxy_verified",
            "supervision_weight": 1.0,
        })
    return targets


def resolve_critic_assisted_exploratory_targets(contexts: list[dict], evaluations: list[dict]) -> list[dict]:
    """EXPLORATORY path only — a completely separate model/training run from
    resolve_verified_only_targets. Uses the critic's own top grid pick for every context regardless
    of verification status. Its output must never be merged into, averaged with, or used as a
    fallback for the official verified-only targets; callers that train the OFFICIAL
    AdaptiveThresholdNetwork must not call this function at all.
    """
    by_context: dict[str, list[dict]] = {}
    for row in evaluations:
        by_context.setdefault(row["context_id"], []).append(row)

    targets = []
    for context in contexts:
        rows = by_context.get(context["context_id"], [])
        if not rows:
            continue
        critic_best = min(rows, key=lambda row: row["rank_within_context"])
        targets.append({
            "context_id": context["context_id"],
            "class_id": context["class_id"],
            "label": context["label"],
            "context_source": context["context_source"],
            "target_threshold": critic_best["candidate_threshold"],
            "critic_grid_threshold": critic_best["candidate_threshold"],
            "target_source": "critic_only",
            "supervision_weight": 1.0,  # full weight WITHIN this exploratory-only path only
        })
    return targets


def diversify_verification_candidates(
    evaluations: list[dict], top_k: int, quantile_fractions: list[float], candidate_pool_size: int,
) -> list[dict]:
    """Selects which grid evaluations get proxy-verified for one context: critic top-K PLUS
    quantile-spaced selection-count targets PLUS the two boundary (smallest/largest selected_count)
    candidates — never critic top-K alone. Verifying only where the critic already thinks is good
    would make the critic/proxy correlation check circular. Each returned entry carries
    `selection_reasons` (a threshold can qualify for more than one reason)."""
    if not evaluations:
        return []
    by_rank = sorted(evaluations, key=lambda entry: entry["rank_within_context"])
    chosen: dict[float, set] = {}

    def mark(entry: dict, reason: str) -> None:
        chosen.setdefault(entry["candidate_threshold"], set()).add(reason)

    for entry in by_rank[:top_k]:
        mark(entry, "critic_top_k")

    for fraction in quantile_fractions:
        target_count = max(1, round(candidate_pool_size * fraction))
        closest = min(evaluations, key=lambda entry: abs(entry["selected_count"] - target_count))
        mark(closest, "quantile_spaced")

    by_size = sorted(evaluations, key=lambda entry: entry["selected_count"])
    mark(by_size[0], "boundary")
    mark(by_size[-1], "boundary")

    # A literal 50%-of-pool selection target is included explicitly so verification always covers
    # the OLD baseline's own operating point, not just the new critic's preferences.
    baseline_target_count = max(1, round(candidate_pool_size * 0.5))
    baseline_closest = min(evaluations, key=lambda entry: abs(entry["selected_count"] - baseline_target_count))
    mark(baseline_closest, "baseline_target_ratio")

    by_threshold = {entry["candidate_threshold"]: entry for entry in evaluations}
    return [
        {**by_threshold[threshold], "selection_reasons": sorted(reasons)}
        for threshold, reasons in sorted(chosen.items())
    ]


def compute_verified_context_counts(
    verified_train_targets: list[dict], verified_held_out_targets: list[dict], labels: list[str]
) -> tuple[dict[str, int], dict[str, int]]:
    """Per-class counts of proxy_verified contexts, split by train/held-out source. The single
    source of truth both the eligibility gate and the governance decision are built from."""
    train_counts = {label: 0 for label in labels}
    held_out_counts = {label: 0 for label in labels}
    for target in verified_train_targets:
        train_counts[target["label"]] += 1
    for target in verified_held_out_targets:
        held_out_counts[target["label"]] += 1
    return train_counts, held_out_counts


def eligible_classes_for_official_training(
    train_counts: dict[str, int], held_out_counts: dict[str, int], min_verified_contexts_per_class: int
) -> set[str]:
    """A class is eligible for the OFFICIAL network only if it independently clears the minimum on
    BOTH sides — enough proxy-verified TRAIN contexts to learn from, and enough proxy-verified,
    image-disjoint HELD-OUT contexts to actually evaluate generalization on. Neither side alone is
    sufficient: training data without held-out evidence can't be trusted, and held-out evidence
    without training data has nothing to evaluate."""
    return {
        label for label in train_counts
        if train_counts[label] >= min_verified_contexts_per_class
        and held_out_counts.get(label, 0) >= min_verified_contexts_per_class
    }


def filter_targets_to_eligible_classes(targets: list[dict], eligible_classes: set[str]) -> list[dict]:
    """Restrict official fitting/evaluation to classes that cleared both evidence gates."""
    return [target for target in targets if target["label"] in eligible_classes]


def choose_full_policy(
    results: list[dict], expected_policies: list[str], expected_seeds: list[int],
    tie_noise_band: float, simplicity_order: list[str],
) -> dict:
    """Select a winner from complete ASISM-tuning full-policy measurements.

    A policy needs exactly one finite measurement for every frozen seed. Policies within the
    pre-registered noise band are resolved by the frozen simplicity order.
    """
    by_policy: dict[str, dict[int, float]] = {policy: {} for policy in expected_policies}
    for row in results:
        policy, seed = str(row["policy"]), int(row["seed"])
        if policy not in by_policy or seed not in expected_seeds:
            continue
        utility = float(row["measured_utility"])
        if not np.isfinite(utility):
            raise ValueError(f"non-finite measured utility for policy={policy}, seed={seed}")
        if seed in by_policy[policy]:
            raise ValueError(f"duplicate full-policy result for policy={policy}, seed={seed}")
        by_policy[policy][seed] = utility
    missing = {
        policy: sorted(set(expected_seeds) - set(values))
        for policy, values in by_policy.items() if set(values) != set(expected_seeds)
    }
    if missing:
        raise ValueError(f"incomplete full-policy verification results: {missing}")
    means = {policy: float(np.mean(list(values.values()))) for policy, values in by_policy.items()}
    best_mean = max(means.values())
    tied = {policy for policy, value in means.items() if best_mean - value <= tie_noise_band}
    ordered = [policy for policy in simplicity_order if policy in tied]
    winner = ordered[0] if ordered else sorted(tied)[0]
    return {
        "winning_policy": winner,
        "mean_measured_utility_by_policy": means,
        "best_mean_measured_utility": best_mean,
        "tie_noise_band": float(tie_noise_band),
        "tied_policies": sorted(tied),
        "tie_break_order": list(simplicity_order),
    }


def determine_per_class_official_method(
    verified_train_targets: list[dict],
    verified_held_out_targets: list[dict],
    labels: list[str],
    min_verified_contexts_per_class: int,
    acceptance_passed: bool,
) -> dict[str, str]:
    """Three-way per-class governance — no other outcome is possible:

    - ZERO verified TRAIN contexts for a class -> the OLD, honestly-named
      fixed_target_ratio_threshold_distillation_baseline_v1, REGARDLESS of how many held-out
      measurements exist. Held-out evidence must never be used to decide or construct a threshold —
      it exists only to evaluate generalization of something already built from train data. A class
      with train_count==0 has nothing to build a hard_proxy_best_among_verified threshold from
      either (see aggregate_hard_proxy_best_threshold, which only ever reads train targets).
    - Some verified TRAIN contexts, but below the pre-registered minimum on either side (train or
      image-disjoint held-out) -> hard_proxy_best_among_verified (the median measured-best threshold
      across that class's verified TRAIN contexts; no network trained/trusted on too little
      verified evidence).
    - Enough verified contexts on BOTH sides AND the frozen acceptance criteria passed ->
      adaptive_threshold_network.

    `verified_train_targets`/`verified_held_out_targets` must both come from
    resolve_verified_only_targets — never mix in critic-only exploratory targets here.
    """
    train_counts, held_out_counts = compute_verified_context_counts(verified_train_targets, verified_held_out_targets, labels)
    result = {}
    for label in labels:
        train_count, held_out_count = train_counts[label], held_out_counts[label]
        if train_count == 0:
            result[label] = "fixed_target_ratio_threshold_distillation_baseline_v1"
        elif train_count < min_verified_contexts_per_class or held_out_count < min_verified_contexts_per_class:
            result[label] = "hard_proxy_best_among_verified"
        else:
            result[label] = "adaptive_threshold_network" if acceptance_passed else "hard_proxy_best_among_verified"
    return result


def aggregate_hard_proxy_best_threshold(verified_train_targets: list[dict], label: str) -> float | None:
    """Final per-class threshold for hard_proxy_best_among_verified: the MEDIAN of
    proxy_best_among_verified_candidates (target_threshold) across that class's verified TRAIN-pool
    contexts ONLY. Held-out contexts are never read here — they evaluate generalization, they do not
    get a vote in what the operative threshold is. Median (not mean) is used for robustness against
    any single noisy proxy measurement. Returns None if the class has no verified train targets."""
    values = [target["target_threshold"] for target in verified_train_targets if target["label"] == label]
    if not values:
        return None
    return float(np.median(values))


def held_out_generalization_metrics(
    predicted_threshold: float, target_threshold: float, scores_for_context: dict[str, float]
) -> dict:
    """Absolute threshold error and selected-set Jaccard between a network's predicted threshold and
    the context's real target, evaluated on a held-out (val-pool-sourced) context the network never
    trained on. Utility regret (which needs the critic) is computed by the caller and merged in."""
    absolute_error = abs(predicted_threshold - target_threshold)
    predicted_selected = {image_id for image_id, score in scores_for_context.items() if score >= predicted_threshold}
    target_selected = {image_id for image_id, score in scores_for_context.items() if score >= target_threshold}
    union = predicted_selected | target_selected
    jaccard = (len(predicted_selected & target_selected) / len(union)) if union else 1.0
    return {"absolute_threshold_error": float(absolute_error), "selected_set_jaccard": float(jaccard)}


CRITERIA_DEFINITIONS = {
    "critic_proxy_median_spearman_min": {
        "unit": "spearman_rho (dimensionless, [-1, 1])",
        "definition": (
            "Median, across classes, of the WITHIN-class Spearman correlation between "
            "critic_predicted_utility and measured_utility, computed only on that class's "
            "proxy_verified candidates. The primary correlation criterion — pooled_spearman is "
            "diagnostic only and never gates acceptance, because pooling across classes with "
            "different utility scales can hide a class that individually correlates poorly."
        ),
    },
    "median_critic_predicted_utility_regret_max": {
        "unit": "critic_predicted_utility units (same scale as the SetUtilityNetwork's output)",
        "definition": (
            "Median, across held-out (bootstrap_val_pool) contexts, of (grid-best "
            "critic_predicted_utility - critic_predicted_utility at the network's own predicted "
            "threshold for that context)."
        ),
    },
    "threshold_stability_std_max": {
        "unit": "threshold probability std (the threshold itself is in [0, 1])",
        "definition": (
            "Median, across classes, of the standard deviation of the network's predicted "
            "threshold across that class's held-out contexts."
        ),
    },
}


def freeze_acceptance_criteria(criteria: dict, calibration_artifact_hash: str | None = None) -> dict:
    """Freezes acceptance-criteria numbers with a timestamp + config hash BEFORE any held-out
    evaluation is run. enforce_acceptance_criteria must be called against exactly this frozen
    payload — never against criteria re-read from a config that could have changed after results
    were seen. Tagged provisional_pre_registered_defaults: these numbers are a starting point set
    before any production evidence exists, not numbers validated by prior production runs — they
    must not be treated as settled science, and must NEVER be adjusted after seeing held-out or
    production results (that would be exactly the p-hacking this freezing mechanism exists to
    prevent)."""
    return {
        "criteria_status": "provisional_pre_registered_defaults",
        "criteria": dict(criteria),
        "criteria_definitions": {key: CRITERIA_DEFINITIONS[key] for key in criteria if key in CRITERIA_DEFINITIONS},
        "criteria_frozen_at": datetime.now(timezone.utc).isoformat(),
        "criteria_config_hash": hash_dict(dict(criteria), length=32),
        "calibration_artifact_hash": calibration_artifact_hash,
    }


def enforce_acceptance_criteria(evidence: dict, frozen: dict) -> dict:
    """Pure pass/fail evaluation of measured evidence against a FROZEN criteria payload (from
    freeze_acceptance_criteria). Returns a dict with a per-criterion verdict and an overall "passed"
    — never raises, so the caller can record a failing result rather than crash. Gates ONLY on
    critic_proxy_median_spearman_min (per-class median) — never on a pooled correlation."""
    criteria = frozen["criteria"]
    results = {}
    checks = {
        "critic_proxy_median_spearman_min": ("median_spearman", lambda value, bound: value is not None and value >= bound),
        "median_critic_predicted_utility_regret_max": (
            "median_critic_predicted_utility_regret", lambda value, bound: value is not None and value <= bound
        ),
        "threshold_stability_std_max": ("threshold_stability_std", lambda value, bound: value is not None and value <= bound),
    }
    for criterion_key, (evidence_key, comparator) in checks.items():
        if criterion_key not in criteria:
            continue
        bound = criteria[criterion_key]
        measured = evidence.get(evidence_key)
        results[criterion_key] = {
            "bound": bound,
            "measured": measured,
            "passed": bool(comparator(measured, bound)),
        }
    return {
        "criteria_status": frozen["criteria_status"],
        "criteria_frozen_at": frozen["criteria_frozen_at"],
        "criteria_config_hash": frozen["criteria_config_hash"],
        "per_criterion": results,
        "passed": all(entry["passed"] for entry in results.values()) if results else False,
    }


def critic_proxy_correlation_per_class(evaluations: list[dict], measurements: list[dict], contexts: list[dict]) -> dict:
    """Spearman correlation between critic_predicted_utility and measured_utility, computed WITHIN
    each class separately (never pooled across classes for the primary number — pooling is reported
    only as an explicit diagnostic). Requires >= 2 verified (context, threshold) pairs for a class to
    report a correlation at all; otherwise that class is simply absent from per_class_spearman."""
    import math as _math

    import scipy.stats as _stats

    label_by_context = {context["context_id"]: context["label"] for context in contexts}
    critic_by_key = {(row["context_id"], row["candidate_threshold"]): row["critic_predicted_utility"] for row in evaluations}

    by_label: dict[str, list[tuple[float, float]]] = {}
    for row in measurements:
        label = label_by_context.get(row["context_id"])
        critic_value = critic_by_key.get((row["context_id"], row["candidate_threshold"]))
        if label is None or critic_value is None:
            continue
        by_label.setdefault(label, []).append((critic_value, row["measured_utility"]))

    per_class_spearman: dict[str, float | None] = {}
    n_verified_by_class: dict[str, int] = {}
    for label, pairs in by_label.items():
        n_verified_by_class[label] = len(pairs)
        if len(pairs) < 2:
            per_class_spearman[label] = None
            continue
        predicted, measured = zip(*pairs)
        correlation = _stats.spearmanr(predicted, measured)
        statistic = getattr(correlation, "statistic", None)
        value = float(statistic) if statistic is not None else float(correlation[0])
        per_class_spearman[label] = None if _math.isnan(value) else value

    available = [value for value in per_class_spearman.values() if value is not None]
    macro_spearman = float(np.mean(available)) if available else None
    median_spearman = float(np.median(available)) if available else None

    all_pairs = [pair for pairs in by_label.values() for pair in pairs]
    pooled_spearman_diagnostic_only = None
    if len(all_pairs) >= 2:
        predicted, measured = zip(*all_pairs)
        correlation = _stats.spearmanr(predicted, measured)
        statistic = getattr(correlation, "statistic", None)
        value = float(statistic) if statistic is not None else float(correlation[0])
        pooled_spearman_diagnostic_only = None if _math.isnan(value) else value

    return {
        "per_class_spearman": per_class_spearman,
        "n_verified_by_class": n_verified_by_class,
        "macro_spearman": macro_spearman,
        "median_spearman": median_spearman,
        "pooled_spearman_diagnostic_only": pooled_spearman_diagnostic_only,
    }


def build_policy_selected_manifest(
    per_class_thresholds: dict[str, float | None],
    ranking_scores: dict[str, float],
    intended_by_id: dict[str, dict],
    image_ids: list[str],
    primary_labels: list[str],
) -> list[str]:
    """Combine per-class thresholds into ONE final selected set for a full-policy verification run.
    An image is selected if its ranking score clears the MINIMUM applicable threshold among the
    diseases it was intended for — the same min(applicable_thresholds) multi-label rule
    06_learn_thresholds_select.py already uses, kept identical here so a full-policy comparison
    across (baseline, hard_proxy_best, network, ...) is apples-to-apples on selection LOGIC, varying
    only the per-class thresholds themselves. An image with no applicable class threshold (e.g. a
    No-Finding recipe, or every applicable class currently unset) is never selected."""
    selected = []
    for image_id in image_ids:
        intended = intended_by_id.get(image_id, {})
        applicable = [
            per_class_thresholds[label] for label in primary_labels
            if int(intended.get(label, 0)) == 1 and per_class_thresholds.get(label) is not None
        ]
        if not applicable:
            continue
        threshold = min(applicable)
        if ranking_scores.get(image_id, 0.0) >= threshold:
            selected.append(image_id)
    return sorted(selected)


## `scripts/asism/models.py`

*138 lines*

**The three trainable networks.** Deliberately small - the training signal is ~96 measured subsets,
not a large dataset.

- **`SetUtilityNetwork`** - a permutation-invariant Deep Sets model over an image subset. Default
  pooling is masked mean. `set_superset_summary` + `encode_set` implement the **C1** ablation
  (masked mean *and* std pooling plus a fixed full-pool summary concatenated to the utility head,
  after Xie et al., ICLR 2024) - flag `superset_conditioning`, **default off**.
  It learns *only* from measured subset-level downstream utility.
- **`MultiSignalUtilityRankingNetwork`** - an MLP `9 -> 128 -> 64 -> 32 -> 1`. The 9 inputs are the
  admitted feature columns (similarity x3, IQA x3, uncertainty, explainability, agreement;
  distinctiveness's column is computed and gated but not yet in the configured feature set). **One**
  scalar utility target, optimised by Smooth-L1 + 0.5 x `pairwise_ranking_loss`. It is *not* a
  multi-objective / Pareto model.
- **`AdaptiveThresholdNetwork`** - a 16-dim class embedding concatenated with the 10-dim context
  -> 64 -> 32 -> 1 -> sigmoid. Per-class, context-dependent admission thresholds.

The key anti-pseudo-replication point: image scores are **distilled from marginal set utility**. No
subset AUROC is ever copied onto each of its member images.

`soft_selection_gate` is a differentiable relaxation of thresholding - present and tested, but not
used by the pipeline, which thresholds hard.

In [ ]:
# ===== FILE: scripts/asism/models.py =====
"""Trainable components for learned ASISM.

The set model learns only from measured subset-level downstream utility.  Image scores are then
distilled from marginal set utility; no subset AUROC is copied onto every member image.
"""
from __future__ import annotations

import torch
from torch import nn


def mlp(dimensions: list[int], dropout: float = 0.0, final_sigmoid: bool = False) -> nn.Sequential:
    layers: list[nn.Module] = []
    for index, (source, target) in enumerate(zip(dimensions, dimensions[1:])):
        last = index == len(dimensions) - 2
        layers.append(nn.Linear(source, target))
        if not last:
            layers.extend([nn.LayerNorm(target), nn.ReLU()])
            if dropout:
                layers.append(nn.Dropout(dropout))
    if final_sigmoid:
        layers.append(nn.Sigmoid())
    return nn.Sequential(*layers)


class SetUtilityNetwork(nn.Module):
    """Permutation-invariant Deep Sets predictor for measured AUROC delta.

    Default: masked MEAN pooling of per-image encodings (the frozen pipeline; the ``(n-1)``
    size-normalization in ``05_train_learned_asism.py::marginal_targets`` is derived from it).

    ``superset_conditioning=True`` (ablation only, Phase-11): the set is summarised by masked mean
    AND std pooling — both order-invariant — and a fixed summary vector of the full candidate pool
    is concatenated to the utility-head input. This follows Xie et al., ICLR 2024 ("Enhancing Neural
    Subset Selection"): a set-utility target that depends on the ground set benefits from a richer
    permutation-invariant sufficient statistic than the mean alone. NOTE: with std pooling the
    marginal-target size-normalization is only first-order correct for the mean component, so this
    flag is evaluated as an ablation, never silently swapped into the frozen path.
    """

    def __init__(self, input_dim: int, image_hidden=(128, 64), utility_hidden=(32,),
                 superset_conditioning: bool = False):
        super().__init__()
        self.superset_conditioning = bool(superset_conditioning)
        self.image_encoder = mlp([input_dim, *image_hidden], dropout=0.2)
        pooled_dim = image_hidden[-1] * (2 if self.superset_conditioning else 1)
        head_input = pooled_dim + (2 * input_dim if self.superset_conditioning else 0)
        self.utility_head = mlp([head_input, *utility_hidden, 1])
        if self.superset_conditioning:
            self.register_buffer("superset_summary", torch.zeros(2 * input_dim))

    def set_superset_summary(self, summary) -> None:
        if not self.superset_conditioning:
            raise ValueError("set_superset_summary called but superset_conditioning is False")
        vector = torch.as_tensor(summary, dtype=self.superset_summary.dtype,
                                 device=self.superset_summary.device)
        if vector.shape != self.superset_summary.shape:
            raise ValueError(
                f"superset summary must have shape {tuple(self.superset_summary.shape)}, "
                f"got {tuple(vector.shape)}"
            )
        self.superset_summary.copy_(vector)

    def encode_set(self, features: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        encoded = self.image_encoder(features)
        weights = mask.unsqueeze(-1).to(encoded.dtype)
        counts = weights.sum(1).clamp_min(1.0)
        mean = (encoded * weights).sum(1) / counts
        if not self.superset_conditioning:
            return mean
        variance = (weights * (encoded - mean.unsqueeze(1)) ** 2).sum(1) / counts
        return torch.cat([mean, variance.clamp_min(0.0).sqrt()], dim=-1)

    def forward(self, features: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        pooled = self.encode_set(features, mask)
        if self.superset_conditioning:
            summary = self.superset_summary.unsqueeze(0).expand(pooled.shape[0], -1)
            pooled = torch.cat([pooled, summary], dim=-1)
        return self.utility_head(pooled).squeeze(-1)


class MultiSignalUtilityRankingNetwork(nn.Module):
    """Small MLP that fuses the Go/No-Go-admitted ASISM signals (similarity, IQA, uncertainty,
    explainability, agreement, distinctiveness) into ONE scalar image-utility score.

    Deliberately NOT a multi-objective model: there is a single regression/ranking target (the
    distilled marginal set-utility from SetUtilityNetwork), optimized by both a Smooth-L1 term and a
    pairwise ranking term. MMoE / SDMGrad / NHDE are cited in the literature review only to make that
    contrast explicit — none of their multi-task/Pareto machinery is used here.

    ``input_dim`` is the number of Go/No-Go-admitted signal FEATURE COLUMNS, not the number of
    signals: some signals contribute several columns. With the default ``learned_asism.feature_columns``
    that is nine (similarity x3, IQA x3, uncertainty, explainability, agreement; distinctiveness's
    column is not yet wired into the learned feature set). Hidden layers are 128 -> 64 -> 32 -> 1.
    """

    def __init__(self, input_dim: int, hidden=(128, 64, 32), dropout=0.2):
        super().__init__()
        # `network` is load-bearing: it determines state_dict keys, so existing ranking_model.pt
        # checkpoints stay loadable across the rename. Do not rename this attribute.
        self.network = mlp([input_dim, *hidden, 1], dropout=dropout)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.network(features).squeeze(-1)


class AdaptiveThresholdNetwork(nn.Module):
    """Produces a [0,1] class-aware threshold from class identity and class context."""

    def __init__(self, number_of_classes: int, context_dim: int, embedding_dim=16,
                 hidden=(64, 32), dropout=0.1):
        super().__init__()
        self.class_embedding = nn.Embedding(number_of_classes, embedding_dim)
        self.network = mlp([context_dim + embedding_dim, *hidden, 1], dropout, final_sigmoid=True)

    def forward(self, class_ids: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        return self.network(torch.cat([self.class_embedding(class_ids), context], dim=-1)).squeeze(-1)


def soft_selection_gate(scores: torch.Tensor, thresholds: torch.Tensor, temperature: float) -> torch.Tensor:
    """Differentiable relaxation of thresholding, for a future end-to-end-trained selector.

    NOT ON THE CURRENT PATH: 06/08b threshold hard, so nothing in the pipeline calls this.
    Retained (and unit-tested) as the entry point an end-to-end variant would need.
    """
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    return torch.sigmoid((scores - thresholds) / temperature)


def pairwise_ranking_loss(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    delta_target = targets[:, None] - targets[None, :]
    valid = delta_target.abs() > 1e-12
    if not valid.any():
        return predictions.sum() * 0.0
    signs = delta_target.sign()[valid]
    delta_prediction = (predictions[:, None] - predictions[None, :])[valid]
    return torch.nn.functional.softplus(-signs * delta_prediction).mean()


## `scripts/asism/04_build_utility_subsets.py`

*195 lines*

**Learned ASISM step 1 - design the experiment that measures utility.** Two phases, and the phase
split is the point.

`--phase feasibility` runs `preflight_check_candidate_pool_inputs` (are the signal artifacts and
the Go/No-Go report there?) and then `pool_feasibility_report` - can this candidate pool actually
support the configured subset design? It **fails closed** if not, before any GPU time is spent.

`--phase build` splits the pool into disjoint train-role and val-role image pools
(`split_image_pool`) and builds ~120 controlled subsets (`build_role_conditioned_subsets`): random,
single-signal and mixed compositions, sizes 100-250, drawn across quantile bands. Each subset
records its `role`, which is what later guarantees zero image overlap between what the Set-Utility
Network trains on and what it is early-stopped on.

`subset_design_config_hash` pins the design so a changed design forces a re-measure.

In [ ]:
# ===== FILE: scripts/asism/04_build_utility_subsets.py =====
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    active_feature_columns, build_role_conditioned_subsets, pool_feasibility_report, split_image_pool, verify_built_subsets,
)
from scripts.asism.candidate_pool import load_candidate_pool  # noqa: E402
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.artifact_contracts import stage2_paths, stage3_paths  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json  # noqa: E402


def preflight_check_candidate_pool_inputs(cfg) -> None:
    """Fail with one combined report naming every missing upstream artifact and its command,
    rather than erroring on the first one and hiding the rest."""
    missing: list[str] = []

    scores_dir = Path(cfg.paths.scores_dir)
    if not scores_dir.is_dir() or not any(scores_dir.glob("*_scores.parquet")):
        missing.append(
            f"  - ASISM signal score artifacts: {scores_dir}/*_scores.parquet\n"
            "    -> python scripts/asism/01_compute_signals.py"
        )

    gonogo_path = Path(cfg.paths.gonogo_report)
    if not gonogo_path.is_file():
        missing.append(
            f"  - Go/No-Go report: {gonogo_path}\n"
            "    -> python scripts/asism/02_gonogo.py"
        )

    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    namespace = str(cfg.split_namespace)
    stage2_manifest = stage2_paths(stage2_cfg, namespace)["manifest_path"]
    if not stage2_manifest.is_file():
        missing.append(
            f"  - Stage 2 synthetic generation manifest: {stage2_manifest}\n"
            "    -> python scripts/generate/01_sample_label_recipes.py\n"
            "    -> python scripts/generate/02_generate_synthetic_images.py"
        )

    if missing:
        raise SystemExit(
            "UPSTREAM GATE: learned-ASISM needs artifacts that don't exist yet for namespace "
            f"{cfg.split_namespace!r}:\n\n" + "\n".join(missing) +
            "\n\nThese require GPU generation and signal computation (RunPod) — not available "
            "locally. Run the listed commands in order, then re-run this command."
        )


def subset_design_config_hash(design) -> str:
    """Hashes only the fields that determine feasibility, so an unrelated config edit elsewhere in
    the file doesn't spuriously invalidate a still-valid feasibility report."""
    relevant = {
        "subset_sizes": list(design.subset_sizes),
        "total_subsets": int(design.total_subsets),
        "random_fraction": float(design.random_fraction),
        "single_signal_fraction": float(design.single_signal_fraction),
        "mixed_fraction": float(design.mixed_fraction),
        "quantile_bins": int(design.quantile_bins),
        "val_pool_fraction": float(design.val_pool_fraction),
        "feasibility_thresholds": OmegaConf.to_container(design.feasibility_thresholds, resolve=True),
    }
    return hash_dict(relevant, length=32)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--phase", choices=["feasibility", "build"], default="feasibility")
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths:
            cfg.paths[key] = str(value)
    cfg.split_namespace = namespace

    learned = cfg.learned_asism
    design = learned.subset_design
    val_fraction = float(design.val_pool_fraction)
    report_path = Path(cfg.paths.subset_design_report)
    config_hash = subset_design_config_hash(design)

    preflight_check_candidate_pool_inputs(cfg)

    if args.phase == "feasibility":
        merged, intended, surviving = load_candidate_pool(cfg)
        columns = active_feature_columns(list(learned.feature_columns), surviving)
        thresholds = OmegaConf.to_container(design.feasibility_thresholds, resolve=True)
        report = pool_feasibility_report(
            merged, columns, intended, list(PRIMARY_ENDPOINT_LABELS),
            list(design.subset_sizes), int(design.quantile_bins), val_fraction, int(design.seed),
            int(design.total_subsets), thresholds,
        )
        # Compute-budget failure is evaluated here too so --phase build can gate on ONE report,
        # not re-derive it — reuses 04b's own estimator so the two scripts never disagree.
        budget_module = _load_04b()
        budget_estimate = budget_module.compute_budget_estimate(cfg, int(design.total_subsets))
        if not budget_estimate["within_budget"]:
            report["failures"].append(
                f"compute estimate above cap: {budget_estimate['estimated_gpu_hours']}h > "
                f"{budget_estimate['max_gpu_hours']}h for {budget_estimate['total_proxy_runs']} "
                "proxy runs implied by subset_design.total_subsets"
            )
            report["passed"] = False
        report["compute_budget_estimate"] = budget_estimate
        report["config_hash"] = config_hash
        report["surviving_signals"] = sorted(surviving)
        report["active_feature_columns"] = columns
        report["git_commit_hash"] = get_git_commit_hash()

        report_path.parent.mkdir(parents=True, exist_ok=True)
        with open(report_path, "w", encoding="utf-8") as handle:
            json.dump(report, handle, indent=2, sort_keys=True)
        print(json.dumps(report, indent=2, sort_keys=True), flush=True)
        print(f"\n-> {report_path}", flush=True)
        if report["passed"]:
            print("PASSED. Review this before --phase build.", flush=True)
            return 0
        print(f"FAILED — {len(report['failures'])} failure(s). --phase build will refuse to run "
              "until subset_design/feasibility_thresholds are revised and this is re-run.", flush=True)
        return 2

    # --phase build: fail closed.
    if not report_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: no feasibility report at {report_path}.\n"
            "Run: python scripts/asism/04_build_utility_subsets.py --phase feasibility"
        )
    report = read_json(report_path)
    if report.get("config_hash") != config_hash:
        raise SystemExit(
            "STALE FEASIBILITY REPORT: configs/stage3_asism.yaml's subset_design/"
            "feasibility_thresholds changed since the report was generated "
            f"(report config_hash={report.get('config_hash')!r}, current={config_hash!r}).\n"
            "Re-run: python scripts/asism/04_build_utility_subsets.py --phase feasibility"
        )
    if not report.get("passed", False):
        raise SystemExit(
            f"FEASIBILITY GATE: {report_path} reports {len(report.get('failures', []))} failure(s):\n"
            + "\n".join(f"  - {reason}" for reason in report.get("failures", []))
            + "\n\nsubset_design/feasibility_thresholds must be revised in configs/stage3_asism.yaml "
              "and --phase feasibility re-run BEFORE --phase build. No automatic adjustment."
        )

    merged, _, surviving = load_candidate_pool(cfg)
    columns = active_feature_columns(list(learned.feature_columns), surviving)
    if report.get("surviving_signals") != sorted(surviving) or report.get("active_feature_columns") != columns:
        raise SystemExit(
            "STALE FEASIBILITY REPORT: Go/No-Go admitted signals changed since feasibility was measured.\n"
            "Re-run: python scripts/asism/04_build_utility_subsets.py --phase feasibility"
        )
    train_frame, val_frame = split_image_pool(merged, val_fraction, int(design.seed))
    train_total = round(int(design.total_subsets) * (1 - val_fraction))
    val_total = int(design.total_subsets) - train_total
    records = build_role_conditioned_subsets(
        train_frame, val_frame, columns, train_total, val_total,
        list(design.subset_sizes), float(design.random_fraction),
        float(design.single_signal_fraction), float(design.mixed_fraction),
        int(design.seed), quantile_bins=int(design.quantile_bins),
    )
    verify_built_subsets(records, train_total, val_total)  # last check before anything is written

    output = Path(cfg.paths.utility_subsets)
    output.parent.mkdir(parents=True, exist_ok=True)
    with open(output, "x", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")
    n_train = sum(1 for r in records if r["role"] == "train")
    n_val = sum(1 for r in records if r["role"] == "val")
    print(f"{len(records)} frozen subset recipes ({n_train} train-role, {n_val} val-role) -> {output}")
    print("Next: run identical proxy training for every recipe and write utility_results.jsonl.")
    return 0


def _load_04b():
    import importlib.util
    script = Path(__file__).with_name("04b_evaluate_utility_subsets.py")
    spec = importlib.util.spec_from_file_location("utility_subsets_04b", script)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/04b_evaluate_utility_subsets.py`

*133 lines*

**Learned ASISM step 2 - the expensive one. This is where "utility" stops being a proxy signal and
becomes a measurement.**

For every frozen subset recipe it trains a real (small) proxy classifier on
`classifier_train` + the subset, trains a real-only baseline with the **same** budget, and records
`augmented_macro_AUROC - real_only_macro_AUROC`. Evaluation is restricted to
`asism_tuning_heldout` - never `final_eval_heldout`.

Engineering that matters: `--phase estimate` computes the run count and GPU-hour estimate and
applies the budget gate *before* any training; `--phase run` is **resumable at
(subset, fold, seed) granularity**, so a dropped pod costs one cell, not the whole matrix.

After an exposure filter (each image must appear in at least `minimum_image_exposures` subsets),
roughly 96 usable `(subset, utility)` pairs remain. Those 96 numbers are the entire supervision
budget for everything that follows.

In [ ]:
# ===== FILE: scripts/asism/04b_evaluate_utility_subsets.py =====
#!/usr/bin/env python3
"""Measure real-only and real+subset macro-AUROC for every frozen utility recipe.

Runs are resumable at (subset, fold, seed) granularity. The same proxy budget is used for the
baseline and augmented run; evaluation is restricted to asism_tuning_heldout.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import read_jsonl  # noqa: E402
from scripts.utils.artifact_contracts import stage2_paths, stage3_paths  # noqa: E402
from scripts.utils.classifier import TrainingBudget, evaluate_classifier, records_from_split, records_from_synthetic_manifest, train_classifier  # noqa: E402
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import hash_dict  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402


def compute_budget_estimate(cfg, n_recipes: int) -> dict:
    """Pre-execution accounting, mirroring 03_tune_freeze_select.py's gate: must be computed and
    checked BEFORE any proxy run, never reduced after seeing results."""
    budget = cfg.learned_asism.compute_budget
    hours_each = float(budget.hours_per_proxy_run_estimate)
    total_runs = 1 + n_recipes  # 1 real-only baseline (shared across recipes) + 1 run per recipe
    estimated_hours = total_runs * hours_each
    max_hours = float(budget.max_gpu_hours)
    return {
        "n_recipes": n_recipes,
        "total_proxy_runs": total_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
        "formula": "total = 1 (real-only baseline) + n_recipes",
    }


def enforce_compute_budget(cfg, n_recipes: int) -> dict:
    estimate = compute_budget_estimate(cfg, n_recipes)
    if not estimate["within_budget"]:
        raise SystemExit(
            "COMPUTE BUDGET GATE: learned-ASISM utility-subset evaluation exceeds its declared budget.\n"
            f"  estimated: {estimate['estimated_gpu_hours']} GPU-hours "
            f"({estimate['total_proxy_runs']} proxy runs x {estimate['hours_per_proxy_run_estimate']}h)\n"
            f"  budget:    {estimate['max_gpu_hours']} GPU-hours\n\n"
            "Reduce learned_asism.subset_design.total_subsets in configs/stage3_asism.yaml (or pass "
            "--limit) BEFORE running --phase run.\n"
            "Reducing it after seeing results would let the outcome influence the search design."
        )
    return estimate


def train_and_score(train_records, eval_records, cfg, seed, checkpoint, tag):
    budget = TrainingBudget(max_steps=int(cfg.tuning.coarse.proxy_max_steps),
                            batch_size=int(cfg.tuning.proxy.batch_size),
                            learning_rate=float(cfg.tuning.proxy.learning_rate), seed=seed,
                            eval_every_n_steps=0)
    model, _, _ = train_classifier(
        train_records=train_records, val_records=[], budget=budget,
        dropout_p=float(cfg.tuning.proxy.dropout_p),
        pretrained_source=str(cfg.tuning.proxy.pretrained_source),
        resolution=int(cfg.tuning.proxy.resolution), progress_desc=tag,
        checkpoint_path=checkpoint, resume_from=checkpoint if checkpoint.is_file() else None,
        checkpoint_metadata={"dataset_id": tag, "condition_or_draw_id": tag,
                             "config_hash": hash_dict({"tag": tag, "seed": seed}, length=64),
                             "label_policy": "chexpert_mask_uncertain_v1"})
    return float(evaluate_classifier(model, eval_records, int(cfg.tuning.proxy.resolution))["macro_auroc"])


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--limit", type=int, default=None, help="Pilot-only number of recipes")
    parser.add_argument("--phase", choices=["estimate", "run"], default="run")
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3"); stage2 = load_named_config("stage2_generation.yaml", "stage2")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    for key, value in stage2_paths(stage2, namespace).items():
        if key in stage2.paths: stage2.paths[key] = str(value)
    if args.phase == "estimate":
        # Budget must be checked BEFORE the subsets exist (docs/stages2_to_5_plan.md §4.7's rule:
        # the search space is sized before any results are seen, never after). Uses the CONFIGURED
        # total_subsets, not an actual utility_subsets.jsonl — that file need not exist yet.
        n_recipes = int(cfg.learned_asism.subset_design.total_subsets)
        if args.limit is not None:
            n_recipes = min(n_recipes, args.limit)
        estimate = compute_budget_estimate(cfg, n_recipes)
        print(json.dumps(estimate, indent=2), flush=True)
        if not estimate["within_budget"]:
            print(
                "\nOVER BUDGET — reduce learned_asism.subset_design.total_subsets in "
                "configs/stage3_asism.yaml (or pass --limit) before running --phase run.",
                flush=True,
            )
            return 2
        print("\nWithin budget. Proceed: --phase run", flush=True)
        return 0

    recipes = read_jsonl(Path(cfg.paths.utility_subsets))
    if args.limit is not None: recipes = recipes[:args.limit]
    enforce_compute_budget(cfg, len(recipes))
    output = Path(cfg.paths.utility_results); output.parent.mkdir(parents=True, exist_ok=True)
    completed = {(row["subset_id"], int(row["fold"]), int(row["seed"])) for row in read_jsonl(output)} if output.is_file() else set()
    stage1 = load_stage1_config(); image_root = Path(stage1.paths.images_dir) / namespace
    real = records_from_split(load_split("classifier_train", namespace, purpose="schema_validation", caller="learned_asism_utility"), image_root / "classifier_train")
    evaluation = records_from_split(load_split("asism_tuning_heldout", namespace, purpose="schema_validation", caller="learned_asism_utility"), image_root / "asism_tuning_heldout")
    checkpoint_dir = Path(cfg.paths.learned_dir) / "proxy_checkpoints"; checkpoint_dir.mkdir(parents=True, exist_ok=True)
    seed = int(cfg.learned_asism.subset_design.seed)
    baseline_checkpoint = checkpoint_dir / f"real_only_seed{seed}.pt"
    baseline_score = train_and_score(real, evaluation, cfg, seed, baseline_checkpoint, f"learned-real-only-s{seed}")
    with open(output, "a", encoding="utf-8", buffering=1) as handle:
        for index, recipe in enumerate(recipes):
            key = (recipe["subset_id"], 0, seed)
            if key in completed: continue
            synthetic = records_from_synthetic_manifest(Path(stage2.paths.manifest_path), Path(stage2.paths.images_dir), recipe["image_ids"])
            checkpoint = checkpoint_dir / f"{recipe['subset_id']}_seed{seed}.pt"
            augmented = train_and_score(real + synthetic, evaluation, cfg, seed, checkpoint, f"learned-{recipe['subset_id']}-s{seed}")
            row = {"subset_id": recipe["subset_id"], "fold": 0, "seed": seed,
                   "role": recipe.get("role"),
                   "real_only_macro_auroc": baseline_score, "augmented_macro_auroc": augmented,
                   "utility_delta": augmented - baseline_score}
            handle.write(json.dumps(row, sort_keys=True) + "\n")
            print(f"[{index + 1}/{len(recipes)}] {recipe['subset_id']}: utility={row['utility_delta']:+.6f}", flush=True)
    return 0


if __name__ == "__main__": raise SystemExit(main())


## `scripts/asism/05_train_learned_asism.py`

*432 lines*

**Learned ASISM step 3 - turn 96 measured subset utilities into a per-image ranker.**

1. **Train the Set-Utility Network** on the measured subsets. The train/val split is **not** a
   random split of subsets - subsets share images, so that would leak. It splits by the `role` each
   subset carries from `04 --phase build` (`split_subsets_by_role`), and
   `image_overlap_fraction` is asserted to confirm zero overlap.
2. **Generate per-image targets** (`marginal_targets`): the size-normalised leave-one-out marginal
   utility `(U(S) - U(S\i)) x (n-1)`, averaged over the subsets each image appeared in. The Banzhaf
   MSR alternative is computed from the same data and its Spearman agreement recorded in the
   manifest.
3. **Normalise the targets** (C2, default `standardize`) so one noisy measurement cannot dominate.
4. **Train the ranking network** (`train_ranker`) with Smooth-L1 + pairwise ranking loss, reporting
   `pairwise_ranking_accuracy`.

`padded_batch` handles variable-size subsets with a mask. **The script intentionally stops when
proxy results are missing** - that refusal is what prevents fabricated image targets.

In [ ]:
# ===== FILE: scripts/asism/05_train_learned_asism.py =====
#!/usr/bin/env python3
"""Train set utility and image ranking models from measured proxy-subset utility.

The command intentionally stops when proxy results are missing. This prevents fabricated image
targets. Image targets are estimated as mean leave-one-out marginal utility across the measured
subsets in which each image appeared.

Train/validation split is NOT a random split of subsets — subsets share images, so that would leak.
Every subset already carries a "role" ("train" | "val") from 04_build_utility_subsets.py's
--phase build, which built train-role and val-role subsets from two disjoint image pools. Splitting
by that role therefore guarantees zero image overlap between what the SetUtilityNetwork is trained
on and what it is validated on, and gives the ranking network the same disjoint split for free
(an image can only ever appear in train-role OR val-role subsets, never both).
"""
from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path

import numpy as np
import scipy.stats
import torch
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    active_feature_columns, banzhaf_msr_targets, contributing_signals, loo_vs_banzhaf_diagnostic,
    normalize_ranking_targets, read_jsonl, safe_feature_frame, validate_utility_results,
)
from scripts.asism.candidate_pool import load_candidate_pool  # noqa: E402
from scripts.asism.models import MultiSignalUtilityRankingNetwork, SetUtilityNetwork, pairwise_ranking_loss  # noqa: E402
from scripts.utils.artifact_contracts import stage3_paths  # noqa: E402
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json, sha256_file, write_frozen_json  # noqa: E402
from scripts.utils.seed import set_seed  # noqa: E402


def padded_batch(records, feature_lookup, device):
    maximum = max(len(row["image_ids"]) for row in records)
    dimension = len(next(iter(feature_lookup.values())))
    features = torch.zeros(len(records), maximum, dimension, dtype=torch.float32, device=device)
    mask = torch.zeros(len(records), maximum, dtype=torch.bool, device=device)
    for row_index, row in enumerate(records):
        vectors = [feature_lookup[image_id] for image_id in row["image_ids"] if image_id in feature_lookup]
        if vectors:
            tensor = torch.as_tensor(np.asarray(vectors), dtype=torch.float32, device=device)
            features[row_index, :len(vectors)] = tensor
            mask[row_index, :len(vectors)] = True
    return features, mask


def split_subsets_by_role(subsets: list[dict]) -> tuple[list[dict], list[dict]]:
    missing_role = [row["subset_id"] for row in subsets if "role" not in row]
    if missing_role:
        raise SystemExit(
            f"{len(missing_role)} subset(s) have no 'role' field (e.g. {missing_role[:3]}). "
            "Rebuild utility_subsets.jsonl with 04_build_utility_subsets.py --phase build."
        )
    train = [row for row in subsets if row["role"] == "train"]
    val = [row for row in subsets if row["role"] == "val"]
    return train, val


def image_overlap_fraction(subsets_a: list[dict], subsets_b: list[dict]) -> float:
    images_a = {image_id for row in subsets_a for image_id in row["image_ids"]}
    images_b = {image_id for row in subsets_b for image_id in row["image_ids"]}
    union = images_a | images_b
    if not union:
        return 0.0
    return len(images_a & images_b) / len(union)


@torch.no_grad()
def predict_utility(model, subsets, lookup, device) -> list[float]:
    model.eval()
    if not subsets:
        return []
    x, mask = padded_batch(subsets, lookup, device)
    return model(x, mask).cpu().numpy().tolist()


def train_set_model(model, train_subsets, val_subsets, utility_by_id, lookup, config, device,
                    min_val_subsets: int) -> dict:
    """Trains only on train_subsets. Early-stops on val_subsets MSE (image-disjoint by construction)
    using early_stopping_patience — this is fitting a fixed, already-measured dataset, not the
    proxy-classifier candidate search that docs/stages2_to_5_plan.md §4.7/§7.1 forbids early
    stopping for, so patience-based stopping is appropriate here.

    Image-disjoint validation is a design invariant, not an optional nicety: if val_subsets doesn't
    meet min_val_subsets (feasibility_thresholds.min_val_subsets), this refuses to train at all
    rather than silently skipping early stopping and validation reporting.
    """
    if len(val_subsets) < min_val_subsets:
        raise SystemExit(
            f"Learned ASISM requires >= {min_val_subsets} val-role subsets for image-disjoint "
            f"validation (feasibility_thresholds.min_val_subsets); got {len(val_subsets)}. "
            "Training refuses to proceed without real held-out validation — rebuild "
            "utility_subsets.jsonl (04_build_utility_subsets.py --phase build) with a "
            "val_pool_fraction/total_subsets that satisfies this, or revise the threshold with "
            "explicit sign-off, before retrying."
        )

    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config.learning_rate),
                                  weight_decay=float(config.weight_decay))
    rng = np.random.default_rng(42)
    patience = int(config.early_stopping_patience)
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    total_epochs = int(config.epochs)
    stopped_at_epoch = total_epochs

    for epoch in range(total_epochs):
        model.train()
        order = rng.permutation(len(train_subsets))
        for start in range(0, len(order), int(config.batch_size)):
            rows = [train_subsets[index] for index in order[start:start + int(config.batch_size)]]
            x, mask = padded_batch(rows, lookup, device)
            target = torch.tensor([utility_by_id[row["subset_id"]] for row in rows], device=device)
            prediction = model(x, mask)
            loss = float(config.utility_mse_weight) * torch.nn.functional.mse_loss(prediction, target)
            loss = loss + float(config.pairwise_ranking_weight) * pairwise_ranking_loss(prediction, target)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

        val_predictions = predict_utility(model, val_subsets, lookup, device)
        val_targets = [utility_by_id[row["subset_id"]] for row in val_subsets]
        val_loss = float(np.mean((np.asarray(val_predictions) - np.asarray(val_targets)) ** 2))
        if val_loss < best_val_loss - 1e-9:
            best_val_loss = val_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                stopped_at_epoch = epoch + 1
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_predictions = predict_utility(model, val_subsets, lookup, device)
    val_targets = [utility_by_id[row["subset_id"]] for row in val_subsets]
    mae = float(np.mean(np.abs(np.asarray(val_predictions) - np.asarray(val_targets))))
    spearman, spearman_undefined_reason = None, None
    correlation = scipy.stats.spearmanr(val_predictions, val_targets)
    statistic = getattr(correlation, "statistic", None)
    raw_spearman = float(statistic) if statistic is not None else float(correlation[0])
    if math.isnan(raw_spearman):
        # scipy returns NaN when predictions or targets are (numerically) constant — correlation is
        # undefined, not zero. NaN is not valid JSON, so this must never reach the manifest as a
        # bare float; None + an explicit reason is the only honest representation.
        spearman_undefined_reason = "constant_predictions_or_targets"
    else:
        spearman = raw_spearman

    return {
        "n_train_subsets": len(train_subsets),
        "n_val_subsets": len(val_subsets),
        "image_overlap_fraction": image_overlap_fraction(train_subsets, val_subsets),
        "early_stopped_at_epoch": stopped_at_epoch,
        "spearman": spearman,
        "spearman_undefined_reason": spearman_undefined_reason,
        "mae": mae,
    }


@torch.no_grad()
def marginal_targets(model, subsets, lookup, device):
    """Mean SIZE-NORMALIZED predicted U(S)-U(S without i); never copy the subset target to members.

    The raw leave-one-out difference is not comparable across subsets of different sizes, so it
    cannot be averaged as-is. SetUtilityNetwork pools its per-image encodings by MEAN
    (`set_utility_network.pooling: "mean"`, models.py::SetUtilityNetwork.encode_set), so for a subset
    S of size n with mean encoding m_S:

        m_S - m_(S\\i) = (f(x_i) - m_S) / (n - 1)

    and therefore U(S) - U(S\\i) shrinks like 1/(n-1). With subset_sizes spanning 100..250, the same
    image quality would yield targets differing by ~2.5x purely from which subsets the image landed
    in — noise injected straight into the ranking network's supervision, unrelated to image quality.

    Multiplying by (n - 1) removes that factor and leaves a size-invariant quantity proportional to
    how far the image's encoding deviates from its subset's mean, which is the quantity that
    actually carries "is this image better than its peers". Only then is averaging across an image's
    subsets meaningful.

    `raw_contribution * (n - 1)` is exact for the linear part of the head and a first-order
    approximation for its nonlinearity; both are far closer to size-invariant than the unscaled
    difference, which is size-dependent by construction.
    """
    totals, counts = {}, {}
    model.eval()
    for row in subsets:
        ids = [image_id for image_id in row["image_ids"] if image_id in lookup]
        if len(ids) < 2:
            continue
        size_normalizer = len(ids) - 1
        full_x, full_mask = padded_batch([{"image_ids": ids}], lookup, device)
        full = float(model(full_x, full_mask).item())
        for image_id in ids:
            reduced = [candidate for candidate in ids if candidate != image_id]
            x, mask = padded_batch([{"image_ids": reduced}], lookup, device)
            contribution = (full - float(model(x, mask).item())) * size_normalizer
            totals[image_id] = totals.get(image_id, 0.0) + contribution
            counts[image_id] = counts.get(image_id, 0) + 1
    return {image_id: totals[image_id] / counts[image_id] for image_id in totals}, counts


def train_ranker(model, features, targets, config, device):
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config.learning_rate))
    ids = list(targets)
    rng = np.random.default_rng(43)
    model.train()
    for _ in range(int(config.epochs)):
        rng.shuffle(ids)
        for start in range(0, len(ids), int(config.batch_size)):
            batch = ids[start:start + int(config.batch_size)]
            x = torch.tensor(np.asarray([features[i] for i in batch]), dtype=torch.float32, device=device)
            y = torch.tensor([targets[i] for i in batch], dtype=torch.float32, device=device)
            prediction = model(x)
            loss = torch.nn.functional.smooth_l1_loss(prediction, y)
            loss += float(config.pairwise_ranking_weight) * pairwise_ranking_loss(prediction, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()


@torch.no_grad()
def pairwise_ranking_accuracy(model, features, targets, device) -> float | None:
    """Fraction of correctly ordered pairs among images with distinct targets. 0.5 = chance level.

    This measures how well the ranker reproduces the SetUtilityNetwork's OWN marginal-utility
    estimates on held-out images — i.e. distillation fidelity — not independent evidence of
    downstream classifier utility. That independent evidence, if it exists, comes from proxy
    verification against real training runs (Phase 2 §2.7/§2.8), not from this metric.
    """
    model.eval()
    ids = list(targets)
    if len(ids) < 2:
        return None
    x = torch.tensor(np.asarray([features[i] for i in ids]), dtype=torch.float32, device=device)
    predictions = model(x).cpu().numpy()
    target_values = np.asarray([targets[i] for i in ids])
    correct, total = 0, 0
    for i in range(len(ids)):
        delta_target = target_values[i + 1:] - target_values[i]
        delta_prediction = predictions[i + 1:] - predictions[i]
        valid = np.abs(delta_target) > 1e-12
        if not valid.any():
            continue
        total += int(valid.sum())
        correct += int((np.sign(delta_target[valid]) == np.sign(delta_prediction[valid])).sum())
    return (correct / total) if total else None


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    cfg.split_namespace = namespace
    # BUG FIXED 2026-08-21: torch's global RNG was never seeded, so SetUtilityNetwork's and
    # MultiSignalUtilityRankingNetwork's weight initialization (and dropout) differed every run —
    # confirmed to occasionally collapse the ranker enough that 06_learn_thresholds_select.py
    # selects 0 images. Reuses learned_asism.subset_design.seed (already the canonical seed for
    # this pipeline's subset construction, per 04_build_utility_subsets.py) rather than adding a
    # second seed knob that could silently drift out of sync with it.
    set_seed(int(cfg.learned_asism.subset_design.seed))
    subset_path, result_path = Path(cfg.paths.utility_subsets), Path(cfg.paths.utility_results)
    if not subset_path.is_file() or not result_path.is_file():
        raise SystemExit("Learned ASISM needs frozen utility_subsets.jsonl and measured utility_results.jsonl; run Stage 3 proxy experiments first.")
    subsets, results = read_jsonl(subset_path), read_jsonl(result_path)
    train_subsets, val_subsets = split_subsets_by_role(subsets)
    result_frame = validate_utility_results(subsets, results)
    utility = dict(zip(result_frame.subset_id, result_frame.utility_delta))

    merged, _, surviving = load_candidate_pool(cfg)
    columns = active_feature_columns(list(cfg.learned_asism.feature_columns), surviving)
    design_report = read_json(Path(cfg.paths.subset_design_report))
    if design_report.get("surviving_signals") != sorted(surviving) or \
            design_report.get("active_feature_columns") != columns:
        raise SystemExit(
            "UPSTREAM GATE: utility subsets were designed for a different Go/No-Go feature set. "
            "Re-run 04_build_utility_subsets.py --phase feasibility and --phase build before training."
        )
    normalized, normalization = safe_feature_frame(merged, columns)
    lookup = {str(image_id): row.to_numpy(np.float32) for image_id, (_, row) in zip(merged.image_id, normalized.iterrows())}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_cfg = cfg.learned_asism.set_utility_network
    superset_conditioning = bool(set_cfg.get("superset_conditioning", False))
    set_model = SetUtilityNetwork(
        len(columns), tuple(set_cfg.image_hidden_dims), tuple(set_cfg.utility_hidden_dims),
        superset_conditioning=superset_conditioning,
    ).to(device)
    if superset_conditioning:
        # Xie et al., ICLR 2024: concatenate a fixed permutation-invariant summary of the FULL
        # candidate pool. Computed from the frozen normalized feature frame — the same pool every
        # subset is drawn from; no held-out split is read.
        pool_matrix = normalized.to_numpy(np.float64)
        set_model.set_superset_summary(
            np.concatenate([pool_matrix.mean(axis=0), pool_matrix.std(axis=0)]).astype(np.float32)
        )
    min_val_subsets = int(cfg.learned_asism.subset_design.feasibility_thresholds.min_val_subsets)
    validation = train_set_model(set_model, train_subsets, val_subsets, utility, lookup, set_cfg, device,
                                 min_val_subsets)

    minimum = int(cfg.learned_asism.subset_design.minimum_image_exposures)

    # Leave-one-out marginal targets (the current, default supervision for the ranking network).
    loo_train, train_exposures = marginal_targets(set_model, train_subsets, lookup, device)
    loo_val, val_exposures = marginal_targets(set_model, val_subsets, lookup, device)
    loo_train = {key: value for key, value in loo_train.items() if train_exposures[key] >= minimum}
    loo_val = {key: value for key, value in loo_val.items() if val_exposures[key] >= minimum}

    # Data-Banzhaf MSR targets from the SAME measured subsets — no extra proxy training, no GPU.
    # Always computed so the LOO-vs-Banzhaf rank-agreement diagnostic can be recorded; only USED as
    # the ranking target when explicitly selected (banzhaf stays optional until that diagnostic is
    # reviewed — docs/stages2_to_5_plan.md §4).
    banzhaf_train, _ = banzhaf_msr_targets(train_subsets, utility, minimum_in_subsets=minimum)
    banzhaf_val, _ = banzhaf_msr_targets(val_subsets, utility, minimum_in_subsets=minimum)
    # Keep only images present in the frozen feature lookup, exactly as the leave-one-out path does
    # (marginal_targets skips ids missing from `lookup`); the ranker can only be fed images it has a
    # feature vector for.
    banzhaf_train = {key: value for key, value in banzhaf_train.items() if key in lookup}
    banzhaf_val = {key: value for key, value in banzhaf_val.items() if key in lookup}
    banzhaf_diagnostic = loo_vs_banzhaf_diagnostic(loo_train, banzhaf_train)

    rank_cfg = cfg.learned_asism.ranking_network
    target_source = str(rank_cfg.get("target_source", "leave_one_out"))
    if target_source not in {"leave_one_out", "banzhaf_msr"}:
        raise SystemExit(
            f"learned_asism.ranking_network.target_source must be 'leave_one_out' or 'banzhaf_msr', "
            f"got {target_source!r}"
        )
    if target_source == "banzhaf_msr":
        train_targets, val_targets_raw = banzhaf_train, banzhaf_val
    else:
        train_targets, val_targets_raw = loo_train, loo_val
    if len(train_targets) < 2:
        raise SystemExit(
            f"Too few train-role images have sufficient subset exposure for image-ranking training "
            f"(target_source={target_source})."
        )

    # Transform the distilled target before the Smooth-L1 + pairwise loss so one noisy proxy
    # measurement cannot dominate training on ~96 images (order-preserving; pairwise term unchanged).
    target_normalization = str(rank_cfg.get("target_normalization", "standardize"))
    target_winsorize = float(rank_cfg.get("target_winsorize_quantile", 0.0))
    train_targets = normalize_ranking_targets(train_targets, target_normalization, target_winsorize)
    val_targets_raw = normalize_ranking_targets(val_targets_raw, target_normalization, target_winsorize)

    ranker = MultiSignalUtilityRankingNetwork(len(columns), tuple(rank_cfg.hidden_dims), float(rank_cfg.dropout)).to(device)
    train_ranker(ranker, lookup, train_targets, rank_cfg, device)
    ranker_pairwise_accuracy = pairwise_ranking_accuracy(ranker, lookup, val_targets_raw, device)

    output = Path(cfg.paths.learned_dir); output.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": set_model.state_dict(), "input_dim": len(columns),
                "superset_conditioning": superset_conditioning}, output / "set_utility_model.pt")
    torch.save({"state_dict": ranker.state_dict(), "input_dim": len(columns)}, output / "ranking_model.pt")
    with open(output / "image_marginal_targets.jsonl", "w", encoding="utf-8") as handle:
        for image_id, target in sorted(train_targets.items()):
            handle.write(json.dumps({"image_id": image_id, "marginal_utility": target,
                                     "exposures": train_exposures.get(image_id, 0), "role": "train",
                                     "target_source": target_source,
                                     "target_normalization": target_normalization}) + "\n")
        for image_id, target in sorted(val_targets_raw.items()):
            handle.write(json.dumps({"image_id": image_id, "marginal_utility": target,
                                     "exposures": val_exposures.get(image_id, 0), "role": "val",
                                     "target_source": target_source,
                                     "target_normalization": target_normalization}) + "\n")
    signals_used = contributing_signals(columns)
    manifest = {"schema_version": 1, "method": "set_utility_to_marginal_ranking_v1", "frozen": True,
                "feature_columns": columns, "normalization": normalization,
                "contributing_signals": signals_used,
                # Mirrors 02_gonogo.py's asism_variant_status rule for the WEIGHTED selector, applied
                # here to the learned one: a ranking network fed by fewer than three admitted signals
                # is reported as an ablation/alternative, not as the primary method (§4.6).
                "learned_variant_status": (
                    "primary" if len(signals_used) >= 3 else
                    f"reduced_variant — only {len(signals_used)} signal(s) ({signals_used}) survived "
                    "Go/No-Go and feed the ranking network; report this selector as an "
                    "ablation/alternative rather than the primary method (§4.6)"
                ),
                "utility_definition": "augmented_macro_auroc-real_only_macro_auroc",
                "n_subsets": len(subsets), "n_train_subsets": len(train_subsets), "n_val_subsets": len(val_subsets),
                "n_rank_targets": len(train_targets),
                "subset_sha256": sha256_file(subset_path), "results_sha256": sha256_file(result_path),
                "config_hash": hash_dict(OmegaConf.to_container(cfg.learned_asism, resolve=True), length=64),
                "validation": validation,
                "target_generation": {
                    "method": target_source,
                    "default": "leave_one_out",
                    "target_normalization": target_normalization,
                    "target_winsorize_quantile": target_winsorize,
                    "set_superset_conditioning": superset_conditioning,
                    "banzhaf_status": (
                        "in_use" if target_source == "banzhaf_msr"
                        else "optional_pending_diagnostic_review"
                    ),
                    "n_leave_one_out_train_targets": len(loo_train),
                    "n_banzhaf_train_targets": len(banzhaf_train),
                    "loo_vs_banzhaf_diagnostic": banzhaf_diagnostic,
                    "note": (
                        "Banzhaf MSR targets are computed from the same measured subsets as the "
                        "leave-one-out targets — no extra proxy training, no GPU cost. Switch the "
                        "ranking network onto them (target_source: banzhaf_msr) only after reviewing "
                        "loo_vs_banzhaf_diagnostic."
                    ),
                },
                "ranker_validation": {
                    "pairwise_accuracy": ranker_pairwise_accuracy,
                    "measures": "distillation_fidelity_not_downstream_utility",
                    "n_val_images": len(val_targets_raw),
                }}
    write_frozen_json(output / "learned_training_manifest.json", manifest)
    print(f"Learned ASISM models -> {output}")
    print(f"Set-utility validation: spearman={validation['spearman']}, mae={validation['mae']}, "
          f"image_overlap_fraction={validation['image_overlap_fraction']}")
    print(f"Ranker held-out pairwise accuracy: {ranker_pairwise_accuracy}")
    print(f"Ranking target source: {target_source} (default leave_one_out). "
          f"LOO-vs-Banzhaf Spearman={banzhaf_diagnostic['spearman']} "
          f"over {banzhaf_diagnostic['n_images_compared']} images "
          f"(reason={banzhaf_diagnostic['spearman_undefined_reason']}).")
    print("Next: learn/freeze class thresholds and emit learned selected_manifest.jsonl.")
    return 0


if __name__ == "__main__": raise SystemExit(main())


## `scripts/asism/06_learn_thresholds_select.py`

*142 lines*

**Learned ASISM step 4 - the fixed-ratio learned threshold. An intermediate/ablation artifact, not
a Stage 4 condition.**

`optimal_threshold` picks, per class, the threshold that best serves a frozen utility/budget
objective at a *fixed target selection ratio*; `context_vector` builds the class context it is
conditioned on. A hard technical safety gate always runs first, and targets are chosen on ASISM
tuning data only.

Its output (`learned/selected_manifest.jsonl`) is kept for the ablation table and is deliberately
left untouched by step 09. The **adaptive** path that actually feeds condition C continues at
`07_build_threshold_contexts.py`.

In [ ]:
# ===== FILE: scripts/asism/06_learn_thresholds_select.py =====
#!/usr/bin/env python3
"""Learn class-aware thresholds from utility-derived ranking scores and select images.

Threshold targets are chosen on ASISM tuning data only by a frozen utility/budget objective, then
distilled into the requested adaptive network. A hard technical safety gate always runs first.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import apply_feature_frame  # noqa: E402
from scripts.asism.candidate_pool import load_candidate_pool  # noqa: E402
from scripts.asism.models import AdaptiveThresholdNetwork, MultiSignalUtilityRankingNetwork  # noqa: E402
from scripts.utils.artifact_contracts import stage3_paths  # noqa: E402
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import read_json, sha256_file, write_frozen_json  # noqa: E402
from scripts.utils.seed import set_seed  # noqa: E402


def context_vector(scores: np.ndarray, prevalence: float, candidates: int, target_count: int) -> np.ndarray:
    q25, q50, q75 = np.quantile(scores, [0.25, 0.5, 0.75]) if len(scores) else (0, 0, 0)
    return np.asarray([prevalence, np.log1p(candidates), scores.mean() if len(scores) else 0,
                       scores.std() if len(scores) else 0, q25, q50, q75,
                       target_count / max(candidates, 1)], dtype=np.float32)


def optimal_threshold(scores: np.ndarray, target_count: int, budget_weight: float) -> float:
    """Frozen one-dimensional search: retain utility while respecting the target count."""
    if not len(scores): return 1.0
    best = None
    for threshold in np.linspace(0.0, 1.0, 101):
        selected = scores[scores >= threshold]
        utility = float(selected.mean()) if len(selected) else -1.0
        budget_error = abs(len(selected) - target_count) / max(target_count, 1)
        objective = utility - budget_weight * budget_error
        candidate = (objective, -threshold, threshold)
        if best is None or candidate > best: best = candidate
    return float(best[2])


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    cfg.split_namespace = namespace
    # BUG FIXED 2026-08-21: this script never seeded torch's global RNG — AdaptiveThresholdNetwork's
    # init was non-deterministic across runs. Confirmed to occasionally produce thresholds so high
    # that ZERO images clear them (see the guard added below). Same fix as 05/08_train_*.py.
    set_seed(int(cfg.learned_asism.subset_design.seed))
    learned_dir = Path(cfg.paths.learned_dir)
    training_manifest = read_json(learned_dir / "learned_training_manifest.json")

    merged, intended, _ = load_candidate_pool(cfg)
    columns = list(training_manifest["feature_columns"])
    normalized = apply_feature_frame(merged, columns, training_manifest["normalization"])
    checkpoint = torch.load(learned_dir / "ranking_model.pt", map_location="cpu", weights_only=True)
    rank_cfg = cfg.learned_asism.ranking_network
    ranker = MultiSignalUtilityRankingNetwork(len(columns), tuple(rank_cfg.hidden_dims), float(rank_cfg.dropout))
    ranker.load_state_dict(checkpoint["state_dict"]); ranker.eval()
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32))).numpy()
    merged = merged.copy(); merged["learned_ranking_score"] = 1.0 / (1.0 + np.exp(-raw))

    threshold_cfg = cfg.learned_asism.threshold_network
    contexts, targets, class_ids = [], [], []
    per_class_candidates = {}
    for class_id, label in enumerate(PRIMARY_ENDPOINT_LABELS):
        ids = [image_id for image_id in merged.image_id if int(intended.get(str(image_id), {}).get(label, 0)) == 1]
        candidates = merged.loc[merged.image_id.isin(ids), "learned_ranking_score"].to_numpy()
        per_class_candidates[label] = set(map(str, ids))
        target_count = min(int(threshold_cfg.max_selected_per_label), max(int(threshold_cfg.min_selected_per_label), len(candidates) // 2))
        contexts.append(context_vector(candidates, len(candidates) / max(len(merged), 1), len(candidates), target_count))
        targets.append(optimal_threshold(candidates, target_count, float(threshold_cfg.budget_weight)))
        class_ids.append(class_id)
    x = torch.tensor(np.asarray(contexts)); y = torch.tensor(targets, dtype=torch.float32); ids_tensor = torch.tensor(class_ids)
    model = AdaptiveThresholdNetwork(len(PRIMARY_ENDPOINT_LABELS), x.shape[1], int(threshold_cfg.class_embedding_dim),
                                     tuple(threshold_cfg.hidden_dims), float(threshold_cfg.dropout))
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(threshold_cfg.learning_rate))
    model.train()
    for _ in range(int(threshold_cfg.epochs)):
        prediction = model(ids_tensor, x)
        loss = torch.nn.functional.mse_loss(prediction, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): learned_thresholds = model(ids_tensor, x).numpy()

    selected, rejected = set(), []
    safety = cfg.learned_asism.safety
    for _, row in merged.iterrows():
        image_id = str(row.image_id)
        if bool(safety.reject_invalid_iqa) and "iqa_valid" in row and not bool(row.iqa_valid):
            rejected.append({"image_id": image_id, "reason": "invalid_iqa_safety_gate"}); continue
        if bool(safety.reject_near_duplicates) and bool(row.get("novelty_is_near_duplicate", False)):
            rejected.append({"image_id": image_id, "reason": "near_duplicate_safety_gate"}); continue
        applicable = [learned_thresholds[i] for i, label in enumerate(PRIMARY_ENDPOINT_LABELS)
                      if image_id in per_class_candidates[label]]
        threshold = min(applicable) if applicable else float(np.median(learned_thresholds))
        if float(row.learned_ranking_score) >= threshold:
            selected.add(image_id)
        else:
            rejected.append({"image_id": image_id, "reason": "below_learned_class_threshold",
                             "ranking_score": float(row.learned_ranking_score), "threshold": float(threshold)})
    if not selected:
        # Matches 09_finalize_learned_selection.py's identical guard for the adaptive policy. A
        # silently-empty selected_manifest.jsonl is a valid-looking file that downstream code
        # (Stage 4 condition builders) would only catch later via an unrelated "no synthetic images
        # found" error, far from this actual cause — fail here, at the source, instead.
        raise SystemExit(
            "FINALIZATION GATE: fixed-ratio learned threshold selected ZERO images. This is a real "
            "failure, not empty-by-design: check learned_thresholds against learned_ranking_score's "
            "actual distribution (a degenerate ranker or an unreasonably strict "
            "threshold_network.budget_weight/min_selected_per_label are the usual causes)."
        )
    with open(Path(cfg.paths.learned_selected_manifest), "x", encoding="utf-8") as handle:
        for image_id in sorted(selected):
            score = float(merged.loc[merged.image_id.astype(str) == image_id, "learned_ranking_score"].iloc[0])
            handle.write(json.dumps({"image_id": image_id, "ranking_score": score, "selector": "learned_asism_v1"}) + "\n")
    with open(Path(cfg.paths.learned_rejected_log), "x", encoding="utf-8") as handle:
        for row in rejected: handle.write(json.dumps(row) + "\n")
    torch.save({"state_dict": model.state_dict(), "context_dim": x.shape[1]}, learned_dir / "threshold_model.pt")
    write_frozen_json(learned_dir / "learned_selection_manifest.json", {
        "schema_version": 1, "method": "fixed_target_ratio_threshold_distillation_baseline_v1", "frozen": True,
        "thresholds": dict(zip(PRIMARY_ENDPOINT_LABELS, map(float, learned_thresholds))),
        "threshold_search_targets": dict(zip(PRIMARY_ENDPOINT_LABELS, map(float, targets))),
        "selected_count": len(selected), "ranking_checkpoint_sha256": sha256_file(learned_dir / "ranking_model.pt")})
    print(f"Learned ASISM selected {len(selected)}/{len(merged)} images -> {cfg.paths.learned_selected_manifest}")
    return 0


if __name__ == "__main__": raise SystemExit(main())


## `scripts/asism/07_build_threshold_contexts.py`

*263 lines*

**Learned ASISM step 5 - build the contexts the threshold network will be conditioned on. Trains
nothing.**

It loads the **frozen** ranker and critic from step 05 as fixed, read-only models
(`load_frozen_models`), then for each of the 11 primary labels:

- builds bootstrap resample contexts from the train-role and val-role image pools **separately**
  (the same disjoint split as before), each a 10-dim vector of score-distribution statistics + real
  patient prevalence + budget, honestly tagged `independent_clinical_sample: False`;
- runs a critic-guided `hard_threshold_grid_search` over candidate thresholds;
- writes `threshold_contexts.jsonl` (one row per context),
  `threshold_candidate_evaluations.jsonl` (one row per threshold tried), and
  `proxy_verification_plan.json` - which candidates *would* be proxy-verified, plus the GPU estimate
  (`verification_compute_estimate`).

The plan diversifies candidates across the grid rather than taking only the critic's top-k, so
verification cannot be rigged by a critic that is systematically wrong in one direction.

In [ ]:
# ===== FILE: scripts/asism/07_build_threshold_contexts.py =====
#!/usr/bin/env python3
"""Build threshold contexts + hard-grid-search candidate thresholds for learned ASISM (Phase 2).

Uses the FROZEN ranker + critic from 05_train_learned_asism.py as fixed, read-only models: this
script never trains anything. For each of the 11 primary labels, builds bootstrap resample contexts
from the train-role and val-role image pools SEPARATELY (same disjoint split as Phase 1b), runs a
hard threshold grid search scored by the critic, and writes:

  threshold_contexts.jsonl              — one row per context (see bootstrap_class_contexts)
  threshold_candidate_evaluations.jsonl — one row per threshold tried within a context
  proxy_verification_plan.json          — which top-K candidates WOULD be proxy-verified, and the
                                          GPU-hour cost of doing so. Nothing here is executed;
                                          07b_verify_thresholds_proxy.py --phase run does that,
                                          separately, and refuses to run without an explicit flag.

critic_predicted_utility is exactly what it says: the frozen critic's prediction, not a measured
value. Nothing here is presented as ground truth.
"""
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import torch
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    apply_feature_frame, bootstrap_class_contexts, diversify_verification_candidates,
    hard_threshold_grid_search, real_class_support_context, split_image_pool,
)
from scripts.asism.candidate_pool import load_candidate_pool  # noqa: E402,F401  (used by 07b/08/08b/09 via this module)
from scripts.asism.models import MultiSignalUtilityRankingNetwork, SetUtilityNetwork  # noqa: E402
from scripts.utils.artifact_contracts import stage3_paths  # noqa: E402
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, sha256_file  # noqa: E402


def preflight_check_phase1_outputs(cfg) -> None:
    """Phase 2 depends on Phase 1's frozen ranker/critic. Fail with a clear message, not a raw
    FileNotFoundError, if they don't exist yet."""
    learned_dir = Path(cfg.paths.learned_dir)
    required = ["set_utility_model.pt", "ranking_model.pt", "learned_training_manifest.json"]
    missing = [name for name in required if not (learned_dir / name).is_file()]
    if missing:
        raise SystemExit(
            "UPSTREAM GATE: threshold-context building needs Phase 1's trained models, missing:\n"
            + "\n".join(f"  - {learned_dir / name}" for name in missing)
            + "\n\n-> python scripts/asism/05_train_learned_asism.py"
        )


def load_frozen_models(cfg, columns: list[str]):
    learned_dir = Path(cfg.paths.learned_dir)
    training_manifest = read_json(learned_dir / "learned_training_manifest.json")

    set_cfg = cfg.learned_asism.set_utility_network
    critic_checkpoint = torch.load(learned_dir / "set_utility_model.pt", map_location="cpu", weights_only=True)
    critic = SetUtilityNetwork(
        len(columns), tuple(set_cfg.image_hidden_dims), tuple(set_cfg.utility_hidden_dims),
        superset_conditioning=bool(critic_checkpoint.get("superset_conditioning", False)),
    )
    critic.load_state_dict(critic_checkpoint["state_dict"])
    critic.eval()
    for parameter in critic.parameters():
        parameter.requires_grad_(False)

    rank_cfg = cfg.learned_asism.ranking_network
    ranker = MultiSignalUtilityRankingNetwork(len(columns), tuple(rank_cfg.hidden_dims), float(rank_cfg.dropout))
    ranker_checkpoint = torch.load(learned_dir / "ranking_model.pt", map_location="cpu", weights_only=True)
    ranker.load_state_dict(ranker_checkpoint["state_dict"])
    ranker.eval()

    return critic, ranker, training_manifest


def make_critic_utility_fn(critic, feature_lookup: dict, device):
    @torch.no_grad()
    def critic_utility_fn(image_ids: list[str]) -> float:
        vectors = [feature_lookup[image_id] for image_id in image_ids if image_id in feature_lookup]
        if not vectors:
            return float("-inf")
        features = torch.tensor(np.asarray(vectors), dtype=torch.float32, device=device).unsqueeze(0)
        mask = torch.ones(1, len(vectors), dtype=torch.bool, device=device)
        return float(critic(features, mask).item())
    return critic_utility_fn


def verification_compute_estimate(cfg, n_planned: int) -> dict:
    """Per-context threshold verification ONLY. Full-policy verification (after per-class thresholds
    are combined across all 11 diseases) has its OWN independent budget and its own fail-closed
    execution gate — see 08_train_threshold_network.py's plan_full_policy_verification and
    08b_verify_full_policy_proxy.py. The two are never summed into one number: they are separately
    approved, separately budgeted production steps."""
    budget = cfg.learned_asism.threshold_network.verification_compute_budget
    hours_each = float(budget.hours_per_proxy_run_estimate)
    estimated_hours = n_planned * hours_each
    max_hours = float(budget.max_gpu_hours)
    return {
        "n_planned_threshold_evaluations": n_planned,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths:
            cfg.paths[key] = str(value)
    cfg.split_namespace = namespace

    preflight_check_phase1_outputs(cfg)

    baseline_path = Path(__file__).with_name("04_build_utility_subsets.py")
    spec = importlib.util.spec_from_file_location("build_utility_subsets_04", baseline_path)
    baseline = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(baseline)
    baseline.preflight_check_candidate_pool_inputs(cfg)
    merged, intended, _ = baseline.load_candidate_pool(cfg)

    tn_cfg = cfg.learned_asism.threshold_network
    columns = list(read_json(Path(cfg.paths.learned_dir) / "learned_training_manifest.json")["feature_columns"])
    critic, ranker, training_manifest = load_frozen_models(cfg, columns)
    ranking_checkpoint_hash = sha256_file(Path(cfg.paths.learned_dir) / "ranking_model.pt")
    critic_checkpoint_hash = sha256_file(Path(cfg.paths.learned_dir) / "set_utility_model.pt")

    normalized = apply_feature_frame(merged, columns, training_manifest["normalization"])
    device = torch.device("cpu")
    feature_lookup = {
        str(image_id): row.to_numpy(np.float32)
        for image_id, (_, row) in zip(merged.image_id, normalized.iterrows())
    }
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32)))
    merged = merged.copy()
    merged["learned_ranking_score"] = (1.0 / (1.0 + np.exp(-raw.numpy())))
    ranking_scores = dict(zip(merged.image_id.astype(str), merged["learned_ranking_score"]))
    critic_utility_fn = make_critic_utility_fn(critic, feature_lookup, device)

    val_fraction = float(cfg.learned_asism.subset_design.val_pool_fraction)
    seed = int(cfg.learned_asism.subset_design.seed)
    train_pool, val_pool = split_image_pool(merged, val_fraction, seed)

    budget_context = {
        "target_synthetic_to_real_ratio": float(tn_cfg.target_synthetic_to_real_ratio),
        "min_selected_per_label": int(tn_cfg.min_selected_per_label),
        "max_selected_per_label": int(tn_cfg.max_selected_per_label),
    }
    t_grid = np.arange(0.0, 1.0 + 1e-9, float(tn_cfg.t_grid_step))
    top_k = int(tn_cfg.top_k_per_context)
    quantile_fractions = [float(fraction) for fraction in tn_cfg.verification_quantile_fractions]

    all_contexts: list[dict] = []
    all_evaluations: list[dict] = []
    planned_evaluations: list[dict] = []

    for class_id, label in enumerate(PRIMARY_ENDPOINT_LABELS):
        real_prevalence = real_class_support_context(namespace, [label])["labels"][label]
        for pool, source in ((train_pool, "bootstrap_train_pool"), (val_pool, "bootstrap_val_pool")):
            contexts = bootstrap_class_contexts(
                pool, class_id, label, intended, "learned_ranking_score", source,
                int(tn_cfg.n_bootstrap_contexts_per_class), float(tn_cfg.bootstrap_min_fraction),
                float(tn_cfg.bootstrap_max_fraction), ranking_checkpoint_hash, critic_checkpoint_hash,
                real_prevalence, budget_context, seed + class_id,
            )
            all_contexts.extend(contexts)
            for context in contexts:
                try:
                    evaluations = hard_threshold_grid_search(
                        critic_utility_fn, ranking_scores, context["image_ids"], t_grid,
                        float(tn_cfg.budget_weight), int(tn_cfg.max_selected_per_label),
                        min_selected=int(tn_cfg.min_selected_per_label),
                    )
                except ValueError:
                    continue  # every threshold produced an empty subset for this context; skip it
                for evaluation in evaluations:
                    evaluation["context_id"] = context["context_id"]

                diversified = diversify_verification_candidates(
                    evaluations, top_k, quantile_fractions, len(context["image_ids"]),
                )
                planned_by_threshold = {entry["candidate_threshold"]: entry for entry in diversified}

                for evaluation in evaluations:
                    planned_entry = planned_by_threshold.get(evaluation["candidate_threshold"])
                    evaluation["proxy_verification_status"] = "planned" if planned_entry else "not_planned"
                    evaluation["selection_reasons"] = planned_entry["selection_reasons"] if planned_entry else []
                    all_evaluations.append(evaluation)
                    if planned_entry:
                        # Same selection rule as hard_threshold_grid_search's own truncation, so
                        # 07b can deterministically re-derive the identical hard subset later.
                        candidates = [image_id for image_id in context["image_ids"]
                                     if ranking_scores.get(image_id, 0.0) >= evaluation["candidate_threshold"]]
                        subset_image_ids = sorted(candidates, key=lambda i: -ranking_scores.get(i, 0.0))[
                            :evaluation["selected_count"]]
                        planned_evaluations.append({
                            "context_id": context["context_id"],
                            "candidate_threshold": evaluation["candidate_threshold"],
                            "selection_reasons": evaluation["selection_reasons"],
                            "hard_subset_image_ids_hash": hash_dict({"image_ids": sorted(subset_image_ids)}, length=32),
                        })

    output_dir = Path(cfg.paths.learned_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Exclusive-create ("x"): these are FROZEN, critic-only artifacts. 07b measures proxy results
    # into a completely separate file (threshold_proxy_measurements.jsonl) and must never be able to
    # append to or rewrite this one.
    with open(Path(cfg.paths.threshold_contexts), "x", encoding="utf-8") as handle:
        for context in all_contexts:
            handle.write(json.dumps(context, sort_keys=True) + "\n")
    with open(Path(cfg.paths.threshold_candidate_evaluations), "x", encoding="utf-8") as handle:
        for evaluation in all_evaluations:
            handle.write(json.dumps(evaluation, sort_keys=True) + "\n")

    estimate = verification_compute_estimate(cfg, len(planned_evaluations))
    reason_counts: dict[str, int] = {}
    for entry in planned_evaluations:
        for reason in entry["selection_reasons"]:
            reason_counts[reason] = reason_counts.get(reason, 0) + 1
    plan = {
        "schema_version": 1,
        "frozen": True,
        "top_k_per_context": top_k,
        "verification_quantile_fractions": quantile_fractions,
        "n_contexts": len(all_contexts),
        "n_planned_by_selection_reason": reason_counts,
        "planned_evaluations": [
            {"context_id": e["context_id"], "candidate_threshold": e["candidate_threshold"],
             "selection_reasons": e["selection_reasons"]}
            for e in planned_evaluations
        ],
        **estimate,
        "git_commit_hash": get_git_commit_hash(),
    }
    with open(Path(cfg.paths.proxy_verification_plan), "x", encoding="utf-8") as handle:
        json.dump(plan, handle, indent=2, sort_keys=True)

    print(f"{len(all_contexts)} contexts, {len(all_evaluations)} threshold evaluations, "
          f"{len(planned_evaluations)} planned for proxy verification (by reason: {reason_counts})")
    print(f"Threshold-verification budget: {estimate['n_planned_threshold_evaluations']} runs "
          f"({estimate['estimated_gpu_hours']}h, within_budget={estimate['within_budget']}). "
          "Full-policy verification is a separate, independently-budgeted step (see 08's plan).")
    print(f"-> {cfg.paths.threshold_contexts}\n-> {cfg.paths.threshold_candidate_evaluations}\n"
          f"-> {cfg.paths.proxy_verification_plan}")
    print("Nothing was executed. Next (on RunPod, after budget sign-off): "
          "07b_verify_thresholds_proxy.py --phase run --i-understand-this-trains-real-models")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/07b_verify_thresholds_proxy.py`

*206 lines*

**Learned ASISM step 6 - the only script in the threshold pipeline that trains real proxy
classifiers.**

`--phase run` **refuses to start** - before touching any config, data or model - unless called with
`--i-understand-this-trains-real-models`. That flag exists specifically so that no test, CI job or
accidental invocation can trigger real GPU training. `--phase estimate` never requires it, because
it never trains.

For each planned `(context_id, candidate_threshold)`: `rebuild_hard_subset` reconstructs the
*identical* subset step 07's grid search scored (same rule, no drift), then `train_and_score` trains
real + subset against a real-only baseline and appends the measured utility. Resumable; evaluated on
`asism_tuning_heldout` only; gated by its own independent compute budget.

In [ ]:
# ===== FILE: scripts/asism/07b_verify_thresholds_proxy.py =====
#!/usr/bin/env python3
"""Execute the proxy_verification_plan.json produced by 07_build_threshold_contexts.py.

This is the ONLY script in the threshold-learning pipeline that trains real proxy classifiers.
--phase run REFUSES to start (before touching any config, data, or model) unless called with
--i-understand-this-trains-real-models. That flag exists specifically so nothing — a test, a CI
job, an accidental invocation — can trigger real GPU training by mistake. --phase estimate never
requires it, because it never trains anything.

For each planned (context_id, candidate_threshold): rebuilds the identical hard subset (same rule
as 07's grid search), trains real + hard-subset vs a real-only baseline, and appends the MEASURED
utility to threshold_proxy_measurements.jsonl — a SEPARATE file from
threshold_candidate_evaluations.jsonl. This script never opens threshold_candidate_evaluations.jsonl
for writing: that file is 07's frozen, critic-only artifact, and mixing measured results into it is
exactly the confusion this split exists to prevent.
"""
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import read_jsonl  # noqa: E402
from scripts.utils.artifact_contracts import stage2_paths, stage3_paths  # noqa: E402
from scripts.utils.classifier import (  # noqa: E402
    TrainingBudget, evaluate_classifier, records_from_split, records_from_synthetic_manifest, train_classifier,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402


def compute_budget_estimate(cfg, n_planned: int) -> dict:
    budget = cfg.learned_asism.threshold_network.verification_compute_budget
    hours_each = float(budget.hours_per_proxy_run_estimate)
    total_runs = 1 + n_planned  # 1 shared real-only baseline + 1 run per planned evaluation
    estimated_hours = total_runs * hours_each
    max_hours = float(budget.max_gpu_hours)
    return {
        "total_proxy_runs": total_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
    }


def enforce_compute_budget(cfg, n_planned: int) -> dict:
    estimate = compute_budget_estimate(cfg, n_planned)
    if not estimate["within_budget"]:
        raise SystemExit(
            "COMPUTE BUDGET GATE: threshold proxy verification exceeds its declared budget.\n"
            f"  estimated: {estimate['estimated_gpu_hours']}h ({estimate['total_proxy_runs']} runs "
            f"x {estimate['hours_per_proxy_run_estimate']}h), budget: {estimate['max_gpu_hours']}h\n"
            "Reduce learned_asism.threshold_network.top_k_per_context or "
            "n_bootstrap_contexts_per_class and re-run 07_build_threshold_contexts.py."
        )
    return estimate


def rebuild_hard_subset(context: dict, threshold: float, ranking_scores: dict[str, float], selected_count: int) -> list[str]:
    """Deterministically reproduces the exact hard subset 07's grid search selected for this
    (context, threshold) pair — same rule: score >= threshold, truncated to the top `selected_count`
    by ranking_score."""
    candidates = [image_id for image_id in context["image_ids"] if ranking_scores.get(image_id, 0.0) >= threshold]
    return sorted(candidates, key=lambda i: -ranking_scores.get(i, 0.0))[:selected_count]


def train_and_score(train_records, eval_records, cfg, seed, checkpoint, tag):
    budget = TrainingBudget(max_steps=int(cfg.tuning.coarse.proxy_max_steps),
                            batch_size=int(cfg.tuning.proxy.batch_size),
                            learning_rate=float(cfg.tuning.proxy.learning_rate), seed=seed,
                            eval_every_n_steps=0)
    model, _, _ = train_classifier(
        train_records=train_records, val_records=[], budget=budget,
        dropout_p=float(cfg.tuning.proxy.dropout_p), pretrained_source=str(cfg.tuning.proxy.pretrained_source),
        resolution=int(cfg.tuning.proxy.resolution), progress_desc=tag,
        checkpoint_path=checkpoint, resume_from=checkpoint if checkpoint.is_file() else None,
        checkpoint_metadata={"dataset_id": tag, "condition_or_draw_id": tag,
                             "config_hash": hash_dict({"tag": tag, "seed": seed}, length=64),
                             "label_policy": "chexpert_mask_uncertain_v1"})
    return float(evaluate_classifier(model, eval_records, int(cfg.tuning.proxy.resolution))["macro_auroc"])


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--phase", choices=["estimate", "run"], default="estimate")
    parser.add_argument(
        "--i-understand-this-trains-real-models", action="store_true",
        help="Required for --phase run. Confirms real GPU proxy-classifier training is about to start.",
    )
    args = parser.parse_args()

    if args.phase == "run" and not args.i_understand_this_trains_real_models:
        raise SystemExit(
            "REFUSED: --phase run launches real proxy-classifier training (GPU/RunPod cost).\n"
            "Pass --i-understand-this-trains-real-models to confirm. This guard runs before any "
            "config, data, or model is touched — nothing about this command can accidentally train."
        )

    cfg = load_named_config("stage3_asism.yaml", "stage3")
    stage2 = load_named_config("stage2_generation.yaml", "stage2")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    cfg.split_namespace = namespace
    for key, value in stage2_paths(stage2, namespace).items():
        if key in stage2.paths: stage2.paths[key] = str(value)

    plan_path = Path(cfg.paths.proxy_verification_plan)
    if not plan_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: no proxy verification plan at {plan_path}.\n"
            "Run: python scripts/asism/07_build_threshold_contexts.py"
        )
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    n_planned = len(plan["planned_evaluations"])

    if args.phase == "estimate":
        estimate = compute_budget_estimate(cfg, n_planned)
        print(json.dumps(estimate, indent=2))
        return 0 if estimate["within_budget"] else 2

    enforce_compute_budget(cfg, n_planned)

    contexts = {row["context_id"]: row for row in read_jsonl(Path(cfg.paths.threshold_contexts))}
    evaluations = read_jsonl(Path(cfg.paths.threshold_candidate_evaluations))  # read-only reference
    evaluations_by_key = {(row["context_id"], row["candidate_threshold"]): row for row in evaluations}
    measurements_path = Path(cfg.paths.threshold_proxy_measurements)
    already_measured = {
        (row["context_id"], row["candidate_threshold"])
        for row in (read_jsonl(measurements_path) if measurements_path.is_file() else [])
    }

    baseline_module_path = Path(__file__).with_name("07_build_threshold_contexts.py")
    spec = importlib.util.spec_from_file_location("build_threshold_contexts_07", baseline_module_path)
    context_builder = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(context_builder)
    context_builder.preflight_check_phase1_outputs(cfg)
    merged, _, _ = context_builder.load_candidate_pool(cfg)
    columns = list(read_json(Path(cfg.paths.learned_dir) / "learned_training_manifest.json")["feature_columns"])
    _, ranker, training_manifest = context_builder.load_frozen_models(cfg, columns)
    normalized = context_builder.apply_feature_frame(merged, columns, training_manifest["normalization"])
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32)))
    ranking_scores: dict[str, float] = dict(
        zip(merged.image_id.astype(str), (1.0 / (1.0 + np.exp(-raw.numpy())))))

    stage1 = load_stage1_config()
    image_root = Path(stage1.paths.images_dir) / namespace
    real = records_from_split(
        load_split("classifier_train", namespace, purpose="schema_validation", caller="threshold_proxy_verification"),
        image_root / "classifier_train",
    )
    evaluation_records = records_from_split(
        load_split("asism_tuning_heldout", namespace, purpose="schema_validation", caller="threshold_proxy_verification"),
        image_root / "asism_tuning_heldout",
    )
    checkpoint_dir = Path(cfg.paths.learned_dir) / "threshold_proxy_checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    seed = int(cfg.learned_asism.subset_design.seed)
    baseline_checkpoint = checkpoint_dir / f"real_only_seed{seed}.pt"
    baseline_score = train_and_score(real, evaluation_records, cfg, seed, baseline_checkpoint, f"threshold-real-only-s{seed}")

    stage2_manifest = Path(stage2.paths.manifest_path)
    stage2_images = Path(stage2.paths.images_dir)

    with open(measurements_path, "a", encoding="utf-8") as handle:
        for index, planned in enumerate(plan["planned_evaluations"]):
            key = (planned["context_id"], planned["candidate_threshold"])
            if key in already_measured:
                continue
            context = contexts[planned["context_id"]]
            evaluation = evaluations_by_key.get(key)
            if evaluation is None:
                continue
            subset_ids = rebuild_hard_subset(context, planned["candidate_threshold"], ranking_scores,
                                             evaluation["selected_count"])
            synthetic = records_from_synthetic_manifest(stage2_manifest, stage2_images, subset_ids)
            tag = f"threshold-{planned['context_id']}-t{planned['candidate_threshold']:.2f}-s{seed}"
            checkpoint = checkpoint_dir / f"{hash_dict({'tag': tag}, length=32)}.pt"
            measured = train_and_score(real + synthetic, evaluation_records, cfg, seed, checkpoint, tag)
            row = {
                "context_id": planned["context_id"],
                "candidate_threshold": planned["candidate_threshold"],
                "selection_reasons": planned.get("selection_reasons", []),
                "selected_count": evaluation["selected_count"],
                "real_only_macro_auroc": baseline_score,
                "augmented_macro_auroc": measured,
                "measured_utility": measured - baseline_score,
                "seed": seed,
            }
            handle.write(json.dumps(row, sort_keys=True) + "\n")
            print(f"[{index + 1}/{n_planned}] {tag}: measured_utility={row['measured_utility']:+.6f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/08_train_threshold_network.py`

*369 lines*

**Learned ASISM step 7 - train the Adaptive Threshold Network, with the project's strictest
governance. Two separate models that are never merged:**

- **`verified_only_official`** - trained only if at least one class is *eligible*: it independently
  clears `min_verified_contexts_per_class` on **both** verified train contexts and verified,
  **image-disjoint** held-out contexts. A class with zero verified train contexts never receives an
  official threshold from this path. Held-out measurements are used **only to evaluate** a threshold
  already built from train data - never to choose or construct one.
- **`critic_assisted_exploratory`** - a separate model on critic-only targets, always trained, always
  labelled exploratory.

Then the **three-tier per-class decision** (`determine_per_class_official_method`): sufficient
verified evidence *and* passing acceptance criteria -> the learned network; 1-2 verified contexts ->
the median best proxy-verified threshold; 0 -> the fixed baseline.

`evaluate_held_out`, `critic_predicted_utility_regret`, `threshold_stability` and
`compute_official_acceptance_evidence` produce the evidence; `enforce_acceptance_criteria` checks it
against criteria frozen *beforehand*. Finally `plan_full_policy_verification` writes the plan for
step 08b.

In [ ]:
# ===== FILE: scripts/asism/08_train_threshold_network.py =====
#!/usr/bin/env python3
"""Train AdaptiveThresholdNetwork(s) for learned ASISM (Phase 2) — two SEPARATE, never-merged paths.

verified_only_official: trained ONLY if at least one class is ELIGIBLE — meaning it independently
    clears min_verified_contexts_per_class on BOTH verified TRAIN contexts and verified,
    image-disjoint HELD-OUT contexts (eligible_classes_for_official_training). A class with zero
    verified TRAIN contexts never gets an official threshold from this path or from
    hard_proxy_best_among_verified — held-out measurements are NEVER used to decide or construct a
    threshold, only to evaluate one already built from train data.
critic_assisted_exploratory: a SEPARATE model trained on critic-only targets, always run (contexts
    always exist), written to its own checkpoint and its own target file. Never merged into, or
    substituted for, the official path's output.

Given this pass has 07b never executed (no GPU training), the expected, CORRECT outcome is that
every class falls back to fixed_target_ratio_threshold_distillation_baseline_v1: zero proxy-verified
targets exist anywhere, so official_trained is False and eligible_classes is empty. That is not a
bug in this script — it is what "no production evidence yet" is supposed to look like.

Full-policy verification (after per-class thresholds are combined across all 11 diseases) is
PLANNED here (plan_full_policy_verification) but never executed — see
08b_verify_full_policy_proxy.py, which has its own independent compute budget and the same
fail-closed --i-understand-this-trains-real-models guard as 07b.
"""
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    aggregate_hard_proxy_best_threshold, compute_verified_context_counts, critic_proxy_correlation_per_class,
    determine_per_class_official_method, eligible_classes_for_official_training, enforce_acceptance_criteria,
    filter_targets_to_eligible_classes,
    freeze_acceptance_criteria, held_out_generalization_metrics, read_jsonl,
    resolve_critic_assisted_exploratory_targets, resolve_verified_only_targets,
)
from scripts.asism.models import AdaptiveThresholdNetwork  # noqa: E402
from scripts.utils.artifact_contracts import stage3_paths  # noqa: E402
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import read_json, sha256_file, write_frozen_json  # noqa: E402
from scripts.utils.seed import set_seed  # noqa: E402


def train_threshold_network(model, targets: list[dict], device, epochs: int, learning_rate: float):
    """Full-batch weighted-MSE distillation. weight_decay=0.0 explicitly: AdamW's default (0.01)
    would move parameters every step even with a zero-weight/zero-gradient loss."""
    if not targets:
        raise ValueError("train_threshold_network: no training targets")
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.0)
    class_ids = torch.tensor([t["class_id"] for t in targets], dtype=torch.long, device=device)
    context = torch.tensor([t["context_features"] for t in targets], dtype=torch.float32, device=device)
    target_values = torch.tensor([t["target_threshold"] for t in targets], dtype=torch.float32, device=device)
    weights = torch.tensor([t["supervision_weight"] for t in targets], dtype=torch.float32, device=device)
    model.train()
    for _ in range(epochs):
        prediction = model(class_ids, context)
        loss = (weights * (prediction - target_values) ** 2).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return model


@torch.no_grad()
def predict_thresholds(model, targets: list[dict], device) -> list[float]:
    if not targets:
        return []
    model.eval()
    class_ids = torch.tensor([t["class_id"] for t in targets], dtype=torch.long, device=device)
    context = torch.tensor([t["context_features"] for t in targets], dtype=torch.float32, device=device)
    return model(class_ids, context).cpu().numpy().tolist()


def critic_predicted_utility_regret(critic_utility_fn, ranking_scores: dict, context: dict,
                                    grid_best_utility: float, predicted_threshold: float,
                                    max_selected: int) -> float:
    candidates = [i for i in context["image_ids"] if ranking_scores.get(i, 0.0) >= predicted_threshold]
    if not candidates:
        return float("inf")
    if len(candidates) > max_selected:
        candidates = sorted(candidates, key=lambda i: -ranking_scores.get(i, 0.0))[:max_selected]
    return grid_best_utility - float(critic_utility_fn(candidates))


def threshold_stability(targets: list[dict], predictions: list[float]) -> dict:
    by_class: dict[int, list[float]] = {}
    for target, prediction in zip(targets, predictions):
        by_class.setdefault(target["class_id"], []).append(prediction)
    stds = {class_id: float(np.std(values)) for class_id, values in by_class.items() if len(values) >= 2}
    return {"per_class_std": stds, "median_std": float(np.median(list(stds.values()))) if stds else None}


def evaluate_held_out(model, held_out_targets, contexts_by_id, ranking_scores, evaluations_by_context,
                      critic_utility_fn, max_selected, device):
    predictions = predict_thresholds(model, held_out_targets, device)
    results = []
    for target, predicted in zip(held_out_targets, predictions):
        context = contexts_by_id[target["context_id"]]
        scores = {image_id: ranking_scores.get(image_id, 0.0) for image_id in context["image_ids"]}
        metrics = held_out_generalization_metrics(float(predicted), target["target_threshold"], scores)
        grid_best = min(evaluations_by_context.get(target["context_id"], [{"critic_predicted_utility": 0.0}]),
                        key=lambda row: row.get("rank_within_context", 0))
        metrics["critic_predicted_utility_regret"] = critic_predicted_utility_regret(
            critic_utility_fn, ranking_scores, context, grid_best["critic_predicted_utility"],
            float(predicted), max_selected,
        )
        results.append({**metrics, "context_id": target["context_id"], "label": target["label"]})
    return results, predictions


def compute_official_acceptance_evidence(held_out_generalization: list[dict], correlation: dict, stability: dict) -> dict:
    """Acceptance evidence using explicitly critic-predicted (not proxy-measured) utility regret
    by the network's own threshold vs the grid optimum), NEVER from absolute_threshold_error (a
    probability-space distance between two thresholds, not a utility measure) — conflating the two
    was a real bug in an earlier version of this script."""
    finite_regrets = [abs(entry["critic_predicted_utility_regret"]) for entry in held_out_generalization
                      if np.isfinite(entry["critic_predicted_utility_regret"])]
    return {
        "median_spearman": correlation["median_spearman"],
        "median_critic_predicted_utility_regret": float(np.median(finite_regrets)) if finite_regrets else None,
        "threshold_stability_std": stability["median_std"],
    }


def save_checkpoint(model, path: Path, path_label: str, tn_cfg, context_dim: int) -> str:
    torch.save({
        "state_dict": model.state_dict(),
        "schema_version": 1,
        "path": path_label,
        "number_of_classes": len(PRIMARY_ENDPOINT_LABELS),
        "context_dim": context_dim,
        "embedding_dim": int(tn_cfg.class_embedding_dim),
        "hidden_dims": list(tn_cfg.hidden_dims),
        "dropout": float(tn_cfg.dropout),
    }, path)
    return sha256_file(path)


def plan_full_policy_verification(per_class_official_method: dict[str, str], full_policy_cfg) -> dict:
    """PLAN ONLY — no execution. Which full-policy variants would need proxy verification once
    thresholds are combined across all 11 diseases, and the independent GPU-hour cost of doing so.
    Actual execution is a separate, fail-closed step: 08b_verify_full_policy_proxy.py --phase run."""
    variants = sorted(
        {
            "fixed_target_ratio_threshold_distillation_baseline_v1",
            "literal_top_50_percent",
            "freematch_style_adaptive_percentile",
        }
        | set(per_class_official_method.values())
    )
    n_seeds = int(full_policy_cfg.n_seeds)
    n_runs = len(variants) * n_seeds
    budget = full_policy_cfg.compute_budget
    hours_each = float(budget.hours_per_proxy_run_estimate)
    estimated_hours = n_runs * hours_each
    max_hours = float(budget.max_gpu_hours)
    return {
        "status": "plan_only_not_executed",
        "policy_variants": variants,
        "n_seeds": n_seeds,
        "estimated_runs": n_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
        "next_step": (
            "python scripts/asism/08b_verify_full_policy_proxy.py --phase run "
            "--i-understand-this-trains-real-models (only after independent budget sign-off)"
        ),
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    cfg.split_namespace = namespace
    # BUG FIXED 2026-08-21: this script never seeded torch's global RNG at all — see the identical
    # fix and its rationale in 05_train_learned_asism.py.
    set_seed(int(cfg.learned_asism.subset_design.seed))

    contexts_path = Path(cfg.paths.threshold_contexts)
    evaluations_path = Path(cfg.paths.threshold_candidate_evaluations)
    if not contexts_path.is_file() or not evaluations_path.is_file():
        raise SystemExit(
            "UPSTREAM GATE: threshold contexts/evaluations missing.\n"
            "Run: python scripts/asism/07_build_threshold_contexts.py"
        )
    contexts = read_jsonl(contexts_path)
    evaluations = read_jsonl(evaluations_path)
    measurements_path = Path(cfg.paths.threshold_proxy_measurements)
    measurements = read_jsonl(measurements_path) if measurements_path.is_file() else []
    tn_cfg = cfg.learned_asism.threshold_network
    labels = list(PRIMARY_ENDPOINT_LABELS)

    context_builder_path = Path(__file__).with_name("07_build_threshold_contexts.py")
    spec = importlib.util.spec_from_file_location("build_threshold_contexts_07", context_builder_path)
    context_builder = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(context_builder)
    context_builder.preflight_check_phase1_outputs(cfg)
    merged, _, _ = context_builder.load_candidate_pool(cfg)
    columns = list(read_json(Path(cfg.paths.learned_dir) / "learned_training_manifest.json")["feature_columns"])
    critic, ranker, training_manifest = context_builder.load_frozen_models(cfg, columns)
    normalized = context_builder.apply_feature_frame(merged, columns, training_manifest["normalization"])
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32)))
    ranking_scores: dict[str, float] = dict(
        zip(merged.image_id.astype(str), (1.0 / (1.0 + np.exp(-raw.numpy())))))
    feature_lookup = {
        str(image_id): row.to_numpy(np.float32)
        for image_id, (_, row) in zip(merged.image_id, normalized.iterrows())
    }
    critic_utility_fn = context_builder.make_critic_utility_fn(critic, feature_lookup, torch.device("cpu"))

    features_by_context = {c["context_id"]: c["context_features"] for c in contexts}
    contexts_by_id = {c["context_id"]: c for c in contexts}
    evaluations_by_context: dict[str, list[dict]] = {}
    for row in evaluations:
        evaluations_by_context.setdefault(row["context_id"], []).append(row)
    device = torch.device("cpu")
    min_verified = int(tn_cfg.min_verified_contexts_per_class)

    # ---- Path A: verified_only_official ------------------------------------------------------
    verified_targets = resolve_verified_only_targets(contexts, measurements)
    for target in verified_targets:
        target["context_features"] = features_by_context[target["context_id"]]
    verified_train = [t for t in verified_targets if t["context_source"] == "bootstrap_train_pool"]
    verified_held_out = [t for t in verified_targets if t["context_source"] == "bootstrap_val_pool"]

    train_counts, held_out_counts = compute_verified_context_counts(verified_train, verified_held_out, labels)
    eligible_classes = eligible_classes_for_official_training(train_counts, held_out_counts, min_verified)
    official_train = filter_targets_to_eligible_classes(verified_train, eligible_classes)
    official_held_out = filter_targets_to_eligible_classes(verified_held_out, eligible_classes)
    # Training requires (a) at least one class independently clearing the minimum on BOTH train and
    # held-out sides, and (b) actual verified train data to fit on. Neither alone is enough.
    official_trained = bool(eligible_classes) and bool(official_train)

    official_held_out_generalization: list[dict] = []
    official_stability: dict = {"per_class_std": {}, "median_std": None}
    official_correlation = critic_proxy_correlation_per_class(evaluations, measurements, contexts)
    acceptance_gate: dict = {"passed": False, "criteria_status": "not_evaluated_no_eligible_class"}
    official_checkpoint_hash = None

    if official_trained:
        official_model = AdaptiveThresholdNetwork(
            len(labels), len(official_train[0]["context_features"]),
            int(tn_cfg.class_embedding_dim), tuple(tn_cfg.hidden_dims), float(tn_cfg.dropout),
        ).to(device)
        train_threshold_network(official_model, official_train, device, int(tn_cfg.epochs), float(tn_cfg.learning_rate))
        official_held_out_generalization, official_predictions = evaluate_held_out(
            official_model, official_held_out, contexts_by_id, ranking_scores, evaluations_by_context,
            critic_utility_fn, int(tn_cfg.max_selected_per_label), device,
        )
        official_stability = threshold_stability(official_held_out, official_predictions) if official_held_out else official_stability
        evidence = compute_official_acceptance_evidence(official_held_out_generalization, official_correlation, official_stability)
        frozen_criteria = freeze_acceptance_criteria(dict(tn_cfg.acceptance_criteria))
        acceptance_gate = enforce_acceptance_criteria(evidence, frozen_criteria)

        official_checkpoint_hash = save_checkpoint(
            official_model, Path(cfg.paths.official_threshold_checkpoint), "verified_only_official",
            tn_cfg, len(official_train[0]["context_features"]),
        )

    # ---- Path B: critic_assisted_exploratory (always runs, never official) ------------------
    exploratory_targets = resolve_critic_assisted_exploratory_targets(contexts, evaluations)
    for target in exploratory_targets:
        target["context_features"] = features_by_context[target["context_id"]]
    exploratory_train = [t for t in exploratory_targets if t["context_source"] == "bootstrap_train_pool"]
    exploratory_held_out = [t for t in exploratory_targets if t["context_source"] == "bootstrap_val_pool"]
    exploratory_held_out_generalization: list[dict] = []
    exploratory_checkpoint_hash = None
    if exploratory_train:
        exploratory_model = AdaptiveThresholdNetwork(
            len(labels), len(exploratory_train[0]["context_features"]),
            int(tn_cfg.class_embedding_dim), tuple(tn_cfg.hidden_dims), float(tn_cfg.dropout),
        ).to(device)
        train_threshold_network(exploratory_model, exploratory_train, device, int(tn_cfg.epochs), float(tn_cfg.learning_rate))
        exploratory_held_out_generalization, _ = evaluate_held_out(
            exploratory_model, exploratory_held_out, contexts_by_id, ranking_scores, evaluations_by_context,
            critic_utility_fn, int(tn_cfg.max_selected_per_label), device,
        )
        exploratory_checkpoint_hash = save_checkpoint(
            exploratory_model, Path(cfg.paths.exploratory_threshold_checkpoint), "critic_assisted_exploratory",
            tn_cfg, len(exploratory_train[0]["context_features"]),
        )

    # ---- Governance + hard-proxy-best aggregation (TRAIN contexts only, never held-out) -------
    per_class_official_method = determine_per_class_official_method(
        verified_train, verified_held_out, labels, min_verified, bool(acceptance_gate["passed"]),
    )
    hard_proxy_best_thresholds = {
        label: aggregate_hard_proxy_best_threshold(verified_train, label)
        for label, method in per_class_official_method.items()
        if method == "hard_proxy_best_among_verified"
    }
    full_policy_plan = plan_full_policy_verification(per_class_official_method, tn_cfg.full_policy_verification)

    # ---- Write SEPARATE verified/exploratory target files (never one shared file) -------------
    output_dir = Path(cfg.paths.learned_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with open(Path(cfg.paths.threshold_training_targets_verified), "x", encoding="utf-8") as handle:
        for target in verified_targets:
            handle.write(json.dumps(target, sort_keys=True) + "\n")
    with open(Path(cfg.paths.threshold_training_targets_exploratory), "x", encoding="utf-8") as handle:
        for target in exploratory_targets:
            handle.write(json.dumps(target, sort_keys=True) + "\n")

    manifest = {
        "schema_version": 3,
        "method": "verified_hard_class_aware_threshold_optimization_with_supervised_distillation",
        "frozen": True,
        "official_path": {
            "trained": official_trained,
            "eligible_classes": sorted(eligible_classes),
            "train_counts": train_counts,
            "held_out_counts": held_out_counts,
            "n_train_targets": len(verified_train),
            "n_held_out_targets": len(verified_held_out),
            "n_eligible_train_targets": len(official_train),
            "n_eligible_held_out_targets": len(official_held_out),
            "checkpoint_path": str(cfg.paths.official_threshold_checkpoint) if official_trained else None,
            "checkpoint_sha256": official_checkpoint_hash,
            "held_out_generalization": official_held_out_generalization,
            "threshold_stability": official_stability,
            "critic_proxy_correlation": official_correlation,
            "acceptance_gate": acceptance_gate,
        },
        "exploratory_path": {
            "trained": bool(exploratory_train),
            "note": "critic_assisted_exploratory — NEVER used as the official selector, ablation only",
            "n_train_targets": len(exploratory_train),
            "n_held_out_targets": len(exploratory_held_out),
            "checkpoint_path": str(cfg.paths.exploratory_threshold_checkpoint) if exploratory_train else None,
            "checkpoint_sha256": exploratory_checkpoint_hash,
            "held_out_generalization": exploratory_held_out_generalization,
        },
        "per_class_official_method": per_class_official_method,
        "hard_proxy_best_thresholds": hard_proxy_best_thresholds,
        "any_class_uses_network_officially": any(
            v == "adaptive_threshold_network" for v in per_class_official_method.values()
        ),
        "full_policy_verification_plan": full_policy_plan,
        "targets_verified_sha256": sha256_file(Path(cfg.paths.threshold_training_targets_verified)),
        "targets_exploratory_sha256": sha256_file(Path(cfg.paths.threshold_training_targets_exploratory)),
    }
    write_frozen_json(Path(cfg.paths.adaptive_threshold_manifest), manifest)
    with open(Path(cfg.paths.full_policy_verification_plan), "x", encoding="utf-8") as handle:
        json.dump(full_policy_plan, handle, indent=2, sort_keys=True)

    print(f"Official path trained: {official_trained} (eligible classes: {sorted(eligible_classes)})")
    print(f"Per-class official method: {per_class_official_method}")
    print(f"hard_proxy_best_thresholds: {hard_proxy_best_thresholds}")
    print(f"Full-policy verification plan (NOT executed): {full_policy_plan['estimated_runs']} runs, "
          f"{full_policy_plan['estimated_gpu_hours']}h, within_budget={full_policy_plan['within_budget']}")
    print(f"-> {cfg.paths.adaptive_threshold_manifest}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/08b_verify_full_policy_proxy.py`

*316 lines*

**Learned ASISM step 8 - measure whole *policies*, not individual thresholds.** Same fail-closed
discipline as 07b (`--i-understand-this-trains-real-models`), with a **separate, independently
approved** compute budget that is never summed with 07b's.

For each candidate policy it builds the **final combined multi-label `selected_manifest`** (per-class
thresholds combined via `min(applicable thresholds)`), trains real + selection, and measures utility
on `asism_tuning_heldout` across several seeds. The five policies:

1. fixed baseline threshold
2. hard-proxy-best among verified
3. the adaptive network (`predict_network_thresholds_per_class`)
4. `literal_top_50_percent`
5. `freematch_style_adaptive_percentile` - the **C4** policy:
   `percentile_threshold_per_class` scaled by range-normalised real prevalence
   (`real_prevalence_contexts_from_threshold_contexts`), plus the per-class selection floor.

`validate_policy_selection` checks each policy's output is well-formed before it is trusted.

In [ ]:
# ===== FILE: scripts/asism/08b_verify_full_policy_proxy.py =====
#!/usr/bin/env python3
"""Execute full_policy_verification_plan.json produced by 08_train_threshold_network.py.

Same fail-closed discipline as 07b_verify_thresholds_proxy.py: --phase run REFUSES to start (before
touching any config, data, or model) unless called with --i-understand-this-trains-real-models.
--phase estimate never requires it.

INDEPENDENT budget from 07b: learned_asism.threshold_network.full_policy_verification.compute_budget.
Never summed with 07b's per-context threshold-verification budget — they are separately approved,
separately executed production steps.

For each planned policy variant, builds the FINAL combined multi-label selected_manifest (per-class
thresholds combined via min(applicable_thresholds) — see build_policy_selected_manifest), then trains
real + that manifest's synthetic images vs a shared real-only baseline, for the configured number of
seeds, and appends results to full_policy_verification_results.jsonl.
"""
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    build_policy_selected_manifest, class_aware_context_vector, enforce_per_class_selection_floor,
    freematch_style_percentile_per_class, read_jsonl,
)
from scripts.asism.models import AdaptiveThresholdNetwork  # noqa: E402
from scripts.utils.artifact_contracts import stage2_paths, stage3_paths  # noqa: E402
from scripts.utils.classifier import (  # noqa: E402
    TrainingBudget, evaluate_classifier, records_from_split, records_from_synthetic_manifest, train_classifier,
)
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import hash_dict, read_json  # noqa: E402
from scripts.utils.splits import load_split  # noqa: E402


def compute_budget_estimate(cfg, n_variants: int, n_seeds: int) -> dict:
    budget = cfg.learned_asism.threshold_network.full_policy_verification.compute_budget
    hours_each = float(budget.hours_per_proxy_run_estimate)
    total_runs = 1 + n_variants * n_seeds  # 1 shared real-only baseline + variant*seed runs
    estimated_hours = total_runs * hours_each
    max_hours = float(budget.max_gpu_hours)
    return {
        "n_variants": n_variants,
        "n_seeds": n_seeds,
        "total_runs": total_runs,
        "hours_per_proxy_run_estimate": hours_each,
        "estimated_gpu_hours": round(estimated_hours, 3),
        "max_gpu_hours": max_hours,
        "within_budget": bool(estimated_hours <= max_hours),
    }


def enforce_compute_budget(cfg, n_variants: int, n_seeds: int) -> dict:
    estimate = compute_budget_estimate(cfg, n_variants, n_seeds)
    if not estimate["within_budget"]:
        raise SystemExit(
            "COMPUTE BUDGET GATE: full-policy verification exceeds its declared, INDEPENDENT budget "
            "(learned_asism.threshold_network.full_policy_verification.compute_budget).\n"
            f"  estimated: {estimate['estimated_gpu_hours']}h ({estimate['total_runs']} runs), "
            f"budget: {estimate['max_gpu_hours']}h\n"
            "Reduce full_policy_verification.n_variants/n_seeds and re-run "
            "08_train_threshold_network.py to regenerate the plan."
        )
    return estimate


def percentile_threshold_per_class(percentile: float, ranking_scores: dict, intended_by_id: dict,
                                   image_ids: list, primary_labels: list) -> dict:
    """literal_top_50_percent (percentile=50.0): the per-class score value at the given percentile
    among that class's own candidates — a real, computed threshold, not a fixed constant."""
    result = {}
    for label in primary_labels:
        class_scores = [ranking_scores[image_id] for image_id in image_ids
                        if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1]
        result[label] = float(np.percentile(class_scores, percentile)) if class_scores else None
    return result


def real_prevalence_contexts_from_threshold_contexts(contexts: list[dict], primary_labels: list[str]) -> dict:
    """Recover and validate the exact frozen prevalence inputs embedded by 07."""
    result = {}
    for label in primary_labels:
        values = [row.get("real_prevalence_context") for row in contexts if row.get("label") == label]
        if not values or any(value is None for value in values):
            raise ValueError(f"missing frozen real_prevalence_context for {label}; rebuild contexts with 07")
        canonical = json.dumps(values[0], sort_keys=True)
        if any(json.dumps(value, sort_keys=True) != canonical for value in values[1:]):
            raise ValueError(f"inconsistent frozen real_prevalence_context for {label}")
        result[label] = values[0]
    return result


def validate_policy_selection(policy_name: str, thresholds: dict, selected_ids: list[str], primary_labels: list[str]) -> None:
    """Reject unresolved or empty synthetic policies before any proxy training starts."""
    missing = [label for label in primary_labels if thresholds.get(label) is None]
    if missing:
        raise ValueError(f"policy {policy_name} has unresolved thresholds for: {missing}")
    if not selected_ids:
        raise ValueError(f"policy {policy_name} selected zero synthetic images")


def predict_network_thresholds_per_class(manifest: dict, checkpoint_path: Path, ranking_scores: dict,
                                          intended_by_id: dict, image_ids: list, primary_labels: list,
                                          real_prevalence_by_label: dict) -> dict:
    """adaptive_threshold_network: inference from the OFFICIAL checkpoint on a deployment-time
    context vector (built from the CURRENT full candidate pool for that class, not a training-time
    bootstrap resample). Only classes governance actually assigned this method get a prediction;
    every other class is None here regardless of what the network might output for it — this
    function must never be used to smuggle a network opinion into a class it wasn't approved for."""
    result = {label: None for label in primary_labels}
    if not checkpoint_path.is_file():
        return result
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = AdaptiveThresholdNetwork(
        checkpoint["number_of_classes"], checkpoint["context_dim"], checkpoint["embedding_dim"],
        tuple(checkpoint["hidden_dims"]), checkpoint["dropout"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    per_class_official_method = manifest["per_class_official_method"]
    for class_id, label in enumerate(primary_labels):
        if per_class_official_method.get(label) != "adaptive_threshold_network":
            continue
        class_scores = [ranking_scores[image_id] for image_id in image_ids
                        if int(intended_by_id.get(image_id, {}).get(label, 0)) == 1]
        if not class_scores:
            continue
        context = class_aware_context_vector(
            class_scores, real_prevalence_by_label[label],
            {"target_synthetic_to_real_ratio": 1.0},
        )
        with torch.no_grad():
            prediction = model(torch.tensor([class_id]), torch.tensor([context], dtype=torch.float32))
        result[label] = float(prediction.item())
    return result


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--phase", choices=["estimate", "run"], default="estimate")
    parser.add_argument(
        "--i-understand-this-trains-real-models", action="store_true",
        help="Required for --phase run. Confirms real GPU proxy-classifier training is about to start.",
    )
    args = parser.parse_args()

    if args.phase == "run" and not args.i_understand_this_trains_real_models:
        raise SystemExit(
            "REFUSED: --phase run launches real proxy-classifier training (GPU/RunPod cost).\n"
            "Pass --i-understand-this-trains-real-models to confirm. This guard runs before any "
            "config, data, or model is touched — nothing about this command can accidentally train."
        )

    cfg = load_named_config("stage3_asism.yaml", "stage3")
    stage2 = load_named_config("stage2_generation.yaml", "stage2")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths: cfg.paths[key] = str(value)
    cfg.split_namespace = namespace
    for key, value in stage2_paths(stage2, namespace).items():
        if key in stage2.paths: stage2.paths[key] = str(value)

    plan_path = Path(cfg.paths.full_policy_verification_plan)
    if not plan_path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: no full-policy verification plan at {plan_path}.\n"
            "Run: python scripts/asism/08_train_threshold_network.py"
        )
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
    n_variants, n_seeds = len(plan["policy_variants"]), int(plan["n_seeds"])

    if args.phase == "estimate":
        estimate = compute_budget_estimate(cfg, n_variants, n_seeds)
        print(json.dumps(estimate, indent=2))
        return 0 if estimate["within_budget"] else 2

    enforce_compute_budget(cfg, n_variants, n_seeds)

    manifest = read_json(Path(cfg.paths.adaptive_threshold_manifest))
    context_builder_path = Path(__file__).with_name("07_build_threshold_contexts.py")
    spec = importlib.util.spec_from_file_location("build_threshold_contexts_07", context_builder_path)
    context_builder = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(context_builder)
    context_builder.preflight_check_phase1_outputs(cfg)
    merged, intended, _ = context_builder.load_candidate_pool(cfg)
    columns = list(read_json(Path(cfg.paths.learned_dir) / "learned_training_manifest.json")["feature_columns"])
    _, ranker, training_manifest = context_builder.load_frozen_models(cfg, columns)
    normalized = context_builder.apply_feature_frame(merged, columns, training_manifest["normalization"])
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32)))
    ranking_scores: dict[str, float] = dict(
        zip(merged.image_id.astype(str), (1.0 / (1.0 + np.exp(-raw.numpy())))))
    image_ids = list(ranking_scores)

    stage1 = load_stage1_config()
    image_root = Path(stage1.paths.images_dir) / namespace
    real = records_from_split(
        load_split("classifier_train", namespace, purpose="schema_validation", caller="full_policy_verification"),
        image_root / "classifier_train",
    )
    evaluation_records = records_from_split(
        load_split("asism_tuning_heldout", namespace, purpose="schema_validation", caller="full_policy_verification"),
        image_root / "asism_tuning_heldout",
    )
    checkpoint_dir = Path(cfg.paths.learned_dir) / "full_policy_checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def train_and_score(train_records, checkpoint, tag, seed):
        budget = TrainingBudget(max_steps=int(cfg.tuning.coarse.proxy_max_steps),
                                batch_size=int(cfg.tuning.proxy.batch_size),
                                learning_rate=float(cfg.tuning.proxy.learning_rate), seed=seed,
                                eval_every_n_steps=0)
        model, _, _ = train_classifier(
            train_records=train_records, val_records=[], budget=budget,
            dropout_p=float(cfg.tuning.proxy.dropout_p), pretrained_source=str(cfg.tuning.proxy.pretrained_source),
            resolution=int(cfg.tuning.proxy.resolution), progress_desc=tag,
            checkpoint_path=checkpoint, resume_from=checkpoint if checkpoint.is_file() else None,
            checkpoint_metadata={"dataset_id": tag, "condition_or_draw_id": tag,
                                 "config_hash": hash_dict({"tag": tag, "seed": seed}, length=64),
                                 "label_policy": "chexpert_mask_uncertain_v1"})
        return float(evaluate_classifier(model, evaluation_records, int(cfg.tuning.proxy.resolution))["macro_auroc"])

    results_path = Path(cfg.paths.full_policy_verification_results)
    completed = {(r["policy"], r["seed"]) for r in read_jsonl(results_path)} if results_path.is_file() else set()
    stage2_manifest, stage2_images = Path(stage2.paths.manifest_path), Path(stage2.paths.images_dir)
    primary_labels = list(PRIMARY_ENDPOINT_LABELS)
    threshold_context_rows = read_jsonl(Path(cfg.paths.threshold_contexts))
    real_prevalence_by_label = real_prevalence_contexts_from_threshold_contexts(
        threshold_context_rows, primary_labels,
    )

    # Resolved ONCE — thresholds don't vary by seed. The OLD baseline uses its own frozen selection
    # artifact directly (06's learned_selected_manifest.jsonl), not a per-class threshold combined
    # here, because its selection logic is not the min(applicable_thresholds) rule this script uses
    # for the other variants.
    selected_ids_by_policy: dict[str, list[str]] = {}
    baseline_selection_manifest_path = Path(cfg.paths.learned_dir) / "learned_selection_manifest.json"
    if not baseline_selection_manifest_path.is_file():
        raise SystemExit("FULL-POLICY GATE: fixed-ratio selection manifest is missing; run 06 first")
    baseline_thresholds = dict(read_json(baseline_selection_manifest_path)["thresholds"])
    for policy_name in plan["policy_variants"]:
        if policy_name == "hard_proxy_best_among_verified":
            thresholds = dict(baseline_thresholds)
            thresholds.update({k: v for k, v in manifest["hard_proxy_best_thresholds"].items() if v is not None})
        elif policy_name == "literal_top_50_percent":
            thresholds = percentile_threshold_per_class(50.0, ranking_scores, intended, image_ids, primary_labels)
        elif policy_name == "freematch_style_adaptive_percentile":
            freematch_cfg = cfg.learned_asism.threshold_network.freematch_style
            thresholds = freematch_style_percentile_per_class(
                ranking_scores, intended, image_ids, primary_labels, real_prevalence_by_label,
                float(freematch_cfg.base_percentile), float(freematch_cfg.min_percentile),
            )
            thresholds = enforce_per_class_selection_floor(
                thresholds, ranking_scores, intended, image_ids, primary_labels,
                int(cfg.learned_asism.threshold_network.min_selected_per_label),
            )
        elif policy_name == "adaptive_threshold_network":
            network_thresholds = predict_network_thresholds_per_class(
                manifest, Path(cfg.paths.official_threshold_checkpoint), ranking_scores, intended,
                image_ids, primary_labels, real_prevalence_by_label,
            )
            thresholds = dict(baseline_thresholds)
            thresholds.update({k: v for k, v in manifest["hard_proxy_best_thresholds"].items() if v is not None})
            thresholds.update({k: v for k, v in network_thresholds.items() if v is not None})
        elif policy_name == "fixed_target_ratio_threshold_distillation_baseline_v1":
            baseline_manifest_path = Path(cfg.paths.learned_selected_manifest)
            selected_ids_by_policy[policy_name] = (
                sorted(row["image_id"] for row in read_jsonl(baseline_manifest_path))
                if baseline_manifest_path.is_file() else []
            )
            continue
        else:
            thresholds = {label: None for label in primary_labels}
        selected_ids_by_policy[policy_name] = build_policy_selected_manifest(
            thresholds, ranking_scores, intended, image_ids, primary_labels,
        )
        validate_policy_selection(policy_name, thresholds, selected_ids_by_policy[policy_name], primary_labels)

    if not selected_ids_by_policy.get("fixed_target_ratio_threshold_distillation_baseline_v1", []):
        raise SystemExit("FULL-POLICY GATE: fixed-ratio baseline selection is missing or empty; run 06 first")

    with open(results_path, "a", encoding="utf-8") as handle:
        for seed_offset in range(n_seeds):
            seed = int(cfg.learned_asism.subset_design.seed) + seed_offset
            baseline_checkpoint = checkpoint_dir / f"real_only_seed{seed}.pt"
            baseline_score = train_and_score(real, baseline_checkpoint, f"full-policy-real-only-s{seed}", seed)

            for policy_name in plan["policy_variants"]:
                if (policy_name, seed) in completed:
                    continue
                selected_ids = selected_ids_by_policy[policy_name]
                synthetic = records_from_synthetic_manifest(stage2_manifest, stage2_images, selected_ids)
                tag = f"full-policy-{policy_name}-s{seed}"
                checkpoint = checkpoint_dir / f"{hash_dict({'tag': tag}, length=32)}.pt"
                measured = train_and_score(real + synthetic, checkpoint, tag, seed)
                row = {
                    "policy": policy_name, "seed": seed, "n_selected": len(selected_ids),
                    "real_only_macro_auroc": baseline_score, "augmented_macro_auroc": measured,
                    "measured_utility": measured - baseline_score,
                }
                handle.write(json.dumps(row, sort_keys=True) + "\n")
                print(f"[{policy_name} seed={seed}] measured_utility={row['measured_utility']:+.6f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/asism/09_finalize_learned_selection.py`

*165 lines*

**Learned ASISM step 9 - commit to exactly one policy.** Trains nothing and never opens
`final_eval_heldout`.

It consumes the complete 08b proxy measurements, applies the **pre-registered tie rule**
(`tie_noise_band` + `simplicity_order`: within the noise band, prefer the simpler policy), commits to
one winner, and writes the frozen `adaptive_selected_manifest.jsonl` - the exact image list that
becomes Stage 4's condition C. The fixed-ratio output from step 06 is left untouched.

The tie rule being pre-registered is what stops "pick whichever policy won" from becoming
"pick whichever policy won on this particular noisy measurement".

In [ ]:
# ===== FILE: scripts/asism/09_finalize_learned_selection.py =====
#!/usr/bin/env python3
"""Freeze the winning learned-ASISM policy and publish its Stage-4 selection.

This step never trains a model and never opens final_eval_heldout. It consumes the complete
full-policy proxy measurements made on asism_tuning_heldout, applies the pre-registered tie rule,
and writes a new adaptive manifest. The fixed-ratio output from 06 remains untouched.
"""
from __future__ import annotations

import importlib.util
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.asism.learned import (  # noqa: E402
    build_policy_selected_manifest, choose_full_policy, enforce_per_class_selection_floor,
    freematch_style_percentile_per_class, read_jsonl,
)
from scripts.utils.artifact_contracts import (  # noqa: E402
    current_code_identity_hash, namespace_identity, require_generation_complete, stage3_paths,
)
from scripts.utils.config import load_named_config  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import read_json, sha256_file, write_frozen_json  # noqa: E402


def _load_neighbor(filename: str, module_name: str):
    path = Path(__file__).with_name(filename)
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--namespace", default=None)
    args = parser.parse_args()
    cfg = load_named_config("stage3_asism.yaml", "stage3")
    namespace = args.namespace or str(cfg.split_namespace)
    for key, value in stage3_paths(cfg, namespace).items():
        if key in cfg.paths:
            cfg.paths[key] = str(value)
    cfg.split_namespace = namespace

    required = [
        Path(cfg.paths.full_policy_verification_plan),
        Path(cfg.paths.full_policy_verification_results),
        Path(cfg.paths.adaptive_threshold_manifest),
        Path(cfg.paths.threshold_contexts),
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise SystemExit(f"FINALIZATION GATE: missing upstream artifacts: {missing}")

    plan = read_json(Path(cfg.paths.full_policy_verification_plan))
    results = read_jsonl(Path(cfg.paths.full_policy_verification_results))
    fp_cfg = cfg.learned_asism.threshold_network.full_policy_verification
    seeds = [int(cfg.learned_asism.subset_design.seed) + index for index in range(int(plan["n_seeds"]))]
    decision = choose_full_policy(
        results, list(plan["policy_variants"]), seeds,
        float(fp_cfg.finalization.tie_noise_band), list(fp_cfg.finalization.simplicity_order),
    )

    context_builder = _load_neighbor("07_build_threshold_contexts.py", "threshold_context_builder_07")
    verifier = _load_neighbor("08b_verify_full_policy_proxy.py", "full_policy_verifier_08b")
    context_builder.preflight_check_phase1_outputs(cfg)
    merged, intended, _ = context_builder.load_candidate_pool(cfg)
    # Provenance recorded in the frozen selection manifest below; Stage 4 checks these before it
    # trains on the selection. Resolved before anything is written, so a mismatch leaves no output.
    identity = namespace_identity(namespace)
    _, generation = require_generation_complete(load_named_config("stage2_generation.yaml", "stage2"), namespace)
    columns = list(read_json(Path(cfg.paths.learned_dir) / "learned_training_manifest.json")["feature_columns"])
    _, ranker, training_manifest = context_builder.load_frozen_models(cfg, columns)
    normalized = context_builder.apply_feature_frame(merged, columns, training_manifest["normalization"])
    with torch.no_grad():
        raw = ranker(torch.tensor(normalized.to_numpy(np.float32)))
    ranking_scores = dict(zip(merged.image_id.astype(str), 1.0 / (1.0 + np.exp(-raw.numpy()))))
    image_ids = list(ranking_scores)
    labels = list(PRIMARY_ENDPOINT_LABELS)
    threshold_manifest = read_json(Path(cfg.paths.adaptive_threshold_manifest))

    baseline_manifest_path = Path(cfg.paths.learned_dir) / "learned_selection_manifest.json"
    baseline_ids_path = Path(cfg.paths.learned_selected_manifest)
    if not baseline_manifest_path.is_file() or not baseline_ids_path.is_file():
        raise SystemExit("FINALIZATION GATE: fixed-ratio baseline artifacts are missing; run 06 first")
    baseline_thresholds = dict(read_json(baseline_manifest_path)["thresholds"])
    winner = decision["winning_policy"]
    thresholds = None
    if winner == "fixed_target_ratio_threshold_distillation_baseline_v1":
        selected_ids = sorted(row["image_id"] for row in read_jsonl(baseline_ids_path))
    else:
        if winner == "literal_top_50_percent":
            thresholds = verifier.percentile_threshold_per_class(50.0, ranking_scores, intended, image_ids, labels)
        elif winner == "freematch_style_adaptive_percentile":
            contexts = read_jsonl(Path(cfg.paths.threshold_contexts))
            prevalence = verifier.real_prevalence_contexts_from_threshold_contexts(contexts, labels)
            freematch_cfg = cfg.learned_asism.threshold_network.freematch_style
            thresholds = freematch_style_percentile_per_class(
                ranking_scores, intended, image_ids, labels, prevalence,
                float(freematch_cfg.base_percentile), float(freematch_cfg.min_percentile),
            )
            thresholds = enforce_per_class_selection_floor(
                thresholds, ranking_scores, intended, image_ids, labels,
                int(cfg.learned_asism.threshold_network.min_selected_per_label),
            )
        elif winner == "hard_proxy_best_among_verified":
            thresholds = dict(baseline_thresholds)
            thresholds.update({k: v for k, v in threshold_manifest["hard_proxy_best_thresholds"].items() if v is not None})
        elif winner == "adaptive_threshold_network":
            contexts = read_jsonl(Path(cfg.paths.threshold_contexts))
            prevalence = verifier.real_prevalence_contexts_from_threshold_contexts(contexts, labels)
            network = verifier.predict_network_thresholds_per_class(
                threshold_manifest, Path(cfg.paths.official_threshold_checkpoint), ranking_scores,
                intended, image_ids, labels, prevalence,
            )
            thresholds = dict(baseline_thresholds)
            thresholds.update({k: v for k, v in threshold_manifest["hard_proxy_best_thresholds"].items() if v is not None})
            thresholds.update({k: v for k, v in network.items() if v is not None})
        else:
            raise SystemExit(f"FINALIZATION GATE: unsupported winning policy {winner}")
        selected_ids = build_policy_selected_manifest(thresholds, ranking_scores, intended, image_ids, labels)
        verifier.validate_policy_selection(winner, thresholds, selected_ids, labels)

    if not selected_ids:
        raise SystemExit(f"FINALIZATION GATE: winning policy {winner} selected zero images")
    with open(Path(cfg.paths.adaptive_selected_manifest), "x", encoding="utf-8") as handle:
        for image_id in selected_ids:
            handle.write(json.dumps({
                "image_id": image_id,
                "ranking_score": float(ranking_scores[image_id]),
                "selector": "adaptive_learned_asism_v2",
                "winning_policy": winner,
            }, sort_keys=True) + "\n")

    write_frozen_json(Path(cfg.paths.adaptive_selection_manifest), {
        "schema_version": 1,
        "method": "class_aware_adaptive_threshold_v2",
        "frozen": True,
        "split_namespace": namespace,
        "namespace_class": identity["namespace_class"],
        "split_manifest_hash": identity["split_manifest_hash"],
        "generation_manifest_sha256": generation["generation_manifest_sha256"],
        "code_identity_sha256": current_code_identity_hash(),
        "winning_policy": winner,
        "policy_decision": decision,
        "thresholds": thresholds,
        "selected_count": len(selected_ids),
        "full_policy_results_sha256": sha256_file(Path(cfg.paths.full_policy_verification_results)),
        "adaptive_threshold_manifest_sha256": sha256_file(Path(cfg.paths.adaptive_threshold_manifest)),
        "ranking_checkpoint_sha256": sha256_file(Path(cfg.paths.learned_dir) / "ranking_model.pt"),
        "selected_manifest_sha256": sha256_file(Path(cfg.paths.adaptive_selected_manifest)),
    })
    print(f"Final adaptive policy: {winner}; selected {len(selected_ids)} images")
    print(f"-> {cfg.paths.adaptive_selected_manifest}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 9 - Stage 4: the thesis conditions

Three classifiers under one frozen protocol. Only the training data differs.

## `scripts/classify/01_train_conditions.py`

*640 lines*

**Stage 4 - train the thesis conditions.**

| Condition | Training data |
|---|---|
| **A** | real only (`classifier_train`) |
| **B** | real + **all** Stage 2 synthetic images |
| **C** | real + the finalized ASISM adaptive selection |

**Primary comparison: C vs B** - does *selecting* synthetic images with ASISM beat using all of them
unselected?

**The KNOWN LIMITATION is stated in the file itself, and you should read it before reading the
results.** C is a strict subset of B, so a C-over-B gain has two competing explanations this design
cannot separate: (a) ASISM chose *good* images, or (b) using *fewer* synthetic images helps
regardless of which ones, because unselected synthetic data is noisy. Distinguishing them needs a
condition that draws |C| images at random with C's label profile - optional condition **D**. The
matched-draw machinery (`build_matched_random_draw`, `profile_of`, `condition_d.*`) is retained and
working for exactly that purpose, but D is not enabled by default.

Fairness machinery: `training_plan` computes the run matrix; every enabled condition and seed gets
the **same optimizer-step budget**; model selection is on `classifier_val` only. Four manifests are
frozen before training starts (protocol, label policy, threshold policy, seed plan), so the protocol
cannot be adjusted after seeing results. Runs are resumable per `(condition, seed, draw)` tag.

In [ ]:
# ===== FILE: scripts/classify/01_train_conditions.py =====
#!/usr/bin/env python3
"""Stage 4 — train the thesis conditions A/B/C.

The three arms the thesis specifies for Stage 5:

    A  real only                     classifier_train
    B  real + ALL synthetic          classifier_train + every Stage 2 image
    C  real + ASISM-selected         classifier_train + the finalized adaptive selection

Primary comparison: C vs B — does selecting synthetic images with ASISM beat using all of them
unselected?

KNOWN LIMITATION — no matched-random control. C is a strict subset of B, so a C-over-B gain has two
competing explanations that this design cannot separate: (a) ASISM chose *good* images, or (b) using
*fewer* synthetic images helps regardless of which ones, because unselected synthetic data is noisy.
Distinguishing them needs a condition drawing |C| images at random with C's label profile
(optional condition D). The matched-draw machinery below (build_matched_random_draw, profile_of,
condition_d.*) is retained and working for exactly that purpose, but D is not enabled by default.
See docs/stages2_to_5_plan.md §7.

Fairness (§7.1): every run — all conditions, all seeds — uses the SAME optimizer-step budget, batch
size, augmentation, and checkpoint-selection rule. All model selection is on classifier_val;
final_eval_heldout is never touched here.

Usage:
    python scripts/classify/01_train_conditions.py --condition A
    python scripts/classify/01_train_conditions.py --condition all
    python scripts/classify/01_train_conditions.py --plan-only            # run/cost accounting
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.experiment_registry import ExperimentRun  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, get_library_versions, hash_dict, read_json, sha256_file, write_frozen_json, write_json  # noqa: E402
from scripts.utils.splits import load_split, split_provenance  # noqa: E402
from scripts.utils.artifact_contracts import (  # noqa: E402
    current_code_identity_hash, namespace_identity, require_generation_complete,
    require_manifest_fields, stage2_paths, stage3_paths, stage4_paths,
)

CONDITIONS = ["A", "B", "C"]


def load_configs():
    stage4 = load_named_config("stage4_classifier.yaml", "stage4")
    stage3 = load_named_config("stage3_asism.yaml", "stage3")
    stage2 = load_named_config("stage2_generation.yaml", "stage2")
    return stage4, stage3, stage2


def load_all_synthetic_ids(stage2_cfg) -> list[str]:
    path = Path(stage2_cfg.paths.manifest_path)
    if not path.is_file():
        raise SystemExit(f"UPSTREAM GATE: missing generation manifest {path}")
    ids = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                ids.append(json.loads(line)["image_id"])
    return ids


def load_selected_ids(stage3_cfg, selector: str = "weighted") -> list[str]:
    paths = {
        "weighted": stage3_cfg.paths.selected_manifest,
        "adaptive": stage3_cfg.paths.adaptive_selected_manifest,
        "fixed_ratio": stage3_cfg.paths.learned_selected_manifest,
    }
    if selector not in paths:
        raise ValueError(f"unknown selector {selector}")
    path = Path(paths[selector])
    if not path.is_file():
        raise SystemExit(
            f"UPSTREAM GATE: ASISM selection not found at {path}\n"
            "Run the weighted selector or the learned-ASISM selection pipeline, as applicable."
        )
    ids = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                ids.append(json.loads(line)["image_id"])
    return ids


def intended_lookup(stage2_cfg) -> dict[str, dict]:
    lookup = {}
    with open(Path(stage2_cfg.paths.manifest_path), encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                entry = json.loads(line)
                lookup[entry["image_id"]] = entry["intended_label_vector"]
    return lookup


def profile_of(image_ids: list[str], intended: dict[str, dict]) -> dict:
    """Class-distribution profile used to match a random-draw control against a reference condition."""
    per_label = {label: 0 for label in PRIMARY_ENDPOINT_LABELS}
    single, multi, none = 0, 0, 0
    for image_id in image_ids:
        vector = intended.get(image_id, {})
        positives = [l for l in PRIMARY_ENDPOINT_LABELS if int(vector.get(l, 0)) == 1]
        for label in positives:
            per_label[label] += 1
        if len(positives) == 0:
            none += 1
        elif len(positives) == 1:
            single += 1
        else:
            multi += 1
    total = max(len(image_ids), 1)
    return {
        "total": len(image_ids),
        "per_label_positive": per_label,
        "single_label": single,
        "multi_label": multi,
        "no_finding": none,
        "single_fraction": single / total,
        "multi_fraction": multi / total,
    }


def build_matched_random_draw(
    all_ids: list[str],
    target_profile: dict,
    intended: dict[str, dict],
    seed: int,
    config,
) -> tuple[list[str], dict]:
    """One matched-random draw for condition D.

    Greedy per-label matching against the target's marginal positive counts, then a top-up to the
    exact total. Joint label-vector matching is ATTEMPTED (by preferring candidates whose exact
    vector the target also contains) but never allowed to make matching infeasible — a strict joint
    match is unachievable for rare combinations and would silently shrink the draw.

    Residual imbalance is measured and returned, never hidden.
    """
    matching = config.condition_d.matching
    rng = np.random.default_rng(seed)
    available = [i for i in all_ids]
    rng.shuffle(available)

    target_counts = dict(target_profile["per_label_positive"])
    target_total = int(target_profile["total"])

    by_vector: dict[tuple, list[str]] = {}
    for image_id in available:
        vector = intended.get(image_id, {})
        key = tuple(sorted(l for l in PRIMARY_ENDPOINT_LABELS if int(vector.get(l, 0)) == 1))
        by_vector.setdefault(key, []).append(image_id)

    chosen: list[str] = []
    chosen_set: set[str] = set()
    counts = {label: 0 for label in PRIMARY_ENDPOINT_LABELS}

    if bool(matching.match_per_label_marginal):
        for label in sorted(PRIMARY_ENDPOINT_LABELS, key=lambda l: target_counts[l]):
            need = target_counts[label]
            if need <= 0:
                continue
            candidates = [
                image_id
                for image_id in available
                if image_id not in chosen_set
                and int(intended.get(image_id, {}).get(label, 0)) == 1
            ]
            for image_id in candidates:
                if counts[label] >= need or len(chosen) >= target_total:
                    break
                chosen.append(image_id)
                chosen_set.add(image_id)
                for other in PRIMARY_ENDPOINT_LABELS:
                    if int(intended.get(image_id, {}).get(other, 0)) == 1:
                        counts[other] += 1

    # Top up to the exact total, preferring No-Finding vectors if the target has them.
    if len(chosen) < target_total:
        remaining = [i for i in available if i not in chosen_set]
        need_no_finding = target_profile["no_finding"] - sum(
            1 for i in chosen if not any(
                int(intended.get(i, {}).get(l, 0)) == 1 for l in PRIMARY_ENDPOINT_LABELS
            )
        )
        no_finding_pool = [
            i for i in remaining
            if not any(int(intended.get(i, {}).get(l, 0)) == 1 for l in PRIMARY_ENDPOINT_LABELS)
        ]
        for image_id in no_finding_pool[: max(0, need_no_finding)]:
            if len(chosen) >= target_total:
                break
            chosen.append(image_id)
            chosen_set.add(image_id)
        for image_id in remaining:
            if len(chosen) >= target_total:
                break
            if image_id not in chosen_set:
                chosen.append(image_id)
                chosen_set.add(image_id)

    chosen = chosen[:target_total]

    # Deterministic repair: hill-climb swaps against the frozen marginal and single/multi target.
    # No reseeding/retry is used; ties follow the seeded `available` order above.
    def positive_set(image_id):
        return frozenset(label for label in PRIMARY_ENDPOINT_LABELS if int(intended.get(image_id, {}).get(label, 0)) == 1)
    chosen_vectors = [positive_set(image_id) for image_id in chosen]
    current_counts = {label: sum(label in vector for vector in chosen_vectors) for label in PRIMARY_ENDPOINT_LABELS}
    current_single = sum(len(vector) == 1 for vector in chosen_vectors)
    current_multi = sum(len(vector) > 1 for vector in chosen_vectors)
    def mismatch_counts(label_counts, single, multi):
        label_error = sum(abs(label_counts[label] - target_counts[label]) / max(target_counts[label], 1) for label in PRIMARY_ENDPOINT_LABELS)
        shape_error = abs(single / max(target_total, 1) - target_profile["single_fraction"]) + abs(multi / max(target_total, 1) - target_profile["multi_fraction"])
        return label_error + shape_error
    current_score = mismatch_counts(current_counts, current_single, current_multi)
    outside = [image_id for image_id in available if image_id not in chosen_set][:500]
    for _ in range(min(100, max(target_total, 1))):
        best = None
        for chosen_index, old_id in list(enumerate(chosen))[:500]:
            old_vector = chosen_vectors[chosen_index]
            for new_id in outside:
                new_vector = positive_set(new_id)
                trial_counts = {label: current_counts[label] - (label in old_vector) + (label in new_vector) for label in PRIMARY_ENDPOINT_LABELS}
                trial_single = current_single - (len(old_vector) == 1) + (len(new_vector) == 1)
                trial_multi = current_multi - (len(old_vector) > 1) + (len(new_vector) > 1)
                score = mismatch_counts(trial_counts, trial_single, trial_multi)
                if score + 1e-12 < current_score and (best is None or score < best[0]):
                    best = (score, chosen_index, old_id, new_id, new_vector, trial_counts, trial_single, trial_multi)
        if best is None:
            break
        current_score, chosen_index, old_id, new_id, new_vector, current_counts, current_single, current_multi = best
        chosen[chosen_index] = new_id
        chosen_vectors[chosen_index] = new_vector
        outside.remove(new_id)
        outside.append(old_id)
        chosen_set.remove(old_id)
        chosen_set.add(new_id)
    achieved = profile_of(chosen, intended)

    tolerance = float(matching.per_label_tolerance_fraction)
    residual = {}
    within_tolerance = True
    for label in PRIMARY_ENDPOINT_LABELS:
        target_count = target_counts[label]
        achieved_count = achieved["per_label_positive"][label]
        allowed = max(1.0, tolerance * max(target_count, 1))
        delta = achieved_count - target_count
        residual[label] = {
            "target": target_count,
            "achieved": achieved_count,
            "delta": delta,
            "within_tolerance": bool(abs(delta) <= allowed),
        }
        if abs(delta) > allowed:
            within_tolerance = False

    single_delta = abs(achieved["single_fraction"] - target_profile["single_fraction"])
    single_ok = single_delta <= float(matching.single_multi_tolerance_fraction)

    valid_pool = set(all_ids)
    constraints = {
        "exact_total": achieved["total"] == target_total,
        "per_label_marginals": within_tolerance,
        "single_multi_proportion": single_ok,
        "no_duplicate_ids": len(chosen) == len(set(chosen)),
        "all_ids_in_valid_pool": set(chosen).issubset(valid_pool),
    }
    unsatisfied = [name for name, passed in constraints.items() if not passed]
    return chosen, {
        "seed": seed,
        "target_total": target_total,
        "achieved_total": achieved["total"],
        "count_matched_exactly": bool(achieved["total"] == target_total),
        "per_label_residual": residual,
        "all_labels_within_tolerance": within_tolerance,
        "single_fraction_target": target_profile["single_fraction"],
        "single_fraction_achieved": achieved["single_fraction"],
        "single_fraction_delta": single_delta,
        "single_multi_within_tolerance": bool(single_ok),
        "joint_vector_matching_attempted": bool(matching.attempt_joint_vector_matching),
        "note": "Residual imbalance is recorded, not hidden; exact joint matching is not forced.",
        "constraints": constraints,
        "passed": not unsatisfied,
        "unsatisfied_constraints": unsatisfied,
        "draw_id_hash": hash_dict({"seed": seed, "ids": chosen}, length=64),
    }


def training_plan(stage4_cfg) -> dict:
    seeds = list(stage4_cfg.seeds)
    n_draws = int(stage4_cfg.condition_d.n_draws)
    all_runs = {
        "A": len(seeds), "B": len(seeds), "C": len(seeds),
        "D": n_draws * len(seeds),
    }
    enabled = list(stage4_cfg.get("conditions", CONDITIONS))
    runs = {condition: all_runs[condition] for condition in enabled}
    total = sum(runs.values())
    return {
        "runs_per_condition": runs,
        "total_runs": total,
        "seeds": seeds,
        "n_d_draws": n_draws,
        "optimizer_steps_per_run": int(stage4_cfg.training.max_steps),
        "total_optimizer_steps": total * int(stage4_cfg.training.max_steps),
        "formula": "sum of enabled per-condition runs; D contributes n_draws x |seeds|",
    }


def build_records(condition: str, draw_ids: list[str] | None, cfgs, namespace: str):
    from scripts.utils.classifier import records_from_split, records_from_synthetic_manifest

    stage4_cfg, stage3_cfg, stage2_cfg = cfgs
    stage1_cfg = load_stage1_config()
    images_dir = Path(stage1_cfg.paths.images_dir) / namespace

    frame = load_split(
        "classifier_train", namespace, purpose="schema_validation", caller="stage4"
    )
    real_records = records_from_split(frame, images_dir / "classifier_train")
    if not real_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed classifier_train images under "
            f"{images_dir / 'classifier_train'}"
        )

    synthetic_ids: list[str] | None = None
    if condition == "B":
        synthetic_ids = None  # all
    elif condition == "C":
        synthetic_ids = load_selected_ids(stage3_cfg, selector="adaptive")
    elif condition == "D":
        synthetic_ids = draw_ids

    synthetic_records = []
    if condition != "A":
        synthetic_records = records_from_synthetic_manifest(
            Path(stage2_cfg.paths.manifest_path),
            Path(stage2_cfg.paths.images_dir),
            synthetic_ids,
        )
        if not synthetic_records:
            raise SystemExit(
                f"UPSTREAM GATE: condition {condition} needs synthetic images but none were found."
            )

    val_frame = load_split("classifier_val", namespace, purpose="schema_validation", caller="stage4")
    val_records = records_from_split(val_frame, images_dir / "classifier_val")
    if not val_records:
        raise SystemExit(
            f"UPSTREAM GATE: no preprocessed classifier_val images under "
            f"{images_dir / 'classifier_val'}"
        )

    return real_records + synthetic_records, val_records


def run_one(condition: str, seed: int, draw_index: int | None, draw_ids, cfgs, namespace: str) -> dict:
    from scripts.utils.classifier import TrainingBudget, train_classifier

    stage4_cfg, _, _ = cfgs
    train_records, val_records = build_records(condition, draw_ids, cfgs, namespace)

    tag = f"{condition}" + (f"_draw{draw_index}" if draw_index is not None else "") + f"_seed{seed}"
    checkpoint_path = Path(stage4_cfg.paths.checkpoints_dir) / f"condition_{tag}.pt"

    budget = TrainingBudget(
        max_steps=int(stage4_cfg.training.max_steps),
        batch_size=int(stage4_cfg.training.batch_size),
        learning_rate=float(stage4_cfg.training.learning_rate),
        weight_decay=float(stage4_cfg.training.weight_decay),
        eval_every_n_steps=int(stage4_cfg.training.eval_every_n_steps),
        seed=seed,
    )

    config_block = {
        "stage4": OmegaConf.to_container(stage4_cfg, resolve=True),
        "condition": condition,
        "draw_index": draw_index,
        "seed": seed,
    }

    with ExperimentRun(
        stage=f"stage4_condition_{condition}",
        config=config_block,
        dataset_version=split_provenance(namespace)["split_manifest_hash"],
    ) as run:
        model, accounting, history = train_classifier(
            train_records=train_records,
            val_records=val_records,
            budget=budget,
            dropout_p=float(stage4_cfg.model.dropout_p),
            pretrained_source=str(stage4_cfg.model.pretrained_source),
            resolution=int(stage4_cfg.model.resolution),
            progress_desc=f"cond-{tag}",
            checkpoint_path=checkpoint_path,
            resume_from=checkpoint_path if checkpoint_path.is_file() else None,
            checkpoint_metadata={
                "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
                "split_namespace": namespace, "dataset_id": tag,
                "condition_or_draw_id": f"{condition}:{draw_index}",
                "config_hash": hash_dict(config_block), "label_policy": "chexpert_mask_uncertain_v1",
                "code_version": get_git_commit_hash(), "code_identity_sha256": current_code_identity_hash(),
            },
        )
        accounting.dataset_id = tag
        accounting.config_hash = hash_dict(config_block)
        accounting.sampling_policy = str(stage4_cfg.training.sampling_policy)
        best = max(history, key=lambda entry: entry["macro_auroc"]) if history else {}
        run.set_checkpoint_path(checkpoint_path)
        run.set_metrics({"best_val_macro_auroc": best.get("macro_auroc"), **accounting.to_dict()})

    return {
        "tag": tag,
        "condition": condition,
        "draw_index": draw_index,
        "seed": seed,
        "checkpoint": accounting.extra.get("best_checkpoint_path", str(checkpoint_path)),
        "resume_checkpoint": str(checkpoint_path),
        "best_val_macro_auroc": best.get("macro_auroc"),
        "accounting": accounting.to_dict(),
        "eval_history": history,
    }


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--condition", choices=CONDITIONS + ["all"], default="all")
    parser.add_argument("--namespace", default=None)
    parser.add_argument("--plan-only", action="store_true", help="Print run/cost accounting and exit")
    args = parser.parse_args()

    cfgs = load_configs()
    stage4_cfg, stage3_cfg, stage2_cfg = cfgs
    namespace = args.namespace or str(stage4_cfg.split_namespace)
    stage4_cfg.split_namespace = namespace
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    for key, value in stage3_paths(stage3_cfg, namespace).items():
        if key in stage3_cfg.paths:
            stage3_cfg.paths[key] = str(value)
    for key, value in stage2_paths(stage2_cfg, namespace).items():
        if key in stage2_cfg.paths:
            stage2_cfg.paths[key] = str(value)

    plan = training_plan(stage4_cfg)
    print("Stage 4 run plan (§7 frozen seed policy)", flush=True)
    print(json.dumps(plan, indent=2), flush=True)
    if args.plan_only:
        return 0

    identity = namespace_identity(namespace)
    _, generation_completion = require_generation_complete(stage2_cfg, namespace)
    # The frozen Stage 3 selection is 09's Learned-ASISM manifest (the sole selector; Stage 5 checks
    # the same file). The weighted selector's asism_frozen_manifest.json (03) is not produced by the
    # pipeline, so gating on it made Stage 4 unrunnable.
    require_manifest_fields(Path(stage3_cfg.paths.adaptive_selection_manifest), {
        "split_namespace": namespace, "namespace_class": identity["namespace_class"],
        "split_manifest_hash": identity["split_manifest_hash"],
        "generation_manifest_sha256": generation_completion["generation_manifest_sha256"],
        "code_identity_sha256": current_code_identity_hash(), "frozen": True,
    }, "frozen Learned ASISM selection")

    results_dir = Path(stage4_cfg.paths.results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    # Freeze the experiment protocol before any training run starts (§9 step 11).
    protocol = {
        "stage": "stage4_frozen_protocol",
        "frozen": True,
        "split_provenance": split_provenance(namespace),
        "model": OmegaConf.to_container(stage4_cfg.model, resolve=True),
        "training": OmegaConf.to_container(stage4_cfg.training, resolve=True),
        "seeds": list(stage4_cfg.seeds),
        "condition_d": OmegaConf.to_container(stage4_cfg.condition_d, resolve=True),
        "threshold_policy": OmegaConf.to_container(stage4_cfg.threshold_policy, resolve=True),
        "label_policy": {
            "classifier_target_labels": 14,
            "primary_endpoint_labels": PRIMARY_ENDPOINT_LABELS,
            "no_finding_in_primary": False,
            "support_devices_in_primary": False,
        },
        "uncertainty_policy": "raw preserved; -1 and blank masked in loss and metrics",
        "fairness_protocol": "equal optimizer steps across every enabled condition and seed",
        "primary_endpoint": f"macro-AUROC over the {len(PRIMARY_ENDPOINT_LABELS)} primary labels on final_eval_heldout",
        "primary_comparison": "C vs B (ASISM-selected synthetic images vs all synthetic images)",
        "known_limitation": (
            "C is a strict subset of B, and no matched-random control condition is enabled. A "
            "C-over-B gain therefore cannot distinguish ASISM's selection quality from the effect "
            "of simply using fewer synthetic images."
        ),
        "multiplicity": {"confirmatory": "holm_bonferroni", "exploratory": "benjamini_hochberg"},
        "run_plan": plan,
        "git_commit_hash": get_git_commit_hash(),
        "code_identity_sha256": current_code_identity_hash(),
        "environment_versions": get_library_versions(),
        "serialized_config": OmegaConf.to_container(stage4_cfg, resolve=True),
    }
    protocol_path = Path(stage4_cfg.paths.protocol_manifest)
    if protocol_path.exists():
        existing = read_json(protocol_path)
        if existing != protocol:
            raise SystemExit("Frozen experiment protocol already exists with different content; use a new versioned protocol/run ID")
    else:
        write_frozen_json(protocol_path, protocol)
    freeze_components = {
        Path(stage4_cfg.paths.label_policy_manifest): protocol["label_policy"],
        Path(stage4_cfg.paths.threshold_policy_manifest): protocol["threshold_policy"],
        Path(stage4_cfg.paths.seed_plan_manifest): {"seeds": protocol["seeds"], "run_plan": plan},
        Path(stage4_cfg.paths.d_matching_policy_manifest): protocol["condition_d"],
    }
    for artifact_path, payload in freeze_components.items():
        frozen_payload = {"schema_version": 1, "protocol_manifest_hash": sha256_file(protocol_path),
                          "config": payload, "environment_versions": protocol.get("environment_versions", {})}
        if artifact_path.exists():
            existing = read_json(artifact_path)
            if existing != {**frozen_payload, "frozen": True}:
                raise SystemExit(f"Frozen methodology artifact differs and cannot be overwritten: {artifact_path}")
        else:
            write_frozen_json(artifact_path, frozen_payload)
    print(f"\nFrozen protocol -> {stage4_cfg.paths.protocol_manifest}", flush=True)

    enabled_conditions = list(stage4_cfg.get("conditions", CONDITIONS))
    if args.condition != "all" and args.condition not in enabled_conditions:
        raise SystemExit(f"Condition {args.condition} is disabled by this explicit configuration: {enabled_conditions}")
    conditions = enabled_conditions if args.condition == "all" else [args.condition]
    if "C" in conditions:
        require_manifest_fields(
            Path(stage3_cfg.paths.adaptive_selection_manifest),
            {"frozen": True, "method": "class_aware_adaptive_threshold_v2"},
            "adaptive learned ASISM selection",
        )
    seeds = list(stage4_cfg.seeds)
    all_results = []
    completed_dir = results_dir / "completed_runs"
    completed_dir.mkdir(exist_ok=True)

    def execute_or_resume(condition, seed, draw_index=None, ids=None):
        tag = f"{condition}" + (f"_draw{draw_index}" if draw_index is not None else "") + f"_seed{seed}"
        expected_run_identity = {
            "split_manifest_hash": split_provenance(namespace)["split_manifest_hash"],
            "split_namespace": namespace, "tag": tag,
            "code_identity_sha256": current_code_identity_hash(),
        }
        record_path = completed_dir / f"{tag}.json"
        if record_path.is_file():
            record = read_json(record_path)
            checkpoint = Path(record.get("checkpoint", ""))
            history_hash = hash_dict({"history": record.get("result", {}).get("eval_history", [])}, length=64)
            compatible = all(record.get(key) == value for key, value in expected_run_identity.items())
            if (record.get("status") == "complete" and compatible and checkpoint.is_file()
                    and record.get("checkpoint_hash") == sha256_file(checkpoint)
                    and record.get("metric_history_hash") == history_hash):
                return record["result"]
            raise SystemExit(f"Invalid/corrupt completed-run record for {tag}; refusing silent reuse: {record_path}")
        result = run_one(condition, seed, draw_index, ids, cfgs, namespace)
        checkpoint = Path(result["checkpoint"])
        history_hash = hash_dict({"history": result["eval_history"]}, length=64)
        completion = {"schema_version": 2, "status": "complete", **expected_run_identity, "checkpoint": str(checkpoint),
                      "checkpoint_hash": sha256_file(checkpoint), "metric_history_hash": history_hash,
                      "accounting": result["accounting"], "split_provenance": split_provenance(namespace), "result": result}
        write_json(record_path, completion)
        return result

    # Matched-random draws are built against the reference condition's actual selection.
    draw_specs: list[tuple[int, list[str], dict]] = []
    if "D" in conditions:
        intended = intended_lookup(stage2_cfg)
        selected_ids = load_selected_ids(stage3_cfg)
        target_profile = profile_of(selected_ids, intended)
        all_ids = load_all_synthetic_ids(stage2_cfg)
        pool = [i for i in all_ids if i not in set(selected_ids)] or all_ids

        for draw_index, draw_seed in enumerate(list(stage4_cfg.condition_d.draw_seeds)):
            ids, report = build_matched_random_draw(
                pool, target_profile, intended, draw_seed, stage4_cfg
            )
            draw_specs.append((draw_index, ids, report))
            if not report["passed"]:
                raise SystemExit(
                    f"Condition D matching infeasible for draw {draw_index}, seed {draw_seed}; "
                    f"refusing training. Unsatisfied constraints: {report['unsatisfied_constraints']}; "
                    f"details={json.dumps(report, sort_keys=True)}"
                )
            write_json(results_dir / f"condition_d_draw_{draw_index}.json", {"draw_index": draw_index, "image_ids": ids, **report})

        write_json(
            results_dir / "condition_d_matching_report.json",
            {
                "target_profile_from_condition_C": target_profile,
                "n_draws": len(draw_specs),
                "draws": [report for _, _, report in draw_specs],
            },
        )
        print(f"Condition D: {len(draw_specs)} matched draws built", flush=True)
        for _, _, report in draw_specs:
            print(
                f"  seed {report['seed']}: n={report['achieved_total']}/{report['target_total']} "
                f"labels_within_tolerance={report['all_labels_within_tolerance']}",
                flush=True,
            )

    for condition in conditions:
        if condition == "D":
            for draw_index, ids, _ in draw_specs:
                for seed in seeds:
                    all_results.append(execute_or_resume(condition, seed, draw_index, ids))
        else:
            for seed in seeds:
                all_results.append(execute_or_resume(condition, seed))

    write_json(
        results_dir / "stage4_training_results.json",
        {
            "results": all_results,
            "run_plan": plan,
            "protocol_manifest": str(stage4_cfg.paths.protocol_manifest),
            "git_commit_hash": get_git_commit_hash(),
        },
    )
    print(f"\n{len(all_results)} runs complete -> {results_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 10 - Stage 5: protected final evaluation and analysis

`final_eval_heldout` is opened exactly once, under an auditable run id - then the statistics run locally off the pod.

## `scripts/eval/stage5_evaluate.py`

*477 lines*

**Stage 5 - the protected final evaluation. The only place `final_eval_heldout` is read for
outcomes, and it is wrapped in more gates than anything else in the project.**

`enforce_preconditions` refuses to run unless *all* of these hold: the production split manifest
exists **and** is frozen **and** passed its support check; the recorded split hash matches the
current manifest; the Stage 4 frozen protocol manifest exists; the Stage 3 frozen selection manifest
exists; the required A/B/C checkpoints exist with their hashes recorded; the classification-threshold
policy is frozen; and an explicit `--final-eval-run-id` was supplied.

That run id is what unlocks outcome-bearing access through
`splits.assert_final_eval_access_allowed`. `register_final_evaluation` writes the evaluation context
first and `mark_final_outcome_access` records the moment the data was actually opened - so the single
permitted read is itself auditable.

Then: `predict_condition` runs inference per condition/seed, `select_thresholds` applies the frozen
threshold policy (`max_youden_j`, chosen on `classifier_val` **only**),
`validate_prediction_frame` checks the exact expected image and patient ids are present, and
`publish_predictions_atomic` + `validate_completed_predictions` write and re-verify the prediction
Parquets. It computes no comparisons itself - that is the next file's job.

In [ ]:
# ===== FILE: scripts/eval/stage5_evaluate.py =====
#!/usr/bin/env python3
"""Stage 5 — protected final evaluation on final_eval_heldout (docs/stages2_to_5_plan.md §8).

Refuses to run unless every precondition in §8 holds:
  - the production split manifest exists AND is frozen AND passed its support check;
  - the recorded split hash matches the current manifest;
  - the frozen experiment-protocol manifest exists (Stage 4);
  - the frozen Learned-ASISM selection manifest exists (Stage 3);
  - the required A/B/C checkpoints exist, with their hashes recorded;
  - the classification-threshold policy is frozen;
  - an explicit --final-eval-run-id is supplied.

The run id is what unlocks outcome-bearing access to final_eval_heldout through
scripts/utils/splits.assert_final_eval_access_allowed(). No other code path can obtain it.

TECHNICAL RESUME vs. METHODOLOGICAL RE-EVALUATION (§8.4): predictions are written incrementally,
and an interrupted run may resume from the same frozen artifacts and complete the same partial
prediction files without altering the protocol. That is an engineering property. What is NOT
permitted is inspecting results and then re-running under a changed method — a genuine
methodological error requires returning to the development stage, re-freezing, and declaring a NEW
run id, which produces a separate result rather than overwriting this one.

Usage:
    python scripts/eval/stage5_evaluate.py --final-eval-run-id stage5-2026-08-07-a --namespace production
"""

from __future__ import annotations

import argparse
import json
import os
import tempfile
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.config import load_named_config, load_stage1_config  # noqa: E402
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS, build_label_arrays  # noqa: E402
from scripts.utils.manifest import get_git_commit_hash, hash_dict, read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.splits import (  # noqa: E402
    load_split,
    read_split_manifest,
    require_frozen_production_split_run,
    split_provenance,
)
from scripts.utils.artifact_contracts import current_code_identity_hash, stage3_paths, stage4_paths  # noqa: E402


def require_smoke_final_eval_authorization(namespace: str, manifest: dict, project_root: Path) -> dict:
    """Authorize fixture or real-data engineering smoke without weakening production guards.

    The historical fixture smoke keeps its existing marker.  A real-data smoke requires a
    namespace- and split-hash-bound marker, so a marker created for one development split cannot
    unlock another split (and a production-class namespace is always refused).
    """
    if manifest.get("namespace_class") != "dev":
        raise SystemExit("Smoke Stage 5 requires a dev-class split")

    if namespace == "dev-smoke-v1" and (project_root / "SMOKE_ONLY.json").is_file():
        return {"kind": "fixture", "upstream_code_identity_sha256": current_code_identity_hash()}

    marker_path = project_root / "REAL_DATA_SMOKE_ONLY.json"
    if not marker_path.is_file():
        raise SystemExit(
            "Real-data smoke Stage 5 requires REAL_DATA_SMOKE_ONLY.json in PROJECT_ROOT"
        )
    marker = read_json(marker_path)
    expected = {
        "schema_version": 1,
        "purpose": "real-data-engineering-smoke-only",
        "namespace": namespace,
        "split_manifest_hash": manifest.get("manifest_hash"),
        "not_scientific_evidence": True,
    }
    mismatches = {
        key: {"expected": value, "actual": marker.get(key)}
        for key, value in expected.items()
        if marker.get(key) != value
    }
    if mismatches:
        raise SystemExit(f"Real-data smoke authorization marker mismatch: {mismatches}")
    upstream_identity = marker.get("upstream_code_identity_sha256")
    if not isinstance(upstream_identity, str) or len(upstream_identity) != 64:
        raise SystemExit(
            "Real-data smoke authorization marker requires a 64-character "
            "upstream_code_identity_sha256"
        )
    return {"kind": "real_data", "upstream_code_identity_sha256": upstream_identity}


def enforce_preconditions(namespace: str, run_id: str) -> dict:
    """All of §8's execution guards. Any failure aborts before final_eval_heldout is opened."""
    failures: list[str] = []
    evidence: dict = {"final_eval_run_id": run_id, "namespace": namespace}

    if not run_id:
        failures.append("no --final-eval-run-id supplied")

    # 1-2. Production splits frozen, support check passed, hash recorded.
    try:
        smoke_mode = os.environ.get("THESIS_SMOKE_MODE") == "1"
        if smoke_mode:
            manifest = read_split_manifest(namespace)
            smoke_authorization = require_smoke_final_eval_authorization(
                namespace, manifest, Path(os.environ.get("PROJECT_ROOT", "."))
            )
            evidence["smoke_only_not_scientific_evidence"] = True
            evidence["smoke_kind"] = smoke_authorization["kind"]
            evidence["upstream_code_identity_sha256"] = smoke_authorization[
                "upstream_code_identity_sha256"
            ]
        else:
            manifest = require_frozen_production_split_run(namespace)
        evidence["split_manifest_hash"] = manifest.get("manifest_hash")
        evidence["split_support_check_passed"] = manifest.get("support_check", {}).get("passed")
        evidence["split_frozen"] = manifest.get("frozen")
    except SystemExit as exc:
        failures.append(f"split manifest: {exc}")

    expected_code_identity = evidence.get(
        "upstream_code_identity_sha256", current_code_identity_hash()
    )

    stage4_cfg = load_named_config("stage4_classifier.yaml", "stage4")
    stage3_cfg = load_named_config("stage3_asism.yaml", "stage3")
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    for key, value in stage3_paths(stage3_cfg, namespace).items():
        if key in stage3_cfg.paths:
            stage3_cfg.paths[key] = str(value)

    # 3. Frozen experiment protocol.
    protocol_path = Path(stage4_cfg.paths.protocol_manifest)
    if not protocol_path.is_file():
        failures.append(f"frozen experiment-protocol manifest missing: {protocol_path}")
    else:
        protocol = read_json(protocol_path)
        if not protocol.get("frozen"):
            failures.append("experiment-protocol manifest exists but is not marked frozen")
        evidence["protocol_manifest_hash"] = sha256_file(protocol_path)
        if protocol.get("split_provenance", {}).get("split_manifest_hash") != evidence.get("split_manifest_hash"):
            failures.append("experiment protocol split hash differs from the requested frozen split")
        if protocol.get("code_identity_sha256") != expected_code_identity:
            failures.append("experiment protocol code identity differs from the current source tree")
        # 6. Threshold policy frozen.
        if not protocol.get("threshold_policy"):
            failures.append("classification-threshold policy is not recorded in the frozen protocol")
        else:
            evidence["threshold_policy"] = protocol["threshold_policy"]
        frozen_component_hashes = {}
        for key in ("label_policy_manifest", "threshold_policy_manifest", "seed_plan_manifest", "d_matching_policy_manifest"):
            component_path = Path(stage4_cfg.paths[key])
            if not component_path.is_file():
                failures.append(f"frozen methodology component missing: {component_path}")
                continue
            component = read_json(component_path)
            if not component.get("frozen") or component.get("protocol_manifest_hash") != evidence["protocol_manifest_hash"]:
                failures.append(f"frozen methodology component incompatible: {component_path}")
            frozen_component_hashes[key] = sha256_file(component_path)
        evidence["frozen_component_hashes"] = frozen_component_hashes

    # 4. Learned-ASISM selection is the sole Stage-3 selector in this thesis protocol.
    asism_path = Path(stage3_cfg.paths.adaptive_selection_manifest)
    if not asism_path.is_file():
        failures.append(f"ASISM freeze manifest missing: {asism_path}")
    else:
        asism = read_json(asism_path)
        if not asism.get("frozen") or asism.get("method") != "class_aware_adaptive_threshold_v2":
            failures.append("Learned ASISM selection manifest is not a compatible frozen artifact")
        evidence["learned_asism_manifest_hash"] = sha256_file(asism_path)

    # C is the finalized Learned-ASISM selector.
    if "C" in list(stage4_cfg.get("conditions", [])):
        learned_path = Path(stage3_cfg.paths.adaptive_selection_manifest)
        if not learned_path.is_file():
            failures.append(f"adaptive learned ASISM selection manifest missing: {learned_path}")
        else:
            learned = read_json(learned_path)
            if not learned.get("frozen") or learned.get("method") != "class_aware_adaptive_threshold_v2":
                failures.append("adaptive learned ASISM selection manifest is not a compatible frozen artifact")

    # 5. Required checkpoints.
    results_path = Path(stage4_cfg.paths.results_dir) / "stage4_training_results.json"
    if not results_path.is_file():
        failures.append(f"Stage 4 training results missing: {results_path}")
    else:
        results = read_json(results_path)["results"]
        import torch
        checkpoint_hashes = {}
        missing = []
        for entry in results:
            path = Path(entry["checkpoint"])
            if path.is_file():
                checkpoint_hashes[entry["tag"]] = sha256_file(path)
                payload = torch.load(path, map_location="cpu", weights_only=False)
                provenance = payload.get("provenance", {})
                if payload.get("checkpoint_role") != "best_validation_model":
                    failures.append(f"checkpoint is not a persisted best-validation model: {entry['tag']}")
                if provenance.get("split_manifest_hash") != evidence.get("split_manifest_hash"):
                    failures.append(f"checkpoint split provenance mismatch: {entry['tag']}")
                if provenance.get("code_identity_sha256") != expected_code_identity:
                    failures.append(f"checkpoint code identity mismatch: {entry['tag']}")
            else:
                missing.append(entry["tag"])
        if missing:
            failures.append(f"{len(missing)} required checkpoint(s) missing: {missing[:5]}")
        expected = read_json(results_path).get("run_plan", {}).get("total_runs")
        if expected and len(results) != expected:
            failures.append(f"expected {expected} runs, found {len(results)}")
        evidence["checkpoint_hashes"] = checkpoint_hashes
        evidence["n_checkpoints"] = len(checkpoint_hashes)

    if failures:
        raise SystemExit(
            "STAGE 5 REFUSED — preconditions not met (docs/stages2_to_5_plan.md §8):\n"
            + "\n".join(f"  - {failure}" for failure in failures)
            + "\n\nfinal_eval_heldout was NOT opened."
        )
    return evidence


def predict_condition(entry: dict, records: list[dict], stage4_cfg) -> np.ndarray:
    from scripts.utils.classifier import build_model, predict_probabilities, require_torch

    torch = require_torch()
    payload = torch.load(entry["checkpoint"], map_location="cpu", weights_only=False)
    model = build_model(
        len(CLASSIFIER_TARGET_LABELS),
        float(stage4_cfg.model.dropout_p),
        str(stage4_cfg.model.pretrained_source),
    )
    model.load_state_dict(payload["model_state"])
    probabilities, _ = predict_probabilities(
        model, records, int(stage4_cfg.model.resolution)
    )
    return probabilities


def select_thresholds(entry: dict, namespace: str, stage4_cfg) -> dict[str, float]:
    """Thresholds chosen on classifier_val ONLY (frozen policy), never on final_eval_heldout."""
    from scripts.utils.classifier import records_from_split
    from scripts.utils.metrics import sensitivity_specificity_f1

    stage1_cfg = load_stage1_config()
    val_frame = load_split("classifier_val", namespace, purpose="schema_validation", caller="stage5")
    val_records = records_from_split(
        val_frame, Path(stage1_cfg.paths.images_dir) / namespace / "classifier_val"
    )
    probabilities = predict_condition(entry, val_records, stage4_cfg)

    frame = pd.DataFrame([record["labels"] for record in val_records])
    for label in CLASSIFIER_TARGET_LABELS:
        if label not in frame.columns:
            frame[label] = np.nan
    _, targets, masks = build_label_arrays(frame[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)

    thresholds = {}
    for index, label in enumerate(CLASSIFIER_TARGET_LABELS):
        mask = masks[:, index].astype(bool)
        if mask.sum() == 0:
            thresholds[label] = 0.5
            continue
        y_true = targets[mask, index].astype(int)
        y_score = probabilities[mask, index]
        best_threshold, best_j = 0.5, -np.inf
        for candidate in np.linspace(0.05, 0.95, 19):
            sensitivity, specificity, _ = sensitivity_specificity_f1(y_score, y_true, candidate)
            if np.isnan(sensitivity) or np.isnan(specificity):
                continue
            youden_j = sensitivity + specificity - 1
            if youden_j > best_j:
                best_j, best_threshold = youden_j, float(candidate)
        thresholds[label] = best_threshold
    return thresholds


def validate_prediction_frame(frame: pd.DataFrame, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    required = ["image_id", "patient_id", *CLASSIFIER_TARGET_LABELS]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"prediction schema missing columns: {missing}")
    if len(frame) != len(expected_image_ids):
        raise ValueError(f"wrong prediction row count: {len(frame)} != {len(expected_image_ids)}")
    if frame["image_id"].duplicated().any():
        raise ValueError("duplicate prediction image IDs")
    if set(frame["image_id"].astype(str)) != set(map(str, expected_image_ids)):
        raise ValueError("prediction image-ID set mismatch")
    if set(frame["patient_id"].astype(str)) != set(map(str, expected_patient_ids)):
        raise ValueError("prediction patient-ID set mismatch")


def publish_predictions_atomic(path: Path, frame: pd.DataFrame, provenance: dict, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp.parquet", dir=path.parent)
    os.close(fd)
    temporary = Path(temp_name)
    try:
        frame.to_parquet(temporary, index=False)
        validate_prediction_frame(pd.read_parquet(temporary), expected_image_ids, expected_patient_ids)
        os.replace(temporary, path)
        sidecar = {"schema_version": 1, "status": "complete", "prediction_sha256": sha256_file(path),
                   "row_count": len(frame), "image_id_set_hash": hash_dict({"ids": sorted(map(str, expected_image_ids))}, length=64),
                   "patient_id_set_hash": hash_dict({"ids": sorted(set(map(str, expected_patient_ids)))}, length=64), **provenance}
        write_json(path.with_suffix(".complete.json"), sidecar)
    finally:
        temporary.unlink(missing_ok=True)


def validate_completed_predictions(path: Path, expected_provenance: dict, expected_image_ids: list[str], expected_patient_ids: list[str]) -> None:
    sidecar_path = path.with_suffix(".complete.json")
    if not path.is_file() or not sidecar_path.is_file():
        raise ValueError("prediction file and completion sidecar are both required")
    sidecar = read_json(sidecar_path)
    if sidecar.get("prediction_sha256") != sha256_file(path):
        raise ValueError("prediction file hash mismatch/corruption")
    mismatch = {k: (sidecar.get(k), v) for k, v in expected_provenance.items() if sidecar.get(k) != v}
    if mismatch:
        raise ValueError(f"prediction provenance mismatch: {mismatch}")
    validate_prediction_frame(pd.read_parquet(path), expected_image_ids, expected_patient_ids)


def register_final_evaluation(context_path: Path, registration: dict) -> dict:
    """Register a new run or validate an identical technical resume."""
    identity_keys = (
        "final_eval_run_id", "namespace", "split_manifest_hash", "asism_manifest_hash",
        "protocol_manifest_hash", "checkpoint_hashes", "threshold_policy_hash",
        "code_git_version", "code_identity_sha256",
    )
    if context_path.exists():
        existing = read_json(context_path)
        for key in identity_keys:
            if existing.get(key) != registration.get(key):
                raise SystemExit(f"Registered final-evaluation context is incompatible at {key}; technical resume refused")
        return existing
    for prior_path in context_path.parent.parent.glob("*/final_eval_registration.json"):
        prior = read_json(prior_path)
        if prior.get("split_manifest_hash") == registration["split_manifest_hash"] and prior.get("status") in {"outcome_accessed", "complete"}:
            raise SystemExit("Final outcomes were already accessed under another run ID; a corrected run requires a new linked protocol freeze")
    write_json(context_path, registration)
    return registration


def mark_final_outcome_access(context_path: Path) -> dict:
    registration = read_json(context_path)
    registration["status"] = "outcome_accessed"
    registration["first_outcome_access_at_utc"] = registration.get("first_outcome_access_at_utc") or datetime.now(timezone.utc).isoformat()
    write_json(context_path, registration)
    return registration


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--final-eval-run-id", required=True)
    parser.add_argument("--namespace", default="production")
    parser.add_argument("--bootstrap-resamples", type=int, default=1000)
    args = parser.parse_args()

    run_id = args.final_eval_run_id
    namespace = args.namespace

    print("Stage 5 precondition checks...", flush=True)
    evidence = enforce_preconditions(namespace, run_id)
    print("  all preconditions satisfied", flush=True)

    stage4_cfg = load_named_config("stage4_classifier.yaml", "stage4")
    for key, value in stage4_paths(stage4_cfg, namespace).items():
        stage4_cfg.paths[key] = str(value)
    stage1_cfg = load_stage1_config()

    output_dir = Path(stage4_cfg.paths.project_root) / "outputs" / "stage5" / namespace / run_id
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions_dir = output_dir / "predictions"
    predictions_dir.mkdir(exist_ok=True)

    results = read_json(Path(stage4_cfg.paths.results_dir) / "stage4_training_results.json")["results"]
    # Freeze concrete classifier_val-selected values before any outcome-bearing final-data access.
    thresholds_path = output_dir / "frozen_threshold_values.json"
    if not thresholds_path.exists():
        threshold_values = {entry["tag"]: select_thresholds(entry, namespace, stage4_cfg) for entry in results}
        write_json(thresholds_path, {"frozen": True, "selected_on": "classifier_val", "values": threshold_values,
                                    "policy": dict(stage4_cfg.threshold_policy)})
    threshold_hash = sha256_file(thresholds_path)

    context_path = output_dir / "final_eval_registration.json"
    registration = {
        "final_eval_run_id": run_id, "namespace": namespace, "split_manifest_hash": evidence["split_manifest_hash"],
        "asism_manifest_hash": evidence["learned_asism_manifest_hash"], "protocol_manifest_hash": evidence["protocol_manifest_hash"],
        "checkpoint_hashes": evidence["checkpoint_hashes"], "threshold_policy_hash": threshold_hash,
        "code_git_version": get_git_commit_hash(), "code_identity_sha256": current_code_identity_hash(),
        "registered_at_utc": datetime.now(timezone.utc).isoformat(), "status": "registered",
    }
    registration = register_final_evaluation(context_path, registration)

    # This is the ONLY outcome-bearing read of final_eval_heldout in the codebase, and it is
    # unlocked solely by the explicit run id (§1.6).
    from scripts.utils.classifier import records_from_split

    eval_frame = load_split(
        "final_eval_heldout",
        namespace,
        purpose="load_labels_for_evaluation",
        final_eval_run_id=run_id,
        final_eval_context_path=context_path,
        caller="stage5_evaluate",
    )
    registration = mark_final_outcome_access(context_path)
    eval_records = records_from_split(
        eval_frame, Path(stage1_cfg.paths.images_dir) / namespace / "final_eval_heldout"
    )
    if not eval_records:
        raise SystemExit(
            "UPSTREAM GATE: no preprocessed final_eval_heldout images found. "
            "Stage 1 preprocessing must have run for this split."
        )
    print(f"  final_eval_heldout: {len(eval_records)} images, "
          f"{eval_frame['patient_id'].nunique()} patients", flush=True)

    # Technical resume: per-condition prediction files are written incrementally and reused.
    for entry in results:
        prediction_path = predictions_dir / f"{entry['tag']}.parquet"
        prediction_provenance = {
            "checkpoint_hash": evidence["checkpoint_hashes"][entry["tag"]],
            "split_hash": evidence["split_manifest_hash"], "protocol_hash": evidence["protocol_manifest_hash"],
            "threshold_hash": threshold_hash, "code_version": get_git_commit_hash(),
        }
        if prediction_path.is_file():
            try:
                validate_completed_predictions(prediction_path, prediction_provenance,
                                               [r["image_id"] for r in eval_records], [r["patient_id"] for r in eval_records])
            except Exception as exc:
                raise SystemExit(f"Existing predictions for {entry['tag']} are partial/corrupt/incompatible; refusing reuse: {exc}") from exc
            print(f"  [{entry['tag']}] safely resumed from validated predictions", flush=True)
            continue
        print(f"  [{entry['tag']}] predicting...", flush=True)
        probabilities = predict_condition(entry, eval_records, stage4_cfg)
        frame = pd.DataFrame(probabilities, columns=CLASSIFIER_TARGET_LABELS)
        frame.insert(0, "image_id", [record["image_id"] for record in eval_records])
        frame.insert(1, "patient_id", [record["patient_id"] for record in eval_records])
        publish_predictions_atomic(prediction_path, frame, prediction_provenance,
                                   [r["image_id"] for r in eval_records], [r["patient_id"] for r in eval_records])

    write_json(
        output_dir / "final_eval_run_manifest.json",
        {
            "final_eval_run_id": run_id,
            "run_at_utc": datetime.now(timezone.utc).isoformat(),
            "namespace": namespace,
            "preconditions": evidence,
            "n_eval_images": len(eval_records),
            "n_eval_patients": int(eval_frame["patient_id"].nunique()),
            "predictions_dir": str(predictions_dir),
            "split_provenance": split_provenance(namespace),
            "git_commit_hash": get_git_commit_hash(),
            "protocol_note": (
                "One frozen final-evaluation protocol. Technical resume is permitted; "
                "methodological re-evaluation is not. A genuine methodological error requires "
                "re-freezing and a NEW run id, reported alongside this one."
            ),
        },
    )
    registration["status"] = "complete"
    registration["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(context_path, registration)

    print(f"\nPredictions written -> {predictions_dir}", flush=True)
    print("Next (no GPU required):", flush=True)
    print(f"  python scripts/eval/compare_conditions.py --run-dir {output_dir}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/eval/compare_conditions.py`

*349 lines*

**Stage 5 analysis - the statistics and result tables.** Reads *only* the prediction Parquet files
written by `stage5_evaluate.py`. No torch, no GPU, so figures and interpretation can be iterated
locally, off the pod.

Implements the frozen statistical policy:

- patient-level **paired bootstrap** for effect sizes and 95% CIs;
- **Holm-Bonferroni** for the CONFIRMATORY family (primary endpoint, primary comparison);
- **Benjamini-Hochberg** FDR for EXPLORATORY analyses, labelled as such everywhere;
- an effect size reported alongside every p-value.

`parse_tag` decodes run tags (`C_seed43`, `D_draw2_seed44`), and condition D - if it was ever run -
is summarised **across** its draws (mean and spread), never as a single cherry-picked draw.

In [ ]:
# ===== FILE: scripts/eval/compare_conditions.py =====
#!/usr/bin/env python3
"""Stage 5 analysis — condition comparison, statistics, and result tables (§8).

Reads only the prediction parquet files written by stage5_evaluate.py. No torch, no GPU: the whole
analysis layer runs locally, which is what keeps interpretation and figure iteration off the pod.

Implements the frozen statistical policy (§8.2):
  - patient-level paired bootstrap for effect sizes and 95% CIs;
  - Holm-Bonferroni for the CONFIRMATORY family (primary endpoint / primary comparison);
  - Benjamini-Hochberg FDR for EXPLORATORY analyses, labelled as such everywhere;
  - effect sizes reported alongside every p-value.

Condition D is summarised ACROSS its five draws (mean and spread), never as a single draw.

Usage:
    python scripts/eval/compare_conditions.py --run-dir outputs/stage5/<run_id>
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.labels import (  # noqa: E402
    CLASSIFIER_TARGET_LABELS,
    PRIMARY_ENDPOINT_LABELS,
    build_label_arrays,
)
from scripts.utils.manifest import read_json, sha256_file, write_json  # noqa: E402
from scripts.utils.metrics import (  # noqa: E402
    auroc,
    benjamini_hochberg,
    full_metric_suite,
    holm_bonferroni,
    paired_bootstrap_difference,
    patient_level_bootstrap,
)

TAG_PATTERN = re.compile(r"^(?P<condition>[A-D])(?:_draw(?P<draw>\d+))?_seed(?P<seed>\d+)$")

# The CONFIRMATORY family, fixed before results are seen (§8.1/§8.2). Everything else is
# exploratory and corrected with FDR.
CONFIRMATORY_COMPARISONS = [("C", "B")]
SECONDARY_COMPARISONS = [("A", "B"), ("A", "C")]


def parse_tag(tag: str) -> dict | None:
    match = TAG_PATTERN.match(tag)
    if not match:
        return None
    return {
        "condition": match.group("condition"),
        "draw": int(match.group("draw")) if match.group("draw") is not None else None,
        "seed": int(match.group("seed")),
    }


def load_predictions(run_dir: Path) -> tuple[dict[str, list[pd.DataFrame]], pd.DataFrame]:
    predictions_dir = run_dir / "predictions"
    if not predictions_dir.is_dir():
        raise SystemExit(f"No predictions directory at {predictions_dir}")

    by_condition: dict[str, list[pd.DataFrame]] = defaultdict(list)
    reference = None
    for path in sorted(predictions_dir.glob("*.parquet")):
        info = parse_tag(path.stem)
        if info is None:
            continue
        sidecar_path = path.with_suffix(".complete.json")
        if not sidecar_path.is_file():
            raise SystemExit(f"Prediction completion sidecar missing: {sidecar_path}")
        sidecar = read_json(sidecar_path)
        if sidecar.get("prediction_sha256") != sha256_file(path):
            raise SystemExit(f"Prediction hash mismatch: {path}")
        frame = pd.read_parquet(path)
        frame.attrs.update(info)
        by_condition[info["condition"]].append(frame)
        reference = frame if reference is None else reference

    if reference is None:
        raise SystemExit(f"No prediction files found in {predictions_dir}")
    return by_condition, reference


def build_truth(run_dir: Path, reference: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Ground-truth arrays for the evaluation rows, aligned to the prediction row order."""
    manifest = read_json(run_dir / "final_eval_run_manifest.json")
    namespace = manifest["namespace"]

    # Non-outcome access: we already hold the outcome-bearing predictions; this reloads the labels
    # for metric computation inside the same declared run.
    from scripts.utils.splits import load_split

    frame = load_split(
        "final_eval_heldout",
        namespace,
        purpose="load_labels_for_evaluation",
        final_eval_run_id=manifest["final_eval_run_id"],
        final_eval_context_path=run_dir / "final_eval_registration.json",
        caller="compare_conditions",
    )
    from scripts.utils.identifiers import sanitize_image_id

    frame["image_id"] = frame["Path"].apply(sanitize_image_id)
    aligned = reference[["image_id"]].merge(frame, on="image_id", how="left")

    for label in CLASSIFIER_TARGET_LABELS:
        if label not in aligned.columns:
            aligned[label] = np.nan
    raw, targets, masks = build_label_arrays(aligned[CLASSIFIER_TARGET_LABELS], CLASSIFIER_TARGET_LABELS)
    return raw, targets, masks


def macro_auroc_for(probabilities: np.ndarray, targets: np.ndarray, masks: np.ndarray,
                    rows: np.ndarray) -> float:
    """Masked macro-AUROC over the 11 primary labels for a subset of rows."""
    values = []
    for label in PRIMARY_ENDPOINT_LABELS:
        index = CLASSIFIER_TARGET_LABELS.index(label)
        mask = masks[rows, index].astype(bool)
        if mask.sum() == 0:
            continue
        y_true = targets[rows][mask, index].astype(int)
        y_score = probabilities[rows][mask, index]
        if (y_true == 1).sum() == 0 or (y_true == 0).sum() == 0:
            continue
        value = auroc(y_score, y_true)
        if not np.isnan(value):
            values.append(value)
    return float(np.mean(values)) if values else float("nan")


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--bootstrap-resamples", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    run_dir = Path(args.run_dir)
    by_condition, reference = load_predictions(run_dir)
    _, targets, masks = build_truth(run_dir, reference)
    patient_ids = reference["patient_id"].to_numpy()

    label_columns = CLASSIFIER_TARGET_LABELS

    # Per-condition mean probabilities across seeds (and, for D, across draws AND seeds).
    condition_probabilities: dict[str, np.ndarray] = {}
    condition_spread: dict[str, dict] = {}
    for condition, frames in by_condition.items():
        stacked = np.stack([frame[label_columns].to_numpy() for frame in frames], axis=0)
        condition_probabilities[condition] = stacked.mean(axis=0)

        per_run_macro = []
        for frame in frames:
            value = macro_auroc_for(
                frame[label_columns].to_numpy(), targets, masks, np.arange(len(frame))
            )
            per_run_macro.append(
                {"draw": frame.attrs.get("draw"), "seed": frame.attrs.get("seed"),
                 "macro_auroc": value}
            )
        values = [entry["macro_auroc"] for entry in per_run_macro if not np.isnan(entry["macro_auroc"])]
        condition_spread[condition] = {
            "n_runs": len(frames),
            "per_run": per_run_macro,
            "mean_macro_auroc": float(np.mean(values)) if values else float("nan"),
            "std_macro_auroc": float(np.std(values, ddof=1)) if len(values) > 1 else float("nan"),
            "min_macro_auroc": float(np.min(values)) if values else float("nan"),
            "max_macro_auroc": float(np.max(values)) if values else float("nan"),
        }

    # Full metric suite per condition.
    per_condition_metrics = {}
    for condition, probabilities in condition_probabilities.items():
        thresholds_path = next(
            (run_dir / "predictions").glob(f"{condition}*seed*.thresholds.json"), None
        )
        thresholds = read_json(thresholds_path)["thresholds"] if thresholds_path else {}
        suite = full_metric_suite(probabilities, targets, masks, thresholds, label_columns)

        bootstrap = patient_level_bootstrap(
            patient_ids,
            lambda rows, p=probabilities: macro_auroc_for(p, targets, masks, rows),
            n_resamples=args.bootstrap_resamples,
            seed=args.seed,
        )
        suite["macro_auroc_bootstrap"] = bootstrap
        suite["across_run_spread"] = condition_spread[condition]
        per_condition_metrics[condition] = suite

    # Paired comparisons.
    def compare(left: str, right: str) -> dict | None:
        if left not in condition_probabilities or right not in condition_probabilities:
            return None
        return paired_bootstrap_difference(
            patient_ids,
            lambda rows: macro_auroc_for(condition_probabilities[left], targets, masks, rows),
            lambda rows: macro_auroc_for(condition_probabilities[right], targets, masks, rows),
            n_resamples=args.bootstrap_resamples,
            seed=args.seed,
        )

    confirmatory, exploratory = {}, {}
    for left, right in CONFIRMATORY_COMPARISONS:
        result = compare(left, right)
        if result:
            confirmatory[f"{left}_vs_{right}"] = result
    for left, right in SECONDARY_COMPARISONS:
        result = compare(left, right)
        if result:
            exploratory[f"{left}_vs_{right}"] = result

    # Per-label exploratory comparisons for the primary comparison pair.
    per_label_exploratory = {}
    if "C" in condition_probabilities and "D" in condition_probabilities:
        for label in PRIMARY_ENDPOINT_LABELS:
            index = label_columns.index(label)

            def label_auroc(rows, probabilities, idx=index):
                mask = masks[rows, idx].astype(bool)
                if mask.sum() == 0:
                    return float("nan")
                y_true = targets[rows][mask, idx].astype(int)
                if (y_true == 1).sum() == 0 or (y_true == 0).sum() == 0:
                    return float("nan")
                return auroc(probabilities[rows][mask, idx], y_true)

            result = paired_bootstrap_difference(
                patient_ids,
                lambda rows: label_auroc(rows, condition_probabilities["C"]),
                lambda rows: label_auroc(rows, condition_probabilities["D"]),
                n_resamples=max(200, args.bootstrap_resamples // 5),
                seed=args.seed,
            )
            per_label_exploratory[f"C_vs_D::{label}"] = result

    confirmatory_corrected = holm_bonferroni(
        {name: result["p_value"] for name, result in confirmatory.items()}
    )
    exploratory_corrected = benjamini_hochberg(
        {
            **{name: result["p_value"] for name, result in exploratory.items()},
            **{name: result["p_value"] for name, result in per_label_exploratory.items()},
        }
    )

    report = {
        "run_dir": str(run_dir),
        "primary_endpoint": f"macro-AUROC over the {len(PRIMARY_ENDPOINT_LABELS)} primary disease labels",
        "primary_comparison": "C vs B (ASISM-selected synthetic images vs all synthetic images)",
        "n_eval_rows": len(reference),
        "n_eval_patients": int(pd.Series(patient_ids).nunique()),
        "per_condition": per_condition_metrics,
        "confirmatory": {
            name: {**result, "correction": confirmatory_corrected.get(name, {})}
            for name, result in confirmatory.items()
        },
        "exploratory": {
            name: {**result, "correction": exploratory_corrected.get(name, {})}
            for name, result in {**exploratory, **per_label_exploratory}.items()
        },
        "statistical_policy": {
            "resampling_unit": "patient",
            "confirmatory_correction": "holm_bonferroni",
            "exploratory_correction": "benjamini_hochberg_fdr",
            "effect_sizes_reported": True,
            "policy_frozen_before_results": True,
        },
    }
    write_json(run_dir / "stage5_comparison_report.json", report)

    # Human-readable tables.
    rows = []
    for condition in sorted(per_condition_metrics):
        suite = per_condition_metrics[condition]
        bootstrap = suite["macro_auroc_bootstrap"]
        spread = suite["across_run_spread"]
        rows.append(
            {
                "condition": condition,
                "macro_auroc": round(suite["macro_auroc_primary"], 4),
                "ci_lower": round(bootstrap["ci_lower"], 4),
                "ci_upper": round(bootstrap["ci_upper"], 4),
                "across_run_mean": round(spread["mean_macro_auroc"], 4),
                "across_run_std": (
                    round(spread["std_macro_auroc"], 4)
                    if not np.isnan(spread["std_macro_auroc"]) else None
                ),
                "n_runs": spread["n_runs"],
                "macro_auprc": round(suite["macro_auprc_primary"], 4),
                "micro_auroc": round(suite["micro_auroc_primary"], 4),
                "mean_brier": round(suite["mean_brier_primary"], 4),
                "mean_ece": round(suite["mean_ece_primary"], 4),
            }
        )
    condition_table = pd.DataFrame(rows)
    condition_table.to_csv(run_dir / "table_conditions.csv", index=False)

    comparison_rows = []
    for family, source, corrected in (
        ("confirmatory", confirmatory, confirmatory_corrected),
        ("exploratory", {**exploratory, **per_label_exploratory}, exploratory_corrected),
    ):
        for name, result in source.items():
            correction = corrected.get(name, {})
            comparison_rows.append(
                {
                    "comparison": name,
                    "family": family,
                    "effect_size": round(result["effect_size"], 4),
                    "ci_lower": round(result["ci_lower"], 4),
                    "ci_upper": round(result["ci_upper"], 4),
                    "p_value": round(result["p_value"], 5),
                    "adjusted_p": (
                        round(correction.get("adjusted_p_value"), 5)
                        if correction.get("adjusted_p_value") is not None
                        and not np.isnan(correction.get("adjusted_p_value", np.nan))
                        else None
                    ),
                    "significant": correction.get("rejected"),
                    "correction": correction.get("method"),
                }
            )
    comparison_table = pd.DataFrame(comparison_rows)
    comparison_table.to_csv(run_dir / "table_comparisons.csv", index=False)

    print(
        f"\nCondition summary (primary endpoint: macro-AUROC over {len(PRIMARY_ENDPOINT_LABELS)} disease labels)",
        flush=True,
    )
    print(condition_table.to_string(index=False), flush=True)
    print("\nComparisons", flush=True)
    print(comparison_table.to_string(index=False), flush=True)
    print(f"\nReport -> {run_dir / 'stage5_comparison_report.json'}", flush=True)
    print(f"Tables -> {run_dir / 'table_conditions.csv'}, {run_dir / 'table_comparisons.csv'}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 11 - Smoke pipelines

How the whole chain gets exercised without CheXpert and without a GPU. These validate **engineering connectivity only**; their outputs are permanently marked non-scientific.

## `scripts/smoke/00_build_fixture.py`

*48 lines*

**Creates the deterministic, synthetic, CheXpert-shaped input used only by the `dev-smoke-v1`
namespace** - a handful of generated grayscale images plus a matching `train.csv`/`valid.csv`.

No real patient data is involved, and everything it produces is permanently marked non-scientific.
This is what lets the whole Stage 1->5 chain be exercised on a laptop.

In [ ]:
# ===== FILE: scripts/smoke/00_build_fixture.py =====
#!/usr/bin/env python3
"""Create the deterministic synthetic CheXpert-shaped input used only by dev-smoke-v1."""
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS


def main() -> int:
    root = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()
    if root.name != "dev-smoke-v1":
        raise SystemExit(f"Smoke fixture PROJECT_ROOT must end in dev-smoke-v1, got {root}")
    raw = root / "data" / "chexpert" / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    rows = []
    for index in range(60):
        patient = f"patient{index:05d}"
        relative = Path("CheXpert-v1.0-small") / "train" / patient / "study1" / "view1_frontal.jpg"
        # Runtime resolvers strip the CheXpert-v1.0-small prefix before joining raw_dir.
        destination = raw / Path(*relative.parts[1:])
        destination.parent.mkdir(parents=True, exist_ok=True)
        yy, xx = np.mgrid[:256, :256]
        array = np.clip(25 + (xx + yy) * 0.25 + (index % 7) * 4, 0, 255).astype(np.uint8)
        Image.fromarray(array, mode="L").save(destination, quality=92)
        row = {"Path": relative.as_posix(), "Sex": "Male" if index % 2 else "Female",
               "Age": 35 + index % 45, "Frontal/Lateral": "Frontal", "AP/PA": "PA"}
        for label_index, label in enumerate(CLASSIFIER_TARGET_LABELS):
            row[label] = int((index + label_index) % 3 == 0)
        row["No Finding"] = int(not any(row[label] for label in CLASSIFIER_TARGET_LABELS[1:13]))
        rows.append(row)
    pd.DataFrame(rows).to_csv(raw / "train.csv", index=False)
    pd.DataFrame(rows[:4]).to_csv(raw / "valid.csv", index=False)
    marker = root / "SMOKE_ONLY.json"
    marker.write_text('{"namespace":"dev-smoke-v1","scientific_evidence":false}\n', encoding="utf-8")
    print(f"Smoke fixture ready: {root} (60 patients; never production evidence)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/smoke/run_smoke_pipeline.py`

*198 lines*

**The resumable Stage 1-5 fixture pipeline.** `--phase local` runs everything that needs no GPU;
the GPU phase adds real (tiny) SDXL training and generation.

`run_if_missing` makes each step skip when its output already exists, so the smoke run is resumable.
`require` / `require_successful_preprocessing` assert each step's outputs before the next step
starts, so a failure is reported at the step that caused it rather than three steps later.

This validates **engineering connectivity only** - that the artifacts, gates, hashes and manifests
all line up. It says nothing about scientific results, and its outputs are marked accordingly.

In [ ]:
# ===== FILE: scripts/smoke/run_smoke_pipeline.py =====
#!/usr/bin/env python3
"""Resumable Stage 1-5 fixture pipeline; outputs are permanently marked non-scientific."""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[2]
NAMESPACE = "dev-smoke-v1"


def require(path: Path, label: str) -> None:
    if not path.exists():
        raise SystemExit(f"SMOKE GATE FAILED: missing {label}: {path}")
    print(f"  validated {label}: {path}", flush=True)


def require_successful_preprocessing(root: Path, split: str) -> None:
    log = root / f"{split}_preprocessing_log.jsonl"
    require(log, f"{split} preprocessing log")
    rows = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()]
    failed = [row for row in rows if row.get("action") == "failed"]
    kept = [row for row in rows if row.get("kept")]
    if failed or not kept:
        raise SystemExit(f"SMOKE GATE FAILED: {split} has failed={len(failed)}, kept={len(kept)}")


def run(command: list[str], env: dict[str, str]) -> None:
    print("\n+ " + subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=REPO, env=env, check=True)


def run_if_missing(command: list[str], output_path: Path, env: dict[str, str]) -> None:
    """Skip a step whose output already exists, rather than reruning it into a FileExistsError.

    Several learned-ASISM scripts (04 build, 06, 09) write their single output file with mode "x"
    (exclusive create) — a deliberate guard against silently overwriting a real result. That is
    correct for a hand-invoked production run, but an ORCHESTRATOR that may be resumed after a
    partial failure must check first, exactly like this file already does for Stage 1's LoRA
    checkpoint above. Without this, resuming this pipeline after any later step fails throws away
    all of this step's (possibly expensive) work for no reason.
    """
    if output_path.exists():
        print(f"  skip (already exists): {subprocess.list2cmdline(command)}", flush=True)
        return
    run(command, env)


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--phase", choices=["local", "runpod", "all"], default="all")
    parser.add_argument("--workspace", default=str(REPO / "outputs" / "smoke" / NAMESPACE))
    parser.add_argument("--approve-smoke-pilot", action="store_true",
                        help="Explicitly approve fixture pilot after visual review; never approves production")
    args = parser.parse_args()
    workspace = Path(args.workspace).resolve()
    if workspace.name != NAMESPACE or "production" in str(workspace).lower():
        raise SystemExit("Smoke workspace must end in dev-smoke-v1 and must not contain 'production'")
    workspace.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PROJECT_ROOT": str(workspace), "THESIS_CONFIG_OVERLAY": str(REPO / "configs" / "smoke_e2e.yaml"),
           "THESIS_SMOKE_MODE": "1", "PYTHONUNBUFFERED": "1"}
    py = sys.executable

    if args.phase in {"local", "all"}:
        run([py, "scripts/smoke/00_build_fixture.py"], env)
        split_manifest = workspace / "data/chexpert/processed/splits" / NAMESPACE / "split_manifest_v2.json"
        if not split_manifest.is_file():
            run([py, "scripts/data/02b_build_sixway_splits.py", "--namespace", "dev", "--run-id", NAMESPACE], env)
        require(split_manifest, "six-way split manifest")
        run([py, "scripts/data/03_preprocess_images.py", "--namespace", NAMESPACE], env)
        for split in ("gen_train", "gen_val", "classifier_train", "classifier_val", "asism_tuning_heldout", "final_eval_heldout"):
            require(workspace / "data/chexpert/processed/images_768" / NAMESPACE / f"{split}_preprocessing_manifest.json", f"{split} preprocessing manifest")
            require_successful_preprocessing(workspace / "data/chexpert/processed/images_768" / NAMESPACE, split)
        run([py, "scripts/data/04_generate_captions.py", "--namespace", NAMESPACE, "--splits", "gen_train", "gen_val"], env)
        for split in ("gen_train", "gen_val"):
            caption_path = workspace / "data/chexpert/processed/captions" / NAMESPACE / f"{split}_captions.jsonl"
            require(caption_path, f"{split} captions")
            if not caption_path.read_text(encoding="utf-8").strip():
                raise SystemExit(f"SMOKE GATE FAILED: empty captions at {caption_path}")
        print("\nLOCAL SMOKE PHASE COMPLETE. No thesis data was accessed.", flush=True)
        if args.phase == "local":
            return 0

    require(workspace / "SMOKE_ONLY.json", "fixture-only marker")
    split_manifest = workspace / "data/chexpert/processed/splits" / NAMESPACE / "split_manifest_v2.json"
    require(split_manifest, "split manifest")

    stage1_run = workspace / "checkpoints/stage1_lora_sdxl/smoke-stage1-v1"
    final_lora = stage1_run / "final"
    if not (final_lora / "metadata.json").is_file():
        command = ["accelerate", "launch", "--config_file", "configs/accelerate_config.yaml",
                   "scripts/train/train_lora_sdxl.py", "--run-id", "smoke-stage1-v1"]
        latest = stage1_run / "latest.json"
        if latest.is_file():
            checkpoint_dir = json.loads(latest.read_text(encoding="utf-8"))["checkpoint_dir"]
            command += ["--resume-from", checkpoint_dir]
        run(command, env)
    require(final_lora / "metadata.json", "Stage 1 final LoRA metadata")
    env["SMOKE_LORA_DIR"] = str(final_lora)

    # Up to 400 recipes (the smoke quota yields ~270): the learned-ASISM feasibility gate below needs
    # several candidates per label in BOTH the train and the 20% val image pools; 40 left most labels
    # with 0-1 val candidates (measured on the A10 GPU smoke). Pilot review itself still only
    # samples stage2.pilot.num_images (4) regardless of this limit.
    run([py, "scripts/generate/01_sample_label_recipes.py", "--namespace", NAMESPACE, "--limit", "400"], env)
    synthetic = workspace / "data/chexpert/synthetic" / NAMESPACE
    require(synthetic / "recipes_manifest.json", "Stage 2 recipe manifest")
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--mode", "pilot", "--namespace", NAMESPACE], env)
    require(synthetic / "pilot/pilot_approval_manifest.json", "pilot checks")
    if not args.approve_smoke_pilot:
        print("\nSTOP: inspect the four fixture pilot images, then rerun this same command with --approve-smoke-pilot.", flush=True)
        return 3
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--approve-pilot", "--reviewer", "smoke-fixture-review",
         "--notes", "Fixture-only pipeline validation", "--namespace", NAMESPACE], env)
    run([py, "scripts/generate/02_generate_synthetic_images.py", "--mode", "full", "--namespace", NAMESPACE], env)
    require(synthetic / "generation_complete.json", "Stage 2 completion")

    run([py, "scripts/classify/00_train_auxiliary_classifier.py", "--namespace", NAMESPACE, "--max-steps", "2", "--batch-size", "2"], env)
    require(workspace / "checkpoints/auxiliary_classifier" / NAMESPACE / "auxiliary_classifier.best.pt", "auxiliary best checkpoint")
    run([py, "scripts/asism/01_compute_signals.py", "--signal", "all", "--namespace", NAMESPACE], env)
    for signal in ("similarity", "iqa", "uncertainty", "explainability", "agreement"):
        require(synthetic / "scores" / f"{signal}_scores.provenance.json", f"{signal} provenance")
    run([py, "scripts/asism/02_gonogo.py", "--namespace", NAMESPACE], env)
    require(synthetic / "asism/gonogo_report.json", "Go/No-Go report")

    # Learned ASISM (docs/stages2_to_5_plan.md §4.9) on the REAL signals just computed above — the
    # thesis's novel contribution, and the ONE part of this file never previously exercised
    # end-to-end (the CPU-only tier in 01b_learned_asism_cpu_smoke.py covers 04->05->06 on a
    # FABRICATED pool; this covers the full 04-09 chain, including real (tiny) proxy verification,
    # on REAL generated images). UNVERIFIED BY EXECUTION as of 2026-08-21 — no GPU was available to
    # run this locally. If `--phase feasibility` below fails, that is expected on a first attempt:
    # fix subset_design/feasibility_thresholds in configs/smoke_e2e.yaml (stage3.learned_asism) and
    # rerun feasibility ONLY — it needs no GPU and does not repeat Stage 1/2/aux/signals.
    learned = synthetic / "asism/learned"
    run([py, "scripts/asism/04_build_utility_subsets.py", "--phase", "feasibility", "--namespace", NAMESPACE], env)
    require(learned / "subset_design_report.json", "subset design feasibility report")
    run_if_missing(
        [py, "scripts/asism/04_build_utility_subsets.py", "--phase", "build", "--namespace", NAMESPACE],
        learned / "utility_subsets.jsonl", env,
    )
    require(learned / "utility_subsets.jsonl", "utility subsets")
    run([py, "scripts/asism/04b_evaluate_utility_subsets.py", "--phase", "estimate", "--namespace", NAMESPACE], env)
    run_if_missing(
        [py, "scripts/asism/04b_evaluate_utility_subsets.py", "--phase", "run", "--namespace", NAMESPACE],
        learned / "utility_results.jsonl", env,
    )
    require(learned / "utility_results.jsonl", "measured utility results")
    run([py, "scripts/asism/05_train_learned_asism.py", "--namespace", NAMESPACE], env)
    require(learned / "learned_training_manifest.json", "learned training manifest")
    require(learned / "ranking_model.pt", "ranking model checkpoint")
    run_if_missing(
        [py, "scripts/asism/06_learn_thresholds_select.py", "--namespace", NAMESPACE],
        learned / "selected_manifest.jsonl", env,
    )
    require(learned / "selected_manifest.jsonl", "fixed-ratio learned selection (condition-G ablation reference)")

    run([py, "scripts/asism/07_build_threshold_contexts.py", "--namespace", NAMESPACE], env)
    require(learned / "threshold_contexts.jsonl", "threshold contexts")
    run([py, "scripts/asism/07b_verify_thresholds_proxy.py", "--phase", "estimate", "--namespace", NAMESPACE], env)
    run_if_missing(
        [py, "scripts/asism/07b_verify_thresholds_proxy.py", "--phase", "run",
         "--i-understand-this-trains-real-models", "--namespace", NAMESPACE],
        learned / "threshold_proxy_measurements.jsonl", env,
    )
    require(learned / "threshold_proxy_measurements.jsonl", "threshold proxy measurements")
    run([py, "scripts/asism/08_train_threshold_network.py", "--namespace", NAMESPACE], env)
    require(learned / "adaptive_threshold_manifest.json", "adaptive threshold manifest")
    run([py, "scripts/asism/08b_verify_full_policy_proxy.py", "--phase", "estimate", "--namespace", NAMESPACE], env)
    run_if_missing(
        [py, "scripts/asism/08b_verify_full_policy_proxy.py", "--phase", "run",
         "--i-understand-this-trains-real-models", "--namespace", NAMESPACE],
        learned / "full_policy_verification_results.jsonl", env,
    )
    require(learned / "full_policy_verification_results.jsonl", "full-policy verification results")
    run_if_missing(
        [py, "scripts/asism/09_finalize_learned_selection.py", "--namespace", NAMESPACE],
        learned / "adaptive_selected_manifest.jsonl", env,
    )
    require(learned / "adaptive_selected_manifest.jsonl", "condition C: finalized Learned ASISM selection")

    run([py, "scripts/classify/01_train_conditions.py", "--condition", "all", "--namespace", NAMESPACE], env)
    require(workspace / "outputs/stage4" / NAMESPACE / "stage4_training_results.json", "Stage 4 A/B/C results")
    run([py, "scripts/eval/stage5_evaluate.py", "--final-eval-run-id", "smoke-final-v1", "--namespace", NAMESPACE,
         "--bootstrap-resamples", "20"], env)
    stage5 = workspace / "outputs/stage5" / NAMESPACE / "smoke-final-v1"
    require(stage5 / "final_eval_run_manifest.json", "Stage 5 smoke manifest")
    run([py, "scripts/eval/compare_conditions.py", "--run-dir", str(stage5), "--bootstrap-resamples", "20"], env)
    require(stage5 / "stage5_comparison_report.json", "Stage 5 comparison report")
    print("\nEND-TO-END SMOKE COMPLETE — FIXTURE ENGINEERING EVIDENCE ONLY, NEVER THESIS RESULTS.", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## `scripts/smoke/01b_learned_asism_cpu_smoke.py`

*302 lines*

**A CPU-only integration smoke for the learned-ASISM pipeline: 04 -> 04b(fabricated) -> 05 -> 06.**

It exists because of a real coverage gap found in the supervisor review of 2026-08-21: the
end-to-end smoke never exercised `scripts/asism/04`-`09` - the thesis's novel contribution - because
doing so through the real pipeline requires a real (if tiny) SDXL run.

This script closes that gap **without touching SDXL**, by fabricating a self-consistent synthetic
candidate pool and score tables (`build_fixture`, `fabricate_utility_results`) and then running the
**real** 04, 05 and 06 scripts as subprocesses against them. It genuinely exercises the production
code path for: Go/No-Go feature removal (one signal is deliberately marked excluded), the subset
feasibility gate, role-disjoint pool splitting, set-utility training, LOO target generation, and
threshold selection.

What it does **not** validate: any scientific claim. The utilities are fabricated.

In [ ]:
# ===== FILE: scripts/smoke/01b_learned_asism_cpu_smoke.py =====
#!/usr/bin/env python3
"""CPU-only integration smoke for the Learned ASISM pipeline: 04 -> 04b(fake) -> 05 -> 06.

Requested by the supervisor review of 2026-08-21: the existing end-to-end smoke
(scripts/smoke/run_smoke_pipeline.py) never exercises scripts/asism/04-09 — the thesis's novel
contribution — because doing so through the real pipeline requires a real (if tiny) SDXL run. This
script closes that gap WITHOUT touching SDXL, by fabricating a self-consistent synthetic candidate
pool and score tables, then running the REAL 04, 05, and 06 scripts as subprocesses against them.

What this DOES exercise, with the real production code path:
  - Go/No-Go feature removal: one signal (explainability) is deliberately marked excluded, so this
    proves active_feature_columns() actually narrows the model's inputs (the bug fixed 2026-08-21)
    rather than merely having a code path for it.
  - 04's feasibility gate, failing closed on an under-sized pool before being tuned to pass.
  - 04's disjoint train/val image pools (verified again explicitly below, on top of 05's own check).
  - 05's training manifest, including learned_variant_status, which must read "reduced_variant"
    here (4 of 5 signals survived) — proving the manifest accurately reports reduced coverage.
  - 05's ranking-model checkpoint and image_marginal_targets.jsonl.
  - 06's fixed-ratio learned selection manifest.

What this does NOT exercise (needs a real, if small, GPU run — the "learned GPU smoke" tier):
  04b's real proxy-classifier training, 07-09's threshold learning and proxy verification, and
  condition C end to end. utility_results.jsonl here is FABRICATED, not measured.

STOP if any assertion fails. This is engineering connectivity evidence only — never a medical or
generative-quality claim, and never scientific evidence about ASISM's selection quality.
"""
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path(__file__).resolve().parents[2]
NAMESPACE = "dev-smoke-v1"
# pool_feasibility_report checks EVERY PRIMARY_ENDPOINT_LABEL, not just whichever ones a fixture
# happens to use — so the fixture must cover all 11, not a convenient subset. N_CANDIDATES is sized
# so each of the 11 labels + No-Finding clears feasibility_thresholds.min_candidates_per_class in
# BOTH the train and val pools (see configs/smoke_e2e.yaml -> stage3.learned_asism).
N_CANDIDATES = 900  # ~75/label pre-split; empirically the smallest round number with headroom
                     # against random shuffle variance across all 11 labels x 2 pools

sys.path.insert(0, str(REPO))


def build_fixture(workspace: Path, env: dict[str, str]) -> None:
    import numpy as np

    os.environ.update(env)  # so the imports below see PROJECT_ROOT / THESIS_CONFIG_OVERLAY too

    from scripts.asism.signals import write_score_artifact
    from scripts.utils.artifact_contracts import (
        asism_score_provenance, current_code_identity_hash, namespace_identity, stage2_paths, stage3_paths,
    )
    from scripts.utils.config import load_named_config
    from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS
    from scripts.utils.manifest import sha256_file, write_json

    stage2_cfg = load_named_config("stage2_generation.yaml", "stage2")
    stage3_cfg = load_named_config("stage3_asism.yaml", "stage3")
    s2_paths = stage2_paths(stage2_cfg, NAMESPACE)
    s3_paths = stage3_paths(stage3_cfg, NAMESPACE)
    # Mirror EXACTLY the config mutation every 0x_*.py script applies before computing provenance
    # (namespace-resolved absolute paths written back onto cfg.paths, split_namespace pinned) — the
    # provenance hash below must match what those scripts independently recompute in their own
    # subprocess, and config_sha256() hashes the WHOLE resolved config, so any unmirrored mutation
    # here silently produces a different hash and a "Stale/incompatible" false-positive rejection.
    for key, value in s3_paths.items():
        if key in stage3_cfg.paths:
            stage3_cfg.paths[key] = str(value)
    stage3_cfg.split_namespace = NAMESPACE
    s2_paths["root"].mkdir(parents=True, exist_ok=True)
    s3_paths["scores_dir"].mkdir(parents=True, exist_ok=True)
    s3_paths["asism_dir"].mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(7)

    # 1) A fabricated Stage 2 generation manifest: image_id + intended_label_vector, single-label
    #    recipes cycling through ALL 11 PRIMARY_ENDPOINT_LABELS plus a No-Finding (all-zero) recipe
    #    so every label the feasibility gate checks has real candidates in both the train and val
    #    pools — this fixture is about the learned-ASISM DATA PLUMBING, not recipe realism.
    image_ids = [f"smoke_synth_{index:04d}" for index in range(N_CANDIDATES)]
    recipe_pool = list(PRIMARY_ENDPOINT_LABELS) + [None]
    recipes = (recipe_pool * (N_CANDIDATES // len(recipe_pool) + 1))[:N_CANDIDATES]
    rng.shuffle(recipes)
    with open(s2_paths["manifest_path"], "w", encoding="utf-8") as handle:
        for image_id, label in zip(image_ids, recipes):
            vector = {lab: int(lab == label) for lab in PRIMARY_ENDPOINT_LABELS}
            handle.write(json.dumps({"image_id": image_id, "intended_label_vector": vector}) + "\n")

    # 2) A self-consistent (but not SDXL-derived) Stage 2 completion record. require_generation_
    #    complete() only checks that the recorded hashes match files that exist and that the split/
    #    code identity match the CURRENT run — it never requires the manifest to have come from a
    #    real generation job, so a structurally honest fixture is legitimate here.
    identity = namespace_identity(NAMESPACE)
    completion = {
        **identity,
        "status": "complete",
        "generation_manifest_sha256": sha256_file(s2_paths["manifest_path"]),
        "code_identity_sha256": current_code_identity_hash(),
        "lora_checkpoint_sha256": "0" * 64,  # placeholder: no real checkpoint exists in this fixture
        "smoke_only": True,
        "fixture": "01b_learned_asism_cpu_smoke.py",
    }
    write_json(s2_paths["generation_completion"], completion)

    # asism_score_provenance() re-reads Stage 2's completion from disk (via require_generation_
    # complete), so the fixture's provenance is computed the SAME WAY production computes it —
    # never hand-assembled, which is exactly what caused the first failed attempt at this fixture
    # (a placeholder asism_config_sha256 that require_score_artifact correctly rejected).
    provenance = asism_score_provenance(stage3_cfg, stage2_cfg, NAMESPACE)

    # 3) Five synthetic score tables, one column set per signal, matching signals.py's real schema
    #    so downstream code sees realistic shapes. Values are pure noise — never claimed to reflect
    #    generative quality.
    def frame(columns_and_ranges):
        import pandas as pd
        data = {"image_id": image_ids}
        for column, (low, high) in columns_and_ranges.items():
            data[column] = rng.uniform(low, high, size=N_CANDIDATES)
        return pd.DataFrame(data)

    signal_frames = {
        "similarity": frame({
            "similarity_knn_mean": (0.3, 0.9), "similarity_top1": (0.4, 0.95),
            "similarity_topk_spread": (0.0, 0.3),
        }),
        "iqa": frame({
            "iqa_composite": (0.2, 1.0), "iqa_sharpness": (10.0, 200.0), "iqa_contrast_std": (10.0, 60.0),
        }),
        "uncertainty": frame({
            "uncertainty_mean_std": (0.01, 0.2), "uncertainty_max_std": (0.02, 0.3),
            "uncertainty_entropy": (0.05, 0.6),
        }),
        "agreement": frame({"agreement_score": (-0.3, 0.9)}),
        # "explainability" is DELIBERATELY OMITTED — this is the Go/No-Go exclusion case below.
    }
    signal_frames["uncertainty"]["uncertainty_band"] = rng.choice(
        ["low", "moderate", "extreme"], size=N_CANDIDATES, p=[0.4, 0.45, 0.15]
    )
    for signal, signal_frame in signal_frames.items():
        write_score_artifact(
            signal_frame, s3_paths["scores_dir"] / f"{signal}_scores.parquet", signal, provenance
        )

    # 4) A hand-written Go/No-Go report: explainability EXCLUDED (its parquet does not exist), the
    #    other four INCLUDED. This is the exact case that used to crash 04/05/07/09 with
    #    "Missing learned-ASISM feature columns" before active_feature_columns() was added.
    write_json(s3_paths["gonogo_report"], {
        "surviving_signals": ["agreement", "iqa", "similarity", "uncertainty"],
        "excluded_signals": ["explainability"],
        "ablation_only_signals": [],
        "asism_variant_status": "reduced_variant — fewer than 5 signals survived (smoke fixture)",
        "smoke_only": True,
    })

    print(f"Fixture built: {N_CANDIDATES} synthetic candidates, 4/5 signals surviving Go/No-Go.")


def fabricate_utility_results(workspace: Path, env: dict[str, str]) -> None:
    """Stand in for 04b's real proxy-classifier training with a deterministic, clearly-fake
    macro-AUROC delta per subset. NEVER measured, NEVER claimed as evidence about ASISM quality —
    only shaped like utility_results.jsonl so 05 has something to train against on CPU."""
    import numpy as np

    os.environ.update(env)
    from scripts.asism.learned import read_jsonl
    from scripts.utils.artifact_contracts import stage3_paths
    from scripts.utils.config import load_named_config

    stage3_cfg = load_named_config("stage3_asism.yaml", "stage3")
    paths = stage3_paths(stage3_cfg, NAMESPACE)
    subsets = read_jsonl(paths["utility_subsets"])
    rng = np.random.default_rng(11)
    with open(paths["utility_results"], "w", encoding="utf-8") as handle:
        for row in subsets:
            real_only = float(rng.uniform(0.55, 0.65))
            # A synthetic "signal": larger subsets get a slightly larger, noisy uplift, purely so
            # 05's training loss has something non-degenerate to fit. Not a claim about real data.
            uplift = 0.01 + 0.00005 * len(row["image_ids"]) + float(rng.normal(0, 0.01))
            handle.write(json.dumps({
                "subset_id": row["subset_id"], "fold": 0, "seed": int(row.get("seed", 0)),
                "real_only_macro_auroc": real_only,
                "augmented_macro_auroc": real_only + uplift,
                "synthetic_smoke_only": True,
            }) + "\n")
    print(f"Fabricated utility_results.jsonl for {len(subsets)} subsets (SMOKE ONLY, not measured).")


def run(command: list[str], env: dict[str, str]) -> None:
    print("\n+ " + subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=REPO, env=env, check=True)


def require(path: Path, label: str) -> None:
    if not path.exists():
        raise SystemExit(f"SMOKE GATE FAILED: missing {label}: {path}")
    print(f"  validated {label}: {path}", flush=True)


def reset_prior_run(workspace: Path) -> None:
    """Delete this script's own output from any earlier run before rebuilding the fixture.

    Every artifact under here (04's utility_subsets.jsonl, 06's selected_manifest.jsonl, ...) is
    written with mode "x" (exclusive create) by production code — a deliberate, correct guard
    against silently overwriting a real result. That guard makes THIS script fail with
    FileExistsError on every rerun unless it clears its own prior output first: the fixture is
    entirely synthetic and rebuilt fresh every invocation, so there is nothing here worth
    preserving between runs, unlike a real production selected_manifest.jsonl.
    """
    target = workspace / "data" / "chexpert" / "synthetic" / NAMESPACE
    if not target.is_dir():
        return
    if NAMESPACE not in str(target) or "smoke" not in str(target).lower():
        raise SystemExit(f"REFUSING to delete a path that doesn't look like a smoke fixture: {target}")
    shutil.rmtree(target)
    print(f"Cleared prior run's fixture output: {target}", flush=True)


def main() -> int:
    workspace = REPO / "outputs" / "smoke" / NAMESPACE
    if not (workspace / "SMOKE_ONLY.json").is_file():
        raise SystemExit(
            "Run `python scripts/smoke/run_smoke_pipeline.py --phase local` first (builds the "
            "fixture splits this script's Stage-2 fixture is layered on top of)."
        )
    reset_prior_run(workspace)
    env = {
        **os.environ, "PROJECT_ROOT": str(workspace),
        # Reuses the SAME overlay as run_smoke_pipeline.py (proven to resolve split/namespace paths
        # correctly), extended with a `stage3.learned_asism` block sized for this fixture's 150
        # candidates — see configs/smoke_e2e.yaml.
        "THESIS_CONFIG_OVERLAY": str(REPO / "configs" / "smoke_e2e.yaml"),
        "PYTHONUNBUFFERED": "1",
    }
    py = sys.executable

    build_fixture(workspace, env)

    run([py, "scripts/asism/04_build_utility_subsets.py", "--phase", "feasibility", "--namespace", NAMESPACE], env)
    run([py, "scripts/asism/04_build_utility_subsets.py", "--phase", "build", "--namespace", NAMESPACE], env)

    from scripts.utils.artifact_contracts import stage3_paths
    from scripts.utils.config import load_named_config
    stage3_cfg = load_named_config("stage3_asism.yaml", "stage3")
    paths = stage3_paths(stage3_cfg, NAMESPACE)
    require(paths["utility_subsets"], "utility_subsets.jsonl")

    fabricate_utility_results(workspace, env)
    require(paths["utility_results"], "fabricated utility_results.jsonl")

    run([py, "scripts/asism/05_train_learned_asism.py", "--namespace", NAMESPACE], env)
    require(paths["learned_dir"] / "learned_training_manifest.json", "learned training manifest")
    require(paths["learned_dir"] / "ranking_model.pt", "ranking model checkpoint")
    require(paths["learned_dir"] / "set_utility_model.pt", "set-utility model checkpoint")

    from scripts.utils.manifest import read_json
    manifest = read_json(paths["learned_dir"] / "learned_training_manifest.json")
    if sorted(manifest.get("contributing_signals", [])) != ["agreement", "iqa", "similarity", "uncertainty"]:
        raise SystemExit(
            f"SMOKE GATE FAILED: expected exactly the 4 surviving signals in contributing_signals "
            f"(explainability excluded), got {manifest.get('contributing_signals')}. This is the "
            "exact bug fixed 2026-08-21 — a missing signal must shrink the feature/signal set, "
            "never crash and never silently keep a 5th signal."
        )
    # 4 of 5 signals clears the >=3 primary-vs-reduced_variant threshold (02_gonogo.py's rule,
    # mirrored for the learned selector) — "primary" IS the correct status here. The threshold
    # boundary itself (2 vs 3 signals) is unit-tested directly in test_learned_asism.py; this
    # integration smoke's job is proving the plumbing doesn't crash when a signal is missing, which
    # the successful run up to this point already demonstrates.
    if manifest.get("learned_variant_status") != "primary":
        raise SystemExit(
            f"SMOKE GATE FAILED: 4 contributing signals should report status='primary'; got "
            f"{manifest.get('learned_variant_status')!r}."
        )
    if manifest["validation"]["image_overlap_fraction"] != 0.0:
        raise SystemExit(
            f"SMOKE GATE FAILED: train/val image pools must be disjoint; overlap fraction = "
            f"{manifest['validation']['image_overlap_fraction']}"
        )
    print(
        f"  verified: {len(manifest['contributing_signals'])}/5 signals contributed "
        f"({manifest['contributing_signals']}), status={manifest['learned_variant_status']!r}, "
        f"train/val pools disjoint"
    )

    run([py, "scripts/asism/06_learn_thresholds_select.py", "--namespace", NAMESPACE], env)
    require(paths["learned_dir"] / "learned_selection_manifest.json", "fixed-ratio learned selection manifest")
    require(paths["learned_selected_manifest"], "fixed-ratio learned selected_manifest.jsonl")

    print("\nCPU LEARNED-ASISM INTEGRATION SMOKE COMPLETE.")
    print("Fixture data and fabricated utility results only — NOT a medical, generative-quality, or")
    print("selection-quality result. Proves 04 -> 05 -> 06 connect and the Go/No-Go feature-removal")
    print("fix works end to end. 07-09 and condition C still need a GPU smoke pass (see docs/smoke_e2e.md).")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


---

# Part 12 - Tests

The tests are where the project's invariants are written down executably. `tests/run_all.py` is the documented production STOP gate.

## `tests/run_all.py`

*90 lines*

**The production STOP gate.** `docs/runpod_stage2_to_5_commands.md` says "STOP unless every test
passes", and this is the command that decides.

Two hard-won details are worth reading:

- **Suites are discovered, never hand-listed.** A hardcoded list once omitted
  `test_learned_asism.py` - 85 tests covering the thesis's novel contribution - and this gate
  happily reported a green "78 passed" while running none of them.
- **Suites that need pytest go through pytest.** A suite using `monkeypatch`, `tmp_path` or
  `pytest.raises` cannot self-run; executing it as a bare script runs *zero* tests and returns 0.
  Detection is by the presence of a `__main__` self-runner.

A suite that produces no parseable summary is treated as **failed**, never as a silent pass.

In [ ]:
# ===== FILE: tests/run_all.py =====
#!/usr/bin/env python3
"""Run every test suite and report a single aggregate result.

Usage:
    python tests/run_all.py
"""

from __future__ import annotations

import os
import re
import subprocess
import sys
from pathlib import Path

# DISCOVERED, never hand-listed. This command is the documented production STOP gate
# (docs/runpod_stage2_to_5_commands.md: "STOP unless ... every test passes"), so a hardcoded list is
# a silent-coverage hazard: test_learned_asism.py — 85 tests covering the thesis's novel
# contribution — was absent from the previous hardcoded list, and this command reported a green
# "78 passed" while never running any of them.
def discover_suites(here: Path) -> list[str]:
    return sorted(path.name for path in here.glob("test_*.py"))


def main() -> int:
    here = Path(__file__).resolve().parent
    repo = here.parent
    TESTS = discover_suites(here)
    if not TESTS:
        print(f"NO TEST SUITES FOUND under {here} — refusing to report success.", flush=True)
        return 1
    print(f"Discovered {len(TESTS)} suite(s): {', '.join(TESTS)}\n", flush=True)

    total_passed, total_failed, failed_suites = 0, 0, []

    for name in TESTS:
        print(f"\n{'=' * 70}\n{name}\n{'=' * 70}", flush=True)
        path = here / name
        # Most suites carry their own `__main__` self-runner so the gate needs no test framework.
        # Suites that use pytest fixtures (monkeypatch, tmp_path) or pytest.raises cannot self-run
        # and MUST go through pytest — running them as bare scripts executes no tests at all, which
        # is how test_learned_asism.py previously contributed a silent zero to this gate.
        self_running = '__name__ == "__main__"' in path.read_text(encoding="utf-8")
        command = (
            [sys.executable, str(path)] if self_running
            else [sys.executable, "-m", "pytest", str(path), "-q"]
        )
        result = subprocess.run(
            command, cwd=repo, capture_output=True, text=True,
            env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
        )
        print(result.stdout, flush=True)
        if result.stderr.strip():
            print(result.stderr, flush=True)

        if not self_running and "No module named pytest" in (result.stdout + result.stderr):
            print(
                f"  {name} needs pytest, which is not installed. Install it "
                "(environment/requirements.txt) — this suite has NOT been run.",
                flush=True,
            )
            failed_suites.append(name)
            continue

        passed = re.search(r"(\d+) passed", result.stdout)
        failed = re.search(r"(\d+) failed", result.stdout)
        counted = False
        if passed:
            total_passed += int(passed.group(1)); counted = True
        if failed:
            total_failed += int(failed.group(1)); counted = True
        if not counted:
            # A suite that reports no countable result is treated as a failure, never as a silent
            # pass — an unparseable summary means we do not know that anything ran.
            print(f"  {name} produced no parseable test summary; treating as FAILED.", flush=True)
            failed_suites.append(name)
            continue
        if result.returncode != 0:
            failed_suites.append(name)

    print(f"\n{'=' * 70}", flush=True)
    print(f"TOTAL: {total_passed} passed, {total_failed} failed across {len(TESTS)} suites", flush=True)
    if failed_suites:
        print(f"FAILED SUITES: {failed_suites}", flush=True)
    print(f"{'=' * 70}", flush=True)
    return 1 if failed_suites else 0


if __name__ == "__main__":
    raise SystemExit(main())


## `tests/fixture_workspace.py`

*23 lines*

**Per-test writable workspaces with unconditional cleanup**, under `tests/_runtime/`. A context
manager that creates a uniquely numbered directory, yields it, and removes it in a `finally` - so a
failing test cannot leave state behind that makes the *next* test pass or fail for the wrong reason.

In [ ]:
# ===== FILE: tests/fixture_workspace.py =====
"""Repository-controlled, per-test writable workspaces with unconditional cleanup."""
from __future__ import annotations

import itertools
import shutil
from contextlib import contextmanager
from pathlib import Path

ROOT = Path(__file__).resolve().parent / "_runtime"
_COUNTER = itertools.count()


@contextmanager
def fixture_workspace(name: str):
    ROOT.mkdir(exist_ok=True)
    path = ROOT / f"{name}-{next(_COUNTER):04d}"
    if path.exists():
        shutil.rmtree(path)
    path.mkdir()
    try:
        yield path
    finally:
        shutil.rmtree(path, ignore_errors=False)


## `tests/test_metrics.py`

*234 lines*

**Tests for `scripts/utils/metrics.py`** - the statistics layer, tested against values computed by
hand rather than against another library's output.

Covers AUROC (including tie handling and the NaN-when-a-class-is-absent contract), average
precision at known values, calibration error, the masking behaviour, patient-level bootstrap, and
the Holm-Bonferroni / Benjamini-Hochberg procedures.

In [ ]:
# ===== FILE: tests/test_metrics.py =====
"""Known-answer tests for scripts/utils/metrics.py.

These use analytically-derived expected values, not fixtures blessed from the code's own output —
a test that only asserts "the code does what it currently does" would not catch a wrong AUROC.
"""

from __future__ import annotations

import sys
sys.dont_write_bytecode = True
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scripts.utils.metrics import (  # noqa: E402
    auroc,
    average_precision,
    benjamini_hochberg,
    expected_calibration_error,
    holm_bonferroni,
    macro_auroc_masked,
    paired_bootstrap_difference,
    patient_level_bootstrap,
    sensitivity_specificity_f1,
)


def test_auroc_perfect_separation():
    scores = np.array([0.1, 0.2, 0.8, 0.9])
    labels = np.array([0, 0, 1, 1])
    assert auroc(scores, labels) == 1.0


def test_auroc_perfect_inversion():
    scores = np.array([0.9, 0.8, 0.2, 0.1])
    labels = np.array([0, 0, 1, 1])
    assert auroc(scores, labels) == 0.0


def test_auroc_all_ties_is_half():
    # Every score identical -> every pair is a tie -> AUROC exactly 0.5 via mid-ranks.
    scores = np.array([0.5, 0.5, 0.5, 0.5])
    labels = np.array([0, 1, 0, 1])
    assert auroc(scores, labels) == 0.5


def test_auroc_known_value():
    # 2 positives (0.6, 0.4), 2 negatives (0.5, 0.3).
    # Pairs (pos,neg): (.6,.5)=1, (.6,.3)=1, (.4,.5)=0, (.4,.3)=1  -> 3/4 = 0.75
    scores = np.array([0.6, 0.4, 0.5, 0.3])
    labels = np.array([1, 1, 0, 0])
    assert auroc(scores, labels) == 0.75


def test_auroc_undefined_when_single_class():
    assert np.isnan(auroc(np.array([0.1, 0.9]), np.array([1, 1])))
    assert np.isnan(auroc(np.array([0.1, 0.9]), np.array([0, 0])))


def test_average_precision_perfect():
    scores = np.array([0.9, 0.8, 0.2, 0.1])
    labels = np.array([1, 1, 0, 0])
    assert average_precision(scores, labels) == 1.0


def test_average_precision_known_value():
    # Ranked: 0.9(pos) 0.8(neg) 0.7(pos). Precision at hits: 1/1=1.0, 2/3=0.667. AP=(1+0.667)/2
    scores = np.array([0.9, 0.8, 0.7])
    labels = np.array([1, 0, 1])
    assert abs(average_precision(scores, labels) - (1.0 + 2 / 3) / 2) < 1e-12


def test_sensitivity_specificity_f1():
    scores = np.array([0.9, 0.8, 0.2, 0.1])
    labels = np.array([1, 0, 1, 0])
    sensitivity, specificity, f1 = sensitivity_specificity_f1(scores, labels, threshold=0.5)
    assert sensitivity == 0.5   # 1 of 2 positives caught
    assert specificity == 0.5   # 1 of 2 negatives correctly rejected
    assert abs(f1 - 0.5) < 1e-12


def test_ece_perfectly_calibrated_is_zero():
    # Predictions of exactly 0 and 1 matching outcomes -> no calibration gap.
    scores = np.array([0.0, 0.0, 1.0, 1.0])
    labels = np.array([0.0, 0.0, 1.0, 1.0])
    assert expected_calibration_error(scores, labels) == 0.0


def test_ece_maximally_miscalibrated_is_one():
    scores = np.array([1.0, 1.0])
    labels = np.array([0.0, 0.0])
    assert abs(expected_calibration_error(scores, labels) - 1.0) < 1e-12


def test_macro_auroc_masked_excludes_masked_positions():
    labels = ["Cardiomegaly", "Edema"]
    # Row 3's Cardiomegaly is masked; if the mask were ignored the AUROC would change.
    probabilities = np.array([[0.9, 0.1], [0.8, 0.2], [0.1, 0.9], [0.2, 0.8]])
    targets = np.array([[1, 0], [1, 0], [0, 1], [0, 1]], dtype=np.float32)
    masks = np.array([[1, 1], [1, 1], [1, 1], [0, 1]], dtype=bool)

    result = macro_auroc_masked(
        probabilities, targets, masks, labels=labels, primary_labels=labels
    )
    assert result["per_label"]["Cardiomegaly"]["effective_n"] == 3
    assert result["per_label"]["Edema"]["effective_n"] == 4
    assert result["macro_auroc"] == 1.0


def test_macro_auroc_excludes_insufficient_support():
    labels = ["Cardiomegaly", "Edema"]
    probabilities = np.array([[0.9, 0.1], [0.8, 0.9]])
    targets = np.array([[1, 1], [0, 1]], dtype=np.float32)
    masks = np.array([[1, 1], [1, 1]], dtype=bool)

    # Edema has no negatives -> ineligible -> excluded from the macro rather than counted as 0.5.
    result = macro_auroc_masked(
        probabilities, targets, masks, labels=labels, primary_labels=labels
    )
    assert result["per_label"]["Edema"]["eligible"] is False
    assert result["n_labels_in_macro"] == 1
    assert result["n_labels_excluded"] == 1


def test_patient_level_bootstrap_resamples_patients_not_rows():
    # 2 patients, 5 rows. Patient A is over-represented by row count; patient-level resampling must
    # move all of a patient's rows together, so only 3 distinct means are reachable.
    patient_ids = np.array(["A", "A", "A", "B", "B"])
    values = np.array([1.0, 1.0, 1.0, 0.0, 0.0])

    def metric(indices):
        return float(values[indices].mean())

    result = patient_level_bootstrap(patient_ids, metric, n_resamples=200, seed=7)
    assert result["n_patients"] == 2
    assert result["point_estimate"] == 0.6
    assert result["ci_lower"] >= 0.0 and result["ci_upper"] <= 1.0


def test_paired_bootstrap_detects_real_difference():
    rng = np.random.default_rng(0)
    n = 300
    patient_ids = np.arange(n).astype(str)
    labels = rng.integers(0, 2, size=n)
    strong = np.where(labels == 1, rng.uniform(0.6, 1.0, n), rng.uniform(0.0, 0.4, n))
    weak = rng.uniform(0.0, 1.0, n)

    result = paired_bootstrap_difference(
        patient_ids,
        lambda idx: auroc(strong[idx], labels[idx]),
        lambda idx: auroc(weak[idx], labels[idx]),
        n_resamples=200,
        seed=1,
    )
    assert result["observed_difference"] > 0.2
    assert result["ci_lower"] > 0.0     # CI excludes zero
    assert result["p_value"] < 0.05


def test_paired_bootstrap_no_difference_for_identical_scores():
    rng = np.random.default_rng(3)
    n = 200
    patient_ids = np.arange(n).astype(str)
    labels = rng.integers(0, 2, size=n)
    scores = rng.uniform(size=n)

    result = paired_bootstrap_difference(
        patient_ids,
        lambda idx: auroc(scores[idx], labels[idx]),
        lambda idx: auroc(scores[idx], labels[idx]),
        n_resamples=100,
        seed=2,
    )
    assert result["observed_difference"] == 0.0
    assert result["p_value"] == 1.0


def test_holm_bonferroni_known_sequence():
    # Classic worked example: p = .01, .02, .03, .04 at alpha .05, n=4.
    # Adjusted: .04, .06, .06, .06 -> only the first is rejected (step-down stops at the first fail).
    result = holm_bonferroni({"a": 0.01, "b": 0.02, "c": 0.03, "d": 0.04}, alpha=0.05)
    assert abs(result["a"]["adjusted_p_value"] - 0.04) < 1e-12
    assert result["a"]["rejected"] is True
    assert result["b"]["rejected"] is False
    assert result["c"]["rejected"] is False
    assert result["d"]["rejected"] is False


def test_holm_bonferroni_is_monotonic():
    result = holm_bonferroni({"a": 0.001, "b": 0.5, "c": 0.02}, alpha=0.05)
    adjusted = sorted((entry["p_value"], entry["adjusted_p_value"]) for entry in result.values())
    values = [pair[1] for pair in adjusted]
    assert values == sorted(values), "adjusted p-values must be non-decreasing in raw p"


def test_benjamini_hochberg_rejects_more_than_holm():
    p_values = {f"t{i}": p for i, p in enumerate([0.001, 0.008, 0.02, 0.03, 0.04])}
    fdr = benjamini_hochberg(p_values, alpha=0.05)
    holm = holm_bonferroni(p_values, alpha=0.05)
    n_fdr = sum(1 for entry in fdr.values() if entry["rejected"])
    n_holm = sum(1 for entry in holm.values() if entry["rejected"])
    assert n_fdr >= n_holm, "FDR should be at least as permissive as Holm at the same alpha"


def test_corrections_label_their_family():
    holm = holm_bonferroni({"a": 0.01}, alpha=0.05)
    fdr = benjamini_hochberg({"a": 0.01}, alpha=0.05)
    assert holm["a"]["family"] == "confirmatory"
    assert fdr["a"]["family"] == "exploratory"


def test_corrections_handle_undefined_p_values():
    result = holm_bonferroni({"a": 0.01, "b": float("nan")}, alpha=0.05)
    assert result["b"]["rejected"] is False
    assert "note" in result["b"]


if __name__ == "__main__":
    import traceback

    tests = [(name, value) for name, value in sorted(globals().items()) if name.startswith("test_")]
    passed, failed = 0, 0
    for name, function in tests:
        try:
            function()
            print(f"  PASS  {name}")
            passed += 1
        except Exception:
            print(f"  FAIL  {name}")
            traceback.print_exc()
            failed += 1
    print(f"\n{passed} passed, {failed} failed, {len(tests)} total")
    raise SystemExit(1 if failed else 0)


## `tests/test_asism_signals.py`

*420 lines*

**Tests for `scripts/asism/signals.py`** - every signal's behaviour on constructed inputs where the
right answer is known: IQA on blank/clipped/blurred images, the hierarchical reference fallback,
agreement's unintended-positive penalty, the uncertainty sign convention, Grad-CAM region overlap,
distinctiveness and duplicate clustering.

It also **pins the `near_duplicate_similarity = 0.95` cut-off** (change C3) so the
literature-grounded threshold cannot drift silently.

In [ ]:
# ===== FILE: tests/test_asism_signals.py =====
"""Behavioural tests for the ASISM signals that need no GPU (docs/stages2_to_5_plan.md §4).

These assert the SEMANTICS the methodology depends on — e.g. that a No-Finding recipe scores high
agreement exactly when disease probabilities are low, and that a memorized near-duplicate is
flagged rather than rewarded — not merely that the functions run.
"""

from __future__ import annotations

import sys
sys.dont_write_bytecode = True
from pathlib import Path

import numpy as np
import pytest
from omegaconf import OmegaConf
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from scripts.asism.signals import (  # noqa: E402
    compute_agreement_scores,
    region_from_cam_mass,
    compute_distinctiveness_scores,
    duplicate_clusters,
    compute_iqa_scores,
    compute_similarity_scores,
    compute_uncertainty_scores,
    expected_region_for,
    hierarchical_reference_indices,
    region_overlap_score,
)
from scripts.utils.config import CONFIGS_DIR  # noqa: E402
from scripts.utils.labels import PRIMARY_ENDPOINT_LABELS  # noqa: E402
from fixture_workspace import fixture_workspace

CONFIG = OmegaConf.load(CONFIGS_DIR / "stage3_asism.yaml")
OmegaConf.resolve(CONFIG)

def _write_image(workspace: Path, name: str, array: np.ndarray) -> Path:
    path = workspace / name
    Image.fromarray(array.astype(np.uint8), mode="RGB").save(path, format="JPEG", quality=95)
    return path


def _intended(positive_labels: list[str]) -> dict:
    return {label: (1 if label in positive_labels else 0) for label in PRIMARY_ENDPOINT_LABELS}


# ---------------------------------------------------------------- IQA (§4.2)

def test_iqa_flags_blank_image():
    blank = np.full((128, 128, 3), 128, dtype=np.uint8)
    with fixture_workspace("iqa-blank") as workspace:
        scores = compute_iqa_scores(_write_image(workspace, "blank.jpg", blank), CONFIG)
    assert bool(scores["iqa_valid"]) is True
    assert bool(scores["iqa_is_near_uniform"]) is True
    assert scores["iqa_composite"] < 0.6, "a blank image must be heavily penalized"


def test_iqa_rewards_structured_image():
    rng = np.random.default_rng(0)
    structured = rng.integers(0, 255, size=(128, 128, 1), dtype=np.uint8).repeat(3, axis=2)
    with fixture_workspace("iqa-structured") as workspace:
        scores = compute_iqa_scores(_write_image(workspace, "structured.jpg", structured), CONFIG)
    assert bool(scores["iqa_is_near_uniform"]) is False
    assert scores["iqa_composite"] > 0.7


def test_iqa_detects_colour_drift():
    # A real CXR is achromatic; strong per-channel divergence is a generation defect.
    rng = np.random.default_rng(1)
    coloured = rng.integers(0, 255, size=(128, 128, 3), dtype=np.uint8)
    with fixture_workspace("iqa-colour") as workspace:
        scores = compute_iqa_scores(_write_image(workspace, "coloured.jpg", coloured), CONFIG)
    assert scores["iqa_channel_spread"] > 2.0


def test_iqa_handles_unreadable_file_without_raising():
    with fixture_workspace("iqa-bad") as workspace:
        bad = workspace / "not_an_image.jpg"
        bad.write_bytes(b"definitely not a jpeg")
        scores = compute_iqa_scores(bad, CONFIG)
    assert bool(scores["iqa_valid"]) is False
    assert "iqa_error" in scores


# ---------------------------------------------------------------- Agreement (§4.5)

def test_agreement_high_when_intended_label_predicted():
    intended = _intended(["Cardiomegaly"])
    probabilities = np.full((1, 12), 0.05)
    probabilities[0, PRIMARY_ENDPOINT_LABELS.index("Cardiomegaly")] = 0.95
    result = compute_agreement_scores(probabilities, [intended], CONFIG)
    assert result.loc[0, "agreement_score"] > 0.9
    assert result.loc[0, "agreement_n_confident_unintended"] == 0


def test_agreement_low_when_intended_label_absent():
    intended = _intended(["Cardiomegaly"])
    probabilities = np.full((1, 12), 0.05)  # the intended label is NOT predicted
    result = compute_agreement_scores(probabilities, [intended], CONFIG)
    assert result.loc[0, "agreement_score"] < 0.2


def test_agreement_penalizes_confident_unintended_labels():
    intended = _intended(["Cardiomegaly"])
    probabilities = np.full((1, 12), 0.05)
    probabilities[0, PRIMARY_ENDPOINT_LABELS.index("Cardiomegaly")] = 0.95
    probabilities[0, PRIMARY_ENDPOINT_LABELS.index("Pneumothorax")] = 0.99  # unintended, confident

    penalized = compute_agreement_scores(probabilities, [intended], CONFIG).loc[0, "agreement_score"]

    clean = np.full((1, 12), 0.05)
    clean[0, PRIMARY_ENDPOINT_LABELS.index("Cardiomegaly")] = 0.95
    unpenalized = compute_agreement_scores(clean, [intended], CONFIG).loc[0, "agreement_score"]

    assert penalized < unpenalized


def test_agreement_no_finding_recipe_is_high_when_all_probabilities_low():
    """A No-Finding recipe is the all-zero intended vector: agreement must be HIGH exactly when the
    classifier sees no disease (§4.5)."""
    intended = _intended([])
    low = np.full((1, 12), 0.02)
    result = compute_agreement_scores(low, [intended], CONFIG)
    assert bool(result.loc[0, "agreement_is_no_finding_recipe"]) is True
    assert result.loc[0, "agreement_score"] > 0.9


def test_agreement_no_finding_recipe_is_low_when_disease_predicted():
    intended = _intended([])
    high = np.full((1, 12), 0.9)
    result = compute_agreement_scores(high, [intended], CONFIG)
    assert result.loc[0, "agreement_score"] < 0.2


# ---------------------------------------------------------------- Uncertainty (§4.3)

def test_uncertainty_bands_are_assigned_by_spread():
    std = np.array([[0.01] * 12, [0.10] * 12, [0.40] * 12])
    mean = np.full((3, 12), 0.5)
    result = compute_uncertainty_scores(std, mean, CONFIG)
    assert list(result["uncertainty_band"]) == ["low", "moderate", "extreme"]


def test_uncertainty_does_not_itself_penalize():
    """The signal reports bands; it must NOT encode 'higher = worse'. Moderate uncertainty may be
    the informative band, and that judgement belongs to the selection policy (§4.3/§4.8)."""
    std = np.array([[0.01] * 12, [0.10] * 12])
    mean = np.full((2, 12), 0.5)
    result = compute_uncertainty_scores(std, mean, CONFIG)
    assert "uncertainty_penalty" not in result.columns
    assert "uncertainty_score" not in result.columns


# ---------------------------------------------------------------- Similarity (§4.1)

def test_similarity_hierarchy_falls_back_when_exact_match_is_rare():
    references = [frozenset({"Cardiomegaly"})] * 3 + [frozenset({"Edema"})] * 40
    # Only 3 exact matches, below min_exact_match_references -> must not return tier1.
    indices, tier = hierarchical_reference_indices(
        _intended(["Cardiomegaly"]), references, min_exact=25
    )
    assert tier != "tier1_exact"
    assert len(indices) > 0, "fallback must always yield a usable reference pool"


def test_similarity_hierarchy_uses_exact_match_when_plentiful():
    references = [frozenset({"Cardiomegaly"})] * 40 + [frozenset({"Edema"})] * 5
    indices, tier = hierarchical_reference_indices(
        _intended(["Cardiomegaly"]), references, min_exact=25
    )
    assert tier == "tier1_exact"
    assert len(indices) == 40


def test_similarity_no_finding_recipe_falls_back_to_class_agnostic():
    references = [frozenset({"Cardiomegaly"})] * 30
    _, tier = hierarchical_reference_indices(_intended([]), references, min_exact=25)
    assert tier == "tier4_class_agnostic"


def test_similarity_flags_near_duplicate_as_memorization():
    """An image nearly identical to ONE reference must be flagged, not rewarded (§4.1)."""
    reference = np.eye(1, 16, 0).repeat(30, axis=0) + np.random.default_rng(0).normal(0, 0.3, (30, 16))
    exact_copy = reference[0:1].copy()

    result = compute_similarity_scores(
        exact_copy,
        reference,
        [frozenset()] * 30,
        [_intended([])],
        CONFIG,
    )
    assert result.loc[0, "similarity_top1"] > 0.99
    assert bool(result.loc[0, "novelty_is_near_duplicate"]) is True
    assert result.loc[0, "novelty_score"] == 0.0, "memorized images get no novelty credit"


def test_similarity_normal_image_is_not_flagged():
    rng = np.random.default_rng(2)
    reference = rng.normal(size=(40, 16))
    synthetic = rng.normal(size=(1, 16))
    result = compute_similarity_scores(
        synthetic, reference, [frozenset()] * 40, [_intended([])], CONFIG
    )
    assert bool(result.loc[0, "novelty_is_near_duplicate"]) is False


def test_near_duplicate_similarity_threshold_is_literature_grounded():
    """The 0.95 near-duplicate cutoff is not hand-tuned: DINOv2 cosine >= 0.95 == "almost
    identical" for medical latent-diffusion output, validated against real patient replication by
    Dar et al., Nature Biomedical Engineering (2025), doi:10.1038/s41551-025-01468-8. A silent
    change to this constant would weaken that citation, so pin it here."""
    assert float(CONFIG.signals.similarity.near_duplicate_similarity) == 0.95
    assert float(CONFIG.signals.similarity.near_duplicate_spread) == 0.02
    assert bool(CONFIG.learned_asism.safety.reject_near_duplicates) is True
    config_text = (CONFIGS_DIR / "stage3_asism.yaml").read_text(encoding="utf-8")
    assert "Nature" in config_text and "10.1038/s41551-025-01468-8" in config_text


# ---------------------------------------------------------------- Explainability (§4.4)

def test_region_overlap_full_inside():
    cam = np.zeros((10, 10))
    cam[5:8, 5:8] = 1.0
    assert region_overlap_score(cam, (0.0, 0.0, 1.0, 1.0)) == 1.0


def test_region_overlap_fully_outside():
    cam = np.zeros((10, 10))
    cam[0:2, 0:2] = 1.0
    assert region_overlap_score(cam, (0.5, 0.5, 1.0, 1.0)) == 0.0


def test_region_overlap_undefined_for_empty_cam():
    assert np.isnan(region_overlap_score(np.zeros((10, 10)), (0.0, 0.0, 1.0, 1.0)))


def test_expected_region_is_pathology_specific():
    cardiomegaly = expected_region_for(_intended(["Cardiomegaly"]), CONFIG)
    effusion = expected_region_for(_intended(["Pleural Effusion"]), CONFIG)
    assert cardiomegaly != effusion, "different pathologies must have different expected regions"


def test_expected_region_no_finding_uses_baseline_box():
    assert expected_region_for(_intended([]), CONFIG) == tuple(
        CONFIG.signals.explainability.baseline_box
    )


def test_expected_region_multilabel_is_union():
    both = expected_region_for(_intended(["Cardiomegaly", "Pleural Effusion"]), CONFIG)
    cardiomegaly = expected_region_for(_intended(["Cardiomegaly"]), CONFIG)
    # The union must be at least as tall as either constituent region.
    assert both[3] >= cardiomegaly[3]


# ----------------------------------------------------------------------------------------------
# §4.6 Distinctiveness (within-synthetic redundancy)
# ----------------------------------------------------------------------------------------------

def _distinct_config(k_neighbors=10, duplicate_similarity=0.95):
    return OmegaConf.create({
        "signals": {"distinctiveness": {
            "k_neighbors": k_neighbors,
            "duplicate_similarity": duplicate_similarity,
            "deduplicate_before_scoring": False,
        }}
    })


def _vec(*positive_labels):
    return {label: int(label in positive_labels) for label in PRIMARY_ENDPOINT_LABELS}


def test_distinctiveness_penalises_within_class_duplicates():
    """Three near-identical images of one class must score LOWER than a genuinely different one."""
    duplicate = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    embeddings = np.stack([
        duplicate,                                  # A
        duplicate + np.array([0.01, 0.0, 0.0]),     # B ~ A
        duplicate + np.array([0.0, 0.01, 0.0]),     # C ~ A
        np.array([0.0, 1.0, 0.0], dtype=np.float32),  # D, orthogonal
    ])
    intended = [_vec("Edema")] * 4
    frame = compute_distinctiveness_scores(embeddings, intended, _distinct_config(k_neighbors=1))

    assert frame["distinctiveness_score"][3] > frame["distinctiveness_score"][0]
    assert frame["distinctiveness_score"][3] > frame["distinctiveness_score"][1]
    assert frame["distinctiveness_score"][3] > frame["distinctiveness_score"][2]
    # A, B and C are near-identical, so they receive near-identical scores -- which is exactly why a
    # per-image threshold cannot keep just one of them, and why duplicate_clusters() exists.
    assert abs(frame["distinctiveness_score"][0] - frame["distinctiveness_score"][1]) < 0.05


def test_distinctiveness_is_computed_within_class_not_across_classes():
    """A rare class must not look 'distinct' merely because fewer images of it were generated.

    Guards the class-size confound: the two Fracture images are identical to each other, so both
    must be scored redundant, even though Edema has five times as many images in the pool.
    """
    common = np.array([1.0, 0.0], dtype=np.float32)
    rare = np.array([0.0, 1.0], dtype=np.float32)
    embeddings = np.stack(
        [common + np.random.default_rng(i).normal(0, 0.3, 2).astype(np.float32) for i in range(10)]
        + [rare, rare.copy()]
    )
    intended = [_vec("Edema")] * 10 + [_vec("Fracture")] * 2
    frame = compute_distinctiveness_scores(embeddings, intended, _distinct_config(k_neighbors=1))

    fracture = frame["distinctiveness_score"][10:].to_numpy()
    edema = frame["distinctiveness_score"][:10].to_numpy()
    assert (fracture < edema.mean()).all(), (
        "identical rare-class images scored as more distinct than a diverse common class -- the "
        "score is leaking class size instead of measuring redundancy"
    )
    assert (frame["distinctiveness_group_size"][:10] == 10).all()
    assert (frame["distinctiveness_group_size"][10:] == 2).all()


def test_distinctiveness_is_undefined_for_a_single_image_class():
    """One image in its class has no peer, so redundancy is undefined -- never a fabricated 1.0."""
    embeddings = np.stack([np.array([1.0, 0.0], dtype=np.float32), np.array([0.0, 1.0], dtype=np.float32)])
    frame = compute_distinctiveness_scores(embeddings, [_vec("Edema"), _vec("Fracture")], _distinct_config())

    assert bool(frame["distinctiveness_is_undefined"].all())
    assert frame["distinctiveness_score"].isna().all()
    assert (frame["distinctiveness_k_used"] == 0).all()


def test_distinctiveness_direction_matches_the_other_signals():
    """Higher must mean BETTER (more unique), like every other admitted signal."""
    embeddings = np.stack([
        np.array([1.0, 0.0], dtype=np.float32),
        np.array([1.0, 0.0], dtype=np.float32),   # exact duplicate of the first
        np.array([0.0, 1.0], dtype=np.float32),   # unrelated
    ])
    intended = [_vec("Edema")] * 3
    frame = compute_distinctiveness_scores(embeddings, intended, _distinct_config(k_neighbors=1))

    assert frame["distinctiveness_score"][2] > frame["distinctiveness_score"][0]
    assert frame["distinctiveness_score"][0] == pytest.approx(0.0, abs=1e-5)
    assert frame["distinctiveness_n_duplicates"][0] >= 1
    assert frame["distinctiveness_n_duplicates"][2] == 0


def test_duplicate_clusters_group_only_within_class_and_skip_singletons():
    """The pre-scoring dedup step: cluster near-identical images so ONE representative can be kept."""
    same = np.array([1.0, 0.0], dtype=np.float32)
    embeddings = np.stack([same, same.copy(), np.array([0.0, 1.0], dtype=np.float32), same.copy()])
    # index 3 is embedding-identical to 0 and 1 but belongs to a DIFFERENT class.
    intended = [_vec("Edema"), _vec("Edema"), _vec("Edema"), _vec("Fracture")]

    clusters = duplicate_clusters(embeddings, intended, duplicate_similarity=0.95)

    assert clusters == [[0, 1]], (
        "expected exactly one within-class duplicate cluster; a cross-class pair must never be "
        "merged, and singletons must not be reported"
    )


# ----------------------------------------------------------------------------------------------
# §4.4 Empirically derived expected regions
# ----------------------------------------------------------------------------------------------

def test_region_from_cam_mass_finds_a_tight_box_around_concentrated_attention():
    """A box derived from attention must actually surround where the attention is."""
    cam = np.zeros((20, 20), dtype=np.float32)
    cam[12:17, 4:9] = 1.0  # lower-left blob
    x0, y0, x1, y1 = region_from_cam_mass(cam, mass_fraction=0.9)

    assert 0.0 <= x0 < x1 <= 1.0 and 0.0 <= y0 < y1 <= 1.0
    assert x0 <= 4 / 20 and x1 >= 9 / 20, "box misses the blob horizontally"
    assert y0 <= 12 / 20 and y1 >= 17 / 20, "box misses the blob vertically"
    assert (x1 - x0) < 0.6 and (y1 - y0) < 0.6, "box is far larger than the attention it encloses"


def test_region_from_cam_mass_rejects_empty_attention():
    """Zero attention must raise, never silently return a whole-image box."""
    with pytest.raises(ValueError):
        region_from_cam_mass(np.zeros((8, 8), dtype=np.float32), mass_fraction=0.8)


def test_derived_regions_take_precedence_but_fall_back_per_label():
    """Precedence is per LABEL: a pathology the derivation skipped keeps its config box."""
    config = OmegaConf.create({"signals": {"explainability": {
        "region_mode": "pathology_specific",
        "baseline_box": [0.20, 0.15, 0.80, 0.85],
        "pathology_regions": {"Edema": [0.10, 0.10, 0.20, 0.20],
                              "Fracture": [0.70, 0.70, 0.90, 0.90]},
    }}})
    derived = {"Edema": [0.30, 0.30, 0.40, 0.40]}

    edema = expected_region_for({"Edema": 1}, config, derived)
    assert edema == (0.30, 0.30, 0.40, 0.40), "derived region should win for a derived label"

    fracture = expected_region_for({"Fracture": 1}, config, derived)
    assert fracture == (0.70, 0.70, 0.90, 0.90), "skipped label must keep its config box"

    # Passing nothing must reproduce the pre-existing behaviour exactly.
    assert expected_region_for({"Edema": 1}, config) == (0.10, 0.10, 0.20, 0.20)


if __name__ == "__main__":
    import traceback

    tests = [(name, value) for name, value in sorted(globals().items()) if name.startswith("test_")]
    passed, failed = 0, 0
    for name, function in tests:
        try:
            function()
            print(f"  PASS  {name}")
            passed += 1
        except Exception:
            print(f"  FAIL  {name}")
            traceback.print_exc()
            failed += 1
    print(f"\n{passed} passed, {failed} failed, {len(tests)} total")
    raise SystemExit(1 if failed else 0)


## `tests/test_learned_asism.py`

*1559 lines*

**The largest suite in the project (~1,500 lines), and the one that matters most** - it covers the
thesis's actual contribution, `scripts/asism/learned.py` and `models.py`.

Among the invariants it enforces: train-role and val-role subsets are image-disjoint; the subset
feasibility gate fails closed; target normalization is order-preserving; LOO and Banzhaf targets are
computed from the same measured subsets; the three-tier per-class governance rule selects the right
threshold source for each evidence level; acceptance criteria are enforced against frozen values;
the FreeMatch policy's leniency ordering follows prevalence; the per-class selection floor actually
raises admission; and the tie rule in `choose_full_policy` prefers the simpler policy within the
noise band.

This is the suite that was silently not running under the old hand-listed `run_all.py`.

In [ ]:
# ===== FILE: tests/test_learned_asism.py =====
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest
import torch
from omegaconf import OmegaConf

from scripts.asism.learned import (
    active_feature_columns,
    aggregate_hard_proxy_best_threshold,
    apply_feature_frame,
    banzhaf_msr_targets,
    bootstrap_class_contexts,
    build_controlled_subsets,
    build_role_conditioned_subsets,
    class_aware_context_vector,
    choose_full_policy,
    compute_verified_context_counts,
    contributing_signals,
    critic_proxy_correlation_per_class,
    determine_per_class_official_method,
    diversify_verification_candidates,
    eligible_classes_for_official_training,
    filter_targets_to_eligible_classes,
    enforce_acceptance_criteria,
    enforce_per_class_selection_floor,
    evaluate_subset_design_feasibility,
    freematch_style_percentile_per_class,
    freeze_acceptance_criteria,
    hard_threshold_grid_search,
    held_out_generalization_metrics,
    loo_vs_banzhaf_diagnostic,
    normalize_ranking_targets,
    pool_feasibility_report,
    real_class_support_context,
    resolve_critic_assisted_exploratory_targets,
    resolve_verified_only_targets,
    safe_feature_frame,
    scientific_status_for_namespace,
    spearman_with_reason,
    split_image_pool,
    validate_utility_results,
    verify_built_subsets,
)
from scripts.asism.models import (
    AdaptiveThresholdNetwork,
    MultiSignalUtilityRankingNetwork,
    SetUtilityNetwork,
    soft_selection_gate,
)


def _load_script(name: str):
    """Load a scripts/asism/*.py module whose filename starts with a digit (not import-able)."""
    path = Path(__file__).resolve().parents[1] / "scripts" / "asism" / name
    spec = importlib.util.spec_from_file_location(name.replace(".py", ""), path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def test_set_utility_is_permutation_invariant():
    torch.manual_seed(1)
    model = SetUtilityNetwork(3, (8, 4), (3,))
    model.eval()
    x = torch.randn(1, 5, 3); mask = torch.ones(1, 5, dtype=torch.bool)
    assert torch.allclose(model(x, mask), model(x[:, [3, 1, 4, 0, 2]], mask), atol=1e-6)


def test_active_feature_columns_excludes_gonogo_rejected_and_ablation_only_signals():
    configured = [
        "similarity_knn_mean", "iqa_composite", "uncertainty_mean_std",
        "explainability_region_overlap", "agreement_score",
    ]
    # Explainability is excluded and uncertainty is ablation-only, so neither
    # may be required by the primary learned selector's merged feature frame.
    assert active_feature_columns(configured, ["similarity", "iqa", "agreement"]) == [
        "similarity_knn_mean", "iqa_composite", "agreement_score",
    ]


def test_soft_gate_and_threshold_shapes():
    model = AdaptiveThresholdNetwork(4, 8, embedding_dim=3, hidden=(5,))
    thresholds = model(torch.tensor([0, 1]), torch.randn(2, 8))
    gate = soft_selection_gate(torch.tensor([0.2, 0.9]), thresholds, 0.2)
    assert thresholds.shape == gate.shape == (2,)
    assert bool(((thresholds >= 0) & (thresholds <= 1)).all())


def test_controlled_subset_design_has_all_families():
    frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(90)],
                          "a": np.linspace(0, 1, 90), "b": np.linspace(1, 0, 90)})
    records = build_controlled_subsets(frame, ["a", "b"], 20, [12], .2, .3, .5, 42)
    assert len(records) == 20
    assert {row["design"]["type"] for row in records} == {"matched_random", "single_signal", "mixed"}
    assert all(len(set(row["image_ids"])) == row["size"] for row in records)


def test_build_controlled_subsets_rejects_inconsistent_fractions():
    frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(90)],
                          "a": np.linspace(0, 1, 90), "b": np.linspace(1, 0, 90)})
    with pytest.raises(ValueError):
        build_controlled_subsets(frame, ["a", "b"], 20, [12], .2, .3, .4, 42)  # sums to .9


def test_build_controlled_subsets_honors_quantile_bins():
    frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(200)],
                          "a": np.linspace(0, 1, 200), "b": np.linspace(1, 0, 200)})
    records = build_controlled_subsets(frame, ["a", "b"], 20, [12], 0.0, 1.0, 0.0, 42, quantile_bins=5)
    bands = {row["design"]["quantile_band"] for row in records if row["design"]["type"] == "single_signal"}
    assert bands, "expected at least one single_signal design"
    assert bands.issubset(set(range(5)))


def test_normalization_and_measured_utility_contract():
    frame = pd.DataFrame({"x": [1.0, np.nan, 3.0], "y": [2.0, 2.0, 2.0]})
    normalized, _ = safe_feature_frame(frame, ["x", "y"])
    assert np.isfinite(normalized.to_numpy()).all()
    subsets = [{"subset_id": "s1"}]
    result = validate_utility_results(subsets, [{"subset_id": "s1", "real_only_macro_auroc": .7,
                                                "augmented_macro_auroc": .72, "fold": 0, "seed": 1}])
    assert abs(result.utility_delta.iloc[0] - .02) < 1e-9


def test_apply_feature_frame_reuses_frozen_stats_not_the_new_pool():
    """The real bug this guards against: re-fitting normalization at selection time silently drifts
    the ranker's inputs away from what it was trained on. apply_feature_frame must use the frozen
    fit stats regardless of what pool it is applied to."""
    pool_a = pd.DataFrame({
        "image_id": [f"a{i}" for i in range(50)],
        "x": np.linspace(0, 1, 50),
        "y": np.linspace(1, 2, 50),
    })
    _, stats = safe_feature_frame(pool_a, ["x", "y"])

    shared_row = {"image_id": "shared", "x": 0.42, "y": 1.7}
    rng = np.random.default_rng(0)
    pool_b = pd.concat([
        pd.DataFrame([shared_row]),
        pd.DataFrame({
            "image_id": [f"b{i}" for i in range(2000)],
            "x": rng.uniform(-5, 5, 2000),
            "y": rng.uniform(-5, 5, 2000),
        }),
    ], ignore_index=True)

    normalized_b = apply_feature_frame(pool_b, ["x", "y"], stats)
    shared_row_normalized = normalized_b.iloc[0]

    manual_x = (shared_row["x"] - stats["mean"]["x"]) / stats["std"]["x"]
    manual_y = (shared_row["y"] - stats["mean"]["y"]) / stats["std"]["y"]
    assert abs(shared_row_normalized["x"] - manual_x) < 1e-9
    assert abs(shared_row_normalized["y"] - manual_y) < 1e-9

    # Refitting on pool_b directly (the old, buggy behavior) would NOT match — proves
    # apply_feature_frame is actually reusing the frozen stats, not silently refitting.
    refit_on_b, _ = safe_feature_frame(pool_b, ["x", "y"])
    assert abs(refit_on_b.iloc[0]["x"] - shared_row_normalized["x"]) > 1e-3


def test_learned_ranking_score_matches_across_pools_through_full_network():
    """Production path end-to-end: apply_feature_frame -> MultiSignalUtilityRankingNetwork -> sigmoid
    (exactly what 06_learn_thresholds_select.py computes as `learned_ranking_score`). The same
    image with the same raw features, same frozen normalization stats, same ranking checkpoint,
    must get the exact same final score whether it sits in a 10-image pool or a 2000-image pool
    with a completely different score distribution."""
    pool_a = pd.DataFrame({
        "image_id": [f"a{i}" for i in range(50)],
        "x": np.linspace(0, 1, 50),
        "y": np.linspace(1, 2, 50),
    })
    _, stats = safe_feature_frame(pool_a, ["x", "y"])

    shared_row = {"image_id": "shared", "x": 0.42, "y": 1.7}
    rng = np.random.default_rng(0)
    small_pool = pd.concat([
        pd.DataFrame([shared_row]),
        pd.DataFrame({"image_id": [f"c{i}" for i in range(10)],
                     "x": rng.uniform(0, 1, 10), "y": rng.uniform(1, 2, 10)}),
    ], ignore_index=True)
    large_pool = pd.concat([
        pd.DataFrame([shared_row]),
        pd.DataFrame({"image_id": [f"b{i}" for i in range(2000)],
                     "x": rng.uniform(-5, 5, 2000), "y": rng.uniform(-5, 5, 2000)}),
    ], ignore_index=True)

    torch.manual_seed(7)
    ranker = MultiSignalUtilityRankingNetwork(input_dim=2, hidden=(4,), dropout=0.0)
    ranker.eval()

    def learned_ranking_score_for_shared_image(pool: pd.DataFrame) -> float:
        normalized = apply_feature_frame(pool, ["x", "y"], stats)
        with torch.no_grad():
            raw = ranker(torch.tensor(normalized.to_numpy(dtype=np.float32)))
        scores = 1.0 / (1.0 + np.exp(-raw.numpy()))
        position = int(np.flatnonzero(pool["image_id"].to_numpy() == "shared")[0])
        return float(scores[position])

    score_in_small_pool = learned_ranking_score_for_shared_image(small_pool)
    score_in_large_pool = learned_ranking_score_for_shared_image(large_pool)
    assert score_in_small_pool == pytest.approx(score_in_large_pool, abs=1e-9)


def test_real_class_support_context_uses_real_patient_prevalence():
    frame = pd.DataFrame({
        "patient_id": ["p1", "p1", "p2", "p3", "p4"],
        "Edema": [1.0, 1.0, 0.0, np.nan, -1.0],
    })
    context = real_class_support_context("dev", ["Edema"], frame=frame)
    assert context["namespace"] == "dev"
    assert context["scientific_status"] == "non-production / development-only"
    assert context["labels"]["Edema"]["positive_patients"] == 1  # p1
    assert context["labels"]["Edema"]["negative_patients"] == 1  # p2
    assert context["labels"]["Edema"]["real_prevalence"] == pytest.approx(0.5)


def test_real_class_support_context_handles_zero_support():
    frame = pd.DataFrame({"patient_id": ["p1", "p2"], "Fracture": [np.nan, -1.0]})
    context = real_class_support_context("dev", ["Fracture"], frame=frame)
    assert context["labels"]["Fracture"]["positive_patients"] == 0
    assert context["labels"]["Fracture"]["negative_patients"] == 0
    assert context["labels"]["Fracture"]["real_prevalence"] is None


def test_scientific_status_for_namespace_only_production_is_production():
    assert scientific_status_for_namespace("production") == "production"
    assert scientific_status_for_namespace("dev") == "non-production / development-only"
    assert scientific_status_for_namespace("dev-smoke-v1") == "non-production / development-only"


def _synthetic_candidate_fixture(n: int = 300, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "image_id": [f"img{i:04d}" for i in range(n)],
        "iqa_composite": rng.uniform(0, 1, n),
        "similarity_knn_mean": rng.uniform(0, 1, n),
    })


def test_split_image_pool_is_disjoint_and_seeded():
    frame = _synthetic_candidate_fixture(300)
    train_a, val_a = split_image_pool(frame, val_fraction=0.2, seed=1)
    assert set(train_a["image_id"]).isdisjoint(set(val_a["image_id"]))
    assert len(train_a) + len(val_a) == len(frame)
    assert abs(len(val_a) / len(frame) - 0.2) < 0.02

    train_b, val_b = split_image_pool(frame, val_fraction=0.2, seed=1)
    assert set(train_a["image_id"]) == set(train_b["image_id"])  # deterministic given the seed


def test_split_image_pool_rejects_invalid_fraction_and_duplicates():
    frame = _synthetic_candidate_fixture(50)
    with pytest.raises(ValueError):
        split_image_pool(frame, val_fraction=0.0, seed=1)
    with pytest.raises(ValueError):
        split_image_pool(frame, val_fraction=1.0, seed=1)
    duplicated = pd.concat([frame, frame.iloc[[0]]], ignore_index=True)
    with pytest.raises(ValueError):
        split_image_pool(duplicated, val_fraction=0.2, seed=1)


def test_build_role_conditioned_subsets_never_mixes_pools():
    # val pool is ~20% of 300 = ~60 images, split across 3 quantile bands (~20 each) by default —
    # sizes must fit within a single band since single_signal designs draw from one band, or
    # _sample now (correctly) raises on an insufficient pool instead of silently undersizing.
    frame = _synthetic_candidate_fixture(300)
    train_frame, val_frame = split_image_pool(frame, val_fraction=0.2, seed=3)
    records = build_role_conditioned_subsets(
        train_frame, val_frame, ["iqa_composite", "similarity_knn_mean"],
        train_total=6, val_total=4, sizes=[10, 15], random_fraction=0.5,
        single_fraction=0.5, mixed_fraction=0.0, seed=7,
    )
    train_ids = {image_id for r in records if r["role"] == "train" for image_id in r["image_ids"]}
    val_ids = {image_id for r in records if r["role"] == "val" for image_id in r["image_ids"]}
    assert train_ids.isdisjoint(val_ids)
    assert train_ids.issubset(set(train_frame["image_id"]))
    assert val_ids.issubset(set(val_frame["image_id"]))
    assert sum(1 for r in records if r["role"] == "train") == 6
    assert sum(1 for r in records if r["role"] == "val") == 4
    assert all(r["subset_id"].startswith(r["role"] + "_") for r in records)


_DEFAULT_THRESHOLDS = {
    "min_candidates_per_class": 20,
    "min_candidates_per_quantile_band": 20,
    "min_val_subsets": 10,
}


def test_pool_feasibility_report_flags_under_sized_val_pool_and_fails_closed():
    frame = _synthetic_candidate_fixture(100)
    intended = {row.image_id: {"Edema": 1} for _, row in frame.iloc[:30].iterrows()}
    report = pool_feasibility_report(
        frame, ["iqa_composite", "similarity_knn_mean"], intended, ["Edema", "Fracture"],
        subset_sizes=[50, 500], quantile_bins=3, val_fraction=0.2, seed=5,
        total_subsets=20, feasibility_thresholds=_DEFAULT_THRESHOLDS,
    )
    assert report["total_candidates"] == 100
    assert report["train_pool_size"] + report["val_pool_size"] == 100
    assert report["candidates_per_label"]["train"]["Edema"] + report["candidates_per_label"]["val"]["Edema"] == 30
    assert report["candidates_per_label"]["train"]["Fracture"] == 0
    assert report["val_pool_fraction_status"] == "provisional_default"
    assert report["train_val_overlap_count"] == 0

    # val pool is ~20 images: requesting a 500-image subset is NOT achievable without replacement,
    # and _sample never uses replacement — so this must surface as a hard failure, not a silent
    # smaller subset.
    val_500 = report["subset_size_achievability"]["val"]["500"]
    assert val_500["achievable_without_replacement"] is False
    assert val_500["achievable_size_if_smaller"] == report["val_pool_size"]
    # a 50-image subset from the ~80-image train pool IS achievable without replacement
    train_50 = report["subset_size_achievability"]["train"]["50"]
    assert train_50["achievable_without_replacement"] is True

    assert report["passed"] is False
    assert any("actual subset size below required" in reason for reason in report["failures"])
    # Fracture has zero candidates anywhere: must trip the min_candidates_per_class failure too.
    assert any("Fracture" in reason and "insufficient candidates per class" in reason
              for reason in report["failures"])


def test_production_scale_pool_supports_revised_subset_sizes_in_each_val_quantile_band():
    """~5k generated images leave ~1k validation images, or ~333 per quantile band.

    The revised largest subset (250) must therefore be feasible without replacement;
    the former 1000-image request was not.
    """
    frame = _synthetic_candidate_fixture(5000)
    intended = {str(image_id): {"Edema": 1} for image_id in frame["image_id"]}
    report = pool_feasibility_report(
        frame, ["iqa_composite", "similarity_knn_mean"], intended, ["Edema"],
        subset_sizes=[100, 200, 250], quantile_bins=3, val_fraction=0.2, seed=42,
        total_subsets=120, feasibility_thresholds=_DEFAULT_THRESHOLDS,
    )
    assert report["passed"] is True
    assert all("actual subset size below required" not in failure for failure in report["failures"])


def test_evaluate_subset_design_feasibility_flags_overlap_as_non_negotiable():
    report = {
        "train_val_overlap_count": 3,
        "candidates_per_label": {"train": {}, "val": {}},
        "candidates_per_quantile_band": {"train": {}, "val": {}},
        "subset_size_achievability": {"train": {}, "val": {}},
        "quantile_band_size_achievability": {"train": {}, "val": {}},
        "planned_val_subsets": 50,
    }
    failures = evaluate_subset_design_feasibility(report, _DEFAULT_THRESHOLDS)
    assert any("overlap" in reason for reason in failures)


def test_evaluate_subset_design_feasibility_passes_clean_report():
    report = {
        "train_val_overlap_count": 0,
        "candidates_per_label": {"train": {"Edema": 100}, "val": {"Edema": 30}},
        "candidates_per_quantile_band": {"train": {"x": {0: 40, 1: 40}}, "val": {"x": {0: 25, 1: 25}}},
        "subset_size_achievability": {
            "train": {"50": {"requested_size": 50, "pool_size": 200,
                             "achievable_without_replacement": True, "achievable_size_if_smaller": 50}},
            "val": {"50": {"requested_size": 50, "pool_size": 60,
                          "achievable_without_replacement": True, "achievable_size_if_smaller": 50}},
        },
        "quantile_band_size_achievability": {"train": {}, "val": {}},
        "planned_val_subsets": 20,
    }
    assert evaluate_subset_design_feasibility(report, _DEFAULT_THRESHOLDS) == []


def test_sample_raises_on_insufficient_pool_instead_of_silently_undersizing():
    """A single quantile band can be much smaller than the whole pool. single_signal/mixed designs
    drawing size=50 from a 5-image band must raise a clear invariant violation, never silently
    return a smaller subset (that must be caught by --phase feasibility before this is ever hit)."""
    small_frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(5)],
                                "a": np.linspace(0, 1, 5), "b": np.linspace(1, 0, 5)})
    # single_fraction=1.0 forces every design to draw from a single (small) quantile band.
    with pytest.raises(ValueError, match="Insufficient pool"):
        build_controlled_subsets(small_frame, ["a", "b"], 5, [50], 0.0, 1.0, 0.0, seed=1, quantile_bins=5)


def test_sample_mixed_design_backfills_partial_per_column_draws_but_still_enforces_total():
    """The 'mixed' design intentionally lets individual columns undershoot their quota (allow_partial)
    because it backfills from the whole frame afterward — this must still succeed when the frame as
    a whole has enough images, even though any single quantile band does not."""
    frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(300)],
                          "a": np.linspace(0, 1, 300), "b": np.linspace(1, 0, 300)})
    records = build_controlled_subsets(frame, ["a", "b"], 6, [80], 0.0, 0.0, 1.0, seed=2, quantile_bins=10)
    for record in records:
        assert record["design"]["type"] == "mixed"
        assert len(set(record["image_ids"])) == record["size"] == 80


def test_verify_built_subsets_detects_duplicate_images_within_a_subset():
    records = [{"subset_id": "train_0", "role": "train", "size": 2, "image_ids": ["i1", "i1"]}]
    with pytest.raises(SystemExit, match="duplicate image_id"):
        verify_built_subsets(records, train_total=1, val_total=0)


def test_verify_built_subsets_detects_size_mismatch():
    records = [{"subset_id": "train_0", "role": "train", "size": 5, "image_ids": ["i1", "i2"]}]
    with pytest.raises(SystemExit, match="declares size=5"):
        verify_built_subsets(records, train_total=1, val_total=0)


def test_verify_built_subsets_detects_train_val_overlap():
    records = [
        {"subset_id": "train_0", "role": "train", "size": 1, "image_ids": ["shared"]},
        {"subset_id": "val_0", "role": "val", "size": 1, "image_ids": ["shared"]},
    ]
    with pytest.raises(SystemExit, match="both a train-role and a val-role"):
        verify_built_subsets(records, train_total=1, val_total=1)


def test_verify_built_subsets_detects_wrong_subset_counts():
    records = [{"subset_id": "train_0", "role": "train", "size": 1, "image_ids": ["i1"]}]
    with pytest.raises(SystemExit, match="planned 2/1"):
        verify_built_subsets(records, train_total=2, val_total=1)


def test_verify_built_subsets_passes_clean_records():
    records = [
        {"subset_id": "train_0", "role": "train", "size": 2, "image_ids": ["i1", "i2"]},
        {"subset_id": "val_0", "role": "val", "size": 1, "image_ids": ["i3"]},
    ]
    verify_built_subsets(records, train_total=1, val_total=1)  # must not raise


def test_04_preflight_reports_missing_artifacts_not_a_raw_traceback():
    """A missing gonogo_report.json (or score/manifest artifacts) must produce an actionable
    SystemExit listing exactly what to run, never a bare FileNotFoundError traceback."""
    module = _load_script("04_build_utility_subsets.py")
    from scripts.utils.artifact_contracts import stage3_paths
    from scripts.utils.config import load_named_config

    cfg = load_named_config("stage3_asism.yaml", "stage3")
    fake_namespace = "unit-test-namespace-that-will-never-exist"
    for key, value in stage3_paths(cfg, fake_namespace).items():
        if key in cfg.paths:
            cfg.paths[key] = str(value)
    cfg.split_namespace = fake_namespace

    with pytest.raises(SystemExit) as excinfo:
        module.preflight_check_candidate_pool_inputs(cfg)
    message = str(excinfo.value)
    assert "UPSTREAM GATE" in message
    assert "01_compute_signals.py" in message
    assert "02_gonogo.py" in message
    assert "02_generate_synthetic_images.py" in message


def test_split_subsets_by_role_requires_role_field_and_splits_correctly():
    module = _load_script("05_train_learned_asism.py")
    subsets = [
        {"subset_id": "a", "image_ids": ["i1"], "role": "train"},
        {"subset_id": "b", "image_ids": ["i2"], "role": "val"},
        {"subset_id": "c", "image_ids": ["i3"], "role": "train"},
    ]
    train, val = module.split_subsets_by_role(subsets)
    assert [row["subset_id"] for row in train] == ["a", "c"]
    assert [row["subset_id"] for row in val] == ["b"]

    with pytest.raises(SystemExit):
        module.split_subsets_by_role([{"subset_id": "x", "image_ids": ["i1"]}])


def test_image_overlap_fraction():
    module = _load_script("05_train_learned_asism.py")
    disjoint_a = [{"image_ids": ["i1", "i2"]}]
    disjoint_b = [{"image_ids": ["i3", "i4"]}]
    assert module.image_overlap_fraction(disjoint_a, disjoint_b) == 0.0

    overlapping_a = [{"image_ids": ["i1", "i2"]}]
    overlapping_b = [{"image_ids": ["i2", "i3"]}]
    assert module.image_overlap_fraction(overlapping_a, overlapping_b) == pytest.approx(1 / 3)


def test_pairwise_ranking_accuracy_perfect_and_none_cases():
    module = _load_script("05_train_learned_asism.py")
    torch.manual_seed(0)

    class PerfectRanker(torch.nn.Module):
        def forward(self, x):
            return x[:, 0]

    features = {"a": np.array([0.1], dtype=np.float32), "b": np.array([0.5], dtype=np.float32),
               "c": np.array([0.9], dtype=np.float32)}
    targets = {"a": 0.0, "b": 1.0, "c": 2.0}  # perfectly correlated with feature order
    accuracy = module.pairwise_ranking_accuracy(PerfectRanker(), features, targets, torch.device("cpu"))
    assert accuracy == pytest.approx(1.0)

    assert module.pairwise_ranking_accuracy(PerfectRanker(), features, {"a": 1.0}, torch.device("cpu")) is None
    assert module.pairwise_ranking_accuracy(PerfectRanker(), features, {"a": 1.0, "b": 1.0}, torch.device("cpu")) is None


def test_train_set_model_reports_zero_overlap_and_validation_metrics_on_disjoint_fixture():
    module = _load_script("05_train_learned_asism.py")
    from scripts.asism.models import SetUtilityNetwork

    rng = np.random.default_rng(0)
    lookup = {f"img{i}": rng.normal(size=3).astype(np.float32) for i in range(40)}
    train_ids = [f"img{i}" for i in range(30)]
    val_ids = [f"img{i}" for i in range(30, 40)]

    def make_subsets(ids, prefix, n):
        rows = []
        for i in range(n):
            chosen = list(rng.choice(ids, size=6, replace=False))
            rows.append({"subset_id": f"{prefix}_{i}", "image_ids": chosen, "role": prefix})
        return rows

    train_subsets = make_subsets(train_ids, "train", 8)
    val_subsets = make_subsets(val_ids, "val", 4)
    utility_by_id = {row["subset_id"]: float(rng.uniform(-0.1, 0.1)) for row in train_subsets + val_subsets}

    config = OmegaConf.create({
        "learning_rate": 1e-3, "weight_decay": 0.0, "epochs": 5, "batch_size": 4,
        "utility_mse_weight": 1.0, "pairwise_ranking_weight": 0.0, "early_stopping_patience": 2,
    })
    torch.manual_seed(0)
    model = SetUtilityNetwork(3, (8, 4), (3,))
    validation = module.train_set_model(model, train_subsets, val_subsets, utility_by_id, lookup, config,
                                        torch.device("cpu"), min_val_subsets=4)
    assert validation["image_overlap_fraction"] == 0.0
    assert validation["n_train_subsets"] == 8
    assert validation["n_val_subsets"] == 4
    assert validation["mae"] is not None
    assert validation["spearman"] is not None
    assert validation["spearman_undefined_reason"] is None
    assert validation["early_stopped_at_epoch"] <= 5


def test_train_set_model_refuses_to_train_below_min_val_subsets():
    """Image-disjoint validation is a design invariant: too few val-role subsets must stop training
    with a clear message, never silently skip validation and keep going."""
    module = _load_script("05_train_learned_asism.py")
    from scripts.asism.models import SetUtilityNetwork

    rng = np.random.default_rng(1)
    lookup = {f"img{i}": rng.normal(size=3).astype(np.float32) for i in range(20)}
    train_subsets = [{"subset_id": f"train_{i}", "image_ids": [f"img{i}", f"img{i+1}"], "role": "train"}
                     for i in range(10)]
    val_subsets = [{"subset_id": "val_0", "image_ids": ["img18", "img19"], "role": "val"}]  # only 1
    utility_by_id = {row["subset_id"]: 0.01 for row in train_subsets + val_subsets}
    config = OmegaConf.create({
        "learning_rate": 1e-3, "weight_decay": 0.0, "epochs": 2, "batch_size": 4,
        "utility_mse_weight": 1.0, "pairwise_ranking_weight": 0.0, "early_stopping_patience": 2,
    })
    model = SetUtilityNetwork(3, (8, 4), (3,))
    with pytest.raises(SystemExit, match="val-role subsets"):
        module.train_set_model(model, train_subsets, val_subsets, utility_by_id, lookup, config,
                               torch.device("cpu"), min_val_subsets=4)


def test_train_set_model_refuses_to_train_with_zero_val_subsets():
    module = _load_script("05_train_learned_asism.py")
    from scripts.asism.models import SetUtilityNetwork

    train_subsets = [{"subset_id": "train_0", "image_ids": ["img0", "img1"], "role": "train"}]
    utility_by_id = {"train_0": 0.01}
    config = OmegaConf.create({
        "learning_rate": 1e-3, "weight_decay": 0.0, "epochs": 2, "batch_size": 4,
        "utility_mse_weight": 1.0, "pairwise_ranking_weight": 0.0, "early_stopping_patience": 2,
    })
    model = SetUtilityNetwork(3, (4,), (3,))
    with pytest.raises(SystemExit, match="val-role subsets"):
        module.train_set_model(model, train_subsets, [], utility_by_id,
                               {"img0": np.zeros(3, dtype=np.float32), "img1": np.zeros(3, dtype=np.float32)},
                               config, torch.device("cpu"), min_val_subsets=4)


def test_train_set_model_reports_undefined_spearman_as_none_with_reason():
    """Constant predictions/targets make Spearman correlation undefined (scipy returns NaN). NaN is
    not valid JSON and must never reach the frozen manifest — this must surface as None + an
    explicit machine-readable reason."""
    module = _load_script("05_train_learned_asism.py")
    from scripts.asism.models import SetUtilityNetwork

    rng = np.random.default_rng(2)
    lookup = {f"img{i}": rng.normal(size=3).astype(np.float32) for i in range(30)}
    train_ids = [f"img{i}" for i in range(20)]
    val_ids = [f"img{i}" for i in range(20, 30)]

    def make_subsets(ids, prefix, n):
        return [{"subset_id": f"{prefix}_{i}", "image_ids": list(rng.choice(ids, size=4, replace=False)),
                "role": prefix} for i in range(n)]

    train_subsets = make_subsets(train_ids, "train", 6)
    val_subsets = make_subsets(val_ids, "val", 4)
    # Constant measured utility for every val subset -> targets have zero variance -> Spearman is
    # mathematically undefined regardless of what the model predicts.
    utility_by_id = {row["subset_id"]: 0.0123 for row in train_subsets + val_subsets}
    config = OmegaConf.create({
        "learning_rate": 1e-3, "weight_decay": 0.0, "epochs": 3, "batch_size": 4,
        "utility_mse_weight": 1.0, "pairwise_ranking_weight": 0.0, "early_stopping_patience": 2,
    })
    torch.manual_seed(0)
    model = SetUtilityNetwork(3, (8, 4), (3,))
    validation = module.train_set_model(model, train_subsets, val_subsets, utility_by_id, lookup, config,
                                        torch.device("cpu"), min_val_subsets=4)
    assert validation["spearman"] is None
    assert validation["spearman_undefined_reason"] == "constant_predictions_or_targets"
    assert validation["mae"] is not None  # MAE is still well-defined even when Spearman isn't


def test_subset_design_config_hash_changes_with_relevant_fields_only():
    module = _load_script("04_build_utility_subsets.py")
    base = {
        "subset_sizes": [250, 500], "total_subsets": 120, "random_fraction": 0.2,
        "single_signal_fraction": 0.3, "mixed_fraction": 0.5, "quantile_bins": 3,
        "val_pool_fraction": 0.2, "feasibility_thresholds": dict(_DEFAULT_THRESHOLDS),
        "seed": 42, "minimum_image_exposures": 3,  # irrelevant fields present but not hashed
    }
    design_a = OmegaConf.create(base)
    design_b = OmegaConf.create(base)
    assert module.subset_design_config_hash(design_a) == module.subset_design_config_hash(design_b)

    changed = dict(base); changed["total_subsets"] = 200
    design_c = OmegaConf.create(changed)
    assert module.subset_design_config_hash(design_a) != module.subset_design_config_hash(design_c)


def test_compute_budget_gate_raises_when_over_budget_and_passes_when_within():
    module = _load_script("04b_evaluate_utility_subsets.py")
    cfg = OmegaConf.create({"learned_asism": {"compute_budget": {
        "max_gpu_hours": 1.0, "hours_per_proxy_run_estimate": 0.15,
    }}})

    over_budget = module.compute_budget_estimate(cfg, n_recipes=100)
    assert over_budget["within_budget"] is False
    with pytest.raises(SystemExit):
        module.enforce_compute_budget(cfg, n_recipes=100)

    within_budget = module.compute_budget_estimate(cfg, n_recipes=2)
    assert within_budget["within_budget"] is True
    module.enforce_compute_budget(cfg, n_recipes=2)  # must not raise


# ---------------------------------------------------------------------------
# Phase 2 — Adaptive Threshold Learning (code + fixtures + unit tests only, no GPU/proxy runs)
# ---------------------------------------------------------------------------

def test_class_aware_context_vector_shape_and_known_slots():
    vector = class_aware_context_vector(
        [0.1, 0.5, 0.9], {"real_prevalence": 0.3, "positive_patients": 50},
        {"target_synthetic_to_real_ratio": 1.5},
    )
    assert len(vector) == 10
    assert vector[7] == pytest.approx(0.3)   # real_prevalence slot
    assert vector[9] == pytest.approx(1.5)   # target_synthetic_to_real_ratio slot


def test_class_aware_context_vector_handles_empty_scores_and_missing_prevalence():
    vector = class_aware_context_vector([], {"real_prevalence": None}, {})
    assert len(vector) == 10
    assert all(np.isfinite(value) for value in vector)


def test_bootstrap_class_contexts_tags_honestly_and_stays_within_one_class_and_pool():
    frame = pd.DataFrame({"image_id": [f"img{i}" for i in range(50)],
                          "learned_ranking_score": np.linspace(0, 1, 50)})
    intended = {f"img{i}": {"Edema": 1} for i in range(30)}
    real_prevalence = {"real_prevalence": 0.2, "positive_patients": 100}
    budget_context = {"target_synthetic_to_real_ratio": 1.0, "min_selected_per_label": 10, "max_selected_per_label": 500}
    contexts = bootstrap_class_contexts(
        frame, class_id=3, label="Edema", intended_by_id=intended, score_column="learned_ranking_score",
        context_source="bootstrap_train_pool", n_replicates=5, min_fraction=0.3, max_fraction=0.8,
        ranking_checkpoint_hash="ckpt-rank", critic_checkpoint_hash="ckpt-critic",
        real_prevalence_context=real_prevalence, budget_context=budget_context, seed=1,
    )
    assert len(contexts) == 5
    edema_positive_ids = {f"img{i}" for i in range(30)}
    for context in contexts:
        assert context["context_source"] == "bootstrap_train_pool"
        assert context["independent_clinical_sample"] is False
        assert context["class_id"] == 3 and context["label"] == "Edema"
        assert set(context["image_ids"]).issubset(edema_positive_ids)
        assert len(set(context["image_ids"])) == len(context["image_ids"])  # no duplicate images
        assert len(context["context_features"]) == 10


def test_bootstrap_class_contexts_rejects_invalid_fraction_range():
    frame = pd.DataFrame({"image_id": ["a"], "score": [0.5]})
    with pytest.raises(ValueError):
        bootstrap_class_contexts(frame, 0, "L", {}, "score", "bootstrap_train_pool", 1, 0.8, 0.3,
                                 "h1", "h2", {}, {}, seed=1)


def test_hard_threshold_grid_search_is_deterministic_and_names_critic_field_explicitly():
    ranking_scores = {"a": 0.9, "b": 0.7, "c": 0.5, "d": 0.3}
    image_ids = list(ranking_scores)

    def critic_fn(selected):
        return len(selected) * 0.1

    t_grid = [0.2, 0.4, 0.6, 0.8]
    result_a = hard_threshold_grid_search(critic_fn, ranking_scores, image_ids, t_grid, 0.1, max_selected=10)
    result_b = hard_threshold_grid_search(critic_fn, ranking_scores, image_ids, t_grid, 0.1, max_selected=10)
    assert result_a == result_b
    assert all("critic_predicted_utility" in entry for entry in result_a)
    assert all("measured_utility" not in entry for entry in result_a)  # never conflated with a measured field


def test_hard_threshold_grid_search_rejects_all_empty_thresholds():
    with pytest.raises(ValueError, match="empty subset"):
        hard_threshold_grid_search(lambda selected: 1.0, {"a": 0.1}, ["a"], [0.99], 0.1, max_selected=10)


def test_hard_threshold_grid_search_respects_max_selected():
    ranking_scores = {f"i{i}": 1.0 for i in range(20)}

    def critic_fn(selected):
        return float(len(selected))

    result = hard_threshold_grid_search(critic_fn, ranking_scores, list(ranking_scores), [0.5], 0.0, max_selected=5)
    assert result[0]["selected_count"] == 5


def test_resolve_verified_only_targets_never_includes_critic_only_contexts():
    """The core separation: a context with zero proxy measurements must be entirely ABSENT from
    the official path's targets — never present with a critic-derived guess at any weight."""
    contexts = [
        {"context_id": "c1", "class_id": 0, "label": "Edema", "context_source": "bootstrap_train_pool"},
        {"context_id": "c2", "class_id": 0, "label": "Edema", "context_source": "bootstrap_train_pool"},
    ]
    measurements = [
        {"context_id": "c1", "candidate_threshold": 0.6, "measured_utility": 0.05},
        {"context_id": "c1", "candidate_threshold": 0.7, "measured_utility": 0.08},
        # c2 has NO measurements at all.
    ]
    targets = resolve_verified_only_targets(contexts, measurements)
    assert len(targets) == 1
    assert targets[0]["context_id"] == "c1"
    assert targets[0]["target_threshold"] == 0.7  # best AMONG VERIFIED, not a claimed global optimum
    assert targets[0]["proxy_best_among_verified_candidates"] == 0.7
    assert targets[0]["n_verified_candidates_in_context"] == 2
    assert targets[0]["target_source"] == "proxy_verified"
    assert "measured_best_threshold" not in targets[0]  # renamed away deliberately


def test_resolve_critic_assisted_exploratory_targets_never_touches_measurements():
    """The exploratory path is a completely separate function signature — it cannot even be called
    with a measurements list, so it structurally cannot leak proxy-verified data into itself, and
    its output must never be fed into official-path training."""
    contexts = [{"context_id": "c1", "class_id": 0, "label": "Edema", "context_source": "bootstrap_train_pool"}]
    evaluations = [
        {"context_id": "c1", "candidate_threshold": 0.6, "critic_predicted_utility": 0.05, "rank_within_context": 1},
        {"context_id": "c1", "candidate_threshold": 0.7, "critic_predicted_utility": 0.03, "rank_within_context": 2},
    ]
    targets = resolve_critic_assisted_exploratory_targets(contexts, evaluations)
    assert targets[0]["target_source"] == "critic_only"
    assert targets[0]["target_threshold"] == 0.6  # critic's own top grid pick
    assert targets[0]["supervision_weight"] == 1.0  # full weight WITHIN the exploratory path only


def test_determine_per_class_official_method_zero_verified_falls_back_to_baseline_not_hard():
    """The bug this guards against: claiming 'hard_class_specific_proxy_verified_thresholds' when
    zero proxy verification ever happened is a contradiction in terms. Zero verified must fall back
    to the honestly-named OLD baseline instead."""
    result = determine_per_class_official_method([], [], ["Edema"], min_verified_contexts_per_class=3,
                                                  acceptance_passed=True)
    assert result["Edema"] == "fixed_target_ratio_threshold_distillation_baseline_v1"


def test_determine_per_class_official_method_some_but_insufficient_verified():
    train_targets = [{"label": "Edema"}]  # only 1, below min_verified_contexts_per_class=3
    result = determine_per_class_official_method(train_targets, [], ["Edema"],
                                                  min_verified_contexts_per_class=3, acceptance_passed=True)
    assert result["Edema"] == "hard_proxy_best_among_verified"


def test_determine_per_class_official_method_requires_both_train_and_held_out_sufficiency():
    """Enough verified TRAIN contexts but insufficient verified HELD-OUT (image-disjoint validation)
    contexts must still fall back — 'enough evidence' means enough on both sides, not just train."""
    train_targets = [{"label": "Edema"}] * 5
    held_out_targets = [{"label": "Edema"}] * 1  # below min=3
    result = determine_per_class_official_method(train_targets, held_out_targets, ["Edema"],
                                                  min_verified_contexts_per_class=3, acceptance_passed=True)
    assert result["Edema"] == "hard_proxy_best_among_verified"


def test_determine_per_class_official_method_network_only_when_sufficient_and_accepted():
    train_targets = [{"label": "Edema"}] * 5
    held_out_targets = [{"label": "Edema"}] * 5
    passed = determine_per_class_official_method(train_targets, held_out_targets, ["Edema"], 3, acceptance_passed=True)
    assert passed["Edema"] == "adaptive_threshold_network"
    failed = determine_per_class_official_method(train_targets, held_out_targets, ["Edema"], 3, acceptance_passed=False)
    assert failed["Edema"] == "hard_proxy_best_among_verified"  # enough data, but acceptance criteria failed


def test_diversify_verification_candidates_includes_more_than_critic_top_k():
    evaluations = [
        {"candidate_threshold": t, "selected_count": count, "rank_within_context": rank}
        for rank, (t, count) in enumerate(
            [(0.9, 5), (0.8, 20), (0.7, 50), (0.5, 100), (0.3, 150), (0.1, 195)], start=1)
    ]
    diversified = diversify_verification_candidates(evaluations, top_k=1, quantile_fractions=[0.5], candidate_pool_size=200)
    reasons = {reason for entry in diversified for reason in entry["selection_reasons"]}
    assert "critic_top_k" in reasons
    assert "quantile_spaced" in reasons
    assert "boundary" in reasons
    assert "baseline_target_ratio" in reasons
    assert len(diversified) > 1  # never just the single critic top-1 pick


def test_diversify_verification_candidates_handles_empty_input():
    assert diversify_verification_candidates([], top_k=2, quantile_fractions=[0.5], candidate_pool_size=100) == []


def test_held_out_generalization_metrics_zero_when_predicted_equals_target():
    scores = {"a": 0.9, "b": 0.5, "c": 0.1}
    metrics = held_out_generalization_metrics(0.5, 0.5, scores)
    assert metrics["absolute_threshold_error"] == 0.0
    assert metrics["selected_set_jaccard"] == 1.0


def test_held_out_generalization_metrics_detects_divergence():
    scores = {"a": 0.9, "b": 0.5, "c": 0.1}
    metrics = held_out_generalization_metrics(0.05, 0.5, scores)
    assert metrics["absolute_threshold_error"] == pytest.approx(0.45)
    assert metrics["selected_set_jaccard"] == pytest.approx(2 / 3)


def test_freeze_and_enforce_acceptance_criteria_pass_and_fail():
    criteria = {"critic_proxy_median_spearman_min": 0.3, "median_critic_predicted_utility_regret_max": 0.01,
               "threshold_stability_std_max": 0.05}
    frozen = freeze_acceptance_criteria(criteria)
    assert frozen["criteria_status"] == "provisional_pre_registered_defaults"
    assert frozen["criteria_frozen_at"]
    assert frozen["criteria_config_hash"]
    assert "critic_proxy_median_spearman_min" in frozen["criteria_definitions"]

    passing = enforce_acceptance_criteria(
        {"median_spearman": 0.5, "median_critic_predicted_utility_regret": 0.005, "threshold_stability_std": 0.02}, frozen)
    assert passing["passed"] is True
    assert passing["criteria_status"] == "provisional_pre_registered_defaults"

    failing = enforce_acceptance_criteria(
        {"median_spearman": 0.1, "median_critic_predicted_utility_regret": 0.005, "threshold_stability_std": 0.02}, frozen)
    assert failing["passed"] is False
    assert failing["per_criterion"]["critic_proxy_median_spearman_min"]["passed"] is False


def test_enforce_acceptance_criteria_none_evidence_fails_not_crashes():
    frozen = freeze_acceptance_criteria({"critic_proxy_median_spearman_min": 0.3})
    result = enforce_acceptance_criteria({"median_spearman": None}, frozen)
    assert result["passed"] is False


def test_critic_proxy_correlation_per_class_never_pools_as_the_primary_number():
    contexts = [
        {"context_id": "c1", "label": "Edema"}, {"context_id": "c2", "label": "Edema"},
        {"context_id": "c3", "label": "Edema"},
        {"context_id": "c4", "label": "Fracture"}, {"context_id": "c5", "label": "Fracture"},
    ]
    evaluations = [
        {"context_id": cid, "candidate_threshold": 0.5, "critic_predicted_utility": value}
        for cid, value in [("c1", 0.1), ("c2", 0.2), ("c3", 0.3), ("c4", 0.9), ("c5", 0.1)]
    ]
    measurements = [
        {"context_id": cid, "candidate_threshold": 0.5, "measured_utility": value}
        for cid, value in [("c1", 0.05), ("c2", 0.15), ("c3", 0.25), ("c4", 0.05), ("c5", 0.5)]
    ]
    result = critic_proxy_correlation_per_class(evaluations, measurements, contexts)
    assert set(result["per_class_spearman"]) <= {"Edema", "Fracture"}
    assert "pooled_spearman_diagnostic_only" in result
    assert result["median_spearman"] is not None or result["macro_spearman"] is not None


def test_critic_proxy_correlation_per_class_needs_two_points_per_class():
    contexts = [{"context_id": "c1", "label": "Edema"}]
    evaluations = [{"context_id": "c1", "candidate_threshold": 0.5, "critic_predicted_utility": 0.1}]
    measurements = [{"context_id": "c1", "candidate_threshold": 0.5, "measured_utility": 0.05}]
    result = critic_proxy_correlation_per_class(evaluations, measurements, contexts)
    assert result["per_class_spearman"]["Edema"] is None
    assert result["n_verified_by_class"]["Edema"] == 1


def test_train_threshold_network_zero_supervision_weight_means_no_learning():
    module = _load_script("08_train_threshold_network.py")
    torch.manual_seed(0)
    targets = [{"class_id": 0, "context_features": [0.1] * 4, "target_threshold": 0.9, "supervision_weight": 0.0}]
    model = AdaptiveThresholdNetwork(2, 4, embedding_dim=2, hidden=(4,))
    before = module.predict_thresholds(model, targets, torch.device("cpu"))
    module.train_threshold_network(model, targets, torch.device("cpu"), epochs=20, learning_rate=0.1)
    after = module.predict_thresholds(model, targets, torch.device("cpu"))
    assert before == pytest.approx(after)  # zero supervision_weight -> zero gradient -> no parameter change


def test_train_threshold_network_positive_weight_moves_prediction_toward_target():
    module = _load_script("08_train_threshold_network.py")
    torch.manual_seed(0)
    targets = [{"class_id": 0, "context_features": [0.1] * 4, "target_threshold": 0.9, "supervision_weight": 1.0}]
    model = AdaptiveThresholdNetwork(2, 4, embedding_dim=2, hidden=(4,))
    before = module.predict_thresholds(model, targets, torch.device("cpu"))[0]
    module.train_threshold_network(model, targets, torch.device("cpu"), epochs=200, learning_rate=0.05)
    after = module.predict_thresholds(model, targets, torch.device("cpu"))[0]
    assert abs(after - 0.9) < abs(before - 0.9)


def test_train_threshold_network_raises_on_empty_targets():
    module = _load_script("08_train_threshold_network.py")
    model = AdaptiveThresholdNetwork(2, 4)
    with pytest.raises(ValueError):
        module.train_threshold_network(model, [], torch.device("cpu"), epochs=5, learning_rate=0.01)


def test_threshold_stability_computes_per_class_std_excluding_singletons():
    module = _load_script("08_train_threshold_network.py")
    targets = [{"class_id": 0}, {"class_id": 0}, {"class_id": 1}]
    predictions = [0.5, 0.7, 0.3]
    result = module.threshold_stability(targets, predictions)
    assert 0 in result["per_class_std"]
    assert 1 not in result["per_class_std"]  # only one context for class 1 -> std undefined, excluded


def test_07_writes_frozen_artifacts_with_exclusive_create_mode():
    """threshold_contexts.jsonl / threshold_candidate_evaluations.jsonl / proxy_verification_plan.json
    must be write-once (07's own critic-only artifacts) — enforced at the open() call, not by
    convention."""
    source = (Path(__file__).resolve().parents[1] / "scripts" / "asism" / "07_build_threshold_contexts.py").read_text(encoding="utf-8")
    assert 'open(Path(cfg.paths.threshold_contexts), "x"' in source
    assert 'open(Path(cfg.paths.threshold_candidate_evaluations), "x"' in source
    assert 'open(Path(cfg.paths.proxy_verification_plan), "x"' in source


def test_07b_never_opens_threshold_candidate_evaluations_for_writing():
    """07b must never mutate 07's frozen critic-only artifact — it writes measurements to a
    completely separate, resumable (append) file instead."""
    source = (Path(__file__).resolve().parents[1] / "scripts" / "asism" / "07b_verify_thresholds_proxy.py").read_text(encoding="utf-8")
    assert 'open(Path(cfg.paths.threshold_candidate_evaluations)' not in source
    assert 'open(measurements_path, "a"' in source


def test_verification_compute_estimate_is_per_context_only_not_full_policy():
    """07's budget must NOT include full-policy runs — that has its own independent budget in 08b."""
    module = _load_script("07_build_threshold_contexts.py")
    cfg = OmegaConf.create({"learned_asism": {"threshold_network": {
        "verification_compute_budget": {"max_gpu_hours": 100.0, "hours_per_proxy_run_estimate": 0.1},
    }}})
    estimate = module.verification_compute_estimate(cfg, n_planned=10)
    assert "n_planned_full_policy_runs" not in estimate
    assert "total_planned_runs" not in estimate
    assert estimate["n_planned_threshold_evaluations"] == 10
    assert estimate["estimated_gpu_hours"] == pytest.approx(1.0)


def test_plan_full_policy_verification_is_plan_only_with_independent_budget():
    module = _load_script("08_train_threshold_network.py")
    full_policy_cfg = OmegaConf.create({
        "n_seeds": 3,
        "compute_budget": {"max_gpu_hours": 10.0, "hours_per_proxy_run_estimate": 0.15},
    })
    per_class_official_method = {"Edema": "adaptive_threshold_network", "Fracture": "hard_proxy_best_among_verified"}
    plan = module.plan_full_policy_verification(per_class_official_method, full_policy_cfg)
    assert "fixed_target_ratio_threshold_distillation_baseline_v1" in plan["policy_variants"]
    assert "literal_top_50_percent" in plan["policy_variants"]
    assert "freematch_style_adaptive_percentile" in plan["policy_variants"]
    assert "adaptive_threshold_network" in plan["policy_variants"]
    assert plan["status"] == "plan_only_not_executed"
    assert plan["estimated_runs"] == len(plan["policy_variants"]) * 3
    assert plan["estimated_gpu_hours"] == pytest.approx(plan["estimated_runs"] * 0.15)
    assert "08b_verify_full_policy_proxy.py" in plan["next_step"]


def test_08b_compute_budget_is_independent_of_07b():
    """08b's estimate must use full_policy_verification.compute_budget, a config block entirely
    separate from 07b's verification_compute_budget."""
    module = _load_script("08b_verify_full_policy_proxy.py")
    cfg = OmegaConf.create({"learned_asism": {"threshold_network": {"full_policy_verification": {
        "compute_budget": {"max_gpu_hours": 1.0, "hours_per_proxy_run_estimate": 0.1},
    }}}})
    over_budget = module.compute_budget_estimate(cfg, n_variants=5, n_seeds=3)
    assert over_budget["within_budget"] is False
    with pytest.raises(SystemExit):
        module.enforce_compute_budget(cfg, n_variants=5, n_seeds=3)

    within_budget = module.compute_budget_estimate(cfg, n_variants=1, n_seeds=1)
    assert within_budget["within_budget"] is True
    module.enforce_compute_budget(cfg, n_variants=1, n_seeds=1)  # must not raise


def test_08b_refuses_real_training_without_confirmation_flag(monkeypatch):
    module = _load_script("08b_verify_full_policy_proxy.py")
    monkeypatch.setattr(sys, "argv", ["08b_verify_full_policy_proxy.py", "--phase", "run"])
    with pytest.raises(SystemExit, match="REFUSED"):
        module.main()


def test_08b_phase_estimate_does_not_require_confirmation_flag(monkeypatch):
    module = _load_script("08b_verify_full_policy_proxy.py")
    monkeypatch.setattr(sys, "argv", ["08b_verify_full_policy_proxy.py", "--phase", "estimate",
                                      "--namespace", "unit-test-namespace-that-will-never-exist"])
    with pytest.raises(SystemExit, match="UPSTREAM GATE"):
        module.main()


def test_08b_phase_run_with_confirmation_flag_reaches_the_next_real_gate(monkeypatch):
    module = _load_script("08b_verify_full_policy_proxy.py")
    monkeypatch.setattr(sys, "argv", [
        "08b_verify_full_policy_proxy.py", "--phase", "run", "--i-understand-this-trains-real-models",
        "--namespace", "unit-test-namespace-that-will-never-exist",
    ])
    with pytest.raises(SystemExit, match="UPSTREAM GATE"):
        module.main()


def test_compute_verified_context_counts_and_eligibility():
    module_targets = [
        {"label": "Edema"}, {"label": "Edema"}, {"label": "Edema"},
        {"label": "Fracture"},
    ]
    held_out_targets = [
        {"label": "Edema"}, {"label": "Edema"}, {"label": "Edema"},
        {"label": "Fracture"}, {"label": "Fracture"}, {"label": "Fracture"},
    ]
    train_counts, held_out_counts = compute_verified_context_counts(module_targets, held_out_targets, ["Edema", "Fracture"])
    assert train_counts == {"Edema": 3, "Fracture": 1}
    assert held_out_counts == {"Edema": 3, "Fracture": 3}
    eligible = eligible_classes_for_official_training(train_counts, held_out_counts, min_verified_contexts_per_class=3)
    assert eligible == {"Edema"}  # Fracture fails on train_count even though held_out is sufficient


def test_aggregate_hard_proxy_best_threshold_uses_train_only_never_held_out():
    verified_train = [
        {"label": "Edema", "target_threshold": 0.5, "context_source": "bootstrap_train_pool"},
        {"label": "Edema", "target_threshold": 0.7, "context_source": "bootstrap_train_pool"},
        {"label": "Edema", "target_threshold": 0.6, "context_source": "bootstrap_train_pool"},
    ]
    result = aggregate_hard_proxy_best_threshold(verified_train, "Edema")
    assert result == pytest.approx(0.6)  # median
    assert aggregate_hard_proxy_best_threshold(verified_train, "Fracture") is None


def test_determine_per_class_official_method_zero_train_falls_back_even_with_held_out_data():
    """The exact bug this guards against: held-out measurements existing must NEVER cause anything
    other than the fixed-ratio baseline when there is zero verified TRAIN evidence."""
    held_out_only = [{"label": "Edema"}] * 5  # plenty of held-out, but...
    result = determine_per_class_official_method([], held_out_only, ["Edema"],
                                                  min_verified_contexts_per_class=3, acceptance_passed=True)
    assert result["Edema"] == "fixed_target_ratio_threshold_distillation_baseline_v1"


def test_compute_official_acceptance_evidence_uses_utility_regret_not_threshold_error():
    module = _load_script("08_train_threshold_network.py")
    held_out_generalization = [
        {"absolute_threshold_error": 0.9, "critic_predicted_utility_regret": 0.01},
        {"absolute_threshold_error": 0.8, "critic_predicted_utility_regret": 0.03},
    ]
    correlation = {"median_spearman": 0.5}
    stability = {"median_std": 0.02}
    evidence = module.compute_official_acceptance_evidence(held_out_generalization, correlation, stability)
    assert evidence["median_critic_predicted_utility_regret"] == pytest.approx(0.02)


def test_compute_official_acceptance_evidence_ignores_infinite_regret():
    module = _load_script("08_train_threshold_network.py")
    held_out_generalization = [
        {"critic_predicted_utility_regret": float("inf")},
        {"critic_predicted_utility_regret": 0.02},
    ]
    evidence = module.compute_official_acceptance_evidence(held_out_generalization, {"median_spearman": None}, {"median_std": None})
    assert evidence["median_critic_predicted_utility_regret"] == pytest.approx(0.02)


def test_save_checkpoint_writes_file_and_returns_hash():
    module = _load_script("08_train_threshold_network.py")
    model = AdaptiveThresholdNetwork(2, 4, embedding_dim=2, hidden=(4,))
    tn_cfg = OmegaConf.create({"class_embedding_dim": 2, "hidden_dims": [4], "dropout": 0.1})
    path = Path(__file__).with_name("checkpoint_test_artifact.pt")
    try:
        checkpoint_hash = module.save_checkpoint(model, path, "verified_only_official", tn_cfg, context_dim=4)
        assert path.is_file()
        assert len(checkpoint_hash) == 64
        loaded = torch.load(path, map_location="cpu", weights_only=True)
        assert loaded["path"] == "verified_only_official"
        assert loaded["context_dim"] == 4
    finally:
        path.unlink(missing_ok=True)


def test_official_targets_are_filtered_to_eligible_classes_only():
    targets = [{"label": "Edema", "context_id": "e"}, {"label": "Fracture", "context_id": "f"}]
    assert filter_targets_to_eligible_classes(targets, {"Edema"}) == [targets[0]]


def test_full_policy_choice_requires_complete_results_and_uses_frozen_tie_order():
    rows = [
        {"policy": "adaptive", "seed": 42, "measured_utility": 0.020},
        {"policy": "adaptive", "seed": 43, "measured_utility": 0.020},
        {"policy": "baseline", "seed": 42, "measured_utility": 0.019},
        {"policy": "baseline", "seed": 43, "measured_utility": 0.019},
    ]
    decision = choose_full_policy(rows, ["adaptive", "baseline"], [42, 43], 0.002, ["baseline", "adaptive"])
    assert decision["winning_policy"] == "baseline"
    with pytest.raises(ValueError, match="incomplete"):
        choose_full_policy(rows[:-1], ["adaptive", "baseline"], [42, 43], 0.002, ["baseline", "adaptive"])


def test_08b_requires_frozen_prevalence_context_and_rejects_empty_policy():
    module = _load_script("08b_verify_full_policy_proxy.py")
    contexts = [
        {"label": "Edema", "real_prevalence_context": {"real_prevalence": 0.2, "positive_patients": 10}},
        {"label": "Edema", "real_prevalence_context": {"real_prevalence": 0.2, "positive_patients": 10}},
    ]
    result = module.real_prevalence_contexts_from_threshold_contexts(contexts, ["Edema"])
    assert result["Edema"]["real_prevalence"] == pytest.approx(0.2)
    with pytest.raises(ValueError, match="missing frozen"):
        module.real_prevalence_contexts_from_threshold_contexts([], ["Edema"])
    with pytest.raises(ValueError, match="zero synthetic"):
        module.validate_policy_selection("adaptive", {"Edema": 0.5}, [], ["Edema"])


def test_stage4_routes_the_thesis_selected_synthetic_condition_to_learned_asism():
    source = (Path(__file__).resolve().parents[1] / "scripts" / "classify" / "01_train_conditions.py").read_text(
        encoding="utf-8"
    )
    assert 'CONDITIONS = ["A", "B", "C"]' in source
    assert 'condition == "C"' in source and 'selector="adaptive"' in source
    assert 'conditions: [A, B, C]' in (Path(__file__).resolve().parents[1] / "configs" / "stage4_classifier.yaml").read_text(encoding="utf-8")


def test_finalizer_never_reads_final_eval_and_publishes_new_adaptive_artifacts():
    source = (Path(__file__).resolve().parents[1] / "scripts" / "asism" / "09_finalize_learned_selection.py").read_text(
        encoding="utf-8"
    )
    assert "load_split(" not in source
    assert "cfg.paths.adaptive_selected_manifest" in source
    assert "cfg.paths.adaptive_selection_manifest" in source
    assert "cfg.paths.learned_selected_manifest" in source  # read-only baseline input


def test_08_writes_two_separate_target_files_not_one_combined_file():
    """Verified and exploratory targets must be written to physically separate files — a loader
    that opens the wrong one gets a clean 'file not found', not silently-wrong rows from a shared
    file it forgot to filter."""
    source = (Path(__file__).resolve().parents[1] / "scripts" / "asism" / "08_train_threshold_network.py").read_text(encoding="utf-8")
    assert 'open(Path(cfg.paths.threshold_training_targets_verified), "x"' in source
    assert 'open(Path(cfg.paths.threshold_training_targets_exploratory), "x"' in source
    assert "cfg.paths.threshold_training_targets)" not in source  # the old single combined path is gone


def test_percentile_threshold_per_class_computes_real_values():
    module = _load_script("08b_verify_full_policy_proxy.py")
    ranking_scores = {"a": 0.1, "b": 0.5, "c": 0.9}
    intended = {"a": {"Edema": 1}, "b": {"Edema": 1}, "c": {"Edema": 1}}
    result = module.percentile_threshold_per_class(50.0, ranking_scores, intended, ["a", "b", "c"], ["Edema"])
    assert result["Edema"] == pytest.approx(0.5)


def test_07b_refuses_real_training_without_confirmation_flag(monkeypatch):
    """The core fail-closed guard: --phase run must refuse BEFORE any config/data/model I/O."""
    module = _load_script("07b_verify_thresholds_proxy.py")
    monkeypatch.setattr(sys, "argv", ["07b_verify_thresholds_proxy.py", "--phase", "run"])
    with pytest.raises(SystemExit, match="REFUSED"):
        module.main()


def test_07b_phase_estimate_does_not_require_confirmation_flag(monkeypatch):
    """--phase estimate never trains anything, so it must reach the (missing-plan) upstream gate
    rather than the REFUSED confirmation-flag message."""
    module = _load_script("07b_verify_thresholds_proxy.py")
    monkeypatch.setattr(sys, "argv", ["07b_verify_thresholds_proxy.py", "--phase", "estimate",
                                      "--namespace", "unit-test-namespace-that-will-never-exist"])
    with pytest.raises(SystemExit, match="UPSTREAM GATE"):
        module.main()


def test_07b_phase_run_with_confirmation_flag_reaches_the_next_real_gate(monkeypatch):
    """With the flag passed, the guard must NOT block — it should proceed to the next real check
    (the missing verification plan), never straight into training."""
    module = _load_script("07b_verify_thresholds_proxy.py")
    monkeypatch.setattr(sys, "argv", [
        "07b_verify_thresholds_proxy.py", "--phase", "run", "--i-understand-this-trains-real-models",
        "--namespace", "unit-test-namespace-that-will-never-exist",
    ])
    with pytest.raises(SystemExit, match="UPSTREAM GATE"):
        module.main()


def test_marginal_targets_are_size_normalized_across_subset_sizes():
    r"""The same image, with the same deviation from its subset mean, must get the same marginal
    target whether it sat in a small subset or a large one.

    Regression test for the 1/(n-1) size bias: SetUtilityNetwork pools by MEAN, so the raw
    leave-one-out difference U(S)-U(S\i) scales like 1/(n-1). Averaging raw differences across
    subsets of different sizes therefore encodes "which subset sizes did this image land in" into
    the ranking network's supervision. Without the (n-1) normalization in marginal_targets, the
    small-subset target here is ~(249/99)x the large-subset one and this test fails.
    """
    module = _load_script("05_train_learned_asism.py")
    device = torch.device("cpu")

    class LinearMeanPool(torch.nn.Module):
        """Exactly the pooling contract of SetUtilityNetwork, with an identity head so the
        expected value is analytic rather than an artefact of a random initialization."""

        def forward(self, features, mask):
            weights = mask.unsqueeze(-1).to(features.dtype)
            pooled = (features * weights).sum(1) / weights.sum(1).clamp_min(1.0)
            return pooled[:, 0]

    model = LinearMeanPool()
    probe = "probe"
    lookup = {probe: np.array([1.0], dtype=np.float32)}
    # Peers all sit at 0.0, so the probe's deviation from the subset mean is identical in both
    # subsets; only the subset SIZE differs.
    small_ids, large_ids = [probe], [probe]
    for index in range(99):
        key = f"small_{index}"
        lookup[key] = np.array([0.0], dtype=np.float32)
        small_ids.append(key)
    for index in range(249):
        key = f"large_{index}"
        lookup[key] = np.array([0.0], dtype=np.float32)
        large_ids.append(key)

    small_only, _ = module.marginal_targets(
        model, [{"image_ids": small_ids}], lookup, device
    )
    large_only, _ = module.marginal_targets(
        model, [{"image_ids": large_ids}], lookup, device
    )

    # Analytic values. Raw U(S)-U(S\i) here is exactly 1/n, so after multiplying by (n-1) the
    # target is (n-1)/n: 0.990 at n=100 and 0.996 at n=250.
    assert small_only[probe] == pytest.approx(99 / 100, rel=1e-6)
    assert large_only[probe] == pytest.approx(249 / 250, rel=1e-6)

    # The point of the fix: the two sizes must now agree to well under 1%. Without the (n-1)
    # factor the raw targets are 0.01 vs 0.004 — a 2.5x size bias — and this assertion fails.
    assert small_only[probe] == pytest.approx(large_only[probe], rel=0.01)

    # Averaging across both subsets must land between them, not be dominated by the smaller one.
    combined, exposures = module.marginal_targets(
        model, [{"image_ids": small_ids}, {"image_ids": large_ids}], lookup, device
    )
    assert exposures[probe] == 2
    assert combined[probe] == pytest.approx((99 / 100 + 249 / 250) / 2, rel=1e-6)


# ---------------------------------------------------------------------------
# Multi-signal utility ranking — Data-Banzhaf MSR targets + LOO diagnostic
# ---------------------------------------------------------------------------

_BANZHAF_SUBSETS = [
    {"subset_id": "s0", "role": "train", "image_ids": ["a", "b"]},
    {"subset_id": "s1", "role": "train", "image_ids": ["a", "c"]},
    {"subset_id": "s2", "role": "train", "image_ids": ["b", "c"]},
    {"subset_id": "s3", "role": "train", "image_ids": ["a", "b", "c"]},
]
_BANZHAF_UTILITY = {"s0": 0.10, "s1": 0.20, "s2": 0.30, "s3": 0.40}


def test_banzhaf_msr_targets_is_mean_in_minus_mean_out():
    targets, counts = banzhaf_msr_targets(_BANZHAF_SUBSETS, _BANZHAF_UTILITY)
    # a: in s0,s1,s3 (mean 0.2333...), out s2 (0.30)
    assert targets["a"] == pytest.approx((0.10 + 0.20 + 0.40) / 3 - 0.30)
    assert targets["b"] == pytest.approx((0.10 + 0.30 + 0.40) / 3 - 0.20)
    assert targets["c"] == pytest.approx((0.20 + 0.30 + 0.40) / 3 - 0.10)
    assert counts["a"] == {"in": 3, "out": 1}
    # c is valued highest — it is only ever absent from the lowest-utility subset.
    assert targets["c"] > targets["b"] > targets["a"]


def test_banzhaf_msr_targets_excludes_one_sided_and_low_exposure_images():
    subsets = [
        {"subset_id": "s0", "image_ids": ["ever", "a"]},
        {"subset_id": "s1", "image_ids": ["ever", "b"]},
        {"subset_id": "s2", "image_ids": ["ever", "a"]},
    ]
    utility = {"s0": 0.1, "s1": 0.2, "s2": 0.3}
    targets, counts = banzhaf_msr_targets(subsets, utility)
    assert "ever" not in targets  # in every subset -> no "out" side -> undefined, omitted
    assert counts["ever"] == {"in": 3, "out": 0}
    assert set(targets) == {"a", "b"}

    # minimum_in_subsets filter: b appears in only one subset.
    stricter, _ = banzhaf_msr_targets(subsets, utility, minimum_in_subsets=2)
    assert set(stricter) == {"a"}


def test_banzhaf_msr_targets_raises_on_missing_measured_utility():
    with pytest.raises(ValueError, match="no measured utility"):
        banzhaf_msr_targets(_BANZHAF_SUBSETS, {"s0": 0.1})


def test_banzhaf_msr_targets_empty_input():
    assert banzhaf_msr_targets([], {}) == ({}, {})


def test_spearman_with_reason_success_and_undefined_cases():
    rho, reason = spearman_with_reason([1, 2, 3, 4], [10, 20, 30, 40])
    assert rho == pytest.approx(1.0)
    assert reason is None
    assert spearman_with_reason([1.0, 1.0, 1.0], [1.0, 2.0, 3.0]) == (None, "constant_predictions_or_targets")
    assert spearman_with_reason([1.0], [2.0]) == (None, "fewer_than_two_paired_points")


def test_loo_vs_banzhaf_diagnostic_reports_rank_agreement_over_shared_images():
    loo = {"a": 0.1, "b": 0.2, "c": 0.3, "only_loo": 0.9}
    banzhaf = {"a": 1.0, "b": 2.0, "c": 3.0, "only_banzhaf": -5.0}
    diagnostic = loo_vs_banzhaf_diagnostic(loo, banzhaf)
    assert diagnostic["n_images_compared"] == 3  # a, b, c
    assert diagnostic["spearman"] == pytest.approx(1.0)
    assert diagnostic["spearman_undefined_reason"] is None
    assert diagnostic["n_leave_one_out_targets"] == 4
    assert diagnostic["n_banzhaf_targets"] == 4


# ---------------------------------------------------------------------------
# FreeMatch-style training-free class-adaptive percentile threshold
# ---------------------------------------------------------------------------

def _freematch_pool():
    image_ids = [f"i{i}" for i in range(10)]
    ranking_scores = {image_id: index / 9 for index, image_id in enumerate(image_ids)}
    intended = {image_id: {"Common": 1, "Rare": 1} for image_id in image_ids}
    return ranking_scores, intended, image_ids


def test_freematch_style_percentile_gives_rare_classes_more_lenient_thresholds():
    ranking_scores, intended, image_ids = _freematch_pool()
    prevalence = {"Common": {"real_prevalence": 0.40}, "Rare": {"real_prevalence": 0.02}}
    result = freematch_style_percentile_per_class(
        ranking_scores, intended, image_ids, ["Common", "Rare"], prevalence,
        base_percentile=50.0, min_percentile=10.0,
    )
    # Identical score distributions per class; only real prevalence differs.
    assert result["Common"] == pytest.approx(float(np.percentile(list(ranking_scores.values()), 50.0)))
    assert result["Rare"] < result["Common"]  # rarer real class -> more lenient (lower) cut


def test_freematch_style_percentile_falls_back_to_base_on_missing_or_zero_prevalence():
    ranking_scores, intended, image_ids = _freematch_pool()
    prevalence = {"Common": {"real_prevalence": 0.40}, "Rare": {"real_prevalence": None}}
    result = freematch_style_percentile_per_class(
        ranking_scores, intended, image_ids, ["Common", "Rare"], prevalence,
    )
    base = float(np.percentile(list(ranking_scores.values()), 50.0))
    assert result["Rare"] == pytest.approx(base)  # unknown prevalence never buys leniency


def test_freematch_style_percentile_none_for_class_without_candidates():
    ranking_scores, intended, image_ids = _freematch_pool()
    result = freematch_style_percentile_per_class(
        ranking_scores, intended, image_ids, ["Common", "Absent"],
        {"Common": {"real_prevalence": 0.4}, "Absent": {"real_prevalence": 0.1}},
    )
    assert result["Absent"] is None


def test_freematch_style_percentile_rejects_inverted_percentile_bounds():
    ranking_scores, intended, image_ids = _freematch_pool()
    with pytest.raises(ValueError):
        freematch_style_percentile_per_class(
            ranking_scores, intended, image_ids, ["Common"], {"Common": {"real_prevalence": 0.4}},
            base_percentile=10.0, min_percentile=50.0,
        )


def test_freematch_style_wired_as_full_policy_variant_in_08b_and_09():
    root = Path(__file__).resolve().parents[1] / "scripts" / "asism"
    source_08b = (root / "08b_verify_full_policy_proxy.py").read_text(encoding="utf-8")
    source_09 = (root / "09_finalize_learned_selection.py").read_text(encoding="utf-8")
    for source in (source_08b, source_09):
        assert 'freematch_style_adaptive_percentile' in source
        assert 'freematch_style_percentile_per_class(' in source
        assert 'enforce_per_class_selection_floor(' in source


# ---------------------------------------------------------------------------
# C1 — SetUtilityNetwork superset conditioning (Xie et al., ICLR 2024) — ablation flag
# ---------------------------------------------------------------------------

def test_set_utility_default_is_unchanged_mean_pooling():
    model = SetUtilityNetwork(3, (8, 4), (3,))
    assert model.superset_conditioning is False
    assert not hasattr(model, "superset_summary")
    x = torch.randn(2, 5, 3)
    mask = torch.ones(2, 5, dtype=torch.bool)
    assert model(x, mask).shape == (2,)


def test_set_utility_superset_conditioning_is_permutation_invariant():
    torch.manual_seed(3)
    model = SetUtilityNetwork(3, (8, 4), (3,), superset_conditioning=True)
    model.set_superset_summary(np.arange(6, dtype=np.float32) / 6)
    model.eval()
    x = torch.randn(1, 5, 3)
    mask = torch.ones(1, 5, dtype=torch.bool)
    assert torch.allclose(model(x, mask), model(x[:, [3, 1, 4, 0, 2]], mask), atol=1e-6)


def test_set_utility_superset_summary_setter_validates_and_survives_state_dict():
    model = SetUtilityNetwork(2, (4,), (3,), superset_conditioning=True)
    with pytest.raises(ValueError):
        model.set_superset_summary(np.zeros(3, dtype=np.float32))  # wrong length (needs 2*input_dim)
    model.set_superset_summary(np.array([1.0, 2.0, 3.0, 4.0], dtype=np.float32))
    reloaded = SetUtilityNetwork(2, (4,), (3,), superset_conditioning=True)
    reloaded.load_state_dict(model.state_dict())
    assert torch.equal(reloaded.superset_summary, model.superset_summary)
    with pytest.raises(ValueError):
        SetUtilityNetwork(2, (4,), (3,)).set_superset_summary(np.zeros(4, dtype=np.float32))


# ---------------------------------------------------------------------------
# C2 — ranking-target normalization (2D-OOB, NeurIPS 2024; Chi et al., ICML 2026)
# ---------------------------------------------------------------------------

def test_normalize_ranking_targets_standardize_rank_none_and_order_preserved():
    raw = {"a": -3.0, "b": 0.0, "c": 0.1, "d": 50.0}
    order = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]

    none = normalize_ranking_targets(raw, "none")
    assert none == raw

    standardized = normalize_ranking_targets(raw, "standardize")
    values = np.array([standardized[k] for k in raw])
    assert values.mean() == pytest.approx(0.0, abs=1e-9)
    assert values.std() == pytest.approx(1.0, abs=1e-9)
    assert [k for k, _ in sorted(standardized.items(), key=lambda kv: kv[1])] == order

    ranked = normalize_ranking_targets(raw, "rank")
    assert min(ranked.values()) == pytest.approx(0.0)
    assert max(ranked.values()) == pytest.approx(1.0)
    assert [k for k, _ in sorted(ranked.items(), key=lambda kv: kv[1])] == order


def test_normalize_ranking_targets_winsorize_clips_the_outlier():
    raw = {f"i{n}": float(n) for n in range(10)}
    raw["outlier"] = 1000.0
    clipped = normalize_ranking_targets(raw, "none", winsorize_quantile=0.1)
    assert clipped["outlier"] < 1000.0
    assert clipped["outlier"] == pytest.approx(np.quantile(list(raw.values()), 0.9))
    with pytest.raises(ValueError):
        normalize_ranking_targets(raw, "none", winsorize_quantile=0.7)


def test_normalize_ranking_targets_empty_and_unknown_method():
    assert normalize_ranking_targets({}, "standardize") == {}
    with pytest.raises(ValueError):
        normalize_ranking_targets({"a": 1.0, "b": 2.0}, "zscore")


# ---------------------------------------------------------------------------
# C4 — per-class selection floor (SST class-fairness, IP&M 2025)
# ---------------------------------------------------------------------------

def _floor_pool(n_per_class: int = 8):
    image_ids = [f"i{n}" for n in range(n_per_class)]
    ranking_scores = {image_id: index / (n_per_class - 1) for index, image_id in enumerate(image_ids)}
    intended = {image_id: {"Rare": 1} for image_id in image_ids}
    return ranking_scores, intended, image_ids


def test_enforce_per_class_selection_floor_admits_the_minimum():
    ranking_scores, intended, image_ids = _floor_pool(8)
    thresholds = {"Rare": 0.95}  # only 1 candidate clears this
    adjusted = enforce_per_class_selection_floor(thresholds, ranking_scores, intended, image_ids, ["Rare"], 4)
    selected = sum(1 for i in image_ids if ranking_scores[i] >= adjusted["Rare"])
    assert selected >= 4
    assert adjusted["Rare"] < 0.95


def test_enforce_per_class_selection_floor_noop_when_already_enough():
    ranking_scores, intended, image_ids = _floor_pool(8)
    thresholds = {"Rare": 0.1}  # 8 candidates clear this
    adjusted = enforce_per_class_selection_floor(thresholds, ranking_scores, intended, image_ids, ["Rare"], 4)
    assert adjusted["Rare"] == 0.1


def test_enforce_per_class_selection_floor_handles_none_and_too_few_candidates():
    ranking_scores, intended, image_ids = _floor_pool(3)
    thresholds = {"Rare": 0.99, "Absent": None}
    adjusted = enforce_per_class_selection_floor(
        thresholds, ranking_scores, intended, image_ids, ["Rare", "Absent"], 10
    )
    assert adjusted["Absent"] is None
    # Only 3 candidates exist for Rare; threshold drops to the lowest so all 3 are admitted.
    assert adjusted["Rare"] == pytest.approx(min(ranking_scores.values()))


def test_freematch_style_percentile_range_normalises_leniency():
    image_ids = [f"i{n}" for n in range(20)]
    ranking_scores = {image_id: index / 19 for index, image_id in enumerate(image_ids)}
    intended = {image_id: {"Rare": 1, "Mid": 1, "Common": 1} for image_id in image_ids}
    prevalence = {
        "Rare": {"real_prevalence": 0.02},
        "Mid": {"real_prevalence": 0.20},
        "Common": {"real_prevalence": 0.40},
    }
    result = freematch_style_percentile_per_class(
        ranking_scores, intended, image_ids, ["Rare", "Mid", "Common"], prevalence,
        base_percentile=50.0, min_percentile=10.0,
    )
    scores = list(ranking_scores.values())
    # Rarest present -> min_percentile; most common -> base_percentile; Mid strictly between.
    assert result["Rare"] == pytest.approx(float(np.percentile(scores, 10.0)))
    assert result["Common"] == pytest.approx(float(np.percentile(scores, 50.0)))
    assert result["Rare"] < result["Mid"] < result["Common"]


# ---------------------------------------------------------------------------
# --namespace must reach cfg.split_namespace (read by load_candidate_pool)
# ---------------------------------------------------------------------------

class _StopAfterNamespaceSetup(Exception):
    """Raised by a patched call right after a script's namespace setup, so nothing else runs."""


# Script -> the first call its main() makes after the namespace setup, patched to stop there.
# 09 needs no patch: its next statement is a missing-artifact gate that exits on its own.
_LEARNED_STAGES_USING_CANDIDATE_POOL = {
    "05_train_learned_asism.py": "set_seed",
    "06_learn_thresholds_select.py": "set_seed",
    "07b_verify_thresholds_proxy.py": "stage2_paths",
    "08_train_threshold_network.py": "set_seed",
    "08b_verify_full_policy_proxy.py": "stage2_paths",
    "09_finalize_learned_selection.py": None,
}


@pytest.mark.parametrize("script", sorted(_LEARNED_STAGES_USING_CANDIDATE_POOL))
def test_learned_stage_uses_cli_namespace_not_config_default(script, monkeypatch):
    """Regression test: each learned stage rewrites its paths for --namespace, and must also set
    cfg.split_namespace to it — load_candidate_pool() reads cfg.split_namespace to choose which split
    manifest, generation manifest and score provenance to validate against. Without that, a
    production run (--namespace production-thesis-v1) silently used the YAML default
    (dev-smoke-v1). The namespace here deliberately differs from the default, which the GPU smoke
    cannot exercise (its namespace IS the default)."""
    module = _load_script(script)
    requested = "dev-namespace-regression-v1"
    captured = {}
    real_load = module.load_named_config

    def load_and_capture(filename, section):
        config = real_load(filename, section)
        if section == "stage3":
            captured["cfg"] = config
        return config

    def stop(*_args, **_kwargs):
        raise _StopAfterNamespaceSetup

    monkeypatch.setattr(module, "load_named_config", load_and_capture)
    stop_at = _LEARNED_STAGES_USING_CANDIDATE_POOL[script]
    if stop_at:
        monkeypatch.setattr(module, stop_at, stop)
    monkeypatch.setattr(sys, "argv", [script, "--namespace", requested])

    with pytest.raises((_StopAfterNamespaceSetup, SystemExit)):
        module.main()

    config_default = str(real_load("stage3_asism.yaml", "stage3").split_namespace)
    assert config_default != requested, "test needs a namespace different from the YAML default"
    assert str(captured["cfg"].split_namespace) == requested
    assert Path(captured["cfg"].paths.scores_dir).parts[-2] == requested


## `tests/test_pipeline_contracts.py`

*329 lines*

**Tests the structural rules the pipeline depends on**, not individual functions: the label-set
definitions and their sizes, the patient-level support rule, split partition disjointness and
exactness, the `final_eval_heldout` purpose-token guard (including that outcome purposes are refused
without a run id), and the namespace validation rules.

In [ ]:
# ===== FILE: tests/test_pipeline_contracts.py =====
"""Contract and guard tests for the Stage 2-5 pipeline (docs/stages2_to_5_plan.md).

Two things are verified here:

1. The LEAKAGE GUARDS actually refuse. A guard that is only documented is not a guard, so each one
   is exercised and asserted to fail.
2. The Stage 5 ANALYSIS LAYER works end-to-end on explicitly-labelled FIXTURES. Fixtures are not
   scientific validation and are never presented as results — they prove the statistical code paths
   execute and produce the expected structure, nothing more.
"""

from __future__ import annotations

import json
import os
import subprocess
import sys
sys.dont_write_bytecode = True
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO))

from scripts.utils.labels import (  # noqa: E402
    CLASSIFIER_TARGET_LABELS,
    GENERATION_TARGET_LABELS,
    INSUFFICIENT_SUPPORT_LABELS,
    PRIMARY_ENDPOINT_LABELS,
    build_label_arrays,
    check_support_rule,
    normalize_label,
    patient_level_support,
)
from scripts.utils.metrics import full_metric_suite  # noqa: E402
from scripts.utils.splits import (  # noqa: E402
    FinalEvalAccessViolation,
    assert_final_eval_access_allowed,
)
from fixture_workspace import fixture_workspace


# ------------------------------------------------------------------ label policy (§5, §6)

def test_primary_endpoint_excludes_no_finding_and_support_devices():
    assert len(CLASSIFIER_TARGET_LABELS) == 14
    assert len(PRIMARY_ENDPOINT_LABELS) == 11
    assert "No Finding" not in PRIMARY_ENDPOINT_LABELS
    assert "Support Devices" not in PRIMARY_ENDPOINT_LABELS
    assert "Pleural Other" not in PRIMARY_ENDPOINT_LABELS
    assert "No Finding" in CLASSIFIER_TARGET_LABELS
    assert "Support Devices" in CLASSIFIER_TARGET_LABELS
    assert "Pleural Other" in CLASSIFIER_TARGET_LABELS


def test_generation_target_labels_restores_insufficient_support_labels():
    assert len(GENERATION_TARGET_LABELS) == 12
    assert "Pleural Other" in GENERATION_TARGET_LABELS
    assert set(GENERATION_TARGET_LABELS) == set(PRIMARY_ENDPOINT_LABELS) | set(INSUFFICIENT_SUPPORT_LABELS)
    assert "No Finding" not in GENERATION_TARGET_LABELS
    assert "Support Devices" not in GENERATION_TARGET_LABELS


def test_uncertain_and_blank_are_masked_never_converted():
    frame = pd.DataFrame({label: [1, 0, -1, None] for label in CLASSIFIER_TARGET_LABELS})
    raw, target, mask = build_label_arrays(frame)
    # Row 0 = positive, row 1 = negative -> masked in; rows 2 (-1) and 3 (blank) -> masked out.
    assert mask[0, 0] and mask[1, 0]
    assert not mask[2, 0], "-1 must be masked, never mapped to 0 or 1"
    assert not mask[3, 0], "blank must be masked"
    assert raw[2, 0] == -1, "raw -1 must be preserved verbatim"
    assert raw[3, 0] == -2, "blank must stay distinguishable from an explicit -1"


def test_normalize_label_handles_all_forms():
    assert normalize_label(1) == 1
    assert normalize_label("0") == 0
    assert normalize_label(-1.0) == -1
    assert normalize_label(None) is None
    assert normalize_label("") is None
    assert normalize_label(float("nan")) is None


def test_patient_level_support_counts_patients_not_images():
    # One patient with 5 positive studies must count as ONE positive patient.
    frame = pd.DataFrame(
        {
            "patient_id": ["p1"] * 5 + ["p2", "p3"],
            "Cardiomegaly": [1, 1, 1, 1, 1, 0, 0],
        }
    )
    support = patient_level_support(frame, ["Cardiomegaly"])
    row = support.iloc[0]
    assert row["positive_patients"] == 1, "patient-level, not image-level"
    assert row["positive_images"] == 5
    assert row["negative_patients"] == 2


def test_patient_positive_wins_over_negative():
    # A patient with one positive and one negative study is a POSITIVE patient.
    frame = pd.DataFrame({"patient_id": ["p1", "p1"], "Edema": [1, 0]})
    support = patient_level_support(frame, ["Edema"])
    assert support.iloc[0]["positive_patients"] == 1
    assert support.iloc[0]["negative_patients"] == 0


def test_support_rule_reports_all_failures_not_just_first():
    support = pd.DataFrame(
        [
            {"label": "A", "positive_patients": 10, "negative_patients": 500},
            {"label": "B", "positive_patients": 500, "negative_patients": 5},
            {"label": "C", "positive_patients": 500, "negative_patients": 500},
        ]
    )
    passed, failures = check_support_rule(support, 50, 50)
    assert passed is False
    assert len(failures) == 2, "must report every failing label, not abort on the first"
    assert {f["label"] for f in failures} == {"A", "B"}


# ------------------------------------------------------------------ leakage guards (§1.6)

def test_final_eval_guard_allows_split_construction():
    assert_final_eval_access_allowed("split_construction", caller="test") is None
    assert_final_eval_access_allowed("support_check", caller="test") is None
    assert_final_eval_access_allowed("hash_verification", caller="test") is None


def test_final_eval_guard_refuses_outcome_access_without_run_id():
    for purpose in ("load_images", "load_labels_for_evaluation", "compute_outcome_statistics",
                    "model_selection"):
        try:
            assert_final_eval_access_allowed(purpose, caller="test")
            raise AssertionError(f"guard did not refuse outcome purpose {purpose!r}")
        except FinalEvalAccessViolation:
            pass


def test_final_eval_guard_allows_outcome_access_with_run_id():
    assert_final_eval_access_allowed(
        "load_labels_for_evaluation", final_eval_run_id="stage5-run-1", caller="test"
    )


def test_final_eval_guard_rejects_unknown_purpose():
    try:
        assert_final_eval_access_allowed("sneaky_peek", caller="test")
        raise AssertionError("guard accepted an undeclared purpose")
    except FinalEvalAccessViolation:
        pass


def test_stage5_refuses_without_frozen_preconditions():
    """Stage 5 must abort BEFORE opening final_eval_heldout when preconditions are unmet."""
    with fixture_workspace("stage5-refusal") as workspace:
        environment = {**os.environ, "PROJECT_ROOT": str(workspace)}
        result = subprocess.run(
            [sys.executable, "scripts/eval/stage5_evaluate.py",
             "--final-eval-run-id", "test-run", "--namespace", "production-isolated-v1"],
            cwd=REPO, capture_output=True, text=True, env=environment,
        )
    assert result.returncode != 0, "Stage 5 must refuse when preconditions are unmet"
    combined = result.stdout + result.stderr
    assert "STAGE 5 REFUSED" in combined or "Missing split manifest" in combined
    assert "final_eval_heldout was NOT opened" in combined or "split manifest" in combined


def test_chexpert_pretrained_initialization_is_rejected():
    """A CheXpert-pretrained checkpoint cannot be shown disjoint from our evaluation patients (§2)."""
    from scripts.utils.classifier import build_model

    try:
        build_model(14, 0.2, "chexpert_densenet121")
        raise AssertionError("CheXpert-pretrained initialization was not rejected")
    except SystemExit as exc:
        assert "Refusing CheXpert-pretrained" in str(exc)


# ------------------------------------------------------------------ split builder (§1)

def test_split_fractions_sum_to_one():
    from scripts.utils.splits import SPLIT_NAMES, load_splits_config

    config = load_splits_config()
    total = sum(float(config.fractions[name]) for name in SPLIT_NAMES)
    assert abs(total - 1.0) < 1e-9, f"frozen fractions must sum to 1.0, got {total}"


def test_six_way_partition_is_exact_and_disjoint():
    import importlib.util

    spec = importlib.util.spec_from_file_location(
        "builder", REPO / "scripts" / "data" / "02b_build_sixway_splits.py"
    )
    builder = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(builder)

    patients = [f"patient{i:05d}" for i in range(1000)]
    fractions = {
        "gen_train": 0.60, "gen_val": 0.10, "classifier_train": 0.15,
        "classifier_val": 0.05, "asism_tuning_heldout": 0.05, "final_eval_heldout": 0.05,
    }
    groups = builder.partition_patients(patients, fractions, seed=42)

    assert sum(len(ids) for ids in groups.values()) == 1000, "partition must be exact"
    builder.assert_disjoint_and_exact(groups, set(patients))  # raises on any leak

    # Deterministic: same seed -> same partition.
    again = builder.partition_patients(patients, fractions, seed=42)
    assert groups == again


def test_partition_detects_injected_leak():
    import importlib.util

    spec = importlib.util.spec_from_file_location(
        "builder", REPO / "scripts" / "data" / "02b_build_sixway_splits.py"
    )
    builder = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(builder)

    groups = {"a": {"p1", "p2"}, "b": {"p2", "p3"}}  # p2 leaks across both
    try:
        builder.assert_disjoint_and_exact(groups, {"p1", "p2", "p3"})
        raise AssertionError("disjointness assertion failed to detect an injected leak")
    except SystemExit as exc:
        assert "SPLIT LEAK" in str(exc)


# ------------------------------------------------------------------ Stage 5 analysis on FIXTURES

def test_stage5_analysis_layer_on_fixtures():
    """End-to-end statistical layer on FIXTURE data.

    This is explicitly NOT a scientific result: probabilities are synthesised so that one condition
    is genuinely better than another, and the test asserts the machinery detects that. It proves the
    code path works; it says nothing about ASISM.
    """
    rng = np.random.default_rng(0)
    n_patients, per_patient = 120, 2
    n_rows = n_patients * per_patient

    patient_ids = np.repeat([f"p{i:04d}" for i in range(n_patients)], per_patient)
    truth = pd.DataFrame(
        {label: rng.integers(0, 2, n_rows) for label in CLASSIFIER_TARGET_LABELS}
    )
    _, targets, masks = build_label_arrays(truth)

    def make_probabilities(signal_strength: float) -> np.ndarray:
        probabilities = np.zeros((n_rows, len(CLASSIFIER_TARGET_LABELS)))
        for index, label in enumerate(CLASSIFIER_TARGET_LABELS):
            y = truth[label].to_numpy()
            noise = rng.uniform(0, 1, n_rows)
            probabilities[:, index] = np.clip(
                signal_strength * y + (1 - signal_strength) * noise, 0.001, 0.999
            )
        return probabilities

    strong = make_probabilities(0.75)
    weak = make_probabilities(0.05)

    suite = full_metric_suite(strong, targets, masks)
    assert suite["macro_auroc_primary"] > 0.7
    assert len(suite["primary_labels"]) == 11
    assert set(suite["secondary_labels"]) == {"No Finding", "Support Devices", "Pleural Other"}
    for label, entry in suite["per_label"].items():
        assert "effective_n" in entry, f"{label} must report effective N"

    from scripts.utils.metrics import auroc, paired_bootstrap_difference

    def macro(probabilities, rows):
        values = []
        for label in PRIMARY_ENDPOINT_LABELS:
            index = CLASSIFIER_TARGET_LABELS.index(label)
            mask = masks[rows, index].astype(bool)
            y = targets[rows][mask, index].astype(int)
            if (y == 1).sum() and (y == 0).sum():
                values.append(auroc(probabilities[rows][mask, index], y))
        return float(np.mean(values)) if values else float("nan")

    comparison = paired_bootstrap_difference(
        patient_ids,
        lambda rows: macro(strong, rows),
        lambda rows: macro(weak, rows),
        n_resamples=200,
        seed=1,
    )
    assert comparison["effect_size"] > 0.1
    assert comparison["ci_lower"] > 0, "a real difference must produce a CI excluding zero"
    assert comparison["p_value"] < 0.05


def test_fixture_data_is_never_written_to_production_paths():
    """Guard against a test accidentally polluting real artifact directories."""
    from scripts.utils.splits import load_splits_config

    config = load_splits_config()
    production_dir = Path(config.paths.splits_root) / "production"
    # The test suite must not have created production splits as a side effect.
    if production_dir.exists():
        manifest = production_dir / "split_manifest_v2.json"
        if manifest.is_file():
            data = json.loads(manifest.read_text(encoding="utf-8"))
            assert data.get("split_namespace") == "production"
            assert data.get("input_csv_rows", 0) > 100000, (
                "a production split manifest exists but was built from a small CSV — "
                "it must only be built from the full cohort"
            )


if __name__ == "__main__":
    import traceback

    tests = [(name, value) for name, value in sorted(globals().items()) if name.startswith("test_")]
    passed, failed = 0, 0
    for name, function in tests:
        try:
            function()
            print(f"  PASS  {name}")
            passed += 1
        except Exception:
            print(f"  FAIL  {name}")
            traceback.print_exc()
            failed += 1
    print(f"\n{passed} passed, {failed} failed, {len(tests)} total")
    raise SystemExit(1 if failed else 0)


## `tests/test_provenance_contracts.py`

*186 lines*

**Tests the provenance chain** - that a stage really does refuse artifacts produced with different
settings. Covers manifest field mismatches, score-artifact hash mismatches, code-identity
mismatches, split-manifest integrity self-hashing, the `write_frozen_json` write-once rule, and the
final-evaluation context validation.

In [ ]:
# ===== FILE: tests/test_provenance_contracts.py =====
"""Producer/consumer and refusal contracts added during the code-completeness pass."""
from __future__ import annotations

import importlib.util
import json
import sys
sys.dont_write_bytecode = True
from pathlib import Path
from types import SimpleNamespace

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO))

from scripts.asism.signals import aggregate_pathology_overlaps
from scripts.utils.artifact_contracts import (
    ArtifactContractError, require_manifest_fields, require_score_artifact, stage2_paths,
    stage3_paths, validate_namespace_run_id,
)
from scripts.utils.manifest import sha256_file
from scripts.utils.splits import FinalEvalAccessViolation, load_split, validate_final_eval_context
from fixture_workspace import fixture_workspace


def load_script(relative: str, name: str):
    spec = importlib.util.spec_from_file_location(name, REPO / relative)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader
    spec.loader.exec_module(module)
    return module


def test_versioned_namespace_contract_refuses_mutable_aliases():
    for value, kind in (("dev", "dev"), ("production", "production"), ("dev-v1", "production")):
        try:
            validate_namespace_run_id(value, kind)
        except ArtifactContractError:
            pass
        else:
            raise AssertionError(f"accepted mutable/mismatched namespace {value}")
    validate_namespace_run_id("dev-smoke-v1", "dev")
    validate_namespace_run_id("production-thesis-v1", "production")


def test_stage2_and_stage3_paths_are_namespace_disjoint():
    cfg = SimpleNamespace(paths=SimpleNamespace(synthetic_root="synthetic"))
    assert stage2_paths(cfg, "dev-v1")["manifest_path"] != stage2_paths(cfg, "production-v1")["manifest_path"]
    assert stage3_paths(cfg, "dev-v1")["scores_dir"] != stage3_paths(cfg, "production-v1")["scores_dir"]


def test_manifest_contract_rejects_stale_field():
    with fixture_workspace("stale-manifest") as workspace:
        path = workspace / "manifest.json"
        path.write_text(json.dumps({"split_manifest_hash": "old"}), encoding="utf-8")
        try:
            require_manifest_fields(path, {"split_manifest_hash": "new"}, "fixture")
        except ArtifactContractError as exc:
            assert "Stale/incompatible" in str(exc)
        else:
            raise AssertionError("stale manifest was accepted")


def test_score_sidecar_hash_is_enforced():
    with fixture_workspace("score-refusal") as workspace:
        path = workspace / "similarity_scores.parquet"
        path.write_bytes(b"not-a-valid-completed-score")
        try:
            require_score_artifact(path, "similarity", {"split_namespace": "production-v1"})
        except ArtifactContractError:
            pass
        else:
            raise AssertionError("score without compatible sidecar was accepted")


def test_stale_unnamespaced_stage2_recipes_are_explicitly_rejected():
    module = load_script("scripts/generate/02_generate_synthetic_images.py", "stage2_generation_test")
    with fixture_workspace("legacy-recipes") as root:
        (root / "label_recipes.csv").write_text("recipe_id\nlegacy\n", encoding="utf-8")
        cfg = SimpleNamespace(paths=SimpleNamespace(synthetic_root=str(root)))
        try:
            module.validate_generation_inputs(cfg, "dev-v1", stage2_paths(cfg, "dev-v1"))
        except ArtifactContractError as exc:
            assert "stale unnamespaced" in str(exc)
        else:
            raise AssertionError("legacy recipes were accepted")


def test_final_eval_metadata_purpose_never_returns_rows():
    try:
        load_split("final_eval_heldout", "production-v1", purpose="schema_validation", caller="test")
    except FinalEvalAccessViolation as exc:
        assert "split_metadata" in str(exc)
    else:
        raise AssertionError("metadata purpose exposed final rows")


def test_arbitrary_final_eval_context_is_refused():
    with fixture_workspace("bad-final-context") as workspace:
        path = workspace / "context.json"
        path.write_text(json.dumps({"final_eval_run_id": "anything", "namespace": "production-v1", "status": "registered"}), encoding="utf-8")
        try:
            validate_final_eval_context(path, "anything", "production-v1")
        except FinalEvalAccessViolation:
            pass
        else:
            raise AssertionError("arbitrary run ID unlocked final data")


def test_full_sha256_is_not_truncated():
    assert len(sha256_file(REPO / "scripts/utils/classifier.py")) == 64


def test_real_data_smoke_stage5_marker_is_bound_to_dev_namespace_and_split():
    module = load_script("scripts/eval/stage5_evaluate.py", "stage5_real_smoke_authorization_test")
    manifest = {"namespace_class": "dev", "manifest_hash": "split-hash"}
    with fixture_workspace("real-smoke-stage5-marker") as workspace:
        marker = {
            "schema_version": 1,
            "purpose": "real-data-engineering-smoke-only",
            "namespace": "dev-real-smoke-v2",
            "split_manifest_hash": "split-hash",
            "not_scientific_evidence": True,
            "upstream_code_identity_sha256": "a" * 64,
        }
        (workspace / "REAL_DATA_SMOKE_ONLY.json").write_text(json.dumps(marker), encoding="utf-8")
        authorization = module.require_smoke_final_eval_authorization(
            "dev-real-smoke-v2", manifest, workspace
        )
        assert authorization == {
            "kind": "real_data", "upstream_code_identity_sha256": "a" * 64
        }

        marker["split_manifest_hash"] = "different-split"
        (workspace / "REAL_DATA_SMOKE_ONLY.json").write_text(json.dumps(marker), encoding="utf-8")
        try:
            module.require_smoke_final_eval_authorization("dev-real-smoke-v2", manifest, workspace)
        except SystemExit as exc:
            assert "marker mismatch" in str(exc)
        else:
            raise AssertionError("real-data smoke marker was accepted for the wrong split")

        try:
            module.require_smoke_final_eval_authorization(
                "production-thesis-v1",
                {"namespace_class": "production", "manifest_hash": "split-hash"},
                workspace,
            )
        except SystemExit as exc:
            assert "dev-class" in str(exc)
        else:
            raise AssertionError("real-data smoke marker unlocked a production split")


def test_multilabel_gradcam_aggregation_uses_every_positive():
    mean, minimum = aggregate_pathology_overlaps({"Edema": 0.9, "Pneumonia": 0.1})
    assert mean == 0.5 and minimum == 0.1


def test_asism_grid_contains_equal_and_every_individual_signal_baseline():
    module = load_script("scripts/asism/03_tune_freeze_select.py", "asism_tuning_test")
    signals = ["similarity", "iqa", "explainability", "agreement"]
    grid = module.candidate_weight_grid([*signals, "uncertainty"], 12, 42)
    assert any(len({row[name] for name in signals}) == 1 and row[signals[0]] > 0 for row in grid)
    for signal in signals:
        assert any(row[signal] == 1.0 and sum(row.values()) == 1.0 for row in grid), signal
    assert any(row.get("__uncertainty_band_only__") == 1.0 for row in grid)


def test_split_builder_initializes_out_dir_before_use():
    source = (REPO / "scripts/data/02b_build_sixway_splits.py").read_text(encoding="utf-8")
    assignment = source.index("out_dir = resolve_splits_dir")
    first_use = source.index("if out_dir.exists()")
    terminal = source.index("raise SystemExit(main())")
    assert assignment < first_use < terminal


if __name__ == "__main__":
    import traceback
    tests = [(name, value) for name, value in sorted(globals().items()) if name.startswith("test_")]
    passed = failed = 0
    for name, function in tests:
        try:
            function(); print(f"  PASS  {name}", flush=True); passed += 1
        except Exception:
            print(f"  FAIL  {name}", flush=True); traceback.print_exc(); failed += 1
    print(f"\n{passed} passed, {failed} failed, {len(tests)} total")
    raise SystemExit(1 if failed else 0)


## `tests/test_roundtrip_contracts.py`

*249 lines*

**End-to-end round-trip tests on real fixture workspaces**: write an artifact with the production
code, read it back with the production reader, and assert the contract holds - including that a
tampered or stale artifact is rejected rather than silently accepted.

In [ ]:
# ===== FILE: tests/test_roundtrip_contracts.py =====
"""CPU-only positive and negative producer/consumer round trips."""
from __future__ import annotations

import importlib.util
import json
import sys
sys.dont_write_bytecode = True
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
from PIL import Image
from omegaconf import OmegaConf

REPO = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO))

from fixture_workspace import fixture_workspace
from scripts.asism.signals import write_score_artifact
from scripts.utils.artifact_contracts import ArtifactContractError, config_sha256, require_score_artifact, stage2_paths
from scripts.utils.classifier import TrainingBudget, train_classifier
from scripts.utils.labels import CLASSIFIER_TARGET_LABELS
from scripts.utils.manifest import sha256_file, write_json
from scripts.utils.splits import validate_final_eval_context


def load_script(relative: str, name: str):
    spec = importlib.util.spec_from_file_location(name, REPO / relative)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader
    spec.loader.exec_module(module)
    return module


def tiny_model(num_labels, _dropout, _source, seed=42):
    import torch
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(3 * 16 * 16, num_labels))


def records(workspace: Path, count: int = 6):
    result = []
    for index in range(count):
        path = workspace / f"image-{index}.png"
        array = np.full((16, 16, 3), 30 + index * 20, dtype=np.uint8)
        Image.fromarray(array).save(path)
        result.append({
            "image_id": f"i{index}", "patient_id": f"p{index}", "image_path": str(path),
            "labels": {label: index % 2 for label in CLASSIFIER_TARGET_LABELS}, "is_synthetic": False,
        })
    return result


def test_auxiliary_best_checkpoint_roundtrip_and_refusal():
    import torch
    import scripts.utils.classifier as classifier
    module = load_script("scripts/asism/01_compute_signals.py", "signal_loader_roundtrip")
    original = classifier.build_model
    classifier.build_model = tiny_model
    try:
        with fixture_workspace("aux-roundtrip") as workspace:
            resume = workspace / "aux.pt"
            best = workspace / "aux.best.pt"
            provenance = {"split_namespace": "dev-smoke-v1", "split_manifest_hash": "split", "code_identity_sha256": "code"}
            torch.save({"checkpoint_schema_version": 2, "checkpoint_role": "best_validation_model",
                        "model_state": tiny_model(14, 0.0, "none").state_dict(), "provenance": provenance,
                        "model_config": {"dropout_p": 0.0, "pretrained_source": "none", "resolution": 16}}, best)
            write_json(resume.with_suffix(".manifest.json"), {"best_checkpoint_sha256": sha256_file(best)})
            module.auxiliary_checkpoint_path = lambda _config, _namespace: resume
            model, resolution = module.load_auxiliary_classifier(SimpleNamespace(), "dev-smoke-v1", provenance)
            assert resolution == 16 and model.training is False
            bad = {**provenance, "split_manifest_hash": "different"}
            try:
                module.load_auxiliary_classifier(SimpleNamespace(), "dev-smoke-v1", bad)
            except SystemExit as exc:
                assert "provenance mismatch" in str(exc)
            else:
                raise AssertionError("incompatible auxiliary checkpoint was accepted")
    finally:
        classifier.build_model = original


def test_classifier_resume_and_best_checkpoint_persistence():
    import torch
    import scripts.utils.classifier as classifier
    original = classifier.build_model
    classifier.build_model = tiny_model
    try:
        with fixture_workspace("classifier-resume") as workspace:
            dataset = records(workspace)
            checkpoint = workspace / "resume.pt"
            metadata = {"split_manifest_hash": "split", "split_namespace": "dev-smoke-v1", "config_hash": "cfg"}
            first = TrainingBudget(1, 2, 1e-3, 0.0, 1, 7)
            _, accounting1, _ = train_classifier(dataset, dataset, first, 0.0, "none", 16,
                                                   checkpoint_path=checkpoint, checkpoint_metadata=metadata)
            best = Path(accounting1.extra["best_checkpoint_path"])
            assert checkpoint.is_file() and best.is_file()
            second = TrainingBudget(2, 2, 1e-3, 0.0, 1, 7)
            model, accounting2, _ = train_classifier(dataset, dataset, second, 0.0, "none", 16,
                                                       checkpoint_path=checkpoint, resume_from=checkpoint,
                                                       checkpoint_metadata=metadata)
            assert accounting2.optimizer_steps == 2
            payload = torch.load(best, map_location="cpu", weights_only=False)
            assert payload["checkpoint_role"] == "best_validation_model"
            model.load_state_dict(payload["model_state"])
            try:
                train_classifier(dataset, dataset, second, 0.0, "none", 16, checkpoint_path=checkpoint,
                                 resume_from=checkpoint, checkpoint_metadata={**metadata, "config_hash": "wrong"})
            except ValueError as exc:
                assert "incompatible classifier checkpoint provenance" in str(exc)
            else:
                raise AssertionError("incompatible resume was accepted")
    finally:
        classifier.build_model = original


def test_stage2_generation_manifest_to_stage3_contract():
    producer = load_script("scripts/generate/02_generate_synthetic_images.py", "generation_producer_roundtrip")
    consumer = load_script("scripts/asism/01_compute_signals.py", "generation_consumer_roundtrip")
    with fixture_workspace("generation-roundtrip") as workspace:
        lora = workspace / "lora"; lora.mkdir(); (lora / "weights.safetensors").write_bytes(b"fixture")
        write_json(lora / "metadata.json", {"split_namespace": "dev-smoke-v1", "split_manifest_hash": "split"})
        config = OmegaConf.create({"paths": {"synthetic_root": str(workspace / "synthetic")},
                                   "recipes": {"seed": 1}, "generation": {"seed": 2},
                                   "checkpoint": {"lora_weights_dir": str(lora)}})
        paths = stage2_paths(config, "dev-smoke-v1"); paths["root"].mkdir(parents=True)
        pd.DataFrame({"recipe_id": ["syn-0", "syn-1"]}).to_csv(paths["recipes_path"], index=False)
        producer.namespace_identity = lambda _namespace: {"split_namespace": "dev-smoke-v1", "namespace_class": "dev",
                                                           "split_manifest_hash": "split", "split_manifest_version": 2,
                                                           "split_manifest_file_sha256": "file", "split_frozen": False}
        producer.current_code_identity_hash = lambda: "code"
        write_json(paths["recipes_manifest"], {"schema_version": 2, "split_namespace": "dev-smoke-v1",
                   "namespace_class": "dev", "split_manifest_hash": "split",
                   "recipe_config_sha256": config_sha256(config.recipes),
                   "recipes_csv_sha256": sha256_file(paths["recipes_path"]), "code_identity_sha256": "code"})
        keys = producer.validate_generation_inputs(config, "dev-smoke-v1", paths)
        completion = {**keys, "num_rows": 2}
        frame = pd.DataFrame([{"image_id": f"syn-{i}", **keys} for i in range(2)])
        consumer.validate_generation_manifest_rows(frame, completion)
        frame.loc[1, "split_manifest_hash"] = "stale"
        try:
            consumer.validate_generation_manifest_rows(frame, completion)
        except SystemExit as exc:
            assert "provenance mismatch" in str(exc)
        else:
            raise AssertionError("stale generation row reached Stage 3")


def test_score_sidecar_gonogo_and_tuning_roundtrip():
    gonogo = load_script("scripts/asism/02_gonogo.py", "gonogo_roundtrip")
    tuning = load_script("scripts/asism/03_tune_freeze_select.py", "tuning_roundtrip")
    with fixture_workspace("score-roundtrip") as workspace:
        provenance = {"split_namespace": "dev-smoke-v1", "split_manifest_hash": "split", "code_identity_sha256": "code"}
        frame = pd.DataFrame({"image_id": [f"i{i}" for i in range(25)], "iqa_composite": np.linspace(0, 1, 25)})
        path = workspace / "iqa_scores.parquet"
        write_score_artifact(frame, path, "iqa", provenance)
        require_score_artifact(path, "iqa", provenance)
        checks = {"technical_validity": gonogo.check_technical_validity(frame, "iqa"),
                  "numerical_stability": {"passed": True}, "missing_rate": {"passed": True},
                  "reproducibility": {"passed": True}, "directionality": {"passed": True},
                  "redundancy": {"passed": True}, "usefulness": {"passed": True}}
        assert gonogo.decide(checks)[0] == "include"
        cfg = SimpleNamespace(paths=SimpleNamespace(scores_dir=str(workspace)), _expected_score_provenance=provenance)
        merged = tuning.load_merged_scores(cfg, ["iqa"])
        assert merged["image_id"].tolist() == frame["image_id"].tolist()
        sidecar = path.with_suffix(".provenance.json")
        altered = json.loads(sidecar.read_text(encoding="utf-8")); altered["split_manifest_hash"] = "wrong"
        sidecar.write_text(json.dumps(altered), encoding="utf-8")
        try:
            tuning.load_merged_scores(cfg, ["iqa"])
        except ArtifactContractError:
            pass
        else:
            raise AssertionError("tuning accepted stale score provenance")


def test_gonogo_reproducibility_accepts_score_sidecar_contract():
    gonogo = load_script("scripts/asism/02_gonogo.py", "gonogo_reproducibility_contract")
    with fixture_workspace("gonogo-reproducibility") as workspace:
        sidecar = workspace / "iqa_scores.provenance.json"
        sidecar.write_text(
            json.dumps(
                {
                    "schema_version": 2,
                    "signal": "iqa",
                    "n_rows": 4,
                    "asism_config_sha256": "a" * 64,
                    "git_commit_hash": "deadbeef",
                }
            ),
            encoding="utf-8",
        )

        result = gonogo.check_reproducibility(workspace, "iqa", object())

        assert result["passed"] is True
        assert result["missing_provenance_fields"] == []
        assert result["asism_config_sha256"] == "a" * 64


def test_stage5_registration_resume_and_second_run_refusal():
    module = load_script("scripts/eval/stage5_evaluate.py", "stage5_registration_roundtrip")
    with fixture_workspace("stage5-registration") as workspace:
        common = {"namespace": "dev-smoke-v1", "split_manifest_hash": "split", "asism_manifest_hash": "asism",
                  "protocol_manifest_hash": "protocol", "checkpoint_hashes": {"A": "hash"},
                  "threshold_policy_hash": "threshold", "code_git_version": "git", "code_identity_sha256": "code",
                  "registered_at_utc": "fixed", "status": "registered"}
        first_path = workspace / "run-1" / "final_eval_registration.json"
        first = {**common, "final_eval_run_id": "run-1"}
        module.register_final_evaluation(first_path, first)
        validate_final_eval_context(first_path, "run-1", "dev-smoke-v1")
        assert module.register_final_evaluation(first_path, first)["final_eval_run_id"] == "run-1"
        prediction_path = first_path.parent / "predictions" / "A.parquet"
        prediction_frame = pd.DataFrame({"image_id": ["i1", "i2"], "patient_id": ["p1", "p2"],
                                         **{label: [0.1, 0.9] for label in CLASSIFIER_TARGET_LABELS}})
        prediction_provenance = {"checkpoint_hash": "checkpoint", "split_hash": "split",
                                 "protocol_hash": "protocol", "threshold_hash": "threshold", "code_version": "git"}
        module.publish_predictions_atomic(prediction_path, prediction_frame, prediction_provenance,
                                          ["i1", "i2"], ["p1", "p2"])
        module.validate_completed_predictions(prediction_path, prediction_provenance, ["i1", "i2"], ["p1", "p2"])
        try:
            module.validate_completed_predictions(prediction_path, {**prediction_provenance, "checkpoint_hash": "wrong"},
                                                  ["i1", "i2"], ["p1", "p2"])
        except ValueError as exc:
            assert "provenance mismatch" in str(exc)
        else:
            raise AssertionError("incompatible Stage 5 technical resume was accepted")
        module.mark_final_outcome_access(first_path)
        second_path = workspace / "run-2" / "final_eval_registration.json"
        try:
            module.register_final_evaluation(second_path, {**common, "final_eval_run_id": "run-2"})
        except SystemExit as exc:
            assert "already accessed" in str(exc)
        else:
            raise AssertionError("second methodological run was accepted")


if __name__ == "__main__":
    import traceback
    tests = [(n, f) for n, f in sorted(globals().items()) if n.startswith("test_")]
    passed = failed = 0
    for name, function in tests:
        try:
            function(); print(f"  PASS  {name}"); passed += 1
        except Exception:
            print(f"  FAIL  {name}"); traceback.print_exc(); failed += 1
    print(f"\n{passed} passed, {failed} failed, {len(tests)} total")
    raise SystemExit(1 if failed else 0)


---

# Part 13 - Cloud execution (Modal)

Running the repository's own scripts on rented GPUs, without forking the code.

## `cloud/modal_stage1.py`

*491 lines*

**Stage 1 on Modal, running the repository's own scripts unmodified.**

The design note is the interesting part: rather than porting Stage 1 into a notebook, it runs
`02b_build_sixway_splits.py` and `train_lora_sdxl.py` *inside* the container, so the provenance chain
(frozen split manifest -> LoRA metadata -> generation manifest -> ...) is produced **natively** and
no Stage 1 -> Stage 2 adapter is needed.

Layout: `/root/repo` holds `scripts/`, `configs/`, `environment/` copied from the checkout (the code
identity); `/vol/project` is `PROJECT_ROOT` on a persistent Modal Volume (data, caches, checkpoints,
logs). Nothing under `scripts/` or `configs/` is edited - overlays are written instead
(`_write_overlay`). Entry points: `prepare_data`, `smoke`, `train` (with `_resume_args` /
`_latest_step` for resumption across container restarts), `status`.

In [ ]:
# ===== FILE: cloud/modal_stage1.py =====
"""Stage 1 (SDXL + LoRA) on Modal — runs the repository's own scripts, unmodified.

Why Modal runs the repo scripts directly (not the Kaggle notebook): the repo's Stage 2-5 gates
check a provenance chain (frozen split manifest -> LoRA metadata -> generation manifest -> ...).
Running `02b_build_sixway_splits.py` and `train_lora_sdxl.py` here produces that chain natively,
so no Stage-1 -> Stage-2 adapter is needed.

Layout inside every container
    /root/repo        scripts/ configs/ environment/ copied from this checkout (the code identity)
    /vol/project      PROJECT_ROOT, on a persistent Modal Volume (data, caches, checkpoints, logs)

Nothing under scripts/ or configs/ is edited. Production-only values (namespace, raw_dir,
dev_subset off) are supplied through THESIS_CONFIG_OVERLAY, a file on the volume that
scripts/utils/config.py merges at load time, so `code_identity()` stays equal to this checkout.

One-time setup (on your machine):
    pip install modal
    python -m modal setup
    modal secret create kaggle KAGGLE_USERNAME=<your-user> KAGGLE_KEY=<your-key>

Run from the repository root. `--detach` keeps the job running if your laptop sleeps or the
terminal closes; follow it on the Modal dashboard or with `modal app logs chest-synth-stage1`.

    modal run --detach cloud/modal_stage1.py::prepare_data   # CPU, a few hours, ~$1
    modal run cloud/modal_stage1.py::smoke                   # GPU ~20 min: train, checkpoint, resume
    modal run --detach cloud/modal_stage1.py::train --hours 6
    modal run cloud/modal_stage1.py::status

GPU: A10 (24 GB, bf16) by default — the Starter plan without a payment method allows only T4, L4
and A10. To use another, set STAGE1_GPU before `modal run`, e.g. on Windows cmd:  set STAGE1_GPU=L4
"""

from __future__ import annotations

import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import modal

REPO_LOCAL = Path(__file__).resolve().parents[1]
REPO = "/root/repo"
VOL_MOUNT = "/vol"
PROJECT_ROOT = Path(VOL_MOUNT) / "project"
OVERLAY = PROJECT_ROOT / "thesis_overlay.yaml"
RAW_LOCATION = PROJECT_ROOT / "raw_location.json"

NAMESPACE = "production-thesis-v1"
KAGGLE_SLUG = "ashery/chexpert"
SPLITS = ["gen_train", "gen_val", "classifier_train", "classifier_val",
          "asism_tuning_heldout", "final_eval_heldout"]

CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "stage1_lora_sdxl"
SMOKE_CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "stage1_smoke"
LOG_DIR = PROJECT_ROOT / "logs" / "modal"

# The Starter plan without a payment method may use only T4, L4 and A10 (L40S/A100/H100 are
# refused at launch; see cloud/modal_gpu_probe.py). A10 and L4 both run the frozen bf16 protocol.
GPU = os.environ.get("STAGE1_GPU", "A10")
# $/hour, from modal.com/pricing (per-second rates x 3600). Used only for printed estimates.
GPU_PRICE_PER_HOUR = {"T4": 0.59, "L4": 0.80, "A10": 1.10, "A10G": 1.10, "L40S": 1.95,
                      "A100-40GB": 2.10, "A100-80GB": 2.50, "H100": 3.95}
CPU_PRICE_CORE_HOUR, MEM_PRICE_GIB_HOUR = 0.0000131 * 3600, 0.00000222 * 3600

# Training is GPU-bound (99% utilization measured on A10) and reads pre-cached latents with no
# DataLoader workers, so 2 cores are enough. RAM holds both latent and prompt caches: size it from
# the "trainer loads both caches into RAM" line prepare_data prints (Modal bills max(request, use)).
TRAIN_CPU = 2.0
TRAIN_MEMORY_GIB = int(os.environ.get("STAGE1_MEMORY_GIB", "32"))


def _hourly(cpu: float, memory_gib: float) -> float:
    """Full $/hour of a container: GPU + requested CPU + requested memory."""
    return GPU_PRICE_PER_HOUR.get(GPU, 0) + cpu * CPU_PRICE_CORE_HOUR + memory_gib * MEM_PRICE_GIB_HOUR

# Latent moments are (8, res/8, res/8) fp16; prompt embeddings are (77x2048 + 1280) fp16.
LATENT_BYTES_768 = 8 * 96 * 96 * 2
PROMPT_BYTES = (77 * 2048 + 1280) * 2

_CODE_IGNORE = ["**/__pycache__", "**/*.pyc"]

image = (
    modal.Image.debian_slim(python_version="3.11")
    # Frozen matrix (environment/RUNPOD_MATRIX.md): torch 2.8.0 / torchvision 0.23.0 / CUDA 12.8.
    .pip_install("torch==2.8.0", "torchvision==0.23.0",
                 index_url="https://download.pytorch.org/whl/cu128")
    .pip_install_from_requirements(str(REPO_LOCAL / "environment" / "requirements.txt"))
    .env({"STAGE1_GPU": GPU})   # so the container prints cost estimates for the GPU actually chosen
    .add_local_dir(REPO_LOCAL / "scripts", f"{REPO}/scripts", ignore=_CODE_IGNORE)
    .add_local_dir(REPO_LOCAL / "configs", f"{REPO}/configs", ignore=_CODE_IGNORE)
    .add_local_dir(REPO_LOCAL / "environment", f"{REPO}/environment", ignore=_CODE_IGNORE)
)

volume = modal.Volume.from_name("chest-synth-thesis", create_if_missing=True, version=2)
app = modal.App("chest-synth-stage1")


# ------------------------------------------------------------------------------------------------
# helpers (run inside the container)
# ------------------------------------------------------------------------------------------------

def _utcnow() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def _env() -> dict[str, str]:
    env = dict(os.environ)
    env.update({
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "THESIS_CONFIG_OVERLAY": str(OVERLAY),
        "HF_HOME": str(PROJECT_ROOT / ".cache" / "huggingface"),
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "KAGGLEHUB_CACHE": str(PROJECT_ROOT / ".cache" / "kagglehub"),
        "PYTHONUNBUFFERED": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "TQDM_MININTERVAL": "60",   # one progress line a minute instead of one per step
    })
    return env


def _log_path(name: str) -> Path:
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    return LOG_DIR / f"{name}_{_utcnow()}.log"


def _run(cmd: list[str], log: Path) -> None:
    """Run a repo script from /root/repo, streaming output to stdout and a log on the volume."""
    print(f"\n$ {' '.join(cmd)}", flush=True)
    with open(log, "a", encoding="utf-8") as handle:
        handle.write(f"\n$ {' '.join(cmd)}\n")
        proc = subprocess.Popen(cmd, cwd=REPO, env=_env(), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed (exit {code}): {' '.join(cmd)} — see {log}")


def _find_data_root(base: Path) -> Path:
    """Same search as scripts/data/00_download_dataset.py::find_data_root."""
    for candidate in (base, base / "CheXpert-v1.0-small", base / "CheXpert-v1.0"):
        if (candidate / "train.csv").exists() and (candidate / "train").is_dir():
            return candidate
    for train_csv in base.rglob("train.csv"):
        if (train_csv.parent / "train").is_dir():
            return train_csv.parent
    raise FileNotFoundError(f"No CheXpert train.csv with a sibling train/ under {base}")


def _write_overlay(raw_root: Path) -> None:
    """Production values for configs/stage1_lora_sdxl.yaml, applied without editing the repo.

    These are exactly the three 'revert for production' lines run_dev_subset.sh documents, plus
    raw_dir pointing at the dataset on the volume (no symlinks, no second copy of 11 GB)."""
    OVERLAY.parent.mkdir(parents=True, exist_ok=True)
    OVERLAY.write_text(
        "stage1:\n"
        "  dev_subset:\n"
        "    enabled: false\n"
        "  split:\n"
        f"    namespace: \"{NAMESPACE}\"\n"
        f"    input_csv: \"{raw_root / 'train.csv'}\"\n"
        "  paths:\n"
        f"    raw_dir: \"{raw_root}\"\n",
        encoding="utf-8",
    )


def _require_overlay() -> None:
    if not OVERLAY.is_file():
        raise SystemExit(f"{OVERLAY} is missing. Run prepare_data first:\n"
                         "  modal run --detach cloud/modal_stage1.py::prepare_data")


def _read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _resume_args(ckpt_root: Path) -> tuple[list[str], str | None, int]:
    """Mirror of scripts/train/launch_resumable.sh: continue the latest run from its newest
    full checkpoint. Returns (extra CLI args, run_id, resumed step)."""
    latest_run = ckpt_root / "latest_run.json"
    if not latest_run.is_file():
        return [], None, 0
    run_id = _read_json(latest_run)["run_id"]
    latest = ckpt_root / run_id / "latest.json"
    if not latest.is_file():
        return ["--run-id", run_id], run_id, 0
    info = _read_json(latest)
    return ["--resume-from", info["checkpoint_dir"], "--run-id", run_id], run_id, int(info["step"])


def _latest_step(ckpt_root: Path) -> int:
    _, _, step = _resume_args(ckpt_root)
    return step


def _train_cmd(*overrides: str, extra: list[str] | None = None) -> list[str]:
    return ["accelerate", "launch", "--config_file", "configs/accelerate_config.yaml",
            "scripts/train/train_lora_sdxl.py", *(extra or []),
            f"split.namespace={NAMESPACE}", *overrides]


# ------------------------------------------------------------------------------------------------
# 1. data preparation (CPU)
# ------------------------------------------------------------------------------------------------

@app.function(image=image, volumes={VOL_MOUNT: volume},
              secrets=[modal.Secret.from_name("kaggle")],
              # Billed at max(request, usage). 03_preprocess_images.py is single-process and the
              # split build is a pandas groupby over 224k rows, so 2 cores / 8 GiB is enough.
              cpu=2.0, memory=8192, timeout=24 * 3600)
def prepare_data(splits: str = "all") -> None:
    """Download CheXpert, verify it, build + freeze the production split, preprocess, caption.

    Every step is idempotent: re-running skips what is already done (the split run is immutable,
    03_preprocess_images.py resumes per split, captions are rebuilt deterministically)."""
    # kagglehub runs in THIS process, so its cache location must be set here, not only in the
    # subprocess env — otherwise the 11 GB lands on ephemeral container disk and is lost on exit.
    os.environ.update(_env())
    log = _log_path("prepare_data")
    wanted = SPLITS if splits == "all" else splits.split(",")

    if RAW_LOCATION.is_file():
        raw_root = Path(_read_json(RAW_LOCATION)["raw_root"])
        print(f"dataset already on the volume: {raw_root}", flush=True)
    else:
        import kagglehub

        print(f"downloading {KAGGLE_SLUG} via kagglehub into the volume ...", flush=True)
        download = Path(kagglehub.dataset_download(KAGGLE_SLUG))
        raw_root = _find_data_root(download)
        for archive in download.rglob("*.zip"):   # extracted already; don't keep a second copy
            archive.unlink()
        RAW_LOCATION.write_text(json.dumps({"raw_root": str(raw_root), "slug": KAGGLE_SLUG,
                                            "downloaded_at_utc": _utcnow()}, indent=2))
        volume.commit()
    _write_overlay(raw_root)

    _run([sys.executable, "scripts/data/01_verify_download.py"], log)

    split_dir = PROJECT_ROOT / "data" / "chexpert" / "processed" / "splits" / NAMESPACE
    if split_dir.exists():
        print(f"split run {NAMESPACE} already frozen at {split_dir} — immutable, reusing", flush=True)
    else:
        _run([sys.executable, "scripts/data/02b_build_sixway_splits.py", "--namespace", "production",
              "--run-id", NAMESPACE, "--input-csv", str(raw_root / "train.csv"), "--freeze"], log)
    volume.commit()

    _run([sys.executable, "scripts/data/03_preprocess_images.py", "--namespace", NAMESPACE,
          "--splits", *wanted], log)
    volume.commit()
    _run([sys.executable, "scripts/data/04_generate_captions.py", "--namespace", NAMESPACE,
          "--splits", "gen_train", "gen_val"], log)
    volume.commit()

    manifest = _read_json(split_dir / "split_manifest_v2.json")
    print("\n" + "=" * 78)
    print(f"SPLIT  {NAMESPACE}  frozen={manifest.get('frozen')}  "
          f"support={'PASSED' if manifest.get('support_check', {}).get('passed') else 'FAILED'}")
    print(f"       manifest_hash={manifest['manifest_hash']}")
    for name, n in manifest.get("patients_per_split", {}).items():
        print(f"       {name:24s} {n:>7,} patients  {manifest['images_per_split'][name]:>8,} images")

    captions_dir = PROJECT_ROOT / "data" / "chexpert" / "processed" / "captions" / NAMESPACE
    total_gb = 0.0
    for split in ("gen_train", "gen_val"):
        path = captions_dir / f"{split}_captions.jsonl"
        records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        unique = {c for r in records for c in r["caption_variants"]}
        latent_gb = len(records) * LATENT_BYTES_768 / 1024 ** 3
        prompt_gb = len(unique) * PROMPT_BYTES / 1024 ** 3
        total_gb += latent_gb + prompt_gb
        print(f"CACHE  {split:10s} {len(records):>7,} images -> latents {latent_gb:5.1f} GB, "
              f"{len(unique):>7,} unique captions -> prompts {prompt_gb:5.1f} GB")
    print(f"       trainer loads both caches into RAM: ~{total_gb:.0f} GB "
          f"(train() requests 32 GB and can grow; tell Claude if this is far above 32)")
    print("=" * 78)
    print("Next: review the split above, then  modal run cloud/modal_stage1.py::smoke")


# ------------------------------------------------------------------------------------------------
# 2. smoke test (GPU): train -> checkpoint -> resume -> final export, on 64 images
# ------------------------------------------------------------------------------------------------

@app.function(image=image, gpu=GPU, volumes={VOL_MOUNT: volume},
              cpu=TRAIN_CPU, memory=16384, timeout=2 * 3600)
def smoke(samples: int = 64, optimizer: str = "adamw") -> None:
    """Exercises every Stage 1 code path the long run depends on, in isolated directories
    (checkpoints/stage1_smoke, separate small caches), so it can never be resumed into by train().

    Leg A trains 10 steps and checkpoints at step 10. Leg B resumes from that checkpoint and runs
    to step 20. Passing means bf16, the optimizer, caching, save_state, load_state and the final
    LoRA export all work on this GPU before any paid long run starts."""
    _require_overlay()
    log = _log_path("smoke")
    shutil.rmtree(SMOKE_CKPT_ROOT, ignore_errors=True)
    common = [f"optimizer.name={optimizer}",
              "checkpointing.save_every_n_steps=10", "validation.val_every_n_steps=10",
              f"paths.checkpoints_dir={SMOKE_CKPT_ROOT}",
              f"paths.logs_dir={PROJECT_ROOT / 'logs' / 'stage1_smoke'}"]
    started = time.time()

    print("\n### smoke leg A: fresh run to step 10", flush=True)
    _run(_train_cmd("training.max_train_steps=10", "training.min_train_steps=10", *common,
                    extra=["--max-samples-per-split", str(samples)]), log)
    resume, run_id, step = _resume_args(SMOKE_CKPT_ROOT)
    if step != 10:
        raise SystemExit(f"SMOKE FAILED: expected a checkpoint at step 10, found step {step}")

    print(f"\n### smoke leg B: resume run {run_id} from step {step} to step 20", flush=True)
    _run(_train_cmd("training.max_train_steps=20", "training.min_train_steps=20", *common,
                    extra=["--max-samples-per-split", str(samples), *resume]), log)
    final = SMOKE_CKPT_ROOT / run_id / "final"
    meta = _read_json(final / "metadata.json")
    weights = sorted(final.glob("*.safetensors"))
    if meta.get("step") != 20 or not weights:
        raise SystemExit(f"SMOKE FAILED: final step={meta.get('step')}, weights={weights}")
    volume.commit()

    minutes = (time.time() - started) / 60
    print("\n" + "=" * 78)
    print(f"SMOKE PASSED on {GPU}: trained 10 -> resumed -> 20 -> exported "
          f"{[w.name for w in weights]}")
    print(f"  split_manifest_hash in LoRA metadata: {meta.get('split_manifest_hash')}")
    print(f"  wall time {minutes:.1f} min  (~${minutes / 60 * _hourly(TRAIN_CPU, 16):.2f})")
    print("Next:  modal run --detach cloud/modal_stage1.py::train --hours 6")
    print("=" * 78)


# ------------------------------------------------------------------------------------------------
# 3. full training (GPU), resumable across invocations
# ------------------------------------------------------------------------------------------------

@app.function(image=image, gpu=GPU, volumes={VOL_MOUNT: volume},
              # Training reads pre-cached latents, so CPU load is light; RAM holds both caches.
              # If Modal preempts the container, re-running train() resumes from the last checkpoint.
              cpu=TRAIN_CPU, memory=TRAIN_MEMORY_GIB * 1024, timeout=24 * 3600)
def train(hours: float = 6.0, max_steps: int = 15000, optimizer: str = "adamw") -> None:
    """Continue (or start) the production Stage 1 run.

    Stops cleanly at the first checkpoint written after `hours` have passed, so no steps are ever
    thrown away; call it again to continue. The first invocation also builds the VAE latent and
    text-embedding caches (a one-time GPU cost, reused by every later invocation).

    max_steps: 15000 matches the protocol used in the Kaggle notebook and the config's
    min_train_steps; configs/stage1_lora_sdxl.yaml's max_train_steps (30000) is only an upper
    bound. Keep it identical across invocations — it also sets the cosine LR schedule length.
    optimizer: 'adamw' follows run_dev_subset.sh (the pinned bitsandbytes 0.45.2 has no CUDA 12.8
    kernel); the config documents adamw as the in-protocol fallback for adamw_8bit."""
    _require_overlay()
    if not 0 < hours <= 22:
        raise SystemExit("--hours must be in (0, 22]; the container hard timeout is 24 h")

    resume, run_id, start_step = _resume_args(CKPT_ROOT)
    if run_id and (CKPT_ROOT / run_id / "final" / "metadata.json").is_file():
        meta = _read_json(CKPT_ROOT / run_id / "final" / "metadata.json")
        print(f"Stage 1 already COMPLETE: run {run_id}, step {meta['step']}. Nothing to do.")
        return

    price = _hourly(TRAIN_CPU, TRAIN_MEMORY_GIB)
    print("=" * 78)
    print(f"STAGE 1  namespace={NAMESPACE}  gpu={GPU}  target={max_steps:,} steps")
    print(f"  resume   : {'run ' + run_id + ' from step ' + str(start_step) if resume else 'fresh start'}")
    print(f"  budget   : stop at the first checkpoint after {hours} h  (~${hours * price:.0f}, "
          f"plus up to one checkpoint interval)")
    print("=" * 78, flush=True)

    log = _log_path("train")
    cmd = _train_cmd(f"optimizer.name={optimizer}",
                     f"training.max_train_steps={max_steps}",
                     f"training.min_train_steps={min(15000, max_steps)}",
                     extra=resume)
    print(f"$ {' '.join(cmd)}", flush=True)

    started = time.time()
    deadline = started + hours * 3600
    stop_reason = {"value": None}
    handle = open(log, "a", encoding="utf-8")
    # New session so the whole accelerate -> python process group can be signalled together.
    proc = subprocess.Popen(cmd, cwd=REPO, env=_env(), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)

    def watchdog() -> None:
        step_at_deadline = None
        while proc.poll() is None:
            time.sleep(60)
            if time.time() < deadline:
                continue
            step = _latest_step(CKPT_ROOT)
            if step_at_deadline is None:
                step_at_deadline = step
                print(f"\n[watchdog] {hours} h reached at checkpoint step {step}; "
                      "stopping after the next checkpoint is written", flush=True)
            elif step > step_at_deadline:
                stop_reason["value"] = f"time budget reached; stopped cleanly after checkpoint {step}"
                print(f"\n[watchdog] {stop_reason['value']}", flush=True)
                time.sleep(30)   # let retention cleanup finish after latest.json is written
                os.killpg(proc.pid, signal.SIGTERM)
                try:
                    proc.wait(timeout=180)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL)
                return

    threading.Thread(target=watchdog, daemon=True).start()
    for line in proc.stdout:
        print(line, end="", flush=True)
        handle.write(line)
    code = proc.wait()
    handle.close()
    volume.commit()

    elapsed_h = (time.time() - started) / 3600
    end_step = _latest_step(CKPT_ROOT)
    _, run_id, _ = _resume_args(CKPT_ROOT)
    complete = bool(run_id) and (CKPT_ROOT / run_id / "final" / "metadata.json").is_file()
    print("\n" + "=" * 78)
    if complete:
        print(f"STAGE 1 COMPLETE — final LoRA in {CKPT_ROOT / run_id / 'final'}")
    elif stop_reason["value"]:
        print(f"PAUSED: {stop_reason['value']}")
    else:
        print(f"TRAINER EXITED with code {code} before completion — read {log}")
    print(f"  steps    : {start_step:,} -> {end_step:,} of {max_steps:,} this session")
    print(f"  elapsed  : {elapsed_h:.2f} h  (~${elapsed_h * price:.2f} on {GPU})")
    if end_step > start_step and not complete:
        rate = (end_step - start_step) / elapsed_h
        left_h = (max_steps - end_step) / rate
        print(f"  remaining: ~{left_h:.1f} h  (~${left_h * price:.0f})  at {rate:,.0f} steps/h")
        print("  continue : modal run --detach cloud/modal_stage1.py::train --hours 6")
    print("=" * 78)
    if code != 0 and not stop_reason["value"]:
        raise SystemExit(code)


# ------------------------------------------------------------------------------------------------
# 4. status (CPU, cheap)
# ------------------------------------------------------------------------------------------------

@app.function(image=image, volumes={VOL_MOUNT: volume}, cpu=1, memory=2048, timeout=900)
def status() -> None:
    """What is on the volume and how far Stage 1 has got."""
    print(f"overlay      : {'present' if OVERLAY.is_file() else 'MISSING (run prepare_data)'}")
    if RAW_LOCATION.is_file():
        print(f"dataset      : {_read_json(RAW_LOCATION)['raw_root']}")
    split_manifest = (PROJECT_ROOT / "data" / "chexpert" / "processed" / "splits" / NAMESPACE
                      / "split_manifest_v2.json")
    if split_manifest.is_file():
        m = _read_json(split_manifest)
        print(f"split        : frozen={m.get('frozen')} hash={m['manifest_hash'][:16]}...")
    images = PROJECT_ROOT / "data" / "chexpert" / "processed" / "images_768" / NAMESPACE
    for split in SPLITS:
        d = images / split
        if d.is_dir():
            print(f"  preprocessed {split:24s} {sum(1 for _ in d.glob('*.jpg')):>8,} images")

    _, run_id, step = _resume_args(CKPT_ROOT)
    if run_id:
        final = CKPT_ROOT / run_id / "final" / "metadata.json"
        snaps = sorted((CKPT_ROOT / run_id / "lora_weights").glob("step_*"),
                       key=lambda p: int(p.name.split("_")[-1]))
        print(f"stage 1 run  : {run_id}")
        print(f"  checkpoint : step {step:,}   complete={final.is_file()}")
        print(f"  lora snaps : {[p.name for p in snaps][-5:]}")
    else:
        print("stage 1 run  : not started")

    for label, path in [("raw dataset", PROJECT_ROOT / ".cache" / "kagglehub"),
                        ("hf models", PROJECT_ROOT / ".cache" / "huggingface"),
                        ("processed", PROJECT_ROOT / "data" / "chexpert" / "processed"),
                        ("checkpoints", PROJECT_ROOT / "checkpoints")]:
        if path.exists():
            out = subprocess.run(["du", "-sh", str(path)], capture_output=True, text=True).stdout
            print(f"disk {label:12s}: {out.split()[0] if out else '?'}")

    logs = sorted(LOG_DIR.glob("train_*.log"))
    if logs:
        tail = logs[-1].read_text(encoding="utf-8", errors="replace").splitlines()[-8:]
        print(f"\nlast train log ({logs[-1].name}):")
        for line in tail:
            print(f"  {line[-160:]}")


## `cloud/modal_bench.py`

*494 lines*

**Before/after benchmarks on Modal, with synthetic data.** Every benchmark runs the **original** and
the **optimized** behaviour through the same repo code, in the same container, on the same synthetic
inputs (`_synthetic_cxr_jpegs`, `_set_mode`), so the comparison isolates the change. No CheXpert data
and no Kaggle secret needed.

Reports per mode: wall time, throughput, GPU utilization sampled from `nvidia-smi` (`GpuSampler`),
peak VRAM, peak host RSS, estimated cost (`_cost`) - **plus an equivalence check of the outputs**,
which is what stops a "speed-up" that quietly changed the numbers from looking like a win.

In [ ]:
# ===== FILE: cloud/modal_bench.py =====
"""Before/after benchmarks for the pipeline optimizations, on Modal, with synthetic data.

Every benchmark runs the ORIGINAL behaviour and the OPTIMIZED behaviour through the same repo
code, in the same container, on the same synthetic inputs, so the comparison isolates the change.
No CheXpert data and no Kaggle secret are needed.

    modal run cloud/modal_bench.py::bench_classifier
    set BENCH_GPU=L4 && modal run cloud/modal_bench.py::bench_classifier    (Windows cmd)

Reported per mode: wall time, throughput, GPU utilization (sampled from nvidia-smi), peak VRAM,
peak host RSS, estimated cost; plus an equivalence check of the outputs.
"""

from __future__ import annotations

import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

import modal

REPO_LOCAL = Path(__file__).resolve().parents[1]
REPO = "/root/repo"
GPU = os.environ.get("BENCH_GPU", "A10")   # Starter plan: only T4, L4, A10 are allowed
GPU_PRICE_PER_HOUR = {"T4": 0.59, "L4": 0.80, "A10": 1.10, "A10G": 1.10, "L40S": 1.95,
                      "A100-40GB": 2.10, "A100-80GB": 2.50, "H100": 3.95}
CPU_CORES, MEMORY_GIB = 4.0, 16
CPU_PRICE_CORE_HOUR, MEM_PRICE_GIB_HOUR = 0.0000131 * 3600, 0.00000222 * 3600
_CODE_IGNORE = ["**/__pycache__", "**/*.pyc"]

# Identical layer chain to cloud/modal_stage1.py, so Modal reuses the cached image.
image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("torch==2.8.0", "torchvision==0.23.0",
                 index_url="https://download.pytorch.org/whl/cu128")
    .pip_install_from_requirements(str(REPO_LOCAL / "environment" / "requirements.txt"))
    .env({"STAGE1_GPU": GPU})
    .add_local_dir(REPO_LOCAL / "scripts", f"{REPO}/scripts", ignore=_CODE_IGNORE)
    .add_local_dir(REPO_LOCAL / "configs", f"{REPO}/configs", ignore=_CODE_IGNORE)
    .add_local_dir(REPO_LOCAL / "environment", f"{REPO}/environment", ignore=_CODE_IGNORE)
    .add_local_dir(REPO_LOCAL / "tests", f"{REPO}/tests", ignore=_CODE_IGNORE + ["_runtime/**"])
)
app = modal.App("chest-synth-bench")

# Same Volume and Hugging Face cache path as production (cloud/modal_stage1.py), so SDXL is
# downloaded once and reused by both the benchmarks and the real run.
volume = modal.Volume.from_name("chest-synth-thesis", create_if_missing=True, version=2)
VOL_MOUNT = "/vol"
PROJECT_ROOT = "/vol/project"
HF_CACHE = f"{PROJECT_ROOT}/.cache/huggingface"
SDXL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
SDXL_REVISION = "462165984030d82259a11f4367a4eed129e94a7b"
# Only the files diffusers/transformers load for this pipeline (fp32 weights, as the repo loads
# them). The full repo snapshot also carries ONNX/OpenVINO exports and single-file checkpoints
# (~70 GB); none of those are ever read.
SDXL_FILES = ["model_index.json", "scheduler/*", "tokenizer/*", "tokenizer_2/*",
              "text_encoder/config.json", "text_encoder/model.safetensors",
              "text_encoder_2/config.json", "text_encoder_2/model.safetensors",
              "vae/config.json", "vae/diffusion_pytorch_model.safetensors",
              "unet/config.json", "unet/diffusion_pytorch_model.safetensors"]


def _repo_env() -> None:
    sys.path.insert(0, REPO)
    os.chdir(REPO)
    os.environ.update({"PROJECT_ROOT": PROJECT_ROOT, "HF_HOME": HF_CACHE,
                       "HF_HUB_ENABLE_HF_TRANSFER": "1", "HF_HUB_OFFLINE": "1"})
    os.environ.pop("THESIS_CONFIG_OVERLAY", None)


@app.function(image=image, cpu=4.0, memory=8192, timeout=3600)
def run_tests(smoke: bool = True) -> None:
    """CPU-only: the repo's own test suite and CPU smoke pipeline, against the code as it is now
    on your machine (the same checks the RunPod STOP gate requires)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    env.pop("THESIS_CONFIG_OVERLAY", None)
    commands = [[sys.executable, "-m", "compileall", "-q", "scripts", "tests"],
                [sys.executable, "tests/run_all.py"]]
    if smoke:
        commands.append([sys.executable, "scripts/smoke/run_smoke_pipeline.py", "--phase", "local"])
    for command in commands:
        print(f"\n$ {' '.join(command)}", flush=True)
        code = subprocess.run(command, cwd=REPO, env=env).returncode
        print(f"exit {code}", flush=True)
        if code != 0:
            raise SystemExit(f"FAILED: {' '.join(command)}")
    print("\nALL CHECKS PASSED")


@app.function(image=image, cpu=2.0, memory=4096, timeout=1800)
def run_namespace_regression() -> None:
    """CPU: the namespace regression test on the current code (must pass) and on a copy with the
    `cfg.split_namespace = namespace` fix removed from the six learned stages (must fail) — proof
    that the test detects the bug rather than passing vacuously."""
    import shutil

    test = "tests/test_learned_asism.py::test_learned_stage_uses_cli_namespace_not_config_default"
    stages = ["05_train_learned_asism.py", "06_learn_thresholds_select.py",
              "07b_verify_thresholds_proxy.py", "08_train_threshold_network.py",
              "08b_verify_full_policy_proxy.py", "09_finalize_learned_selection.py"]
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    env.pop("THESIS_CONFIG_OVERLAY", None)

    print("### with the fix (current code): expect 6 passed", flush=True)
    fixed = subprocess.run([sys.executable, "-m", "pytest", "-q", test], cwd=REPO, env=env).returncode

    control = Path("/tmp/repo_without_fix")
    shutil.rmtree(control, ignore_errors=True)
    shutil.copytree(REPO, control)
    for name in stages:
        path = control / "scripts" / "asism" / name
        lines = path.read_text(encoding="utf-8").splitlines(keepends=True)
        kept = [line for line in lines if line.strip() != "cfg.split_namespace = namespace"]
        assert len(kept) == len(lines) - 1, f"expected exactly one fix line in {name}"
        path.write_text("".join(kept), encoding="utf-8")
    print("\n### without the fix (negative control): expect 6 failed", flush=True)
    unfixed = subprocess.run([sys.executable, "-m", "pytest", "-q", test], cwd=control, env=env).returncode

    print(f"\nwith fix exit={fixed} (0 = all passed) | without fix exit={unfixed} (non-zero = detected)")
    if fixed != 0 or unfixed == 0:
        raise SystemExit("namespace regression test did not behave as expected")
    print("NAMESPACE REGRESSION TEST: passes with the fix, fails without it")


@app.function(image=image, gpu=GPU, volumes={VOL_MOUNT: volume}, cpu=CPU_CORES,
              memory=32 * 1024, timeout=3 * 3600)
def run_gpu_smoke() -> None:
    """GPU: the repo's full end-to-end smoke (scripts/smoke/run_smoke_pipeline.py --phase all) on
    the 60-patient fixture — Stage 1 LoRA -> Stage 2 -> auxiliary classifier -> all ASISM signals ->
    learned ASISM 04-09 -> Stage 4 -> Stage 5, every GPU code path at toy size. Fixture pilot is
    approved with --approve-smoke-pilot (namespace dev-smoke-v1 only; never production)."""
    workspace = Path(REPO) / "outputs" / "smoke" / "dev-smoke-v1"
    hf_link = workspace / ".cache" / "huggingface"
    hf_link.parent.mkdir(parents=True, exist_ok=True)
    if not hf_link.exists():
        hf_link.symlink_to(HF_CACHE, target_is_directory=True)   # reuse the SDXL already on the volume
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_HUB_ENABLE_HF_TRANSFER": "1",
           "TQDM_MININTERVAL": "30"}
    env.pop("THESIS_CONFIG_OVERLAY", None)
    started = time.time()
    code = subprocess.run([sys.executable, "scripts/smoke/run_smoke_pipeline.py", "--phase", "all",
                           "--approve-smoke-pilot"], cwd=REPO, env=env).returncode
    minutes = (time.time() - started) / 60
    print(f"\nGPU SMOKE exit={code} in {minutes:.1f} min (~${_cost(minutes * 60):.2f} on {_detected_gpu()})")
    if code != 0:
        raise SystemExit(f"GPU smoke failed (exit {code})")


@app.function(image=image, volumes={VOL_MOUNT: volume}, cpu=2.0, memory=4096, timeout=3600)
def download_models() -> None:
    """CPU-only: fetch the pinned SDXL files into the shared cache (so no GPU idles on a download)."""
    from huggingface_hub import snapshot_download

    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    started = time.time()
    path = snapshot_download(SDXL_ID, revision=SDXL_REVISION, cache_dir=HF_CACHE,
                             allow_patterns=SDXL_FILES)
    volume.commit()
    size = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024 ** 3
    print(f"SDXL {SDXL_REVISION[:8]} -> {path}  ({size:.1f} GiB, {time.time() - started:.0f}s)")


# ------------------------------------------------------------------------------------------------
# measurement helpers
# ------------------------------------------------------------------------------------------------

class GpuSampler:
    """Samples GPU utilization (%) from nvidia-smi, and the TOTAL resident memory of this process
    plus all its children (DataLoader workers included), while a block runs."""

    def __init__(self, interval: float = 0.25):
        self.interval, self.samples, self.rss_peak, self._stop = interval, [], 0, threading.Event()

    def _tree_rss(self) -> int:
        """Physical RAM actually used by the container. Summing RSS over forked DataLoader workers
        would count the parent's copy-on-write pages once per worker, so prefer the cgroup
        counter; fall back to main RSS + each child's unique (USS) memory."""
        for path in ("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"):
            try:
                return int(Path(path).read_text().strip())
            except (OSError, ValueError):
                continue
        import psutil

        me = psutil.Process()
        total = me.memory_info().rss
        for child in me.children(recursive=True):
            try:
                total += child.memory_full_info().uss
            except psutil.Error:
                pass
        return total

    def _loop(self) -> None:
        while not self._stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu",
                                  "--format=csv,noheader,nounits"], capture_output=True, text=True)
            try:
                self.samples.append(float(out.stdout.strip().splitlines()[0]))
            except (ValueError, IndexError):
                pass
            self.rss_peak = max(self.rss_peak, self._tree_rss())
            time.sleep(self.interval)

    def __enter__(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=5)

    @property
    def mean(self) -> float:
        return round(sum(self.samples) / len(self.samples), 1) if self.samples else float("nan")


def _detected_gpu() -> str:
    """The GPU this container really got (BENCH_GPU is only read on your machine)."""
    out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.upper()
    for key in ("H100", "A100", "L40S", "A10", "L4", "T4"):
        if key in out:
            return "A100-40GB" if key == "A100" else key
    return GPU


def _cost(seconds: float) -> float:
    hourly = (GPU_PRICE_PER_HOUR.get(_detected_gpu(), 0) + CPU_CORES * CPU_PRICE_CORE_HOUR
              + MEMORY_GIB * MEM_PRICE_GIB_HOUR)
    return seconds / 3600 * hourly


def _measure(label: str, fn):
    import torch

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    with GpuSampler() as sampler:
        started = time.perf_counter()
        result = fn()
        torch.cuda.synchronize()
        seconds = time.perf_counter() - started
    stats = {"seconds": round(seconds, 2), "gpu_util_pct": sampler.mean,
             "peak_vram_gib": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2),
             "peak_ram_gib": round(sampler.rss_peak / 1024 ** 3, 2),
             "cost_usd": round(_cost(seconds), 4)}
    print(f"  {label:34s} {json.dumps(stats)}", flush=True)
    return result, stats


def _synthetic_cxr_jpegs(directory: Path, count: int, size: int = 768, seed: int = 0) -> list[Path]:
    """CXR-like 768px JPEGs (smooth anatomy-scale structure + fine noise, grey RGB, quality 95 —
    the same size/format 03_preprocess_images.py writes), so decode cost matches the real data."""
    import numpy as np
    from PIL import Image

    directory.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size] / size
    paths = []
    for index in range(count):
        path = directory / f"img_{index:05d}.jpg"
        paths.append(path)
        if path.exists():
            continue
        fx, fy, phase = rng.uniform(2, 9), rng.uniform(2, 9), rng.uniform(0, 6.3)
        base = 0.5 + 0.25 * np.sin(fx * 6.28 * xx + phase) * np.cos(fy * 6.28 * yy)
        base += rng.normal(0, 0.06, (size, size))
        gray = np.clip(base * 255, 0, 255).astype(np.uint8)
        Image.fromarray(gray, mode="L").convert("RGB").save(path, format="JPEG", quality=95)
    return paths


def _set_mode(optimized: bool) -> None:
    """Toggle the optimization flags the repo code reads; '0' restores the original behaviour."""
    import torch

    os.environ["THESIS_LOADER_WORKERS"] = "" if optimized else "0"
    os.environ["THESIS_CUDNN_BENCHMARK"] = "1" if optimized else "0"
    torch.backends.cudnn.benchmark = False   # the repo code turns it on when optimized


# ------------------------------------------------------------------------------------------------
# 1. classifier data loading (scripts/utils/classifier.py)
# ------------------------------------------------------------------------------------------------

@app.function(image=image, gpu=GPU, cpu=CPU_CORES, memory=MEMORY_GIB * 1024, timeout=3600)
def bench_classifier(n_train: int = 1536, n_eval: int = 1536, steps: int = 48,
                     mc_passes: int = 5, mc_images: int = 256) -> dict:
    """Original (num_workers=0) vs optimized (workers + pinned memory + cudnn.benchmark) for the
    three ways the pipeline uses the classifier: training steps, a full evaluation pass, and MC
    Dropout passes. Equivalence: batches must be bit-identical; predictions and trained weights
    may differ only by GPU floating-point noise (measured by repeating the original run)."""
    import numpy as np
    import torch
    from torch.utils.data import DataLoader

    sys.path.insert(0, REPO)
    os.chdir(REPO)
    import scripts.utils.classifier as clf

    print(f"GPU {_detected_gpu()}  cpu={CPU_CORES}  os.cpu_count={os.cpu_count()}  torch {torch.__version__}")
    rng = np.random.default_rng(1)
    paths = _synthetic_cxr_jpegs(Path("/tmp/bench_cxr"), n_train + n_eval)
    labels = clf.CLASSIFIER_TARGET_LABELS
    choices = np.array([1.0, 0.0, -1.0, np.nan])

    def make(p):
        return {"image_path": str(p), "is_synthetic": False,
                "labels": {l: float(rng.choice(choices, p=[.2, .5, .1, .2])) for l in labels}}

    train_records = [make(p) for p in paths[:n_train]]
    eval_records = [make(p) for p in paths[n_train:]]
    mc_records = eval_records[:mc_images]

    # --- equivalence of the data pipeline itself: identical batches, identical order ----------
    def first_batches(optimized: bool, k: int = 12):
        _set_mode(optimized)
        ds = clf.CXRRecordDataset(train_records, resolution=320)
        loader = DataLoader(ds, batch_size=32, shuffle=True,
                            generator=torch.Generator().manual_seed(7),
                            **clf._loader_settings("cuda", None))
        out = []
        for i, b in enumerate(loader):
            out.append(b)
            if i + 1 == k:
                break
        return out

    a, b = first_batches(False), first_batches(True)
    batches_identical = all(torch.equal(x[key], y[key]) for x, y in zip(a, b)
                            for key in ("image", "target", "mask", "index"))
    print(f"data pipeline: first {len(a)} batches bit-identical = {batches_identical}")

    budget_kw = dict(max_steps=steps, batch_size=32, learning_rate=1e-4, eval_every_n_steps=0, seed=42)

    # Warm-up (not measured): CUDA context, ImageNet weight download, first-kernel compilation.
    # Without it the first measured mode would pay one-off costs and look slower than it is.
    _set_mode(False)
    warm = clf.train_classifier(train_records[:64], [], clf.TrainingBudget(**{**budget_kw, "max_steps": 2}),
                                0.2, "imagenet", 320, progress_desc="warmup")[0]
    clf.predict_probabilities(warm, eval_records[:64], 320)
    del warm
    torch.cuda.empty_cache()

    results, models, predictions = {}, {}, {}
    for mode in ("original", "original_repeat", "optimized"):
        optimized = mode == "optimized"
        _set_mode(optimized)
        print(f"\n[{mode}] loader={clf._loader_settings('cuda', None)}")
        model, train_stats = _measure("train %d steps" % steps, lambda: clf.train_classifier(
            train_records, [], clf.TrainingBudget(**budget_kw), 0.2, "imagenet", 320,
            progress_desc=mode)[0])
        probs, eval_stats = _measure("eval pass %d imgs" % n_eval,
                                     lambda: clf.predict_probabilities(model, eval_records, 320)[0])
        (_, mc_std), mc_stats = _measure("mc-dropout %dx%d imgs" % (mc_passes, mc_images),
                                         lambda: clf.predict_probabilities(
                                             model, mc_records, 320, mc_dropout_passes=mc_passes))
        results[mode] = {
            "train": {**train_stats, "img_per_s": round(steps * 32 / train_stats["seconds"], 1)},
            "eval": {**eval_stats, "img_per_s": round(n_eval / eval_stats["seconds"], 1)},
            "mc_dropout": {**mc_stats, "img_per_s": round(mc_passes * mc_images / mc_stats["seconds"], 1)},
        }
        models[mode] = {k: v.detach().float().cpu() for k, v in model.state_dict().items()
                        if v.dtype.is_floating_point}
        predictions[mode] = probs

    def weight_diff(x, y):
        return max(float((models[x][k] - models[y][k]).abs().max()) for k in models[x])

    equivalence = {
        "batches_bit_identical": batches_identical,
        "weights_maxabs_optimized_vs_original": weight_diff("optimized", "original"),
        "weights_maxabs_noise_floor_original_vs_repeat": weight_diff("original_repeat", "original"),
        "probs_maxabs_optimized_vs_original": float(np.abs(predictions["optimized"] - predictions["original"]).max()),
        "probs_maxabs_noise_floor": float(np.abs(predictions["original_repeat"] - predictions["original"]).max()),
    }
    speedup = {phase: round(results["original"][phase]["seconds"] / results["optimized"][phase]["seconds"], 2)
               for phase in ("train", "eval", "mc_dropout")}
    report = {"gpu": _detected_gpu(), "results": results, "speedup": speedup, "equivalence": equivalence}
    print("\n" + json.dumps({"speedup": speedup, "equivalence": equivalence}, indent=2))
    return report


def _load_script(relative: str, name: str):
    import importlib.util

    spec = importlib.util.spec_from_file_location(name, f"{REPO}/{relative}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# ------------------------------------------------------------------------------------------------
# 4. Stage 1 training step: gradient checkpointing and micro-batch shape (trainer's own model)
# ------------------------------------------------------------------------------------------------

@app.function(image=image, gpu=GPU, volumes={VOL_MOUNT: volume}, cpu=CPU_CORES,
              memory=32 * 1024, timeout=3600)
def bench_stage1_step(variants: str = "8x2:on,16x1:on,4x4:off", steps: int = 6,
                      max_train_steps: int = 15000) -> dict:
    """Seconds per optimizer step and peak VRAM for the Stage 1 step (768px, bf16, LoRA r32, EMA),
    built by the trainer's own build_models(). Each variant is micro-batch x accumulation, so every
    optimizer step still averages the loss over 16 samples; ':on'/':off' is gradient checkpointing.
    8x2:on is the frozen configuration. Cached-latent training reads no images, so synthetic
    latents and embeddings give the same compute as the real run."""
    import torch
    import torch.nn.functional as F
    from accelerate import Accelerator
    from diffusers.optimization import get_scheduler

    _repo_env()
    trainer = _load_script("scripts/train/train_lora_sdxl.py", "trainer")
    from scripts.utils.config import load_stage1_config

    cfg = load_stage1_config()
    res = int(cfg.data.resolution)
    models = trainer.build_models(cfg, torch.bfloat16)
    for frozen_only in ("vae", "text_encoder_one", "text_encoder_two"):
        models.pop(frozen_only)
    unet, scheduler = models["unet"], models["noise_scheduler"]

    # Accumulation is done by hand (loss / A per micro-batch, one optimizer step) so the batch
    # shape can change between variants without rebuilding the model; the arithmetic per optimizer
    # step is the same the trainer performs through accelerator.accumulate().
    accelerator = Accelerator(gradient_accumulation_steps=1, mixed_precision="bf16")
    params = [p for p in unet.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=float(cfg.optimizer.learning_rate), weight_decay=float(cfg.optimizer.weight_decay))
    lr_scheduler = get_scheduler(str(cfg.optimizer.lr_scheduler), optimizer=optimizer,
                                 num_warmup_steps=int(cfg.optimizer.lr_warmup_steps), num_training_steps=max_train_steps)
    unet, optimizer, lr_scheduler = accelerator.prepare(unet, optimizer, lr_scheduler)
    ema = trainer.LoraEMA(accelerator.unwrap_model(unet), float(cfg.training.ema.decay))
    device, dtype = accelerator.device, torch.bfloat16

    def batch_of(size):
        return {"latents": torch.randn(size, 4, res // 8, res // 8, device=device, dtype=dtype),
                "prompt_embeds": torch.randn(size, 77, 2048, device=device, dtype=dtype),
                "pooled_embeds": torch.randn(size, 1280, device=device, dtype=dtype),
                "time_ids": torch.tensor([[res, res, 0, 0, res, res]] * size, device=device, dtype=dtype)}

    def optimizer_steps(n, micro, accumulation, fake):
        for _ in range(n):
            for _ in range(accumulation):
                noise = torch.randn_like(fake["latents"])
                timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (micro,), device=device).long()
                noisy = scheduler.add_noise(fake["latents"], noise, timesteps)
                pred = unet(noisy, timesteps, encoder_hidden_states=fake["prompt_embeds"],
                            added_cond_kwargs={"text_embeds": fake["pooled_embeds"], "time_ids": fake["time_ids"]},
                            return_dict=False)[0]
                accelerator.backward(F.mse_loss(pred.float(), noise.float()) / accumulation)
            accelerator.clip_grad_norm_(params, float(cfg.optimizer.max_grad_norm))
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            ema.update(accelerator.unwrap_model(unet))

    price = GPU_PRICE_PER_HOUR.get(_detected_gpu(), 0)
    results = {}
    for variant in [v.strip() for v in variants.split(",") if v.strip()]:
        shape, gc = variant.split(":")
        micro, accumulation = (int(x) for x in shape.split("x"))
        raw = accelerator.unwrap_model(unet)
        raw.enable_gradient_checkpointing() if gc == "on" else raw.disable_gradient_checkpointing()
        optimizer.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        try:
            fake = batch_of(micro)
            optimizer_steps(1, micro, accumulation, fake)   # warm-up
            _, stats = _measure(f"{variant}: {steps} optimizer steps",
                                lambda: optimizer_steps(steps, micro, accumulation, fake))
            per_step = stats["seconds"] / steps
            results[variant] = {**stats, "s_per_step": round(per_step, 3),
                                "stage1_hours": round(per_step * max_train_steps / 3600, 1),
                                "stage1_gpu_cost_usd": round(per_step * max_train_steps / 3600 * price, 1)}
        except torch.cuda.OutOfMemoryError as exc:
            results[variant] = {"oom": True, "error": str(exc).splitlines()[0][:160],
                                "peak_vram_gib": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2)}
            print(f"  {variant}: CUDA OOM on {_detected_gpu()}")
        finally:
            fake = None
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    report = {"gpu": _detected_gpu(),
              "vram_total_gib": round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1),
              "results": results}
    print()
    print(json.dumps(report, indent=2))
    return report


## `cloud/modal_gpu_probe.py`

*22 lines*

**A 20-line utility** that answers "which GPU types will this Modal workspace actually let me
launch?" by running `nvidia-smi` for a few seconds on the requested type. Written because a Starter
plan without a payment method refuses some GPU types at launch time.

In [ ]:
# ===== FILE: cloud/modal_gpu_probe.py =====
"""Which GPU types this Modal workspace may use (the Starter plan without a payment method
refuses some types at launch). Runs `nvidia-smi` for a few seconds on the requested GPU.

    set PROBE_GPU=L4 && modal run cloud/modal_gpu_probe.py
"""

import os

import modal

GPU = os.environ.get("PROBE_GPU", "T4")
app = modal.App(f"gpu-probe-{GPU.lower()}")


@app.function(gpu=GPU, image=modal.Image.debian_slim(), timeout=120)
def probe() -> str:
    import subprocess

    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print(f"PROBE_OK {GPU}: {out}")
    return out


---

# Part 14 - Environment and drivers

Dependencies, and the one script that runs the whole pipeline end to end.

## `environment/requirements.txt`

*31 lines*

**The dependency set.** After a verified working setup, freeze exact versions with
`pip freeze > environment/requirements-lock.txt` - the repository intentionally keeps the loose
requirements and the exact lock as two separate files.

```text
# RunPod application dependencies. The base image supplies the exact torch stack documented in
# RUNPOD_MATRIX.md; do not reinstall torch from this file.

diffusers==0.34.0
transformers==4.49.0
accelerate==1.2.1
peft==0.15.0
bitsandbytes==0.45.2
safetensors==0.5.3
pillow==11.1.0
numpy==1.26.4
pandas==2.2.3
opencv-python-headless==4.11.0.86
scipy==1.15.1
torchmetrics[image]==1.6.1
torchxrayvision==1.3.4
omegaconf==2.3.0
tensorboard==2.18.0
wandb==0.19.6
pyarrow==19.0.0
huggingface_hub==0.28.1
hf-transfer==0.1.9
kagglehub==0.3.8
tqdm==4.67.1
python-dotenv==1.0.1
timm==1.0.19

# Required by the production STOP gate (`python tests/run_all.py`): test_learned_asism.py uses
# pytest fixtures and cannot self-run, so without pytest on the pod the learned-ASISM suite — the
# tests covering the thesis's novel contribution — cannot execute at all.
pytest==8.3.4

```

## `run_dev_subset.sh`

*107 lines*

**The canonical end-to-end pipeline, and the most useful single file for understanding execution
order.** A one-shot dev-subset shakedown of Stages 1-5 for namespace `dev-10k-v1`.

**Re-runnable by design:** every completed step drops a marker in `.pipeline_state/<namespace>/` and
is skipped next time, so if it dies at step N you just run it again and it resumes there. Delete a
marker to force one step to re-run. An `ERR` trap names the step that failed.

The real order it runs:

```
00_download -> 01b_dev_subset -> 02b_splits -> 03_preprocess -> 04_captions
  -> 10_stage1_lora (accelerate, FOREGROUND) -> 11_set_lora_ckpt
  -> 20_recipes -> 21_pilot -> 22_approve_pilot -> 23_generate_full
  -> 30_aux_classifier -> 31_signals -> 32_gonogo
  -> 33_feasibility -> 34_build_subsets -> 35_eval_subsets -> 36_train_learned
  -> 37_fixed_thresh -> 38_contexts -> 39_verify_thresh -> 40_thresh_net
  -> 41_full_policy -> 42_finalize
  -> 50_conditions -> 51_stage5_eval -> 52_compare
```

Note what is **absent**: `scripts/asism/03_tune_freeze_select.py`. It is not a pipeline step.

Two workarounds are documented inline and worth knowing: Stage 1 runs in the **foreground** (not via
`launch_resumable.sh`, which returns immediately and would let Stage 2 start before a checkpoint
exists), and `optimizer.name=adamw` is forced because the RunPod pytorch-2.8/cu128 image ships a
`bitsandbytes` without a CUDA binary.

```bash
#!/usr/bin/env bash
# One-shot dev-subset shakedown pipeline (Stages 1-5) for namespace dev-10k-v1.
#
# Re-runnable: every completed step drops a marker in .pipeline_state/dev-10k-v1/ and is skipped
# on the next run, so if the script dies at step N you just run it again and it resumes at N.
# To force a step to re-run, delete its marker (e.g. rm .pipeline_state/dev-10k-v1/36_train_learned.done).
#
# Run it detached so it survives an SSH drop:
#     export PROJECT_ROOT=/workspace/chest-synth-thesis
#     cd $PROJECT_ROOT
#     nohup bash run_dev_subset.sh > logs/dev10k_run.log 2>&1 &
#     tail -f logs/dev10k_run.log
#
# For the FULL production run instead: revert the three dev_subset lines in
# configs/stage1_lora_sdxl.yaml, use `--namespace production --run-id production-thesis-v1 --freeze`
# in 02b, `--namespace production-thesis-v1` everywhere else, and a >= 150 GB persistent volume.
set -Eeuo pipefail

export PROJECT_ROOT="${PROJECT_ROOT:-/workspace/chest-synth-thesis}"
cd "$PROJECT_ROOT"
mkdir -p logs

NS=dev-10k-v1
STATE="$PROJECT_ROOT/.pipeline_state/$NS"
mkdir -p "$STATE"
AUTO_APPROVE_PILOT="${AUTO_APPROVE_PILOT:-1}"   # dev shakedown: auto-approve the Stage-2 pilot.
                                               # Set to 0 to pause and inspect the pilot images.

CURRENT_STEP="(init)"
trap 'echo "!!! FAILED at step: ${CURRENT_STEP} (exit $?). Fix it, then re-run this script to resume."; exit 1' ERR

run () {                        # run <marker-name> <command...>
  local name="$1"; shift
  CURRENT_STEP="$name"
  if [ -f "$STATE/$name.done" ]; then echo ">>> SKIP  $name"; return 0; fi
  echo; echo "================ $name  ::  $*"; date
  "$@"
  touch "$STATE/$name.done"
  echo ">>> OK    $name"
}

echo "### dev-subset pipeline | namespace=$NS | PROJECT_ROOT=$PROJECT_ROOT | $(date)"
python -c "import torch; assert torch.cuda.is_available(), 'CUDA not available'; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'OK')"

# ---------------- data + Stage 1 (LoRA) ----------------
run 00_download       python scripts/data/00_download_dataset.py
run 01b_dev_subset    python scripts/data/01b_build_dev_subset.py
run 02b_splits        python scripts/data/02b_build_sixway_splits.py --namespace dev --run-id "$NS"
run 03_preprocess     python scripts/data/03_preprocess_images.py --namespace "$NS" --splits gen_train gen_val classifier_train classifier_val asism_tuning_heldout final_eval_heldout
run 04_captions       python scripts/data/04_generate_captions.py --namespace "$NS" --splits gen_train gen_val

# Stage 1 LoRA in the FOREGROUND so the pipeline waits for it. (launch_resumable.sh backgrounds
# training and returns immediately, which lets Stage 2 start before any checkpoint exists.)
# STAGE1_MAX_STEPS keeps the dev shakedown short; the production run uses the config's 15000-30000.
STAGE1_MAX_STEPS="${STAGE1_MAX_STEPS:-2000}"
# optimizer.name=adamw: the RunPod pytorch-2.8/cu128 image ships a bitsandbytes without a CUDA
# binary. train_lora_sdxl.py now probes a real AdamW8bit step and falls back on its own, so this
# override is no longer required to avoid the crash -- it is kept as an explicit, recorded choice
# so this pipeline's optimizer does not depend on what a given pod image happens to ship.
run 10_stage1_lora    accelerate launch --config_file configs/accelerate_config.yaml scripts/train/train_lora_sdxl.py split.namespace="$NS" optimizer.name=adamw training.max_train_steps="$STAGE1_MAX_STEPS" training.min_train_steps="$STAGE1_MAX_STEPS"

# Point Stage 2 at the LoRA checkpoint step 10 produced (02_generate_synthetic_images.py reads
# configs/stage2_generation.yaml directly and takes no CLI override).
set_lora_ckpt () {
  local rid ckpt
  rid=$(python3 -c "import json; print(json.load(open('checkpoints/stage1_lora_sdxl/latest_run.json'))['run_id'])")
  ckpt="checkpoints/stage1_lora_sdxl/$rid/final"
  [ -d "$ckpt" ] || ckpt=$(ls -d "checkpoints/stage1_lora_sdxl/$rid/lora_weights/step_"* 2>/dev/null | sort -V | tail -1)
  [ -n "${ckpt:-}" ] && [ -d "$ckpt" ] || { echo "no LoRA checkpoint under checkpoints/stage1_lora_sdxl/$rid"; return 1; }
  sed -i "s|^\(\s*lora_weights_dir:\).*|\1 $ckpt|" configs/stage2_generation.yaml
  grep -n "lora_weights_dir:" configs/stage2_generation.yaml
}
run 11_set_lora_ckpt  set_lora_ckpt

# ---------------- Stage 2 (synthetic generation) ----------------
run 20_recipes        python scripts/generate/01_sample_label_recipes.py --namespace "$NS"
run 21_pilot          python scripts/generate/02_generate_synthetic_images.py --mode pilot --namespace "$NS"
if [ "$AUTO_APPROVE_PILOT" = "1" ]; then
  run 22_approve_pilot python scripts/generate/02_generate_synthetic_images.py --approve-pilot --reviewer walaa --notes "dev-subset shakedown" --namespace "$NS"
else
  echo ">>> PAUSED for pilot review. Inspect the 4 pilot images, then rerun:  AUTO_APPROVE_PILOT=1 bash $0"
  exit 0
fi
run 23_generate_full  python scripts/generate/02_generate_synthetic_images.py --mode full --namespace "$NS"

# ---------------- auxiliary classifier + Stage 3 (ASISM) ----------------
run 30_aux_classifier python scripts/classify/00_train_auxiliary_classifier.py --namespace "$NS" --max-steps 8000
run 31_signals        python scripts/asism/01_compute_signals.py --signal all --namespace "$NS"
run 32_gonogo         python scripts/asism/02_gonogo.py --namespace "$NS"
run 33_feasibility    python scripts/asism/04_build_utility_subsets.py --phase feasibility --namespace "$NS"
run 34_build_subsets  python scripts/asism/04_build_utility_subsets.py --phase build --namespace "$NS"
run 35_eval_subsets   python scripts/asism/04b_evaluate_utility_subsets.py --phase run --namespace "$NS"
run 36_train_learned  python scripts/asism/05_train_learned_asism.py --namespace "$NS"
run 37_fixed_thresh   python scripts/asism/06_learn_thresholds_select.py --namespace "$NS"
run 38_contexts       python scripts/asism/07_build_threshold_contexts.py --namespace "$NS"
run 39_verify_thresh  python scripts/asism/07b_verify_thresholds_proxy.py --phase run --i-understand-this-trains-real-models --namespace "$NS"
run 40_thresh_net     python scripts/asism/08_train_threshold_network.py --namespace "$NS"
run 41_full_policy    python scripts/asism/08b_verify_full_policy_proxy.py --phase run --i-understand-this-trains-real-models --namespace "$NS"
run 42_finalize       python scripts/asism/09_finalize_learned_selection.py --namespace "$NS"

# ---------------- Stage 4 + Stage 5 ----------------
run 50_conditions     python scripts/classify/01_train_conditions.py --condition all --namespace "$NS"
run 51_stage5_eval    python scripts/eval/stage5_evaluate.py --final-eval-run-id dev10k-final-v1 --namespace "$NS"
run 52_compare        python scripts/eval/compare_conditions.py --run-dir "outputs/stage5/$NS/dev10k-final-v1"

echo
echo "### DONE. Stage 5 comparison in: outputs/stage5/$NS/dev10k-final-v1/"

```

---

# Part 15 - Design documents (verbatim)

The rationale behind every decision above. Reproduced unmodified; the `.md` files in `docs/` remain the source of truth.

## `README.md`

*116 lines*

# Adaptive Quality-Aware Synthetic Data Selection Framework

Master's thesis project (AI, Zewail City University): fine-tuning SDXL with LoRA on CheXpert
chest X-rays, generating synthetic data, and adaptively selecting/weighting it (ASISM) before
classifier training and evaluation. See `docs/proposal.md` for the full 5-stage framework and
`docs/literature_review.md` for the grounding literature.

The repository contains executable code for Stages 1--5. Production thesis results still require
the real CheXpert data, trained checkpoints, generated images, and GPU execution; committed smoke
artifacts validate engineering connectivity only.

Stage 5 compares three conditions on `final_eval_heldout` (`configs/stage4_classifier.yaml` →
`conditions: [A, B, C]`): **A** real only, **B** real + all synthetic, **C** real + ASISM-
selected synthetic. There is no separate weighted-baseline selector or condition — ASISM is the full
module (`docs/stages2_to_5_plan.md` §4.9), including the Multi-Signal Utility Ranking Network and
Adaptive Threshold Learning below. Primary comparison: **C vs. B**. See
`docs/stages2_to_5_plan.md` §7.1 for a known limitation of that comparison (no matched-random
control) still pending a supervisor decision.

Learned-ASISM order (after Stage 2, auxiliary classifier, signal computation, and Go/No-Go):

```bash
python scripts/asism/04_build_utility_subsets.py --phase feasibility
python scripts/asism/04_build_utility_subsets.py --phase build
python scripts/asism/04b_evaluate_utility_subsets.py --phase estimate
python scripts/asism/04b_evaluate_utility_subsets.py --phase run
python scripts/asism/05_train_learned_asism.py
python scripts/asism/06_learn_thresholds_select.py
# 06 is the fixed-ratio learned threshold — an intermediate/ablation artifact, not a Stage 4
# condition. The adaptive path that feeds condition C continues with:
python scripts/asism/07_build_threshold_contexts.py
python scripts/asism/07b_verify_thresholds_proxy.py --phase estimate
python scripts/asism/07b_verify_thresholds_proxy.py --phase run --i-understand-this-trains-real-models
python scripts/asism/08_train_threshold_network.py
python scripts/asism/08b_verify_full_policy_proxy.py --phase estimate
python scripts/asism/08b_verify_full_policy_proxy.py --phase run --i-understand-this-trains-real-models
python scripts/asism/09_finalize_learned_selection.py
```

`scripts/asism/03_tune_freeze_select.py` (the pre-learned weighted-score selector) is retained as
the historical baseline the learned module replaced, and is exercised only by its own tests. Nothing
imports it and nothing in the Stage 4/5 pipeline consumes its output — do not run it as a pipeline
step. The shared signal-merge used by the learned stages (04–09) is
`scripts/asism/candidate_pool.py`, not this script.

## Environment

Target hardware: RunPod, 1x NVIDIA A40 (48GB), PyTorch 2.8.0 / CUDA 12.8 template. All paths that
must survive a pod restart (data, checkpoints, logs, HF cache) are expected to live on the
persistent volume — set `PROJECT_ROOT` (see `configs/stage1_lora_sdxl.yaml`) to wherever this repo
is cloned (e.g. `/workspace/chest-synth-thesis` on RunPod, or the local checkout when developing).

```bash
pip install -r environment/requirements.txt
```

After a verified working setup, freeze exact versions:

```bash
pip freeze > environment/requirements-lock.txt
```

## Pipeline (Stage 1)

Run from the repo root, in order. `NS` is the split namespace every later stage is pinned to
(`dev-<run>` or `production-<run>`):

```bash
NS=production-thesis-v1
python scripts/data/00_download_dataset.py                  # kagglehub download + link into data/chexpert/raw/ (idempotent)
python scripts/data/01_verify_download.py                   # sanity-check the extracted CheXpert-v1.0-small archive
python scripts/data/02b_build_sixway_splits.py --namespace production --run-id "$NS" --freeze
python scripts/data/03_preprocess_images.py --namespace "$NS" --splits gen_train gen_val classifier_train classifier_val asism_tuning_heldout final_eval_heldout
python scripts/data/04_generate_captions.py --namespace "$NS" --splits gen_train gen_val
bash scripts/train/launch_resumable.sh                      # resumable SDXL LoRA training (checks checkpoints/.../latest.json)
```

`scripts/data/02_build_patient_splits.py` is the superseded schema-v1 three-way builder and is not a
pipeline step; `configs/splits.yaml` replaced it. For the full Stage 1–5 sequence, including the dev
namespace, see `run_dev_subset.sh` — it is the executable version of this list.

Evaluation/monitoring:

```bash
python scripts/eval/generate_probe_samples.py --checkpoint <path>
python scripts/eval/compute_fid_clipscore.py --checkpoint <path>
```

## Dataset

CheXpert-v1.0-small, obtained from Kaggle (`ashery/chexpert`) via `scripts/data/00_download_dataset.py`,
which downloads it with `kagglehub`, symlinks (or copies) it into `data/chexpert/raw/`, and runs
`01_verify_download.py` automatically. Requires Kaggle API credentials configured for `kagglehub`
(e.g. `~/.kaggle/kaggle.json` or the `KAGGLE_USERNAME`/`KAGGLE_KEY` env vars). Not included in this repo.
See `docs/stage1_plan.md` §6 for the exact expected structure.

## Layout

```
data/chexpert/{raw,processed}/   # dataset (gitignored except splits/manifests)
configs/                         # YAML configs — single source of truth for tunables
scripts/data/                    # download, verify, six-way splits, preprocess, captions
scripts/train/                   # Stage 1 SDXL + LoRA
scripts/generate/                # Stage 2 label recipes + synthetic generation
scripts/asism/                   # Stage 3 signals, Go/No-Go, learned ASISM (04–09)
scripts/classify/                # auxiliary classifier + Stage 4 conditions
scripts/eval/                    # Stage 1 monitoring + Stage 5 evaluation and analysis
scripts/smoke/                   # CPU/GPU fixture pipelines (non-scientific by construction)
scripts/utils/                   # shared config, provenance, labels, splits, metrics, classifier
tests/                           # contract + unit suites; tests/run_all.py is the STOP gate
cloud/                           # Modal entry points (Stage 1, benchmarks, GPU probe)
notebooks/                       # thesis_full_project_code.ipynb — the whole project, one notebook
checkpoints/stage1_lora_sdxl/    # resumable + inference-ready LoRA checkpoints (gitignored, per-run metadata tracked)
logs/, outputs/                  # TensorBoard logs, probe sample grids (gitignored)
docs/                            # proposal, literature review, this stage's design plan
```

## `docs/project_overview.md`

*198 lines*

# Project overview

A single reference for the whole pipeline as it currently stands, including the four
literature-driven changes (C1–C4). For the design rationale and citations see
`literature_review.md`; for the staged runtime plan see `stages2_to_5_plan.md`.

---

## 1. One-paragraph summary

Fine-tune a diffusion foundation model on chest X-rays, generate label-conditioned synthetic
images, then **select** which of those synthetic images to add to a real training set so that a
multi-label disease classifier improves — fairly across classes, and demonstrably because of
*which* images were chosen rather than *how many*. The selection stage (ASISM) scores every
synthetic image on six quality signals, learns a task-grounded utility from measured downstream
AUROC changes, ranks images, and applies a class-specific admission threshold chosen on
proxy evidence only. `final_eval_heldout` is opened once, for the final Stage 4/5 comparison.

---

## 2. Data splits (frozen)

Six patient-disjoint splits, built by `scripts/data/02b_build_sixway_splits.py`:

| Split | Used for |
|---|---|
| `gen_train`, `gen_val` | Stage 1 LoRA fine-tuning |
| `classifier_train` | real training data for every proxy and real classifier |
| `classifier_val` | model selection during classifier training |
| `asism_tuning_heldout` | **all** Stage 3 tuning evidence (proxy AUROC, threshold policy choice) |
| `final_eval_heldout` | opened once, Stage 4/5 only — never by ASISM |

**Frozen support rule:** every primary-endpoint label has ≥ 50 positive and ≥ 50 negative
patients in every split where it drives a decision. Patient IDs never cross splits.

---

## 3. Stages

### Stage 1 — generation model
`scripts/train/train_lora_sdxl.py`. SDXL fine-tuned with LoRA on CheXpert-derived captions
(`scripts/data/04_generate_captions.py`), bf16, gradient checkpointing, **768**-resolution
training (`configs/stage1_lora_sdxl.yaml` → `data.resolution`; `stage1_plan.md`'s "Resolution:
768, not 1024" note explains why neither 1024 nor 512). Output: a frozen LoRA adapter.

### Stage 2 — synthetic generation
`scripts/generate/01_sample_label_recipes.py` → `02_generate_synthetic_images.py`. Label recipes
are sampled under **rarity-aware per-class quotas**, single- and multi-label compositions,
co-occurrence constraints, and a cap on positives per recipe. Generation uses DPM-Solver
multistep, ~40 steps, guidance ≈ 7. Output: a synthetic image pool (~5k in the production
namespace) plus a manifest with each image's `intended_label_vector`.

### Stage 3 — ASISM (this thesis)

`scripts/asism/01`–`09`.

#### 3.1 Six per-image signals (`01_compute_signals.py`)

| # | Signal | Method | Feature columns |
|---|---|---|---|
| 1 | **Similarity** | DINOv2 ViT-S/14 embeddings; k-NN (k = 15) vs. a per-label real reference pool; hierarchical fallback when exact-match references are scarce | `similarity_knn_mean`, `similarity_top1`, `similarity_topk_spread` |
| 2 | **Image quality** | blank / clipping / low-contrast / uniform-border / blur (Laplacian) checks | `iqa_composite`, `iqa_sharpness`, `iqa_contrast_std` |
| 3 | **Uncertainty** | auxiliary DenseNet121 + MC-Dropout, 20 passes; predictive std over the primary labels; moderate uncertainty is *not* penalised | `uncertainty_mean_std` |
| 4 | **Explainability** | Grad-CAM from the auxiliary classifier vs. empirically derived expected pathology regions | `explainability_region_overlap` |
| 5 | **Agreement** | mean P(intended positives) − penalty · mean P(confident *unintended* positives) | `agreement_score` |
| 6 | **Distinctiveness** | within-class k-NN redundancy (k = 10); `1 − mean top-k similarity` | *(design signal; its dedicated feature column is the piece still being wired into the learned feature set — the learned ranker currently consumes the 9 columns of signals 1–5)* |

A **near-duplicate / memorisation flag** fires when `similarity_top1 ≥ 0.95`; such images get no
novelty credit and are rejected (`reject_near_duplicates`). The 0.95 cut-off is grounded in
Dar et al., *Nature Biomedical Engineering* 2025 (**C3**).

#### 3.2 Go/No-Go gate (`02_gonogo.py`)

Each signal must pass checks on missingness, distinct-value count, a directionality probe,
redundancy correlation with other signals, and reproducibility. Verdict per signal:
`include` (model input), `ablation_only` (kept for ablation, excluded from the frozen selector),
or `exclude` (dropped). Frozen in `gonogo_report.json` / `asism_frozen_manifest.json`.

#### 3.3 Subset-utility measurement (`04` + `04b`, GPU)

The candidate pool is split into **train-role** and **val-role** image pools that are disjoint by
construction (`split_image_pool`). About 120 controlled subsets (random / single-signal / mixed,
sizes 100–250) are built, and each subset's **real downstream utility** is measured as
`augmented_macro_AUROC − real_only_macro_AUROC` — a proxy classifier trained on
`classifier_train` + the subset, evaluated on `asism_tuning_heldout` only. After an exposure
filter (each image in ≥ 3 subsets), ≈ 96 usable `(subset, utility)` pairs remain.

#### 3.4 Learned utility and ranking (`05_train_learned_asism.py`)

- **Set-Utility Network** — permutation-invariant Deep Sets, trained on the measured subset
  utilities (train-role subsets, early-stopped on val-role). Default: masked mean pooling.
  **C1 (ablation flag, default off):** masked mean *and* std pooling plus a fixed full-pool
  summary concatenated to the utility head (Xie et al., ICLR 2024).
- **Target generation** — per-image supervision for the ranker is the size-normalised
  leave-one-out marginal utility `(U(S) − U(S∖i))·(n−1)` by default. A **Data-Banzhaf MSR**
  estimator (`banzhaf_msr_targets`) — mean utility of subsets containing an image minus mean
  utility of subsets not containing it — is computed from the same measured subsets at no extra
  GPU cost; the LOO-vs-Banzhaf Spearman correlation is recorded in the training manifest, and
  Banzhaf stays opt-in until that diagnostic is reviewed.
  **C2 (default on):** the active target is transformed before the loss —
  `target_normalization ∈ {"standardize", "rank", "none"}` (default `"standardize"`), plus an
  optional `target_winsorize_quantile` — so one noisy proxy measurement cannot dominate training
  on ≈ 96 points. All transforms are order-preserving. `"none"` restores the prior behaviour.
- **Multi-Signal Utility Ranking Network** — small MLP, `9 → 128 → 64 → 32 → 1` (9 = the admitted
  signal feature columns: similarity ×3, IQA ×3, uncertainty, explainability, agreement), one scalar
  utility target, optimised by Smooth-L1 + 0.5 · pairwise-ranking loss. It is *not* a
  multi-objective / Pareto model.

#### 3.5 Adaptive class threshold (`06`, `07`/`07b`, `08`/`08b`, `09`)

- `06_learn_thresholds_select.py` — a fixed-target-ratio learned threshold; an internal ablation
  reference, **not** a Stage 4 condition.
- `07`/`07b` — per-class bootstrap **contexts** (resamples of one image pool, honestly tagged
  `independent_clinical_sample: False`), each a 10-dim vector (score-distribution stats + real
  patient prevalence + budget). A critic-guided hard-threshold grid search proposes candidates;
  `07b` (GPU) proxy-verifies a *diversified* subset of the grid (not the critic's top-k alone).
- `08_train_threshold_network.py` — **Adaptive Threshold Network**: 16-dim class embedding +
  context → 64 → 32 → 1 → sigmoid. Trained by supervised distillation on proxy-verified targets
  only. **Three-tier per-class governance** (`determine_per_class_official_method`):

  | Verified evidence for a class | Threshold source |
  |---|---|
  | ≥ 3 proxy-verified contexts on **both** train and image-disjoint held-out sides, and frozen acceptance criteria pass | learned Adaptive Threshold Network |
  | 1–2 verified contexts | median best proxy-verified threshold |
  | 0 verified contexts | fixed baseline threshold |

- `08b` (GPU) — for each candidate **threshold policy** it builds the final multi-label
  `selected_manifest`, trains real + selection, and measures utility on `asism_tuning_heldout`
  over several seeds. Policies: fixed baseline, hard-proxy-best, adaptive network,
  `literal_top_50_percent`, and `freematch_style_adaptive_percentile`.
  **C4:** the FreeMatch-style policy scales per-class leniency by *range-normalised* real
  prevalence (rarest present class → most lenient), and `enforce_per_class_selection_floor`
  lowers a class's threshold after thresholding if it would otherwise admit fewer than
  `min_selected_per_label` — the one-time-curation analogue of SST's class-fairness term
  (Zhao et al., IP&M 2025).
- `09_finalize_learned_selection.py` — applies the pre-registered tie rule
  (`tie_noise_band` + `simplicity_order`) to the `08b` measurements, commits to **one** winning
  policy, and writes the frozen `adaptive_selected_manifest.jsonl`. It never opens
  `final_eval_heldout`.

### Stages 4–5 — final comparison
`scripts/classify/01_train_conditions.py`, `scripts/eval/stage5_evaluate.py`. Real classifiers,
equal optimiser-step budget across all conditions and seeds, model selection on `classifier_val`,
final metric on `final_eval_heldout`:

Enabled conditions are `configs/stage4_classifier.yaml` → `conditions: [A, B, C]`:

| Condition | Training data | State |
|---|---|---|
| **A** | real only (`classifier_train`) | enabled |
| **B** | real + **all** Stage 2 synthetic images | enabled |
| **C** | real + ASISM-selected synthetic (`adaptive_selected_manifest`) | enabled |
| **D** | real + a random synthetic subset with **C's per-class size and composition** | **INACTIVE** — machinery retained and working, not enabled; see §7.1 |

Primary comparison **C vs B**. **D** would isolate selection quality from subset size, which is the
§7.1 limitation of that comparison — it is not currently run, and re-enabling it requires supervisor
sign-off. Confirmatory tests use Holm–Bonferroni; exploratory use Benjamini–Hochberg.

---

## 4. The four literature-driven changes (C1–C4)

| ID | Change | Paper | State |
|---|---|---|---|
| **C1** | Set-Utility Network: optional mean+std pooling + full-pool summary | Xie et al., ICLR 2024 | flag `superset_conditioning`, **default off**, ablation only |
| **C2** | Ranking-target normalization (`standardize` / `rank` / `none`) + winsorize | 2D-OOB (NeurIPS 2024); Chi et al. (ICML 2026); Banzhaf robustness | **default `standardize`** — the one change to the frozen path; `"none"` reverts |
| **C3** | `near_duplicate_similarity = 0.95` grounded + pinned by a test | Dar et al., Nature Biomed. Eng. 2025 | documentation + test, **no logic change** |
| **C4** | FreeMatch-style policy: range-normalised leniency + per-class selection floor | SST, IP&M 2025 | active **inside the FreeMatch policy only** (`08b`/`09`) |

C5 (importance-weighted redundancy, InfoMax ICLR 2025) and C6 (inter-class similarity margin,
CosSIF CiBM 2024) are designed but not implemented; they would be ablation flags, default off.

---

## 5. Invariants the design enforces

- **Image-disjoint everywhere.** Train-role vs. val-role subsets share no images; threshold
  contexts come from one pool only.
- **Proxy-only tuning.** Every Stage 3 decision uses `asism_tuning_heldout`; `final_eval_heldout`
  is opened once, for Stage 4/5.
- **No pseudo-replication.** The ranker is never trained on a subset AUROC copied onto its member
  images; it is distilled from the Set-Utility Network's measured set-level utility.
- **Frozen manifests + hashes** at every step; a config change alters the config hash and forces
  a re-run from the affected stage forward.
- **Conservative governance.** The learned threshold network is used for a class only with
  sufficient verified, image-disjoint evidence *and* passing pre-registered acceptance criteria;
  otherwise the pipeline falls back.
- **One winning policy.** The fixed / FreeMatch / learned threshold policies do not each become a
  final-evaluation condition; proxy evidence selects one, which becomes Condition C.

---

## 6. Test and run status

- `python tests/run_all.py` — 186 passed / 0 failed (includes all C1–C4 tests).
- `python scripts/smoke/run_smoke_pipeline.py --phase local` — passes on CPU.
- Full smoke and the real `01`→`09` sequence need a GPU (Stage 1 SDXL, and the `--phase run`
  proxy-training steps `04b` / `07b` / `08b`).

## `docs/proposal.md`

*70 lines*

# Thesis Project Context

## Who I am
I am a Master's student in AI at Zewail City University. This is my master's thesis project.
I am implementing this framework myself, step by step, in order to generate results. After I
finish getting results, I will write the paper based on this implementation and the results.

## Thesis Title
An Adaptive Quality-Aware Synthetic Data Selection Framework for Medical Image Classification
Using Diffusion Foundation Models

## Dataset
CheXpert-v1.0-small — 224,316 chest radiographs from 65,240 patients, frontal and lateral views,
with 14 pathology labels (including uncertainty labels: -1, 0, 1). Located at `data/chexpert/`.

## Framework Overview
The framework consists of five stages. Implement them **one stage at a time, in order** —
do not jump ahead to a later stage until the current one is working and I've confirmed it.

### Stage 1 — Generative Model Setup
- Stable Diffusion XL (SDXL)
- LoRA Fine-Tuning on CheXpert chest X-rays

### Stage 2 — Synthetic Data Generation
- Generate synthetic medical images (chest X-rays) using the fine-tuned SDXL+LoRA model

### Stage 3 — Adaptive Synthetic Image Selection Module (ASISM) [Novel Contribution]
This is the core novel contribution of the thesis. It combines multiple algorithms to decide
which synthetic images are good enough to keep:
- CLIP / DINOv2 Similarity Scoring
- Image Quality Assessment (IQA)
- Uncertainty Estimation (Monte Carlo Dropout or Deep Ensembles)
- Grad-CAM / Score-CAM Explainability Verification
- Multi-Signal Utility Ranking Network (Novel)
- Adaptive Threshold Learning (Novel)

**Naming revision (2026-08-21) — terminology only, no method change.** The fifth component was
originally listed as "Multi-Objective Ranking Network". It was renamed after code review found the
original name technically incorrect for what is implemented: the network has **one output head**
trained against **one scalar target** (marginal utility distilled from the set-utility model), and
the quality signals are input *features*, not objectives. There is no Pareto front and no
multi-objective optimizer, so a reviewer in optimization would flag the original name on first read
of the source. The method, architecture, and training procedure are unchanged — only the name.
Genuine multi-objective variants (Pareto non-dominated selection, or a multi-head network with MGDA
gradient balancing) are recorded as future options in `docs/stages2_to_5_plan.md` §4.9, not
implemented.

### Stage 4 — Classifier Training
- Train the classifier using only the synthetic images selected by ASISM

### Stage 5 — Evaluation
Compare classifier performance across three training conditions:
- Real images only
- Real + all synthetic images (unfiltered)
- Real + selected synthetic images (ASISM-filtered)

## What I need from you
- Implement actual, runnable code for each stage — not just descriptions or pseudocode.
- Read `docs/literature_review.md` before starting, and ground design choices (e.g. what
  the ASISM module should look for, what baselines to compare against) in the papers
  summarized there — especially Rehman et al. (LoRA-tuned SD on CheXpert/MIMIC-CXR) and
  Niemeijer et al. (TSynD, uncertainty-guided synthesis), since my thesis directly extends
  the gap they leave open: neither optimizes how synthetic and real data should be
  selected and mixed after generation.
- If something in current published literature (post your training data) would change an
  implementation choice, flag it — but don't assume you have live search; tell me what to
  look up myself if you're not sure your information is current.
- Work stage by stage. After each stage, stop and let me review results before continuing.
- The end goal is a full set of experimental results (Stage 5 comparisons) that I will use
  to write a research paper for publication.

## `docs/novelty_target_decision.md`

*74 lines*

# Supervisor decision required before learned ASISM components

## Resolution status (2026-08-21)

**Option 1 below has been implemented in code**, as of commit `d14488f`: `scripts/asism/models.py`
(`SetUtilityNetwork`, `MultiSignalUtilityRankingNetwork`, `AdaptiveThresholdNetwork`), the pipeline
`scripts/asism/04_build_utility_subsets.py` through `09_finalize_learned_selection.py`, and
`scripts/asism/learned.py`. This is documented as `docs/stages2_to_5_plan.md` §4.9 (v4 revision
note) and reflected in `configs/stage3_asism.yaml` and Stage 4 condition C
(`docs/stages2_to_5_plan.md` §7).

**What this resolves:** the pseudo-replication concern raised below. `MultiSignalUtilityRankingNetwork`
(renamed from `MultiObjectiveRankingNetwork` — it fuses the admitted signals into one scalar utility
target, it is not a multi-task/Pareto model) is never trained on a subset-level AUROC copied onto
every member image; it is distilled from `SetUtilityNetwork`'s measured *set-level* utility only —
either the leave-one-out marginal or the optional Data-Banzhaf MSR estimate over the same measured
subsets (see `learned.py`'s module docstring, `banzhaf_msr_targets`, and `resolve_verified_only_targets`,
which admits a target only from a proxy-*measured* candidate, never a critic guess). `AdaptiveThresholdNetwork` is trained per class only where that class clears
`min_verified_contexts_per_class` on both proxy-verified train and image-disjoint held-out evidence
(`determine_per_class_official_method`); otherwise selection falls back to
`hard_proxy_best_among_verified` or the fixed-ratio baseline — a class is never handed to the network
on thin evidence.

**What remains open:** this document's original ask was for the supervisor to select the target
*and approve the additional ASISM tuning budget* before implementation proceeded. That approval was
not recorded before this code was written — implementation proceeded ahead of the sign-off this
document itself required. Before treating the learned selector (Stage 4 condition C) as anything
more than a candidate method for the thesis, get explicit supervisor confirmation of: (a) Option 1
as the accepted target/claim, (b) the compute budget actually spent (`configs/stage3_asism.yaml` →
`compute_budget`, `verification_compute_budget`, `full_policy_verification.compute_budget`), and (c)
whether to add the matched-random control described in `docs/stages2_to_5_plan.md` §7.1 — without
it, a C-over-B result cannot separate ASISM's ranking quality from the effect of using fewer
synthetic images. This status update records what exists; it is not a substitute for that approval.

**v2 note (2026-08-21):** the weighted-score selector referred to below is no longer part of the
thesis pipeline (§4.9 / §7 v3 revision notes) — the thesis defines ASISM as the full module
including both learned components, so there is no weighted-baseline condition. The memo below is
preserved as written for the historical record.

The original decision memo is preserved below, unchanged.

---

The repository does not currently contain a scientifically defensible supervised target for an
image-level ranking network or a class-aware threshold model. The available labels say what SDXL
was prompted to generate; they do not say whether adding a particular synthetic image improves a
real-image classifier. The bounded proxy search produces a noisy *subset-level* macro-AUROC for a
small number of candidate selectors. Treating that value as if it were an image-level target would
duplicate it across every selected image, create pseudo-replication, and invent causal credit that
the experiment did not observe.

For that reason no neural model has been added under the names “Multi-Objective Ranking Network” or
“Adaptive Threshold Learning.” The current `weighted_score_baseline` remains exactly a baseline.

Two defensible alternatives are available:

1. **Pre-registered weakly supervised set-utility learning.** A small permutation-invariant network
   consumes sets of normalized ASISM vectors and predicts patient-fold proxy macro-AUROC. A
   differentiable class-aware gate supplies thresholds. Training and validation use nested,
   patient-disjoint ASISM tuning folds, and comparisons retain individual, equal-weight,
   tuned-weight, and fixed-percentile baselines. This requires many more pre-registered subset
   trials; its target is learned set utility, not ground-truth image quality.
2. **Independent image-level utility labels.** Obtain blinded clinician pairwise quality/usefulness
   judgments, or pre-register a costly marginal-contribution/influence experiment in which each
   image or small batch receives an out-of-fold classifier-utility estimate. Train a ranking loss
   and class-aware threshold model on those targets with patient/source-group separation. This has
   a clearer image-level claim but needs new annotations or substantially more GPU computation.

Minimal decision: the supervisor must choose the intended target and claim—set-level proxy utility
or independently measured image-level utility—and approve the additional ASISM tuning budget/data
collection. Alternatively, the supervisor may explicitly approve weighted-score and proxy-tuned
selection as baselines and remove the literal learned-network requirements. Pareto ranking is not
implemented. Implementation should
not proceed until that decision is frozen in a new protocol version.

## `docs/stage1_plan.md`

*366 lines*

# Stage 1 Implementation Plan: SDXL + LoRA Fine-Tuning on CheXpert

## Context

This is Stage 1 of a 5-stage Master's thesis framework ("An Adaptive Quality-Aware Synthetic
Data Selection Framework for Medical Image Classification Using Diffusion Foundation Models").
The project directory is currently empty of code — no `.py`/`.ipynb` files, no dependency files,
no git repo, no dataset on disk (only `Proposal · MD`, `Literature review .md`, and a
`.claude/agents/` folder with three generic, unrelated subagent templates that specify no
diffusion/LoRA tooling or hardware conventions to build on).

Stage 1's job is to fine-tune SDXL with LoRA on CheXpert chest X-rays so that later stages can
generate controllable, label-conditioned synthetic images (Stage 2), score/select them (Stage 3 —
the thesis's novel ASISM contribution), train a classifier on selected synthetic data (Stage 4),
and run a real-vs-synthetic comparison (Stage 5). Because every later stage depends on what
Stage 1 produces and how it's structured, this plan optimizes not just for a working fine-tune,
but for **provenance and reproducibility** (so Stage 5's comparison isn't contaminated),
**resumability** (RunPod pods can be interrupted), and **clean handoff** (Stage 2 must reuse
Stage 1's exact captioning logic and weights deterministically).

Confirmed constraints from the user:
- **Hardware:** RunPod, PyTorch 2.8.0 template, 1× A40 (48GB VRAM), CUDA 12.8, 9 vCPU, 50GB RAM,
  80GB ephemeral container disk + a separate persistent volume. VS Code over SSH; Claude Code runs
  on the instance directly.
- **Dataset:** CheXpert-v1.0-small, not yet downloaded, to be obtained from Kaggle.
- **Captioning:** structured label-to-text templates (matching Rehman et al.'s tabular-to-text
  approach), not VLM-generated captions.
- Every design decision below states the choice, why, alternatives considered, why they were
  rejected, and trade-offs — per explicit user request.

Grounding papers (from `Literature review .md`): **Rehman et al.** (LoRA-tuned SD on
CheXpert/MIMIC-CXR, tabular-to-text prompts, 24GB→8GB memory reduction vs. full fine-tuning,
Grad-CAM-verified shift away from shortcut features) is the direct template, extended here from
SD to SDXL. **Niemeijer et al.** (TSynD) establishes that downstream stages need *controllable,
label-conditioned* generation, not just realism — so Stage 1 must preserve exact
label↔caption↔image provenance. The literature's repeated warning that **realism ≠ diagnostic
correctness** (Pozzi et al., Hosseini & Serag) is why Stage 1's validation is deliberately scoped
to training stability/plausibility only, deferring diagnostic fidelity to Stage 3/5.

No code is written in this planning turn. This plan is implemented only after explicit approval.

---

## 1. Papers and design choices Stage 1 is based on

- **Rehman et al.**: adopt the mechanism (LoRA on a diffusion UNet + structured label→text
  captions), swap base model SD→SDXL (the thesis's stated extension), swap available fields
  (age/sex/frontal-lateral + 14 CheXpert pathologies, since race isn't a CheXpert field).
- **Niemeijer et al. (TSynD)**: doesn't change Stage 1's architecture, but requires Stage 1 to
  preserve label↔caption↔image provenance so Stage 2 can do targeted/conditional generation and
  Stage 3 can later reason about which label combinations were under/over-represented.
- **Structural-conditioning theme** (edges/masks improve anatomical consistency): CheXpert has no
  segmentation masks, so ControlNet-style conditioning is explicitly **deferred**, not dropped —
  revisit if Stage 3's Grad-CAM/quality review later reveals systematic anatomical inconsistency.
- **"Realism ≠ diagnostic correctness"**: shapes §9 — Stage 1 validates training stability and
  qualitative plausibility only; diagnostic fidelity is Stage 3 (Grad-CAM/IQA) and Stage 5's job.

## 2. SDXL model variant

**Choice: `stabilityai/stable-diffusion-xl-base-1.0` UNet only (no refiner), pinned revision
hash, VAE upcast to fp32 during encode/decode.**

| Alternative | Why rejected |
|---|---|
| Base + refiner two-stage | Refiner specializes in natural-photo high-frequency detail touch-up; no evidence it helps grayscale radiographs, and doubles the LoRA-training surface. |
| SDXL-Turbo / SDXL-Lightning (distilled) | Distilled for few-step *inference* latency, which offline batch generation (Stage 2) doesn't need; training tooling for distilled variants is less mature; risks breaking the distillation guarantee. |
| SSD-1B / Vega (smaller distilled UNet) | Exist to fit smaller GPUs — irrelevant with 48GB headroom; reduced capacity risks losing fine pathology texture that Grad-CAM/classifier stages depend on. |
| SD 1.5 / SD 2.1 | Thesis title commits to SDXL; SDXL's dual text encoder and native 1024px training give more expressive conditioning than what Rehman et al. (SD) used — this thesis's stated extension. |
| fp16 checkpoint + `sdxl-vae-fp16-fix` | Valid alternative to the well-documented SDXL-VAE fp16 NaN bug; kept as a documented Stage-2 speed option, not Stage-1 default — fp32 VAE upcast is diffusers' standard, dependency-free mitigation. |

**Trade-off:** SDXL is heavier than SD1.5 and has two text encoders to manage; mitigated by the
A40's headroom and by caching text embeddings (§8).

## 3. LoRA configuration

**Choice: rank 32, alpha 32 (scale 1.0), target = UNet self+cross attention projections
(`to_q,to_k,to_v,to_out.0`) only, text encoders frozen, dropout 0.0, LR 1e-4 with warmup+cosine
decay, AdamW8bit (bitsandbytes) primary with plain AdamW fallback.**

| Decision | Why | Alternatives rejected |
|---|---|---|
| Rank 32 | Middle ground for 14-pathology, multi-view diversity — more capacity than single-subject DreamBooth (rank 4–8) needs, without eroding LoRA's memory-efficiency motivation (the reason Rehman et al. chose LoRA at all) via near-full-rank (64–128). | Rank 4–8 underfits label diversity; rank 64–128 defeats the memory-efficiency point and bloats per-checkpoint files. |
| Alpha = rank | Keeps alpha/rank scaling identity simple; tune convergence via LR only. | Alpha = 2×rank: no principled justification, conflates two knobs. |
| Self+cross attention only | Field-standard for SDXL LoRA; cross-attention binds prompt→content, self-attention governs spatial/anatomical layout — literature emphasizes anatomical consistency. | Cross-attention-only leaves spatial reasoning to natural-image-pretrained weights (poor match for CXR layout statistics). LoRA on conv/ResNet blocks: unnecessary complexity for unvalidated gain. |
| Text encoders frozen | Constrained template vocabulary doesn't need "medical understanding," just stable distinguishable embeddings; avoids catastrophic forgetting that would hurt Stage 2's generalization to unseen label combos; mirrors Rehman et al.'s efficiency framing. | Text-encoder LoRA (rank 8–16): adds VRAM/complexity to a second sensitive component with no evidence yet it's needed — **documented fallback** if v1 samples show weak conditioning on rare labels (e.g. Lung Lesion, Pleural Other). |
| Dropout 0.0 | Dataset scale (100k+ images) plus low-rank re-parameterization already regularizes. | Dropout 0.1: unnecessary at this scale, slows convergence; revisit only if val loss diverges from train loss. |
| LR 1e-4, warmup+cosine | Standard, well-tested starting point for rank-16–32 SDXL UNet LoRA; LoRA tolerates higher LR than full fine-tuning. | Prodigy (adaptive, LR-free): less battle-tested for SDXL LoRA, harder to debug; kept as a documented experiment, not default. |
| AdamW8bit primary | Extra memory headroom for batch/resolution/caching. | Plain AdamW: fine too since LoRA's trainable param count is small — kept as explicit fallback given real risk that bitsandbytes wheels lag brand-new CUDA 12.8/torch 2.8. Adafactor: not a validated choice in the SDXL-LoRA ecosystem. |

## 4. Folder structure

Rule: **everything that must survive a pod restart lives on the persistent volume** (conventionally
`/workspace`); container disk holds only OS/disposable scratch. Git repo is initialized on the
persistent volume (code/configs/small manifests tracked; raw/processed images, checkpoints, caches
`.gitignore`d — each checkpoint's own metadata JSON records the git commit hash for provenance
without committing binaries).

```
/workspace/chest-synth-thesis/
  data/chexpert/raw/                    # untouched Kaggle extract
  data/chexpert/processed/images_768/   # preprocessed images
  data/chexpert/processed/splits/       # gen_train.csv, gen_val.csv, classifier_heldout.csv, split_manifest.json
  data/chexpert/processed/captions/     # train_captions.jsonl, caption_template_version.json
  configs/stage1_lora_sdxl.yaml, accelerate_config.yaml, dataset_config.yaml
  scripts/data/       01_verify_download.py, 02_build_patient_splits.py, 03_preprocess_images.py, 04_generate_captions.py
  scripts/train/      train_lora_sdxl.py, launch_resumable.sh
  scripts/eval/       generate_probe_samples.py, compute_fid_clipscore.py
  scripts/utils/      caption_builder.py (importable — reused verbatim by Stage 2), seed.py, manifest.py
  checkpoints/stage1_lora_sdxl/<run_id>/checkpoint-<step>/, lora_weights/, latest.json, final/
  logs/stage1_lora_sdxl/<run_id>/       # tensorboard events
  outputs/stage1_samples/<run_id>/      # fixed-seed probe grids
  environment/requirements.txt, requirements-lock.txt
  docs/literature_review.md, proposal.md
  .cache/huggingface/                   # HF_HOME redirected here, not container disk
  .gitignore, README.md
```

`run_id` = timestamp + short git sha + config hash.

**Disk budget:** raw CheXpert-small ~11GB, processed/padded images ~10–20GB, SDXL base weights
~7–14GB, cached VAE latents+text embeddings for ~130k images ~15–20GB, checkpoints (LoRA-only
weights tiny; full resumable states capped at last 3) a few GB. Realistic total ~60–100GB —
**provision a ≥150–200GB persistent volume.**

## 5. Python libraries and versions

Torch 2.8.0+cu128 is already provided by the RunPod template — do not reinstall/downgrade.

| Package | Pin | Notes |
|---|---|---|
| diffusers | `>=0.31,<0.35` | Reference SDXL LoRA training script + PEFT-backed LoRA, reused by Stage 2 |
| transformers | `>=4.44,<4.50` | Two SDXL text encoders |
| accelerate | `>=0.34` | Mixed precision, grad accumulation, save/load_state for resume |
| peft | `>=0.13` | LoRA backend used by diffusers |
| bitsandbytes | `>=0.43` | **Verify CUDA 12.8 compat at setup**; fallback to `torch.optim.AdamW` if it fails |
| xformers | **not installed** | Use PyTorch 2.x SDPA (`AttnProcessor2_0`) instead — matches torch version by construction, avoids a cu128 wheel-availability risk |
| safetensors, pillow, numpy, pandas, opencv-python-headless, scipy | latest stable | Preprocessing |
| torchmetrics[image] | latest stable | FID, CLIP-score |
| torchxrayvision | latest stable | CXR-domain feature extractor for domain-appropriate FID (§9) |
| omegaconf | latest stable | Nested YAML config + CLI overrides |
| tensorboard | latest stable | Primary, offline-durable logging |
| wandb | latest stable (optional) | Supplementary remote dashboard |
| huggingface_hub, tqdm, python-dotenv | latest stable | Pinned model download, secrets, progress |

**Currency caveat:** exact compatible-version matrix for SDXL LoRA + torch 2.8/cu128 should be
verified with a Day-1 smoke test (few steps, few images), then frozen via
`pip freeze > requirements-lock.txt` — versions above are reasonable anchors, not guarantees.

## 6. Dataset preprocessing pipeline

- **Verification first:** since the exact Kaggle mirror isn't confirmed yet, `01_verify_download.py`
  checks structure rather than a checksum: row counts (224,316 train + 234 valid), column schema
  matches known CheXpert fields, spot-check image openability/dimensions, compare label prevalence
  against published CheXpert statistics.
- **View filtering:** **frontal-only default for v1.** Downstream classifier benchmarks
  predominantly use frontal images; mixing frontal+lateral without an explicit view tag would blur
  output distribution given their different pathology-appearance statistics. Lateral images are
  filtered (not deleted), flagged in the manifest — reversible. Rejected default alternative:
  include both with view as a caption tag (roughly doubles conditioning combinatorics for v1).
- **Resolution: 768, not 1024.** Source images are ~320–390px; upsampling to 1024 adds no real
  information and costs more compute. 512 considered but discards more available detail than
  necessary. First ablation candidate, not fixed.
- **Aspect handling: aspect-preserving resize + minimal letterbox padding, NOT non-uniform
  stretch-to-square and NOT center-crop.** Non-uniform stretch directly distorts the
  cardiothoracic ratio (central to Cardiomegaly assessment) — unacceptable given downstream stages
  depend on faithful pathology content. Center-crop risks losing peripheral pathology (e.g. a
  costophrenic-angle effusion, apical pneumothorax).
- **No horizontal flip augmentation** — flips invert left-right anatomy (heart position, aortic
  arch, gastric bubble); must be explicitly disabled since diffusers' reference script may default
  to flips.
- **Quality filtering:** verify openability, normalize grayscale→3-channel RGB, discard/flag
  images below a minimum resolution, flag likely-corrupt frames (near-uniform pixel statistics).
  Every filtering decision logged with image ID + reason — nothing silently dropped.
- **Normalization:** pixels to [-1, 1] before VAE encoding; VAE fp32 upcast (§2).
- **Patient-level split (most consequential decision in Stage 1):** split by patient ID, not image
  ID, into `gen_train` (~70%), `gen_val` (~10%, held out from training, used for val loss/qualitative
  monitoring), and `classifier_heldout` (~20%, **never touched by Stage 1 at all**). This quarantine
  is what keeps Stage 5's eventual real-vs-synthetic comparison uncontaminated — if the generator
  had ever seen a patient later used in Stage 5's real-only test set, that's a bias risk. The
  official CheXpert `valid.csv` (234 curated images) is similarly reserved untouched, for possible
  benchmark comparability later. Split seed, ratios, and code version persisted in
  `split_manifest.json`.

## 7. Caption generation strategy

Template: `"A {view} chest X-ray of a {age_bucket} {sex} patient. Findings: {finding_clause}. {device_clause}"`
- `age_bucket`: decade-binned (not raw integer) to reduce vocabulary sparsity while still encoding
  demographics, echoing Rehman et al.
- `finding_clause`: positive labels (see uncertainty policy below) sorted into CheXpert's canonical
  column order, joined with natural "X, Y, and Z" grammar (more stable CLIP embeddings than a comma
  token-dump); "no acute cardiopulmonary abnormality" when `No Finding == 1`, which is treated as
  authoritative over any spuriously co-positive fields (a known NLP-label-extraction artifact).
- `device_clause`: "Support devices are present." appended separately, not folded into pathology
  list, since it's not a disease finding.

**Uncertainty label (-1) policy — chosen: omit from the finding clause (treat as "not mentioned,"
neither asserted present nor absent).**
- **Why:** a -1 label means the original radiologist wasn't confident from the actual image —
  i.e. no reliable visual signature exists for that case. Asserting "possible X" in the caption
  binds a text token to a visually inconsistent signal, weakening the clean caption↔image
  association that Stage 2/3 need for controllable, trustworthy generation.
- **Rejected alternative 1** — hedged text ("possible X"): more literal, but pollutes the concept
  as above.
- **Rejected alternative 2** — blanket U-Ones/U-Zeros mapping: factually false for a real fraction
  of cases; worse for image-text pair training than for classifier label-noise tolerance, since it
  actively teaches a wrong association rather than just adding noise to a loss average.
- **Trade-off:** reduces effective positive-caption prevalence for labels with high -1 rates (e.g.
  Enlarged Cardiomediastinum, Lung Opacity) — accepted limitation. The raw ternary label is
  preserved unmodified in the manifest regardless, so this is purely a captioning-time policy,
  revisable later without redoing image preprocessing.
- **Template diversity:** 3–5 paraphrase variants randomly selected per image per epoch (label
  semantics fixed) — cheap, standard mitigation against memorizing prompt structure.

## 8. Training pipeline

- **Base implementation:** adapt diffusers' `examples/text_to_image/train_text_to_image_lora_sdxl.py`
  (not kohya-ss sd-scripts — kohya is oriented toward artistic/anime LoRA workflows, harder to
  integrate custom label-driven captioning and less citable/auditable for a thesis methodology
  section than the HF-maintained diffusers+accelerate+PEFT stack). Custom layer adds: CheXpert
  `Dataset` class, RunPod-aware resume logic, YAML-driven config, TensorBoard/W&B hooks, periodic
  qualitative + FID/CLIP-score callbacks.
- **Execution:** single A40 via HF Accelerate (single-process) — even without multi-GPU need,
  Accelerate provides the `save_state`/`load_state` resume primitives the RunPod-interruption
  requirement needs.
- **Precision: bf16** (not fp16) — A40 is Ampere, supports bf16 natively; avoids the numerical
  range issues behind the well-documented SDXL-VAE fp16 NaN bug, combined with fp32 VAE upcast.
- **Attention:** PyTorch 2.x SDPA, not xformers (per §5 rationale).
- **Batch size / grad accumulation for 48GB:** only LoRA params carry optimizer state (frozen
  base contributes activations only), so activation memory dominates, not parameter memory — SDXL
  LoRA is known to fit even 24GB cards at modest batch sizes. **Default: resolution 768, batch size
  8, grad accumulation 2 (effective 16)** with gradient checkpointing on. First lever to raise if
  headroom allows: batch size. First lever to drop on OOM (before resolution): batch size.
- **Latent/embedding caching:** precompute/cache VAE latents + text embeddings once per
  image/template-variant (captions are deterministic-from-labels) — removes VAE/text-encoder
  forward passes from the training hot loop, trading ~15–20GB disk for faster steps and lower peak
  memory.
- **EMA:** enabled on LoRA adapter weights only (decay ~0.9999) — cheap since LoRA weights are
  small; treated as an ablation toggle, not fixed. **Warmup is on** (`training.ema.warmup`): the
  decay at update k is `min(0.9999, (1+k)/(10+k))`, diffusers' `EMAModel` schedule. A fixed 0.9999
  from step 1 averages in the zero-initialised LoRA, which does nothing: `0.9999**n` of that no-op
  init survives in the shadow (~82% at 2,000 steps, ~22% at 15,000), and the exported `final/`
  LoRA is the shadow. With warmup the shadow tracks roughly the last 10% of steps instead (at
  15,000 steps the decay is ~0.9994, so the 0.9999 cap is not reached within the planned budget).
  The update count is saved with the EMA state; resume refuses a checkpoint whose EMA schedule
  differs from the config. Tested in `tests/test_stage1_ema.py`.
- **Duration:** step-based, not epoch-based (gen_train is 100k+ images even after frontal-only
  filtering). Initial budget ~15,000–30,000 steps; actual stop point driven by val loss plateau +
  stable qualitative probes (§9), not a pre-decided epoch count.

## 9. Validation strategy

Deliberately scoped to *training convergence + controllability*, not clinical correctness
(explicitly deferred to Stage 3/5 per the literature's realism≠correctness warning):
1. **Training stability:** train + `gen_val` denoising MSE loss trends down, plateaus, no
   NaNs/divergence.
2. **Qualitative plausibility:** fixed probe-prompt set (one per pathology-positive, no-finding,
   a few multi-label combos), fixed seed, regenerated every checkpoint interval for direct
   checkpoint-to-checkpoint comparison — manually reviewed by the user, not formal radiologist
   review (explicitly out of scope for Stage 1).
3. **Informal directional conditioning check:** e.g. "cardiomegaly"-conditioned samples should
   visibly show larger cardiac silhouette more often than "no finding" samples — a sanity check,
   not a statistical claim.
4. **CLIP-score trend**, tracked relative to itself across checkpoints only — flagged as a weak
   signal since general-domain CLIP wasn't trained on medical images; useful for regressions, not
   an absolute bar.
5. **FID trend**, two variants: standard Inception-FID (field-standard, comparable to other work,
   but caveated as a poor fit for grayscale medical images), and a **domain FID using
   `torchxrayvision`'s DenseNet121 features** as the primary/more meaningful variant — a more
   literature-appropriate response to "FID doesn't reflect diagnostic content."

Explicitly NOT claimed by Stage 1 validation: diagnostic correctness, faithful pathology anatomy,
absence of shortcut learning — those are Stage 3/5 questions.

## 10. Checkpoints and logging

- **Cadence:** full resumable Accelerate state (optimizer state, LoRA weights, RNG, scheduler,
  step/epoch counters) every 500–1000 steps — affordable since only LoRA params carry optimizer
  state. **Retention:** last 3 full-resumable checkpoints; **all** lightweight inference-ready
  LoRA-weights-only snapshots kept (tiny, tens of MB) for possible Stage 2 multi-checkpoint sampling.
- **Naming:** `checkpoints/stage1_lora_sdxl/<run_id>/checkpoint-<step>/`. Each checkpoint's metadata
  JSON records step, epoch, wall-clock time, git commit hash, fully resolved config, dataset split
  manifest hash, seed, exact library versions. `latest.json` always points at the newest
  full-resumable checkpoint.
- **Resume on RunPod:** all paths point to the persistent volume by construction (§4), so nothing
  needed for resume lives only on ephemeral container disk. `launch_resumable.sh` checks
  `latest.json` on start, passes `--resume_from_checkpoint` if present. Training launched under
  `nohup`/`tmux` with output to a persistent-volume log file, so VS Code SSH disconnects don't kill
  training. **Recommend deliberately killing training early once and verifying resume actually
  works**, before committing to a long run.
- **Logging:** **TensorBoard required baseline** (fully offline, no network dependency, event files
  on persistent volume, inspectable via VS Code extension or port-forwarding). **W&B optional
  supplement** (via Accelerate's multi-tracker support) for cross-run comparison during rank/alpha/LR
  sweeps and remote progress checks — not a replacement for TensorBoard, since it needs network
  egress/API key and an external account-linked history.

## 11. Expected outputs (handoff to Stage 2)

- Final LoRA adapter weights (UNet-only safetensors) + manifest: base SDXL revision hash, LoRA
  config, step count, split manifest reference, library/version pins.
- `caption_builder.py` as an **importable module**, not a one-off script — Stage 2 must call the
  identical function to build generation prompts; any drift from train-time phrasing degrades
  conditioning fidelity.
- Patient-level split manifests, especially `classifier_heldout.csv`, intact and untouched by
  Stage 1.
- Training logs/metrics archive (loss curves, probe sample grids, FID/CLIP-score trends) as a
  durable record for the thesis write-up.
- `requirements-lock.txt` and the exact resolved config YAML for the final run.

## 12. Risks and alternatives

| Risk | Mitigation |
|---|---|
| OOM despite headroom planning | Priority order: reduce batch size → increase grad accumulation → drop resolution 768→512 → more aggressive gradient checkpointing. Latent caching already reduces peak memory. |
| Caption-template shortcut learning (e.g. burned-in L/R film markers) | Not Stage 1's job to solve (that's Rehman et al./Stage 3 Grad-CAM's role) — Stage 1 preserves enough provenance for Stage 3 to trace any suspicious correlation. |
| CheXpert label noise (NLP-extracted, not human-verified) | Propagates into caption noise by construction; accepted, documented limitation. Human-curated `valid.csv` preserved untouched for future use needing cleaner labels. |
| Disk exhaustion on RunPod | Budgeted ~60–100GB realistic use (§4); provision 150–200GB volume; checkpoint retention capped; latent caching is a disk-vs-compute toggle, disable if tight. |
| Pod interruption mid-training | Frequent cheap checkpoints, all paths on persistent volume, nohup/tmux survival, deliberate resume test before long runs (§10). |
| xformers/torch mismatch | Avoided by design — SDPA default instead of a cu128-fragile xformers wheel. |
| bitsandbytes/CUDA 12.8 compatibility | Smoke-test at setup; plain AdamW fallback is low-cost since LoRA's trainable param count is small. |
| diffusers/PEFT/Accelerate API churn | Pin exact versions after Day-1 smoke test, freeze via `requirements-lock.txt` rather than trusting memorized version numbers. |

**Higher-level alternatives rejected:**
- **Full UNet fine-tuning:** contradicts thesis title (LoRA specified) and Rehman et al.'s
  motivating memory-efficiency finding; higher overfitting risk on templated captions.
- **DreamBooth-style fine-tuning:** designed for few-shot single-concept personalization, not a
  large multi-label dataset needing fine-grained multi-attribute control.
- **ControlNet/structural conditioning:** valuable per the literature, but CheXpert has no
  segmentation masks (would need a separate lung-segmentation sub-project) — deferred, revisit if
  Stage 3 reveals systematic anatomical inconsistency LoRA-only can't fix.
- **CXR-domain-pretrained backbone** (e.g. RoentGen-style) instead of vanilla SDXL: real
  alternative, but rejected as primary since the thesis commits to SDXL and no widely validated
  SDXL-native CXR checkpoint is confidently known to exist; kept as a possible future ablation.

---

## Critical files to be created

- `scripts/data/02_build_patient_splits.py` — patient-level split logic guarding Stage 5's
  eventual real-vs-synthetic validity; must be built and verified first.
- `scripts/utils/caption_builder.py` — importable captioning module reused identically by Stage 1
  (training) and Stage 2 (generation prompts); highest-leverage correctness point for
  train/generation consistency.
- `scripts/train/train_lora_sdxl.py` — core LoRA training loop (diffusers/Accelerate-based),
  encoding the §3/§8 decisions.
- `configs/stage1_lora_sdxl.yaml` — single source of truth for every tunable decision (rank,
  alpha, LR, resolution, batch size, checkpoint cadence).
- `scripts/data/03_preprocess_images.py` — aspect-preserving resize/pad + quality filtering (§6),
  directly protecting diagnostic content (e.g. cardiothoracic ratio) later stages depend on.

## Verification (once implemented)

1. Run `01_verify_download.py` against the extracted Kaggle download; confirm row counts and label
   schema match expectations.
2. Run `02_build_patient_splits.py`; confirm no patient ID appears in more than one split, and
   `classifier_heldout` size matches the target ~20%.
3. Run `03_preprocess_images.py` and `04_generate_captions.py` on a small sample (~100 images);
   manually inspect a handful of processed images (no stretch distortion, no lost anatomy at
   letterbox edges) and captions (correct finding clause, correct -1 omission, correct device
   clause).
4. Day-1 smoke test: run `train_lora_sdxl.py` for a few dozen steps on a small subset; confirm no
   OOM, no NaNs, checkpoint save succeeds, and `pip freeze` gets locked to `requirements-lock.txt`.
5. Deliberately kill the smoke-test run and restart via `launch_resumable.sh`; confirm training
   resumes from `latest.json` with matching step/loss continuity — validates the RunPod-resume
   requirement before any long run is trusted.
6. After a longer run begins accumulating checkpoints, inspect the fixed-seed probe grids across
   checkpoints for the informal directional-conditioning sanity check (§9.3).

## `docs/stages2_to_5_plan.md`

*583 lines*

# Stages 2–5 Specification: Splits, Generation, ASISM, Classifier Experiments, Evaluation

This document is the **frozen methodology specification** implemented by the Stage 2–5 code in this
repository. Decisions here are fixed, not open. Where a value is configurable, the config key is
named and its **frozen v1 default** is stated.

Style follows `docs/stage1_plan.md`: decision → rationale → alternatives rejected → trade-off.

ASISM (Stage 3) is a **proposed adaptive multi-signal framework designed to investigate whether the
identified synthetic-selection gap can be addressed** (`docs/literature_review.md`: Niemeijer et
al./TSynD leave synthetic-real selection and balancing unresolved). Whether it does is an empirical
question, answered by the pre-specified primary comparison in §8, not asserted here.

---

## 1. Split architecture — FROZEN

### 1.1 Why the previous architecture was replaced

The superseded split (`split_manifest.json`, schema v1) allocated `gen_train` 0.70 + `gen_val` 0.10
+ `classifier_heldout` 0.20 = 100% of patients, with `asism_tuning_heldout` and `final_eval_heldout`
derived as children of `classifier_heldout`. That partition leaves no population for a classifier
development split, and deriving one from `gen_train` would mean the headline classifiers' early-stopping
and threshold decisions were made on patients the generator itself trained on.

**Replacement: six direct, mutually exclusive, patient-level top-level partitions**, drawn from one
deterministic patient list with one recorded seed. **No split is derived from another.**

| Alternative rejected | Why |
|---|---|
| Carve `classifier_train`/`classifier_val` from `gen_train` | Classifier development data would be patients the generator trained on — generator/classifier development contamination. |
| Reuse `asism_tuning_heldout` as `classifier_val` | Collapses ASISM tuning and classifier model selection onto one population. |
| Reuse `gen_val` as `classifier_val` | `gen_val` is the generator's monitoring split; same contamination argument. |
| Keep v1 and forgo a classifier development split | Forces headline-condition model selection onto training data (invalid) or a heldout split (leakage). |

### 1.2 Frozen v2 allocation

| Split | Fraction | Role |
|---|---|---|
| `gen_train` | **0.48** | Stage 1 LoRA training; Stage 3 real-reference pool (§4.1); auxiliary classifier training (§2) |
| `gen_val` | **0.10** | Stage 1 monitoring; auxiliary classifier validation (§2) |
| `classifier_train` | **0.15** | Real component of A/B/C training (§7); real component of ASISM proxy training (§4.7) |
| `classifier_val` | **0.09** | A/B/C early stopping, checkpoint selection, threshold selection, hyperparameter/model selection — identical rules across all conditions |
| `asism_tuning_heldout` | **0.09** | ASISM Go/No-Go evidence (§4.6) and proxy-search evaluation (§4.7) |
| `final_eval_heldout` | **0.09** | Stage 5 final evaluation only (§8) |

Config: `configs/splits.yaml` → `fractions.*`. Sum asserted == 1.0 at build time.

**v2 revision note (production run against the full CheXpert-v1.0-small cohort, 2026-08-08):**
the original v1 allocation (60/10/15/5/5/5) failed the §1.3 support-feasibility rule for
`Lung Lesion` and `Atelectasis` — their confident-negative patient population is real but thin
enough (≈660 and ≈580 patients dataset-wide, respectively) that a 5% decision-bearing split fell
short of the required 50. Raising `classifier_val`/`asism_tuning_heldout`/`final_eval_heldout` from
5% to 9% each (funded by lowering `gen_train` from 60% to 48%) clears both with margin. A third
failing label, `Pleural Other`, could not be fixed this way — its dataset-wide confident-negative
population is only ≈100 patients, so no split-fraction choice gives four disjoint decision-bearing
splits 50 each. `Pleural Other` is excluded from `primary_endpoint_label_set` instead (§5.2) rather
than distorting the split policy further for a label no fraction choice can support.

### 1.3 Support-feasibility rule — FROZEN

Before production splits are frozen, each label in `primary_endpoint_label_set` (§5.2) must satisfy,
**in every split where that label drives a decision or metric** (`classifier_train`,
`classifier_val`, `asism_tuning_heldout`, `final_eval_heldout`):

- **≥ 50 positive patients**, and
- **≥ 50 negative patients**.

**Patient counts, not image counts, determine eligibility.** Image counts are reported as
supplementary information only. Positivity is at patient level: a patient is positive for a label if
any of their studies carries a positive (`1`) value for it.

If the verified full dataset fails this rule under the frozen fractions, the split-freeze command
**stops at the freeze boundary**, emits the complete support report, and proposes the smallest
justified fraction adjustment. Frozen fractions are never silently modified.

### 1.4 Development vs. production namespaces — FROZEN

Split artifacts are namespaced by `split_namespace`:

- **`production`** — requires the source CSV to pass the dataset-integrity contract for the complete
  intended CheXpert-v1.0-small training cohort (`configs/dataset_config.yaml` →
  `source.expected_train_rows`, within tolerance). Written to `splits/production/`. Only these are
  thesis splits.
- **`dev`** — the same 60/10/15/5/5/5 policy applied to the development subset, for pipeline smoke
  testing. Written to `splits/dev/`. Rare-label support insufficiency is **marked in the manifest**
  (`support_check.passed = false`, per-label failures listed). Never called production-frozen.

The two namespaces never overwrite each other. Downstream stages record which namespace they
consumed and refuse to mix.

### 1.5 Superseded artifacts

Schema-v1 artifacts (`splits/*.csv`, `splits/split_manifest.json`) are **retained, not deleted**, and
marked superseded by `splits/SUPERSEDED_v1.md`. No Stage 2–5 code reads them. New manifests carry
`manifest_version: 2` and a `split_namespace` field; a mismatch between an artifact's recorded split
hash and the current frozen manifest is a **hard error** via
`scripts/utils/artifact_contracts.py` (`require_manifest_fields`, `require_score_artifact`,
`require_generation_complete`).

### 1.6 `final_eval_heldout` access rule — FROZEN

`final_eval_heldout` **is** accessed during the one-time split-construction procedure, for exactly:
deterministic patient assignment; disjointness and exact-partition assertions; the §1.3
positive/negative patient-support check; schema/integrity validation; hashing and manifest creation.

**After the manifest is frozen**, no code may load its images, load its labels for outcome
computation, compute outcome statistics, or make any model/method decision from it before Stage 5.
Before Stage 5, only non-outcome-bearing existence/schema/hash verification is permitted.

This is enforced programmatically by `scripts/utils/splits.py::assert_final_eval_access_allowed()`,
which requires an explicit access-purpose token and refuses outcome-bearing purposes unless a Stage 5
final-evaluation run ID is supplied. The claim "the file is never accessed" would be false and is not
made.

---

## 2. Auxiliary real-only reference classifier — ASISM prerequisite

Stage 3's uncertainty (§4.3), explainability (§4.4), and agreement (§4.5) signals require a
real-data classifier. This is that dependency: trained once, frozen, reused. It is **not** condition
A; condition A is trained separately, from scratch, on `classifier_train` under §7's frozen protocol.

- **Data:** `gen_train` train / `gen_val` validate. Never `classifier_train`, `classifier_val`, or
  any heldout split. Rationale: the generator's own splits are the correct home for a scorer of the
  generator's output, and this leaves the entire classifier development pool unspent for A/B/C.
- **Initialization:** default **ImageNet-pretrained** DenseNet121. A CXR-pretrained checkpoint is
  permitted only with documented training-set provenance proving no overlap with our evaluation
  patients. CheXpert-pretrained checkpoints are **rejected by default** — most publish no
  patient-level provenance, so overlap with `final_eval_heldout` cannot be excluded. Provenance
  (source, hash, datasets, license) is recorded in the run manifest.
- **Architecture:** DenseNet121 with dropout **activatable at inference**, so MC Dropout (§4.3) is
  genuinely available rather than nominally specified.
- **Label semantics:** identical to §5/§6 — the scorer speaks the same label language as the
  experiments it feeds.

---

## 3. Stage 2 — synthetic generation

**Terminology:** conditioning labels are `intended_label_vector`, never "ground truth." §4.5
measures intent-vs-content agreement; it does not establish clinical truth.

**Recipe validity — auditable, every decision recorded with a reason:**
1. **Empirical co-occurrence** from `gen_train` at or above `min_support_patients`.
2. **Medical-rule overrides**, both directions: an allow-list for clinically plausible combinations
   rare in the data, and a block-list for combinations judged likely CheXpert NLP-extraction noise.
3. **`No Finding` recipes are the all-zero intended vector over `GENERATION_TARGET_LABELS`**, and
   are mutually exclusive with any positive pathology.
4. **`Support Devices`** may be carried as a conditioning/context attribute. It does **not** affect
   the primary disease-agreement score (§7) unless `secondary_agreement.enabled` is turned on.
5. No `-1` intent is ever encoded in a recipe.
6. Frontal-only, matching Stage 1's `view_filter`.
7. Age/sex coverage and per-recipe quotas are config-driven, oversampling rare classes subject to
   the support rule.

**v3 revision note (2026-08-08):** recipe eligibility (co-occurrence mining, quotas, the
single-label guarantee, and the caption row) is keyed to `GENERATION_TARGET_LABELS`
(`scripts/utils/labels.py`) — `PRIMARY_ENDPOINT_LABELS` plus `INSUFFICIENT_SUPPORT_LABELS` — not
`PRIMARY_ENDPOINT_LABELS` alone. `Pleural Other`'s exclusion from the primary endpoint (§5.2)
reflects real-cohort support scarcity for *evaluation*; it is not a reason to also stop generating
or training on synthetic examples of it. §4.5 agreement scoring remains scoped to
`PRIMARY_ENDPOINT_LABELS` only, unchanged — as does the matched-random marginal-matching logic, if
the §7.1 control is re-enabled.

**Pilot gate — FROZEN:** full production generation **refuses to start** unless a versioned
`pilot_approval_manifest.json` exists, records a completed pilot, records passing automatic artifact
checks, and carries an explicit approval record. Enforced at runtime in
`02_generate_synthetic_images.py`.

**Captions** are built by `scripts/utils/caption_builder.py` verbatim — the same module Stage 1
trained with. Any drift between train-time and generation-time phrasing degrades conditioning
fidelity (`docs/stage1_plan.md` §11).

---

## 4. Stage 3 — ASISM

### 4.0 Artifacts — FROZEN

Five independent versioned parquet artifacts under `data/chexpert/synthetic/scores/`:
`similarity_scores.parquet`, `iqa_scores.parquet`, `uncertainty_scores.parquet`,
**`explainability_scores.parquet`**, `agreement_scores.parquet`. Each carries `image_id`, its
scores, and a schema-version + provenance block. Merged by `image_id` at §4.7 over the signals that
cleared §4.6. Artifacts of excluded-but-valid signals are retained for audit.

### 4.1 Similarity
Hierarchical reference retrieval: exact `intended_label_vector` match → nearest multilabel
neighborhood → pathology prototypes → class-agnostic CXR-realism fallback, so no image goes
unscored. **k-NN aggregation**, not single-nearest-neighbor. A **separate** near-duplicate/
memorization score (high single-NN similarity with low rank-k spread) keeps "realistic" and
"memorized" distinguishable.

### 4.2 IQA
Generic no-reference IQA is a baseline component, not the main signal. Domain sub-scores:
blur/sharpness, clipping, contrast, histogram abnormality, border/padding artifact, anatomical
completeness, grayscale consistency. Whether the generic component adds anything beyond the domain
checks is an ablation, not an assumption.

### 4.3 Uncertainty
**High uncertainty is not equated with low quality.** Bands: very-low (possibly redundant), moderate
(possibly most informative — TSynD's premise), extreme **in combination with** poor
similarity/quality/agreement (the actual OOD flag). Mechanism: MC Dropout on the frozen auxiliary
classifier. Deep Ensembles is a documented upgrade path, not the default.

### 4.4 Explainability
Grad-CAM against the auxiliary classifier, scored by overlap with an expected region. The fixed
central-thorax box is an explicit **baseline**; pathology-specific expected regions are configured
per label. **Region overlap is an explainability-plausibility signal, not proof of diagnostic
correctness** — stated wherever the score is reported.

### 4.5 Intended-label agreement
```
agreement = mean P(intended positive disease labels)
          − penalty × mean P(confidently predicted unintended disease labels)
```
For a **`No Finding` recipe**, agreement is high when predicted probability is low across all 11
primary disease labels. Computed over the 11 primary labels only; `Support Devices` is excluded
unless `secondary_agreement.enabled`. Kept **separate** from uncertainty, similarity, IQA, and
explainability — a confident correct reading and an uncertain correct reading are different facts.

**A proxy label-consistency signal, never clinical proof.** A generator and a classifier can share a
bias and agree while both are wrong.

### 4.6 Go/No-Go gate — FROZEN, applies to all five signals

Each signal is checked for: technical validity; score directionality; numerical stability;
reproducibility; missing/invalid-output rate; redundancy/correlation with other signals; downstream
usefulness. Evidence comes from `asism_tuning_heldout` only.

- **Fails validity/stability/reproducibility** → excluded from the frozen selector; exclusion and
  reason recorded; artifact retained if technically valid.
- **Passes but weak/redundant** → retained as **ablation-only**, not necessarily weighted into the
  selector.
- **A signal is never forced into the selector merely because it was implemented.**
- If a component is excluded, the manifest explicitly records whether the resulting ASISM variant
  remains the primary method or becomes an ablation/alternative.

### 4.7 Tuning — FROZEN data flow and bounded search

**Proxy data flow (frozen):**
- Proxy training data: **`classifier_train` + the actually-constructed candidate synthetic subset.**
  `gen_train` is **not** the real component of proxy training.
- Proxy evaluation data: **`asism_tuning_heldout` only.**
- Fixed optimizer-step budget; **no candidate-specific early stopping.**
- `classifier_val` is **not** used to rank ASISM candidates.
- `final_eval_heldout` is never used.

**Two-stage bounded search:**
- **Stage A, coarse screening** — fixed small trial count, one fixed proxy seed, small step budget.
- **Stage B, shortlist validation** — 3–5 retained configurations, stronger fixed budget,
  pre-declared folds/seeds, simplicity-preferring tie rule.

**Before execution:**
```
total proxy runs = coarse_trials × coarse_folds × coarse_seeds
                 + shortlist_size × shortlist_folds × shortlist_seeds
estimated GPU-hours = total proxy runs × hours_per_proxy_run
```
A configurable compute-budget gate blocks execution if the estimate exceeds budget; the search space
is reduced **before** any results are seen, never after. Search execution resumes at trial
granularity.

### 4.8 Adaptive selection
Ordered policy: (1) absolute **minimum quality floor** over surviving signals, applied to everything
first; (2) per-label **class-specific quota** informed by real prevalence and target synthetic:real
ratio; (3) **min/max accepted count** per class; (4) uncertainty-band awareness. **A poor image is
never accepted merely because its pathology is rare** — rarity changes how many images compete for a
quota, never whether the floor applies.

### 4.9 Learned ASISM extension (v4 revision note, 2026-08-21) — resolves `docs/novelty_target_decision.md`

**This is ASISM as the thesis defines it** — the **five** scoring signals (§4.1 similarity, §4.2
IQA, §4.3 uncertainty, §4.4 explainability, §4.5 intended-label agreement) feeding the two novel
learned components, "Multi-Signal Utility Ranking Network (Novel)" and "Adaptive Threshold Learning
(Novel)". It implements **Option 1** of `docs/novelty_target_decision.md` (pre-registered weakly
supervised set-utility learning).

**v5 revision note (2026-08-21) — naming, no method change.** The ranking component was renamed from
"Multi-Objective Ranking Network" to **Multi-Signal Utility Ranking Network (MSURN)**, and the class
`MultiObjectiveRankingNetwork` to `MultiSignalUtilityRankingNetwork`. Reason: the implemented model
is a single MLP with **one output head**, trained against **one scalar target** (`marginal_utility`)
with smooth-L1 + pairwise-ranking as two loss *terms* on that same target. The nine signal columns
are input *features*, not objectives. There is no Pareto front and no multi-objective gradient
balancing, so "multi-objective" was incorrect in the sense the optimization literature uses it, and
`docs/novelty_target_decision.md` already recorded that "Pareto ranking is not implemented".
Architecture, loss, hyperparameters, configs and all measured behaviour are unchanged; the rename is
checkpoint-safe because `state_dict` keys derive from the unchanged `self.network` attribute.

**Future options (not implemented, not scope).** Two routes would make a multi-objective claim
literally true if ever wanted: (a) Pareto non-dominated sorting over (predicted utility,
`novelty_score`, `iqa_composite`) at *selection* time, leaving the network untouched; or (b) a
multi-head network predicting separately-measured objectives with MGDA-style gradient balancing at
*training* time. Both require defining and independently measuring ≥2 genuinely conflicting
objectives, and (b) requires re-running the Stage 3 experiments.

**v3 revision note (2026-08-21):** §4.7's `weighted_score_baseline` is **no longer part of the
thesis pipeline**. It predates the learned components and was never specified by the thesis, which
defines ASISM as the full module including both learned networks. `03_tune_freeze_select.py` is
retained for reference; no Stage 4 condition consumes its output, and the learned stages no longer
import it (they load candidates via `scripts/asism/candidate_pool.py`).
Pipeline (`scripts/asism/04` through `09`, config: `configs/stage3_asism.yaml` →
`signals.ranking_network` / `signals.threshold_network`):

1. **`04_build_utility_subsets.py` / `04b_evaluate_utility_subsets.py`** — build controlled
   candidate subsets (random, single-signal, mixed compositions) from an image pool split
   train/val-role and disjoint by construction (`split_image_pool`), then measure each subset's real
   downstream utility: `augmented_macro_auroc − real_only_macro_auroc` on the proxy protocol already
   frozen by §4.7 (same `classifier_train`-based proxy training, `asism_tuning_heldout`-only
   evaluation — no new leakage surface).
2. **`05_train_learned_asism.py`** — trains `SetUtilityNetwork` (permutation-invariant Deep Sets) on
   *measured* subset-level utility only, then distills per-image ranking scores
   (`MultiSignalUtilityRankingNetwork` — the admitted signals fused into one scalar utility target,
   not a multi-objective model; trained with Smooth-L1 + a pairwise ranking loss against the
   distilled scores) — never by copying a subset's AUROC onto its member images. This is the
   mechanism that avoids the pseudo-replication problem `novelty_target_decision.md` raised. The
   distillation target is the leave-one-out marginal by default; `banzhaf_msr_targets` computes a
   Data-Banzhaf (Wang & Jia, AISTATS 2023) alternative from the *same* measured subsets (no extra
   proxy training) and 05 records the two targets' Spearman rank agreement, keeping Banzhaf opt-in
   (`ranking_network.target_source`) until that diagnostic is reviewed.
3. **`06_learn_thresholds_select.py`** — the fixed-ratio learned threshold. An intermediate stage of
   the pipeline and an internal ablation reference; it is **not** a Stage 4 condition (the former
   condition G was removed — §7 v3 revision note).
4. **`07`/`07b`/`08`/`08b`** — builds bootstrap class contexts (honestly tagged
   `independent_clinical_sample: False` — they are resamples of one candidate pool, not new clinical
   evidence), a critic-guided hard-threshold grid search, proxy-verifies a diversified (not
   critic-only) subset of that grid, and trains `AdaptiveThresholdNetwork` (class-aware, via class
   embedding + context vector) against verified-only targets.
5. **`09_finalize_learned_selection.py`** — per class, `determine_per_class_official_method` picks
   exactly one of three outcomes, never a blend: `fixed_target_ratio_threshold_distillation_baseline_v1`
   (zero verified train contexts for that class), `hard_proxy_best_among_verified` (some verified
   train evidence but below `min_verified_contexts_per_class` on either side), or
   `adaptive_threshold_network` (enough verified evidence on both train and held-out sides **and**
   the frozen acceptance criteria in `configs/stage3_asism.yaml` → `acceptance_criteria` pass).
   The finalizer then runs the pre-registered `choose_full_policy` tie rule over the full-policy
   proxy measurements (asism_tuning_heldout only) to pick **one** winning threshold policy — the
   competing variants are `fixed_target_ratio_…baseline_v1`, `hard_proxy_best_among_verified`,
   `adaptive_threshold_network`, `literal_top_50_percent`, and the training-free
   `freematch_style_adaptive_percentile` (FreeMatch-adapted; rarer real-population classes get a
   more lenient admission percentile). The single winner is what Stage 4 **condition C** consumes;
   the variants do **not** each become separate final-evaluation conditions.

**Governance status:** implemented but **not yet supervisor-approved** — see the resolution note at
the top of `docs/novelty_target_decision.md`. Not yet run on production data (§11).

**Interaction with §7/§8:** `09`'s output (`adaptive_selected_manifest`) is the selection consumed by
Stage 4 **condition C** — the thesis's "real + selected synthetic" arm, and the only synthetic-
selection condition that survives the §7 v3 revision note. `06`'s `learned_selected_manifest` is
retained on disk as an ablation reference but feeds no Stage 4 condition.

---

## 5. Label policy — FROZEN

### 5.1 `classifier_target_label_set`
All **14** CheXpert observations. This is the classifier's output space; all 14 are predicted and
reported.

### 5.2 `primary_endpoint_label_set`
The **11** disease labels: the 14 minus `No Finding`, `Support Devices`, and `Pleural Other`.

`Enlarged Cardiomediastinum`, `Cardiomegaly`, `Lung Opacity`, `Lung Lesion`, `Edema`,
`Consolidation`, `Pneumonia`, `Atelectasis`, `Pneumothorax`, `Pleural Effusion`, `Fracture`.

- **`No Finding`** — an absence-of-disease meta-label, not a pathology. Secondary outcome only.
- **`Support Devices`** — a device-presence label, not a disease, and the highest-prevalence, easiest
  label in CheXpert; including it would flatter the macro-average without measuring diagnostic
  performance. Secondary outcome only.
- **`Pleural Other`** — **v2 revision note (2026-08-08):** excluded for insufficient support, not for
  a clinical-relevance reason like the two above. On the full production CheXpert cohort only ≈100
  patients dataset-wide carry a confident negative label for it, so no §1.2 fraction choice can give
  every decision-bearing split the §1.3 minimum of 50 negative patients. Reported as a secondary
  outcome with its support counts, per §5.3.

All three remain in the output space; all three are excluded from the **primary endpoint**.

### 5.3 Minimum support for reporting
The §1.3 rule (≥50 positive and ≥50 negative **patients**) also governs metric eligibility. A label
below threshold in the evaluating split is excluded from the primary macro-average **by this
pre-specified rule** and reported separately with its support counts. Support is a property of the
split, computed at split-build time — so the rule never requires inspecting model performance.

---

## 6. Real-label uncertainty policy — FROZEN

CheXpert labels are `1` / `0` / `-1` (uncertain) / blank (not mentioned).

**Policy: preserve raw labels; mask `-1` and blank in training loss and in evaluation.**

Three fields stored separately, never collapsed:

| Field | Meaning |
|---|---|
| `raw_label` | Original value, unmodified: `1`, `0`, `-1`, blank |
| `training_target` | Value fed to the loss under the frozen mapping |
| `loss_mask` / `eval_mask` | Per-label boolean: does this label participate |

`-1` is **never** silently converted to `0` or `1`. U-Ones/U-Zeros is a documented alternative, not
the default: it asserts a fact the original radiologist declined to assert. Masking applies
identically to training, validation, threshold selection, final evaluation, and metric computation.
**Effective patient N is reported for every metric.**

---

## 7. Stage 4 — conditions A/B/C — FROZEN (v3 revision note 2026-08-21)

Architecture, initialization, label policy (§5), uncertainty policy (§6), optimizer family, batch
size, augmentation, checkpoint-selection rule, validation rule, and threshold-selection rule are
**identical across all conditions**. Real component is `classifier_train`; all model selection is on
`classifier_val`.

| Condition | Real | Synthetic |
|---|---|---|
| **A** | `classifier_train` | none |
| **B** | `classifier_train` | all Stage 2 images |
| **F** | `classifier_train` | Learned ASISM selection (§4.9 `adaptive_selected_manifest`) |

**v3 revision note (2026-08-21) — supersedes the v2 note.** The condition set was reduced from A–G
to **A/B/C** to match the three arms the thesis specifies for Stage 5 (real only / real + all
synthetic / real + selected synthetic). Conditions removed and why:

| Removed | Was | Why removed |
|---|---|---|
| **C** | real + weighted-baseline ASISM | The thesis does not specify a weighted ASISM selector; ASISM's Stage-3 definition is the full module including the two learned components. C was an implementation-history artifact, not a thesis arm. |
| **G** | real + fixed-ratio learned baseline | An internal ablation of F's threshold, not a thesis arm. |
| **E** | synthetic only | Not among the thesis's three arms. |
| **D** | matched-random control | Removed as a consequence of removing C (it was matched to C). **See the limitation below — this removal has a scientific cost the other three do not.** |

`configs/stage4_classifier.yaml` → `conditions: [A, B, C]`. Total runs = 3 conditions × 3 seeds =
**9** (superseding the 33 and, before it, 27 stated in earlier revisions).

### 7.1 Known limitation — no matched-random control

**C is a strict subset of B.** A C-over-B improvement therefore admits two explanations this design
cannot separate:

1. **ASISM selected good images** — the thesis claim, and
2. **using fewer synthetic images is simply better**, whichever ones they are, because unselected
   synthetic data is noisy enough that dilution alone helps.

Explanation 2 would produce a C-over-B win even with a *random* selector of the same size. C vs. B
is therefore evidence that *selection helps*, but **not** evidence that *ASISM's ranking is good* —
which is the actual novel contribution. The thesis's three-arm summary is the headline comparison,
not a complete experimental design, and a committee is likely to raise exactly this point.

**Resolving it** requires one added condition: matched-random draws of |F| images sharing F's
per-label profile, trained under the identical protocol. The machinery already exists and is
retained in working order (`01_train_conditions.py`: `build_matched_random_draw`, `profile_of`,
`configs/stage4_classifier.yaml` → `condition_d.*`); it needs re-pointing from the deleted C to F,
plus `n_draws × 3` additional runs (5 draws → 15 runs, total 24). **This is a supervisor decision
and must be settled before Stage 4 executes**, not after results are seen.

**Fairness protocol (frozen): equal optimizer steps** across every enabled condition — not equal
epochs. At fixed epochs, a larger dataset silently receives more gradient updates, conflating "more
data" with "more training." Equal steps isolates the data-composition effect that C vs. B exists to
measure. Recorded per run: optimizer steps, epochs, effective dataset size, real sample exposures,
synthetic sample exposures, interleaving/sampling policy, model seed, dataset ID, config hash,
checkpoint hash.

---

## 8. Stage 5 — final evaluation — FROZEN

**Primary endpoint:** macro-AUROC over `primary_endpoint_label_set` (§5.2) on `final_eval_heldout`.

**Primary comparison: C vs. B** — Learned-ASISM-selected synthetic images vs. all synthetic images
unselected. **Supporting comparisons:** A vs. B and A vs. C. Implemented in
`scripts/eval/compare_conditions.py`:

```
CONFIRMATORY_COMPARISONS = [("F", "B")]
SECONDARY_COMPARISONS    = [("A", "B"), ("A", "F")]
```

Document and code agree. With a single confirmatory comparison, Holm–Bonferroni reduces to the
uncorrected p-value for C vs. B; the correction machinery stays in place so that adding a
confirmatory comparison later (e.g. the matched-random control in §7.1) is a configuration change,
not a reanalysis.

**v3 revision note (2026-08-21) — supersedes the v2 note.** Earlier revisions of this section froze
**C vs. D** as primary and then recorded a conflict with a code-side **F vs. C**. Both are obsolete:
conditions C, D, E and G no longer exist (§7 v3 revision note), so neither comparison is defined.
The conflict is resolved by deletion, not by adjudication.

**What C vs. B can and cannot establish.** It tests whether selecting synthetic images beats using
them all. It does **not** isolate the quality of ASISM's ranking from the effect of simply using
fewer synthetic images — see §7.1. Any Stage 5 write-up must state this limitation explicitly
alongside the C vs. B result, or add the matched-random control described there.

Whether the learned selector (F) is accepted as a thesis method at all remains open in
`docs/novelty_target_decision.md`; this correction family assumes it is.

**Statistics:** patient-level paired bootstrap for effect sizes and 95% CIs. **Holm–Bonferroni** for
the pre-specified confirmatory family; **Benjamini–Hochberg FDR** for exploratory analyses, labelled
as such. **Effect sizes are reported alongside every p-value/CI.** The confirmatory family and
correction configuration are frozen before Stage 4 and are not revised after seeing results.

**Reporting:** per-label AUROC; macro-AUROC (primary); micro-AUROC; AUPRC; sensitivity/specificity;
F1; calibration (Brier, ECE); patient-level bootstrap CIs; corrected paired comparisons with effect
sizes; **effective patient N after masking for every metric**; `No Finding` and `Support Devices`
reported separately; the §7.1 limitation stated alongside the C vs. B result (or, if the
matched-random control is enabled, its full across-draw distribution); leave-one-signal-out ASISM
component-contribution analysis.

**Execution guards (frozen).** Stage 5 refuses to run unless: the production split manifest is
frozen; the final split hash matches; the frozen experiment-protocol manifest exists; the ASISM
freeze manifest exists; all required A/B/C checkpoint hashes exist; the classification-threshold
policy is frozen; and an explicit final-evaluation run ID is supplied.

**Technical resume vs. methodological re-evaluation.** Predictions are written incrementally and
resumably; a technical interruption may resume from the same frozen artifacts and complete the same
partial prediction files without altering the protocol. Aggregate results must never be used to
modify ASISM, classifier selection, thresholds, or the protocol. Once any final-evaluation result
has been inspected, the frozen methodology cannot be changed and the same evaluation cannot be re-run
for methodological reasons. A genuine methodological error requires returning to the relevant
development stage, documenting the change, re-freezing affected artifacts, and treating the result as
a **new pre-specified evaluation run** — reported alongside the original, never silently replacing it.

---

## 9. Execution order

1. Build six-way splits; run the §1.3 support check.
2. Validate and freeze the versioned split manifest.
3. Stage 1, using `gen_train`/`gen_val` only.
4. Stage 2 pilot generation → pilot approval manifest.
5. Stage 2 full generation (gated on §3's pilot gate).
6. Auxiliary classifier (§2), on `gen_train`/`gen_val`.
7. Compute the five ASISM signals (§4.1–§4.5).
8. Go/No-Go gate (§4.6).
9. Learned ASISM (§4.9): `04` subset design → `04b` measured subset utility → `05` set-utility and
   ranking networks → `06`–`08b` threshold learning and proxy verification → `09` finalize. All
   proxy evaluation is on `asism_tuning_heldout` only.
10. Freeze ASISM configuration (§4.9 learned selector, pending the supervisor sign-off tracked in
    `docs/novelty_target_decision.md`).
11. Freeze the experiment protocol (§5, §6, §7, §8), including §7.1's matched-random-control
    decision, which must be settled before any Stage 4 run.
12. Train A/B/C (9 runs; 24 if the §7.1 matched-random control is added).
13. Stage 5 evaluation on `final_eval_heldout`.
14. Patient-level statistics, corrections, tables, figures.

---

## 10. Leakage rules (consolidated)

- `final_eval_heldout` — split construction (§1.6) and Stage 5 only. Never ASISM tuning, Go/No-Go,
  proxy search, hyperparameter tuning, early stopping, checkpoint selection, threshold selection,
  model selection, label-exclusion decisions, or search-space reduction.
- `asism_tuning_heldout` — ASISM Go/No-Go and proxy-search evaluation only.
- `classifier_train` — A/B/C real component and ASISM proxy real component.
- `classifier_val` — A/B/C model selection only; never ASISM candidate ranking; never trained on.
- `gen_train` / `gen_val` — Stage 1, auxiliary classifier, ASISM reference pool. Never A/B/C model
  selection, never evaluation.
- All six splits are patient-disjoint by construction from one deterministic list and one seed.
- Stage 5 bootstrap resampling is patient-level.

---

## 11. Validation status vocabulary

A stage carries one or more of: **implemented**; **locally smoke-tested**; **GPU-ready**;
**production-executed**. Fixtures are not scientific validation. Where the LoRA checkpoint,
generated images, trained classifiers, or a GPU are absent, downstream contracts and executable code
are still implemented and tested with explicitly labelled fixtures, and fail cleanly at real upstream
gates. No fake production artifacts are created and no scientific results are claimed.

---

## 12. Engineering vs. proposed contribution

**Engineering, not claimed as research contribution:** config-driven pipelines; atomic, resumable,
idempotent scripts; versioned manifests and hash-checked lineage; patient-level split discipline;
parquet score artifacts; programmatic leakage guards.

**Proposed research contribution:** an adaptive multi-signal framework — similarity with explicit
memorization awareness (§4.1), multi-faceted technical quality (§4.2), uncertainty treated as
potential informativeness rather than a pure quality penalty (§4.3), explainability plausibility
(§4.4), and intended-label agreement as a proxy consistency check (§4.5) — admitted to the selector
only through the Go/No-Go gate (§4.6), tuned under a bounded compute-accounted search (§4.7), and
applied through a class-aware quota-and-floor policy (§4.8). Whether this constitutes a genuine
contribution is answered by the §8 primary comparison and the component-contribution analysis, not
asserted here.

## `docs/introduction.md`

*97 lines*

# 1 Introduction

Automated classification of medical images is a central task in clinical machine learning, with
broad applications across diverse settings such as disease screening, patient triage, and
diagnostic decision support [1]. Unlike image generation, which produces new samples, classification
assigns a categorical label to an image: a capability that is essential for scalable, automated
diagnosis. Its performance, however, is bounded by the availability of large, diverse, and
accurately annotated training data, which is difficult to obtain in practice because
patient-privacy regulation restricts data sharing, expert annotation is costly, and many
pathologies are rare, so the labelled data any single institution can accumulate is limited [2].
Synthetic image generation has emerged as a leading response to this bottleneck, and diffusion
models in particular have displaced generative adversarial networks as the generative backbone of
choice because of their more stable training and their better preservation of fine anatomical
detail. Recent diffusion models produce chest radiographs and other medical images that are
frequently indistinguishable from real ones in blinded physician review, and can improve downstream
classifiers in data-scarce settings [1]. This has made synthetic-to-real learning, in which a model
is trained or adapted on generated images whose labels are known by construction and then deployed
on real, unseen clinical data, a practical strategy for addressing both data scarcity and
cross-institution domain shift [3, 4].

The output of a diffusion model, however, is uneven. Some generated images are uninformative or
artefactual, some do not express the pathology they were conditioned on, and some are
near-duplicates of specific real training patients: unconditional latent diffusion models have
been shown to replicate or near-replicate real patient images, detectable at a high
feature-similarity threshold [5]. Adding all generated images to a real training set
indiscriminately can therefore degrade rather than improve generalisation, and commonly used
generation-quality metrics such as the Fréchet Inception Distance do not predict downstream
diagnostic performance [6, 7]. The question of which synthetic images should be admitted to a real
training set, and in what per-class balance, is treated only in pieces by recent work: through
single-criterion soft re-weighting for synthetic chest radiographs [2], training-free CLIP-based
filtering for segmentation [8], or classifier-likelihood filtering for an imbalanced grading task
[9]. No existing method combines several complementary quality signals, learns which individual
images most improve a downstream multi-label classifier, and adapts the admission threshold to
each class.

To address this gap, this paper proposes ASISM (Adaptive Quality-Aware Synthetic Data Selection),
a framework that curates the output of a LoRA-fine-tuned diffusion foundation model before it is
used to train a real classifier. ASISM has three components:

1. A multi-signal quality assessment that scores every synthetic image on six complementary
   signals: k-nearest-neighbour similarity to real references in a self-supervised feature space,
   no-reference and domain-specific image quality, Monte-Carlo-dropout predictive uncertainty,
   Grad-CAM alignment with expected pathology regions, agreement between the generated image and
   its intended labels, and within-class distinctiveness. Each signal must independently pass a
   go/no-go admission gate before it can enter the model.

2. A learned utility estimator. A permutation-invariant set-utility network is trained on measured
   subset-level changes in classification AUROC, and its value is distilled into a small
   multi-signal ranking network that assigns one scalar utility score to each image [10]. The
   distillation target is a size-normalised leave-one-out marginal contribution; a Data-Banzhaf
   Maximum-Sample-Reuse estimate, computed from the same measured subsets at no additional cost, is
   provided as a noise-robust alternative.

3. An adaptive per-class admission threshold. A learned threshold network predicts a separate
   admission threshold for each disease label, decided once before final classifier training,
   under a three-tier governance rule that uses the learned threshold only where sufficient
   proxy-verified, image-disjoint evidence exists and otherwise falls back to a verified proxy
   threshold or a fixed baseline [11].

All tuning decisions in ASISM are made on a dedicated proxy split, and the final evaluation split
is never accessed during selection. The framework is evaluated end-to-end on a held-out patient
set by comparing a classifier trained on real data only, a classifier trained on real data plus
all synthetic images, and a classifier trained on real data plus the ASISM-selected subset,
together with a matched-random control that draws the same number of synthetic images per class at
random. This design isolates the benefit of intelligent selection from the effect of simply using
fewer synthetic images, an effect that recent work shows can itself be substantial [12, 13].

The main contributions of this paper are summarized as follows:

i. A multi-signal quality assessment for diffusion-generated medical images that combines six
   heterogeneous perceptual signals: semantic similarity, image quality, predictive uncertainty,
   explanation localisation, intended-label agreement, and within-class distinctiveness, under an
   explicit signal-governance gate. To the best of our knowledge, this is the first framework to
   combine this many complementary signals for the selection of synthetic medical images, and the
   first to use explanation–region alignment as a data-selection signal.

ii. A learned set-level utility model, trained on measured downstream AUROC changes and distilled
   into a per-image ranking network, which avoids the pseudo-replication of copying a set-level
   score onto its member images. A Data-Banzhaf Maximum-Sample-Reuse target is derived from the
   same measured subsets with no additional proxy training, as a noise-robust alternative to the
   leave-one-out marginal.

iii. A learned class-specific admission threshold governed by a conservative three-tier rule,
   together with a strictly proxy-based tuning protocol in which selection decisions never touch
   the final evaluation data. The framework is evaluated against a matched-random control that
   separates the contribution of intelligent selection from that of dataset size, an ablation
   absent from comparable prior work.

The remainder of this paper is organized as follows. Section 2 reviews recent work on
diffusion-based synthetic medical image generation and on post-generation synthetic-data
selection. Section 3 details the proposed ASISM framework, including the six signals, the
set-utility and ranking networks, and the adaptive-threshold governance. Section 4 describes the
dataset, the patient-disjoint split design, and the training and proxy-evaluation configuration.
Section 5 presents the experimental results and the ablation studies. Section 6 concludes the
paper, summarizing the key findings, discussing their implications for synthetic-data curation in
medical imaging, and outlining future research directions. A summary of the proposed framework is
presented in Fig. 1.

## `docs/literature_review.md`

*316 lines*

# 2 Literature review

Deploying deep learning in medical image analysis is constrained above all by data: patient-privacy
regulation, the cost of expert annotation, and the rarity of many pathologies together cap the
labelled data that any single institution can accumulate. Synthetic image generation has become a
leading response to this bottleneck, and diffusion models have largely displaced generative
adversarial networks as the generative backbone of choice because of their more stable training
and their better preservation of fine anatomical detail. A closely related obstacle is domain
shift, in which a model trained at one institution degrades when deployed at another owing to
differences in scanner vendor, acquisition protocol, and patient population. This has produced a
growing body of synthetic-to-real (S2R) work in which diffusion-generated images, whose labels are
known by construction, are used to train or adapt models intended for real, unseen clinical data.
Most of this work is concerned with generation itself: image fidelity, structural conditioning,
and domain alignment. The question addressed in this thesis arises afterwards, given a large pool
of generated images of uneven quality: which of them should be admitted to a real training set,
and in what per-class balance. In line with supervisory guidance, the review below is restricted
to work published from 2024 onward; the few earlier papers that are the original source of a
technique implemented here are named once, for attribution only, at the end of the section.

Recent generation-side research continues to focus on aligning domains before or during synthesis.
Ji and Chung proposed Stochastic Step Alignment, a diffusion-based unsupervised domain adaptation
method that aligns the intermediate trajectory of the denoising process between source and target
domains rather than only the final image [3]. They reported Dice similarity coefficients of 86.5%
for MR-to-CT and 88.3% for CT-to-MR abdominal segmentation on the CHAOS benchmark, outperforming
previous domain-adaptation approaches. Zeng et al. addressed the stricter, privacy-driven setting
in which source pixels cannot be shared, introducing Reliable Source Approximation, which uses
edge-conditioned diffusion to synthesize source-like images and then applies an uncertainty and
prediction-consistency filter to retain only reliable pseudo-labels before fine-tuning [4]. Their
method achieved 77.83% Dice on vestibular schwannoma segmentation without ever accessing source
data. Gong et al. separated appearance alignment from structural alignment in a 3D diffusion
framework for volumetric segmentation, reporting accuracy close to a target-trained upper bound
[14]. Cho et al. released MediSyn, a generalist text-guided latent diffusion model spanning ten
imaging modalities and six medical specialties, and showed through blinded physician review that
its outputs were frequently indistinguishable from real images while improving downstream
classifier performance in data-scarce settings [1]. In the chest-radiograph domain most relevant
to this thesis, Prakash et al. conditioned a latent diffusion model on text and segmentation masks
and used proxy modelling together with radiologist feedback to raise the training utility of
synthetic CheXpert images, reporting F1 gains of up to 0.150 for classification [15]. Their
emphasis, however, is on the conditioning process rather than on selecting individual images. The
CXR-LT 2024 challenge consolidated long-tailed, multi-label, and zero-shot chest-radiograph
methods, several of which use generative augmentation for rare findings, but as a benchmark it
prescribes no selection procedure [16].

A recurring finding in this recent work is that generation-quality metrics are poor proxies for
clinical usefulness. A 2025 review in The Lancet Digital Health argued that downstream task
performance, not the Fréchet Inception Distance (FID), must be the acceptance criterion for
medical synthetic data [6]. An empirical study on retinal image synthesis showed directly that a
lower FID does not translate into better downstream classification or segmentation when the
synthetic images are used for augmentation [7]. Domain-adapted alternatives have been proposed,
such as the Fréchet Radiomic Distance, which compares real and synthetic medical datasets in a
space of quantitative radiomic features [17], but these remain distributional metrics rather than
per-image signals. Taken together, recent generation-side work either aligns domains at synthesis
time or filters generated data by a single criterion, and none learns which individual synthetic
images most improve a downstream multi-label classifier.

A smaller but rapidly growing body of work addresses the post-generation selection problem
directly. Wang et al. observed that diffusion-generated chest radiographs contain noisy and
uninformative samples, and proposed Informative Data Selection, an Information-Bottleneck-grounded
scheme that assigns a higher training weight to more informative synthetic images without
discarding any [2]. Their approach shares the modality, the generator family, and the goal of
this thesis, but differs in three respects: it uses a single information-theoretic criterion
rather than several heterogeneous signals; it re-weights softly rather than making a hard
admission decision; and it applies no per-class adaptive threshold. Tang et al. proposed a
training-free selection method for semantic segmentation that combines a perturbation-based CLIP
similarity score with a class-balanced annotation-similarity filter, halving the synthetic dataset
while improving the segmenter [8]. This is a close analogue of the similarity front end used
here, but relies on CLIP alone, targets segmentation rather than multi-label classification, and
learns neither a ranking model nor a threshold. Zhang et al. used a classifier-predicted-likelihood
semantic-quality metric to filter synthetic fundus images, adding them per class to
under-represented diabetic-retinopathy grades, and reported an increase in balanced accuracy from
66.8% to 74.2% [9]. Their result validates both the intended-label-agreement signal and the
per-class handling adopted in this thesis, although they use one signal and a fixed per-class
quota rather than a learned threshold. In a non-diffusion setting, the CosSIF method removes
synthetic skin-lesion images whose cosine-similarity feature distributions are less
class-discriminative than those of real images, improving downstream AUC by 1.88% on ISIC-2016
[18], which motivates a within-class-versus-between-class similarity cue.

Two further studies frame the selection problem theoretically. Rezaei et al. showed analytically
that, for linear models, the covariance shift between the target and synthetic distributions
drives generalisation error while the mean shift does not, and that matching the covariance is a
near-optimal selection criterion that also transfers to deep networks [19]. Their work supplies a
principled single-criterion baseline against which a multi-signal approach can be contrasted.
Nguyen et al. proposed Targeted Diffusion Augmentation, which generates faithful synthetic images
only for the real training examples that a model fails to learn early, and showed that augmenting
only 30–40% of the data outperforms augmenting everything [12]. Despite its title, the method
selects which real examples to augment rather than which generated images to admit, a distinction
this thesis states explicitly. Kolossov et al. proved that training on a well-chosen subset can
beat training on the full sample, and that unbiased-reweighting and influence-function selectors
can be substantially sub-optimal [13], which provides the formal justification for comparing a
selected synthetic subset against the use of all synthetic data.

The ranking component of the proposed framework draws on recent work in learned set functions and
data valuation. Xie et al. showed that when the utility of a subset depends on the ground set it
is drawn from, the subset representation must incorporate a permutation-invariant sufficient
statistic of that superset, and that this term accounts for most of the observed gain, with the
largest improvements when subsets are small relative to the pool [10]. That regime matches the
roughly one hundred measured subsets used to train the set-utility model here. The DUPRE framework
makes the same efficiency argument that motivates this thesis's selection stage, learning a model
that predicts subset utility so that a proxy classifier need not be retrained for every candidate
subset [20]. For the per-image target, Sun et al. introduced an out-of-bag marginal-contribution
estimator that is fast and robust to the stochasticity of training and that detects fine-grained
outliers [21]. Xu et al. valued a whole data distribution from small samples using a
maximum-mean-discrepancy formulation, supporting the premise that data carry unequal value but
operating at a coarser granularity than the per-image mechanism used here [22]. Chi et al. unified
Shapley-, Banzhaf-, and leave-one-out-type values and optimised them for the selection objective
directly [23]. Yang et al. provided recent, training-free evidence that fusing a
similarity-matching score with an image-quality-assessment term outperforms either alone [24], and
Loizou and Tsoumakos showed that chunk-level contribution can be estimated with single-iteration
proxies at very large speed-ups, paralleling the bounded-budget subset measurement adopted here
[25]. The Set-LLM architecture demonstrates that architectural permutation-invariance with formal
guarantees remains an active design principle for set-structured inputs [26].

The adaptive-threshold component builds on recent class-adaptive thresholding in semi-supervised
learning, adapted from a per-training-step mechanism into a one-time data-curation decision. The
AllMatch method applies class-adaptive confidence thresholds that exploit all unlabelled data
[27]. Zhao et al. decomposed the per-class threshold into a global term and a local term and added
an explicit class-fairness regulariser that prevents rare classes from being starved of
pseudo-labels [11]; this directly motivates the range-normalised per-class leniency and the hard
per-class selection floor used in this thesis. For the distinctiveness signal, Tan et al.
formulated coreset selection as maximising information content minus an importance-weighted
redundancy term defined by pairwise similarity, solved as a discrete quadratic program [28], and
Maharana et al. showed through graph message passing that raw diversity must be balanced against
per-sample difficulty [29]. Finally, and specific to the medical setting, Dar et al. demonstrated
that unconditional latent diffusion models replicate or near-replicate real training patients,
with near-duplication detectable at a DINOv2 cosine similarity above 0.95 [5]. Their result
grounds the near-duplicate rejection threshold used in this thesis in an independent source rather
than a hand-chosen constant.

Despite this progress, there remain clear shortcomings across the recent literature. Existing
methods for selecting synthetic data act on a single criterion, whether an information-theoretic
bound [2], a CLIP similarity score [8], a classifier likelihood [9], or a distributional
distance [19], and none combines several complementary perceptual signals. Most operate at the
level of the individual image or the whole distribution and do not learn a set-level utility that
is then distilled onto individual images; where a marginal-contribution value is used it is
computed for general classification tasks [21, 23] rather than for synthetic medical images.
Class-adaptive thresholds are studied almost exclusively as a per-training-step mechanism for
semi-supervised learning [27, 11] rather than as a one-time curation decision for a multi-label
medical classifier. Few studies use predictive uncertainty or explanation quality as an explicit
selection signal, and most report predictive accuracy while providing limited interpretability,
which reduces their usefulness for clinical decision support. Therefore, there remains a research
gap in developing a multi-signal, class-adaptive, and interpretable framework for selecting
diffusion-generated medical images for a downstream multi-label classifier.

To address these limitations, this thesis proposes ASISM (Adaptive Quality-Aware Synthetic Data
Selection), a framework that operates on the output of a LoRA-fine-tuned diffusion foundation
model. Each synthetic image is scored on six complementary signals: DINOv2 k-nearest-neighbour
similarity to real references, no-reference and domain-specific image quality, Monte-Carlo-dropout
predictive uncertainty, Grad-CAM alignment with expected pathology regions, agreement between the
generated image and its intended labels, and within-class distinctiveness. Each signal must pass an
independent go/no-go admission gate before it can enter the model. A permutation-invariant
set-utility network is trained on measured subset-level AUROC changes, and its value is distilled
into a small multi-signal ranking network that produces one scalar utility score per image; the
distillation target is a size-normalised leave-one-out marginal, with a Data-Banzhaf
Maximum-Sample-Reuse estimate computed from the same measured subsets as a robust alternative. A
per-class admission threshold is then decided once, before final classifier training, under a
three-tier governance rule that uses a learned threshold network only where sufficient
proxy-verified, image-disjoint evidence exists and otherwise falls back to a verified proxy
threshold or a fixed baseline. All tuning decisions are made on a dedicated proxy split, and the
final evaluation split is never opened during selection. Table 1 summarises the reviewed methods,
their strengths, and their limitations relative to this problem.

## Table 1  Summary of the reviewed 2024–2026 methods, strengths, and limitations

| Authors | Method | Strength | Limitation relative to this thesis |
|---|---|---|---|
| Cho et al. [1] | MediSyn (generalist text-guided latent diffusion) | Broad modality coverage; physician-rated realism | Generation only; no post-generation selection |
| Wang et al. [2] | Informative Data Selection (IB re-weighting) | Same modality and goal; down-weights uninformative synthetic CXR | One criterion; soft re-weighting not hard admission; no per-class threshold; no learned set-utility |
| Ji and Chung [3] | Stochastic Step Alignment (diffusion UDA) | Aligns the denoising trajectory between domains; strong cross-modality Dice | Alignment at generation time; no per-image admission decision |
| Zeng et al. [4] | Reliable Source Approximation (source-free UDA) | Uncertainty and consistency filter for reliable pseudo-labels; no source access | One filtering criterion; segmentation; filters pseudo-labels, not generated images by downstream effect |
| Dar et al. [5] | Memorisation analysis of medical LDMs | Shows near-duplication at DINOv2 cosine > 0.95 | A diagnostic study, not a selection framework |
| The Lancet Digital Health review [6] | Review of generative AI for medical synthesis | Argues downstream performance, not FID, is the right acceptance criterion | Review; proposes no method |
| Retinal FID study [7] | Empirical analysis of FID for augmentation | Shows lower FID does not imply better downstream performance | Diagnostic; not a selector |
| Tang et al. [8] | Training-free CLIP-based selection | Perturbation-based CLIP similarity + class-balanced filter; halves the set | CLIP-only (two filters); segmentation; no learned ranker; no threshold learning |
| Zhang et al. [9] | Classifier-likelihood filtering for DR grading | Per-class filtering of synthetic fundus images; balanced accuracy 66.8 → 74.2% | One signal; fixed per-class quota, not a learned threshold; no learned ranker; no valuation |
| Xie et al. [10] | Superset-conditioned neural subset selection | Permutation-invariant superset statistic is the main source of gain | Drug-discovery subsets; no downstream-AUROC target; no thresholds |
| Zhao et al. [11] | SST self-adaptive thresholding | Global × local per-class threshold with a class-fairness regulariser | Per-training-step SSL; not a one-time curation threshold |
| Nguyen et al. [12] | Targeted Diffusion Augmentation | Augmenting 30–40% of the data beats augmenting everything | Selects which real examples to augment, not which generated images to admit |
| Kolossov et al. [13] | Statistical theory of data selection | Proves a well-chosen subset can beat the full sample | Theory; surrogate-guided; not multi-signal, not synthetic/medical |
| Gong et al. [14] | Diffuse-UDA (appearance + structure aligned 3D diffusion) | Approaches the target-trained upper bound in 3D segmentation | No selection of generated samples by downstream effect |
| Prakash et al. [15] | Mask-conditioned LDM for chest radiographs | Proxy modelling and radiologist feedback raise synthetic utility; CheXpert | Conditioning-focused; no per-image selection framework |
| CXR-LT 2024 [16] | Long-tailed multi-label CXR challenge | Consolidates generative augmentation methods for rare findings | A benchmark, not a selection method |
| Fréchet Radiomic Distance [17] | Radiomic-feature distributional metric | Domain-adapted real-vs-synthetic dataset comparison | A metric, not a per-image signal |
| CosSIF [18] | Cosine-similarity image filtering | Removes less class-discriminative synthetic images; +1.88% AUC (ISIC-2016) | Single modality (skin); GAN generator; one criterion |
| Rezaei et al. [19] | Covariance-matching synthetic-data selection | Analytic criterion; transfers from linear models to deep networks | Single distribution-level criterion; not per-image; no thresholds |
| DUPRE [20] | Data utility prediction for valuation | Predicts subset utility to avoid per-subset retraining | General valuation; no perceptual signals; no ranking distillation |
| Sun et al. [21] | 2D-OOB joint valuation | Out-of-bag marginal-contribution estimator; fast; robust to stochastic training | General/tabular; per-sample; no set-utility distillation; no thresholds |
| Xu et al. [22] | Data distribution valuation | MMD-based valuation of a distribution from small samples | Distribution-level, not per-image; not selection-oriented |
| Chi et al. [23] | Unified data values for selection | Optimises Shapley/Banzhaf/LOO values for the selection objective | General classification; not synthetic/medical; no perceptual signals |
| Yang et al. [24] | GMValuator (similarity + IQA valuation) | Training-free fusion of similarity matching and image-quality assessment | Values real training data for a generator, not generated images for a classifier; two signals |
| Loizou and Tsoumakos [25] | Chunked Data Shapley | Chunk-level contribution with single-iteration proxies; large speed-ups | Tabular focus; no perceptual signals |
| Set-LLM [26] | Permutation-invariant architecture | Formal permutation-invariance guarantees for set inputs | Text-set setting; not utility regression over image sets |
| AllMatch [27] | Class-adaptive SSL thresholding | Class-adaptive thresholds using all unlabelled data | Per-training-step SSL loop; single-label |
| Tan et al. [28] | InfoMax data pruning | Coreset = information − importance-weighted redundancy, as a QP | Single objective; no downstream-utility target; no thresholds |
| Maharana et al. [29] | D² Pruning | Graph message passing balancing diversity and difficulty | Real-data pruning; no synthetic/medical; no learned utility model |

## References

[1] Cho C, et al. MediSyn: a generalist text-guided latent diffusion model for synthetic medical
image generation. arXiv:2405.09806; 2024.

[2] Wang Y, Goel R, Jojic M, Silva AC, Wu T, Yang Y. Informative synthetic data generation for
thorax disease classification. Conference on Uncertainty in Artificial Intelligence (UAI). PMLR
vol. 286; 2025. p. 4489–514.

[3] Ji W, Chung ACS. Stochastic step alignment for diffusion-based unsupervised domain adaptation
in medical image segmentation. Medical Image Computing and Computer-Assisted Intervention (MICCAI);
2024.

[4] Zeng Y, et al. Reliable source approximation: source-free unsupervised domain adaptation for
vestibular schwannoma MRI segmentation. Medical Image Computing and Computer-Assisted Intervention
(MICCAI); 2024.

[5] Dar SUH, et al. Unconditional latent diffusion models memorize patient imaging data:
implications for openly sharing synthetic data. Nature Biomedical Engineering; 2025.
doi:10.1038/s41551-025-01468-8.

[6] Generative artificial intelligence in medical image synthesis: opportunities, challenges, and
future directions. The Lancet Digital Health; 2025.

[7] A pragmatic note on evaluating generative models with Fréchet Inception Distance for retinal
image synthesis. arXiv:2502.17160; 2025.

[8] Tang H, Yu S, Pang J, Zhang B. A training-free synthetic data selection method for semantic
segmentation. Proceedings of the AAAI Conference on Artificial Intelligence; 2025.
doi:10.1609/aaai.v39i7.32777. arXiv:2501.15201.

[9] Zhang H, Heinke A, Nagel ID, Bartsch DUG, Freeman WR, Nguyen TQ, An C. Class-conditioned
image synthesis with diffusion for imbalanced diabetic retinopathy grading. Medical Image
Computing and Computer-Assisted Intervention (MICCAI); 2025.

[10] Xie B, Bian Y, Zhou K, Chen Y, Zhao P, Han B, Meng W, Cheng J. Enhancing neural subset
selection: integrating background information into set representations. International Conference on
Learning Representations (ICLR); 2024. arXiv:2402.03139.

[11] Zhao S, Huang H, Li X, Chen X, Wang R. SST: self-training with self-adaptive thresholding for
semi-supervised learning. Information Processing & Management. 2025;62(5):104158. arXiv:2506.00467.

[12] Nguyen D, Li J, Zheng J, Mirzasoleiman B. Do we need all the synthetic data? Targeted image
augmentation via diffusion models. International Conference on Learning Representations (ICLR); 2026
(accepted). arXiv:2505.21574.

[13] Kolossov G, Montanari A, Tandon P. Towards a statistical theory of data selection under weak
supervision. International Conference on Learning Representations (ICLR); 2024 (Oral).
arXiv:2309.14563.

[14] Gong H, et al. Diffuse-UDA: addressing unsupervised domain adaptation in medical image
segmentation with appearance and structure aligned diffusion models. arXiv:2408.05985; 2024.

[15] Prakash E, Valanarasu JMJ, Chen Z, Reis EP, Johnston A, Pareek A, Bluethgen C, Gatidis S,
Olsen C, Chaudhari A, Ng A, Langlotz C. Evaluating and improving the effectiveness of synthetic
chest X-rays for medical image analysis. Proceedings of the IEEE/CVF International Conference on
Computer Vision (ICCV) Workshops; 2025. p. 4413–21.

[16] CXR-LT 2024: a MICCAI challenge on long-tailed, multi-label, and zero-shot disease
classification from chest X-ray. MICCAI 2024 challenge report; arXiv:2506.07984.

[17] Fréchet Radiomic Distance (FRD): a versatile metric for comparing medical imaging datasets.
arXiv:2412.01496; 2024.

[18] CosSIF: cosine similarity-based image filtering to overcome low inter-class variation in
synthetic medical image datasets. Computers in Biology and Medicine. 2024;172:108317.
arXiv:2307.13842.

[19] Rezaei P, et al. High-dimensional analysis of synthetic data selection. International
Conference on Learning Representations (ICLR); 2026 (accepted). arXiv:2510.08123.

[20] DUPRE: data utility prediction for efficient data valuation. International Conference on
Autonomous Agents and Multiagent Systems (AAMAS); 2025.

[21] Sun Y, Shen X, Kwon Y. 2D-OOB: attributing data contribution through joint valuation
framework. Advances in Neural Information Processing Systems (NeurIPS); 2024. arXiv:2408.03572.

[22] Xu X, Wu Z, Sim RHL, Low BKH, et al. Data distribution valuation. Advances in Neural
Information Processing Systems (NeurIPS); 2024. arXiv:2410.04386.

[23] Chi H, et al. Unifying and optimizing data values for selection via sequential
decision-making. International Conference on Machine Learning (ICML); 2026 (Spotlight; accepted).
arXiv:2502.04554.

[24] Yang J, Deng W, et al. GMValuator: similarity-based data valuation for generative models.
International Conference on Learning Representations (ICLR); 2025. OpenReview:WncnpvJk83.
arXiv:2304.10701.

[25] Loizou A, Tsoumakos D. Chunked Data Shapley: a scalable dataset quality assessment for
machine learning. ACM International Conference on Information and Knowledge Management (CIKM); 2025.
doi:10.1145/3746252.3761305. arXiv:2508.16255.

[26] Set-LLM: a permutation-invariant LLM. Advances in Neural Information Processing Systems
(NeurIPS); 2025. arXiv:2505.15433.

[27] AllMatch: exploiting all unlabeled data for semi-supervised learning. International Joint
Conference on Artificial Intelligence (IJCAI); 2024.

[28] Tan H, Wu S, Huang W, Zhao S, Qi X. Data pruning by information maximization. International
Conference on Learning Representations (ICLR); 2025. OpenReview:93XT0lKOct. arXiv:2506.01701.

[29] Maharana A, Yadav P, Bansal M. D² pruning: message passing for balancing diversity and
difficulty in data pruning. International Conference on Learning Representations (ICLR); 2024.
arXiv:2310.07931.

## Method-origin references (attribution only, pre-2024)

The following papers are the original source of a technique implemented in this thesis. They are
cited once, in the methodology, solely so that the framework can attribute its own components; they
carry no claim about the current state of the art, and the operative citation in each case is the
2024–2026 paper indicated. Zaheer et al. introduced the permutation-invariant set function
`ρ(Σ φ(x))` (Deep Sets, NeurIPS 2017), applied here through the recent set-utility work of Xie et
al. [10] and Set-LLM [26]. Wang and Jia introduced the Data-Banzhaf value and its
Maximum-Sample-Reuse estimator (AISTATS 2023), used here in the form recently unified for selection
by Chi et al. [23] and Sun et al. [21]. Class-adaptive confidence thresholding originates with
FlexMatch (Zhang et al., NeurIPS 2021) and FreeMatch (Wang et al., ICLR 2023), adapted here
following AllMatch [27] and Zhao et al. [11]. Grad-CAM (Selvaraju et al., ICCV 2017) is used only
as a tool; no 2024–2026 work uses explanation–region alignment as a data-selection signal, which
is one contribution of this thesis. Multi-gate Mixture-of-Experts (KDD 2018) and gradient-based
multi-objective optimisation work are deliberately not cited: the multi-signal ranking network has
a single scalar output head and one training target, and is not a multi-task or Pareto model.

## `docs/smoke_e2e.md`

*149 lines*

# Tiny Stage 1-5 end-to-end smoke run

This lane is engineering validation only. It creates 60 deterministic, patient-disjoint synthetic
CheXpert-shaped records under `outputs/smoke/dev-smoke-v1` — a fully disposable workspace: every
byte in it is regenerated by re-running the smoke pipeline, and the SDXL weights it caches under
`.cache/huggingface` are pinned to an immutable revision, so the whole directory can be deleted
to reclaim space (a completed GPU smoke leaves ~22 GB of checkpoints and model cache there).
It never reads `data/chexpert`, never uses a production namespace, and stamps the workspace
`scientific_evidence: false`.

## One orchestrated command

From the repository root:

```bash
python scripts/smoke/run_smoke_pipeline.py --phase all
```

The command prints every subprocess, stops at the first failure, validates each producer artifact,
and resumes completed work. Its first invocation deliberately stops after generating the four pilot
images. Inspect:

```text
outputs/smoke/dev-smoke-v1/data/chexpert/synthetic/dev-smoke-v1/pilot/images/
```

Then resume the same pipeline with explicit fixture-pilot approval:

```bash
python scripts/smoke/run_smoke_pipeline.py --phase all --approve-smoke-pilot
```

The approval is valid only inside the marked smoke workspace and cannot unlock production.

## Local-only preparation

```bash
python tests/run_all.py
python scripts/smoke/run_smoke_pipeline.py --phase local
```

Expected CPU outputs:

| Stage | Input | Validated output |
|---|---|---|
| fixture | deterministic generator seed/rules | `outputs/smoke/dev-smoke-v1/SMOKE_ONLY.json`, 60 source JPEGs and CSV rows |
| split | fixture `train.csv` | six CSVs plus `split_manifest_v2.json` under the smoke workspace |
| preprocessing | six fixture split CSVs | six image directories, manifests, and failure-free logs |
| captions | processed `gen_train`/`gen_val` | two nonempty JSONL caption files and provenance manifests |

No GPU is required. On the tested local machine this phase took under one minute after code import.

## RunPod GPU phase

Use the pinned environment described in `environment/RUNPOD_MATRIX.md`, then either run the entire
command on the pod or copy the smoke workspace from local storage and run only the GPU phase:

```bash
cd /workspace/chest-synth-thesis
pip install -r environment/requirements.txt
python -c "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.version.cuda)"
python tests/run_all.py
python scripts/smoke/run_smoke_pipeline.py --phase all
# inspect the four pilot images; then:
python scripts/smoke/run_smoke_pipeline.py --phase all --approve-smoke-pilot
```

If the CPU workspace was copied to the identical repository-relative location:

```bash
python scripts/smoke/run_smoke_pipeline.py --phase runpod
# inspect pilot, then resume:
python scripts/smoke/run_smoke_pipeline.py --phase runpod --approve-smoke-pilot
```

Expected GPU-path outputs:

| Stage | Input | Validated output |
|---|---|---|
| Stage 1 | smoke images/captions, pinned SDXL | two-step LoRA final checkpoint plus resumable optimizer checkpoints |
| Stage 2 | explicit LoRA and 40 recipes | pilot approval record, ~150-300 full images across all 11 primary labels, generation completion manifest |
| auxiliary | smoke `gen_train`/`gen_val` | two-step resume checkpoint and separate best-validation checkpoint |
| Stage 3 signals | real generated images | five Parquets and five hash/provenance sidecars |
| Go/No-Go | score artifacts | `gonogo_report.json` with admitted/excluded signals |
| Learned ASISM (04–09) | real signals from above | subset feasibility report, measured utility, ranking + threshold-network checkpoints, condition C's `adaptive_selected_manifest.jsonl` |
| Stage 4 | conditions **A/B/C**, one seed | three two-step best checkpoints and atomic completion records |
| Stage 5 | separate three-patient fixture final split | registered protected context, predictions with sidecars, comparison report/tables |

Cold-cache estimate on one A40 48 GB: approximately 30–60 minutes, dominated by model downloads,
SDXL initialization, and the tiny ASISM proxy matrix. Warm-cache resume should be materially faster.
Allow approximately 20–35 GB for pinned model caches, checkpoints, environment packages, and smoke
outputs. These are operational estimates, not measured production costs — the learned-ASISM 04-09
addition (2026-08-21) is not included in them and has never been timed; expect this estimate to be
revised after the first real run.

STOP if any artifact gate fails, if a path contains `production`, if the workspace lacks
`SMOKE_ONLY.json`, if pilot review fails, or if the orchestrator attempts to open real thesis data.

**Learned ASISM 04–09 on this GPU path is UNVERIFIED BY EXECUTION as of 2026-08-21** — no GPU was
available to build it against. Every individual flag/path was checked against the scripts' actual
argparse definitions and `stage3_paths()`, and the CPU tier below proves the 04→05→06 portion works
against a (fabricated) real-shaped pool, but the full 04–09 chain — including real `04b`/`07b`/`08b`
proxy-classifier training — has never actually run. `04 --phase feasibility` is the cheap, no-GPU
first checkpoint: if it fails on the real generated pool, lower `stage3.learned_asism.subset_design`
/ `feasibility_thresholds` in `configs/smoke_e2e.yaml` and rerun *only* that command — it does not
require repeating Stage 1/2/aux/signals. Report the first real result back before trusting this
table.

## Learned ASISM CPU integration smoke (04 → 05 → 06)

```bash
python scripts/smoke/run_smoke_pipeline.py --phase local   # builds the split/caption fixture this needs
python scripts/smoke/01b_learned_asism_cpu_smoke.py
```

CPU-only, no SDXL, under a minute. Fabricates 900 synthetic candidate images (label vectors only —
no JPEGs) and four of the five score signals (`explainability` is deliberately omitted), then runs
the **real** `04_build_utility_subsets.py`, `05_train_learned_asism.py`, and
`06_learn_thresholds_select.py` as subprocesses against that fixture, with `utility_results.jsonl`
fabricated (never a real proxy-classifier measurement) so `05` has something to train against
without a GPU. Explicitly verifies:

- Go/No-Go feature removal: `active_feature_columns()` narrows the ranking network's inputs to the
  4 surviving signals rather than crashing (the bug fixed 2026-08-21) or silently keeping the 5th.
- `contributing_signals` and `learned_variant_status` in the training manifest correctly report
  4 signals / `"primary"` (the primary-vs-`reduced_variant` boundary itself is unit-tested in
  `tests/test_learned_asism.py`, not here).
- Train/val image pools are disjoint (`image_overlap_fraction == 0.0`).
- A ranking-model checkpoint and a non-empty selected-images manifest are produced.

## Scientific limitations

This run demonstrates software connectivity only. It does not validate medical quality,
statistical power, rare-label support, or thesis outcomes. Nothing under this heading — including
the learned-ASISM CPU smoke above — is scientific evidence about ASISM's selection quality.

Current coverage:

| Tier | Covers | Status |
|---|---|---|
| Unit tests | Every learned-ASISM function in isolation | ✅ `tests/test_learned_asism.py`, 88 tests |
| CPU integration smoke | `04` → `05` → `06` connect; Go/No-Go feature removal | ✅ verified passing, `01b_learned_asism_cpu_smoke.py` |
| GPU smoke | `04`–`09` on real generated images, real (tiny) proxy verification, condition **C** trained | ⚠️ built 2026-08-21, **never executed** — no GPU available; see the RunPod GPU phase section above |

Closing the last row needs a GPU-smoke extension of the existing RunPod phase: real (tiny) proxy
classifiers for `04b`/`07b`/`08b` instead of the CPU tier's fabricated numbers, plus
`conditions: [A, B, C]` in the GPU overlay (already set there). Until it exists, condition C and
the thesis's primary comparison (C vs. B) have never been run end to end anywhere, including in
smoke form.

## `docs/runpod_stage2_to_5_commands.md`

*231 lines*

# RunPod production execution handoff

Use one immutable split run ID everywhere. The examples use `production-thesis-v1`; changing any
split assignment requires a new ID. Dev commands use `dev-smoke-v1`. Outputs from the two IDs are
physically separate and every consumer verifies the split, configuration, checkpoint, source-tree,
and upstream artifact hashes.

## Local, no GPU

```powershell
python -m compileall -q scripts tests
python tests/run_all.py
python scripts/data/02b_build_sixway_splits.py --help
python scripts/asism/04_build_utility_subsets.py --help
python scripts/asism/04b_evaluate_utility_subsets.py --help
python scripts/asism/05_train_learned_asism.py --help
python scripts/asism/09_finalize_learned_selection.py --help
python scripts/classify/01_train_conditions.py --plan-only --namespace dev-smoke-v1
```

The plan-only command still requires a valid `dev-smoke-v1` split manifest. A fixture/dev build is:

```powershell
python scripts/data/02b_build_sixway_splits.py --namespace dev --run-id dev-smoke-v1 --input-csv data/chexpert/raw/train.csv
python scripts/data/03_preprocess_images.py --namespace dev-smoke-v1 --splits gen_train gen_val classifier_train classifier_val asism_tuning_heldout final_eval_heldout
python scripts/data/04_generate_captions.py --namespace dev-smoke-v1 --splits gen_train gen_val
```

Preprocessing resumes independently per split and validates its manifest before skipping an image.
Captions are intentionally generated only for the generator splits. These commands need CPU and
roughly the size of the selected source images plus processed JPEGs; they do not establish scientific
validity.

## RunPod setup and immutable inputs

Provision a persistent volume of at least 200 GB. A40 48 GB is the reviewed target. Application
dependencies and immutable model revisions are in `environment/requirements.txt` and
`environment/RUNPOD_MATRIX.md`.

```bash
cd /workspace/chest-synth-thesis
export PROJECT_ROOT=/workspace/chest-synth-thesis
pip install -r environment/requirements.txt
python -c "import torch,torchvision,diffusers,transformers,accelerate,peft,timm,pyarrow; print(torch.__version__,torch.version.cuda,torchvision.__version__,torch.cuda.is_available())"
python -m compileall -q scripts tests
python tests/run_all.py
nvidia-smi
```

STOP unless the matrix is Python 3.11, torch 2.8.0, torchvision 0.23.0, CUDA 12.8, CUDA available,
and every test passes.

## Exact production order

1. Verify the complete CheXpert source and image tree (CPU/I/O):

   ```bash
   python scripts/data/01_verify_download.py
   ```

2. Build and atomically freeze a new patient split (CPU/RAM, normally minutes). This refuses an
   existing run directory and refuses production freeze if the support gate fails:

   ```bash
   python scripts/data/02b_build_sixway_splits.py --namespace production --run-id production-thesis-v1 --freeze
   ```

   STOP and manually review `data/chexpert/processed/splits/production-thesis-v1/split_manifest_v2.json`,
   prevalence deviations, rare-label support, hashes, and patient counts.

3. Process all real splits (CPU/I/O; expect several hours and tens of GB) and build Stage 1 captions:

   ```bash
   python scripts/data/03_preprocess_images.py --namespace production-thesis-v1 --splits gen_train gen_val classifier_train classifier_val asism_tuning_heldout final_eval_heldout
   python scripts/data/04_generate_captions.py --namespace production-thesis-v1 --splits gen_train gen_val
   ```

4. Stage 1 smoke and explicit resume test (GPU). Both use the same immutable namespace:

   ```bash
   bash scripts/train/launch_resumable.sh split.namespace=production-thesis-v1 training.max_train_steps=20 checkpointing.save_every_n_steps=10 validation.val_every_n_steps=10
   # wait for step 10+, stop the worker once, then repeat the identical command and verify continuity
   bash scripts/train/launch_resumable.sh split.namespace=production-thesis-v1 training.max_train_steps=20 checkpointing.save_every_n_steps=10 validation.val_every_n_steps=10
   ```

   The checkpoint interval key is `checkpointing.save_every_n_steps`; an unknown key such as
   `training.checkpointing_steps` is silently added to the config and ignored, so no step-10
   checkpoint would exist and the resume test would test nothing. The first launch builds the
   full gen_train latent cache (~3 h on an A40) before step 1; the full run reuses it.

   STOP and inspect loss, samples, checkpoint metadata, and resume continuity. Then retire the smoke
   run pointer — otherwise `launch_resumable.sh` resumes the finished 20-step smoke run instead of
   starting the full one:

   ```bash
   rm checkpoints/stage1_lora_sdxl/latest_run.json
   bash scripts/train/launch_resumable.sh
   ```

   The full run takes the frozen config as-is (`training.max_train_steps: 30000`, ~22–27 h on an
   A40). Training does not stop early on its own, and the cosine LR schedule is built from
   `max_train_steps`, so relaunch every resume with the **identical** command and no step
   overrides. Select one explicit LoRA checkpoint only after probe review; record its path in
   `configs/stage2_generation.yaml` and commit it **before** step 5 (see "Code freeze" below).

   **Code freeze before step 5.** From `01_sample_label_recipes.py` onward every artifact records
   `code_identity_sha256`, the hash of every file under `scripts/`, `configs/` and `environment/`
   (`scripts/utils/manifest.py::code_identity`, committed or not), and every later consumer refuses
   artifacts made under a different hash. After step 5 starts, any edit to those trees — a YAML
   tweak, a `git pull`, or `pip freeze > environment/requirements-lock.txt` — invalidates the
   synthetic images and everything downstream. So, before step 5:

   - commit `checkpoint.lora_weights_dir` in `configs/stage2_generation.yaml`;
   - settle the §7.1 matched-random-control decision (step 7) and commit any
     `configs/stage4_classifier.yaml` change;
   - capture `pip freeze` outside the repo trees (e.g. `/workspace/pip_freeze.txt`).

   If a later gate needs a config change anyway (for example step 6b's feasibility report failing on
   `subset_sizes`), put the change in a YAML file outside the repo and point `THESIS_CONFIG_OVERLAY`
   at it (sections `stage2`/`stage3`/`stage4`, as in `configs/smoke_e2e.yaml`) instead of editing
   `configs/`; export the same overlay for every later command.

5. Stage 2 recipes, pilot, manual gate, and full generation (GPU; potentially tens of GPU-hours and
   tens of GB depending on recipe count):

   ```bash
   python scripts/generate/01_sample_label_recipes.py --namespace production-thesis-v1
   python scripts/generate/02_generate_synthetic_images.py --mode pilot --namespace production-thesis-v1
   ```

   STOP for visual/manual pilot review. Approval is irreversible provenance, not an automatic check:

   ```bash
   python scripts/generate/02_generate_synthetic_images.py --approve-pilot --reviewer "<name>" --namespace production-thesis-v1
   python -u scripts/generate/02_generate_synthetic_images.py --mode full --namespace production-thesis-v1
   ```

6. Auxiliary classifier and Stage 3 signals (GPU except generic IQA):

   ```bash
   python scripts/classify/00_train_auxiliary_classifier.py --namespace production-thesis-v1 --max-steps 8000
   python scripts/asism/01_compute_signals.py --signal all --namespace production-thesis-v1
   python scripts/asism/02_gonogo.py --namespace production-thesis-v1
   ```

   STOP after Go/No-Go. Review which signals were admitted before continuing: the learned-ASISM
   feature set is derived from exactly those signals, and every stage below is frozen against them.

   `03_tune_freeze_select.py` (the weighted-score selector) is **not part of the thesis pipeline** —
   the thesis's ASISM is the full module including the two learned components, so there is no
   weighted-baseline condition to produce. The script is retained for reference only; do not run it.

6b. Learned ASISM — set-utility model, ranking network, adaptive threshold network
   (`docs/stages2_to_5_plan.md` §4.9). Each `--phase estimate` prints a GPU-hour estimate against
   `configs/stage3_asism.yaml`'s compute-budget gates; `--phase run` is refused without the explicit
   confirmation flag, so nothing here can accidentally launch paid GPU time:

   ```bash
   python scripts/asism/04_build_utility_subsets.py --phase feasibility --namespace production-thesis-v1
   python scripts/asism/04_build_utility_subsets.py --phase build --namespace production-thesis-v1
   python scripts/asism/04b_evaluate_utility_subsets.py --phase estimate --namespace production-thesis-v1
   python scripts/asism/04b_evaluate_utility_subsets.py --phase run --namespace production-thesis-v1
   python scripts/asism/05_train_learned_asism.py --namespace production-thesis-v1
   python scripts/asism/06_learn_thresholds_select.py --namespace production-thesis-v1
   ```

   STOP at `--phase feasibility`: it fails closed if the candidate pool is too small for the
   configured `subset_design.subset_sizes`. Roughly **3,750+ usable synthetic candidates** are needed
   under the current sizes `[100, 200, 250]`. Fix by revising `subset_sizes` (or generating more in
   Stage 2) and re-running feasibility — never by editing the thresholds to make a failing report
   pass.

   `06` produces the fixed-ratio learned threshold — an intermediate artifact and ablation
   reference, not a Stage 4 condition. The adaptive path that feeds condition C continues:

   ```bash
   python scripts/asism/07_build_threshold_contexts.py --namespace production-thesis-v1
   python scripts/asism/07b_verify_thresholds_proxy.py --phase estimate --namespace production-thesis-v1
   python scripts/asism/07b_verify_thresholds_proxy.py --phase run --i-understand-this-trains-real-models --namespace production-thesis-v1
   python scripts/asism/08_train_threshold_network.py --namespace production-thesis-v1
   python scripts/asism/08b_verify_full_policy_proxy.py --phase estimate --namespace production-thesis-v1
   python scripts/asism/08b_verify_full_policy_proxy.py --phase run --i-understand-this-trains-real-models --namespace production-thesis-v1
   python scripts/asism/09_finalize_learned_selection.py --namespace production-thesis-v1
   ```

7. Stage 4 plan and 9 classifier runs — the thesis's three arms A/B/C × 3 seeds
   (`docs/stages2_to_5_plan.md` §7 v3 revision note; GPU; budget before launch, commonly many
   GPU-hours):

   ```bash
   python scripts/classify/01_train_conditions.py --plan-only --namespace production-thesis-v1
   python scripts/classify/01_train_conditions.py --condition all --namespace production-thesis-v1
   ```

   `--condition all` trains every condition in `configs/stage4_classifier.yaml` → `conditions`
   (currently `[A, B, C]`). F will hit an `UPSTREAM GATE` error if step 6b was skipped — that is the
   intended fail-closed behavior, not a bug.

   **Before launching, confirm the §7.1 decision:** C is a strict subset of B, so with only A/B/C
   there is no matched-random control and a C-over-B gain cannot be attributed to ASISM's ranking
   rather than to using fewer synthetic images. Adding that control costs `n_draws × 3` more runs.
   Settle it now — it cannot be added after results are seen.

   Each run resumes from its optimizer checkpoint; validated completed runs are skipped.

8. Register and run the one protected final evaluation (GPU inference), then analyze locally (CPU):

   ```bash
   python scripts/eval/stage5_evaluate.py --final-eval-run-id final-thesis-v1 --namespace production-thesis-v1
   python scripts/eval/compare_conditions.py --run-dir outputs/stage5/production-thesis-v1/final-thesis-v1
   ```

   STOP before registration for final protocol/checkpoint/hash review. The first outcome-bearing read
   changes registration state. Technical resume must reuse the identical run ID and hashes; a second
   methodological evaluation is refused.

## Resume units

| Producer | Resume unit | Completion contract |
|---|---|---|
| preprocessing | image within each split | decoded image plus matching per-split manifest |
| Stage 1 | optimizer step | model/optimizer/RNG plus exact split and config identity |
| Stage 2 | image | valid JPEG plus unique provenance-bearing manifest row |
| auxiliary/proxy classifier | optimizer step | model/optimizer/scheduler/RNG/sampler and best model |
| ASISM signals | signal | Parquet hash plus provenance sidecar |
| ASISM search | trial/fold | compatible trial log and checkpoint provenance |
| Stage 4 | optimizer step and completed run | best checkpoint and atomic completion record |
| Stage 5 | condition | validated Parquet plus completion sidecar |

Current preserved root-level Stage 2 recipes and manifests are legacy artifacts. Current code
explicitly refuses them; regenerate them under the selected versioned namespace.

## `environment/RUNPOD_MATRIX.md`

*25 lines*

# Frozen RunPod dependency and model matrix

Production runs use a RunPod image with Python 3.11, CUDA 12.8, PyTorch 2.8.0,
torchvision 0.23.0, and one NVIDIA A40 48 GB GPU. Verify these exact runtime versions before any
paid training; `environment/requirements.txt` pins every application-layer dependency and must not
replace the image-provided PyTorch packages.

The two remotely resolved model inputs are immutable:

| Component | Repository | Revision | License declared upstream |
|---|---|---|---|
| SDXL base | `stabilityai/stable-diffusion-xl-base-1.0` | `462165984030d82259a11f4367a4eed129e94a7b` | OpenRAIL++ |
| DINOv2 ViT-S/14 | `timm/vit_small_patch14_dinov2.lvd142m` | `936966a8732c5442c9def5d126f2cc4ad4243dba` | Apache-2.0 |

Run this gate immediately after environment installation:

```bash
python -c "import torch,torchvision,diffusers,transformers,accelerate,peft,timm,pyarrow; print(torch.__version__,torch.version.cuda,torchvision.__version__)"
python -m compileall -q scripts tests
python tests/run_all.py
```

STOP if the printed torch/CUDA/torchvision versions differ, CUDA is unavailable, model revisions
cannot be resolved, or any test fails. After the successful pod smoke test, capture `pip freeze`,
the container image identifier, NVIDIA driver version, and `nvidia-smi` output in the run manifest.

## `cloud/BENCHMARKS.md`

*112 lines*

# Pipeline optimization benchmarks (Modal, 2026-09-11)

Measured with `cloud/modal_bench.py` on Modal. Each benchmark runs the original and the optimized
behaviour through the same repository code, in the same container, on the same synthetic inputs
(768 px CXR-like JPEGs; synthetic latents for the Stage 1 step), so the comparison isolates the
change. No CheXpert data was used. Costs include the GPU plus the requested CPU (4 cores) and
memory (16 GiB) at Modal's published rates.

**Available hardware.** The Starter plan without a payment method may use only T4, L4 and A10;
L40S and A100 are refused at launch (`cloud/modal_gpu_probe.py`). All measurements are on A10 and
L4.

## Summary of decisions

| # | Change | Result | Decision |
|---|---|---|---|
| 1 | Classifier DataLoader workers + pinned memory + cudnn.benchmark | 2.2-4.9x faster; outputs identical | **Kept** (`7333044`) |
| 2 | Stage 2: several images per SDXL pipeline call | 1.07x faster; images change (PSNR 23-33 dB) and the original's bit-exact reproducibility is lost | **Rejected** |
| 3a | Stage 1: batched VAE latent caching | 1.05x faster at 3x VRAM; batch 16 OOMs | **Rejected** |
| 3b | Stage 1: free VAE + text encoders after caching | ~2 GB less host RAM for the whole run, no computation change | **Kept** (`33e53ad`) |
| 4 | Stage 1 gradient checkpointing | GC off does not fit in 24 GB (8x2 and 4x4 both OOM) | **GC stays on** (the only feasible setting) |
| - | Stage 1 container: 2 CPU cores instead of 4 | training is GPU-bound (99% util) | **Kept** (`d907986`) |

## 1. Classifier data loading (`scripts/utils/classifier.py`)

Used by the auxiliary classifier, the ASISM proxy runs (04b, 07b, 08b), Stage 4 and Stage 5.
Original: `num_workers=0`, so every 768 px JPEG is decoded and resized inside the training process
while the GPU waits.

| Phase | A10 original | A10 optimized | Speedup | L4 original | L4 optimized | Speedup |
|---|---|---|---|---|---|---|
| Train 48 steps (batch 32) | 30.6 s | 12.7 s | 2.4x | 50.5 s | 23.2 s | 2.2x |
| Eval pass, 1536 images | 23.0 s | 4.7 s | 4.9x | 34.5 s | 10.9 s | 3.2x |
| MC Dropout, 5 x 256 images | 19.1 s | 6.5 s | 3.0x | 29.5 s | 13.5 s | 2.2x |
| GPU utilization (eval) | 17% | 73% | | 17% | — | |
| Peak VRAM | 8.1 GiB | 8.2 GiB | same | 8.1 GiB | 8.2 GiB | same |

**Equivalence.** The first 12 shuffled training batches are bit-identical (images, targets, masks,
order) on both GPUs. On L4 the trained weights and predictions are bit-identical to the original
(max abs difference 0.0). On A10, where the original itself is not bit-reproducible between two
runs (weights differ by up to 0.125), the optimized-vs-original difference (0.090) is inside that
noise floor. `tests/run_all.py`: 194 passed.

**GPU choice for Stages 3-5.** Cost for the same work: A10 $0.0050 per 48 training steps vs L4
$0.0072 (+44%); A10 $0.0019 per 1536-image evaluation vs L4 $0.0034 (+83%). **A10 is both faster
and cheaper.** T4 was not measured (no TF32/bf16, expected to be slower per dollar).

**Projected effect on the pipeline (A10, estimates from the throughputs above):**

| Workload | Original | Optimized |
|---|---|---|
| Auxiliary classifier (8000 steps + 11 evals of gen_val) | ~2.3 h | ~0.8 h |
| 04b utility subsets (121 proxy runs) | ~15.0 h | ~4.4 h |
| 08b full-policy verification (15 runs) | ~1.9 h | ~0.6 h |
| Stage 4 (9 runs x 6000 steps + 13 evals) | ~17.8 h | ~5.7 h |
| **Total** (07b not included) | **~37 h (~$52)** | **~11.5 h (~$16)** |

The original 04b estimate (~7.4 min per proxy run) agrees with the config's own
`hours_per_proxy_run_estimate: 0.15` (9 min), which supports the projection.

## 2. Stage 2 batched generation — rejected

SDXL 768 px, 40 DPM-Solver steps, CFG 7, A10.

| | One image per call (original) | Four per call |
|---|---|---|
| Seconds per image | 6.96 | 6.47 (1.07x) |
| GPU utilization | 99.7% | 99.7% |
| Peak VRAM | 8.8 GiB | 15.2 GiB |

The original is fully deterministic: two runs with the same seed produce byte-identical images.
Batched images start from the same noise but diverge during denoising (PSNR 23-33 dB, mean
absolute pixel difference 2.3-10.5), so a resumed run would no longer reproduce an uninterrupted
one. 7% is not worth that; the generator is unchanged.

## 3. Stage 1 caches

VAE latent cache, 96 images at 768 px, fp32 VAE, A10:

| | Per image (original) | Batch 4 | Batch 8 | Batch 16 |
|---|---|---|---|---|
| Time | 15.9 s | 15.1 s | 15.6 s | 23.0 s (OOM, fell back to 8) |
| Peak VRAM | 1.7 GiB | 6.0 GiB | 11.6 GiB | 18.4 GiB |

The original loop already runs the GPU at 94%, so batching recovers ~5%. Rejected. Freeing the
VAE and both text encoders after the caches are built is kept.

## 4. Stage 1 training step and gradient checkpointing

Trainer's own `build_models()` (SDXL UNet bf16, LoRA r32, EMA), 768 px, synthetic latents. Every
variant averages the loss over 16 samples per optimizer step.

| Variant (micro-batch x accumulation, GC) | A10 s/step | A10 VRAM | L4 s/step | L4 VRAM |
|---|---|---|---|---|
| **8 x 2, GC on (frozen)** | **5.10** | 8.8 GiB | 6.73 | 8.8 GiB |
| 16 x 1, GC on | 4.82 | 11.7 GiB | 7.20 | 11.7 GiB |
| 8 x 2, GC off | OOM (21.3 GiB) | — | — | — |
| 4 x 4, GC off | OOM | — | OOM | — |

GPU utilization 98-99% in every variant that fits. Without gradient checkpointing the step does not
fit in 24 GB even at micro-batch 4, so GC stays on. 16 x 1 is 5.5% faster on A10 but changes the
frozen batch geometry, so it is not applied.

**Stage 1 projection (15,000 steps, A10):** 21.2 h of training plus a one-time latent cache of
~0.17 s per image (~5 h for ~110k gen_train + gen_val images) ≈ 26 h ≈ $36-38 with 2 cores and
24-32 GiB RAM. L4 costs about the same in total but takes ~28 h of training.

## Storage

The pinned SDXL files the pipeline actually loads are 12.9 GiB (`download_models` fetches only
those; the full repository snapshot is ~70 GB). Modal Volumes include 1 TiB/month free, so the
dataset, caches and checkpoints add no storage cost.

---

# Part 16 - Audit log: contradictions found, and what was done

This notebook was built alongside a consistency audit of the whole repository. Recorded here so the
reasoning is not lost in the git log.

## Fixed: the F -> C rename was incomplete

Commit `a8691ce` renamed the ASISM-selected arm from **F** to **C** in the smoke config and smoke
scripts, but not in the documentation or the config comments. Left behind:

| Where | Said | Now says |
|---|---|---|
| `README.md` | "Primary comparison: **F vs. B**" | C vs. B |
| `docs/project_overview.md` | an **A/B/F/D** table, with D as a live arm, in a paragraph that then claimed "primary comparison C vs B" | A/B/C, with D explicitly marked INACTIVE |
| `docs/smoke_e2e.md` | "condition F", "`conditions: [A, B, F]` in the GPU overlay" | condition C; the overlay already said `[A, B, C]` |
| `docs/stages2_to_5_plan.md` | "Supporting comparisons: A vs. B and A vs. **F**" | A vs. C |
| `configs/stage4_classifier.yaml` | "B vs **F** (selected subset)"; the matched-random control "was matched to the removed condition **C**" and should be "re-pointed at condition **F**" | both inverted by the rename and corrected: C exists, F does not |
| 5 code docstrings, `configs/splits.yaml`, the plan's split tables | stale **A-E** / **A-G** condition ranges | A/B/C |

The code itself was already right: `01_train_conditions.py` and `compare_conditions.py` both record
`"C vs B"`. Only the prose had drifted. The `stages2_to_5_plan.md` note that records the *historical*
"F vs C" conflict is deliberately left as the record it is.

## Fixed: an accidentally deleted docstring

`scripts/asism/01_compute_signals.py` lost its 19-line module docstring in commit `0190fa6`, whose
message is *"Add new research papers for NeurIPS 2024 and 2025"* - the docstring deletion was the
commit's only change and is not mentioned. Because the CLI does
`argparse.ArgumentParser(description=__doc__)`, `--help` had been printing no description ever since.
Restored verbatim from `8b69933`.

Two related cases were repairs of *deliberate* removals that left dangling references:
`00c_derive_expected_regions.py` had a comment pointing at "SPLIT DISCIPLINE in the module docstring"
that no longer existed (the rule is now stated where it applies), and
`04_build_utility_subsets.py`'s `preflight_check_candidate_pool_inputs` had a whitespace-only line
where its docstring had been.

## Fixed: stale cross-references

- `03_preprocess_images.py` told the user to run `02_build_patient_splits.py` when a split file was
  missing - but it reads v2 namespaced splits, so the right command is
  `02b_build_sixway_splits.py --namespace <ns>`.
- `README.md`'s Stage 1 pipeline listed `02_build_patient_splits.py` with no namespace flags. The
  real sequence is `02b`, and `run_dev_subset.sh` is its executable form.
- `README.md`'s layout block listed only `scripts/{data,train,eval,utils}` - omitting `asism/`,
  `generate/`, `classify/` and `smoke/`, i.e. most of the thesis.
- `labels.py`'s `intended_vector_to_labels` docstring said agreement scores "all 12 predicted
  probabilities". Agreement is scored over the **11** `PRIMARY_ENDPOINT_LABELS` the function
  iterates; 12 is `GENERATION_TARGET_LABELS`, a different set.

## Removed: code with zero references

Each of these had exactly one occurrence in the repository - its own definition - and no test:

- **`scripts/utils/provenance_gate.py`, the whole module.** Its own docstring said "new Stage 2-5
  scripts should call `validate_upstream_artifact()`" - but Stage 2-5 was built on
  `artifact_contracts.require_manifest_fields` / `require_score_artifact` /
  `require_generation_complete`, which do the same job with per-field mismatch reporting.
  `docs/stages2_to_5_plan.md` 1.5 cited the unused helper as *the* enforcement mechanism and now
  names the ones actually in force.
- `splits.require_frozen_production_splits` - superseded by `require_frozen_production_split_run`.
- `experiment_registry.read_registry`
- `labels.SECONDARY_LABELS`
- `01_compute_signals.PRODUCED_SIGNALS`
- `01_sample_label_recipes.DEVICE_LABEL` - duplicated `caption_builder.DEVICE_COLUMN`
- `stage5_evaluate.file_hash` - the script uses `manifest.sha256_file` throughout

## Kept, but now labelled

Four helpers are unit-tested design alternatives that nothing on the current path calls. They were
kept - the reasoning is worth preserving - but each now says so in its own docstring instead of only
in the git history: `models.soft_selection_gate` (the pipeline thresholds hard),
`signals.duplicate_clusters` (near-duplicates are rejected individually),
`02b.partition_patients` (superseded by `multilabel_partition_patients`), and
`splits.freeze_patient_folds` / `build_patient_folds` (only `03_tune_freeze_select.py` calls them).

## Kept as-is: dormant by design, not dead

These look unused but are deliberate, and are documented as such in the code:

- **Condition D** and its whole matched-draw machinery (`build_matched_random_draw`, `profile_of`,
  the `condition_d` config block, `compare_conditions`' D handling). It is the documented fix for the
  thesis's one structural gap: C is a strict subset of B, so a C-over-B gain cannot be attributed to
  ASISM's *ranking* rather than to simply using *fewer* synthetic images. Re-enabling needs
  supervisor sign-off and `n_draws x 3` extra runs.
- **`06_learn_thresholds_select.py`** - the fixed-ratio threshold, an ablation reference, explicitly
  not a Stage 4 condition.
- **`03_tune_freeze_select.py`** - the historical weighted-score baseline. Not a pipeline step and
  not imported by anything; retained with that stated plainly.
- **`02_build_patient_splits.py`** - the superseded v1 three-way split builder, still imported by
  `01b_build_dev_subset.py` for its `extract_patient_id` helper (which `02b` duplicates identically).
- **The distinctiveness signal** - computed and Go/No-Go-gated, but its feature column is not yet in
  `configs/stage3_asism.yaml` -> `learned_asism.feature_columns`, so the ranker consumes 9 columns,
  not 10. A documented gap, consistent across `models.py`, the config and `project_overview.md`.
- **The superseded v1 split artifacts** and their `SUPERSEDED_v1.md` note - a deliberate audit trail.

## Still open, for a supervisor decision, not a code change

- **No matched-random control.** `docs/stages2_to_5_plan.md` 7.1, and stated in
  `01_train_conditions.py` itself. This is the one substantive limitation of the primary comparison.
- **Condition C has never been run end to end**, not even in smoke form, because the GPU smoke tier
  was built but never executed - no GPU was available. See `docs/smoke_e2e.md`.